# Phase 4 — Professional Python: Packaging, Typing, Testing and the Standard Library

**Outcome:** ship a typed, tested, logged and installable package, and reach for the
right standard-library or third-party module with confidence.

| Unit | Topic |
| --- | --- |
| 4.1 | Package management and project structure |
| 4.2 | The type system (Python's TypeScript) |
| 4.3 | Testing, debugging and logging |
| 4.4 | Standard library essentials |
| 4.5 | File I/O, encodings and serialization |
| 4.6 | Essential third-party libraries |
| Project | Publish the Log Analyzer |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- This phase is about **tools**: uv, pip, mypy, pyright, ruff and friends. Tool demos
  write small throwaway projects under `notebooks/_scratch/` (git-ignored) and run
  the real tool in a child process, printing its real output. Absolute paths are
  shown as `<scratch>` (the unit's scratch folder), `<repo>` or `~`, and timings as
  `<time>`.
- No cell changes the course's own `pyproject.toml` or `uv.lock` (a check at the end
  of Unit 4.1 proves it), and nothing is ever published.
- Packaging demos resolve against a **local wheelhouse** of tiny demo packages built
  in Unit 4.1, so they work offline and give the same answer every time. The only
  cells that need the network (a vulnerability audit and one third-party tool)
  check first and print a skip message when you are offline.
- Tool output can change with the tool's version; the versions used are printed.
- *Predict the output* questions hide the answer in a collapsible block: decide
  first, then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub (or fix the code, where
  the exercise says so), then run the check cell. Until then the checks print FAIL;
  that is expected.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.14*, …).

## What you need from Phases 1–3

Phase 4 turns notebook code into software other people install. If one of these
feels shaky, re-read the unit before going on.

| Earlier idea | Unit | What Phase 4 adds |
| --- | --- | --- |
| Virtual environments, `uv init`, `uv add`, `uv run` | 0.2–0.3 | The whole packaging toolchain: wheels, build backends, lockfiles, workspaces, publishing |
| Functions, type hints on every signature, `Callable` | 1.10–1.12 | Hints become a checked contract: mypy and pyright, generics, `ParamSpec`, `@overload` |
| Decorators (`timer`, `retry`) | 1.12 | Decorators that keep their wrapped function's signature for the type checker |
| Exceptions, `logging` basics | 1.14 | Testing, debugging and logging configuration (Unit 4.3) |
| Protocols, ABCs, dataclasses, enums | 2.3, 2.6 | Structural typing checked statically; `TypedDict`, `Literal`, discriminated unions |
| `Money`, the inventory model | 2.4, Project 2 | Typed to pass `mypy --strict` (Lab 4.2) |
| Packages, `sys.path`, `importlib.metadata`, entry points | 3.3 | `src/` layout, editable installs, `[project.scripts]`, single-sourced `__version__` |
| `pyinspect` (a package without packaging metadata) | Project 3 | The Phase 4 Project makes code like this installable |

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 4.1 — Package Management and Project Structure

**Objectives**

- Read a wheel filename and know whether it installs on a given interpreter; tell
  wheels from sdists and build frontends from build backends.
- Use pip's resolver, editable installs, constraints files and cache, and explain why
  `pip freeze` is not a lockfile.
- Drive a project end to end with uv: `init`, `add`, `lock`, `sync`, `run`, `tool`,
  inline-script metadata, workspaces, `export`, `build` and `publish`.
- Write every table of a `pyproject.toml` from memory, and PEP 440 version
  specifiers that mean what you intend.
- Choose a pinning strategy (application vs library), a project layout (`src/`) and a
  release scheme (SemVer or CalVer); keep the supply chain safe.

**JS bridge: npm and `package.json`**

| npm / Node.js | Python | The difference that bites |
| --- | --- | --- |
| registry.npmjs.org | PyPI (pypi.org) | A release holds several files: one sdist plus wheels per platform |
| `package.json` | `pyproject.toml` | One file for metadata, build config **and** tool settings (ruff, mypy, pytest) |
| `dependencies` | `[project] dependencies` | Same idea |
| `devDependencies` | `[dependency-groups] dev` (PEP 735) | Groups are never published in the package metadata |
| `optionalDependencies` / `peerDependencies` | extras: `[project.optional-dependencies]` | Extras are opt-in feature sets (`pkg[report]`), not "install if it builds" |
| `node_modules/`, nested per package | One flat `site-packages/` per virtual environment | Only **one** version of each package per environment: conflicts are errors, not nested copies |
| `package-lock.json`, `pnpm-lock.yaml` | `uv.lock`, `pylock.toml` (PEP 751) | `uv.lock` is cross-platform (one lock for Linux, macOS, Windows) |
| `npm ci` | `uv sync --locked` / `--frozen` | Same promise: install exactly the lock |
| `npx tool` | `uvx tool` | Same idea; uvx caches tool environments |
| `"bin"` | `[project.scripts]` | uv/pip generate the launcher scripts |
| `"scripts"` + `npm run` | no built-in task runner; `uv run <command>` | Use a Makefile, `just`, `poe` or plain `uv run` |
| `^1.2.3` (caret) | `~=1.2` or `>=1.2,<2` | PEP 440 is not SemVer: there is no caret |
| `.tgz` tarball | wheel (built, may be platform-specific) + sdist (source) | Installing a wheel runs no code; building an sdist does |
| `"engines": {"node": ">=20"}` | `requires-python = ">=3.12"` | The resolver enforces it |
| `npm link` | editable install (`uv sync`, `pip install -e`) | Implemented with a `.pth` file or an import hook |
| `npm audit` | `pip-audit` | Same idea, different database (PyPA advisory DB / OSV) |
| npm / pnpm workspaces | uv workspaces | One lockfile for all members |
| `npm publish` | `uv publish` | Prefer trusted publishing (OIDC) over tokens |

**How the demos run.** Every command runs in a child process inside
`notebooks/_scratch/unit41/` (shown as `<scratch>`), with a scratch uv cache and the
course's own environment variables removed, so nothing touches the course project.
The setup cell builds a **local wheelhouse**, a folder of tiny demo packages
(`rupee-fmt`, `gst-calc`, `table-print`, `win-colors`, `legacy-invoice`) in several
versions. Demo projects point uv and pip at it with `no-index` + `find-links`, the
standard way to install from a folder of wheels, so every resolution is real, offline
and repeatable.

| Demo package | Versions | Depends on |
| --- | --- | --- |
| `rupee-fmt` | 1.0.0, 1.1.0, 1.2.0, 2.0.0, 2.1.0rc1 | nothing (`fmt(amount)` changes output in 2.x) |
| `gst-calc` | 1.0.0, 2.0.0 | `rupee-fmt>=1.0,<2` (1.x) or `>=2` (2.x); `win-colors` on Windows only; extra `report` → `table-print`; console script `gst` |
| `table-print`, `win-colors` | 0.1.0, 1.0.0 | nothing |
| `legacy-invoice` | 1.0.0 | `rupee-fmt<1.1` |

In [2]:
import base64
import email.parser
import hashlib
import importlib.machinery
import importlib.metadata
import importlib.util
import json
import os
import platform
import re
import shlex
import shutil
import socket
import subprocess
import sys
import sysconfig
import tarfile
import textwrap
import tomllib
import zipfile
from collections.abc import Callable
from dataclasses import dataclass, field
from pathlib import Path

REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
UNIT41_DIR = scratch_dir / "unit41"
WHEELHOUSE = UNIT41_DIR / "wheelhouse"
UNIT41_DIR.mkdir(parents=True, exist_ok=True)
VENV_BIN = Path(sys.executable).parent
COURSE_FILES = ("pyproject.toml", "uv.lock")


def file_digest(path: Path) -> str:
    """SHA-256 of a file's bytes, as hex."""
    return hashlib.sha256(path.read_bytes()).hexdigest()


COURSE_DIGESTS_AT_START = {name: file_digest(REPO_ROOT / name) for name in COURSE_FILES}


def tool_env(**overrides: str) -> dict[str, str]:
    """Environment for child tools: plain output, scratch caches, nothing pointing at the course."""
    inherited = {
        key: value
        for key, value in os.environ.items()
        if not key.startswith(("UV_", "PIP_", "VIRTUAL_ENV", "CONDA_", "PYTHONPATH"))
    }
    return {
        **inherited,
        "NO_COLOR": "1",
        "PYTHON_COLORS": "0",
        "UV_NO_PROGRESS": "1",
        "UV_PYTHON": "3.15",
        "UV_PYTHON_DOWNLOADS": "never",
        "UV_CACHE_DIR": str(UNIT41_DIR / "uv-cache"),
        "UV_TOOL_DIR": str(UNIT41_DIR / "uv-tools"),
        "UV_TOOL_BIN_DIR": str(UNIT41_DIR / "uv-tools-bin"),
        "PIP_DISABLE_PIP_VERSION_CHECK": "1",
        "PIP_NO_COLOR": "1",
        "PIP_PROGRESS_BAR": "off",
        "PIP_CACHE_DIR": str(UNIT41_DIR / "pip-cache"),
        "GIT_CEILING_DIRECTORIES": str(scratch_dir),
        **overrides,
    }


TIMING = re.compile(r"\b(in|after) \d+(\.\d+)?(ms|s|µs)\b")
ANSI_ESCAPE = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")


def tidy(text: str) -> str:
    """Show absolute paths as <scratch>, <repo> or ~ and timings as <time>; drop colour codes."""
    for folder, label in ((UNIT41_DIR, "<scratch>"), (REPO_ROOT, "<repo>"), (Path.home(), "~")):
        text = text.replace(f"file://{folder}", label).replace(str(folder), label)
    return ANSI_ESCAPE.sub("", TIMING.sub(r"\1 <time>", text))


def executable(name: str) -> str | None:
    """A path as given, the kernel's python, a tool from the course venv, or one on PATH."""
    if "/" in name:
        return name if Path(name).exists() else None
    if name == "python":
        return sys.executable
    local_tool = VENV_BIN / name
    return str(local_tool) if local_tool.exists() else shutil.which(name)


def run_tool(
    *command: str, cwd: Path = UNIT41_DIR, env: dict[str, str] | None = None, timeout: int = 300
) -> tuple[int, str]:
    """Run a CLI tool in a child process; return (exit code, tidied stdout + stderr)."""
    found = executable(command[0])
    if found is None:
        return 127, f"SKIP: {command[0]} is not installed on this machine"
    completed = subprocess.run(
        [found, *command[1:]],
        cwd=cwd,
        env=env or tool_env(),
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    return completed.returncode, tidy((completed.stdout + completed.stderr).rstrip())


def shell(
    *command: str,
    cwd: Path = UNIT41_DIR,
    env: dict[str, str] | None = None,
    keep: str | None = None,
    max_lines: int | None = None,
) -> int:
    """Print `[folder]$ command` and the tool's real output (only lines matching keep, if given)."""
    exit_code, output = run_tool(*command, cwd=cwd, env=env)
    shown = [Path(command[0]).name if "/" in command[0] else command[0], *command[1:]]
    print(f"[{tidy(str(cwd))}]$ {tidy(shlex.join(shown))}")
    lines = output.splitlines()
    if keep is not None:
        lines = [line for line in lines if re.search(keep, line)]
    if max_lines is not None and len(lines) > max_lines:
        lines = [*lines[:max_lines], f"... ({len(lines) - max_lines} more lines)"]
    if lines:
        print(textwrap.indent("\n".join(lines), "  "))
    if exit_code:
        print(f"  (exit code {exit_code})")
    return exit_code


def fresh_dir(name: str) -> Path:
    """An empty folder <scratch>/<name>, recreated on every run."""
    folder = UNIT41_DIR / name
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    return folder


def write_files(root: Path, files: dict[str, str]) -> Path:
    """Write relative path -> text (dedented) under root; return root."""
    for relative_path, text in files.items():
        target = root / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(textwrap.dedent(text).lstrip("\n"))
    return root


SKIPPED_PARTS = {".venv", "__pycache__", ".pytest_cache", ".mypy_cache"}


def show_tree(root: Path) -> None:
    """Print every file under root (skipping virtual environments and caches)."""
    print(f"{tidy(str(root))}/")
    for path in sorted(root.rglob("*")):
        if path.is_file() and not SKIPPED_PARTS & set(path.relative_to(root).parts):
            print(f"  {path.relative_to(root).as_posix()}")


def show_file(path: Path, *, after: str | None = None) -> None:
    """Print a text file (optionally only from the first line containing after)."""
    text = path.read_text()
    if after is not None and after in text:
        text = text[text.index(after) :]
    print(f"--- {path.name} ---\n{tidy(text.rstrip())}")


@dataclass(frozen=True)
class DemoPackage:
    """A tiny package for the local wheelhouse (never published anywhere)."""

    name: str
    version: str
    source: str
    dependencies: tuple[str, ...] = ()
    extras: dict[str, list[str]] = field(default_factory=dict)
    scripts: dict[str, str] = field(default_factory=dict)

    @property
    def module(self) -> str:
        return self.name.replace("-", "_")

    def files(self) -> dict[str, str]:
        """pyproject.toml and src/<module>/__init__.py for this package."""
        lines = [
            "[project]",
            f'name = "{self.name}"',
            f'version = "{self.version}"',
            'description = "Course demo package (never published)"',
            'requires-python = ">=3.12"',
            'license = "MIT"',
            f"dependencies = {json.dumps(list(self.dependencies))}",
        ]
        if self.extras:
            lines += ["", "[project.optional-dependencies]"]
            lines += [f"{extra} = {json.dumps(reqs)}" for extra, reqs in self.extras.items()]
        if self.scripts:
            lines += ["", "[project.scripts]"]
            lines += [f'{command} = "{target}"' for command, target in self.scripts.items()]
        lines += ["", "[build-system]", 'requires = ["uv_build>=0.13,<0.14"]']
        lines += ['build-backend = "uv_build"']
        return {
            "pyproject.toml": "\n".join(lines) + "\n",
            f"src/{self.module}/__init__.py": textwrap.dedent(self.source).lstrip("\n"),
        }

    def wheel_name(self) -> str:
        return f"{self.module}-{self.version}-py3-none-any.whl"


RUPEE_V1 = """
    def fmt(amount: float) -> str:
        return f"INR {amount:,.2f}"
"""
RUPEE_V2 = """
    def fmt(amount: float, symbol: str = "\u20b9") -> str:
        return f"{symbol}{amount:,.2f}"
"""
GST_SOURCE = """
    import sys

    from rupee_fmt import fmt


    def with_gst(amount: float, rate: float = 0.18) -> float:
        return round(amount * (1 + rate), 2)


    def main() -> None:
        print(fmt(with_gst(float(sys.argv[1]))))
"""
WHEELHOUSE_PACKAGES = [
    *(DemoPackage("rupee-fmt", version, RUPEE_V1) for version in ("1.0.0", "1.1.0", "1.2.0")),
    *(DemoPackage("rupee-fmt", version, RUPEE_V2) for version in ("2.0.0", "2.1.0rc1")),
    DemoPackage(
        "table-print", "0.1.0", "def table(rows):\n    return '\\n'.join(map(str, rows))\n"
    ),
    DemoPackage("win-colors", "1.0.0", "RED = '\\x1b[31m'\n"),
    DemoPackage(
        "gst-calc",
        "1.0.0",
        GST_SOURCE,
        dependencies=("rupee-fmt>=1.0,<2", "win-colors>=1.0; sys_platform == 'win32'"),
        extras={"report": ["table-print>=0.1"]},
        scripts={"gst": "gst_calc:main"},
    ),
    DemoPackage(
        "gst-calc",
        "2.0.0",
        GST_SOURCE,
        dependencies=("rupee-fmt>=2",),
        extras={"report": ["table-print>=0.1"]},
        scripts={"gst": "gst_calc:main"},
    ),
    DemoPackage("legacy-invoice", "1.0.0", "TEMPLATE = 'plain'\n", dependencies=("rupee-fmt<1.1",)),
]


def publish_to_wheelhouse(package: DemoPackage) -> None:
    """Build one demo package (sdist + wheel) into the local wheelhouse with `uv build`."""
    source_root = write_files(
        fresh_dir(f"sources/{package.name}-{package.version}"), package.files()
    )
    exit_code, output = run_tool(
        "uv", "build", "--offline", "--out-dir", str(WHEELHOUSE), cwd=source_root
    )
    if exit_code:
        raise RuntimeError(f"uv build failed for {package.name} {package.version}:\n{output}")


def remove_from_wheelhouse(name: str, version: str) -> None:
    """Delete one release (wheel and sdist) from the local wheelhouse."""
    for path in WHEELHOUSE.glob(f"{name.replace('-', '_')}-{version}[-.]*"):
        path.unlink()


def build_wheelhouse() -> Path:
    """Build every demo package that is missing from the wheelhouse; return its folder."""
    WHEELHOUSE.mkdir(parents=True, exist_ok=True)
    for package in WHEELHOUSE_PACKAGES:
        if not (WHEELHOUSE / package.wheel_name()).exists():
            publish_to_wheelhouse(package)
    return WHEELHOUSE


def network_available(host: str = "pypi.org", port: int = 443, timeout: float = 3.0) -> bool:
    """Can we open a TCP connection to host? (Only two cells in this unit need the network.)"""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


build_wheelhouse()
print(f"Python {platform.python_version()}; {run_tool('uv', '--version')[1]}")
print(f"wheelhouse: {sorted(path.name for path in WHEELHOUSE.glob('*.whl'))}")

Python 3.15.1; uv 0.13.5 (x86_64-unknown-linux-gnu)
wheelhouse: ['gst_calc-1.0.0-py3-none-any.whl', 'gst_calc-2.0.0-py3-none-any.whl', 'legacy_invoice-1.0.0-py3-none-any.whl', 'rupee_fmt-1.0.0-py3-none-any.whl', 'rupee_fmt-1.1.0-py3-none-any.whl', 'rupee_fmt-1.2.0-py3-none-any.whl', 'rupee_fmt-2.0.0-py3-none-any.whl', 'rupee_fmt-2.1.0rc1-py3-none-any.whl', 'table_print-0.1.0-py3-none-any.whl', 'win_colors-1.0.0-py3-none-any.whl']


### 4.1.1 The ecosystem

**PyPI** (the Python Package Index) is Python's npm registry. A *project* has
*releases* (versions), and each release has one or more **distribution files**:

| File | Extension | What it is | Installing it |
| --- | --- | --- | --- |
| **sdist** (source distribution) | `.tar.gz` | Source code + `pyproject.toml` + `PKG-INFO` | Must be **built** first: runs the project's build backend, maybe a C compiler |
| **wheel** (built distribution) | `.whl` (a zip) | Files laid out ready to copy into `site-packages`, plus a `.dist-info/` folder | Unzip and copy: **no code runs** |

Installers prefer a wheel that matches the interpreter and fall back to the sdist.
Which wheel matches is decided by the three **tags** in its filename:

```text
{name}-{version}(-{build})?-{python tag}-{abi tag}-{platform tag}.whl
numpy-2.3.4-cp315-cp315-manylinux_2_28_x86_64.whl
            ^^^^^ ^^^^^ ^^^^^^^^^^^^^^^^^^^^^
            |     |     platform: Linux with glibc >= 2.28 on x86-64
            |     ABI: CPython 3.15's C ABI (cp315t = free-threaded build)
            interpreter: CPython 3.15
```

| Tag example | Meaning |
| --- | --- |
| `py3-none-any` | Pure Python, any Python 3, no C ABI, any OS: one wheel fits all |
| `cp315-cp315-manylinux_2_28_x86_64` | Compiled for CPython 3.15, Linux glibc ≥ 2.28, x86-64 |
| `cp315-cp315t-…` | The **free-threaded** (no-GIL) 3.15 build: the `t` ABI flag (Unit 3.1.5) |
| `cp312-abi3-…` | Uses only the **stable ABI** (`abi3`): one wheel for CPython 3.12 and every later version |
| `…-musllinux_1_2_aarch64` | Linux with musl libc (Alpine) on ARM64 |
| `…-macosx_11_0_arm64`, `…-win_amd64` | macOS on Apple Silicon, 64-bit Windows |

The next cell parses some (illustrative) wheel filenames with `packaging`, the
library pip and uv's ideas come from, and asks whether each would install **here**.

In [3]:
if importlib.util.find_spec("packaging") is None:
    print("SKIP: the packaging library is not installed")
else:
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename

    supported_here = set(sys_tags())
    example_wheels = [
        "requests-2.32.5-py3-none-any.whl",
        "numpy-2.3.4-cp315-cp315-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl",
        "numpy-2.3.4-cp315-cp315t-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl",
        "cryptography-46.0.3-cp311-abi3-manylinux_2_34_x86_64.whl",
        "pydantic_core-2.50.4-cp315-cp315-musllinux_1_2_aarch64.whl",
        "numpy-2.3.4-cp314-cp314-win_amd64.whl",
    ]
    for filename in example_wheels:
        name, version, _build, wheel_tags = parse_wheel_filename(filename)
        tag = sorted(wheel_tags, key=str)[0]
        installs = bool(wheel_tags & supported_here)
        print(
            f"{name:<14} {str(version):<8} {tag.interpreter:<6} {tag.abi:<7} "
            f"{tag.platform:<26} installs here: {installs}"
        )

requests       2.32.5   py3    none    any                        installs here: True
numpy          2.3.4    cp315  cp315   manylinux_2_27_x86_64      installs here: True
numpy          2.3.4    cp315  cp315t  manylinux_2_27_x86_64      installs here: False
cryptography   46.0.3   cp311  abi3    manylinux_2_34_x86_64      installs here: True
pydantic-core  2.50.4   cp315  cp315   musllinux_1_2_aarch64      installs here: False
numpy          2.3.4    cp314  cp314   win_amd64                  installs here: False


A wheel installs when **any** of its tags is in the interpreter's list of supported
tags, which is ordered from most to least specific. The installer picks the wheel
whose best tag comes first in this list. Here is the start of that list for this
kernel:

In [4]:
if importlib.util.find_spec("packaging") is not None:
    from packaging.tags import sys_tags

    all_tags = [str(tag) for tag in sys_tags()]
    print(f"{len(all_tags)} supported tags; the first eight and the last:")
    for tag in [*all_tags[:8], "...", all_tags[-1]]:
        print(f"  {tag}")
print(f"{sys.implementation.cache_tag=}  {sys.abiflags=}")
print(f"{sysconfig.get_config_var('SOABI')=}")

1437 supported tags; the first eight and the last:
  cp315-cp315-linux_x86_64
  cp315-cp315-manylinux_2_43_x86_64
  cp315-cp315-manylinux_2_42_x86_64
  cp315-cp315-manylinux_2_41_x86_64
  cp315-cp315-manylinux_2_40_x86_64
  cp315-cp315-manylinux_2_39_x86_64
  cp315-cp315-manylinux_2_38_x86_64
  cp315-cp315-manylinux_2_37_x86_64
  ...
  py30-none-any
sys.implementation.cache_tag='cpython-315'  sys.abiflags=''
sysconfig.get_config_var('SOABI')='cpython-315-x86_64-linux-gnu'


**manylinux.** Compiled Linux wheels must run on many distributions. A
`manylinux_2_28` wheel promises it only needs glibc 2.28 or newer, so it runs on any
distribution at least that recent. Your glibc version decides which `manylinux_2_X`
tags appear in the list above (`musllinux` is the same idea for Alpine's musl libc).

**abi3, the stable ABI.** An extension compiled against the *limited C API* uses only
the stable ABI and is tagged `cp3X-abi3`: one wheel serves that CPython version and
all later ones. Its files end in `.abi3.so`. The free-threaded build has a different
object layout, so it has its own ABI tag (`cp315t`) and suffixes; 3.15 adds `abi3t`,
a stable ABI for free-threaded builds (*New in 3.15*). The next cell asks
this kernel and, if one is installed, the uv-managed free-threaded 3.15 (Unit 3.1.5)
in a child process.

In [5]:
def find_free_threaded_python() -> str | None:
    """Path of an installed free-threaded 3.15 interpreter, or None."""
    env = {key: value for key, value in tool_env().items() if key != "UV_PYTHON"}
    found = subprocess.run(
        [executable("uv"), "python", "find", "3.15t"], capture_output=True, text=True, env=env
    )
    return found.stdout.strip() if found.returncode == 0 else None


ABI_PROBE = """
import importlib.machinery, platform, sys, sysconfig
soabi = sysconfig.get_config_var("SOABI")
print(platform.python_version(), f"abiflags={sys.abiflags!r}", f"SOABI={soabi}")
print("  extension suffixes:", importlib.machinery.EXTENSION_SUFFIXES)
"""
print("default build:", run_tool("python", "-c", ABI_PROBE)[1])
libc_name, libc_version = platform.libc_ver()
print(f"  C library: {libc_name} {libc_version} -> manylinux_2_X tags up to that glibc")
free_threaded_python = find_free_threaded_python()
if free_threaded_python is None:
    print("SKIP: no free-threaded 3.15 installed (uv python install 3.15t)")
else:
    print("free-threaded build:", run_tool(free_threaded_python, "-c", ABI_PROBE)[1])

default build: 3.15.1 abiflags='' SOABI=cpython-315-x86_64-linux-gnu
  extension suffixes: ['.cpython-315-x86_64-linux-gnu.so', '.abi3-x86_64-linux-gnu.so', '.abi3.so', '.abi3t-x86_64-linux-gnu.so', '.abi3t.so', '.so']
  C library: glibc 2.43 -> manylinux_2_X tags up to that glibc
free-threaded build: 3.15.1 abiflags='t' SOABI=cpython-315t-x86_64-linux-gnu
  extension suffixes: ['.cpython-315t-x86_64-linux-gnu.so', '.abi3t-x86_64-linux-gnu.so', '.abi3t.so', '.so']


**Build frontends and backends (PEP 517 / PEP 518).** Building a wheel is split
between two programs that talk through a small, standard Python API:

| Role | Job | Examples |
| --- | --- | --- |
| **Frontend** | Reads `[build-system]`, creates an isolated build environment, installs `requires`, calls the backend's hooks | pip, uv, `python -m build` |
| **Backend** | Turns the source tree into a wheel or sdist | `setuptools`, `hatchling`, `flit-core`, `pdm-backend`, `poetry-core`, `uv_build`, `maturin` (Rust), `scikit-build-core` (CMake) |

PEP 518 added `[build-system] requires` (what the build needs installed), PEP 517
the hooks (`build_wheel`, `build_sdist`, `get_requires_for_build_wheel`, …) and PEP
660 the editable-install hook (`build_editable`). Any frontend can build any project.

```mermaid
flowchart LR
    F["frontend (uv, pip)"] -->|"reads [build-system]"| P[pyproject.toml]
    F -->|"creates isolated env, installs requires"| E[build env]
    F -->|"calls build_wheel() / build_sdist()"| B["backend (uv_build, hatchling, ...)"]
    B -->|"writes"| W[".whl / .tar.gz"]
    F -->|"installs the wheel"| S[site-packages]
```

<details><summary>ASCII fallback</summary>

```text
frontend (uv, pip) --reads--> pyproject.toml [build-system]
         |--creates isolated env + installs `requires`
         |--calls build_wheel()/build_sdist()--> backend --writes--> .whl/.tar.gz
         '--installs the wheel--> site-packages
```

</details>

**Under the hood: a complete build backend in 40 lines.** A backend is just a module
with those functions. The next cell writes one (`tiny_backend.py`), a project that
uses it through `backend-path` (an in-tree backend, so `requires` is empty), and
builds it with uv as the frontend.

In [6]:
TINY_BACKEND = r"""
    # A minimal PEP 517 + PEP 660 build backend (pure Python, py3-none-any only).
    import base64, hashlib, io, os, tarfile, tomllib, zipfile
    from pathlib import Path

    PROJECT = tomllib.loads(Path("pyproject.toml").read_text())["project"]
    NAME, VERSION = PROJECT["name"], PROJECT["version"]
    MODULE = NAME.replace("-", "_")
    DIST_INFO = f"{MODULE}-{VERSION}.dist-info"
    METADATA = f"Metadata-Version: 2.4\nName: {NAME}\nVersion: {VERSION}\n"

    def _record_line(path, data):
        digest = base64.urlsafe_b64encode(hashlib.sha256(data).digest()).rstrip(b"=")
        return f"{path},sha256={digest.decode()},{len(data)}"

    def _wheel(directory, files):
        files = {**files, f"{DIST_INFO}/METADATA": METADATA.encode(),
                 f"{DIST_INFO}/WHEEL": b"Wheel-Version: 1.0\nGenerator: tiny\n"
                 b"Root-Is-Purelib: true\nTag: py3-none-any\n"}
        record = [_record_line(path, data) for path, data in files.items()]
        name = f"{MODULE}-{VERSION}-py3-none-any.whl"
        with zipfile.ZipFile(Path(directory) / name, "w") as wheel:
            for path, data in files.items():
                wheel.writestr(path, data)
            wheel.writestr(f"{DIST_INFO}/RECORD", "\n".join(record) + f"\n{DIST_INFO}/RECORD,,\n")
        return name

    def get_requires_for_build_wheel(config_settings=None):
        return []

    def build_wheel(wheel_directory, config_settings=None, metadata_directory=None):
        sources = {p.as_posix(): p.read_bytes() for p in Path(MODULE).rglob("*.py")}
        return _wheel(wheel_directory, sources)

    def build_editable(wheel_directory, config_settings=None, metadata_directory=None):
        # PEP 660: the "wheel" holds only a .pth file pointing at the source folder
        return _wheel(wheel_directory, {f"_{MODULE}_editable.pth": (os.getcwd() + "\n").encode()})

    def build_sdist(sdist_directory, config_settings=None):
        name = f"{MODULE}-{VERSION}.tar.gz"
        with tarfile.open(Path(sdist_directory) / name, "w:gz") as sdist:
            sources = ["pyproject.toml", "tiny_backend.py", *map(str, Path(MODULE).rglob("*.py"))]
            for path in sources:
                sdist.add(path, f"{MODULE}-{VERSION}/{path}")
            info = tarfile.TarInfo(f"{MODULE}-{VERSION}/PKG-INFO")
            info.size = len(METADATA)
            sdist.addfile(info, io.BytesIO(METADATA.encode()))
        return name
"""
tiny_root = write_files(
    fresh_dir("tiny-tax"),
    {
        "pyproject.toml": """
            [build-system]
            requires = []                    # nothing to install: the backend is in-tree
            build-backend = "tiny_backend"   # module that defines the hooks
            backend-path = ["."]             # ...found in the project folder itself

            [project]
            name = "tiny-tax"
            version = "0.1.0"
        """,
        "tiny_backend.py": TINY_BACKEND,
        "tiny_tax/__init__.py": "GST_RATE = 0.18\n",
    },
)
shell("uv", "build", "--offline", cwd=tiny_root)

[<scratch>/tiny-tax]$ uv build --offline
  Building source distribution...
  Successfully built dist/tiny_tax-0.1.0.tar.gz
  Successfully built dist/tiny_tax-0.1.0-py3-none-any.whl


0

uv (the frontend) built the sdist first, then a wheel *from* the unpacked sdist, which
proves the sdist is complete. The frontend's side of the protocol is just "import
the backend and call a hook". Calling the hooks yourself in a child interpreter shows
there is nothing more to it, and lists what each archive contains:

In [7]:
HOOK_CALLER = """
import sys, tarfile, zipfile
sys.path.insert(0, ".")          # backend-path = ["."]
import tiny_backend as backend    # build-backend = "tiny_backend"
print("get_requires_for_build_wheel ->", backend.get_requires_for_build_wheel())
wheel = backend.build_wheel("hook-dist")
sdist = backend.build_sdist("hook-dist")
print("build_wheel ->", wheel, zipfile.ZipFile(f"hook-dist/{wheel}").namelist())
print("build_sdist ->", sdist, tarfile.open(f"hook-dist/{sdist}").getnames())
"""
(tiny_root / "hook-dist").mkdir(exist_ok=True)
print(run_tool("python", "-c", HOOK_CALLER, cwd=tiny_root)[1])

get_requires_for_build_wheel -> []
build_wheel -> tiny_tax-0.1.0-py3-none-any.whl ['tiny_tax/__init__.py', 'tiny_tax-0.1.0.dist-info/METADATA', 'tiny_tax-0.1.0.dist-info/WHEEL', 'tiny_tax-0.1.0.dist-info/RECORD']
build_sdist -> tiny_tax-0.1.0.tar.gz ['tiny_tax-0.1.0/pyproject.toml', 'tiny_tax-0.1.0/tiny_backend.py', 'tiny_tax-0.1.0/tiny_tax/__init__.py', 'tiny_tax-0.1.0/PKG-INFO']


### 4.1.2 pip

pip is the reference installer that ships with CPython (through `ensurepip`). uv
projects do not include it, so the demo makes a classic virtual environment with
`python -m venv`, which installs pip from the wheel bundled with the interpreter
(no network needed). pip reads its options from `PIP_*` environment variables, so
the demo sets `PIP_NO_INDEX=1` and `PIP_FIND_LINKS=<wheelhouse>` once.

| npm | pip |
| --- | --- |
| `npm install pkg` | `pip install pkg` |
| `npm install` (from package.json) | `pip install -r requirements.txt` or `pip install .` |
| `npm ls` | `pip list`, `pip show pkg`, `pip freeze` |
| `npm link` | `pip install -e path` |

In [8]:
PIP_VENV = UNIT41_DIR / "pip-venv"
shutil.rmtree(PIP_VENV, ignore_errors=True)
print(run_tool("python", "-m", "venv", str(PIP_VENV))[1] or "venv created")
PIP_PYTHON = str(PIP_VENV / "bin" / "python")
PIP_ENV = tool_env(PIP_NO_INDEX="1", PIP_FIND_LINKS=str(WHEELHOUSE))


def pip(*arguments: str, cwd: Path = UNIT41_DIR, **options: object) -> int:
    """Run pip inside the scratch venv against the local wheelhouse."""
    return shell(PIP_PYTHON, "-m", "pip", *arguments, cwd=cwd, env=PIP_ENV, **options)


pip("--version")
pip("list")

venv created
[<scratch>]$ python -m pip --version
  pip 26.2.2 from <scratch>/pip-venv/lib/python3.15/site-packages/pip (python 3.15)
[<scratch>]$ python -m pip list
  Package Version
  ------- -------
  pip     26.2.2


0

**The resolver.** Since pip 20.3, pip uses a *backtracking* resolver: it picks
candidate versions, and when a later requirement conflicts it backs up and tries
older ones. Below, `gst-calc 1.0.0` accepts `rupee-fmt>=1.0,<2` and `legacy-invoice`
needs `rupee-fmt<1.1`, so the only answer is `rupee-fmt 1.0.0`. `--dry-run` resolves
without installing.

In [9]:
pip("install", "--dry-run", "gst-calc==1.0.0", "legacy-invoice", keep="Would install")

[<scratch>]$ python -m pip install --dry-run gst-calc==1.0.0 legacy-invoice
  Would install gst-calc-1.0.0 legacy-invoice-1.0.0 rupee-fmt-1.0.0


0

`gst-calc 2.0.0` needs `rupee-fmt>=2`, which can never satisfy `legacy-invoice`'s
`<1.1`. npm would nest two copies; Python has one flat environment, so the resolver
reports a conflict and installs nothing:

In [10]:
pip(
    "install",
    "--dry-run",
    "gst-calc==2.0.0",
    "legacy-invoice",
    keep=r"ERROR|requires|conflict|depends",
)

[<scratch>]$ python -m pip install --dry-run gst-calc==2.0.0 legacy-invoice
  The conflict is caused by:
      gst-calc 2.0.0 depends on rupee-fmt>=2
      legacy-invoice 1.0.0 depends on rupee-fmt<1.1
  Additionally, some packages in these conflicts have no matching distributions available for your environment:
  2. remove package versions to allow pip to attempt to solve the dependency conflict
  ERROR: Cannot install gst-calc==2.0.0 and legacy-invoice==1.0.0 because these package versions have conflicting dependencies.
  ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
  (exit code 1)


1

**Editable installs** (`pip install -e path`, PEP 660). The project is "installed"
but its code stays in the working folder, so edits show up without reinstalling,
like `npm link`. pip asks the backend's `build_editable` hook for a wheel; our tiny
backend's editable wheel holds a single `.pth` file naming the source folder, and
`site` adds that folder to `sys.path` at startup (Unit 3.3.5).

In [11]:
pip("install", "-e", str(tiny_root), keep="Successfully|ERROR")
site_packages = next(PIP_VENV.glob("lib/python3*/site-packages"))
for pth_file in site_packages.glob("_tiny*.pth"):
    print(f"{pth_file.name}: {tidy(pth_file.read_text().strip())}")
WHERE = "import tiny_tax; print(tiny_tax.__file__, tiny_tax.GST_RATE)"
print("before edit:", run_tool(PIP_PYTHON, "-c", WHERE)[1])
(tiny_root / "tiny_tax" / "__init__.py").write_text("GST_RATE = 0.12\n")
print("after edit: ", run_tool(PIP_PYTHON, "-c", WHERE)[1], "(no reinstall)")

[<scratch>]$ python -m pip install -e <scratch>/tiny-tax
  Successfully built tiny-tax
_tiny_tax_editable.pth: <scratch>/tiny-tax
before edit: <scratch>/tiny-tax/tiny_tax/__init__.py 0.18
after edit:  <scratch>/tiny-tax/tiny_tax/__init__.py 0.12 (no reinstall)


**`pip freeze` vs a lockfile.** `pip freeze` prints what is installed *right now*
in requirements format. It is a snapshot, not a lockfile:

| `pip freeze > requirements.txt` | A real lockfile (`uv.lock`, `pylock.toml`) |
| --- | --- |
| Only this machine's platform and Python | Resolved for every platform and marker |
| No hashes (unless you add them) | Hashes for every file |
| Mixes runtime and dev tools, and anything installed by hand | Generated from declared dependencies and groups |
| No record of *why* a package is there | Records the dependency graph |
| Editable installs appear as paths | Editable sources recorded properly |

In [12]:
pip("install", "gst-calc==1.0.0", keep="Successfully")
pip("freeze")

[<scratch>]$ python -m pip install gst-calc==1.0.0
[<scratch>]$ python -m pip freeze
  gst-calc==1.0.0
  rupee-fmt==1.2.0
  # Editable install with no version control (tiny-tax==0.1.0)
  -e <scratch>/tiny-tax


0

**Constraints files** (`-c constraints.txt`) cap versions *without* asking for
anything to be installed: a constraint only matters if something else pulls that
package in. Teams use one shared constraints file to keep many services on the same
versions. Here a constraint holds `rupee-fmt` below 1.2 even though `gst-calc`
allows up to `<2`:

In [13]:
(UNIT41_DIR / "constraints.txt").write_text("rupee-fmt<1.2\ntable-print==0.1.0\n")
pip(
    "install",
    "--dry-run",
    "--ignore-installed",
    "-c",
    "constraints.txt",
    "gst-calc==1.0.0",
    keep="Would install",
)

[<scratch>]$ python -m pip install --dry-run --ignore-installed -c constraints.txt gst-calc==1.0.0
  Would install gst-calc-1.0.0 rupee-fmt-1.1.0


0

**The cache.** pip caches downloaded files (HTTP cache) and wheels it built from
sdists, so a second install is fast. `pip cache dir / info / list / purge` manage
it. The demo's `PIP_CACHE_DIR` points into the scratch folder; your real one is
usually `~/.cache/pip`. Installing from local wheels stores nothing, so the cache is
empty. uv's cache (next section) works differently.

In [14]:
pip("cache", "dir")
pip("cache", "info")

[<scratch>]$ python -m pip cache dir
  <scratch>/pip-cache
[<scratch>]$ python -m pip cache info
  Package index page cache location (pip v23.3+): <scratch>/pip-cache/http-v2
  Package index page cache location (older pips): <scratch>/pip-cache/http
  Package index page cache size: 0 bytes
  Number of HTTP files: 0
  Locally built wheels location: <scratch>/pip-cache/wheels
  Locally built wheels size: 0 bytes
  Number of locally built wheels: 0


0

### 4.1.3 uv in depth

uv (from Astral, written in Rust) replaces pip, pip-tools, virtualenv, pipx, pyenv
and twine with one fast binary. You met `uv init`, `uv add` and `uv run` in Unit
0.3; this section covers the rest.

**The global cache.** uv keeps one cache per machine (`uv cache dir`). Installing a
package into a virtual environment **hardlinks** (or copy-on-write clones) files
out of the cache instead of copying them, like pnpm's content-addressed store. Ten
projects using the same package share one copy on disk. The first command below
asks uv for your real cache location; the demos use a scratch cache.

In [15]:
real_env = {key: value for key, value in tool_env().items() if key != "UV_CACHE_DIR"}
shell("uv", "cache", "dir", env=real_env)
shell("uv", "cache", "dir")
shell("uv", "cache", "size", "--human", keep="iB$")  # experimental command in uv 0.13

[<scratch>]$ uv cache dir
  ~/.cache/uv
[<scratch>]$ uv cache dir
  <scratch>/uv-cache
[<scratch>]$ uv cache size --human
  13.7MiB


0

**`uv init`: three kinds of project.** All three create a `pyproject.toml`; they
differ in whether the project is *built and installed* into its own environment.

| Command | Layout | `[build-system]` | Use it for |
| --- | --- | --- | --- |
| `uv init --app --no-package` | `main.py` at the root | none (a *virtual* project: only its dependencies are installed) | Scripts, notebooks, web services you never `pip install` |
| `uv init --app` (packaged on this uv) or `--package` | `src/<name>/__init__.py` + `[project.scripts]` | yes | CLIs and apps that need an entry point |
| `uv init --lib` | `src/<name>/__init__.py` + `py.typed` | yes | Libraries others import |

The course itself is a virtual project (`uv init --bare`, Unit 0.3). Every `uv init`
below passes `--no-workspace` (otherwise uv would offer to add the scratch project
to an enclosing workspace), `--author-from none` (otherwise uv copies your git name
and e-mail into the file) and `--vcs none`.

In [16]:
UV_INIT = ("uv", "init", "--no-workspace", "--author-from", "none", "--vcs", "none", "--no-readme")
init_root = fresh_dir("init-demo")
for kind, project in [
    ("--app --no-package", "shop-app"),
    ("--app", "shop-cli"),
    ("--package", "invoicing-cli"),
    ("--lib", "price-lib"),
]:
    shell(*UV_INIT, *kind.split(), project, cwd=init_root)
show_tree(init_root)

[<scratch>/init-demo]$ uv init --no-workspace --author-from none --vcs none --no-readme --app --no-package shop-app
  Initialized project `shop-app` at `<scratch>/init-demo/shop-app`
[<scratch>/init-demo]$ uv init --no-workspace --author-from none --vcs none --no-readme --app shop-cli
  Initialized project `shop-cli` at `<scratch>/init-demo/shop-cli`
[<scratch>/init-demo]$ uv init --no-workspace --author-from none --vcs none --no-readme --package invoicing-cli
  Initialized project `invoicing-cli` at `<scratch>/init-demo/invoicing-cli`
[<scratch>/init-demo]$ uv init --no-workspace --author-from none --vcs none --no-readme --lib price-lib
  Initialized project `price-lib` at `<scratch>/init-demo/price-lib`
<scratch>/init-demo/
  invoicing-cli/.python-version
  invoicing-cli/pyproject.toml
  invoicing-cli/src/invoicing_cli/__init__.py
  price-lib/.python-version
  price-lib/pyproject.toml
  price-lib/src/price_lib/__init__.py
  price-lib/src/price_lib/py.typed
  shop-app/.python-version


Compare the generated `pyproject.toml` files of a virtual app and a library:

In [17]:
show_file(init_root / "shop-app" / "pyproject.toml")
show_file(init_root / "price-lib" / "pyproject.toml")
show_file(init_root / "invoicing-cli" / "src" / "invoicing_cli" / "__init__.py")

--- pyproject.toml ---
[project]
name = "shop-app"
version = "0.1.0"
description = "Add your description here"
requires-python = ">=3.15"
dependencies = []
--- pyproject.toml ---
[project]
name = "price-lib"
version = "0.1.0"
description = "Add your description here"
requires-python = ">=3.15"
dependencies = []

[build-system]
requires = ["uv_build>=0.13.5,<0.14.0"]
build-backend = "uv_build"
--- __init__.py ---
def main() -> None:
    print("Hello from invoicing-cli!")


**A working project: `invoicing`.** The rest of this section drives one packaged
project. First it is pointed at the wheelhouse through a `[tool.uv]` table (the
same settings a company uses for a private index). Before locking, the newest
`rupee-fmt` 1.x (1.2.0) is removed from the wheelhouse; it is "released" again
later to show how upgrades work.

In [18]:
projects_root = fresh_dir("projects")
shell(*UV_INIT, "--package", "invoicing", cwd=projects_root)
invoicing = projects_root / "invoicing"
with (invoicing / "pyproject.toml").open("a") as pyproject_file:
    pyproject_file.write(
        "\n[tool.uv]\nno-index = true               # never contact PyPI\n"
        'find-links = ["../../wheelhouse"]   # resolve from the local wheelhouse\n'
    )
remove_from_wheelhouse("rupee-fmt", "1.2.0")
print(sorted(path.name for path in WHEELHOUSE.glob("rupee_fmt*.whl")))

[<scratch>/projects]$ uv init --no-workspace --author-from none --vcs none --no-readme --package invoicing
  Initialized project `invoicing` at `<scratch>/projects/invoicing`
['rupee_fmt-1.0.0-py3-none-any.whl', 'rupee_fmt-1.1.0-py3-none-any.whl', 'rupee_fmt-2.0.0-py3-none-any.whl', 'rupee_fmt-2.1.0rc1-py3-none-any.whl']


**`uv add` with version specifiers.** `uv add` edits `pyproject.toml`, re-locks and
syncs the environment in one step. Quote specifiers so the shell does not treat
`<` or `>` as redirection. The four flags decide *where* the requirement goes:

| Command | Written to | Installed by `uv sync`? | Published in the wheel? |
| --- | --- | --- | --- |
| `uv add "gst-calc~=1.0"` | `[project] dependencies` | yes | yes (`Requires-Dist`) |
| `uv add --dev pkg` | `[dependency-groups] dev` | yes (default group) | no |
| `uv add --group docs pkg` | `[dependency-groups] docs` | only with `--group docs` | no |
| `uv add --optional report pkg` | `[project.optional-dependencies] report` | only with `--extra report` | yes, as an extra |

In [19]:
shell("uv", "add", "gst-calc~=1.0", cwd=invoicing)
shell("uv", "add", "--dev", "rupee-fmt>=1.1", cwd=invoicing)
shell("uv", "add", "--group", "docs", "table-print", cwd=invoicing)
shell("uv", "add", "--optional", "report", "table-print>=0.1", cwd=invoicing)

[<scratch>/projects/invoicing]$ uv add 'gst-calc~=1.0'
  Using CPython 3.15.1
  Creating virtual environment at: .venv
  Resolved 4 packages in <time>
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Prepared 1 package in <time>
  Installed 3 packages in <time>
   + gst-calc==1.0.0
   + invoicing==0.1.0 (from <scratch>/projects/invoicing)
   + rupee-fmt==1.1.0
[<scratch>/projects/invoicing]$ uv add --dev 'rupee-fmt>=1.1'
  Resolved 4 packages in <time>
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Prepared 1 package in <time>
  Uninstalled 1 package in <time>
  Installed 1 package in <time>
   ~ invoicing==0.1.0 (from <scratch>/projects/invoicing)
[<scratch>/projects/invoicing]$ uv add --group docs table-print
  Resolved 5 packages in <time>
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Prepare

0

The four tables uv wrote (note that `uv add --dev rupee-fmt>=1.1` kept the user's
specifier, while a bare name gets a lower bound from the version it resolved):

In [20]:
show_file(invoicing / "pyproject.toml", after="dependencies")

--- pyproject.toml ---
dependencies = [
    "gst-calc~=1.0",
]

[project.scripts]
invoicing = "invoicing:main"

[project.optional-dependencies]
report = [
    "table-print>=0.1",
]

[build-system]
requires = ["uv_build>=0.13.5,<0.14.0"]
build-backend = "uv_build"

[tool.uv]
no-index = true               # never contact PyPI
find-links = ["../../wheelhouse"]   # resolve from the local wheelhouse

[dependency-groups]
dev = [
    "rupee-fmt>=1.1",
]
docs = [
    "table-print>=0.1.0",
]


**`uv lock --upgrade-package`.** A lockfile is *stable*: `uv lock` keeps every
locked version that still satisfies `pyproject.toml`, even when newer ones exist.
Below, `rupee-fmt 1.2.0` is "released" into the wheelhouse. Plain `uv lock` changes
nothing; `uv lock --upgrade-package rupee-fmt` moves just that package (and only as
far as the specifiers allow, so 2.x is never chosen). `uv lock --upgrade` would move
everything.

In [21]:
def locked_rupee() -> str:
    """The rupee-fmt version recorded in invoicing/uv.lock."""
    lock_text = (invoicing / "uv.lock").read_text()
    return re.search(r'name = "rupee-fmt"\nversion = "([^"]+)"', lock_text)[1]


print(f"locked before the release: rupee-fmt {locked_rupee()}")
publish_to_wheelhouse(next(p for p in WHEELHOUSE_PACKAGES if p.version == "1.2.0"))
shell("uv", "lock", cwd=invoicing)
print(f"after plain `uv lock`:     rupee-fmt {locked_rupee()}")
shell("uv", "lock", "--upgrade-package", "rupee-fmt", cwd=invoicing)
print(f"after --upgrade-package:   rupee-fmt {locked_rupee()}")

locked before the release: rupee-fmt 1.1.0
[<scratch>/projects/invoicing]$ uv lock
  Resolved 5 packages in <time>
after plain `uv lock`:     rupee-fmt 1.1.0


[<scratch>/projects/invoicing]$ uv lock --upgrade-package rupee-fmt
  Resolved 5 packages in <time>
  Updated rupee-fmt v1.1.0 -> v1.2.0
after --upgrade-package:   rupee-fmt 1.2.0


**`uv sync --locked` vs `--frozen`.** Both promise not to change the lockfile, which
is what CI and Docker builds want:

| Flag | If `pyproject.toml` and `uv.lock` disagree | Use it |
| --- | --- | --- |
| (none) | Re-locks, then installs | Day-to-day development |
| `--locked` | **Fails** with an error | CI: catch a forgotten `uv lock` |
| `--frozen` | Installs the lock **as is**, without checking `pyproject.toml` | Docker layers that copy only `uv.lock`, or when resolution is impossible here |

The demo edits `pyproject.toml` by hand (adding `win-colors` for every platform)
without re-locking:

In [22]:
pyproject_path = invoicing / "pyproject.toml"
original_pyproject = pyproject_path.read_text()
pyproject_path.write_text(
    original_pyproject.replace('"gst-calc~=1.0",', '"gst-calc~=1.0",\n    "win-colors",')
)
shell("uv", "sync", "--locked", cwd=invoicing)
shell("uv", "sync", "--frozen", cwd=invoicing)
pyproject_path.write_text(original_pyproject)  # undo the hand edit
shell("uv", "sync", "--locked", cwd=invoicing)

[<scratch>/projects/invoicing]$ uv sync --locked
  Resolved 5 packages in <time>
  error: The lockfile at `uv.lock` needs to be updated, but `--locked` was provided.

  hint: To update the lockfile, run `uv lock`.
  (exit code 1)


[<scratch>/projects/invoicing]$ uv sync --frozen
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Prepared 1 package in <time>
  Uninstalled 3 packages in <time>
  Installed 2 packages in <time>
   ~ invoicing==0.1.0 (from <scratch>/projects/invoicing)
   - rupee-fmt==1.1.0
   + rupee-fmt==1.2.0
   - table-print==0.1.0


[<scratch>/projects/invoicing]$ uv sync --locked
  Resolved 5 packages in <time>
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Prepared 1 package in <time>
  Uninstalled 1 package in <time>
  Installed 1 package in <time>
   ~ invoicing==0.1.0 (from <scratch>/projects/invoicing)


0

`uv run` syncs first, then runs a command inside the project environment. The
project's console script (`invoicing = "invoicing:main"` in `[project.scripts]`) is
installed as a command. Edit `main()` to use the dependency and run it again; no
reinstall is needed because the project itself is installed *editable*.

In [23]:
shell("uv", "run", "invoicing", cwd=invoicing)
write_files(
    invoicing / "src" / "invoicing",
    {
        "__init__.py": """
    import sys

    from gst_calc import with_gst
    from rupee_fmt import fmt


    def main() -> None:
        amount = float(sys.argv[1]) if len(sys.argv) > 1 else 100.0
        print(f"{fmt(amount)} + GST = {fmt(with_gst(amount))}")
"""
    },
)
shell("uv", "run", "invoicing", "2500", cwd=invoicing)

[<scratch>/projects/invoicing]$ uv run invoicing
  Hello from invoicing!
[<scratch>/projects/invoicing]$ uv run invoicing 2500
  INR 2,500.00 + GST = INR 2,950.00


0

**`uv tool` and `uvx`** are uv's pipx and npx. `uv tool install` puts a CLI into
its own isolated environment and links its commands into a bin folder;
`uvx` (short for `uv tool run`) runs a tool in a cached throwaway environment. The
demo redirects `UV_TOOL_DIR` and `UV_TOOL_BIN_DIR` into the scratch folder, so
nothing lands in your real `~/.local/bin`.

| npm | uv |
| --- | --- |
| `npm install -g pkg` | `uv tool install pkg` |
| `npx pkg` | `uvx pkg` (`uvx --from pkg command` when names differ) |
| `npm ls -g` | `uv tool list` |

In [24]:
shutil.rmtree(UNIT41_DIR / "uv-tools", ignore_errors=True)
shell(
    "uv",
    "tool",
    "install",
    "--offline",
    "../wheelhouse/gst_calc-1.0.0-py3-none-any.whl",
    "--no-index",
    "--find-links",
    "../wheelhouse",
    cwd=projects_root,
)
shell("uv", "tool", "list")
shell(str(UNIT41_DIR / "uv-tools-bin" / "gst"), "1000")
shell(
    "uvx",
    "--offline",
    "--no-index",
    "--find-links",
    "wheelhouse",
    "--from",
    "gst-calc==2.0.0",
    "gst",
    "1000",
)
shell("uv", "tool", "uninstall", "gst-calc")

[<scratch>/projects]$ uv tool install --offline ../wheelhouse/gst_calc-1.0.0-py3-none-any.whl --no-index --find-links ../wheelhouse
  Resolved 2 packages in <time>
  Installed 2 packages in <time>
   + gst-calc==1.0.0 (from <scratch>/wheelhouse/gst_calc-1.0.0-py3-none-any.whl)
   + rupee-fmt==1.2.0
  Installed 1 executable: gst
[<scratch>]$ uv tool list
  gst-calc v1.0.0
  - gst


[<scratch>]$ gst 1000
  INR 1,180.00


[<scratch>]$ uvx --offline --no-index --find-links wheelhouse --from gst-calc==2.0.0 gst 1000
  ₹1,180.00
[<scratch>]$ uv tool uninstall gst-calc
  Uninstalled 1 executable: gst


0

**PEP 723 inline script metadata.** A single-file script can declare its own
dependencies in a special comment block; `uv run script.py` reads it and runs the
script in a cached environment that has exactly those packages. No project, no
virtual environment to manage: ideal for one-off tools. `uv init --script` creates
the block and `uv add --script` edits it.

```python
# /// script
# requires-python = ">=3.12"
# dependencies = ["rupee-fmt>=2"]
# ///
```

In [25]:
scripts_root = fresh_dir("scripts")
shell("uv", "init", "--script", "report.py", cwd=scripts_root)
FIND_LINKS = ("--no-index", "--find-links", str(WHEELHOUSE))
shell("uv", "add", "--script", "report.py", "rupee-fmt>=2", *FIND_LINKS, cwd=scripts_root)
report_script = scripts_root / "report.py"
report_script.write_text(
    report_script.read_text().replace(
        'print("Hello from report.py!")',
        "import importlib.metadata\n    from rupee_fmt import fmt\n\n"
        '    print(fmt(1234567.5), "with rupee-fmt", importlib.metadata.version("rupee-fmt"))',
    )
)
show_file(report_script)
shell("uv", "run", *FIND_LINKS, "report.py", cwd=scripts_root)

[<scratch>/scripts]$ uv init --script report.py
  Initialized script at `report.py`
[<scratch>/scripts]$ uv add --script report.py 'rupee-fmt>=2' --no-index --find-links <scratch>/wheelhouse
  Resolved 1 package in <time>
--- report.py ---
# /// script
# requires-python = ">=3.15"
# dependencies = [
#     "rupee-fmt>=2",
# ]
# ///


def main() -> None:
    import importlib.metadata
    from rupee_fmt import fmt

    print(fmt(1234567.5), "with rupee-fmt", importlib.metadata.version("rupee-fmt"))


if __name__ == "__main__":
    main()


[<scratch>/scripts]$ uv run --no-index --find-links <scratch>/wheelhouse report.py
  ₹1,234,567.50 with rupee-fmt 2.0.0


0

**Workspaces** are uv's version of npm/pnpm workspaces: several packages in one
repository share **one** `uv.lock` (so they agree on every version) and one
environment. The root declares `[tool.uv.workspace] members`; a member depends on
another with `[tool.uv.sources] name = { workspace = true }`, and both are installed
editable. Here `uv init` is run *without* `--no-workspace` inside the root, so it
adds each member automatically.

In [26]:
monorepo = write_files(
    fresh_dir("monorepo"),
    {
        "pyproject.toml": """
    [project]
    name = "monorepo"
    version = "0.1.0"
    requires-python = ">=3.15"

    [tool.uv.workspace]
    members = ["packages/*"]
"""
    },
)
UV_INIT_MEMBER = ("uv", "init", "--author-from", "none", "--vcs", "none", "--no-readme")
shell(*UV_INIT_MEMBER, "--lib", "packages/pricing-core", cwd=monorepo)
shell(*UV_INIT_MEMBER, "--package", "packages/billing-api", cwd=monorepo)
shell("uv", "add", "--offline", "--package", "billing-api", "pricing-core", cwd=monorepo)
show_file(monorepo / "packages" / "billing-api" / "pyproject.toml", after="dependencies")
shell(
    "uv",
    "run",
    "--offline",
    "--package",
    "billing-api",
    "python",
    "-c",
    "import pricing_core; print(pricing_core.hello())",
    cwd=monorepo,
)
workspace_lock = tomllib.loads((monorepo / "uv.lock").read_text())
print(f"one uv.lock for the workspace, members: {workspace_lock['manifest']['members']}")

[<scratch>/monorepo]$ uv init --author-from none --vcs none --no-readme --lib packages/pricing-core
  Project `pricing-core` is already a member of workspace `<scratch>/monorepo`
  Initialized project `pricing-core` at `<scratch>/monorepo/packages/pricing-core`


[<scratch>/monorepo]$ uv init --author-from none --vcs none --no-readme --package packages/billing-api
  Project `billing-api` is already a member of workspace `<scratch>/monorepo`
  Initialized project `billing-api` at `<scratch>/monorepo/packages/billing-api`


[<scratch>/monorepo]$ uv add --offline --package billing-api pricing-core
  Using CPython 3.15.1
  Creating virtual environment at: .venv
  Resolved 3 packages in <time>
     Building billing-api @ <scratch>/monorepo/packages/billing-api
     Building pricing-core @ <scratch>/monorepo/packages/pricing-core
        Built pricing-core @ <scratch>/monorepo/packages/pricing-core
        Built billing-api @ <scratch>/monorepo/packages/billing-api
  Prepared 2 packages in <time>
  Installed 2 packages in <time>
   + billing-api==0.1.0 (from <scratch>/monorepo/packages/billing-api)
   + pricing-core==0.1.0 (from <scratch>/monorepo/packages/pricing-core)
--- pyproject.toml ---
dependencies = [
    "pricing-core",
]

[project.scripts]
billing-api = "billing_api:main"

[build-system]
requires = ["uv_build>=0.13.5,<0.14.0"]
build-backend = "uv_build"

[tool.uv.sources]
pricing-core = { workspace = true }


[<scratch>/monorepo]$ uv run --offline --package billing-api python -c 'import pricing_core; print(pricing_core.hello())'
  Hello from pricing-core!
one uv.lock for the workspace, members: ['billing-api', 'monorepo', 'pricing-core']


**`uv export`** converts `uv.lock` into formats other tools understand:
`requirements.txt` (for pip, Dependabot, older platforms), `pylock.toml` (the PEP 751
standard lockfile, 4.1.6) or a CycloneDX SBOM (`cyclonedx1.5`, for security teams).

In [27]:
shell("uv", "export", "--format", "requirements.txt", "--no-hashes", cwd=invoicing)
shell(
    "uv",
    "export",
    "--format",
    "requirements.txt",
    "--no-dev",
    "--extra",
    "report",
    "--no-hashes",
    "--no-header",
    cwd=invoicing,
)

[<scratch>/projects/invoicing]$ uv export --format requirements.txt --no-hashes
  # This file was autogenerated by uv via the following command:
  #    uv export --format requirements.txt --no-hashes
  -e .
  gst-calc==1.0.0
      # via invoicing
  rupee-fmt==1.2.0
      # via gst-calc
  win-colors==1.0.0 ; sys_platform == 'win32'
      # via gst-calc
  Resolved 5 packages in <time>
[<scratch>/projects/invoicing]$ uv export --format requirements.txt --no-dev --extra report --no-hashes --no-header
  -e .
  gst-calc==1.0.0
      # via invoicing
  rupee-fmt==1.2.0
      # via gst-calc
  table-print==0.1.0
      # via invoicing
  win-colors==1.0.0 ; sys_platform == 'win32'
      # via gst-calc
  Resolved 5 packages in <time>


0

**`uv build`** is a build frontend (4.1.1): it builds an sdist into `dist/`, then a
wheel from that sdist. `uv_build` projects are built by uv's built-in copy of the
backend, with no extra download. 4.1.8 opens both archives.

In [28]:
shell("uv", "build", "--offline", cwd=invoicing)
print(sorted(path.name for path in (invoicing / "dist").iterdir()))

[<scratch>/projects/invoicing]$ uv build --offline
  Building source distribution...
  Successfully built dist/invoicing-0.1.0.tar.gz
  Successfully built dist/invoicing-0.1.0-py3-none-any.whl
['.gitignore', 'invoicing-0.1.0-py3-none-any.whl', 'invoicing-0.1.0.tar.gz']


**`uv publish`** uploads `dist/*` to an index (PyPI by default; `--index` or
`--publish-url` for TestPyPI or a private index). The course never publishes, so
this cell only prints the relevant options from `uv publish --help`. 4.1.8 shows the
full release flow with TestPyPI and trusted publishing.

In [29]:
shell(
    "uv",
    "publish",
    "--help",
    keep=r"^\s+(--index |--publish-url|--trusted-publishing|--check-url|--dry-run|\[FILES\])",
)

[<scratch>]$ uv publish --help
    [FILES]...  Paths to the files to upload. Accepts glob expressions [default: dist/*]
        --index <INDEX>
        --trusted-publishing <TRUSTED_PUBLISHING>
        --publish-url <PUBLISH_URL>
        --check-url <CHECK_URL>
        --dry-run


0

### 4.1.4 `pyproject.toml` anatomy

`pyproject.toml` is TOML (like `package.json` with comments and typed tables). Its
tables come from different standards:

| Table | Standard | Read by | npm analogue |
| --- | --- | --- | --- |
| `[build-system]` | PEP 518 / 517 | build frontends (pip, uv) | none (npm has no build step) |
| `[project]` | PEP 621 | backends, which turn it into wheel `METADATA` | `name`, `version`, `dependencies`, `engines`, `license` |
| `[project.scripts]` | PEP 621 (entry points) | installers, which generate launchers | `"bin"` |
| `[project.optional-dependencies]` | PEP 621 | resolvers (`pkg[extra]`) | `optionalDependencies` / `peerDependencies`, sort of |
| `[project.urls]` | PEP 621 | PyPI's sidebar | `homepage`, `repository` |
| `[dependency-groups]` | PEP 735 | uv, pip ≥ 25.1 (`--group`) | `devDependencies` |
| `[tool.<name>]` | each tool's own docs | uv, ruff, mypy, pytest, … | `"eslintConfig"`, `"jest"` keys |

Here is a complete, annotated example for a CLI called `invoice-kit`:

In [30]:
ANATOMY_PYPROJECT = """
[build-system]                                  # PEP 518: how to build this project
requires = ["uv_build>=0.13,<0.14"]             # installed into the isolated build env
build-backend = "uv_build"                      # module whose PEP 517 hooks build it

[project]                                       # PEP 621: the package metadata
name = "invoice-kit"                            # PyPI name (normalized: invoice_kit)
version = "1.4.0"                               # PEP 440 version (or dynamic = ["version"])
description = "GST invoices from the command line"
readme = "README.md"                            # long description shown on PyPI
requires-python = ">=3.12"                      # resolvers skip this release on older Pythons
license = "MIT"                                 # SPDX expression (PEP 639)
authors = [{ name = "Course Author" }]
classifiers = ["Programming Language :: Python :: 3", "Typing :: Typed"]
dependencies = [                                # runtime requirements -> Requires-Dist
    "gst-calc>=1.0,<2",
    "win-colors>=1.0; sys_platform == 'win32'", # environment marker: Windows only
]

[project.optional-dependencies]                 # extras: pip install "invoice-kit[report]"
report = ["table-print>=0.1"]

[project.scripts]                               # console scripts: command = "module:function"
invoice-kit = "invoice_kit.cli:main"

[project.urls]
Homepage = "https://example.com/invoice-kit"

[dependency-groups]                             # PEP 735: local-only groups, never published
dev = ["rupee-fmt>=1.1", { include-group = "lint" }]
lint = ["table-print"]

[tool.uv]                                       # uv settings for this project
no-index = true
find-links = ["../wheelhouse"]

[tool.ruff]                                     # linter / formatter
line-length = 100
target-version = "py312"

[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B"]

[tool.mypy]                                     # type checker (Unit 4.2)
strict = true
python_version = "3.12"

[tool.pytest.ini_options]                       # test runner (Unit 4.3)
testpaths = ["tests"]
addopts = "-ra --strict-markers"
"""
anatomy = tomllib.loads(ANATOMY_PYPROJECT)
for table, content in anatomy.items():
    keys = list(content)
    print(f"[{table}] -> {keys}")

[build-system] -> ['requires', 'build-backend']
[project] -> ['name', 'version', 'description', 'readme', 'requires-python', 'license', 'authors', 'classifiers', 'dependencies', 'optional-dependencies', 'scripts', 'urls']
[dependency-groups] -> ['dev', 'lint']
[tool] -> ['uv', 'ruff', 'mypy', 'pytest']


`tomllib` (standard library since 3.11) parses TOML into dicts; note how `[tool.uv]`
and `[tool.ruff]` become nested keys of `tool`. The `[tool.*]` tables configure the
tools:

In [31]:
for tool_name, settings in anatomy["tool"].items():
    print(f"tool.{tool_name}: {settings}")
print(f"{anatomy['dependency-groups']=}")

tool.uv: {'no-index': True, 'find-links': ['../wheelhouse']}
tool.ruff: {'line-length': 100, 'target-version': 'py312', 'lint': {'select': ['E', 'F', 'I', 'UP', 'B']}}
tool.mypy: {'strict': True, 'python_version': '3.12'}
tool.pytest: {'ini_options': {'testpaths': ['tests'], 'addopts': '-ra --strict-markers'}}
anatomy['dependency-groups']={'dev': ['rupee-fmt>=1.1', {'include-group': 'lint'}], 'lint': ['table-print']}


**Where each table ends up.** Build the project and read the wheel's `METADATA`:
`[project]` becomes `Name`, `Version`, `Requires-Python`, `License-Expression`,
`Requires-Dist`; each extra becomes `Provides-Extra` plus `Requires-Dist: …; extra ==
"report"`; `[project.scripts]` becomes `entry_points.txt`. `[dependency-groups]` and
`[tool.*]` are **not** in the wheel: they only matter in the source tree.

In [32]:
anatomy_root = write_files(
    fresh_dir("projects/invoice-kit"),
    {
        "pyproject.toml": ANATOMY_PYPROJECT.replace("../wheelhouse", "../../wheelhouse"),
        "README.md": "# invoice-kit\n\nGST invoices from the command line.\n",
        "src/invoice_kit/__init__.py": "",
        "src/invoice_kit/cli.py": "def main() -> None:\n    print('invoice-kit')\n",
        "src/invoice_kit/py.typed": "",
    },
)
shell("uv", "build", "--offline", "--wheel", cwd=anatomy_root, keep="Successfully")
kit_wheel = next((anatomy_root / "dist").glob("*.whl"))
with zipfile.ZipFile(kit_wheel) as wheel:
    metadata_text = wheel.read("invoice_kit-1.4.0.dist-info/METADATA").decode()
    entry_points = wheel.read("invoice_kit-1.4.0.dist-info/entry_points.txt").decode()
print(metadata_text.split("\n\n")[0])  # headers only (the README body follows)
print("--- entry_points.txt ---\n" + entry_points.strip())
print(f"{'dependency-groups' in metadata_text=}  {'ruff' in metadata_text=}")

[<scratch>/projects/invoice-kit]$ uv build --offline --wheel
  Successfully built dist/invoice_kit-1.4.0-py3-none-any.whl
Metadata-Version: 2.4
Name: invoice-kit
Version: 1.4.0
Summary: GST invoices from the command line
Author: Course Author
License-Expression: MIT
Classifier: Programming Language :: Python :: 3
Classifier: Typing :: Typed
Requires-Dist: gst-calc>=1.0,<2
Requires-Dist: win-colors>=1.0 ; sys_platform == 'win32'
Requires-Dist: table-print>=0.1 ; extra == 'report'
Requires-Python: >=3.12
Project-URL: Homepage, https://example.com/invoice-kit
Provides-Extra: report
Description-Content-Type: text/markdown
--- entry_points.txt ---
[console_scripts]
invoice-kit = invoice_kit.cli:main
'dependency-groups' in metadata_text=False  'ruff' in metadata_text=False


The course's own `pyproject.toml` is a *virtual* project: `[project]` without
`[build-system]`, a `dev` group and tool tables. Reading it is fine; no cell in this
notebook writes to it.

In [33]:
course = tomllib.loads((REPO_ROOT / "pyproject.toml").read_text())
print(f"{course['project']['name']=}  {course['project']['requires-python']=}")
print(f"{'build-system' in course=}  (virtual project: not built or installed)")
print(f"{sorted(course['dependency-groups'])=}  {sorted(course['tool'])=}")

course['project']['name']='python-course'  course['project']['requires-python']='>=3.15'
'build-system' in course=False  (virtual project: not built or installed)
sorted(course['dependency-groups'])=['dev']  sorted(course['tool'])=['ruff']


### 4.1.5 Version specifiers (PEP 440)

Python versions follow **PEP 440**, not SemVer. A version looks like
`[N!]N(.N)*[{a|b|rc}N][.postN][.devN]`, and ordering is defined by the standard:
`1.0.dev1 < 1.0a1 < 1.0b1 < 1.0rc1 < 1.0 < 1.0.post1`. A requirement combines a
name, optional extras, specifiers and an optional environment marker:

```text
gst-calc[report] >=1.0,<2 ; python_version >= "3.12"
^name    ^extras ^specifiers ^marker
```

| Operator | Meaning | npm equivalent |
| --- | --- | --- |
| `==1.4.2` | Exactly this version (`==1.4.*` = any 1.4.x) | `1.4.2`, `1.4.x` |
| `>=1.4` | This version or newer | `>=1.4` |
| `~=1.4` | **Compatible release**: `>=1.4, ==1.*` | `^1.4` (for ≥ 1.0) |
| `~=1.4.2` | `>=1.4.2, ==1.4.*` | `~1.4.2` |
| `!=1.5.0` | Anything except this version | `!=` does not exist in npm (`<1.5.0 \|\| >1.5.0`) |
| `<2`, `>`, `<=` | Upper bounds and exclusive comparisons | same |
| `===` | Arbitrary string equality (rare, legacy) | none |

`packaging.specifiers.SpecifierSet` implements these rules exactly as pip and uv do.
Each operator gets its own demo below.

In [34]:
from packaging.specifiers import SpecifierSet
from packaging.version import Version

CANDIDATES = ["1.3.9", "1.4.0", "1.4.2", "1.4.9", "1.5.0", "1.9.1", "2.0.0", "2.1.0rc1"]


def matching(specifier: str) -> list[str]:
    """Which candidate versions satisfy the specifier (pre-releases excluded by default)?"""
    return [str(version) for version in SpecifierSet(specifier).filter(map(Version, CANDIDATES))]


print(f"{matching('==1.4.2')=}")
print(f"{matching('==1.4.*')=}  (wildcard: any 1.4.x)")

matching('==1.4.2')=['1.4.2']
matching('==1.4.*')=['1.4.0', '1.4.2', '1.4.9']  (wildcard: any 1.4.x)


`>=` is a lower bound only: it happily accepts the next major version, breaking changes included.

In [35]:
print(f"{matching('>=1.4')=}")
print(f"{matching('>=1.4,<2')=}  (the usual way to say: 1.4 or newer, but not 2.x)")

matching('>=1.4')=['1.4.0', '1.4.2', '1.4.9', '1.5.0', '1.9.1', '2.0.0']
matching('>=1.4,<2')=['1.4.0', '1.4.2', '1.4.9', '1.5.0', '1.9.1']  (the usual way to say: 1.4 or newer, but not 2.x)


`~=` (*compatible release*) drops the last component you write and pins the rest,
so the number of components you write decides how much can float:

In [36]:
print(f"{matching('~=1.4')=}    (>=1.4, ==1.*)")
print(f"{matching('~=1.4.2')=}  (>=1.4.2, ==1.4.*)")
print(f"{SpecifierSet('~=1.4.2') == SpecifierSet('>=1.4.2,==1.4.*')=}  (different spelling)")
print(f"{Version('1.4.2') in SpecifierSet('~=1.4.2')=}")

matching('~=1.4')=['1.4.0', '1.4.2', '1.4.9', '1.5.0', '1.9.1']    (>=1.4, ==1.*)
matching('~=1.4.2')=['1.4.2', '1.4.9']  (>=1.4.2, ==1.4.*)
SpecifierSet('~=1.4.2') == SpecifierSet('>=1.4.2,==1.4.*')=False  (different spelling)
Version('1.4.2') in SpecifierSet('~=1.4.2')=True


`!=` excludes a known-bad release while still accepting everything around it:

In [37]:
print(f"{matching('>=1.4,!=1.5.0,<2')=}")
print(f"{matching('!=1.4.*')=}  (exclude a whole minor series)")

matching('>=1.4,!=1.5.0,<2')=['1.4.0', '1.4.2', '1.4.9', '1.9.1']
matching('!=1.4.*')=['1.3.9', '1.5.0', '1.9.1', '2.0.0']  (exclude a whole minor series)


**Pre-releases** (`a`, `b`, `rc`, `.dev`) are excluded unless a specifier names one
or you opt in. This keeps `2.1.0rc1` out of production by accident. The same rule
applies in uv: `uv pip compile` picks 2.0.0 unless you pass `--prerelease allow`.

In [38]:
print(f"{matching('>=2')=}")
print(
    f"{[str(v) for v in SpecifierSet('>=2', prereleases=True).filter(map(Version, CANDIDATES))]=}"
)
print(f"{matching('>=2.1.0rc1')=}  (naming a pre-release opts in)")
ordering = ["1.0", "1.0.post1", "1.0rc1", "1.0a1", "1.0.dev1", "1.0b2"]
print(f"{sorted(ordering, key=Version)=}")
requirements_in = UNIT41_DIR / "requirements.in"
requirements_in.write_text("rupee-fmt>=2\n")
for extra_flags in ([], ["--prerelease", "allow"]):
    shell(
        "uv",
        "pip",
        "compile",
        "requirements.in",
        "--no-header",
        "--no-annotate",
        "--no-index",
        "--find-links",
        "wheelhouse",
        "--quiet",
        *extra_flags,
    )

matching('>=2')=['2.0.0']
[str(v) for v in SpecifierSet('>=2', prereleases=True).filter(map(Version, CANDIDATES))]=['2.0.0', '2.1.0rc1']
matching('>=2.1.0rc1')=['2.1.0rc1']  (naming a pre-release opts in)
sorted(ordering, key=Version)=['1.0.dev1', '1.0a1', '1.0b2', '1.0rc1', '1.0', '1.0.post1']
[<scratch>]$ uv pip compile requirements.in --no-header --no-annotate --no-index --find-links wheelhouse --quiet
  rupee-fmt==2.0.0
[<scratch>]$ uv pip compile requirements.in --no-header --no-annotate --no-index --find-links wheelhouse --quiet --prerelease allow
  rupee-fmt==2.1.0rc1


**Environment markers** (PEP 508) make a requirement conditional on where it is
installed: `sys_platform`, `platform_machine`, `python_version`,
`implementation_name`, `os_name` and more. `gst-calc` uses one to pull in
`win-colors` on Windows only. A marker is evaluated against an environment; the
default is the running interpreter.

In [39]:
from packaging.markers import Marker, default_environment

windows_only = Marker("sys_platform == 'win32'")
modern_cpython = Marker("python_version >= '3.13' and implementation_name == 'cpython'")
here = default_environment()
print(f"{here['sys_platform']=}  {here['python_version']=}  {here['platform_machine']=}")
print(f"{windows_only.evaluate()=}  {windows_only.evaluate({'sys_platform': 'win32'})=}")
print(f"{modern_cpython.evaluate()=}")
old_python = Marker('python_version < "3.11"')
print(f"{old_python.evaluate()=}  (e.g. a tomli backport)")

here['sys_platform']='linux'  here['python_version']='3.15'  here['platform_machine']='x86_64'
windows_only.evaluate()=False  windows_only.evaluate({'sys_platform': 'win32'})=True
modern_cpython.evaluate()=True
old_python.evaluate()=False  (e.g. a tomli backport)


**Extras** name optional feature sets. `pkg[extra]` adds the extra's requirements;
inside the published metadata they appear as markers on `extra`. `Requirement`
parses a whole PEP 508 line:

In [40]:
from packaging.requirements import Requirement

requirement = Requirement('gst-calc[report]>=1.0,<2; python_version >= "3.12"')
print(f"{requirement.name=}  {requirement.extras=}")
print(f"{str(requirement.specifier)=}  {str(requirement.marker)=}")
published = Requirement('table-print>=0.1; extra == "report"')
print(
    f"{published.marker.evaluate({'extra': 'report'})=}\n"
    f"{published.marker.evaluate({'extra': ''})=}"
)

requirement.name='gst-calc'  requirement.extras={'report'}
str(requirement.specifier)='<2,>=1.0'  str(requirement.marker)='python_version >= "3.12"'
published.marker.evaluate({'extra': 'report'})=True
published.marker.evaluate({'extra': ''})=False


And for real: installing `gst-calc[report]` adds `table-print`, the plain name does not.

In [41]:
for requirement_text in ("gst-calc==1.0.0", "gst-calc[report]==1.0.0"):
    pip("install", "--dry-run", "--ignore-installed", requirement_text, keep="Would install")

[<scratch>]$ python -m pip install --dry-run --ignore-installed gst-calc==1.0.0
  Would install gst-calc-1.0.0 rupee-fmt-1.2.0
[<scratch>]$ python -m pip install --dry-run --ignore-installed 'gst-calc[report]==1.0.0'
  Would install gst-calc-1.0.0 rupee-fmt-1.2.0 table-print-0.1.0


**Predict the output.**

```python
from packaging.specifiers import SpecifierSet

spec = SpecifierSet("~=2.2")
print([v for v in ["2.1", "2.2", "2.9", "3.0", "2.10"] if v in spec])
```

<details><summary>Answer</summary>

`['2.2', '2.9', '2.10']`. `~=2.2` means `>=2.2, ==2.*`, so 2.1 is too old and 3.0
is a new major series. Versions compare component by component as numbers, so
`2.10` is **newer** than `2.9` (unlike string comparison).

</details>

Run the next cell to check your prediction.

In [42]:
from packaging.specifiers import SpecifierSet

spec = SpecifierSet("~=2.2")
print([v for v in ["2.1", "2.2", "2.9", "3.0", "2.10"] if v in spec])

['2.2', '2.9', '2.10']


### 4.1.6 Lockfiles

`pyproject.toml` says what you *accept* (`gst-calc~=1.0`); a lockfile records what
you *got*: every package, transitive ones included, at an exact version, with file
hashes and where it came from. Installing from the lock gives the same environment
on every machine.

**`uv.lock`** is uv's own format: TOML, **universal** (resolved once for every
platform and Python version the project supports; markers record which packages
apply where) and meant to be committed. Never edit it by hand.

In [43]:
lock = tomllib.loads((invoicing / "uv.lock").read_text())
print(f"{lock['version']=}  {lock['requires-python']=}")
for package in lock["package"]:
    source = package["source"]
    wheels = len(package.get("wheels", []))
    dependencies = [
        dependency["name"] + (f" ; {dependency['marker']}" if "marker" in dependency else "")
        for dependency in package.get("dependencies", [])
    ]
    print(
        f"{package['name']:<12} {package['version']:<7} {tidy(str(source)):<34} "
        f"wheels={wheels} deps={dependencies}"
    )

lock['version']=1  lock['requires-python']='>=3.15'
gst-calc     1.0.0   {'registry': '../../wheelhouse'}   wheels=1 deps=['rupee-fmt', "win-colors ; sys_platform == 'win32'"]
invoicing    0.1.0   {'editable': '.'}                  wheels=0 deps=['gst-calc']
rupee-fmt    1.2.0   {'registry': '../../wheelhouse'}   wheels=1 deps=[]
table-print  0.1.0   {'registry': '../../wheelhouse'}   wheels=1 deps=[]
win-colors   1.0.0   {'registry': '../../wheelhouse'}   wheels=1 deps=[]


Note `win-colors`: it is locked even on Linux, with the marker
`sys_platform == 'win32'` on the edge from `gst-calc`, so a Windows teammate gets the
same lock. `uv sync` only installs what applies to the current machine:

In [44]:
installed = run_tool("uv", "pip", "list", "--format", "freeze", cwd=invoicing)[1]
print(installed)
print(f"{'win-colors' in installed=}  (locked for Windows, not installed here)")

gst-calc==1.0.0
invoicing==0.1.0
rupee-fmt==1.2.0
'win-colors' in installed=False  (locked for Windows, not installed here)


**`pylock.toml` (PEP 751, accepted 2025)** is the *standard* lockfile format that
any installer can read: a flat list of packages, each with its exact files and
hashes. uv exports it from `uv.lock`; pip ≥ 25.1 can write one with `pip lock`
(experimental). Think of `uv.lock` as the rich working format and `pylock.toml` as
the interchange format for other tools and auditors.

In [45]:
shell("uv", "export", "--format", "pylock.toml", "-o", "pylock.toml", "--quiet", cwd=invoicing)
pylock = tomllib.loads((invoicing / "pylock.toml").read_text())
print(f"{pylock['lock-version']=}  {pylock['created-by']=}")
for package in pylock["packages"]:
    files = [*package.get("wheels", []), *([package["sdist"]] if "sdist" in package else [])]
    hashes = [next(iter(file["hashes"].values()))[:12] + "…" for file in files]
    print(
        f"{package['name']:<12} {package.get('version', '(editable)'):<10} "
        f"marker={package.get('marker', '-'):<24} sha256={hashes}"
    )

[<scratch>/projects/invoicing]$ uv export --format pylock.toml -o pylock.toml --quiet
pylock['lock-version']='1.0'  pylock['created-by']='uv'
gst-calc     1.0.0      marker=-                        sha256=['c09ec26299b9…', '5fa5a288e797…']
invoicing    (editable) marker=-                        sha256=[]
rupee-fmt    1.2.0      marker=-                        sha256=['479bc9002b86…', '15ebb6744931…']
win-colors   1.0.0      marker=sys_platform == 'win32'  sha256=['def335bdaa52…', '59e220dec9e4…']


**Pinning strategy: applications vs libraries.** The same project can be both, so
decide per artefact:

| | Application (service, CLI you deploy) | Library (published for others to import) |
| --- | --- | --- |
| `dependencies` | Reasonable ranges (`>=1.4,<2`) | **Wide** ranges: lower bound you tested, upper bound only for known breaks |
| Lockfile | Committed; deploy exactly the lock (`uv sync --locked`) | Committed for CI and contributors, but **users never see it** |
| Upgrades | Deliberate: `uv lock --upgrade-package`, Dependabot/Renovate PRs | Test against the newest *and* the oldest allowed versions |
| Exact pins (`==`) in `pyproject.toml` | Acceptable | Never: they cause conflicts in every app that uses you |

uv can resolve to the **lowest** versions your specifiers allow, which is how a
library tests its lower bounds:

In [46]:
requirements_in.write_text("gst-calc>=1.0,<2\n")
for strategy in ("highest", "lowest"):
    print(f"--resolution {strategy}:")
    shell(
        "uv",
        "pip",
        "compile",
        "requirements.in",
        "--resolution",
        strategy,
        "--no-header",
        "--no-annotate",
        "--no-index",
        "--find-links",
        "wheelhouse",
        "--quiet",
    )

--resolution highest:
[<scratch>]$ uv pip compile requirements.in --resolution highest --no-header --no-annotate --no-index --find-links wheelhouse --quiet
  gst-calc==1.0.0
  rupee-fmt==1.2.0
--resolution lowest:
[<scratch>]$ uv pip compile requirements.in --resolution lowest --no-header --no-annotate --no-index --find-links wheelhouse --quiet
  gst-calc==1.0.0
  rupee-fmt==1.0.0


### 4.1.7 Project layouts

| Flat layout | `src/` layout |
| --- | --- |
| `myproject/mypkg/__init__.py` | `myproject/src/mypkg/__init__.py` |
| `import mypkg` works from the project root **without installing**, because the current folder is on `sys.path` | `import mypkg` works **only** after installing (usually editable) |
| Tests may pass against the working copy while the built wheel is broken (missing files, wrong package list) | Tests run against the installed package, like users |

uv (`--lib`, `--package`), Hatch, PDM and the Python Packaging User Guide default to
`src/`. The demo runs `python -c "import pricing"` from the root of each layout,
without installing anything:

In [47]:
PRICING_MODULE = "def price(amount):\n    return amount\n"
flat_root = write_files(fresh_dir("layouts/flat"), {"pricing/__init__.py": PRICING_MODULE})
src_root = write_files(fresh_dir("layouts/src"), {"src/pricing/__init__.py": PRICING_MODULE})
IMPORT_CHECK = "import pricing; print('imported from', pricing.__file__)"
print("flat:", run_tool("python", "-c", IMPORT_CHECK, cwd=flat_root)[1])
print("src: ", run_tool("python", "-c", IMPORT_CHECK, cwd=src_root)[1].splitlines()[-1])

flat: imported from <scratch>/layouts/flat/pricing/__init__.py
src:  ModuleNotFoundError: No module named 'pricing'. Did you mean: 'profiling'?


**Editable-install mechanics.** `uv sync` installs the project itself in editable
mode (like `pip install -e .`). What actually lands in `site-packages` is tiny: a
`.dist-info/` folder (metadata, so `importlib.metadata` finds the project) with a
`direct_url.json` saying *editable*, and a `.pth` file that adds `src/` to
`sys.path`. setuptools sometimes uses an import-hook finder instead of a plain path
(`__editable___*_finder.py`), which can hide new top-level modules; a `.pth` path
does not.

In [48]:
invoicing_site = next((invoicing / ".venv").glob("lib/python3*/site-packages"))
for path in sorted(invoicing_site.glob("*invoicing*")):
    print(path.name + ("/" if path.is_dir() else ""))
pth_text = next(invoicing_site.glob("*invoicing*.pth")).read_text().strip()
print(f"pth -> {tidy(pth_text)}")
dist_info = next(invoicing_site.glob("invoicing-*.dist-info"))
print(f"direct_url.json -> {tidy((dist_info / 'direct_url.json').read_text())}")

invoicing-0.1.0.dist-info/
invoicing.pth
pth -> <scratch>/projects/invoicing/src
direct_url.json -> {"url":"<scratch>/projects/invoicing","dir_info":{"editable":true}}


**Under the hood: hardlinks from the cache.** The other packages in that
environment were not copied: each file is a hardlink into uv's cache, so its link
count is above 1 (the same inode appears in two places).

In [49]:
rupee_init = next(invoicing_site.glob("rupee_fmt/__init__.py"))
status = rupee_init.stat()
print(f"rupee_fmt/__init__.py: inode {status.st_ino}, {status.st_nlink} links")
linked_in_cache = [
    path
    for path in (UNIT41_DIR / "uv-cache").rglob("__init__.py")
    if path.stat().st_ino == status.st_ino
]
print(f"same inode in the uv cache: {[tidy(str(path))[:60] + '…' for path in linked_in_cache[:1]]}")

rupee_fmt/__init__.py: inode 17586286, 2 links
same inode in the uv cache: ['<scratch>/uv-cache/archive-v0/PHyvdcUWKyA319Lc/rupee_fmt/__i…']


**`__version__` via `importlib.metadata`.** Keep the version in exactly one place,
`pyproject.toml`, and read it at run time from the installed metadata. A bumped
version shows up after the next `uv sync` (which re-writes the metadata):

```python
from importlib.metadata import version

__version__ = version("invoicing")
```

In [50]:
init_file = invoicing / "src" / "invoicing" / "__init__.py"
init_file.write_text(
    'from importlib.metadata import version\n\n__version__ = version("invoicing")\n'
    + init_file.read_text()
)
VERSION_CHECK = ("python", "-c", "import invoicing; print(invoicing.__version__)")
shell("uv", "run", *VERSION_CHECK, cwd=invoicing)
pyproject_path.write_text(
    pyproject_path.read_text().replace('version = "0.1.0"', 'version = "0.2.0"')
)
shell("uv", "run", *VERSION_CHECK, cwd=invoicing)  # re-syncs: metadata rewritten
print(
    f"in this kernel: {importlib.metadata.version('mypy')=}  "
    f"{importlib.metadata.version('pydantic')=}"
)

[<scratch>/projects/invoicing]$ uv run python -c 'import invoicing; print(invoicing.__version__)'
  0.1.0


[<scratch>/projects/invoicing]$ uv run python -c 'import invoicing; print(invoicing.__version__)'
  0.2.0
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Uninstalled 1 package in <time>
  Installed 1 package in <time>
in this kernel: importlib.metadata.version('mypy')='2.4.1'  importlib.metadata.version('pydantic')='2.14.3'


### 4.1.8 Building and publishing

**Wheel contents.** A wheel is a zip file. Everything outside the `.dist-info/`
folder is copied into `site-packages` as is; the `.dist-info/` folder holds the
metadata installers and `importlib.metadata` read:

| File | Content |
| --- | --- |
| `METADATA` | Core metadata (name, version, `Requires-Dist`, `Requires-Python`, README) |
| `WHEEL` | Wheel format version, generator, `Root-Is-Purelib`, the tags |
| `RECORD` | Every file with its SHA-256 and size: the installer checks them and uses it to uninstall |
| `entry_points.txt` | Console scripts and plugin entry points |
| `licenses/` | License files (PEP 639) |

The demo opens the `invoicing` wheel and sdist built in 4.1.3.

In [51]:
shutil.rmtree(invoicing / "dist", ignore_errors=True)
shell("uv", "build", "--offline", cwd=invoicing, keep="Successfully")
invoicing_wheel = next((invoicing / "dist").glob("*.whl"))
invoicing_sdist = next((invoicing / "dist").glob("*.tar.gz"))
with zipfile.ZipFile(invoicing_wheel) as wheel:
    print(f"{invoicing_wheel.name}:")
    for info in wheel.infolist():
        print(f"  {info.filename:<44} {info.file_size:>6} bytes")
    dist_info_name = next(name.split("/")[0] for name in wheel.namelist() if ".dist-info/" in name)
    print(wheel.read(f"{dist_info_name}/WHEEL").decode())

[<scratch>/projects/invoicing]$ uv build --offline
  Successfully built dist/invoicing-0.2.0.tar.gz
  Successfully built dist/invoicing-0.2.0-py3-none-any.whl
invoicing-0.2.0-py3-none-any.whl:
  invoicing/                                        0 bytes
  invoicing/__init__.py                           289 bytes
  invoicing-0.2.0.dist-info/                        0 bytes
  invoicing-0.2.0.dist-info/WHEEL                  80 bytes
  invoicing-0.2.0.dist-info/entry_points.txt       46 bytes
  invoicing-0.2.0.dist-info/METADATA              216 bytes
  invoicing-0.2.0.dist-info/RECORD                385 bytes
Wheel-Version: 1.0
Generator: uv 0.13.5
Root-Is-Purelib: true
Tag: py3-none-any



`METADATA` uses e-mail header syntax, so the standard library's `email.parser` reads
it, which is exactly what `importlib.metadata` does:

In [52]:
with zipfile.ZipFile(invoicing_wheel) as wheel:
    headers = email.parser.Parser().parsestr(wheel.read(f"{dist_info_name}/METADATA").decode())
    entry_points = wheel.read(f"{dist_info_name}/entry_points.txt").decode()
for key in ("Metadata-Version", "Name", "Version", "Requires-Python"):
    print(f"{key}: {headers[key]}")
print(f"Requires-Dist: {headers.get_all('Requires-Dist')}")
print(f"Provides-Extra: {headers.get_all('Provides-Extra')}")
print(f"entry_points.txt: {entry_points.split()}")

Metadata-Version: 2.3
Name: invoicing
Version: 0.2.0
Requires-Python: >=3.15
Requires-Dist: ['gst-calc~=1.0', "table-print>=0.1 ; extra == 'report'"]
Provides-Extra: ['report']
entry_points.txt: ['[console_scripts]', 'invoicing', '=', 'invoicing:main']


**Under the hood: verifying `RECORD`.** Each line is `path,sha256=<urlsafe base64
digest without padding>,size`. Recomputing every digest is what an installer does to
detect a corrupted or tampered wheel:

In [53]:
def record_digest(data: bytes) -> str:
    """The RECORD spelling of a SHA-256 digest."""
    return "sha256=" + base64.urlsafe_b64encode(hashlib.sha256(data).digest()).rstrip(b"=").decode()


with zipfile.ZipFile(invoicing_wheel) as wheel:
    record_lines = wheel.read(f"{dist_info_name}/RECORD").decode().splitlines()
    for line in record_lines:
        path, digest, size = line.rsplit(",", 2)
        if digest:  # RECORD cannot contain its own hash
            actual = record_digest(wheel.read(path))
            print(f"{path:<44} {'OK' if actual == digest else 'MISMATCH'}  {digest[:20]}…")
        else:
            print(f"{path:<44} (no hash: the RECORD file itself)")

invoicing/__init__.py                        OK  sha256=qm-iUS2GXRwSl…
invoicing-0.2.0.dist-info/WHEEL              OK  sha256=si2p8vW03Rnhy…
invoicing-0.2.0.dist-info/entry_points.txt   OK  sha256=kGDIjnJ5Sz4c5…
invoicing-0.2.0.dist-info/METADATA           OK  sha256=S-3jZut3SJGPG…
invoicing-0.2.0.dist-info/RECORD             (no hash: the RECORD file itself)


The **sdist** is a `.tar.gz` of the source tree plus a generated `PKG-INFO` (the same
metadata). It contains `pyproject.toml`, so anyone can rebuild the wheel; uv built
the wheel above from this very archive. `uv_build` stores a normalized copy of
`pyproject.toml` (comments dropped, lists on one line) and keeps your original as
`pyproject.toml.orig`.

In [54]:
with tarfile.open(invoicing_sdist) as sdist:
    for member in sdist.getmembers():
        if member.isfile():
            print(f"  {member.name:<48} {member.size:>6} bytes")
    pkg_info = (
        sdist.extractfile(f"{invoicing_sdist.name.removesuffix('.tar.gz')}/PKG-INFO")
        .read()
        .decode()
    )
print(pkg_info.split("\n\n")[0])

  invoicing-0.2.0/PKG-INFO                            216 bytes
  invoicing-0.2.0/pyproject.toml                      480 bytes
  invoicing-0.2.0/pyproject.toml.orig                 581 bytes
  invoicing-0.2.0/src/invoicing/__init__.py           289 bytes
Metadata-Version: 2.3
Name: invoicing
Version: 0.2.0
Summary: Add your description here
Requires-Dist: gst-calc~=1.0
Requires-Dist: table-print>=0.1 ; extra == 'report'
Requires-Python: >=3.15
Provides-Extra: report



**TestPyPI** (test.pypi.org) is a separate, throwaway copy of PyPI for rehearsing a
release. Declare it as an *explicit* index (never used for resolution unless a
package is pinned to it) with a publish URL, then `uv publish --index testpypi`, and
install from it to check the release works. The course does not run these commands.

```toml
[[tool.uv.index]]
name = "testpypi"
url = "https://test.pypi.org/simple/"
publish-url = "https://test.pypi.org/legacy/"
explicit = true
```

```bash
uv build
uv publish --index testpypi            # token or trusted publishing
uv run --with invoicing --index https://test.pypi.org/simple/ --no-project -- invoicing
```

**Trusted publishing (OIDC).** Instead of storing a long-lived PyPI API token as a CI
secret, you tell PyPI once: "releases of `invoicing` may come from GitHub repository
*owner/repo*, workflow `release.yml`, environment `pypi`". At release time the CI job
asks GitHub for a short-lived **OIDC identity token**, PyPI checks its claims against
that configuration and returns a 15-minute upload token. Nothing secret is stored,
and a leaked token expires almost at once. npm now offers the same thing (trusted
publishing with provenance).

```mermaid
sequenceDiagram
    participant CI as GitHub Actions job
    participant GH as GitHub OIDC provider
    participant PyPI
    CI->>GH: request ID token (audience: pypi)
    GH-->>CI: signed JWT (repo, workflow, environment)
    CI->>PyPI: exchange JWT for an upload token
    PyPI-->>CI: short-lived API token (if claims match the trusted publisher)
    CI->>PyPI: uv publish dist/*
```

<details><summary>ASCII fallback</summary>

```text
CI job --(1) ask for ID token--> GitHub OIDC
CI job <-(2) signed JWT: repo, workflow, env-- GitHub OIDC
CI job --(3) JWT--> PyPI checks the trusted-publisher config
CI job <-(4) 15-minute upload token-- PyPI
CI job --(5) uv publish dist/*--> PyPI
```

</details>

```yaml
# .github/workflows/release.yml (shown, not used by the course)
on:
  release:
    types: [published]
jobs:
  publish:
    runs-on: ubuntu-latest
    environment: pypi
    permissions:
      id-token: write        # allows the job to request the OIDC token
    steps:
      - uses: actions/checkout@v5
      - uses: astral-sh/setup-uv@v7
      - run: uv build
      - run: uv publish      # detects the OIDC environment: --trusted-publishing automatic
```

**SemVer vs CalVer.**

| Scheme | Format | Says | Used by |
| --- | --- | --- | --- |
| SemVer | `MAJOR.MINOR.PATCH` | Compatibility: major = breaking, minor = features, patch = fixes | most libraries, mypy, pydantic, uv (0.x: anything may break) |
| CalVer | `YY.MINOR` or `YYYY.MM.PATCH` | *When* it was released; no compatibility promise | pip, Black, Ubuntu, `certifi` |

Both are valid PEP 440 versions, so specifiers and ordering work the same. A SemVer
bump helper and the versions of tools in this environment:

In [55]:
def bump(version: str, part: str) -> str:
    """Next SemVer version: part is "major", "minor" or "patch"; pre-release parts are dropped."""
    major, minor, patch = Version(version).release[:3]
    match part:
        case "major":
            return f"{major + 1}.0.0"
        case "minor":
            return f"{major}.{minor + 1}.0"
        case "patch":
            return f"{major}.{minor}.{patch + 1}"
    raise ValueError(f"unknown part {part!r}")


print(f"{bump('1.4.2', 'patch')=}  {bump('1.4.2', 'minor')=}  {bump('1.4.2', 'major')=}")
print(f"{Version('2026.11.0') > Version('2026.2.3')=}  (CalVer compares numerically too)")
for distribution in ("pip", "mypy", "pydantic", "certifi"):
    try:
        print(f"  {distribution:<9} {importlib.metadata.version(distribution)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {distribution:<9} not installed")

bump('1.4.2', 'patch')='1.4.3'  bump('1.4.2', 'minor')='1.5.0'  bump('1.4.2', 'major')='2.0.0'
Version('2026.11.0') > Version('2026.2.3')=True  (CalVer compares numerically too)
  pip       26.3
  mypy      2.4.1
  pydantic  2.14.3
  certifi   2026.10.23


**Changelogs.** Keep a human-written `CHANGELOG.md` in the
[Keep a Changelog](https://keepachangelog.com) format: newest release first, an
`Unreleased` section at the top, entries grouped under *Added*, *Changed*,
*Deprecated*, *Removed*, *Fixed*, *Security*. With Conventional Commit messages
(`feat:`, `fix:`, `feat!:`), tools such as `git-cliff`, `commitizen` or
`python-semantic-release` can draft the entries and even pick the next SemVer
version. The idea in a few lines:

In [56]:
def draft_changelog(current_version: str, commit_subjects: list[str]) -> str:
    """Group Conventional Commit subjects into a Keep-a-Changelog section and pick the bump."""
    sections: dict[str, list[str]] = {"Added": [], "Fixed": [], "Changed": []}
    part = "patch"
    for subject in commit_subjects:
        kind, _, description = subject.partition(": ")
        if kind.endswith("!"):
            sections["Changed"].append(f"**Breaking:** {description}")
            part = "major"
        elif kind.startswith("feat"):
            sections["Added"].append(description)
            part = "minor" if part == "patch" else part
        elif kind.startswith("fix"):
            sections["Fixed"].append(description)
    lines = [f"## [{bump(current_version, part)}]"]
    for heading, entries in sections.items():
        lines += [f"### {heading}", *(f"- {entry}" for entry in entries)] if entries else []
    return "\n".join(lines)


print(
    draft_changelog(
        "1.4.2",
        [
            "feat(cli): add --currency option",
            "fix: round GST half up",
            "docs: fix typo",
            "feat!: drop Python 3.11",
        ],
    )
)

## [2.0.0]
### Added
- add --currency option
### Fixed
- round GST half up
### Changed
- **Breaking:** drop Python 3.11


### 4.1.9 Supply-chain hygiene

Every dependency is code you run with your permissions. The main risks and the tool
for each:

| Risk | Defence |
| --- | --- |
| A dependency has a published vulnerability | `pip-audit` (or `uv` + Dependabot/Renovate) in CI |
| You install a look-alike name (`requets`) | Review new dependencies; check name, author, download history |
| A file is swapped in transit or on a mirror | Hash checking (lockfiles, `--require-hashes`) |
| `pyproject.toml` drifts from the code (missing or unused dependencies) | `deptry` |
| A license you cannot ship (copyleft in a closed product) | Read `License-Expression` metadata; license checkers |
| Building an sdist runs arbitrary code | Prefer wheels (`--only-binary :all:`, `uv … --no-build`) |

**`pip-audit`** checks installed packages or a requirements file against the Python
Packaging Advisory Database (via PyPI or OSV). It needs the network, so the cell
checks first. The requirements file pins two deliberately old releases. The
findings change over time as advisories are published.

In [57]:
audit_file = UNIT41_DIR / "audit-requirements.txt"
audit_file.write_text("requests==2.19.0\njinja2==2.10\n")
if not network_available():
    print("SKIP: no network connection to pypi.org; pip-audit needs the advisory database")
else:
    exit_code, audit_output = run_tool(
        "pip-audit",
        "-r",
        str(audit_file),
        "--no-deps",
        "--disable-pip",
        "--progress-spinner",
        "off",
        "--desc",
        "off",
    )
    lines = audit_output.splitlines()
    print(next((line for line in lines if line.startswith("Found")), "no summary line"))
    table = [line for line in lines if re.match(r"(Name |---|requests |jinja2 )", line)]
    print(*list(dict.fromkeys(table))[:8], "...", sep="\n")
    print(f"exit code {exit_code} (1 = vulnerabilities found)")

Found 22 known vulnerabilities in 2 packages
Name     Version ID              Fix Versions
-------- ------- --------------- ------------
requests 2.19.0  PYSEC-2018-28   2.20.0
requests 2.19.0  PYSEC-2023-74   2.31.0
requests 2.19.0  PYSEC-2026-1873 2.32.0
requests 2.19.0  PYSEC-2026-1872 2.32.4
requests 2.19.0  PYSEC-2026-2275 2.33.0
jinja2   2.10    PYSEC-2021-66   2.11.3
...
exit code 1 (1 = vulnerabilities found)


**Typosquatting.** Attackers publish packages whose names are one keystroke away
from popular ones (`requets`, `python-dateutils`, `beautifulsoup`, `colourama` all
happened on PyPI). A quick defence when reviewing a new dependency is to compare it
with well-known names; `difflib` finds near misses:

In [58]:
import difflib

POPULAR = [
    "requests",
    "python-dateutil",
    "beautifulsoup4",
    "colorama",
    "numpy",
    "pandas",
    "urllib3",
    "setuptools",
    "cryptography",
    "pyyaml",
    "django",
    "flask",
]


def looks_like_typosquat(name: str) -> str | None:
    """The popular package this name is suspiciously close to, or None."""
    if name in POPULAR:
        return None
    close = difflib.get_close_matches(name, POPULAR, n=1, cutoff=0.85)
    return close[0] if close else None


for candidate in ["requets", "python-dateutils", "beautifulsoup", "colourama", "pydantic", "numpy"]:
    print(f"{candidate:<18} -> {looks_like_typosquat(candidate)}")

requets            -> requests
python-dateutils   -> python-dateutil
beautifulsoup      -> beautifulsoup4
colourama          -> colorama
pydantic           -> None
numpy              -> None


**Hash checking.** With `--require-hashes`, pip refuses any file whose SHA-256 is not
listed, so a swapped file on a mirror or a compromised index cannot slip in. uv
does the same with the hashes in `uv.lock`. `pip hash` prints the line to paste.
Below, the first install uses a wrong hash and is refused; the second uses the real
one.

In [59]:
wheel_path = WHEELHOUSE / "table_print-0.1.0-py3-none-any.whl"
pip("hash", str(wheel_path))
real_hash = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
for label, digest in [("tampered", "0" * 64), ("genuine", real_hash)]:
    (UNIT41_DIR / "hashed.txt").write_text(f"table-print==0.1.0 --hash=sha256:{digest}\n")
    print(f"--- {label} hash ---")
    pip(
        "install",
        "--require-hashes",
        "--force-reinstall",
        "-r",
        "hashed.txt",
        keep=r"Successfully|ERROR|Expected|Got",
    )

[<scratch>]$ python -m pip hash <scratch>/wheelhouse/table_print-0.1.0-py3-none-any.whl
  <scratch>/wheelhouse/table_print-0.1.0-py3-none-any.whl:
  --hash=sha256:5b2953bee5e74b466a94ef4c59fc1487a69ef8c0e8e4b3e73836c6e4477e6c21
--- tampered hash ---


[<scratch>]$ python -m pip install --require-hashes --force-reinstall -r hashed.txt
  ERROR: THESE PACKAGES DO NOT MATCH THE HASHES FROM THE REQUIREMENTS FILE. If you have updated the package versions, please update the hashes. Otherwise, examine the package contents carefully; someone may have tampered with them.
          Expected sha256 0000000000000000000000000000000000000000000000000000000000000000
               Got        5b2953bee5e74b466a94ef4c59fc1487a69ef8c0e8e4b3e73836c6e4477e6c21
  (exit code 1)
--- genuine hash ---


[<scratch>]$ python -m pip install --require-hashes --force-reinstall -r hashed.txt


**`deptry`** compares the imports in your code with the dependencies you declare:
`DEP001` an import that is not declared (works today only because something else
pulled it in), `DEP002` a declared dependency nobody imports, `DEP003` a transitive
dependency used directly, `DEP004` a dev dependency used in production code. First
the idea in a few lines of `ast` (Unit 3.2), offline:

In [60]:
import ast


def check_dependencies(source: str, declared: list[str]) -> dict[str, list[str]]:
    """Mini deptry: top-level imports vs declared distributions (module = name with _)."""
    imported = {
        alias.name.split(".")[0] if isinstance(node, ast.Import) else node.module.split(".")[0]
        for node in ast.walk(ast.parse(source))
        if isinstance(node, (ast.Import, ast.ImportFrom)) and getattr(node, "level", 0) == 0
        for alias in (node.names if isinstance(node, ast.Import) else [node.names[0]])
    }
    third_party = imported - set(sys.stdlib_module_names)
    declared_modules = {name.replace("-", "_").lower(): name for name in declared}
    return {
        "DEP001 missing": sorted(third_party - set(declared_modules)),
        "DEP002 unused": sorted(declared_modules[m] for m in set(declared_modules) - third_party),
    }


APP_SOURCE = "import json\nimport requests\nfrom rupee_fmt import fmt\n"
print(check_dependencies(APP_SOURCE, ["rupee-fmt", "table-print"]))

{'DEP001 missing': ['requests'], 'DEP002 unused': ['table-print']}


The real `deptry` (a Rust-accelerated tool, not a course dependency) runs through
`uvx`, which needs the network the first time to download it:

In [61]:
deptry_root = write_files(
    fresh_dir("deptry-demo"),
    {
        "pyproject.toml": """
        [project]
        name = "deptry-demo"
        version = "0.1.0"
        dependencies = ["rupee-fmt", "table-print"]
    """,
        "app.py": APP_SOURCE,
    },
)
if not network_available():
    print("SKIP: no network connection; `uvx deptry .` needs to download deptry once")
else:
    shell("uvx", "deptry", ".", "--no-ansi", cwd=deptry_root, keep=r"DEP\d|Found")

[<scratch>/deptry-demo]$ uvx deptry . --no-ansi
  app.py:2:8: DEP001 'requests' imported but missing from the dependency definitions
  pyproject.toml: DEP002 'table-print' defined as a dependency but not used in the codebase
  Found 2 dependency issues.
  (exit code 1)


**Licenses.** Installed packages declare their license in metadata: the modern
`License-Expression` field (an SPDX expression, PEP 639), the older free-text
`License`, or `License ::` classifiers. A first look at what this course installs:

In [62]:
def license_of(distribution_name: str) -> str:
    """Best available license information from installed metadata."""
    metadata = importlib.metadata.metadata(distribution_name)
    if expression := metadata.get("License-Expression"):
        return f"{expression} (SPDX)"
    classifiers = [
        c.split(" :: ")[-1]
        for c in metadata.get_all("Classifier") or []
        if c.startswith("License ::")
    ]
    if classifiers:
        return f"{', '.join(classifiers)} (classifier)"
    return (metadata.get("License") or "unknown").splitlines()[0][:40]


for name in ("mypy", "pyright", "pip-audit", "pydantic", "jupyterlab", "nbformat", "rich"):
    print(f"{name:<11} {license_of(name)}")

mypy        MIT (SPDX)
pyright     MIT
pip-audit   Apache Software License (classifier)
pydantic    MIT (SPDX)
jupyterlab  BSD License (classifier)
nbformat    BSD License (classifier)
rich        MIT License (classifier)


### 4.1.10 Tool comparison

| Python tool | What it does | Lockfile | Closest JS tool |
| --- | --- | --- | --- |
| **pip + venv** | Install into an environment you create; `requirements.txt` | none (freeze is a snapshot) | `npm install` without a lockfile |
| **pip-tools** | `pip-compile` turns `requirements.in` into a fully pinned, hashed `requirements.txt`; `pip-sync` installs exactly it | the compiled `requirements.txt` | `npm shrinkwrap` |
| **Poetry** | Projects, resolver, lockfile, build backend (`poetry-core`), publishing | `poetry.lock` | yarn (classic) |
| **PDM** | Standards-first project manager (PEP 621, PEP 735), plugin system | `pdm.lock` | pnpm |
| **Hatch** | Environments and test matrices, version bumping, the `hatchling` backend | none built in (uses others) | npm scripts + a build tool |
| **Conda** (+ mamba, pixi) | Cross-language binary packages (CUDA, GDAL, R) from conda-forge, its own environments | `environment.yml` / `pixi.lock` | nvm + apt together |
| **uv** | All of pip, pip-tools, virtualenv, pipx, pyenv (`uv python`), twine, workspaces; 10–100× faster | `uv.lock` (+ `pylock.toml` export) | pnpm + npx + nvm in one binary |

Rules of thumb: new projects use **uv**; data science with native non-Python
libraries often needs **conda/pixi**; existing Poetry or PDM projects can stay where
they are (all of them read PEP 621 `[project]` today, so moving is cheap). Which of
these are on this machine:

In [63]:
TOOLS = {
    "pip": "pip",
    "pip-compile": "pip-tools",
    "poetry": "Poetry",
    "pdm": "PDM",
    "hatch": "Hatch",
    "conda": "Conda",
    "pixi": "pixi",
    "uv": "uv",
}
for command, product in TOOLS.items():
    found = executable(command)
    version = run_tool(command, "--version")[1].splitlines()[0] if found else "not installed"
    print(f"{product:<10} {command:<12} {tidy(version)}")

pip        pip          pip 26.3 from <repo>/.venv/lib/python3.15/site-packages/pip (python 3.15)
pip-tools  pip-compile  not installed
Poetry     poetry       not installed
PDM        pdm          not installed
Hatch      hatch        not installed
Conda      conda        not installed
pixi       pixi         not installed
uv         uv           uv 0.13.5 (x86_64-unknown-linux-gnu)


**Check: the course project was never touched.** Every demo above ran in
`notebooks/_scratch/unit41/`. The course's `pyproject.toml` and `uv.lock` still
have the digests recorded by the unit's setup cell:

In [64]:
for name in COURSE_FILES:
    unchanged = file_digest(REPO_ROOT / name) == COURSE_DIGESTS_AT_START[name]
    print(f"{name:<15} unchanged: {unchanged}")

pyproject.toml  unchanged: True
uv.lock         unchanged: True


### Lab 4.1

Four exercises. Complete each stub, then run its check cell. Until then the checks
print FAIL; that is expected. The checks build and run your work with uv in
`notebooks/_scratch/unit41/lab/`, offline, against the local wheelhouse.

#### Exercise 1 — annotate every line of a `pyproject.toml`

Return `PLAIN_PYPROJECT` (below) with a comment on **every** non-blank line,
explaining what the line does. A comment is `# ` followed by at least three words,
either at the end of the line or on its own line directly above it. The TOML must
still parse to exactly the same data.

In [65]:
PLAIN_PYPROJECT = """
[build-system]
requires = ["uv_build>=0.13,<0.14"]
build-backend = "uv_build"

[project]
name = "shipping-rates"
version = "0.3.0"
requires-python = ">=3.12"
dependencies = ["rupee-fmt~=1.1", "win-colors; sys_platform == 'win32'"]

[project.optional-dependencies]
report = ["table-print>=0.1"]

[project.scripts]
ship-quote = "shipping_rates.cli:main"

[dependency-groups]
dev = ["gst-calc>=1,<2"]

[tool.ruff]
line-length = 100
"""


def annotated_pyproject() -> str:
    """PLAIN_PYPROJECT with a comment for every non-blank line."""
    raise NotImplementedError("Lab 4.1 exercise 1: annotate the file")

Run the checks.

In [66]:
def annotation_problems(plain: str, annotated: str) -> list[str]:
    """Lines of plain that have no 3-word comment (inline or on the line above) in annotated."""
    problems: list[str] = []
    comment_above = False
    content_lines = [line.strip() for line in textwrap.dedent(plain).splitlines() if line.strip()]
    seen = 0
    for line in textwrap.dedent(annotated).splitlines():
        stripped = line.strip()
        if stripped.startswith("#"):
            comment_above = len(stripped.lstrip("# ").split()) >= 3
            continue
        if not stripped:
            comment_above = False
            continue
        content, _, inline = stripped.partition(" # ")
        if len(inline.split()) < 3 and not comment_above:
            problems.append(content.strip())
        seen += 1
        comment_above = False
    if seen != len(content_lines):
        problems.append(f"expected {len(content_lines)} TOML lines, found {seen}")
    return problems


run_checks(
    [
        (
            "parses to the same data",
            lambda: (
                tomllib.loads(textwrap.dedent(annotated_pyproject()))
                == tomllib.loads(textwrap.dedent(PLAIN_PYPROJECT))
            ),
            True,
        ),
        (
            "every line annotated",
            lambda: annotation_problems(PLAIN_PYPROJECT, annotated_pyproject()),
            [],
        ),
        (
            "still a build-system table",
            lambda: tomllib.loads(textwrap.dedent(annotated_pyproject()))["build-system"][
                "build-backend"
            ],
            "uv_build",
        ),
    ]
)

FAIL  parses to the same data: raised NotImplementedError: Lab 4.1 exercise 1: annotate the file
FAIL  every line annotated: raised NotImplementedError: Lab 4.1 exercise 1: annotate the file
FAIL  still a build-system table: raised NotImplementedError: Lab 4.1 exercise 1: annotate the file
0/3 checks passed


#### Exercise 2 — a `src/`-layout library with a console script, extras and groups

Return the files (relative path → text) of a project that builds with uv:

| Requirement | Detail |
| --- | --- |
| Name, version, Python | `invoice-tools`, `0.1.0`, `requires-python = ">=3.12"`, no runtime dependencies |
| Build backend | `uv_build` (`requires = ["uv_build>=0.13,<0.14"]`) |
| Layout | `src/invoice_tools/` with `__init__.py`, `cli.py` and `py.typed`; nothing importable at the root |
| `__version__` | read with `importlib.metadata.version("invoice-tools")` in `__init__.py` |
| Console script | `invoice-total` → `invoice_tools.cli:main`, which prints `total: <sum>` of its arguments with two decimals (`invoice-total 10 20.5` prints `total: 30.50`) |
| Extra | `report` → `table-print>=0.1` |
| Dependency group | `dev` → `rupee-fmt>=1` (must **not** appear in the wheel metadata) |

In [67]:
def library_files() -> dict[str, str]:
    """Relative path -> file content for the invoice-tools project."""
    raise NotImplementedError("Lab 4.1 exercise 2: write the project files")

Run the checks. They build your project with `uv build`, then inspect and run the wheel.

In [68]:
LAB_DIR = UNIT41_DIR / "lab"


_learner_builds: dict[str, tuple[Path, dict[str, str]]] = {}


def build_learner_library() -> tuple[Path, dict[str, str]]:
    """Build library_files() with uv (once per distinct content); return (wheel, file texts)."""
    key = json.dumps(library_files(), sort_keys=True)
    if key not in _learner_builds:
        _learner_builds[key] = _build_library()
    return _learner_builds[key]


def _build_library() -> tuple[Path, dict[str, str]]:
    """Write library_files() under lab/, build with uv, read METADATA and entry points."""
    root = write_files(fresh_dir("lab/invoice-tools"), library_files())
    exit_code, output = run_tool("uv", "build", "--offline", cwd=root)
    if exit_code:
        raise RuntimeError(output.splitlines()[-1])
    wheel_path = next((root / "dist").glob("*.whl"))
    with zipfile.ZipFile(wheel_path) as wheel:
        names = wheel.namelist()
        dist_info = next(name.split("/")[0] for name in names if ".dist-info/" in name)
        report = {
            "names": "\n".join(names),
            "metadata": wheel.read(f"{dist_info}/METADATA").decode(),
            "entry_points": wheel.read(f"{dist_info}/entry_points.txt").decode(),
        }
    return wheel_path, report


def run_learner_cli(*arguments: str) -> str:
    """Run invoice-total from the built wheel with uvx."""
    wheel_path, _ = build_learner_library()
    return run_tool(
        "uvx", "--quiet", "--offline", "--from", str(wheel_path), "invoice-total", *arguments
    )[1]


def learner_version() -> str:
    wheel_path, _ = build_learner_library()
    return run_tool(
        "uv",
        "run",
        "--quiet",
        "--offline",
        "--no-project",
        "--with",
        str(wheel_path),
        "python",
        "-c",
        "import invoice_tools; print(invoice_tools.__version__)",
        cwd=LAB_DIR,
    )[1]


run_checks(
    [
        (
            "src layout",
            lambda: sorted(p for p in library_files() if p.startswith("src/invoice_tools/")),
            [
                "src/invoice_tools/__init__.py",
                "src/invoice_tools/cli.py",
                "src/invoice_tools/py.typed",
            ],
        ),
        (
            "nothing importable at the root",
            lambda: [p for p in library_files() if p.endswith(".py") and "/" not in p],
            [],
        ),
        (
            "console script in entry_points.txt",
            lambda: (
                "invoice-total = invoice_tools.cli:main"
                in build_learner_library()[1]["entry_points"]
            ),
            True,
        ),
        (
            "extra published",
            lambda: "Provides-Extra: report" in build_learner_library()[1]["metadata"],
            True,
        ),
        (
            "extra requirement",
            lambda: (
                re.search(
                    r"Requires-Dist: table-print>=0\.1 ?; extra == .report.",
                    build_learner_library()[1]["metadata"],
                )
                is not None
            ),
            True,
        ),
        (
            "dev group not published",
            lambda: "rupee-fmt" in build_learner_library()[1]["metadata"],
            False,
        ),
        (
            "py.typed in the wheel",
            lambda: "invoice_tools/py.typed" in build_learner_library()[1]["names"],
            True,
        ),
        ("invoice-total 10 20.5", lambda: run_learner_cli("10", "20.5"), "total: 30.50"),
        ("__version__ from metadata", learner_version, "0.1.0"),
    ]
)

FAIL  src layout: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  nothing importable at the root: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  console script in entry_points.txt: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  extra published: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  extra requirement: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  dev group not published: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  py.typed in the wheel: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  invoice-total 10 20.5: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  __version__ from metadata: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
0/9 checks passed


#### Exercise 3 — a PEP 723 single-file script

Return the source of a script with an inline metadata block that declares
`requires-python = ">=3.12"` and the single dependency `rupee-fmt>=2`. When run, it
prints two lines: `fmt(1234.5)` from `rupee_fmt`, then `rupee-fmt <version>` using
`importlib.metadata`. The check runs it with `uv run` against the wheelhouse.
Which version should the resolver choose, given that `2.1.0rc1` exists?

In [69]:
def pep723_script() -> str:
    """Source of a self-describing script for `uv run`."""
    raise NotImplementedError("Lab 4.1 exercise 3: write the script")

Run the checks. `read_script_metadata` uses the reference regular expression from PEP 723.

In [70]:
PEP723_BLOCK = re.compile(
    r"(?m)^# /// (?P<type>[a-zA-Z0-9-]+)$\s(?P<content>(^#(| .*)$\s)+)^# ///$"
)


def read_script_metadata(script: str) -> dict[str, object]:
    """Parse the `# /// script` block of a PEP 723 script."""
    script = textwrap.dedent(script)
    match = next((m for m in PEP723_BLOCK.finditer(script) if m["type"] == "script"), None)
    if match is None:
        raise ValueError("no `# /// script` block")
    toml_text = "".join(
        line[2:] if line.startswith("# ") else line[1:]
        for line in match["content"].splitlines(keepends=True)
    )
    return tomllib.loads(toml_text)


def run_learner_script() -> list[str]:
    root = fresh_dir("lab/script")
    (root / "quote.py").write_text(textwrap.dedent(pep723_script()))
    output = run_tool(
        "uv",
        "run",
        "--quiet",
        "--offline",
        "--no-index",
        "--find-links",
        str(WHEELHOUSE),
        "quote.py",
        cwd=root,
    )[1]
    return output.splitlines()[-2:]


run_checks(
    [
        (
            "requires-python",
            lambda: read_script_metadata(pep723_script())["requires-python"],
            ">=3.12",
        ),
        (
            "dependencies",
            lambda: [
                d.replace(" ", "") for d in read_script_metadata(pep723_script())["dependencies"]
            ],
            ["rupee-fmt>=2"],
        ),
        ("output", run_learner_script, ["\u20b91,234.50", "rupee-fmt 2.0.0"]),
    ]
)

FAIL  requires-python: raised NotImplementedError: Lab 4.1 exercise 3: write the script
FAIL  dependencies: raised NotImplementedError: Lab 4.1 exercise 3: write the script
FAIL  output: raised NotImplementedError: Lab 4.1 exercise 3: write the script
0/3 checks passed


#### Exercise 4 — build a wheel and inspect its contents

Write `wheel_summary(wheel_path)` returning a dict with these keys, read only from
the wheel file (use `zipfile` and `email.parser`; do not install anything):

| Key | Value |
| --- | --- |
| `"name"`, `"version"` | from `METADATA` |
| `"tags"` | list of the `Tag:` values in `WHEEL` |
| `"packages"` | sorted top-level folders or modules outside `.dist-info/` |
| `"console_scripts"` | dict command → `module:function` from `entry_points.txt` (empty if none) |
| `"requires_dist"` | list of `Requires-Dist` values |
| `"record_ok"` | `True` if every file listed in `RECORD` with a hash matches its SHA-256 |

The check uses `gst-calc 1.0.0` from the wheelhouse and a tampered copy of it.

In [71]:
def wheel_summary(wheel_path: Path) -> dict[str, object]:
    """Facts about a wheel, read from its files."""
    raise NotImplementedError("Lab 4.1 exercise 4: inspect the wheel")

Run the checks.

In [72]:
GST_WHEEL = build_wheelhouse() / "gst_calc-1.0.0-py3-none-any.whl"


def tampered_wheel() -> Path:
    """A copy of GST_WHEEL whose gst_calc/__init__.py was edited after building."""
    LAB_DIR.mkdir(parents=True, exist_ok=True)
    target = LAB_DIR / GST_WHEEL.name
    with zipfile.ZipFile(GST_WHEEL) as source, zipfile.ZipFile(target, "w") as copy:
        for item in source.infolist():
            data = source.read(item)
            if item.filename == "gst_calc/__init__.py":
                data += b"\nprint('surprise')\n"
            copy.writestr(item, data)
    return target


run_checks(
    [
        ("name", lambda: wheel_summary(GST_WHEEL)["name"], "gst-calc"),
        ("version", lambda: wheel_summary(GST_WHEEL)["version"], "1.0.0"),
        ("tags", lambda: wheel_summary(GST_WHEEL)["tags"], ["py3-none-any"]),
        ("packages", lambda: wheel_summary(GST_WHEEL)["packages"], ["gst_calc"]),
        (
            "console scripts",
            lambda: wheel_summary(GST_WHEEL)["console_scripts"],
            {"gst": "gst_calc:main"},
        ),
        ("three requirements", lambda: len(wheel_summary(GST_WHEEL)["requires_dist"]), 3),
        (
            "one extra requirement",
            lambda: sum("extra ==" in r for r in wheel_summary(GST_WHEEL)["requires_dist"]),
            1,
        ),
        ("RECORD verifies", lambda: wheel_summary(GST_WHEEL)["record_ok"], True),
        ("tampering detected", lambda: wheel_summary(tampered_wheel())["record_ok"], False),
    ]
)

FAIL  name: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  version: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  tags: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  packages: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  console scripts: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  three requirements: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  one extra requirement: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  RECORD verifies: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  tampering detected: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
0/9 checks passed


### Recap — Unit 4.1

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Wheels and tags | `py3-none-any` fits everywhere; `cp315-cp315t-manylinux_2_28_x86_64` fits one interpreter, ABI and platform | prebuilt binaries per platform (`node-pre-gyp`) |
| sdist | Source + `pyproject.toml`; installing it runs the build backend | publishing raw source with an `install` script |
| Frontend / backend | uv and pip call PEP 517 hooks of `uv_build`, `hatchling`, `setuptools`, … | none |
| pip | Backtracking resolver; `-e` editable; `-c` constraints; `freeze` is not a lock | npm |
| uv | `init`, `add`, `lock --upgrade-package`, `sync --locked/--frozen`, `run`, `tool`, `uvx`, scripts, workspaces, `export`, `build`, `publish` | pnpm + npx + nvm |
| `pyproject.toml` | `[build-system]`, `[project]`, scripts, extras, `[dependency-groups]`, `[tool.*]` | `package.json` |
| PEP 440 | `~=1.4` is `>=1.4,==1.*`; pre-releases opt-in; markers and extras | semver ranges |
| Lockfiles | `uv.lock` (universal), `pylock.toml` (standard); apps pin, libraries stay wide | `package-lock.json` |
| Layout | `src/` forces tests to use the installed package; editable = `.pth` | `npm link` |
| Publishing | Inspect the wheel, rehearse on TestPyPI, use trusted publishing; SemVer or CalVer; keep a changelog | `npm publish --provenance` |
| Supply chain | `pip-audit`, hashes, `deptry`, name checks, licenses | `npm audit`, lockfile integrity |

**Further reading**

- [Python Packaging User Guide](https://packaging.python.org/),
  [`pyproject.toml` specification](https://packaging.python.org/en/latest/specifications/pyproject-toml/),
  [Version specifiers](https://packaging.python.org/en/latest/specifications/version-specifiers/),
  [Platform compatibility tags](https://packaging.python.org/en/latest/specifications/platform-compatibility-tags/),
  [Binary distribution format](https://packaging.python.org/en/latest/specifications/binary-distribution-format/)
- [uv documentation](https://docs.astral.sh/uv/) (projects, workspaces, scripts,
  tools, publishing), [pip documentation](https://pip.pypa.io/)
- [`importlib.metadata`](https://docs.python.org/3/library/importlib.metadata.html),
  [`tomllib`](https://docs.python.org/3/library/tomllib.html),
  [`zipfile`](https://docs.python.org/3/library/zipfile.html),
  [`venv`](https://docs.python.org/3/library/venv.html)
- [PEP 440](https://peps.python.org/pep-0440/), [PEP 508](https://peps.python.org/pep-0508/),
  [PEP 517](https://peps.python.org/pep-0517/), [PEP 518](https://peps.python.org/pep-0518/),
  [PEP 621](https://peps.python.org/pep-0621/), [PEP 639](https://peps.python.org/pep-0639/),
  [PEP 660](https://peps.python.org/pep-0660/), [PEP 723](https://peps.python.org/pep-0723/),
  [PEP 735](https://peps.python.org/pep-0735/), [PEP 751](https://peps.python.org/pep-0751/)
- [PyPI trusted publishers](https://docs.pypi.org/trusted-publishers/),
  [Keep a Changelog](https://keepachangelog.com/), [pip-audit](https://github.com/pypa/pip-audit),
  [deptry](https://deptry.com/)

## Unit 4.2 — The Type System (Python's TypeScript)

**Objectives**

- Explain gradual typing: what a type checker checks, what the interpreter ignores,
  and why Python classes are nominal while Protocols are structural.
- Annotate variables, parameters, returns, class and constant attributes, and know
  when annotations are evaluated (*Changed in 3.14*).
- Use collection, union, literal, `TypedDict`, `NewType`, `Self` and generic types;
  narrow unions so the checker can prove your code safe.
- Type callables and decorators precisely (`ParamSpec`, `Concatenate`, `@overload`)
  and write your own narrowing functions (`TypeIs`).
- Read type hints at run time (`Annotated`, `get_type_hints`) the way Pydantic and
  FastAPI do.
- Run mypy and pyright in strict mode, configure them, and adopt them gradually.

**TS bridge: Python's TypeScript**

| TypeScript | Python | The difference that bites |
| --- | --- | --- |
| `tsc` (part of the toolchain) | mypy, pyright (separate tools) | **Python never refuses to run** badly typed code; the interpreter ignores hints |
| Types erased at compile time | Annotations kept at run time as objects | Libraries (Pydantic, FastAPI, dataclasses) read them |
| `let total: number = 0` | `total: int = 0` | `int` and `float` are different types (an `int` is accepted where `float` is expected) |
| `string \| null` (`strictNullChecks`) | `str \| None` | Same idea; mypy always treats `None` strictly |
| `any` | `Any` | Same: disables checking |
| `unknown` | `object` | Must narrow before use |
| `value as Product` | `cast(Product, value)` | Neither checks anything at run time |
| `// @ts-expect-error` | `# type: ignore[code]` | Give the error code so other errors still show |
| Structural typing everywhere | **Nominal** classes, structural `Protocol`s | A class with the right shape is *not* accepted unless you use a Protocol |
| `interface Address { city: string; pin?: string }` | `class Address(TypedDict)` with `NotRequired` | `TypedDict` is only for dicts |
| `"GET" \| "POST"` | `Literal["GET", "POST"]` | Same |
| `function first<T>(xs: T[]): T` | `def first[T](items: Sequence[T]) -> T` | PEP 695 syntax (3.12) |
| `value is Product` (type predicate) | `-> TypeIs[Product]` | 3.13 |
| `never` | `Never` / `NoReturn` | Same |
| `readonly` | `Final`, `ReadOnly`, frozen dataclasses | Checked statically only |
| `.d.ts`, DefinitelyTyped | `.pyi` stubs, typeshed, `types-*` packages | Same idea |
| `"strict": true` in `tsconfig.json` | `mypy --strict`, pyright `strict` | Configure in `pyproject.toml` |

**How the checker demos run.** The interpreter does not check types, so demos that
show checker errors write a small file to `notebooks/_scratch/unit42/` and run the
real mypy and/or pyright on it in a child process: `check(source)` prints both
outputs. Line numbers count from the first line of the source string. Error codes
in brackets (`[arg-type]`, `[reportArgumentType]`) are what you put in an ignore
comment or search for in the docs.

In [73]:
import annotationlib
import functools
import importlib.util
import inspect
import json
import linecache
import os
import re
import subprocess
import sys
import textwrap
import time
import typing
from collections.abc import Callable, Iterable, Mapping, Sequence  # noqa: F401  (lab hints use them)
from dataclasses import dataclass, field, fields
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path
from typing import Annotated, Literal

REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
UNIT42_DIR = scratch_dir / "unit42"
UNIT42_DIR.mkdir(parents=True, exist_ok=True)
VENV_BIN = Path(sys.executable).parent
CHECKER_ENV = {
    **os.environ,
    "NO_COLOR": "1",
    "FORCE_COLOR": "0",
    "MYPY_FORCE_COLOR": "0",
    "PYTHON_COLORS": "0",
}
ANSI_ESCAPE = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")
PAISE = Decimal("0.01")


def write_demo(source: str, name: str, header: str = "") -> Path:
    """Write dedented source to <scratch>/<name>.py and return the path."""
    path = UNIT42_DIR / f"{name}.py"
    path.write_text(header + textwrap.dedent(source).lstrip("\n"))
    return path


def run_mypy(
    source: str, *, name: str = "demo", strict: bool = False, options: tuple[str, ...] = ()
) -> list[str]:
    """Type-check source with mypy in a child process; return its diagnostics, one per line."""
    mypy = VENV_BIN / "mypy"
    if not mypy.exists():
        return ["SKIP: mypy is not installed (uv add --dev mypy)"]
    path = write_demo(source, name)
    command = [
        str(mypy),
        "--no-error-summary",
        "--no-color-output",
        "--no-pretty",
        "--cache-dir",
        str(UNIT42_DIR / ".mypy_cache"),
        *(["--strict"] if strict else []),
        *options,
        path.name,
    ]
    completed = subprocess.run(
        command, cwd=UNIT42_DIR, capture_output=True, text=True, env=CHECKER_ENV, timeout=300
    )
    return (completed.stdout + completed.stderr).strip().splitlines()


def run_pyright(source: str, *, name: str = "demo", strict: bool = False) -> list[str]:
    """Type-check source with pyright (JSON output) in a child process; one line per finding."""
    pyright = VENV_BIN / "pyright"
    if not pyright.exists():
        return ["SKIP: pyright is not installed (uv add --dev pyright)"]
    path = write_demo(source, name, "# pyright: strict\n" if strict else "")
    completed = subprocess.run(
        [str(pyright), "--outputjson", "--pythonpath", sys.executable, path.name],
        cwd=UNIT42_DIR,
        capture_output=True,
        text=True,
        env=CHECKER_ENV,
        timeout=300,
    )
    try:
        report = json.loads(completed.stdout)
    except ValueError:
        return [
            f"SKIP: pyright did not run: {(completed.stderr or completed.stdout).strip()[:120]}"
        ]
    findings = []
    for diagnostic in report["generalDiagnostics"]:
        line_number = diagnostic["range"]["start"]["line"] + 1 - (1 if strict else 0)
        message = " ".join(part.strip() for part in diagnostic["message"].splitlines()[:2])
        rule = f"  [{diagnostic['rule']}]" if diagnostic.get("rule") else ""
        findings.append(f"{path.name}:{line_number}: {diagnostic['severity']}: {message}{rule}")
    return findings


def check(
    source: str,
    *,
    name: str = "demo",
    strict: bool = False,
    checkers: tuple[str, ...] = ("mypy", "pyright"),
) -> None:
    """Print what each checker reports for source (line numbers count from its first line)."""
    for checker in checkers:
        runner = run_mypy if checker == "mypy" else run_pyright
        findings = runner(source, name=name, strict=strict)
        print(f"{checker}{' (strict)' if strict else ''}:")
        print(textwrap.indent("\n".join(findings) or "no issues", "  "))


def source_of(obj: object) -> str:
    """Source code of a function or class, also for classes defined in a notebook cell."""
    try:
        return textwrap.dedent(inspect.getsource(obj))
    except OSError, TypeError:
        if not isinstance(obj, type):
            raise
    # inspect cannot find the file of a notebook class; use the file of one of its functions
    for member in vars(obj).values():
        member = getattr(member, "__func__", None) or getattr(member, "fget", None) or member
        code_object = getattr(inspect.unwrap(member), "__code__", None)
        lines = linecache.getlines(code_object.co_filename) if code_object else []
        if lines:
            start = obj.__firstlineno__ - 1
            while start > 0 and lines[start - 1].lstrip().startswith("@"):
                start -= 1
            return textwrap.dedent("".join(inspect.getblock(lines[start:])))
    # no functions (e.g. an exception class with only a docstring): search the cached cell sources
    header = re.compile(rf"\s*class\s+{obj.__name__}\b")
    for _size, _mtime, lines, _name in reversed(list(linecache.cache.values())):
        if len(lines) >= obj.__firstlineno__ and header.match(lines[obj.__firstlineno__ - 1]):
            return textwrap.dedent("".join(inspect.getblock(lines[obj.__firstlineno__ - 1 :])))
    raise OSError(f"source of {obj.__name__} not found")


LAB_PRELUDE = """
import functools
import typing
import re
import time
from collections.abc import Callable, Iterable, Iterator, Mapping, Sequence
from dataclasses import dataclass, field
from decimal import ROUND_HALF_UP, Decimal
from typing import Annotated, Any, Literal, Self, cast, get_type_hints, overload, reveal_type

PAISE = Decimal("0.01")
"""


@functools.cache
def mypy_report(source: str, strict: bool = True) -> tuple[str, ...]:
    """mypy's lines for LAB_PRELUDE + source (cached: identical source is checked once)."""
    return tuple(run_mypy(LAB_PRELUDE + "\n\n" + source, name="lab_check", strict=strict))


def mypy_errors(*objects: object, strict: bool = True, extra: str = "") -> list[str]:
    """Error lines (no notes) mypy reports for the source of objects, plus extra code."""
    source = "\n\n".join([*(source_of(obj) for obj in objects), textwrap.dedent(extra)])
    return [
        line.split(": error: ", 1)[1] for line in mypy_report(source, strict) if ": error: " in line
    ]


def revealed_types(*objects: object, extra: str = "") -> list[str]:
    """The types mypy reveals for reveal_type(...) calls in extra (after the objects' source)."""
    source = "\n\n".join([*(source_of(obj) for obj in objects), textwrap.dedent(extra)])
    return re.findall(r'Revealed type is "([^"]+)"', "\n".join(mypy_report(source, True)))


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


for tool in ("mypy", "pyright"):
    found = VENV_BIN / tool
    if found.exists():
        version = subprocess.run([str(found), "--version"], capture_output=True, text=True).stdout
        print(f"{tool}: {version.strip()}")
    else:
        print(f"{tool}: not installed; checker cells will print a skip message")

mypy: mypy 2.4.1 (compiled: yes)


pyright: pyright 1.1.414


### 4.2.1 Gradual typing

**PEP 484** (2014) defined the type-hint syntax and its meaning, but left checking to
external tools. Three facts follow:

1. **The interpreter ignores hints.** It stores them and moves on; a badly typed
   call runs (and maybe fails later, or silently does the wrong thing).
2. **Static checkers** (mypy, pyright, and newer ones such as `ty` and Pyrefly)
   read the source without running it and report what cannot be proven safe.
3. **Typing is gradual.** Unannotated code is treated as `Any`, so typed and untyped
   code mix freely and you can add hints one module at a time, like `allowJs` and
   `// @ts-check` in a JavaScript project moving to TypeScript.

The interpreter runs this badly typed call without complaint:

In [74]:
def label(sku: str, quantity: int) -> str:
    """Return a shelf label such as "3 x TEA-500"."""
    return f"{quantity} x {sku}"


print(f"{label('TEA-500', 3)=}")
print(f"{label(3, 'TEA-500')=}  (wrong order: runs anyway, wrong answer)")
print(f"{label.__annotations__=}")

label('TEA-500', 3)='3 x TEA-500'
label(3, 'TEA-500')='TEA-500 x 3'  (wrong order: runs anyway, wrong answer)
label.__annotations__={'sku': <class 'str'>, 'quantity': <class 'int'>, 'return': <class 'str'>}


mypy, given the same code as a file, reports both swapped arguments before anything runs:

In [75]:
check(
    """
    def label(sku: str, quantity: int) -> str:
        return f"{quantity} x {sku}"

    label(3, "TEA-500")
    """,
    checkers=("mypy",),
)

mypy:
  demo.py:4: error: Argument 1 to "label" has incompatible type "int"; expected "str"  [arg-type]
  demo.py:4: error: Argument 2 to "label" has incompatible type "str"; expected "int"  [arg-type]


**Gradual means unannotated code is not checked (by default).** mypy skips the
bodies of functions without annotations and treats their results as `Any`.
`--strict` (or `--check-untyped-defs`) turns that off. pyright *infers* types inside
unannotated functions, so it finds more even in its default mode.

In [76]:
UNTYPED = """
def legacy_total(items):
    return sum(item.price for item in items) + "INR"   # str + number: a bug

def report(prices: list[float]) -> str:
    return legacy_total(prices)                        # Any: accepted silently
"""
check(UNTYPED, name="untyped", checkers=("mypy",))
check(UNTYPED, name="untyped", strict=True, checkers=("mypy",))
check(UNTYPED, name="untyped", checkers=("pyright",))

mypy:
  no issues


mypy (strict):
  untyped.py:1: error: Function is missing a type annotation  [no-untyped-def]
  untyped.py:5: error: Returning Any from function declared to return "str"  [no-any-return]
  untyped.py:5: error: Call to untyped function "legacy_total" in typed context  [no-untyped-call]


pyright:
  untyped.py:2: error: Operator "+" not supported for types "int" and "Literal['INR']"  [reportOperatorIssue]


**Nominal vs structural typing.** TypeScript is *structural*: any value with the
right shape fits an interface or class type. Python classes are *nominal*: a
parameter typed `Duck` accepts `Duck` and its subclasses only. For structural typing
you opt in with a `typing.Protocol` (Unit 2.3.3).

```typescript
class Duck  { quack(): string { return "Quack"; } }
class Robot { quack(): string { return "Beep";  } }
function makeNoise(duck: Duck) { return duck.quack(); }
makeNoise(new Robot());   // OK in TypeScript: same shape
```

In [77]:
check(
    """
    from typing import Protocol

    class Duck:
        def quack(self) -> str:
            return "Quack"

    class Robot:                      # same shape, unrelated class
        def quack(self) -> str:
            return "Beep"

    class Quacker(Protocol):          # structural: anything with quack() -> str
        def quack(self) -> str: ...

    def make_noise(duck: Duck) -> str:
        return duck.quack()

    def make_any_noise(quacker: Quacker) -> str:
        return quacker.quack()

    make_noise(Robot())               # nominal: rejected
    make_any_noise(Robot())           # structural: accepted
    """,
    name="nominal",
    checkers=("mypy",),
)

mypy:
  nominal.py:20: error: Argument 1 to "make_noise" has incompatible type "Robot"; expected "Duck"  [arg-type]


**Predict the output.**

```python
def double(value: int) -> int:
    return value * 2

print(double("ab"), double([0]))
```

<details><summary>Answer</summary>

`abab [0, 0]`. The interpreter never checks hints, and `*` works for strings and
lists. mypy would report both calls (`[arg-type]`); at run time you only find out
when the wrong result causes trouble somewhere else.

</details>

Run the next cell to check your prediction.

In [78]:
def double(value: int) -> int:
    return value * 2


print(double("ab"), double([0]))

abab [0, 0]


### 4.2.2 Annotation syntax

| Where | Syntax | TypeScript |
| --- | --- | --- |
| Variable | `retries: int = 3` | `let retries: number = 3` |
| Declaration only | `customer_name: str` (no value: **no variable is created**) | `let customerName: string;` |
| Parameter, default | `def ship(weight_kg: float, express: bool = False)` | `(weightKg: number, express = false)` |
| `*args`, `**kwargs` | `*skus: str`, `**options: int` (type of **each** item) | `...skus: string[]` |
| Return | `-> Decimal`, `-> None` | `: Decimal`, `: void` |
| Class attribute shared by all instances | `TAX_RATE: ClassVar[Decimal]` | `static TAX_RATE: Decimal` |
| Constant | `MAX_RETRIES: Final = 3` | `const MAX_RETRIES = 3` |

In [79]:
retries: int = 3


class CheckoutSettings:
    currency: str = "INR"
    customer_name: str  # annotation only: recorded, but no attribute is created


def running_total() -> int:
    total: int  # annotation only: no local variable yet
    return total  # noqa: F821  (deliberate: no value was assigned)


def ship(*skus: str, express: bool = False, **weights_kg: float) -> None:
    """Each sku is a str; each keyword argument value is a float."""


print(f"{retries=}  {CheckoutSettings.__annotations__=}")
print(f"{hasattr(CheckoutSettings, 'customer_name')=}")
print(f"{raised_message(running_total)=}")
print(f"{inspect.signature(ship)=}")
print(f"{ship.__annotations__=}")

retries=3  CheckoutSettings.__annotations__={'currency': <class 'str'>, 'customer_name': <class 'str'>}
hasattr(CheckoutSettings, 'customer_name')=False
raised_message(running_total)="UnboundLocalError: cannot access local variable 'total' where it is not associated with a value"
inspect.signature(ship)=<Signature (*skus: str, express: bool = False, **weights_kg: float) -> None>
ship.__annotations__={'skus': <class 'str'>, 'express': <class 'bool'>, 'weights_kg': <class 'float'>, 'return': None}


**`ClassVar` and `Final`.** `ClassVar` marks an attribute that belongs to the class,
not to instances; dataclasses leave `ClassVar` fields out of `__init__`. `Final`
marks a name that must not be re-bound (with no type given, the checker infers it
from the value). Neither is enforced at run time:

In [80]:
from typing import ClassVar, Final

MAX_RETRIES: Final = 3


@dataclass
class InvoiceLine:
    TAX_RATE: ClassVar[Decimal] = Decimal("0.18")  # class-level, not a field
    description: str
    amount: Decimal


line = InvoiceLine("Tea", Decimal("100"))
print(f"{[f.name for f in fields(InvoiceLine)]=}  (TAX_RATE is not a field)")
MAX_RETRIES = 5  # the interpreter allows it; the checker below does not
print(f"{MAX_RETRIES=}")

[f.name for f in fields(InvoiceLine)]=['description', 'amount']  (TAX_RATE is not a field)
MAX_RETRIES=5


In [81]:
check(
    """
    from dataclasses import dataclass
    from decimal import Decimal
    from typing import ClassVar, Final

    MAX_RETRIES: Final = 3
    MAX_RETRIES = 5

    @dataclass
    class InvoiceLine:
        TAX_RATE: ClassVar[Decimal] = Decimal("0.18")
        description: str
        amount: Decimal

    line = InvoiceLine("Tea", Decimal("100"))
    line.TAX_RATE = Decimal("0.05")
    """,
    name="classvar_final",
    checkers=("mypy",),
)

mypy:
  classvar_final.py:6: error: Cannot assign to final name "MAX_RETRIES"  [misc]
  classvar_final.py:15: error: Cannot assign to class variable "TAX_RATE" via instance  [misc]


**Deferred evaluation of annotations (PEP 649 / PEP 749, *Changed in 3.14*).** Until
3.13, annotations were evaluated when the `def` or `class` statement ran, so a class
could not mention a class defined later without quoting it (`"Customer"`) or using
`from __future__ import annotations`. Since 3.14 the compiler stores annotations in
a generated `__annotate__` function and evaluates them **only when someone asks**.
Forward references just work, and modules that never read their annotations do not
pay for them.

In [82]:
class Order:
    customer: Customer  # defined further down: fine in 3.14+

    def ship_to(self) -> Address: ...


print(f"{Order.__annotate__=}")
print(f"{raised_message(lambda: Order.__annotations__)=}  (Customer does not exist yet)")

Order.__annotate__=<function Order.__annotate__ at 0x7f8dee4d0670>
raised_message(lambda: Order.__annotations__)="NameError: name 'Customer' is not defined"  (Customer does not exist yet)


The new standard-library module **`annotationlib`** (*New in 3.14*) reads
annotations in three formats, so tools can cope with names that do not exist yet:

| `Format` | Result for `customer: Customer` before `Customer` exists |
| --- | --- |
| `VALUE` (what `__annotations__` uses) | raises `NameError` |
| `FORWARDREF` | real objects where possible, `ForwardRef('Customer')` otherwise |
| `STRING` | the source text: `'Customer'` |

In [83]:
from annotationlib import Format, get_annotations

print(f"{get_annotations(Order, format=Format.FORWARDREF)=}")
print(f"{get_annotations(Order, format=Format.STRING)=}")
print(f"{get_annotations(Order.ship_to, format=Format.STRING)=}")


class Customer:
    name: str


print(f"{get_annotations(Order, format=Format.VALUE)=}  (now it resolves)")

get_annotations(Order, format=Format.FORWARDREF)={'customer': ForwardRef('Customer', is_class=True, owner=<class '__main__.Order'>)}
get_annotations(Order, format=Format.STRING)={'customer': 'Customer'}
get_annotations(Order.ship_to, format=Format.STRING)={'return': 'Address'}
get_annotations(Order, format=Format.VALUE)={'customer': <class '__main__.Customer'>}  (now it resolves)


> **Note:** `from __future__ import annotations` (PEP 563, stringified annotations)
> is no longer needed on 3.14+ and is expected to be deprecated. Libraries that read
> annotations should call `annotationlib.get_annotations()` or
> `typing.get_type_hints()` rather than touching `__annotations__` directly.

### 4.2.3 Collection types

Built-in collections are generic since 3.9: subscript them with the element type.

| Python | Means | TypeScript |
| --- | --- | --- |
| `list[int]` | list of ints | `number[]` |
| `dict[str, list[User]]` | str keys, values are lists of `User` | `Record<string, User[]>` |
| `set[str]`, `frozenset[str]` | sets of str | `Set<string>` |
| `tuple[int, str]` | **exactly two** items: an int then a str | `[number, string]` |
| `tuple[int, ...]` | **any number** of ints | `readonly number[]` |
| `tuple[()]` | the empty tuple | `[]` |

`typing.List`, `typing.Dict` and friends are deprecated aliases; ruff's `UP006`
rewrites them. At run time, `list[int]` is a `types.GenericAlias` that remembers its
arguments, but `isinstance` cannot check element types:

In [84]:
@dataclass
class User:
    name: str


team_members: dict[str, list[User]] = {"backend": [User("Asha")], "frontend": []}
alias = dict[str, list[User]]
print(f"{type(alias).__name__=}  {typing.get_origin(alias)=}  {typing.get_args(alias)=}")
print(f"{raised_message(lambda: isinstance(team_members, dict[str, list[User]]))=}")
print(f"{typing.get_args(tuple[int, str])=}  {typing.get_args(tuple[int, ...])=}")

type(alias).__name__='GenericAlias'  typing.get_origin(alias)=<class 'dict'>  typing.get_args(alias)=(<class 'str'>, list[__main__.User])
raised_message(lambda: isinstance(team_members, dict[str, list[User]]))='TypeError: isinstance() argument 2 cannot be a parameterized generic'
typing.get_args(tuple[int, str])=(<class 'int'>, <class 'str'>)  typing.get_args(tuple[int, ...])=(<class 'int'>, Ellipsis)


The checker does track element types and tuple lengths:

In [85]:
check(
    """
    from dataclasses import dataclass

    @dataclass
    class User:
        name: str

    point: tuple[int, int] = (1, 2, 3)
    scores: tuple[int, ...] = (90, 85, 77)
    team_members: dict[str, list[User]] = {"backend": [User("Asha")]}
    team_members["frontend"] = User("Ravi")
    team_members["qa"].append("Meena")
    """,
    name="collections_demo",
    checkers=("mypy",),
)

mypy:
  collections_demo.py:7: error: Incompatible types in assignment (expression has type "tuple[int, int, int]", variable has type "tuple[int, int]")  [assignment]
  collections_demo.py:10: error: Incompatible types in assignment (expression has type "User", target has type "list[User]")  [assignment]
  collections_demo.py:11: error: Argument 1 to "append" of "list" has incompatible type "str"; expected "User"  [arg-type]


**Accept abstract types, return concrete ones.** A parameter typed `list[float]`
rejects tuples, sets and generators even when the function only loops over it.
Type parameters with the *least* you need from `collections.abc` and return a
concrete type so callers get every method:

| You only… | Accept | Accepts |
| --- | --- | --- |
| loop once | `Iterable[T]` | list, tuple, set, dict keys, generators, files |
| need `len`, `in` and repeated loops | `Collection[T]` | list, tuple, set, dict views |
| index or slice, read-only | `Sequence[T]` | list, tuple, str, range |
| read a mapping | `Mapping[K, V]` | dict, `MappingProxyType`, `ChainMap` |
| change it in place | `MutableSequence[T]`, `MutableMapping[K, V]` | list / dict |

In [86]:
check(
    """
    from collections.abc import Iterable

    def total_strict(prices: list[float]) -> float:
        return sum(prices)

    def total(prices: Iterable[float]) -> float:
        return sum(prices)

    def prices_with_gst(prices: Iterable[float]) -> list[float]:
        return [round(price * 1.18, 2) for price in prices]   # concrete result

    total_strict((10.0, 20.0))
    total((10.0, 20.0))
    total({10.0, 20.0})
    total(price for price in [10.0, 20.0])
    prices_with_gst((100.0,)).append(5.0)                     # list methods available
    """,
    name="abstract_params",
    checkers=("mypy",),
)

mypy:
  abstract_params.py:12: error: Argument 1 to "total_strict" has incompatible type "tuple[float, float]"; expected "list[float]"  [arg-type]


And at run time the `Iterable` version really does accept all of them:

In [87]:
def total(prices: Iterable[float]) -> float:
    """Sum any iterable of prices."""
    return sum(prices)


print(total([10.0, 20.0]), total((10.0, 20.0)), total({10.0, 20.0}), total(p for p in [10.0, 20.0]))
print(total({"tea": 120.5, "rice": 80.0}.values()))

30.0 30.0 30.0 30.0
200.5


### 4.2.4 Unions and narrowing

`X | Y` (PEP 604, 3.10) means "an `X` or a `Y`"; `X | None` replaces
`Optional[X]`. A union is only safe to use after **narrowing**: a check the checker
understands, after which it knows the more precise type in each branch. These are
the same ideas as TypeScript's control-flow analysis:

| Python check | TypeScript equivalent | Narrows `str \| None` / `int \| str` to |
| --- | --- | --- |
| `if name is None: return` / `if name is not None:` | `if (name === null)` | `str` after the check |
| `if isinstance(value, int):` | `typeof value === "number"` | `int` inside, `str` in `else` |
| `match value:` + `case int():` | `switch` on `typeof` | per `case` |
| `if name:` (truthiness) | `if (name)` | `str` (but `""` is excluded too) |
| early `return` / `raise` / `continue` | same | the rest of the block |
| `assert name is not None` | `name!` (non-null assertion) | `str` (checked at run time unless `-O`) |

First the bug: `find_customer` may return `None`, and the caller forgets. Both
checkers catch it.

In [88]:
UNNARROWED = """
def find_customer(customer_id: int) -> str | None:
    return {1: "Asha", 2: "Ravi"}.get(customer_id)

def greeting(customer_id: int) -> str:
    name = find_customer(customer_id)
    return "Hello, " + name.upper()
"""
check(UNNARROWED, name="unnarrowed")

mypy:
  unnarrowed.py:6: error: Item "None" of "str | None" has no attribute "upper"  [union-attr]


pyright:
  unnarrowed.py:6: error: "upper" is not a known attribute of "None"  [reportOptionalMemberAccess]


At run time the bug only appears for a missing id, which is why the checker is worth having:

In [89]:
def find_customer(customer_id: int) -> str | None:
    """Customer name, or None if unknown."""
    return {1: "Asha", 2: "Ravi"}.get(customer_id)


def greeting(customer_id: int) -> str:
    """Unsafe: forgets that find_customer can return None."""
    name = find_customer(customer_id)
    return "Hello, " + name.upper()


print(f"{greeting(1)=}")
print(f"{raised_message(lambda: greeting(99))=}")

greeting(1)='Hello, ASHA'
raised_message(lambda: greeting(99))="AttributeError: 'NoneType' object has no attribute 'upper'"


Three ways to narrow, each proven safe by both checkers. `reveal_type(x)` is a
checker instruction: it prints the type the checker inferred at that point
(mypy: `note: Revealed type is ...`, pyright: `information: Type of ...`).

In [90]:
check(
    """
    def find_customer(customer_id: int) -> str | None:
        return {1: "Asha", 2: "Ravi"}.get(customer_id)

    def greeting_is_none(customer_id: int) -> str:
        name = find_customer(customer_id)
        if name is None:
            return "Hello, guest"
        reveal_type(name)
        return "Hello, " + name.upper()

    def describe(value: int | str) -> str:
        if isinstance(value, int):
            reveal_type(value)
            return f"{value:,}"
        reveal_type(value)
        return value.strip()

    def describe_match(value: int | str | None) -> str:
        match value:
            case int():
                return f"{value:,}"
            case str():
                return value.strip()
            case None:
                return "-"
    """,
    name="narrowed",
)

mypy:
  narrowed.py:8: note: Revealed type is "str"
  narrowed.py:13: note: Revealed type is "int"
  narrowed.py:15: note: Revealed type is "str"


pyright:
  narrowed.py:8: information: Type of "name" is "str"
  narrowed.py:13: information: Type of "value" is "int"
  narrowed.py:15: information: Type of "value" is "str"


**`Never`, `NoReturn` and `assert_never`.** `Never` (alias `NoReturn`, the
conventional spelling for return types) is the empty type, TypeScript's `never`:

- A function that always raises returns `NoReturn`; code after a call to it is
  unreachable, so it narrows like a `return`.
- After every possibility of a union has been handled, the remaining type is
  `Never`. Passing the value to `assert_never()` (whose parameter is `Never`) makes
  the checker prove the `match` is **exhaustive**: add a new option and forget to
  handle it, and you get an error at that line.

In [91]:
EXHAUSTIVE = """
from typing import Literal, NoReturn, assert_never

type Speed = Literal["standard", "express", "same_day"]   # "same_day" was just added

def fail(message: str) -> NoReturn:
    raise ValueError(message)

def delivery_days(speed: Speed) -> int:
    match speed:
        case "standard":
            return 5
        case "express":
            return 2
        case _:
            assert_never(speed)          # same_day is not handled

def parse_days(text: str | None) -> int:
    if text is None:
        fail("missing value")
    return int(text)                    # text is str here: fail() never returns
"""
check(EXHAUSTIVE, name="exhaustive")

mypy:
  exhaustive.py:15: error: Argument 1 to "assert_never" has incompatible type "Literal['same_day']"; expected "Never"  [arg-type]


pyright:
  exhaustive.py:15: error: Argument of type "Literal['same_day']" cannot be assigned to parameter "arg" of type "Never" in function "assert_never" Type "Literal['same_day']" is not assignable to type "Never"  [reportArgumentType]


At run time `assert_never` raises `AssertionError`, a safety net for values the
checker could not see (data from JSON, for example):

In [92]:
from typing import Literal, Never, NoReturn, assert_never

type Speed = Literal["standard", "express"]


def delivery_days(speed: Speed) -> int:
    """Days to deliver; exhaustive over Speed."""
    match speed:
        case "standard":
            return 5
        case "express":
            return 2
        case _:
            assert_never(speed)


print(f"{delivery_days('express')=}")
print(f"{raised_message(lambda: delivery_days('drone'))=}")
print(f"{NoReturn!r} vs {Never!r}: different objects at run time, the same type to checkers")

delivery_days('express')=2
raised_message(lambda: delivery_days('drone'))="AssertionError: Expected code to be unreachable, but got: 'drone'"
typing.NoReturn vs typing.Never: different objects at run time, the same type to checkers


### 4.2.5 Escape hatches

| Hatch | Effect | TypeScript | Use it when |
| --- | --- | --- | --- |
| `Any` | Everything is allowed, in both directions; checking is off for that value | `any` | Truly dynamic data you will validate elsewhere; gradual migration |
| `object` | Anything can be passed in, but **nothing** can be done with it until you narrow | `unknown` | "I accept anything" parameters (logging, serializers) |
| `cast(T, value)` | Tells the checker "trust me, this is a `T`"; does nothing at run time | `value as T` | After a check the checker cannot follow |
| `# type: ignore[code]` | Silences one error code on one line | `// @ts-expect-error` | Checker bugs, untyped third-party code |

In [93]:
check(
    """
    import json
    from typing import Any, cast

    def log_any(value: Any) -> None:
        print(value.upper())        # Any: no error, may crash at run time

    def log_object(value: object) -> None:
        print(value.upper())        # object: error until narrowed

    def log_narrowed(value: object) -> None:
        if isinstance(value, str):
            print(value.upper())

    config = cast(dict[str, int], json.loads('{"retries": 3}'))
    reveal_type(json.loads("{}"))
    reveal_type(config)
    """,
    name="any_object",
    checkers=("mypy",),
)

mypy:
  any_object.py:8: error: "object" has no attribute "upper"  [attr-defined]
  any_object.py:15: note: Revealed type is "Any"
  any_object.py:16: note: Revealed type is "dict[str, int]"


`cast` is a lie the interpreter does not check: it returns its argument unchanged.
If the data does not match, the error appears later, far from the cause. Prefer
narrowing with `isinstance` or validating with a library such as Pydantic
(4.2.11).

In [94]:
from typing import cast

port = cast(int, "8080")  # the checker now believes port is an int
print(f"{port=}  {type(port).__name__=}")
print(f"{raised_message(lambda: port + 1)=}")

port='8080'  type(port).__name__='str'
raised_message(lambda: port + 1)='TypeError: can only concatenate str (not "int") to str'


**`# type: ignore[code]`.** Always give the error code: a bare `# type: ignore`
hides every error on the line, including ones you introduce later. With
`--warn-unused-ignores` mypy reports ignores that no longer suppress anything, and
`--enable-error-code ignore-without-code` rejects bare ones (both are in `--strict`
or recommended with it). pyright's own form is `# pyright: ignore[ruleName]`.

In [95]:
check(
    """
    def parse_port(text: str) -> int:
        return int(text)

    a: int = parse_port(8080)  # type: ignore[arg-type]
    b: int = parse_port(8080)  # type: ignore[assignment]
    c: int = parse_port("80")  # type: ignore[arg-type]
    d: int = parse_port(8080)  # type: ignore
    """,
    name="ignores",
    checkers=("mypy",),
)
STRICT_IGNORES = ("--warn-unused-ignores", "--enable-error-code", "ignore-without-code")
print("mypy --warn-unused-ignores --enable-error-code ignore-without-code:")
for finding in run_mypy(
    (UNIT42_DIR / "ignores.py").read_text(), name="ignores", options=STRICT_IGNORES
):
    print(f"  {finding}")

mypy:
  ignores.py:5: error: Argument 1 to "parse_port" has incompatible type "int"; expected "str"  [arg-type]
  ignores.py:5: note: Error code "arg-type" not covered by "type: ignore[assignment]" comment
mypy --warn-unused-ignores --enable-error-code ignore-without-code:


  ignores.py:5: error: Unused "type: ignore" comment  [unused-ignore]
  ignores.py:5: error: Argument 1 to "parse_port" has incompatible type "int"; expected "str"  [arg-type]
  ignores.py:5: note: Error code "arg-type" not covered by "type: ignore[assignment]" comment
  ignores.py:6: error: Unused "type: ignore" comment  [unused-ignore]
  ignores.py:7: error: "type: ignore" comment without error code (consider "type: ignore[arg-type]" instead)  [ignore-without-code]


### 4.2.6 Precise types

| Type | Means | TypeScript |
| --- | --- | --- |
| `Literal["GET", "POST"]` | Only these exact values | `"GET" \| "POST"` |
| `LiteralString` | A `str` built only from literals in the source (no user input) | none (a "template literal" brand) |
| `Final` | Never re-bound; with no type given, infers a literal type | `const` |
| `NewType("UserId", int)` | A distinct type for the checker, a plain `int` at run time | branded types (`number & {__brand: "UserId"}`) |
| `Self` | "The class of the current instance" (subclasses included) | `this` type |

**`Literal` and `LiteralString`.** `LiteralString` (PEP 675) protects APIs such as
SQL execution: only strings written in the source, or combined from such strings,
are accepted, so an f-string with user input is rejected. pyright implements it;
mypy currently treats `LiteralString` as plain `str`, a real difference between the
two checkers:

In [96]:
check(
    """
    from typing import Literal, LiteralString

    type Method = Literal["GET", "POST", "DELETE"]

    def request(method: Method, path: str) -> str:
        return f"{method} {path}"

    def run_query(sql: LiteralString) -> None:
        print(sql)

    request("GET", "/orders")
    request("PATCH", "/orders")

    def find_orders(customer: str) -> None:
        run_query("SELECT * FROM orders WHERE customer = %s")      # literal: OK
        run_query(f"SELECT * FROM orders WHERE customer = '{customer}'")
    """,
    name="literals",
)

mypy:
  literals.py:12: error: Argument 1 to "request" has incompatible type "Literal['PATCH']"; expected "Literal['GET', 'POST', 'DELETE']"  [arg-type]


pyright:
  literals.py:12: error: Argument of type "Literal['PATCH']" cannot be assigned to parameter "method" of type "Method" in function "request" Type "Literal['PATCH']" is not assignable to type "Method"  [reportArgumentType]
  literals.py:16: error: Argument of type "str" cannot be assigned to parameter "sql" of type "LiteralString" in function "run_query" "str" is not assignable to "LiteralString"  [reportArgumentType]


**`Final` infers literal types.** `TIMEOUT: Final = 30` has the type `Literal[30]`
where it is used, so it can feed a `Literal`-typed parameter:

In [97]:
check(
    """
    from typing import Final, Literal

    DEFAULT_PAGE_SIZE: Final = 20
    page_size = 20

    def fetch(size: Literal[10, 20, 50]) -> None: ...

    fetch(DEFAULT_PAGE_SIZE)     # Literal[20]: OK
    fetch(page_size)             # int: could be anything
    reveal_type(DEFAULT_PAGE_SIZE)
    """,
    name="final_literal",
    checkers=("mypy",),
)

mypy:
  final_literal.py:9: error: Argument 1 to "fetch" has incompatible type "int"; expected "Literal[10, 20, 50]"  [arg-type]
  final_literal.py:10: note: Revealed type is "Literal[20]?"


**`NewType` and `Self`.** `NewType` creates a distinct type for the checker (so a
`UserId` cannot be passed where an `OrderId` is expected) at zero run-time cost: the
"class" is an identity function. `Self` (3.11) types fluent APIs and alternative
constructors so subclasses get their own type back.

In [98]:
from typing import NewType, Self

UserId = NewType("UserId", int)
OrderId = NewType("OrderId", int)
user_id = UserId(42)
print(f"{user_id=}  {type(user_id).__name__=}  {user_id == 42=}  {UserId=}")


class QueryBuilder:
    """Fluent builder: every method returns the builder itself."""

    def __init__(self) -> None:
        self.conditions: list[str] = []

    def where(self, condition: str) -> Self:
        self.conditions.append(condition)
        return self


class OrderQuery(QueryBuilder):
    def unpaid(self) -> Self:
        return self.where("paid = false")


query = OrderQuery().where("city = 'Pune'").unpaid()  # where() returned OrderQuery
print(f"{type(query).__name__=}  {query.conditions=}")

user_id=42  type(user_id).__name__='int'  user_id == 42=True  UserId=__main__.UserId
type(query).__name__='OrderQuery'  query.conditions=["city = 'Pune'", 'paid = false']


In [99]:
check(
    """
    from typing import NewType, Self

    UserId = NewType("UserId", int)
    OrderId = NewType("OrderId", int)

    def cancel_order(order_id: OrderId) -> None: ...

    cancel_order(UserId(42))
    cancel_order(42)
    cancel_order(OrderId(42))

    class QueryBuilder:
        def where(self, condition: str) -> Self:
            return self

    class OrderQuery(QueryBuilder):
        def unpaid(self) -> Self:
            return self.where("paid = false")

    reveal_type(OrderQuery().where("city = 'Pune'"))
    """,
    name="newtype_self",
    checkers=("mypy",),
)

mypy:
  newtype_self.py:8: error: Argument 1 to "cancel_order" has incompatible type "UserId"; expected "OrderId"  [arg-type]
  newtype_self.py:9: error: Argument 1 to "cancel_order" has incompatible type "int"; expected "OrderId"  [arg-type]
  newtype_self.py:20: note: Revealed type is "newtype_self.OrderQuery"


**`TypedDict`** types dictionaries with a fixed set of string keys, such as JSON
payloads. It is TypeScript's `interface` for objects that stay plain dicts:

| Python | TypeScript |
| --- | --- |
| `class Address(TypedDict):` + `city: str` | `interface Address { city: string }` |
| `pin: NotRequired[str]` | `pin?: string` |
| `class Patch(TypedDict, total=False):` + `name: Required[str]` | all optional except `name` |
| `sku: ReadOnly[str]` (PEP 705, *New in 3.13*) | `readonly sku: string` |
| `def create(**fields: Unpack[ProductFields])` (PEP 692) | `function create(fields: ProductFields)` |

At run time a `TypedDict` *is* a `dict`: no validation, no class instance. The
class only records which keys are required, optional and read-only.

In [100]:
from typing import NotRequired, ReadOnly, Required, TypedDict, Unpack


class Address(TypedDict):
    city: str
    pin: NotRequired[str]


class ProductPatch(TypedDict, total=False):
    sku: Required[ReadOnly[str]]
    name: str
    price: Decimal


def update_product(**changes: Unpack[ProductPatch]) -> dict[str, object]:
    """Apply a partial update; keyword arguments are checked against ProductPatch."""
    return dict(changes)


home: Address = {"city": "Pune"}
print(f"{home=}  {type(home).__name__=}")
print(f"{Address.__required_keys__=}  {Address.__optional_keys__=}")
print(f"{ProductPatch.__required_keys__=}  {ProductPatch.__readonly_keys__=}")
print(f"{update_product(sku='TEA-500', price=Decimal('120.50'))=}")

home={'city': 'Pune'}  type(home).__name__='dict'
Address.__required_keys__=frozenset({'city'})  Address.__optional_keys__=frozenset({'pin'})
ProductPatch.__required_keys__=frozenset({'sku'})  ProductPatch.__readonly_keys__=frozenset({'sku'})
update_product(sku='TEA-500', price=Decimal('120.50'))={'sku': 'TEA-500', 'price': Decimal('120.50')}


The checker enforces the keys, their types, read-only keys and the `**kwargs` names:

In [101]:
check(
    """
    from decimal import Decimal
    from typing import NotRequired, ReadOnly, Required, TypedDict, Unpack

    class Address(TypedDict):
        city: str
        pin: NotRequired[str]

    class ProductPatch(TypedDict, total=False):
        sku: Required[ReadOnly[str]]
        name: str
        price: Decimal

    def update_product(**changes: Unpack[ProductPatch]) -> None: ...

    office: Address = {"pin": "411001"}                   # city missing
    shop: Address = {"city": "Pune", "zip": "411001"}     # unknown key
    patch: ProductPatch = {"sku": "TEA-500", "name": "Tea"}
    patch["sku"] = "TEA-1KG"                             # read-only
    update_product(sku="TEA-500", colour="green")        # unknown keyword
    update_product(price=99)                             # sku missing, int is not Decimal
    """,
    name="typed_dicts",
    checkers=("mypy",),
)

mypy:
  typed_dicts.py:15: error: Missing key "city" for TypedDict "Address"  [typeddict-item]
  typed_dicts.py:16: error: Extra key "zip" for TypedDict "Address"  [typeddict-unknown-key]
  typed_dicts.py:18: error: ReadOnly TypedDict key "sku" TypedDict is mutated  [typeddict-readonly-mutated]
  typed_dicts.py:19: error: Unexpected keyword argument "colour" for "update_product"  [call-arg]
  typed_dicts.py:20: error: Missing named argument "sku" for "update_product"  [call-arg]
  typed_dicts.py:20: error: Argument "price" to "update_product" has incompatible type "int"; expected "Decimal"  [arg-type]


### 4.2.7 Generics

**PEP 695 syntax** (3.12) declares type parameters in square brackets right after
the name, like TypeScript's angle brackets:

| Python 3.12+ | TypeScript |
| --- | --- |
| `def first[T](items: Sequence[T]) -> T:` | `function first<T>(items: T[]): T` |
| `class Stack[T]:` | `class Stack<T>` |
| `type Pair[T] = tuple[T, T]` | `type Pair<T> = [T, T]` |
| `[T: Hashable]` (bound) | `<T extends Hashable>` |
| `[T: (int, float)]` (constraints) | `<T extends number \| bigint>`, roughly |
| `[T = str]` (default, PEP 696, 3.13) | `<T = string>` |

Type parameters are real objects at run time, available as `__type_params__`:

In [102]:
def first[T](items: Sequence[T]) -> T:
    """The first item; the result has the element type of the sequence."""
    return items[0]


class Stack[T]:
    """A last-in, first-out stack of T."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)


type Pair[T] = tuple[T, T]

print(f"{first.__type_params__=}  {Stack.__type_params__=}")
print(f"{Stack[int]=}  {typing.get_args(Stack[int])=}")
print(f"{type(Pair).__name__=}  {Pair.__value__=}  {Pair[float]=}")

first.__type_params__=(T,)  Stack.__type_params__=(T,)
Stack[int]=__main__.Stack[int]  typing.get_args(Stack[int])=(<class 'int'>,)
type(Pair).__name__='TypeAliasType'  Pair.__value__=tuple[T, T]  Pair[float]=__main__.Pair[float]


`type` aliases are evaluated lazily (`__value__` is computed on first access), so
they can be **recursive**: the classic JSON type needs no quotes.

In [103]:
type Json = dict[str, Json] | list[Json] | str | int | float | bool | None


def count_leaves(value: Json) -> int:
    """Number of non-container values in a JSON document."""
    if isinstance(value, dict):
        return sum(count_leaves(item) for item in value.values())
    if isinstance(value, list):
        return sum(count_leaves(item) for item in value)
    return 1


print(f"{count_leaves({'order': {'id': 7, 'lines': [1, 2, {'sku': 'TEA'}]}, 'paid': True})=}")
print(f"{Json.__value__=}")

count_leaves({'order': {'id': 7, 'lines': [1, 2, {'sku': 'TEA'}]}, 'paid': True})=5
Json.__value__=dict[str, __main__.Json] | list[__main__.Json] | str | int | float | bool | None


**Bounds vs constraints, and defaults.** A *bound* `[T: Base]` accepts `Base` or any
subtype and keeps the exact type (`bool` stays `bool`). *Constraints*
`[T: (int, str)]` mean `T` is exactly one of the listed types (a `bool` is solved
as `int`). A *default* (PEP 696, *New in 3.13*) is used when the parameter is not
given, which is how `Generator[int]` can omit its other two parameters.

In [104]:
check(
    """
    from collections.abc import Hashable, Sequence

    def bounded[T: int](value: T) -> T:
        return value

    def constrained[T: (int, str)](value: T) -> T:
        return value

    def dedupe[T: Hashable](items: Sequence[T]) -> list[T]:
        return list(dict.fromkeys(items))

    class Page[T = dict[str, object]]:
        def __init__(self, items: list[T]) -> None:
            self.items = items

    def plain_page() -> Page:
        return Page([])

    reveal_type(bounded(True))
    reveal_type(constrained(True))
    reveal_type(dedupe(["a", "b", "a"]))
    dedupe([[1], [2]])
    reveal_type(plain_page().items)
    """,
    name="bounds",
    checkers=("mypy",),
)

mypy:
  bounds.py:19: note: Revealed type is "bool"
  bounds.py:20: note: Revealed type is "int"
  bounds.py:21: note: Revealed type is "list[str]"
  bounds.py:22: error: Value of type variable "T" of "dedupe" cannot be "list[int]"  [type-var]
  bounds.py:23: note: Revealed type is "list[dict[str, object]]"


In [105]:
class Page[T = dict[str, object]]:
    """A page of results; items default to JSON-like dicts."""


(page_param,) = Page.__type_params__
print(f"{page_param.has_default()=}  {page_param.__default__=}")


def dedupe[T: typing.Hashable](items: Sequence[T]) -> list[T]:
    """Drop repeated items, keeping the first of each."""
    return list(dict.fromkeys(items))


(dedupe_param,) = dedupe.__type_params__
print(f"{dedupe_param.__bound__=}  {dedupe([3, 1, 3, 2, 1])=}")

page_param.has_default()=True  page_param.__default__=dict[str, object]
dedupe_param.__bound__=typing.Hashable  dedupe([3, 1, 3, 2, 1])=[3, 1, 2]


**Variance** answers: if `bool` is a subtype of `int`, is `Box[bool]` a subtype of
`Box[int]`?

| Variance | Rule | Examples | Why |
| --- | --- | --- | --- |
| Covariant | `Box[bool]` **is** a `Box[int]` | `Sequence`, `tuple`, `frozenset`, `Mapping` values, read-only classes | You can only *read* `int`s out of it |
| Invariant | No relation | `list`, `dict`, `set`, mutable classes | Someone could *write* an `int` into your list of `bool`s |
| Contravariant | `Handler[int]` **is** a `Handler[bool]` | `Callable` parameters, write-only sinks | Something that handles any `int` can handle a `bool` |

TypeScript checks method parameters *bivariantly* (unsound on purpose); Python
checkers are strict here. With PEP 695 syntax the checker **infers** variance from
how the class uses `T`.

In [106]:
check(
    """
    from collections.abc import Sequence
    from dataclasses import dataclass

    def average(values: list[float]) -> float:
        return sum(values) / len(values)

    def average_ro(values: Sequence[float]) -> float:
        return sum(values) / len(values)

    flags: list[bool] = [True, False]
    ints: list[int] = [1, 2, 3]
    average(ints)          # list is invariant
    average_ro(ints)       # Sequence is covariant

    @dataclass(frozen=True)
    class ReadOnlyBox[T]:  # only reads T: inferred covariant
        item: T

    class Box[T]:          # reads and writes T: inferred invariant
        def __init__(self, item: T) -> None:
            self.item = item

    def open_int_box(box: ReadOnlyBox[int]) -> int:
        return box.item

    def open_mutable(box: Box[int]) -> int:
        return box.item

    open_int_box(ReadOnlyBox(True))
    bool_box: Box[bool] = Box(True)
    open_mutable(bool_box)
    """,
    name="variance",
    checkers=("mypy",),
)

mypy:
  variance.py:12: error: Argument 1 to "average" has incompatible type "list[int]"; expected "list[float]"  [arg-type]
  variance.py:12: note: "list" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
  variance.py:12: note: Consider using "Sequence" instead, which is covariant
  variance.py:31: error: Argument 1 to "open_mutable" has incompatible type "Box[bool]"; expected "Box[int]"  [arg-type]


**The old `TypeVar` syntax** is what you will read in most existing code (and still
write when supporting Python ≤ 3.11). Same meaning, more ceremony: the type variable
is a module-level object, classes inherit from `Generic[T]`, and variance must be
declared by hand. ruff's `UP046`/`UP047` rewrite it to PEP 695 syntax for
`target-version = "py312"` and later, hence the `noqa` comments.

In [107]:
from typing import Generic, TypeVar

T = TypeVar("T")
T_co = TypeVar("T_co", covariant=True)
Number = TypeVar("Number", int, float)  # constraints
AnyStr_ = TypeVar("AnyStr_", bound=str | bytes)  # bound


class OldBox(Generic[T]):  # noqa: UP046
    def __init__(self, item: T) -> None:
        self.item = item


def old_first(items: Sequence[T]) -> T:  # noqa: UP047
    """Same as first[T], old spelling."""
    return items[0]


print(f"{OldBox.__parameters__=}  {OldBox[int]=}")
print(f"{T_co.__covariant__=}  {Number.__constraints__=}  {AnyStr_.__bound__=}")
new_param = Stack.__type_params__[0]
print(f"PEP 695 parameter: {new_param!r}, infer variance: {new_param.__infer_variance__}")

OldBox.__parameters__=(~T,)  OldBox[int]=__main__.OldBox[int]
T_co.__covariant__=True  Number.__constraints__=(<class 'int'>, <class 'float'>)  AnyStr_.__bound__=str | bytes
PEP 695 parameter: T, infer variance: True


### 4.2.8 Callable types

| Python | TypeScript |
| --- | --- |
| `Callable[[int, str], bool]` | `(a: number, b: string) => boolean` |
| `Callable[..., Decimal]` | `(...args: any[]) => Decimal` |
| `Callable[[], None]` | `() => void` |
| callback `Protocol` with `__call__` | call signature in an interface: `{ (amount: Decimal): Decimal }` |

`Callable` cannot describe keyword names, defaults or overloads. For those, write a
**callback Protocol**: a Protocol whose `__call__` has the exact signature.

In [108]:
check(
    """
    from collections.abc import Callable
    from decimal import Decimal
    from typing import Protocol

    class PriceRule(Protocol):
        def __call__(self, amount: Decimal, *, tier: str = "basic") -> Decimal: ...

    def ten_percent_off(amount: Decimal, *, tier: str = "basic") -> Decimal:
        return amount * Decimal("0.9")

    def wrong_keyword(amount: Decimal, *, level: str = "basic") -> Decimal:
        return amount

    def simple(amount: Decimal) -> Decimal:
        return amount

    def apply(rule: PriceRule, amount: Decimal) -> Decimal:
        return rule(amount, tier="gold")

    def apply_simple(rule: Callable[[Decimal], Decimal], amount: Decimal) -> Decimal:
        return rule(amount)

    apply(ten_percent_off, Decimal("100"))
    apply(wrong_keyword, Decimal("100"))
    apply(simple, Decimal("100"))
    apply_simple(simple, Decimal("100"))
    apply_simple(len, Decimal("100"))
    """,
    name="callables",
    checkers=("mypy",),
)

mypy:
  callables.py:24: error: Argument 1 to "apply" has incompatible type "def wrong_keyword(amount: Decimal, *, level: str = ...) -> Decimal"; expected "PriceRule"  [arg-type]
  callables.py:24: note: "PriceRule.__call__" has type "def __call__(self, amount: Decimal, *, tier: str = ...) -> Decimal"
  callables.py:25: error: Argument 1 to "apply" has incompatible type "Callable[[Decimal], Decimal]"; expected "PriceRule"  [arg-type]
  callables.py:25: note: "PriceRule.__call__" has type "def __call__(self, amount: Decimal, *, tier: str = ...) -> Decimal"
  callables.py:27: error: Argument 1 to "apply_simple" has incompatible type "Callable[[Sized], int]"; expected "Callable[[Decimal], Decimal]"  [arg-type]


**`ParamSpec`: decorators that keep the signature.** A decorator typed with
`Callable[..., Any]` erases what it wraps: callers lose argument checking and the
return type becomes `Any`. `[**P, R]` captures the wrapped function's parameters as
`P` and its return type as `R`, so the wrapper keeps both (Unit 1.12 used this
spelling already).

In [109]:
check(
    """
    import functools
    from collections.abc import Callable
    from typing import Any

    def logged_loose(func: Callable[..., Any]) -> Callable[..., Any]:
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            return func(*args, **kwargs)
        return wrapper

    def logged[**P, R](func: Callable[P, R]) -> Callable[P, R]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            print(f"calling {func.__name__}")
            return func(*args, **kwargs)
        return wrapper

    @logged_loose
    def price_loose(sku: str, quantity: int = 1) -> float:
        return 10.0 * quantity

    @logged
    def price(sku: str, quantity: int = 1) -> float:
        return 10.0 * quantity

    price_loose(42)            # not caught
    price(42)                  # caught
    reveal_type(price)
    reveal_type(price_loose("TEA"))
    """,
    name="paramspec",
    checkers=("mypy",),
)

mypy:
  paramspec.py:26: error: Argument 1 to "price" has incompatible type "int"; expected "str"  [arg-type]
  paramspec.py:27: note: Revealed type is "def (sku: str, quantity: int =) -> float"
  paramspec.py:28: note: Revealed type is "Any"


**`Concatenate`** describes a decorator that **adds or removes** leading
parameters. A classic case is dependency injection: the decorated function takes a
connection as its first parameter, and the wrapper supplies it, so callers do not.
Decorator *factories* (decorators with arguments) just add one more layer:
`def retry(times: int) -> Callable[[Callable[P, R]], Callable[P, R]]`, which is
Lab 4.2 exercise 3.

In [110]:
from typing import Concatenate


class Connection:
    def execute(self, sql: str) -> list[tuple[object, ...]]:
        return [(sql.split()[-1], 1)]


def with_connection[**P, R](func: Callable[Concatenate[Connection, P], R]) -> Callable[P, R]:
    """Supply a fresh Connection as the first argument."""

    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        return func(Connection(), *args, **kwargs)

    return wrapper


@with_connection
def count_orders(connection: Connection, city: str) -> int:
    return len(connection.execute(f"SELECT 1 FROM orders WHERE city = {city}"))


print(f"{count_orders('Pune')=}  {inspect.signature(count_orders)=}")

count_orders('Pune')=1  inspect.signature(count_orders)=<Signature (connection: __main__.Connection, city: str) -> int>


Note that `inspect.signature` (which follows `__wrapped__`) still shows the original
parameters; only the checker knows the connection parameter was removed. mypy
agrees with the run-time behaviour:

In [111]:
check(
    """
    import functools
    from collections.abc import Callable
    from typing import Concatenate

    class Connection: ...

    def with_connection[**P, R](func: Callable[Concatenate[Connection, P], R]) -> Callable[P, R]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            return func(Connection(), *args, **kwargs)
        return wrapper

    @with_connection
    def count_orders(connection: Connection, city: str) -> int:
        return 1

    reveal_type(count_orders)
    count_orders("Pune")
    count_orders(Connection(), "Pune")
    """,
    name="concatenate",
    checkers=("mypy",),
)

mypy:
  concatenate.py:17: note: Revealed type is "def (city: str) -> int"
  concatenate.py:19: error: Too many arguments for "count_orders"  [call-arg]
  concatenate.py:19: error: Argument 1 to "count_orders" has incompatible type "Connection"; expected "str"  [arg-type]


**`@overload`** declares several signatures for one function when the return type
depends on the argument types (TypeScript function overloads). The `@overload`
stubs are for the checker only; the last, undecorated definition is the real
implementation. `typing.get_overloads()` (3.11) lists them at run time.

In [112]:
from typing import get_overloads, overload


@overload
def parse_amount(raw: str) -> Decimal: ...
@overload
def parse_amount(raw: None) -> None: ...
def parse_amount(raw: str | None) -> Decimal | None:
    """Parse "1,234.50" into Decimal; None stays None."""
    return None if raw is None else Decimal(raw.replace(",", ""))


print(f"{parse_amount('1,234.50')=}  {parse_amount(None)=}")
print(f"{len(get_overloads(parse_amount))=}")
check(
    """
    from decimal import Decimal
    from typing import overload

    @overload
    def parse_amount(raw: str) -> Decimal: ...
    @overload
    def parse_amount(raw: None) -> None: ...
    def parse_amount(raw: str | None) -> Decimal | None:
        return None if raw is None else Decimal(raw.replace(",", ""))

    reveal_type(parse_amount("1,234.50"))
    reveal_type(parse_amount(None))
    parse_amount(12)
    """,
    name="overloads",
    checkers=("mypy",),
)

parse_amount('1,234.50')=Decimal('1234.50')  parse_amount(None)=None
len(get_overloads(parse_amount))=2
mypy:
  overloads.py:11: note: Revealed type is "decimal.Decimal"
  overloads.py:12: note: Revealed type is "None"
  overloads.py:13: error: No overload variant of "parse_amount" matches argument type "int"  [call-overload]
  overloads.py:13: note: Possible overload variants:
  overloads.py:13: note:     def parse_amount(raw: str) -> Decimal
  overloads.py:13: note:     def parse_amount(raw: None) -> None


### 4.2.9 Class typing

| Tool | Means | TypeScript |
| --- | --- | --- |
| `type[Product]` | The class `Product` itself (or a subclass), not an instance | `typeof Product` / `new () => Product` |
| `@override` (PEP 698, 3.12) | "This method overrides a base method"; error if it does not | `override` keyword |
| `@final` on a class | Must not be subclassed | none (`sealed` in C#) |
| `@final` on a method | Must not be overridden | none |
| `class Repository[T](Protocol)` | A generic structural interface | `interface Repository<T>` |

At run time `@override` and `@final` only set a marker attribute (`__override__`,
`__final__`); the checker enforces them.

In [113]:
from typing import final, override


class Model:
    def describe(self) -> str:
        return f"{type(self).__name__} stored in {type(self).__name__.lower()}s"


class Product(Model):
    @override
    def describe(self) -> str:
        return "Product stored in the catalogue"


def create[M: Model](model_class: type[M]) -> M:
    """Instantiate any Model subclass; the result has that subclass's type."""
    return model_class()


@final
class Money: ...


print(f"{create(Product).describe()=}  {create(Model).describe()=}")
print(f"{Product.describe.__override__=}  {Money.__final__=}")

create(Product).describe()='Product stored in the catalogue'  create(Model).describe()='Model stored in models'
Product.describe.__override__=True  Money.__final__=True


A generic Protocol plus the class-typing errors the checker reports:

In [114]:
check(
    """
    from typing import Protocol, final, override

    class Repository[T](Protocol):
        def get(self, key: str) -> T | None: ...
        def add(self, key: str, item: T) -> None: ...

    class InMemoryRepository[T]:
        def __init__(self) -> None:
            self._items: dict[str, T] = {}
        def get(self, key: str) -> T | None:
            return self._items.get(key)
        def add(self, key: str, item: T) -> None:
            self._items[key] = item

    def restock(repository: Repository[int], sku: str) -> None:
        repository.add(sku, (repository.get(sku) or 0) + 10)

    restock(InMemoryRepository[int](), "TEA-500")
    restock(InMemoryRepository[str](), "TEA-500")

    class Shipment:
        def cost(self) -> int:
            return 50
        @final
        def tracking_id(self) -> str:
            return "TRK-1"

    class Express(Shipment):
        @override
        def costs(self) -> int:       # typo: overrides nothing
            return 120
        def tracking_id(self) -> str:  # overrides a final method
            return "EXP-1"

    @final
    class Money: ...

    class Rupees(Money): ...

    def build(kind: type[Shipment]) -> Shipment:
        return kind()

    build(Express)
    build(Money)
    """,
    name="class_typing",
    checkers=("mypy",),
)

mypy:
  class_typing.py:19: error: Argument 1 to "restock" has incompatible type "InMemoryRepository[str]"; expected "Repository[int]"  [arg-type]
  class_typing.py:30: error: Method "costs" is marked as an override, but no base method was found with this name  [misc]
  class_typing.py:32: error: Cannot override final attribute "tracking_id" (previously declared in base class "Shipment")  [misc]
  class_typing.py:38: error: Cannot inherit from final class "Money"  [misc]
  class_typing.py:44: error: Argument 1 to "build" has incompatible type "type[Money]"; expected "type[Shipment]"  [arg-type]


### 4.2.10 Custom narrowing

When a check is too complex for the checker to follow, write a function that
returns `bool` and tell the checker what `True` means:

| | `TypeGuard[T]` (PEP 647, 3.10) | `TypeIs[T]` (PEP 742, *New in 3.13*) |
| --- | --- | --- |
| Narrows when `True` | to `T` | to `T` (intersected with the input type) |
| Narrows when `False` | no | **yes**: to "input type minus `T`" |
| `T` must be a subtype of the input type | no (can narrow `list[object]` to `list[str]`) | yes |
| TypeScript | | `function isProduct(x: unknown): x is Product` |

Prefer `TypeIs`; use `TypeGuard` only for narrowing to a non-subtype, typically a
container whose element type you checked.

In [115]:
from typing import TypeGuard, TypeIs


def is_str_list(values: list[object]) -> TypeGuard[list[str]]:
    """True if every item is a str."""
    return all(isinstance(value, str) for value in values)


def is_decimal(value: object) -> TypeIs[Decimal]:
    """True if value is a Decimal."""
    return isinstance(value, Decimal)


print(f"{is_str_list(['a', 'b'])=}  {is_str_list(['a', 1])=}  {is_decimal(Decimal('1'))=}")
check(
    """
    from decimal import Decimal
    from typing import TypeGuard, TypeIs

    def is_str_list(values: list[object]) -> TypeGuard[list[str]]:
        return all(isinstance(value, str) for value in values)

    def is_decimal(value: object) -> TypeIs[Decimal]:
        return isinstance(value, Decimal)

    def handle(values: list[object], amount: Decimal | float) -> None:
        if is_str_list(values):
            reveal_type(values)
        if is_decimal(amount):
            reveal_type(amount)
        else:
            reveal_type(amount)     # TypeIs also narrows the else branch
    """,
    name="type_is",
    checkers=("mypy",),
)

is_str_list(['a', 'b'])=True  is_str_list(['a', 1])=False  is_decimal(Decimal('1'))=True


mypy:
  type_is.py:12: note: Revealed type is "list[str]"
  type_is.py:14: note: Revealed type is "decimal.Decimal"
  type_is.py:16: note: Revealed type is "float"


**Discriminated (tagged) unions.** Give every member of a union a field with a
different `Literal` value; checking that field narrows to one member. This is
exactly TypeScript's tagged union (`type Event = {kind: "paid", …} | {kind: …}`),
and it combines with `match` and `assert_never` for exhaustiveness. It works with
dataclasses, `TypedDict`s and Pydantic models.

In [116]:
@dataclass(frozen=True)
class Paid:
    order_id: int
    amount: Decimal
    kind: Literal["paid"] = "paid"


@dataclass(frozen=True)
class Refunded:
    order_id: int
    reason: str
    kind: Literal["refunded"] = "refunded"


type PaymentEvent = Paid | Refunded


def describe_event(event: PaymentEvent) -> str:
    """One line per event; exhaustive over the tag."""
    match event.kind:
        case "paid":
            return f"order {event.order_id}: paid INR {event.amount}"
        case "refunded":
            return f"order {event.order_id}: refunded ({event.reason})"


for event in (Paid(7, Decimal("499")), Refunded(8, "damaged")):
    print(describe_event(event))

order 7: paid INR 499
order 8: refunded (damaged)


In [117]:
check(
    """
    from dataclasses import dataclass
    from typing import Literal, TypedDict, assert_never

    class CardPayment(TypedDict):
        method: Literal["card"]
        last4: str

    class UpiPayment(TypedDict):
        method: Literal["upi"]
        vpa: str

    class CashPayment(TypedDict):
        method: Literal["cash"]

    type Payment = CardPayment | UpiPayment | CashPayment

    def receipt_line(payment: Payment) -> str:
        if payment["method"] == "card":
            reveal_type(payment)
            return f"card ending {payment['last4']}"
        if payment["method"] == "upi":
            return f"UPI {payment['vpa']}"
        reveal_type(payment)
        return payment["vpa"]            # a cash payment has no vpa
    """,
    name="tagged_unions",
    checkers=("mypy",),
)

mypy:
  tagged_unions.py:19: note: Revealed type is "TypedDict(tagged_unions.CardPayment, {'method': Literal['card'], 'last4': str})"
  tagged_unions.py:23: note: Revealed type is "TypedDict(tagged_unions.CashPayment, {'method': Literal['cash']})"
  tagged_unions.py:24: error: TypedDict "CashPayment" has no key "vpa"  [typeddict-item]


### 4.2.11 `Annotated` and runtime introspection

`Annotated[T, metadata, ...]` (PEP 593) attaches arbitrary objects to a type. The
**checker sees only `T`**; libraries read the metadata at run time. This is how
Pydantic expresses constraints (`Annotated[int, Field(gt=0)]`) and how FastAPI
declares where a parameter comes from (`Annotated[str, Query(max_length=50)]`,
`Annotated[Session, Depends(get_session)]`, Phase 6). TypeScript has no run-time
types at all, which is why JS libraries such as Zod define schemas as values
instead.

| Function | Returns |
| --- | --- |
| `typing.get_type_hints(obj)` | Evaluated hints, `Annotated` **stripped** |
| `typing.get_type_hints(obj, include_extras=True)` | Evaluated hints with `Annotated` kept |
| `typing.get_origin(tp)` | The unsubscripted form: `list` for `list[int]`, `Annotated` for `Annotated[...]`, `types.UnionType` for `X \| Y` |
| `typing.get_args(tp)` | The arguments: `(int,)`, `(int, Range(0, 150))`, … |

In [118]:
from typing import Annotated, get_args, get_origin, get_type_hints


@dataclass(frozen=True)
class Range:
    minimum: int
    maximum: int


@dataclass
class Signup:
    name: Annotated[str, "display name"]
    age: Annotated[int, Range(18, 120)]
    referral: str | None = None


print(f"{get_type_hints(Signup)=}")
hints = get_type_hints(Signup, include_extras=True)
print(f"{hints['age']=}")
print(f"{get_origin(hints['age']) is Annotated=}  {get_args(hints['age'])=}")
print(f"{hints['age'].__metadata__=}")
print(f"{get_origin(hints['referral'])=}  {get_args(hints['referral'])=}")

get_type_hints(Signup)={'name': <class 'str'>, 'age': <class 'int'>, 'referral': str | None}
hints['age']=typing.Annotated[int, Range(minimum=18, maximum=120)]
get_origin(hints['age']) is Annotated=True  get_args(hints['age'])=(<class 'int'>, Range(minimum=18, maximum=120))
hints['age'].__metadata__=(Range(minimum=18, maximum=120),)
get_origin(hints['referral'])=<class 'typing.Union'>  get_args(hints['referral'])=(<class 'str'>, <class 'NoneType'>)


**Under the hood: a mini validator.** Reading the metadata is all a framework does
before deciding what to check. Lab 4.2 exercise 7 builds a complete one; this cell
only walks a function's parameters:

In [119]:
def constraints_of(function: Callable[..., object]) -> dict[str, list[object]]:
    """Parameter name -> metadata objects found in Annotated hints."""
    found: dict[str, list[object]] = {}
    for name, hint in get_type_hints(function, include_extras=True).items():
        if get_origin(hint) is Annotated:
            found[name] = list(hint.__metadata__)
    return found


def register(name: Annotated[str, "1-40 chars"], age: Annotated[int, Range(18, 120)]) -> None:
    """Register a customer (the metadata is for a framework to enforce)."""


print(constraints_of(register))
print(f"{inspect.signature(register).parameters['age'].annotation=}")

{'name': ['1-40 chars'], 'age': [Range(minimum=18, maximum=120)]}
inspect.signature(register).parameters['age'].annotation=typing.Annotated[int, Range(minimum=18, maximum=120)]


**Pydantic** turns hints plus `Annotated` metadata into a validator and parser
(Unit 4.6 and Phase 6 go deeper). Constraints from `Field(...)` and from the
`annotated-types` library (`Gt`, `Len`, …) both work:

In [120]:
if importlib.util.find_spec("pydantic") is None:
    print("SKIP: pydantic is not installed (uv add pydantic)")
else:
    from annotated_types import Len
    from pydantic import BaseModel, Field, ValidationError

    class SignupModel(BaseModel):
        name: Annotated[str, Len(1, 40)]
        age: Annotated[int, Field(ge=18, le=120)]
        referral: str | None = None

    print(SignupModel(name="Asha", age="31"))  # "31" is parsed to 31
    try:
        SignupModel(name="", age=12)
    except ValidationError as error:
        for problem in error.errors():
            print(f"  {problem['loc'][0]}: {problem['msg']}")

name='Asha' age=31 referral=None
  name: String should have at least 1 character
  age: Input should be greater than or equal to 18


FastAPI (Phase 6) reads the same structure to wire HTTP requests to functions:

```python
from typing import Annotated
from fastapi import Depends, FastAPI, Query

app = FastAPI()

@app.get("/orders")
def list_orders(
    city: Annotated[str, Query(max_length=40)],          # from ?city=...
    session: Annotated[Session, Depends(get_session)],   # injected
) -> list[Order]: ...
```

**Predict the output.**

```python
from typing import Annotated, get_type_hints

def ship(weight: Annotated[float, "kg"]) -> None: ...

print(get_type_hints(ship))
print(ship.__annotations__["weight"] == Annotated[float, "kg"])
```

<details><summary>Answer</summary>

`{'weight': <class 'float'>, 'return': <class 'NoneType'>}` then `True`.
`get_type_hints` strips `Annotated` unless you pass `include_extras=True`, and it
turns the `None` return annotation into `NoneType`. `__annotations__` holds the raw
objects, and `Annotated` aliases compare equal when their type and metadata match.

</details>

Run the next cell to check your prediction.

In [121]:
from typing import Annotated, get_type_hints


def ship(weight: Annotated[float, "kg"]) -> None: ...


print(get_type_hints(ship))
print(ship.__annotations__["weight"] == Annotated[float, "kg"])

{'weight': <class 'float'>, 'return': <class 'NoneType'>}
True


### 4.2.12 Running checkers

| | mypy | pyright |
| --- | --- | --- |
| Written in | Python (compiled with mypyc) | TypeScript (runs on Node.js; powers VS Code's Pylance) |
| Install | `uv add --dev mypy` | `uv add --dev pyright` (downloads Node if needed) |
| Default mode | Skips unannotated functions | Checks everything, infers inside untyped code |
| Strict mode | `--strict` or `strict = true` | `typeCheckingMode = "strict"` or `# pyright: strict` |
| Config | `[tool.mypy]` (+ `[[tool.mypy.overrides]]`) | `[tool.pyright]` or `pyrightconfig.json` |
| Plugins | Yes (Django, Pydantic, SQLAlchemy) | No (relies on standard typing features) |
| TypeScript equivalent | `tsc --noEmit` with `"strict": true` in `tsconfig.json` | the same, and pyright is what VS Code runs as you type (like the TS language server) |

`--strict` is a bundle of flags; mypy lists them in its help:

In [122]:
help_text = subprocess.run(
    [str(VENV_BIN / "mypy"), "--help"], capture_output=True, text=True, env=CHECKER_ENV
).stdout
help_text = ANSI_ESCAPE.sub("", help_text)  # Python 3.14+ argparse can colour its help
strict_help = re.search(r"--strict\s+(.*?)(?=\n  --)", help_text, flags=re.S)
print(textwrap.fill(" ".join(strict_help[1].split()), width=96) if strict_help else "not found")

Strict mode; enables the following flags: --disallow-any-generics, --disallow-subclassing- any,
--disallow-untyped-calls, --disallow-untyped- defs, --disallow-incomplete-defs, --check-untyped-
defs, --disallow-untyped-decorators, --warn- redundant-casts, --warn-unused-ignores, --warn-
return-any, --no-implicit-reexport, --strict- equality, --extra-checks


**Configuration in `pyproject.toml`.** A realistic setup for a project that is
partly typed: strict everywhere, except a legacy package that is excluded for now.
Both checkers run from the project folder and read the same file.

In [123]:
typed_shop = UNIT42_DIR / "typed-shop"
files = {
    "pyproject.toml": """
        [tool.mypy]
        strict = true
        files = ["src"]
        mypy_path = "src"
        warn_unreachable = true

        [[tool.mypy.overrides]]
        module = ["legacy", "legacy.*"]
        ignore_errors = true                  # gradual adoption: not checked yet

        [tool.pyright]
        include = ["src"]
        exclude = ["src/legacy"]
        typeCheckingMode = "strict"
        pythonVersion = "3.15"
    """,
    "src/shop/__init__.py": "",
    "src/shop/pricing.py": """
        from decimal import Decimal

        def with_gst(amount: Decimal, rate: Decimal = Decimal("0.18")) -> Decimal:
            return amount * (1 + rate)

        def total(amounts: list[Decimal]) -> Decimal:
            return sum(amounts, Decimal(0))

        def bad_total(amounts):                  # untyped: strict mode rejects it
            return sum(amounts)
    """,
    "src/legacy/__init__.py": "",
    "src/legacy/report.py": "def make_report(rows):\n    return '\\n'.join(rows) + 1\n",
}
for relative_path, text in files.items():
    (typed_shop / relative_path).parent.mkdir(parents=True, exist_ok=True)
    (typed_shop / relative_path).write_text(textwrap.dedent(text).lstrip("\n"))
for command in (["mypy", "--cache-dir", "../.mypy_cache"], ["pyright"]):
    result = subprocess.run(
        [str(VENV_BIN / command[0]), *command[1:]],
        cwd=typed_shop,
        capture_output=True,
        text=True,
        env=CHECKER_ENV,
    )
    print(
        f"$ {command[0]}\n"
        + textwrap.indent(
            ANSI_ESCAPE.sub("", result.stdout).replace(str(typed_shop), "<project>").strip(), "  "
        )
    )

$ mypy
  src/shop/pricing.py:9: error: Function is missing a type annotation  [no-untyped-def]
  Found 1 error in 1 file (checked 4 source files)


$ pyright
  <project>/src/shop/pricing.py
    <project>/src/shop/pricing.py:9:15 - error: Type of parameter "amounts" is unknown (reportUnknownParameterType)
    <project>/src/shop/pricing.py:9:15 - error: Type annotation is missing for parameter "amounts" (reportMissingParameterType)
    <project>/src/shop/pricing.py:10:16 - error: Argument type is unknown
      Argument corresponds to parameter "iterable" in function "sum" (reportUnknownArgumentType)
  3 errors, 0 warnings, 0 informations


**`reveal_type` at run time.** Besides being a checker instruction,
`typing.reveal_type` (3.11) is a real function: it prints the *run-time* type to
stderr and returns its argument unchanged, so you can leave it in while debugging
and run the code. (Remove it afterwards; ruff and the checkers flag it.)

In [124]:
import contextlib
import io

captured = io.StringIO()
with contextlib.redirect_stderr(captured):
    returned = typing.reveal_type([1, 2, 3])
print(f"{returned=}  stderr: {captured.getvalue().strip()!r}")

returned=[1, 2, 3]  stderr: "Runtime type is 'list'"


**Stubs and `py.typed`.** A checker needs types for every library you import:

| Source of types | Example |
| --- | --- |
| Inline hints in a package that ships a **`py.typed`** marker file (PEP 561) | pydantic, httpx, rich |
| Stub files (`.pyi`: signatures only, bodies are `...`) bundled in **typeshed** | the standard library; mypy and pyright ship it |
| A separate stub package `types-<name>` or `<name>-stubs` | `types-requests`, `pandas-stubs` |
| Your own stubs in a folder on `mypy_path` / pyright's `stubPath` | private untyped code |

Without `py.typed`, an installed package's inline hints are **ignored**, as the next
demo shows. It installs a tiny untyped package into a scratch virtual environment
and points mypy at that environment's interpreter.

In [125]:
stub_demo = UNIT42_DIR / "stub-demo"
env_dir = stub_demo / "venv"
if not env_dir.exists():
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(env_dir)], check=True)
site_packages = next(env_dir.glob("lib/python3*/site-packages"))
package_dir = site_packages / "rupee_tools"
package_dir.mkdir(exist_ok=True)
(package_dir / "__init__.py").write_text(
    "def to_paise(amount: float) -> int:\n    return round(amount * 100)\n"
)
(package_dir / "py.typed").unlink(missing_ok=True)
APP = """
from rupee_tools import to_paise

reveal_type(to_paise(12.5))
to_paise("12.50")
"""
ENV_OPTION = ("--python-executable", str(env_dir / "bin" / "python"))
print("without py.typed:", *run_mypy(APP, name="stub_app", options=ENV_OPTION), sep="\n  ")
(package_dir / "py.typed").write_text("")
print("with py.typed:", *run_mypy(APP, name="stub_app", options=ENV_OPTION), sep="\n  ")

without py.typed:
  stub_app.py:1: error: Skipping analyzing "rupee_tools": module is installed, but missing library stubs or py.typed marker  [import-untyped]
  stub_app.py:1: note: See https://mypy.readthedocs.io/en/stable/running_mypy.html#missing-imports
  stub_app.py:3: note: Revealed type is "Any"
with py.typed:
  stub_app.py:3: note: Revealed type is "int"
  stub_app.py:4: error: Argument 1 to "to_paise" has incompatible type "str"; expected "float"  [arg-type]


When you cannot change the package, write a **stub** instead: a `.pyi` file with the
same module path, found through `mypy_path`. mypy's `stubgen` drafts one from the
source (here with `py.typed` removed again, so only the stub provides types):

In [126]:
(package_dir / "py.typed").unlink()
stubs_dir = stub_demo / "stubs"
subprocess.run(
    [str(VENV_BIN / "stubgen"), str(package_dir / "__init__.py"), "-o", str(stubs_dir)],
    capture_output=True,
    text=True,
    check=True,
)
stub_file = next(stubs_dir.rglob("*.pyi"))
print(f"stub written to: stubs/{stub_file.relative_to(stubs_dir).as_posix()}")
print(f"--- {stub_file.name} (generated by stubgen) ---\n{stub_file.read_text().strip()}")
stub_env = {**CHECKER_ENV, "MYPYPATH": str(stubs_dir)}
completed = subprocess.run(
    [
        str(VENV_BIN / "mypy"),
        "--no-error-summary",
        "--cache-dir",
        ".mypy_cache",
        *ENV_OPTION,
        "stub_app.py",
    ],
    cwd=UNIT42_DIR,
    capture_output=True,
    text=True,
    env=stub_env,
)
print("with the stub on MYPYPATH:", *completed.stdout.strip().splitlines(), sep="\n  ")

stub written to: stubs/rupee_tools/__init__.pyi
--- __init__.pyi (generated by stubgen) ---
def to_paise(amount: float) -> int: ...
with the stub on MYPYPATH:
  stub_app.py:3: note: Revealed type is "int"
  stub_app.py:4: error: Argument 1 to "to_paise" has incompatible type "str"; expected "float"  [arg-type]


**Gradual adoption.** Nobody types a large codebase in one go. A path that works:

1. Add mypy (or pyright) to CI in its **default** mode, fix what it finds, keep it
   green.
2. Turn on `strict = true` globally and list every not-yet-clean module in a
   `[[tool.mypy.overrides]]` block with `ignore_errors = true` (as in the config
   demo above). New modules are strict from day one.
3. Remove modules from the override list one at a time (a *ratchet*: the list may
   only shrink). Count errors per module to pick the cheapest next target.
4. Type the boundaries first: public functions, data models, I/O. Inference does the
   rest inside function bodies.
5. Keep `warn_unused_ignores` on so stale `# type: ignore` comments disappear.

A ratchet in a few lines: count mypy errors per file of the `typed-shop` project
when everything (legacy included) is checked strictly.

In [127]:
from collections import Counter

all_strict = subprocess.run(
    [
        str(VENV_BIN / "mypy"),
        "--strict",
        "--no-error-summary",
        "--cache-dir",
        "../.mypy_cache",
        "src",
    ],
    cwd=typed_shop,
    capture_output=True,
    text=True,
    env=CHECKER_ENV,
)
per_file = Counter(
    line.split(":")[0] for line in all_strict.stdout.splitlines() if ": error:" in line
)
for path, error_count in per_file.most_common():
    print(f"{error_count:>3} errors  {path}")

  1 errors  src/shop/pricing.py


### Lab 4.2

Seven exercises. For the stubs, replace `raise NotImplementedError`. Exercises 1, 5
and 6 are different: the code already works but is untyped or badly typed, and you
**fix its types in place**. Every check cell runs mypy (`--strict` unless stated)
on the source of your code, which it gets with `source_of(...)`, and also tests the
behaviour. The checker prelude imports `Callable`, `Iterable`, `Iterator`,
`Mapping`, `Sequence` (from `collections.abc`), `dataclass`, `field`, `Decimal`,
`ROUND_HALF_UP`, `Annotated`, `Any`, `Literal`, `Self`, `cast`, `overload` and
`get_type_hints`, so use those names. Until you finish, the checks print FAIL; that
is expected.

#### Exercise 1 — add precise hints to 10 functions

Annotate the functions in the next cell (change nothing else) so that
`mypy --strict` reports no errors and the hints are exactly these (spaces do not
matter; `T` is a PEP 695 type parameter):

| Function | Hints |
| --- | --- |
| `count_words` | `(text: str) -> dict[str, int]` |
| `average` | `(values: Iterable[float]) -> float` |
| `find_user` | `(users: Mapping[int, str], user_id: int) -> str \| None` |
| `pair_up` | `(names: Sequence[str], scores: Sequence[int]) -> list[tuple[str, int]]` |
| `parse_point` | `(text: str) -> tuple[float, float]` |
| `chunk[T]` | `(items: Sequence[T], size: int) -> list[list[T]]` |
| `merge_settings` | `(defaults: Mapping[str, object], overrides: Mapping[str, object]) -> dict[str, object]` |
| `first_or_default[T]` | `(items: Iterable[T], default: T) -> T` |
| `apply_twice[T]` | `(func: Callable[[T], T], value: T) -> T` |
| `status_text` | `(code: int) -> Literal["ok", "client error", "server error", "unknown"]` |

In [128]:
def count_words(text):
    words = text.lower().split()
    counts = dict.fromkeys(words, 0)
    for word in words:
        counts[word] += 1
    return counts


def average(values):
    total = 0.0
    count = 0
    for value in values:
        total += value
        count += 1
    return total / count if count else 0.0


def find_user(users, user_id):
    return users.get(user_id)


def pair_up(names, scores):
    return list(zip(names, scores, strict=True))


def parse_point(text):
    x_text, y_text = text.split(",")
    return float(x_text), float(y_text)


def chunk(items, size):
    return [list(items[start : start + size]) for start in range(0, len(items), size)]


def merge_settings(defaults, overrides):
    return {**defaults, **overrides}


def first_or_default(items, default):
    for item in items:
        return item
    return default


def apply_twice(func, value):
    return func(func(value))


def status_text(code):
    if 200 <= code < 300:
        return "ok"
    if 400 <= code < 500:
        return "client error"
    if 500 <= code < 600:
        return "server error"
    return "unknown"

Run the checks.

In [129]:
HINT_FUNCTIONS = [
    count_words,
    average,
    find_user,
    pair_up,
    parse_point,
    chunk,
    merge_settings,
    first_or_default,
    apply_twice,
    status_text,
]
EXPECTED_HINTS = {
    "count_words": {"text": "str", "return": "dict[str, int]"},
    "average": {"values": "Iterable[float]", "return": "float"},
    "find_user": {"users": "Mapping[int, str]", "user_id": "int", "return": "str | None"},
    "pair_up": {
        "names": "Sequence[str]",
        "scores": "Sequence[int]",
        "return": "list[tuple[str, int]]",
    },
    "parse_point": {"text": "str", "return": "tuple[float, float]"},
    "chunk": {"items": "Sequence[T]", "size": "int", "return": "list[list[T]]"},
    "merge_settings": {
        "defaults": "Mapping[str, object]",
        "overrides": "Mapping[str, object]",
        "return": "dict[str, object]",
    },
    "first_or_default": {"items": "Iterable[T]", "default": "T", "return": "T"},
    "apply_twice": {"func": "Callable[[T], T]", "value": "T", "return": "T"},
    "status_text": {
        "code": "int",
        "return": 'Literal["ok", "client error", "server error", "unknown"]',
    },
}


def hint_strings(function: Callable[..., object]) -> dict[str, str]:
    """The annotations as written in the source, without spaces."""
    written = annotationlib.get_annotations(function, format=annotationlib.Format.STRING)
    return {name: text.replace(" ", "").replace("'", '"') for name, text in written.items()}


run_checks(
    [
        *(
            (
                f"hints of {function.__name__}",
                lambda function=function: hint_strings(function),
                {
                    name: text.replace(" ", "")
                    for name, text in EXPECTED_HINTS[function.__name__].items()
                },
            )
            for function in HINT_FUNCTIONS
        ),
        ("mypy --strict is clean", lambda: mypy_errors(*HINT_FUNCTIONS), []),
        (
            "behaviour unchanged",
            lambda: (
                count_words("a b a"),
                chunk([1, 2, 3], 2),
                status_text(404),
                first_or_default([], 7),
                apply_twice(lambda n: n * 3, 2),
            ),
            ({"a": 2, "b": 1}, [[1, 2], [3]], "client error", 7, 18),
        ),
    ]
)

FAIL  hints of count_words: expected {'text': 'str', 'return': 'dict[str,int]'}, got {}
FAIL  hints of average: expected {'values': 'Iterable[float]', 'return': 'float'}, got {}
FAIL  hints of find_user: expected {'users': 'Mapping[int,str]', 'user_id': 'int', 'return': 'str|None'}, got {}
FAIL  hints of pair_up: expected {'names': 'Sequence[str]', 'scores': 'Sequence[int]', 'return': 'list[tuple[str,int]]'}, got {}
FAIL  hints of parse_point: expected {'text': 'str', 'return': 'tuple[float,float]'}, got {}
FAIL  hints of chunk: expected {'items': 'Sequence[T]', 'size': 'int', 'return': 'list[list[T]]'}, got {}
FAIL  hints of merge_settings: expected {'defaults': 'Mapping[str,object]', 'overrides': 'Mapping[str,object]', 'return': 'dict[str,object]'}, got {}
FAIL  hints of first_or_default: expected {'items': 'Iterable[T]', 'default': 'T', 'return': 'T'}, got {}
FAIL  hints of apply_twice: expected {'func': 'Callable[[T],T]', 'value': 'T', 'return': 'T'}, got {}
FAIL  hints of status_t

FAIL  mypy --strict is clean: expected [], got ['Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]']
PASS  behaviour unchanged
1/12 checks passed


#### Exercise 2 — generic `Stack[T]` and `Result[T, E]`

Implement the classes below using PEP 695 syntax (keep the signatures):

- `Stack[T]`: `push`, `pop` (raises `IndexError("pop from empty stack")` when
  empty), `peek` (`None` when empty), `__len__`, `is_empty`.
- `Ok[T]` and `Err[E]`: frozen dataclasses holding `value` / `error`, with
  `is_ok()` and `unwrap()`. `Err.unwrap()` raises
  `ValueError(f"unwrap on Err: {self.error!r}")`. `Result[T, E]` is their union
  (given).
- `safe_divide(a, b)` returns `Ok(a / b)` or `Err("division by zero")`.

The checks also ask mypy what `Stack[int]().pop()` and `safe_divide(1, 2)` are, and
expect an error for `Stack[int]().push("tea")`.

In [130]:
class Stack[T]:
    """A last-in, first-out stack."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        raise NotImplementedError("Lab 4.2 exercise 2: Stack.push")

    def pop(self) -> T:
        raise NotImplementedError("Lab 4.2 exercise 2: Stack.pop")

    def peek(self) -> T | None:
        raise NotImplementedError("Lab 4.2 exercise 2: Stack.peek")

    def __len__(self) -> int:
        raise NotImplementedError("Lab 4.2 exercise 2: Stack.__len__")

    def is_empty(self) -> bool:
        raise NotImplementedError("Lab 4.2 exercise 2: Stack.is_empty")


@dataclass(frozen=True)
class Ok[T]:
    value: T

    def is_ok(self) -> bool:
        raise NotImplementedError("Lab 4.2 exercise 2: Ok.is_ok")

    def unwrap(self) -> T:
        raise NotImplementedError("Lab 4.2 exercise 2: Ok.unwrap")


@dataclass(frozen=True)
class Err[E]:
    error: E

    def is_ok(self) -> bool:
        raise NotImplementedError("Lab 4.2 exercise 2: Err.is_ok")

    def unwrap(self) -> typing.Never:
        raise NotImplementedError("Lab 4.2 exercise 2: Err.unwrap")


type Result[T, E] = Ok[T] | Err[E]


def safe_divide(a: float, b: float) -> Result[float, str]:
    """Ok(a / b), or Err("division by zero")."""
    raise NotImplementedError("Lab 4.2 exercise 2: safe_divide")

Run the checks.

In [131]:
GENERIC_USAGE = """
type Result[T, E] = Ok[T] | Err[E]

reveal_type(Stack[int]().pop())
reveal_type(safe_divide(1, 2))
Stack[int]().push("tea")
"""


def stack_trace() -> list[object]:
    stack = Stack[str]()
    stack.push("tea")
    stack.push("rice")
    return [len(stack), stack.peek(), stack.pop(), stack.pop(), stack.is_empty(), stack.peek()]


def generic_errors() -> list[str]:
    errors = mypy_errors(Stack, Ok, Err, safe_divide, extra=GENERIC_USAGE)
    return [error.split("  [")[-1] for error in errors]


run_checks(
    [
        ("Stack behaviour", stack_trace, [2, "rice", "rice", "tea", True, None]),
        (
            "pop on empty",
            lambda: raised_message(lambda: Stack[int]().pop()),
            "IndexError: pop from empty stack",
        ),
        ("Ok", lambda: (Ok(5).is_ok(), Ok(5).unwrap()), (True, 5)),
        (
            "Err",
            lambda: (Err("boom").is_ok(), raised_message(lambda: Err("boom").unwrap())),
            (False, "ValueError: unwrap on Err: 'boom'"),
        ),
        (
            "safe_divide",
            lambda: (safe_divide(1, 4), safe_divide(1, 0)),
            (Ok(0.25), Err("division by zero")),
        ),
        (
            "PEP 695 parameters",
            lambda: (
                [p.__name__ for p in Stack.__type_params__],
                [p.__name__ for p in Result.__type_params__],
            ),
            (["T"], ["T", "E"]),
        ),
        (
            "mypy: types revealed",
            lambda: revealed_types(Stack, Ok, Err, safe_divide, extra=GENERIC_USAGE)[:2],
            ["int", "lab_check.Ok[float] | lab_check.Err[str]"],
        ),
        ("mypy: only the push('tea') error", generic_errors, ["arg-type]"]),
    ]
)

FAIL  Stack behaviour: raised NotImplementedError: Lab 4.2 exercise 2: Stack.push
FAIL  pop on empty: expected 'IndexError: pop from empty stack', got 'NotImplementedError: Lab 4.2 exercise 2: Stack.pop'
FAIL  Ok: raised NotImplementedError: Lab 4.2 exercise 2: Ok.is_ok
FAIL  Err: raised NotImplementedError: Lab 4.2 exercise 2: Err.is_ok
FAIL  safe_divide: raised NotImplementedError: Lab 4.2 exercise 2: safe_divide
PASS  PEP 695 parameters
PASS  mypy: types revealed
PASS  mypy: only the push('tea') error
3/8 checks passed


#### Exercise 3 — a typed `@retry` with `ParamSpec`

Write the decorator factory `retry(times=3, *, exceptions=(Exception,), delay=0.0,
sleep=time.sleep)`, a simpler cousin of Unit 1.12's `retry`:

- Call the function up to `times` times; return the first successful result.
- After a failure with one of `exceptions`, call `sleep(delay)` before the next
  attempt (not after the last); re-raise the last error when attempts run out.
  Other exceptions propagate at once.
- Keep the wrapped function's metadata (`functools.wraps`) **and its signature for
  the checker**: use `[**P, R]` so `mypy --strict` accepts your decorator, keeps
  the parameters of the decorated function and rejects a badly typed call.

In [132]:
def retry[**P, R](
    times: int = 3,
    *,
    exceptions: tuple[type[Exception], ...] = (Exception,),
    delay: float = 0.0,
    sleep: Callable[[float], None] = time.sleep,
) -> Callable[[Callable[P, R]], Callable[P, R]]:
    """Retry the decorated function up to `times` times on the given exceptions."""
    raise NotImplementedError("Lab 4.2 exercise 3: write retry")

Run the checks. `flaky(n)` returns a function that fails `n` times, then succeeds.

In [133]:
SERVICE_DOWN = ConnectionError("down")


def flaky(failures: int, error: Exception = SERVICE_DOWN) -> Callable[[str], str]:
    """A function that raises `error` for its first `failures` calls."""
    calls = {"count": 0}

    def fetch(sku: str) -> str:
        calls["count"] += 1
        if calls["count"] <= failures:
            raise error
        return f"{sku} after {calls['count']} calls"

    return fetch


def retry_run(
    failures: int, times: int, error: Exception = SERVICE_DOWN
) -> tuple[str, list[float]]:
    """Decorate flaky(failures, error); return (result or "Error: message", recorded sleeps)."""
    sleeps: list[float] = []
    decorate = retry(times, exceptions=(ConnectionError,), delay=0.5, sleep=sleeps.append)
    decorated = decorate(flaky(failures, error))
    try:
        outcome = decorated("TEA")
    except Exception as caught:
        outcome = f"{type(caught).__name__}: {caught}"
    return outcome, sleeps


RETRY_USAGE = """
@retry(3, exceptions=(ConnectionError,))
def fetch_price(sku: str, quantity: int = 1) -> Decimal:
    return Decimal("10") * quantity

reveal_type(fetch_price)
fetch_price("TEA", 2)
fetch_price(42)
"""

run_checks(
    [
        ("succeeds on the 3rd call", lambda: retry_run(2, 3), ("TEA after 3 calls", [0.5, 0.5])),
        ("gives up after times", lambda: retry_run(5, 3), ("ConnectionError: down", [0.5, 0.5])),
        ("other errors propagate", lambda: retry_run(5, 3, KeyError("x")), ("KeyError: 'x'", [])),
        ("keeps metadata", lambda: retry()(flaky(0)).__name__, "fetch"),
        (
            "mypy: signature kept",
            lambda: revealed_types(retry, extra=RETRY_USAGE),
            ["def (sku: str, quantity: int =) -> decimal.Decimal"],
        ),
        (
            "mypy: bad call rejected, nothing else",
            lambda: [e.split("  [")[-1] for e in mypy_errors(retry, extra=RETRY_USAGE)],
            ["arg-type]"],
        ),
    ]
)

FAIL  succeeds on the 3rd call: raised NotImplementedError: Lab 4.2 exercise 3: write retry
FAIL  gives up after times: raised NotImplementedError: Lab 4.2 exercise 3: write retry
FAIL  other errors propagate: raised NotImplementedError: Lab 4.2 exercise 3: write retry
FAIL  keeps metadata: raised NotImplementedError: Lab 4.2 exercise 3: write retry


PASS  mypy: signature kept
PASS  mypy: bad call rejected, nothing else
2/6 checks passed


#### Exercise 4 — an `@overload`ed `parse_value`

`parse_value(text, kind="str")` converts text according to `kind`:

| Call | Returns | Checker type |
| --- | --- | --- |
| `parse_value("42", "int")` | `42` | `int` |
| `parse_value("2.5", "float")` | `2.5` | `float` |
| `parse_value("yes", "bool")` | `True` (`"true"`, `"yes"`, `"1"` → `True`; `"false"`, `"no"`, `"0"` → `False`, case-insensitive; anything else `ValueError`) | `bool` |
| `parse_value(" tea ")` | `"tea"` (stripped) | `str` |

Write four `@overload` signatures using `Literal` kinds (`"str"` is the default),
then the implementation. Any other `kind` must be a checker error. Keep everything
inside the function: the check sends only the overloads and the implementation to
mypy, not module-level constants.

In [134]:
def parse_value(text: str, kind: str = "str") -> object:
    """Convert text to str, int, float or bool according to kind."""
    raise NotImplementedError("Lab 4.2 exercise 4: overloads + implementation")

Run the checks.

In [135]:
OVERLOAD_USAGE = """
reveal_type(parse_value("42", "int"))
reveal_type(parse_value("2.5", "float"))
reveal_type(parse_value("yes", "bool"))
reveal_type(parse_value(" tea "))
parse_value("2026-11-07", "date")
"""


def overload_source() -> str:
    """The overloads and the implementation: source of the whole cell block."""
    return (
        "\n".join(source_of(f) for f in typing.get_overloads(parse_value))
        + "\n"
        + source_of(parse_value)
    )


run_checks(
    [
        ("int", lambda: parse_value("42", "int"), 42),
        ("float", lambda: parse_value("2.5", "float"), 2.5),
        (
            "bool",
            lambda: [parse_value(t, "bool") for t in ("Yes", "TRUE", "1", "no", "False", "0")],
            [True, True, True, False, False, False],
        ),
        ("bad bool", lambda: raised_error_name(lambda: parse_value("maybe", "bool")), "ValueError"),
        ("str (default)", lambda: parse_value(" tea "), "tea"),
        ("four overloads", lambda: len(typing.get_overloads(parse_value)), 4),
        (
            "mypy: types",
            lambda: re.findall(
                r'Revealed type is "([^"]+)"',
                "\n".join(mypy_report(overload_source() + textwrap.dedent(OVERLOAD_USAGE))),
            ),
            ["int", "float", "bool", "str"],
        ),
        (
            "mypy: unknown kind rejected",
            lambda: [
                line.split("  [")[-1]
                for line in mypy_report(overload_source() + textwrap.dedent(OVERLOAD_USAGE))
                if ": error:" in line
            ],
            ["call-overload]"],
        ),
    ]
)

FAIL  int: raised NotImplementedError: Lab 4.2 exercise 4: overloads + implementation
FAIL  float: raised NotImplementedError: Lab 4.2 exercise 4: overloads + implementation
FAIL  bool: raised NotImplementedError: Lab 4.2 exercise 4: overloads + implementation
FAIL  bad bool: expected 'ValueError', got 'NotImplementedError'
FAIL  str (default): raised NotImplementedError: Lab 4.2 exercise 4: overloads + implementation
FAIL  four overloads: expected 4, got 0


FAIL  mypy: types: expected ['int', 'float', 'bool', 'str'], got ['object', 'object', 'object', 'object']
FAIL  mypy: unknown kind rejected: expected ['call-overload]'], got []
0/8 checks passed


#### Exercise 5 — fix 8 mypy errors in a provided module

The next cell is a small order-report module. It is annotated, but mypy (default
mode) reports **8 errors**, and some of them are real bugs. Fix the code in place
until mypy reports nothing **and** the behaviour checks pass:

- `parse_line("TEA,2,120.50")` → `LineItem("TEA", 2, Decimal("120.50"))`
- `find_item` returns `None` for an unknown SKU (fix the annotation, not the
  behaviour); `quantity_of` returns `0` for an unknown SKU
- `order_total` returns a `Decimal`; `describe` returns `"TEA x 2"`
- `totals_by_sku` maps each SKU to its line total (a `Decimal`)
- `sample_item()` returns a `LineItem` for `"RICE"`, quantity 1, price `80`
- `cheapest` returns the **SKU** of the item with the lowest unit price

Fixing one error can reveal another (a function that now returns `X | None`
makes its callers narrow); keep going until mypy is silent.

The cell after it shows the 8 errors for the code as given.

In [136]:
@dataclass
class LineItem:
    sku: str
    quantity: int
    unit_price: Decimal

    def total(self) -> Decimal:
        return self.quantity * self.unit_price


def parse_line(raw: str) -> LineItem:
    sku, quantity, price = raw.split(",")
    return LineItem(sku, quantity, Decimal(price))


def find_item(items: list[LineItem], sku: str) -> LineItem:
    for item in items:
        if item.sku == sku:
            return item
    return None


def quantity_of(items: list[LineItem], sku: str) -> int:
    return find_item(items, sku).quantity


def order_total(items: list[LineItem]) -> Decimal:
    total = 0
    for item in items:
        total += item.total()
    return total


def describe(item: LineItem) -> str:
    return item.sku + " x " + item.quantity


def totals_by_sku(items: list[LineItem]) -> dict[str, int]:
    totals: dict[str, int] = {}
    for item in items:
        totals[item.sku] = item.total()
    return totals


def sample_item() -> LineItem:
    return LineItem(sku="RICE", quantity=1, price=Decimal("80"))


def cheapest(items: list[LineItem]) -> str:
    return min(items, key=lambda item: item.unit_price)

The errors as they stand (this cell is information, not a check):

In [137]:
ORDER_REPORT = [
    LineItem,
    parse_line,
    find_item,
    quantity_of,
    order_total,
    describe,
    totals_by_sku,
    sample_item,
    cheapest,
]
for error in mypy_errors(*ORDER_REPORT, strict=False):
    print(error)

Argument 2 to "LineItem" has incompatible type "str"; expected "int"  [arg-type]
Incompatible return value type (got "None", expected "LineItem")  [return-value]
Incompatible types in assignment (expression has type "Decimal", variable has type "int")  [assignment]
Incompatible return value type (got "int", expected "Decimal")  [return-value]
Unsupported operand types for + ("str" and "int")  [operator]
Incompatible types in assignment (expression has type "Decimal", target has type "int")  [assignment]
Unexpected keyword argument "price" for "LineItem"  [call-arg]
Incompatible return value type (got "LineItem", expected "str")  [return-value]


Run the checks.

In [138]:
SAMPLE_LINES = [parse_line("TEA,2,120.50"), parse_line("RICE,1,80")]

run_checks(
    [
        ("mypy is clean", lambda: mypy_errors(*ORDER_REPORT, strict=False), []),
        ("parse_line", lambda: parse_line("TEA,2,120.50"), LineItem("TEA", 2, Decimal("120.50"))),
        ("find_item missing", lambda: find_item(SAMPLE_LINES, "SALT"), None),
        (
            "quantity_of",
            lambda: (quantity_of(SAMPLE_LINES, "TEA"), quantity_of(SAMPLE_LINES, "SALT")),
            (2, 0),
        ),
        ("order_total", lambda: order_total(SAMPLE_LINES), Decimal("321.00")),
        ("order_total type", lambda: type(order_total([])).__name__, "Decimal"),
        ("describe", lambda: describe(SAMPLE_LINES[0]), "TEA x 2"),
        (
            "totals_by_sku",
            lambda: totals_by_sku(SAMPLE_LINES),
            {"TEA": Decimal("241.00"), "RICE": Decimal("80")},
        ),
        ("sample_item", sample_item, LineItem("RICE", 1, Decimal("80"))),
        ("cheapest", lambda: cheapest(SAMPLE_LINES), "RICE"),
    ]
)

FAIL  mypy is clean: expected [], got ['Argument 2 to "LineItem" has incompatible type "str"; expected "int"  [arg-type]', 'Incompatible return value type (got "None", expected "LineItem")  [return-value]', 'Incompatible types in assignment (expression has type "Decimal", variable has type "int")  [assignment]', 'Incompatible return value type (got "int", expected "Decimal")  [return-value]', 'Unsupported operand types for + ("str" and "int")  [operator]', 'Incompatible types in assignment (expression has type "Decimal", target has type "int")  [assignment]', 'Unexpected keyword argument "price" for "LineItem"  [call-arg]', 'Incompatible return value type (got "LineItem", expected "str")  [return-value]']
FAIL  parse_line: expected LineItem(sku='TEA', quantity=2, unit_price=Decimal('120.50')), got LineItem(sku='TEA', quantity='2', unit_price=Decimal('120.50'))
PASS  find_item missing
FAIL  quantity_of: raised AttributeError: 'NoneType' object has no attribute 'quantity'
FAIL  order_tot

#### Exercise 6 — type the Phase 2 model to pass `--strict`

Below is a condensed, **untyped** copy of the Phase 2 inventory model: `Money`
(Lab 2.4), `Product` and `Warehouse` (Phase 2 Project). Add annotations (you may
also add attribute annotations or turn `Product` into a dataclass) until
`mypy --strict` reports nothing. The behaviour must not change.

Hints: `__eq__` takes `other: object`; binary dunders may return `NotImplemented`
while being annotated `-> Money`; empty dict attributes need their type.

In [139]:
class CurrencyMismatchError(ValueError):
    """Two Money values in different currencies were combined."""


class Money:
    __slots__ = ("_amount", "_currency")

    def __init__(self, amount, currency="INR"):
        if isinstance(amount, float):
            raise TypeError("amount must be Decimal, int or str, not float")
        self._amount = Decimal(amount).quantize(PAISE, rounding=ROUND_HALF_UP)
        self._currency = currency

    @property
    def amount(self):
        return self._amount

    @property
    def currency(self):
        return self._currency

    def __repr__(self):
        return f"Money({str(self._amount)!r}, {self._currency!r})"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self._amount, self._currency) == (other._amount, other._currency)

    def __hash__(self):
        return hash((self._amount, self._currency))

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if other._currency != self._currency:
            raise CurrencyMismatchError(f"{self._currency} vs {other._currency}")
        return Money(self._amount + other._amount, self._currency)

    def __mul__(self, factor):
        if isinstance(factor, bool) or not isinstance(factor, (int, Decimal)):
            return NotImplemented
        return Money(self._amount * factor, self._currency)


class InsufficientStockError(Exception):
    def __init__(self, sku, available, requested):
        super().__init__(f"not enough {sku}: {available} available, {requested} requested")
        self.sku, self.available, self.requested = sku, available, requested


class Product:
    def __init__(self, sku, name, unit_price):
        self.sku, self.name, self.unit_price = sku, name, unit_price

    def __repr__(self):
        return f"Product({self.sku!r}, {self.name!r}, {self.unit_price!r})"


class Warehouse:
    def __init__(self, code):
        self.code = code
        self._products = {}
        self._quantities = {}

    def add_product(self, product):
        self._products[product.sku] = product
        self._quantities[product.sku] = 0

    def receive(self, sku, quantity):
        self._quantities[sku] += quantity
        return self._quantities[sku]

    def ship(self, sku, quantity):
        available = self._quantities[sku]
        if quantity > available:
            raise InsufficientStockError(sku, available, quantity)
        self._quantities[sku] = available - quantity
        return self._quantities[sku]

    def stock_value(self):
        return sum(
            (product.unit_price * self._quantities[sku] for sku, product in self._products.items()),
            Money(0),
        )

Run the checks.

In [140]:
INVENTORY_MODEL = [CurrencyMismatchError, Money, InsufficientStockError, Product, Warehouse]


def warehouse_story() -> tuple[object, ...]:
    pune = Warehouse("PNQ")
    pune.add_product(Product("TEA-500", "Assam tea", Money("120.50")))
    pune.add_product(Product("RICE-1K", "Basmati", Money(80)))
    received = pune.receive("TEA-500", 10)
    pune.receive("RICE-1K", 5)
    left = pune.ship("TEA-500", 4)
    shortage = raised_message(lambda: pune.ship("RICE-1K", 9))
    return received, left, shortage, pune.stock_value()


run_checks(
    [
        ("mypy --strict is clean", lambda: mypy_errors(*INVENTORY_MODEL), []),
        (
            "Money arithmetic",
            lambda: (Money("10.005") + Money(5), Money(3) * 2),
            (Money("15.01"), Money("6.00")),
        ),
        (
            "currency mismatch",
            lambda: raised_error_name(lambda: Money(1) + Money(1, "USD")),
            "CurrencyMismatchError",
        ),
        ("float rejected", lambda: raised_error_name(lambda: Money(1.5)), "TypeError"),
        ("NotImplemented path", lambda: raised_error_name(lambda: Money(1) + 1), "TypeError"),
        (
            "warehouse story",
            warehouse_story,
            (
                10,
                6,
                "InsufficientStockError: not enough RICE-1K: 5 available, 9 requested",
                Money("1123.00"),
            ),
        ),
    ]
)

FAIL  mypy --strict is clean: expected [], got ['Function is missing a type annotation  [no-untyped-def]', 'Function is missing a return type annotation  [no-untyped-def]', 'Function is missing a return type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a return type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Call to untyped function "Money" in typed context  [no-untyped-call]', 'Function is missing a type annotation  [no-untyped-def]', 'Call to untyped function "Money" in typed context  [no-untyped-call]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing

#### Exercise 7 — an `Annotated`-driven runtime validator

Write `validate(instance)` for dataclass instances. For every field, read the hint
with `get_type_hints(type(instance), include_extras=True)` and return a list of
error messages (empty when valid), in field order:

| Situation | Message |
| --- | --- |
| The value is not an instance of the field's base type (for `Annotated[int, …]` the base is `int`; `bool` is **not** accepted as `int`) | `"age: expected int, got str"` (then skip that field's constraints) |
| `Range(minimum, maximum)` violated | `"age: 200 is not between 0 and 150"` |
| `MaxLength(limit)` violated | `"name: longer than 20 characters"` |
| `Pattern(regex)` does not `fullmatch` | `"email: does not match [^@]+@[^@]+"` |

Fields whose hint is not `Annotated` only get the type check (and only for plain
classes such as `str`; skip hints like `list[str]` or `str | None`).

In [141]:
@dataclass(frozen=True)
class Range:
    minimum: int
    maximum: int


@dataclass(frozen=True)
class MaxLength:
    limit: int


@dataclass(frozen=True)
class Pattern:
    regex: str


def validate(instance: object) -> list[str]:
    """Error messages for a dataclass instance, driven by Annotated metadata."""
    raise NotImplementedError("Lab 4.2 exercise 7: write validate")

Run the checks.

In [142]:
@dataclass
class CustomerForm:
    name: Annotated[str, MaxLength(20)]
    age: Annotated[int, Range(0, 150)]
    email: Annotated[str, Pattern(r"[^@]+@[^@]+"), MaxLength(40)]
    city: str
    tags: list[str] = field(default_factory=list)


run_checks(
    [
        ("valid", lambda: validate(CustomerForm("Asha", 31, "asha@example.com", "Pune")), []),
        (
            "range",
            lambda: validate(CustomerForm("Asha", 200, "asha@example.com", "Pune")),
            ["age: 200 is not between 0 and 150"],
        ),
        (
            "length and pattern",
            lambda: validate(CustomerForm("A" * 21, 31, "not-an-email", "Pune")),
            ["name: longer than 20 characters", "email: does not match [^@]+@[^@]+"],
        ),
        (
            "wrong types",
            lambda: validate(CustomerForm("Asha", "31", "a@b", 42)),
            ["age: expected int, got str", "city: expected str, got int"],
        ),
        (
            "bool is not int",
            lambda: validate(CustomerForm("Asha", True, "a@b", "Pune")),
            ["age: expected int, got bool"],
        ),
        (
            "several constraints",
            lambda: validate(CustomerForm("Asha", 31, "x" * 41 + "@b", "Pune")),
            ["email: longer than 40 characters"],
        ),
        (
            "generic hints skipped",
            lambda: validate(CustomerForm("Asha", 31, "a@b", "Pune", tags="oops")),
            [],
        ),
    ]
)

FAIL  valid: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  range: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  length and pattern: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  wrong types: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  bool is not int: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  several constraints: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  generic hints skipped: raised NotImplementedError: Lab 4.2 exercise 7: write validate
0/7 checks passed


### Recap — Unit 4.2

| Concept | One-line takeaway | TypeScript equivalent |
| --- | --- | --- |
| Gradual typing | Hints are checked by tools, ignored by the interpreter; untyped code is `Any` | `allowJs`, `// @ts-check` |
| Nominal vs structural | Classes are nominal; `Protocol` opts into structural typing | everything is structural |
| Annotation syntax | `x: int`, `-> T`, `ClassVar`, `Final`; lazy since 3.14 (`annotationlib`) | `: number`, `static`, `const` |
| Collections | `list[int]`, `dict[str, list[User]]`, `tuple[int, str]` vs `tuple[int, ...]`; accept abc, return concrete | `number[]`, `Record`, tuples |
| Unions | `X \| None`, narrow with `is None`, `isinstance`, `match`; `assert_never` for exhaustiveness | control-flow narrowing, `never` |
| Escape hatches | `Any` off, `object` safe, `cast` lies, `# type: ignore[code]` | `any`, `unknown`, `as`, `@ts-expect-error` |
| Precise types | `Literal`, `LiteralString`, `Final`, `NewType`, `Self`, `TypedDict` | literal types, brands, `this`, interfaces |
| Generics | `def f[T]`, `class C[T]`, `type A[T] = ...`; bounds, constraints, defaults; variance inferred | `<T extends …= …>` |
| Callables | `Callable`, callback Protocols, `ParamSpec`, `Concatenate`, `@overload` | function types, overloads |
| Class typing | `type[T]`, `@override`, `@final`, generic Protocols | `typeof C`, `override` |
| Custom narrowing | `TypeIs` (both branches) over `TypeGuard`; tagged unions | `x is T`, discriminated unions |
| `Annotated` | Metadata for run-time libraries; read with `get_type_hints(include_extras=True)` | (none: Zod schemas) |
| Checkers | mypy / pyright strict, `pyproject.toml` config, `py.typed`, stubs, ratchet adoption | `tsc --strict`, `.d.ts` |

**Further reading**

- [`typing`](https://docs.python.org/3/library/typing.html),
  [`annotationlib`](https://docs.python.org/3/library/annotationlib.html),
  [`collections.abc`](https://docs.python.org/3/library/collections.abc.html),
  [Annotations best practices](https://docs.python.org/3/howto/annotations.html)
- [Typing specification](https://typing.python.org/en/latest/spec/) and
  [typing guides](https://typing.python.org/en/latest/guides/)
- [mypy documentation](https://mypy.readthedocs.io/),
  [pyright documentation](https://microsoft.github.io/pyright/),
  [typeshed](https://github.com/python/typeshed)
- [PEP 484](https://peps.python.org/pep-0484/), [PEP 561](https://peps.python.org/pep-0561/),
  [PEP 593](https://peps.python.org/pep-0593/), [PEP 604](https://peps.python.org/pep-0604/),
  [PEP 612](https://peps.python.org/pep-0612/), [PEP 647](https://peps.python.org/pep-0647/),
  [PEP 649](https://peps.python.org/pep-0649/), [PEP 675](https://peps.python.org/pep-0675/),
  [PEP 692](https://peps.python.org/pep-0692/), [PEP 695](https://peps.python.org/pep-0695/),
  [PEP 696](https://peps.python.org/pep-0696/), [PEP 698](https://peps.python.org/pep-0698/),
  [PEP 705](https://peps.python.org/pep-0705/), [PEP 742](https://peps.python.org/pep-0742/),
  [PEP 749](https://peps.python.org/pep-0749/)

## Unit 4.3 — Testing, Debugging and Logging

**Objectives**

- Choose what to test at which level (unit, integration, end to end) and write
  tests in the Arrange-Act-Assert shape.
- Write and run pytest tests: discovery, plain `assert`, `pytest.raises`,
  `pytest.approx`, the everyday CLI flags.
- Share setup with fixtures (scopes, teardown, `conftest.py`, factories, the
  built-in `tmp_path`, `monkeypatch`, `capsys` and `caplog`).
- Generate cases with `parametrize` and Hypothesis; control runs with markers.
- Replace collaborators with mocks, stubs and fakes, patch in the right place, and
  control time.
- Measure line and branch coverage and know what it does not tell you.
- Debug with `breakpoint()`, pdb, post-mortem and `faulthandler`.
- Configure logging properly (`dictConfig`, handlers, formatters, filters, JSON) and
  follow the rules for libraries.
- Gate every commit and pull request with ruff, pre-commit and CI.

**JS bridge: Jest / Vitest → pytest**

| Jest / Vitest | pytest / Python | The difference that bites |
| --- | --- | --- |
| `test("adds", () => {...})`, `it(...)` | `def test_adds(): ...` in `test_*.py` | Found by **naming convention**; no registration call |
| `describe("Cart", ...)` | a module, or a `class TestCart:` | Test classes have no `__init__` and no base class |
| `expect(total).toBe(3)` | `assert total == 3` | Plain `assert`; pytest **rewrites** it to explain failures |
| `expect(fn).toThrow(/bad/)` | `with pytest.raises(ValueError, match="bad"):` | A context manager, not a callback |
| `toBeCloseTo(0.3)` | `== pytest.approx(0.3)` | Works on lists and dicts too |
| `beforeEach` / `afterEach` | fixtures (`yield` for teardown) | Injected **by parameter name** |
| `test.each([...])` | `@pytest.mark.parametrize` | Each case is a separate test with an id |
| `test.skip`, `test.failing` | `@pytest.mark.skip`, `@pytest.mark.xfail` | `skipif(condition, reason=...)` too |
| `jest.fn()`, `vi.fn()` | `unittest.mock.Mock()` | Attributes appear on access; use `autospec` |
| `jest.mock("./rates")` | `mock.patch("shop.checkout.fetch_rate")` | Patch **where the name is looked up** |
| `jest.useFakeTimers()` | inject a clock (or `monkeypatch`) | `datetime.now` cannot be patched directly |
| `--coverage` | `pytest --cov` (pytest-cov) | `--cov-branch` for branch coverage |
| fast-check | Hypothesis | Same idea: properties + shrinking |
| `-t "name"` | `-k "name"` | `-k` takes a boolean expression |
| `console.log` | `logging` (`print` only in scripts) | Configure once at the entry point |
| `debugger;` statement | `breakpoint()` | Disable with `PYTHONBREAKPOINT=0` |
| ESLint + Prettier, husky | ruff (lint + format), pre-commit | One tool replaces several |

**How the tool demos run.** Tests live in files, so each demo writes a tiny project
to `notebooks/_scratch/unit43/<name>/` with `project(name, files)` and runs the
**real pytest** (or coverage, pdb, pre-commit, ruff) in a child process with
`run_pytest(root, *flags)`, printing its output. A fresh interpreter per run means no
stale imports. Paths are shown as `<scratch>`, `<python>` (the interpreter's
install folder) or `~`, and durations as `<time>`.

In [143]:
import ast
import contextlib
import functools
import importlib.util
import inspect
import io
import json
import linecache
import logging
import logging.config
import logging.handlers
import os
import queue
import re
import shlex
import shutil
import subprocess
import sys
import textwrap
import time
import types
from collections.abc import Callable
from datetime import UTC, datetime
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path
from unittest import mock

REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
UNIT43_DIR = scratch_dir / "unit43"
UNIT43_DIR.mkdir(parents=True, exist_ok=True)
VENV_BIN = Path(sys.executable).parent
PYTHON_HOME = Path(sys.base_prefix)
PAISE = Decimal("0.01")


def tool_env(**overrides: str) -> dict[str, str]:
    """Environment for child tools: plain output, scratch caches, no personal git/IPython config."""
    dropped = ("UV_", "PIP_", "VIRTUAL_ENV", "PYTHONPATH", "PYTEST_", "COV", "PYTHONBREAKPOINT")
    inherited = {key: value for key, value in os.environ.items() if not key.startswith(dropped)}
    return {
        **inherited,
        "NO_COLOR": "1",
        "PY_COLORS": "0",
        "PYTHON_COLORS": "0",
        "COLUMNS": "80",
        "PYTHONDONTWRITEBYTECODE": "1",
        "PRE_COMMIT_HOME": str(UNIT43_DIR / "pre-commit-cache"),
        "IPYTHONDIR": str(UNIT43_DIR / "ipython-profile"),
        "GIT_CONFIG_GLOBAL": os.devnull,
        "GIT_CONFIG_NOSYSTEM": "1",
        "GIT_CEILING_DIRECTORIES": str(scratch_dir),
        "GIT_AUTHOR_NAME": "Course Demo",
        "GIT_AUTHOR_EMAIL": "demo@example.com",
        "GIT_COMMITTER_NAME": "Course Demo",
        "GIT_COMMITTER_EMAIL": "demo@example.com",
        **overrides,
    }


TIMING = re.compile(r"\b(in|after) \d+(\.\d+)?(ms|s|µs)\b")
ANSI_ESCAPE = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")
ADDRESS = re.compile(r" at 0x[0-9a-f]+")


def tidy(text: str) -> str:
    """Show paths as <scratch>, <repo>, <python> or ~, timings as <time>, addresses as 0x..."""
    for folder, label in (
        (UNIT43_DIR, "<scratch>"),
        (REPO_ROOT, "<repo>"),
        (PYTHON_HOME, "<python>"),
        (Path.home(), "~"),
    ):
        text = text.replace(str(folder), label)
    return ADDRESS.sub(" at 0x...", ANSI_ESCAPE.sub("", TIMING.sub(r"\1 <time>", text)))


def run_tool(
    *command: str,
    cwd: Path = UNIT43_DIR,
    env: dict[str, str] | None = None,
    input_text: str | None = None,
    merged: bool = False,
    timeout: int = 300,
) -> tuple[int, str]:
    """Run a CLI tool in a child process; return (exit code, tidied stdout + stderr).

    merged=True interleaves stderr with stdout in the order they were written.
    """
    name = command[0]
    found = sys.executable if name == "python" else str(VENV_BIN / name)
    if not Path(found).exists():
        found = shutil.which(name)
    if found is None:
        return 127, f"SKIP: {name} is not installed on this machine"
    completed = subprocess.run(
        [found, *command[1:]],
        cwd=cwd,
        env={**(env or tool_env()), **({"PYTHONUNBUFFERED": "1"} if merged else {})},
        input=input_text,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT if merged else subprocess.PIPE,
        text=True,
        timeout=timeout,
    )
    return completed.returncode, tidy((completed.stdout + (completed.stderr or "")).rstrip())


def shell(
    *command: str,
    cwd: Path = UNIT43_DIR,
    env: dict[str, str] | None = None,
    input_text: str | None = None,
    merged: bool = False,
    keep: str | None = None,
    max_lines: int | None = None,
) -> None:
    """Print `[folder]$ command` and the tool's real output (only lines matching keep, if given)."""
    exit_code, output = run_tool(*command, cwd=cwd, env=env, input_text=input_text, merged=merged)
    print(f"[{tidy(str(cwd))}]$ {shlex.join(command)}")
    lines = output.splitlines()
    if keep is not None:
        lines = [line for line in lines if re.search(keep, line)]
    if max_lines is not None and len(lines) > max_lines:
        lines = [*lines[:max_lines], f"... ({len(lines) - max_lines} more lines)"]
    if lines:
        print(textwrap.indent("\n".join(lines), "  ", lambda line: True))
    if exit_code:
        print(f"  (exit code {exit_code})")


def fresh_dir(name: str) -> Path:
    """An empty folder <scratch>/<name>, recreated on every run."""
    folder = UNIT43_DIR / name
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    return folder


def write_files(root: Path, files: dict[str, str]) -> Path:
    """Write relative path -> text (dedented) under root; return root."""
    for relative_path, text in files.items():
        target = root / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(textwrap.dedent(text).lstrip("\n"))
    return root


def show_file(path: Path) -> None:
    """Print a text file with a header line."""
    print(f"--- {path.name} ---\n{tidy(path.read_text().rstrip())}")


PYTEST_CONFIG = """
[tool.pytest.ini_options]
pythonpath = ["."]
testpaths = ["tests"]
"""


def project(name: str, files: dict[str, str]) -> Path:
    """A fresh scratch test project: pyproject.toml (pytest config) plus the given files."""
    return write_files(fresh_dir(name), {"pyproject.toml": PYTEST_CONFIG, **files})


def run_pytest(root: Path, *arguments: str, **shell_options: object) -> None:
    """Run the real pytest in root (a child process) and print its output."""
    shell("pytest", *arguments, cwd=root, merged=True, **shell_options)


class FrozenClock(logging.Filter):
    """Demo-only filter: stamp every record with one fixed UTC time so outputs never change."""

    def __init__(self, moment: datetime = datetime(2026, 11, 7, 9, 30, tzinfo=UTC)) -> None:
        super().__init__()
        self.timestamp = moment.timestamp()

    def filter(self, record: logging.LogRecord) -> bool:
        record.created, record.msecs = self.timestamp, 0.0
        return True


def source_of(obj: object) -> str:
    """Source code of a function or class, also for classes defined in a notebook cell."""
    try:
        return textwrap.dedent(inspect.getsource(obj))
    except OSError, TypeError:
        if not isinstance(obj, type):
            raise
    for member in vars(obj).values():
        member = getattr(member, "__func__", None) or getattr(member, "fget", None) or member
        code_object = getattr(inspect.unwrap(member), "__code__", None)
        lines = linecache.getlines(code_object.co_filename) if code_object else []
        if lines:
            start = obj.__firstlineno__ - 1
            while start > 0 and lines[start - 1].lstrip().startswith("@"):
                start -= 1
            return textwrap.dedent("".join(inspect.getblock(lines[start:])))
    raise OSError(f"source of {obj.__name__} not found")


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


for tool in ("pytest", "coverage", "pre-commit", "ruff"):
    exit_code, version = run_tool(tool, "--version")
    print(f"{tool}: {version.splitlines()[0] if version else 'not installed'}")
for package in ("pytest_cov", "hypothesis"):
    found = importlib.util.find_spec(package) is not None
    print(f"{package}: {'installed' if found else 'not installed; its cells print a skip message'}")

pytest: pytest 9.2.0
coverage: Coverage.py, version 7.17.2 with C extension


pre-commit: pre-commit 4.7.0
ruff: ruff 0.17.3
pytest_cov: installed
hypothesis: installed


**The code under test.** Most demos test a small `shop` package written into each
scratch project: `shop/money.py` holds the shared `Money` value type from Phase 2
(Decimal amounts rounded half-up to the paisa, currency checks, `+ - *`, `sum()`
support, ordering) and `shop/billing.py` holds `split_bill` from Lab 1.3. Lab 4.3
adds an exchange-rate client and the Phase 2 inventory model.
`shop_files("money", "billing")` returns the files to write.

In [144]:
MONEY_MODULE = """
    \"\"\"The shared Money value type (Lab 2.4, exercise 1; Phase 2 Project, Step 1).\"\"\"

    from __future__ import annotations

    import functools
    from decimal import ROUND_HALF_UP, Decimal

    PAISE = Decimal("0.01")


    class CurrencyMismatchError(ValueError):
        \"\"\"Two Money values in different currencies were combined or compared.\"\"\"


    @functools.total_ordering
    class Money:
        __slots__ = ("_amount", "_currency")

        def __init__(self, amount: Decimal | int | str, currency: str = "INR") -> None:
            if isinstance(amount, float):
                raise TypeError("amount must be Decimal, int or str, not float")
            if not (len(currency) == 3 and currency.isalpha() and currency.isupper()):
                raise ValueError(f"invalid currency code: {currency!r}")
            self._amount = Decimal(amount).quantize(PAISE, rounding=ROUND_HALF_UP)
            self._currency = currency

        @property
        def amount(self) -> Decimal:
            return self._amount

        @property
        def currency(self) -> str:
            return self._currency

        def __repr__(self) -> str:
            return f"Money({str(self._amount)!r}, {self._currency!r})"

        def __str__(self) -> str:
            return f"{self._currency} {self._amount:,.2f}"

        def _same_currency(self, other: Money) -> None:
            if other._currency != self._currency:
                raise CurrencyMismatchError(
                    f"currency mismatch: {self._currency} vs {other._currency}"
                )

        def __eq__(self, other: object) -> bool:
            if not isinstance(other, Money):
                return NotImplemented
            return (self._amount, self._currency) == (other._amount, other._currency)

        def __hash__(self) -> int:
            return hash((self._amount, self._currency))

        def __lt__(self, other: object) -> bool:
            if not isinstance(other, Money):
                return NotImplemented
            self._same_currency(other)
            return self._amount < other._amount

        def __add__(self, other: object) -> Money:
            if not isinstance(other, Money):
                return NotImplemented
            self._same_currency(other)
            return Money(self._amount + other._amount, self._currency)

        def __radd__(self, other: object) -> Money:
            if isinstance(other, int) and not isinstance(other, bool) and other == 0:
                return self  # sum() starts from 0
            return NotImplemented

        def __sub__(self, other: object) -> Money:
            if not isinstance(other, Money):
                return NotImplemented
            self._same_currency(other)
            return Money(self._amount - other._amount, self._currency)

        def __mul__(self, factor: object) -> Money:
            if isinstance(factor, bool) or not isinstance(factor, (int, Decimal)):
                return NotImplemented
            return Money(self._amount * factor, self._currency)

        __rmul__ = __mul__

        def __neg__(self) -> Money:
            return Money(-self._amount, self._currency)

        def __bool__(self) -> bool:
            return bool(self._amount)
"""

BILLING_MODULE = """
    \"\"\"split_bill from Lab 1.3.\"\"\"

    from decimal import ROUND_HALF_UP, Decimal

    PAISE = Decimal("0.01")


    def split_bill(total_amount: str, people_count: int) -> list[Decimal]:
        \"\"\"Split a rupee amount into people_count parts, exact to the paisa.\"\"\"
        if people_count < 1:
            raise ValueError(f"people_count must be at least 1, got {people_count}")
        total_in_paise = int(Decimal(total_amount).quantize(PAISE, rounding=ROUND_HALF_UP) * 100)
        base_share_in_paise, leftover_paise = divmod(total_in_paise, people_count)
        return [
            (Decimal(base_share_in_paise + (1 if index < leftover_paise else 0)) / 100).quantize(
                PAISE
            )
            for index in range(people_count)
        ]
"""


def shop_files(*modules: str) -> dict[str, str]:
    """Files of a small `shop` package with the requested modules ("money", "billing")."""
    sources = {"money": MONEY_MODULE, "billing": BILLING_MODULE}
    return {"shop/__init__.py": "", **{f"shop/{name}.py": sources[name] for name in modules}}


print(f"{sorted(shop_files('money', 'billing'))=}")

sorted(shop_files('money', 'billing'))=['shop/__init__.py', 'shop/billing.py', 'shop/money.py']


### 4.3.1 Testing strategy

**The test pyramid.** Many fast, isolated tests at the bottom; fewer slower tests
that wire real parts together; a handful of end-to-end tests at the top. The shape
is about **cost and feedback**: a unit test runs in microseconds and points at one
function; an end-to-end failure says "something is broken" after seconds or
minutes.

```mermaid
flowchart TB
    E2E["End-to-end: a few<br/>real HTTP + DB + browser"] --> INT
    INT["Integration: some<br/>service + real SQLite, API + test client"] --> UNIT
    UNIT["Unit: many<br/>pure functions, one class, collaborators replaced"]
```

<details><summary>ASCII version</summary>

```text
        /\        end-to-end   (few, slow, broad)
       /  \
      /----\      integration  (some)
     /      \
    /--------\    unit         (many, fast, precise)
```

</details>

| Level | What it exercises | Example in this course | Speed | When it fails, you learn |
| --- | --- | --- | --- | --- |
| Unit | One function or class; collaborators replaced or trivial | `Money("0.125") == Money("0.13")` | µs | exactly which rule broke |
| Integration | Several real parts together (service + repository, app + DB) | `InventoryService` with an SQLite repository | ms | the parts disagree (SQL, wiring, serialization) |
| End-to-end | The deployed system through its real interface | `POST /orders` on a running FastAPI app (Phase 6) | s | the system is broken for users |

> **Note:** the "testing trophy" (Kent C. Dodds) puts more weight on integration
> tests. Both agree on the important part: most tests should be fast and
> deterministic, and each level should test what only it can.

**Arrange-Act-Assert (AAA).** Every test has three parts, in this order, separated
by a blank line: set up the inputs (*arrange*), do **one** thing (*act*), check the
outcome (*assert*). A pytest test is just a function, so the kernel can call it
directly; a passing test returns `None`, a failing one raises `AssertionError`.

In [145]:
def apply_coupon(total_in_paise: int, percent: int) -> int:
    """Take percent off a total (integer paise, rounded down in the customer's favour)."""
    return total_in_paise - total_in_paise * percent // 100


def test_coupon_takes_the_percentage_off_the_total() -> None:
    """AAA: one behaviour, one act, explicit expected values."""
    # Arrange
    total_in_paise, percent = 67_900, 10

    # Act
    payable = apply_coupon(total_in_paise, percent)

    # Assert
    assert payable == 61_110


print(f"{test_coupon_takes_the_percentage_off_the_total()=}  (None: passed)")

test_coupon_takes_the_percentage_off_the_total()=None  (None: passed)


**What makes a test good** (the same in Jest and pytest):

| Property | Means | Smell when missing |
| --- | --- | --- |
| Fast | milliseconds, so you run it on every save | people stop running the suite |
| Isolated | no shared state; any order, in parallel | passes alone, fails in the suite |
| Deterministic | same result every run (seeded, fixed clock, no network) | "flaky" reruns |
| Behaviour-focused | checks what the caller sees, not private details | refactors break tests |
| Readable | the name states the rule; the failure says what broke | `test_1`, `test_money_works` |

The first real run: a project with one module and one test file. Note the header
(Python, pytest and plugin versions, `rootdir`, the config file), one character per
test (`.` pass, `F` fail, `E` error, `s` skip, `x` xfail), and the summary line.

In [146]:
first_run = project(
    "first_run",
    {
        **shop_files("billing"),
        "tests/test_billing.py": """
            from decimal import Decimal

            from shop.billing import split_bill


            def test_even_split():
                assert split_bill("90", 3) == [Decimal("30.00")] * 3


            def test_leftover_goes_to_first_people():
                assert split_bill("100", 3) == [
                    Decimal("33.34"), Decimal("33.33"), Decimal("33.33")
                ]
        """,
    },
)
show_file(first_run / "pyproject.toml")
run_pytest(first_run)

--- pyproject.toml ---
[tool.pytest.ini_options]
pythonpath = ["."]
testpaths = ["tests"]


[<scratch>/first_run]$ pytest
  ============================= test session starts ==============================
  platform linux -- Python 3.15.1, pytest-9.2.0, pluggy-1.7.0
  rootdir: <scratch>/first_run
  configfile: pyproject.toml
  testpaths: tests
  plugins: platformdirs-4.13.5, hypothesis-6.170.15, anyio-4.15.1, cov-7.1.0
  collected 2 items
  
  tests/test_billing.py ..                                                 [100%]
  
  ============================== 2 passed in <time> ===============================


### 4.3.2 pytest fundamentals

**Discovery.** pytest collects tests by convention; no registration, no imports of
a test framework needed for simple tests.

| Rule | Default | Configure with |
| --- | --- | --- |
| Where to look | the `testpaths` folders, else the current folder, recursively | `testpaths = ["tests"]` |
| Test files | `test_*.py` or `*_test.py` | `python_files` |
| Test classes | `Test*` classes **without** an `__init__` | `python_classes` |
| Test functions/methods | `test_*` | `python_functions` |
| Root folder | the folder of the config file (`pyproject.toml`, `pytest.ini`, …) | `--rootdir` |
| Import path | `pythonpath = ["."]` puts the project root on `sys.path` | `--import-mode` |

> **Note:** `python -m pytest` also adds the current folder to `sys.path`; the plain
> `pytest` command does not. Set `pythonpath` in the config so both behave alike.

`--collect-only -q` lists what pytest would run, without running it. `helpers.py`,
the class with an `__init__`, and `check_total` are not collected:

In [147]:
discovery = project(
    "discovery",
    {
        "tests/test_cart.py": """
            def test_empty_cart_total_is_zero():
                assert sum([]) == 0


            def check_total():  # not test_*: ignored
                assert False


            class TestDiscounts:
                def test_member_discount(self):
                    assert round(100 * 0.9) == 90


            class TestWithInit:  # has __init__: pytest warns and skips it
                def __init__(self):
                    self.rate = 0.9

                def test_never_collected(self):
                    assert False
        """,
        "tests/orders_test.py": "def test_order_id_is_text():\n    assert str(42) == '42'\n",
        "tests/helpers.py": "def test_looks_like_a_test():\n    assert False\n",
    },
)
run_pytest(discovery, "--collect-only", "-q")

[<scratch>/discovery]$ pytest --collect-only -q
  tests/orders_test.py::test_order_id_is_text
  tests/test_cart.py::test_empty_cart_total_is_zero
  tests/test_cart.py::TestDiscounts::test_member_discount
  
  =============================== warnings summary ===============================
  tests/test_cart.py:14
    <scratch>/discovery/tests/test_cart.py:14: PytestCollectionWarning: cannot collect test class 'TestWithInit' because it has a __init__ constructor (from: tests/test_cart.py)
      class TestWithInit:  # has __init__: pytest warns and skips it
  
  -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
  3 tests collected in <time>


**Assertion rewriting.** pytest uses plain `assert`, yet a failure shows the values
involved, a diff for lists, dicts and strings, and the sub-expressions. It does this
by **rewriting the `assert` statements of test modules** when importing them (an
import hook, Unit 3.3). Jest needs `expect(...).toEqual(...)` to get the same
report.

In [148]:
rewriting = project(
    "rewriting",
    {
        "tests/test_reports.py": """
            def test_order_summary():
                expected = {"sku": "TEA-500", "quantity": 3, "status": "paid"}
                actual = {"sku": "TEA-500", "quantity": 2, "status": "paid"}
                assert actual == expected


            def test_skus():
                assert ["TEA-500", "MUG-02", "JAR-750"] == ["TEA-500", "MUG-03", "JAR-750"]


            def test_function_result():
                def tax(amount):
                    return amount * 18 // 100

                assert tax(250) == 46
        """,
    },
)
run_pytest(rewriting, "-q", "--no-header", "--tb=short")

[<scratch>/rewriting]$ pytest -q --no-header --tb=short
  FFF                                                                      [100%]
  =================================== FAILURES ===================================
  ______________________________ test_order_summary ______________________________
  tests/test_reports.py:4: in test_order_summary
      assert actual == expected
  E   AssertionError: assert {'sku': 'TEA-...atus': 'paid'} == {'sku': 'TEA-...atus': 'paid'}
  E     
  E     Omitting 2 identical items, use -vv to show
  E     Differing items:
  E     {'quantity': 2} != {'quantity': 3}
  E     Use -v to get more diff
  __________________________________ test_skus ___________________________________
  tests/test_reports.py:8: in test_skus
      assert ["TEA-500", "MUG-02", "JAR-750"] == ["TEA-500", "MUG-03", "JAR-750"]
  E   AssertionError: assert ['TEA-500', '...2', 'JAR-750'] == ['TEA-500', '...3', 'JAR-750']
  E     
  E     At index 1 diff: 'MUG-02' != 'MUG-03'
  E   

**Under the hood: the rewritten `assert`.** pytest's import hook calls
`rewrite_asserts` on the module's AST before compiling it. Running it by hand on a
one-line test shows what the interpreter actually executes: the comparison is
stored in temporaries, and on failure pytest formats each value with `saferepr`.
The `@py_…` names are deliberately invalid identifiers, so they can never clash with
your own names.

In [149]:
from _pytest.assertion.rewrite import rewrite_asserts  # pytest internals: may change

test_source = "def test_total():\n    total = 1 + 1\n    assert total == 3\n"
syntax_tree = ast.parse(test_source)
rewrite_asserts(syntax_tree, test_source.encode())
rewritten = ast.unparse(syntax_tree)
print("\n".join(line[:110] for line in rewritten.splitlines()))

import builtins as @py_builtins
import _pytest.assertion.rewrite as @pytest_ar

def test_total():
    total = 1 + 1
    @py_assert2 = 3
    @py_assert1 = total == @py_assert2
    if not @py_assert1:
        @py_format4 = @pytest_ar._call_reprcompare(('==',), (@py_assert1,), ('%(py0)s == %(py3)s',), (total, @
        @py_format6 = ('' + 'assert %(py5)s') % {'py5': @py_format4}
        raise AssertionError(@pytest_ar._format_explanation(@py_format6))
    @py_assert1 = @py_assert2 = None


> **Warning:** `python -O` strips every `assert` statement. That is fine for tests
> (pytest never runs with `-O`) but it is why production code must not use `assert`
> for validation: raise `ValueError` instead.

**`pytest.raises`** checks that a block raises an exception of a type (or a
subclass). `match=` is a regular expression searched in `str(error)`, and the
`ExceptionInfo` object gives you the exception afterwards for more checks. pytest's
helpers also work outside pytest, so the kernel can try them directly:

In [150]:
import pytest

with pytest.raises(ValueError, match=r"at least 1, got 0") as error_info:
    raise ValueError("people_count must be at least 1, got 0")
print(f"{error_info.type.__name__=}, {str(error_info.value)=}")

try:
    with pytest.raises(ValueError):
        int("42")  # no exception: the check itself fails
except BaseException as failure:  # pytest's Failed derives from BaseException
    print(f"{type(failure).__name__}: {failure}")

error_info.type.__name__='ValueError', str(error_info.value)='people_count must be at least 1, got 0'
Failed: DID NOT RAISE ValueError


The same checks as a test file. `pytest.raises` also accepts a tuple of types, and
`ExceptionInfo.group_contains` checks exception groups (Unit 1.14). The failing
test shows the report when the message does not match:

In [151]:
raises_demo = project(
    "raises_demo",
    {
        **shop_files("money", "billing"),
        "tests/test_errors.py": """
            import pytest

            from shop.billing import split_bill
            from shop.money import CurrencyMismatchError, Money


            def test_zero_people_is_rejected():
                with pytest.raises(ValueError, match="at least 1"):
                    split_bill("100", 0)


            def test_currency_mismatch_is_a_value_error():
                with pytest.raises(ValueError) as error_info:
                    Money(1, "INR") + Money(1, "USD")
                assert error_info.type is CurrencyMismatchError
                assert "INR vs USD" in str(error_info.value)


            def test_float_message():
                with pytest.raises(TypeError, match="no floats allowed"):
                    Money(0.1)
        """,
    },
)
run_pytest(raises_demo, "-q", "--no-header", "--tb=line")

[<scratch>/raises_demo]$ pytest -q --no-header --tb=line
  ..F                                                                      [100%]
  =================================== FAILURES ===================================
  E   AssertionError: Regex pattern did not match.
        Expected regex: 'no floats allowed'
        Actual message: 'amount must be Decimal, int or str, not float'
  <scratch>/raises_demo/tests/test_errors.py:20: AssertionError: Regex pattern did not match.
  =========================== short test summary info ============================
  FAILED tests/test_errors.py::test_float_message - AssertionError: Regex patte...
  1 failed, 2 passed in <time>
  (exit code 1)


**`pytest.approx`** compares floats with a tolerance (default: relative `1e-6`, or
absolute `1e-12` near zero), like Jest's `toBeCloseTo`, and also works element-wise
for sequences and dict values. Use `rel=` or `abs=` to set the tolerance. For money,
use `Decimal` and exact comparisons instead.

In [152]:
print(f"{0.1 + 0.2 == 0.3=}")
print(f"{0.1 + 0.2 == pytest.approx(0.3)=}")
print(f"{[0.1 + 0.2, 0.2 + 0.4] == pytest.approx([0.3, 0.6])=}")
print(f"{ {'tax': 18.000001} == pytest.approx({'tax': 18.0})=}")
print(f"{101 == pytest.approx(100, rel=0.05)=}, {101 == pytest.approx(100, abs=0.5)=}")
print(f"{pytest.approx(0.3)!r}")

0.1 + 0.2 == 0.3=False
0.1 + 0.2 == pytest.approx(0.3)=True
[0.1 + 0.2, 0.2 + 0.4] == pytest.approx([0.3, 0.6])=True
 {'tax': 18.000001} == pytest.approx({'tax': 18.0})=True
101 == pytest.approx(100, rel=0.05)=True, 101 == pytest.approx(100, abs=0.5)=False
0.3 ± 3.0e-07


**Predict the output.**

```python
import pytest

print(0.1 + 0.2 == pytest.approx(0.3), [0.1 + 0.2] == [0.3], 1e-13 == pytest.approx(0))
```

<details><summary>Answer</summary>

`True False True`. `approx` applies a tolerance; the plain list comparison uses
exact `==` on the floats inside (`0.30000000000000004 != 0.3`). Near zero a relative
tolerance is useless (any percentage of 0 is 0), so `approx` also has an absolute
tolerance of `1e-12`, which `1e-13` is within.

</details>

Run the next cell to check your prediction.

In [153]:
print(0.1 + 0.2 == pytest.approx(0.3), [0.1 + 0.2] == [0.3], 1e-13 == pytest.approx(0))

True False True


**Everyday CLI flags.** One demo project with five tests (one of them failing, one
that prints) is used for each flag below.

| Flag | Meaning | Jest / Vitest |
| --- | --- | --- |
| `-k "expr"` | run tests whose names match a boolean expression (`and`, `or`, `not`) | `-t` |
| `-x` | stop at the first failure (`--maxfail=N` for N) | `--bail` |
| `-v` / `-q` | one line per test / minimal output | `--verbose` |
| `--lf` | rerun only the tests that failed last time (`--ff`: failed first) | `--onlyFailures` |
| `-s` | do not capture `print` output (same as `--capture=no`) | (prints always shown) |
| `--tb=short\|line\|no` | traceback style | |
| `-ra` | summary of everything that did not pass | |
| `--durations=5` | the five slowest tests | |

In [154]:
flags_demo = project(
    "flags_demo",
    {
        "tests/test_checkout.py": """
            def test_cart_total():
                assert 240 * 2 + 199 == 679


            def test_cart_total_with_coupon():
                assert 679 - 50 == 629


            def test_invoice_rounding():
                assert round(2.675, 2) == 2.68  # fails: 2.675 is stored as 2.67499...


            def test_shipping_is_free_above_500():
                print("debug: checking the free-shipping rule")
                assert 679 > 500


            def test_invoice_number_format():
                assert f"INV-{7:05d}" == "INV-00007"
        """,
    },
)
run_pytest(flags_demo, "-q", "--no-header", "--tb=no")

[<scratch>/flags_demo]$ pytest -q --no-header --tb=no
  ..F..                                                                    [100%]
  =========================== short test summary info ============================
  FAILED tests/test_checkout.py::test_invoice_rounding - assert 2.67 == 2.68
  1 failed, 4 passed in <time>
  (exit code 1)


`-k` selects tests by name with a boolean expression: `coupon or shipping` matches
two tests, `not total` deselects the two with "total" in their names.

In [155]:
run_pytest(flags_demo, "-q", "--no-header", "--tb=no", "-k", "coupon or shipping")
run_pytest(flags_demo, "-q", "--no-header", "--tb=no", "-k", "not total")

[<scratch>/flags_demo]$ pytest -q --no-header --tb=no -k 'coupon or shipping'
  ..                                                                       [100%]
  2 passed, 3 deselected in <time>


[<scratch>/flags_demo]$ pytest -q --no-header --tb=no -k 'not total'
  F..                                                                      [100%]
  =========================== short test summary info ============================
  FAILED tests/test_checkout.py::test_invoice_rounding - assert 2.67 == 2.68
  1 failed, 2 passed, 2 deselected in <time>
  (exit code 1)


`-x` stops at the first failure (useful with a long suite), and `-v` prints one line
per test with its **node id** (`file::Class::test[params]`), which you can paste
back on the command line to run a single test.

In [156]:
run_pytest(flags_demo, "-x", "-q", "--no-header", "--tb=no")
run_pytest(flags_demo, "-v", "--no-header", "--tb=no")

[<scratch>/flags_demo]$ pytest -x -q --no-header --tb=no
  ..F
  =========================== short test summary info ============================
  FAILED tests/test_checkout.py::test_invoice_rounding - assert 2.67 == 2.68
  !!!!!!!!!!!!!!!!!!!!!!!!!! stopping after 1 failures !!!!!!!!!!!!!!!!!!!!!!!!!!!
  1 failed, 2 passed in <time>
  (exit code 1)


[<scratch>/flags_demo]$ pytest -v --no-header --tb=no
  ============================= test session starts ==============================
  collecting ... collected 5 items
  
  tests/test_checkout.py::test_cart_total PASSED                           [ 20%]
  tests/test_checkout.py::test_cart_total_with_coupon PASSED               [ 40%]
  tests/test_checkout.py::test_invoice_rounding FAILED                     [ 60%]
  tests/test_checkout.py::test_shipping_is_free_above_500 PASSED           [ 80%]
  tests/test_checkout.py::test_invoice_number_format PASSED                [100%]
  
  =========================== short test summary info ============================
  FAILED tests/test_checkout.py::test_invoice_rounding - assert 2.67 == 2.68
  ========================= 1 failed, 4 passed in <time> ==========================
  (exit code 1)


`--lf` ("last failed") reads `.pytest_cache/`, which the previous run wrote, and
runs only the tests that failed then. In a red-green loop you fix, rerun `--lf`, and
finally run everything.

In [157]:
run_pytest(flags_demo, "--lf", "-q", "--no-header", "--tb=no")
print(f"{sorted(path.name for path in (flags_demo / '.pytest_cache' / 'v' / 'cache').iterdir())=}")

[<scratch>/flags_demo]$ pytest --lf -q --no-header --tb=no
  F                                                                        [100%]
  =========================== short test summary info ============================
  FAILED tests/test_checkout.py::test_invoice_rounding - assert 2.67 == 2.68
  1 failed in <time>
  (exit code 1)
sorted(path.name for path in (flags_demo / '.pytest_cache' / 'v' / 'cache').iterdir())=['lastfailed', 'nodeids']


**Output capture.** pytest captures `print` output and shows it only for failing
tests, so passing tests stay quiet. `-s` turns capturing off (handy with print
debugging or `breakpoint()`); `-rP` shows captured output of passing tests in the
summary instead.

In [158]:
run_pytest(flags_demo, "-q", "--no-header", "-k", "shipping")
run_pytest(flags_demo, "-q", "--no-header", "-k", "shipping", "-s")

[<scratch>/flags_demo]$ pytest -q --no-header -k shipping
  .                                                                        [100%]
  1 passed, 4 deselected in <time>


[<scratch>/flags_demo]$ pytest -q --no-header -k shipping -s
  debug: checking the free-shipping rule
  .
  1 passed, 4 deselected in <time>


**Running pytest from a notebook.** Three options:

| Option | How | Trade-off |
| --- | --- | --- |
| Child process (this course) | `subprocess.run(["pytest", ...])` | Fresh interpreter every run, exactly like CI; tests must live in files |
| In the kernel | `pytest.main([...])` returns an exit code | Fast, but test modules and the code under test stay in `sys.modules`: **edits are not picked up** |
| `ipytest` (third party) | `%%ipytest` cell magic runs tests defined in cells | Convenient for teaching; not used here |

The trap of `pytest.main`: the kernel imports `pricing.py` once. After the file
changes, a second `pytest.main` still sees the old function, because Python caches
modules (Unit 3.3). The cell cleans up `sys.modules` and `sys.path` afterwards.

In [159]:
in_kernel = project(
    "in_kernel",
    {
        "pricing.py": "def with_gst(amount):\n    return amount * 118 // 100\n",
        "tests/test_pricing.py": (
            "from pricing import with_gst\n\n\ndef test_gst():\n    assert with_gst(100) == 118\n"
        ),
    },
)
path_before, modules_before = list(sys.path), set(sys.modules)
arguments = [
    str(in_kernel),
    "-q",
    "-p",
    "no:cacheprovider",
    "--capture=sys",
    "--rootdir",
    str(in_kernel),
]
with contextlib.redirect_stdout(io.StringIO()):
    first_exit = pytest.main(arguments)
(in_kernel / "pricing.py").write_text(
    "def with_gst(amount):\n    return amount * 112 // 100  # bug\n"
)
with contextlib.redirect_stdout(io.StringIO()):
    second_exit = pytest.main(arguments)
print(f"{first_exit!r}, {second_exit!r}  (the bug is invisible: pricing was cached)")
print(f"{'pricing' in sys.modules=}")
for name in set(sys.modules) - modules_before:
    if name in ("pricing", "test_pricing") or name.startswith("tests"):
        del sys.modules[name]
sys.path[:] = path_before
print(f"after clean-up: {'pricing' in sys.modules=}")

<ExitCode.OK: 0>, <ExitCode.OK: 0>  (the bug is invisible: pricing was cached)
'pricing' in sys.modules=True
after clean-up: 'pricing' in sys.modules=False


The same edited project in a child process catches the bug at once, which is why
every demo in this unit runs pytest that way:

In [160]:
run_pytest(in_kernel, "-q", "--no-header", "--tb=line", "-p", "no:cacheprovider")

[<scratch>/in_kernel]$ pytest -q --no-header --tb=line -p no:cacheprovider
  F                                                                        [100%]
  =================================== FAILURES ===================================
  E   assert 112 == 118
       +  where 112 = with_gst(100)
  <scratch>/in_kernel/tests/test_pricing.py:5: assert 112 == 118
  =========================== short test summary info ============================
  FAILED tests/test_pricing.py::test_gst - assert 112 == 118
  1 failed in <time>
  (exit code 1)


### 4.3.3 Fixtures

A **fixture** is a function decorated with `@pytest.fixture` that provides a value
to tests. A test asks for it by **naming it as a parameter**; pytest calls the
fixture, passes the result in, and (for `yield` fixtures) cleans up afterwards.
Compared with Jest's `beforeEach`, fixtures are explicit (you see what each test
uses), composable (fixtures can request other fixtures) and lazy (only the fixtures
a test asks for run).

In [161]:
fixture_basics = project(
    "fixture_basics",
    {
        **shop_files("money"),
        "tests/test_basket.py": """
            import pytest

            from shop.money import Money


            @pytest.fixture
            def basket():
                return [Money("240"), Money("199"), Money("49.50")]


            @pytest.fixture
            def basket_total(basket):  # a fixture using another fixture
                return sum(basket)


            def test_total(basket_total):
                assert basket_total == Money("488.50")


            def test_each_test_gets_a_fresh_basket(basket):
                basket.append(Money(1))
                assert len(basket) == 4
        """,
    },
)
run_pytest(fixture_basics, "-v", "--no-header")

[<scratch>/fixture_basics]$ pytest -v --no-header
  ============================= test session starts ==============================
  collecting ... collected 2 items
  
  tests/test_basket.py::test_total PASSED                                  [ 50%]
  tests/test_basket.py::test_each_test_gets_a_fresh_basket PASSED          [100%]
  
  ============================== 2 passed in <time> ===============================


**Scopes.** By default a fixture runs once per test (`scope="function"`). Wider
scopes share one instance: `"class"`, `"module"`, `"package"`, `"session"` (the whole
run). Use wide scopes for expensive, read-only resources (a database engine, a
parsed catalogue) and keep anything a test mutates function-scoped.
`--setup-show` prints every SETUP and TEARDOWN with its scope letter (S, M, C, F), so
you can watch the sharing:

In [162]:
scopes = project(
    "scopes",
    {
        "tests/test_scopes.py": """
            import pytest


            @pytest.fixture(scope="session")
            def catalogue():
                return {"TEA-500": 240, "MUG-02": 199}


            @pytest.fixture(scope="module")
            def price_list(catalogue):
                return sorted(catalogue.values())


            @pytest.fixture
            def cart():
                return []


            def test_cheapest(price_list):
                assert price_list[0] == 199


            def test_add(cart, catalogue):
                cart.append(catalogue["TEA-500"])
                assert cart == [240]


            def test_cart_starts_empty(cart):
                assert cart == []
        """,
    },
)
run_pytest(scopes, "--setup-show", "-q", "--no-header")

[<scratch>/scopes]$ pytest --setup-show -q --no-header
  
  SETUP    S catalogue
      SETUP    M price_list (fixtures used: catalogue)
          tests/test_scopes.py::test_cheapest (fixtures used: catalogue, price_list) .
          SETUP    F cart
          tests/test_scopes.py::test_add (fixtures used: cart, catalogue) .
          TEARDOWN F cart
          SETUP    F cart
          tests/test_scopes.py::test_cart_starts_empty (fixtures used: cart) .
          TEARDOWN F cart
      TEARDOWN M price_list
  TEARDOWN S catalogue
  3 passed in <time>


**`yield` fixtures: setup and teardown in one place.** Code before `yield` is setup,
the yielded value goes to the test, code after `yield` is teardown and runs **even
when the test fails** (like a context manager's `finally`, Unit 1.14). Here an
in-memory SQLite connection is created, filled, handed over and closed;
`request.addfinalizer(...)` is the older callback style.

In [163]:
teardown = project(
    "teardown",
    {
        "tests/test_orders_db.py": """
            import sqlite3

            import pytest


            @pytest.fixture
            def orders_db():
                connection = sqlite3.connect(":memory:")
                connection.execute("CREATE TABLE orders (id INTEGER, total INTEGER)")
                connection.executemany("INSERT INTO orders VALUES (?, ?)", [(1, 679), (2, 240)])
                print("\\nsetup: database ready")
                yield connection
                connection.close()
                print("teardown: database closed")


            def test_order_count(orders_db):
                assert orders_db.execute("SELECT COUNT(*) FROM orders").fetchone() == (2,)


            def test_failing_test_still_tears_down(orders_db):
                assert orders_db.execute("SELECT SUM(total) FROM orders").fetchone() == (0,)
        """,
    },
)
run_pytest(teardown, "-q", "--no-header", "-s", "--tb=line")

[<scratch>/teardown]$ pytest -q --no-header -s --tb=line
  
  setup: database ready
  .teardown: database closed
  
  setup: database ready
  Fteardown: database closed
  
  =================================== FAILURES ===================================
  E   assert (919,) == (0,)
        
        At index 0 diff: 919 != 0
        Use -v to get more diff
  <scratch>/teardown/tests/test_orders_db.py:22: assert (919,) == (0,)
  =========================== short test summary info ============================
  FAILED tests/test_orders_db.py::test_failing_test_still_tears_down - assert (...
  1 failed, 1 passed in <time>
  (exit code 1)


**`conftest.py`: sharing fixtures.** Fixtures defined in a `conftest.py` are
available to every test in that folder and below, **without importing** them.
Conftests nest: a fixture in `tests/api/conftest.py` overrides one with the same
name from `tests/conftest.py` for the tests under `tests/api/`. `pytest --fixtures`
lists every fixture a test could use, with its docstring and where it is defined.

In [164]:
conftest_demo = project(
    "conftest_demo",
    {
        "tests/conftest.py": """
            import pytest


            @pytest.fixture
            def currency():
                \"\"\"Default currency for tests.\"\"\"
                return "INR"


            @pytest.fixture
            def customer():
                \"\"\"A sample customer record.\"\"\"
                return {"name": "Asha", "city": "Pune"}
        """,
        "tests/test_shop.py": """
            def test_default_currency(currency, customer):
                assert (currency, customer["city"]) == ("INR", "Pune")
        """,
        "tests/api/conftest.py": """
            import pytest


            @pytest.fixture
            def currency():
                \"\"\"The public API reports prices in USD.\"\"\"
                return "USD"
        """,
        "tests/api/test_api.py": """
            def test_api_currency(currency, customer):
                assert (currency, customer["name"]) == ("USD", "Asha")
        """,
    },
)
run_pytest(conftest_demo, "-v", "--no-header")
run_pytest(
    conftest_demo,
    "--fixtures",
    "-q",
    "tests/api",
    keep=r"conftest|currency|customer|INR|USD|sample",
)

[<scratch>/conftest_demo]$ pytest -v --no-header
  ============================= test session starts ==============================
  collecting ... collected 2 items
  
  tests/api/test_api.py::test_api_currency PASSED                          [ 50%]
  tests/test_shop.py::test_default_currency PASSED                         [100%]
  
  ============================== 2 passed in <time> ===============================


[<scratch>/conftest_demo]$ pytest --fixtures -q tests/api
  ------------------------ fixtures defined from conftest ------------------------
  customer -- tests/conftest.py:10
      A sample customer record.
  currency -- tests/conftest.py:4
      Default currency for tests.
  currency -- tests/api/conftest.py:4
      The public API reports prices in USD.


**`autouse=True`** applies a fixture to every test in its scope without the tests
asking for it. Use it sparingly, for things every test needs and nobody should
forget: resetting global state, seeding `random`, blocking the network. Here it
seeds `random` before each test, so both tests see the same "random" order:

In [165]:
autouse_demo = project(
    "autouse_demo",
    {
        "tests/conftest.py": """
            import random

            import pytest


            @pytest.fixture(autouse=True)
            def seeded_random():
                random.seed(42)
        """,
        "tests/test_draw.py": """
            import random


            def test_lucky_draw():
                assert random.sample(["Asha", "Ravi", "Meera", "John"], 2) == ["Asha", "John"]


            def test_same_draw_again():
                assert random.randint(1, 100) == 82
        """,
    },
)
run_pytest(autouse_demo, "-q", "--no-header")

[<scratch>/autouse_demo]$ pytest -q --no-header
  ..                                                                       [100%]
  2 passed in <time>


Both tests hard-code "random" results and pass on every run: the autouse fixture
re-seeds `random` before *each* test, so the second test does not depend on how many
numbers the first one drew. The values are the same on every machine for a given
Python version (the Mersenne Twister algorithm is fixed; `sample`'s use of it could
change between versions).

**Factory fixtures.** When a test needs *several* objects, or needs to choose their
details, a fixture can return a **function** that builds them. The factory can track
what it made and clean it up after the test.

In [166]:
factory_demo = project(
    "factory_demo",
    {
        **shop_files("money"),
        "tests/test_factory.py": """
            import pytest

            from shop.money import Money


            @pytest.fixture
            def make_order():
                created = []

                def build(*prices, customer="Asha"):
                    order = {"id": len(created) + 1, "customer": customer,
                             "lines": [Money(price) for price in prices]}
                    created.append(order)
                    return order

                yield build
                print(f"\\nteardown: {len(created)} orders cleaned up")


            def test_two_orders(make_order):
                first = make_order("240", "199")
                second = make_order("49.50", customer="Ravi")
                assert (first["id"], second["id"]) == (1, 2)
                assert sum(first["lines"]) == Money("439")
        """,
    },
)
run_pytest(factory_demo, "-q", "--no-header", "-s")

[<scratch>/factory_demo]$ pytest -q --no-header -s
  .
  teardown: 2 orders cleaned up
  
  1 passed in <time>


**Built-in fixtures.** pytest ships fixtures for the most common needs:

| Fixture | Gives you | Typical use |
| --- | --- | --- |
| `tmp_path` | a fresh `pathlib.Path` folder per test (`tmp_path_factory` for wider scopes) | code that reads or writes files |
| `monkeypatch` | `setattr`, `setenv`, `delenv`, `setitem`, `chdir`, `syspath_prepend`, all undone after the test | replace a function, set an environment variable |
| `capsys` | `readouterr()` → `(out, err)` of what the test printed | testing CLI output |
| `caplog` | `records`, `record_tuples`, `messages`, `text`, `set_level`, `at_level` | asserting on log messages |

`tmp_path` gives each test its own empty folder under the system temp directory;
pytest keeps the last three runs for inspection.

In [167]:
tmp_path_demo = project(
    "tmp_path_demo",
    {
        "shop/report.py": """
            from pathlib import Path


            def write_report(folder: Path, totals: dict[str, int]) -> Path:
                path = folder / "report.csv"
                rows = "".join(f"{sku},{total}\\n" for sku, total in totals.items())
                path.write_text("sku,total\\n" + rows)
                return path
        """,
        "shop/__init__.py": "",
        "tests/test_report.py": """
            from shop.report import write_report


            def test_write_report(tmp_path):
                path = write_report(tmp_path, {"TEA-500": 480, "MUG-02": 199})
                assert path.read_text().splitlines() == ["sku,total", "TEA-500,480", "MUG-02,199"]
                assert [item.name for item in tmp_path.iterdir()] == ["report.csv"]
        """,
    },
)
run_pytest(tmp_path_demo, "-q", "--no-header")

[<scratch>/tmp_path_demo]$ pytest -q --no-header
  .                                                                        [100%]
  1 passed in <time>


`monkeypatch` changes something for one test and **undoes it automatically**:
environment variables, attributes of modules or objects, dict items, the working
folder. Here it sets `SHOP_CURRENCY` and replaces the slow `fetch_rate` function with
a lambda (note the *target*: the module where `price_in` looks the name up, more in
4.3.5).

In [168]:
monkeypatch_demo = project(
    "monkeypatch_demo",
    {
        "shop/__init__.py": "",
        "shop/pricing.py": """
            import os
            import time


            def fetch_rate(currency: str) -> float:
                time.sleep(5)  # pretend this calls a slow web service
                raise ConnectionError("no network in tests")


            def price_in(amount_in_inr: float) -> str:
                currency = os.environ.get("SHOP_CURRENCY", "INR")
                if currency == "INR":
                    return f"INR {amount_in_inr:.2f}"
                return f"{currency} {amount_in_inr * fetch_rate(currency):.2f}"
        """,
        "tests/test_pricing.py": """
            import os

            from shop import pricing


            def test_default_currency(monkeypatch):
                monkeypatch.delenv("SHOP_CURRENCY", raising=False)
                assert pricing.price_in(100) == "INR 100.00"


            def test_usd_with_fake_rate(monkeypatch):
                monkeypatch.setenv("SHOP_CURRENCY", "USD")
                monkeypatch.setattr(pricing, "fetch_rate", lambda currency: 0.012)
                assert pricing.price_in(100) == "USD 1.20"


            def test_patches_were_undone():
                assert "SHOP_CURRENCY" not in os.environ
                assert pricing.fetch_rate.__name__ == "fetch_rate"
        """,
    },
)
run_pytest(monkeypatch_demo, "-v", "--no-header")

[<scratch>/monkeypatch_demo]$ pytest -v --no-header
  ============================= test session starts ==============================
  collecting ... collected 3 items
  
  tests/test_pricing.py::test_default_currency PASSED                      [ 33%]
  tests/test_pricing.py::test_usd_with_fake_rate PASSED                    [ 66%]
  tests/test_pricing.py::test_patches_were_undone PASSED                   [100%]
  
  ============================== 3 passed in <time> ===============================


`capsys` captures what the code under test prints, so a CLI's output can be
asserted exactly. `readouterr()` returns everything printed since the last call and
resets the buffer. (`capfd` does the same at the file-descriptor level, for output
from C extensions and child processes.)

In [169]:
capsys_demo = project(
    "capsys_demo",
    {
        "tests/test_cli_output.py": """
            import sys


            def print_receipt(lines):
                for name, price in lines:
                    print(f"{name:<10}{price:>8.2f}")
                if not lines:
                    print("empty cart", file=sys.stderr)


            def test_receipt(capsys):
                print_receipt([("Tea", 240), ("Mug", 199)])
                captured = capsys.readouterr()
                assert captured.out == "Tea         240.00\\nMug         199.00\\n"
                assert captured.err == ""


            def test_empty_cart_warns_on_stderr(capsys):
                print_receipt([])
                assert capsys.readouterr() == ("", "empty cart\\n")
        """,
    },
)
run_pytest(capsys_demo, "-q", "--no-header")

[<scratch>/capsys_demo]$ pytest -q --no-header
  ..                                                                       [100%]
  2 passed in <time>


`caplog` captures log records (4.3.9) emitted during the test.
`record_tuples` gives `(logger name, level number, message)` triples, which are
easy to compare; `caplog.set_level(...)` or `with caplog.at_level(...)` lowers the
threshold so DEBUG and INFO records are captured too.

In [170]:
caplog_demo = project(
    "caplog_demo",
    {
        "tests/test_logging.py": """
            import logging

            logger = logging.getLogger("shop.orders")


            def cancel_order(order_id, paid):
                logger.info("cancelling order %s", order_id)
                if paid:
                    logger.warning("order %s was paid: refund needed", order_id)


            def test_paid_order_logs_a_warning(caplog):
                cancel_order(7, paid=True)
                assert caplog.record_tuples == [
                    ("shop.orders", logging.WARNING, "order 7 was paid: refund needed")
                ]


            def test_info_is_captured_when_enabled(caplog):
                with caplog.at_level(logging.INFO, logger="shop.orders"):
                    cancel_order(8, paid=False)
                assert caplog.messages == ["cancelling order 8"]
        """,
    },
)
run_pytest(caplog_demo, "-q", "--no-header")

[<scratch>/caplog_demo]$ pytest -q --no-header
  ..                                                                       [100%]
  2 passed in <time>


Why did the first test see only the warning? Its logger has no level, so the root
logger's default `WARNING` applies and the INFO record is never created (4.3.9).
`at_level` lowers it for the second test.

### 4.3.4 Parametrization and markers

**`@pytest.mark.parametrize`** runs one test function once per case, like Jest's
`test.each`. Each case becomes a separate test with its own id, pass/fail and
`-k` selection. Give **ids** so failures read well: with `ids=[...]`, with
`pytest.param(..., id="...")`, or with an `ids=` function. Without them pytest
builds ids from the values (`0.125-0.13`) or, for objects, from the parameter names
(`amount0`).

In [171]:
parametrize_demo = project(
    "parametrize_demo",
    {
        **shop_files("money"),
        "tests/test_money_rounding.py": """
            from decimal import Decimal

            import pytest

            from shop.money import Money


            @pytest.mark.parametrize(
                ("raw_amount", "expected"),
                [("0.125", "0.13"), ("0.124", "0.12"), ("-0.125", "-0.13"), ("7", "7.00")],
                ids=["half-up", "below-half", "negative", "whole-rupees"],
            )
            def test_rounds_half_up_to_paise(raw_amount, expected):
                assert Money(raw_amount).amount == Decimal(expected)


            @pytest.mark.parametrize(
                ("left", "right", "expected"),
                [
                    pytest.param(Money("240"), Money("199"), Money("439"), id="rupees"),
                    pytest.param(Money("0.10"), Money("0.20"), Money("0.30"), id="paise-exact"),
                    (Money("1"), Money("-1"), Money("0")),
                ],
            )
            def test_addition(left, right, expected):
                assert left + right == expected
        """,
    },
)
run_pytest(parametrize_demo, "-v", "--no-header")

[<scratch>/parametrize_demo]$ pytest -v --no-header
  ============================= test session starts ==============================
  collecting ... collected 7 items
  
  tests/test_money_rounding.py::test_rounds_half_up_to_paise[half-up] PASSED [ 14%]
  tests/test_money_rounding.py::test_rounds_half_up_to_paise[below-half] PASSED [ 28%]
  tests/test_money_rounding.py::test_rounds_half_up_to_paise[negative] PASSED [ 42%]
  tests/test_money_rounding.py::test_rounds_half_up_to_paise[whole-rupees] PASSED [ 57%]
  tests/test_money_rounding.py::test_addition[rupees] PASSED               [ 71%]
  tests/test_money_rounding.py::test_addition[paise-exact] PASSED          [ 85%]
  tests/test_money_rounding.py::test_addition[left2-right2-expected2] PASSED [100%]
  
  ============================== 7 passed in <time> ===============================


Two more forms: an **`ids` function** that turns each value into an id, and
**stacked** `parametrize` decorators, which run the cartesian product (here 3
currencies × 2 amounts = 6 tests). `-k` can then pick cases by id.

In [172]:
stacked_demo = project(
    "stacked_demo",
    {
        **shop_files("money"),
        "tests/test_currencies.py": """
            import pytest

            from shop.money import Money


            @pytest.mark.parametrize("currency", ["INR", "USD", "EUR"])
            @pytest.mark.parametrize(
                "amount", ["0.01", "999999.99"], ids=lambda value: f"amt{value}"
            )
            def test_round_trip_through_str(amount, currency):
                assert str(Money(amount, currency)).startswith(currency + " ")
        """,
    },
)
run_pytest(stacked_demo, "-v", "--no-header")
run_pytest(stacked_demo, "-v", "--no-header", "-k", "USD and amt0.01")

[<scratch>/stacked_demo]$ pytest -v --no-header
  ============================= test session starts ==============================
  collecting ... collected 6 items
  
  tests/test_currencies.py::test_round_trip_through_str[amt0.01-INR] PASSED [ 16%]
  tests/test_currencies.py::test_round_trip_through_str[amt0.01-USD] PASSED [ 33%]
  tests/test_currencies.py::test_round_trip_through_str[amt0.01-EUR] PASSED [ 50%]
  tests/test_currencies.py::test_round_trip_through_str[amt999999.99-INR] PASSED [ 66%]
  tests/test_currencies.py::test_round_trip_through_str[amt999999.99-USD] PASSED [ 83%]
  tests/test_currencies.py::test_round_trip_through_str[amt999999.99-EUR] PASSED [100%]
  
  ============================== 6 passed in <time> ===============================


[<scratch>/stacked_demo]$ pytest -v --no-header -k 'USD and amt0.01'
  ============================= test session starts ==============================
  collecting ... collected 6 items / 5 deselected / 1 selected
  
  tests/test_currencies.py::test_round_trip_through_str[amt0.01-USD] PASSED [100%]
  
  ======================= 1 passed, 5 deselected in <time> ========================


**Built-in markers** change how a test runs:

| Marker | Effect | Use for |
| --- | --- | --- |
| `@pytest.mark.skip(reason=...)` | never runs; reported as `s` | a test that cannot run yet |
| `pytest.skip("...")` inside a test | stops it as skipped at run time | conditions known only at run time |
| `@pytest.mark.skipif(condition, reason=...)` | skips when the condition is true | OS, Python version, missing optional package |
| `pytest.importorskip("module")` | skips the test or module when an import fails | optional dependencies |
| `@pytest.mark.xfail(reason=..., strict=True, raises=...)` | expected failure: `x` when it fails, `X` (XPASS) when it passes; `strict=True` turns XPASS into a failure | known bugs with a ticket |

`-ra` prints the reason for every test that did not simply pass:

In [173]:
markers_demo = project(
    "markers_demo",
    {
        "tests/test_markers.py": """
            import sys

            import pytest


            @pytest.mark.skip(reason="payment sandbox not set up yet")
            def test_card_payment():
                ...


            @pytest.mark.skipif(sys.platform == "win32", reason="POSIX file permissions")
            def test_report_is_private():
                assert True


            @pytest.mark.skipif(sys.version_info < (3, 14), reason="needs uuid7 (3.14+)")
            def test_order_ids_are_time_ordered():
                import uuid
                assert uuid.uuid7().version == 7


            def test_needs_optional_package():
                pytest.importorskip("shop_pdf_export", reason="optional PDF plug-in not installed")


            @pytest.mark.xfail(reason="BUG-142: float rounding", raises=AssertionError)
            def test_known_bug():
                assert round(2.675, 2) == 2.68


            @pytest.mark.xfail(reason="fixed already?", strict=True)
            def test_fixed_bug():
                assert round(2.5) == 2
        """,
    },
)
run_pytest(markers_demo, "-ra", "-q", "--no-header")

[<scratch>/markers_demo]$ pytest -ra -q --no-header
  s..sxF                                                                   [100%]
  =================================== FAILURES ===================================
  ________________________________ test_fixed_bug ________________________________
  [XPASS(strict)] fixed already?
  =========================== short test summary info ============================
  SKIPPED [1] tests/test_markers.py:6: payment sandbox not set up yet
  SKIPPED [1] tests/test_markers.py:23: optional PDF plug-in not installed
  XFAIL tests/test_markers.py::test_known_bug - BUG-142: float rounding
  FAILED tests/test_markers.py::test_fixed_bug - [XPASS(strict)] fixed already?
  1 failed, 2 passed, 2 skipped, 1 xfailed in <time>
  (exit code 1)


The last test passed although it was marked `xfail(strict=True)`, so pytest reports
it as a failure ("XPASS(strict)"): the bug it documented is gone and the marker
should be removed. (`round(2.5) == 2` because Python rounds halves to even, Unit
1.3.)

**Custom markers** label tests so you can select them with `-m`, like tags.
Register them in the config (`markers = [...]`) and run with `--strict-markers`, so
a typo such as `@pytest.mark.solw` is an error instead of a silently ignored label.

In [174]:
custom_markers = project(
    "custom_markers",
    {
        "pyproject.toml": """
            [tool.pytest.ini_options]
            pythonpath = ["."]
            testpaths = ["tests"]
            addopts = "--strict-markers"
            markers = [
                "slow: takes more than a second",
                "integration: needs a real database",
            ]
        """,
        "tests/test_suite.py": """
            import pytest


            def test_fast_rule():
                assert 1 + 1 == 2


            @pytest.mark.slow
            def test_full_catalogue_import():
                assert sum(range(10_000)) == 49_995_000


            @pytest.mark.integration
            @pytest.mark.slow
            def test_orders_table():
                assert True
        """,
    },
)
run_pytest(custom_markers, "-m", "not slow", "-q", "--no-header")
run_pytest(custom_markers, "-m", "integration or not slow", "-v", "--no-header")
run_pytest(custom_markers, "--markers", keep=r"^@pytest\.mark\.(slow|integration)")

[<scratch>/custom_markers]$ pytest -m 'not slow' -q --no-header
  .                                                                        [100%]
  1 passed, 2 deselected in <time>


[<scratch>/custom_markers]$ pytest -m 'integration or not slow' -v --no-header
  ============================= test session starts ==============================
  collecting ... collected 3 items / 1 deselected / 2 selected
  
  tests/test_suite.py::test_fast_rule PASSED                               [ 50%]
  tests/test_suite.py::test_orders_table PASSED                            [100%]
  
  ======================= 2 passed, 1 deselected in <time> ========================
[<scratch>/custom_markers]$ pytest --markers
  @pytest.mark.slow: takes more than a second
  @pytest.mark.integration: needs a real database


With `--strict-markers`, a misspelt marker stops collection with a clear error:

In [175]:
(custom_markers / "tests" / "test_typo.py").write_text(
    "import pytest\n\n\n@pytest.mark.solw\ndef test_typo():\n    assert True\n"
)
run_pytest(custom_markers, "-q", "--no-header", keep=r"solw|error|Error")

[<scratch>/custom_markers]$ pytest -q --no-header
  'solw' not found in `markers` configuration option
  ERROR tests/test_typo.py - Failed: 'solw' not found in `markers` configuratio...
  !!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
  1 error in <time>
  (exit code 2)


### 4.3.5 Test doubles

A **test double** stands in for a real collaborator (an HTTP API, a mailer, the
clock, a database) so a unit test is fast, deterministic and focused. The names
come from Gerard Meszaros's *xUnit Test Patterns*:

| Double | What it is | Checks | Example |
| --- | --- | --- | --- |
| Dummy | passed but never used | nothing | `None` for an unused logger argument |
| Stub | returns canned answers | the **result** of the code under test | `rates.get` always returns `83.12` |
| Fake | a working, simplified implementation | the result (state) | `InMemoryInventoryRepository` instead of PostgreSQL |
| Spy | a real or stub object that records calls | calls, after the fact | `Mock(wraps=real_mailer)` |
| Mock | pre-programmed with expected calls, verified | the **interaction** | "`send` was called once with this address" |

In Python, `unittest.mock` provides one flexible class (`Mock`) that serves as stub,
spy or mock depending on how you use it, much like `jest.fn()`. Fakes you write
yourself. The rows below each get a demo.

| API | What it does |
| --- | --- |
| `Mock()` | any attribute or call returns another `Mock`; records every call |
| `MagicMock()` | `Mock` plus the magic methods (`len`, iteration, `with`, `[]`) |
| `AsyncMock()` | calling it returns an awaitable (*New in 3.8*) |
| `side_effect` | raise an exception, return successive values, or call a function |
| call assertions | `assert_called_once_with`, `assert_any_call`, `assert_has_calls`, `call_args_list`, `call_count`, `assert_not_called` |
| `patch(target)` | replace a name **where it is looked up**, then restore it |
| `autospec` | the double has the real object's signature and attributes |

#### `Mock`

Every attribute you touch and every call you make on a `Mock` returns another
`Mock`, and every call is recorded. `return_value` makes it a stub. The danger:
a misspelt method name "works" too, so a test can pass while checking nothing.

In [176]:
mailer = mock.Mock()
mailer.send.return_value = "queued"

result = mailer.send("asha@example.com", subject="Order 7 shipped")
print(f"{result=}")
print(f"{mailer.send.call_args=}")
print(f"{mailer.send.call_count=}, {mailer.method_calls=}")
print(f"{type(mailer.sned('typo')).__name__=}  (a typo silently returns a Mock)")
print(raised_message(lambda: mailer.assert_called_once()))
print(raised_message(lambda: mailer.asert_called_once()))  # misspelt assert*: caught

result='queued'
mailer.send.call_args=call('asha@example.com', subject='Order 7 shipped')
mailer.send.call_count=1, mailer.method_calls=[call.send('asha@example.com', subject='Order 7 shipped')]
type(mailer.sned('typo')).__name__='Mock'  (a typo silently returns a Mock)
AssertionError: Expected 'mock' to have been called once. Called 0 times.
Calls: [call.send('asha@example.com', subject='Order 7 shipped'), call.sned('typo')].
AttributeError: 'asert_called_once' is not a valid assertion. Use a spec for the mock if 'asert_called_once' is meant to be an attribute


**Predict the output.**

```python
from unittest import mock

client = mock.Mock()
print(client.get is client.get, client() is client(), client.get() is client.post())
```

<details><summary>Answer</summary>

`True True False`. A child attribute is created on first access and then cached, so
`client.get` is the same object every time. Every call returns the mock's single
`return_value` (itself a `Mock`, created on the first call), so `client()` is always
the same object. `client.get()` and `client.post()` are calls on two different
children, each with its own `return_value`.

</details>

Run the next cell to check your prediction.

In [177]:
client = mock.Mock()
print(client.get is client.get, client() is client(), client.get() is client.post())

True True False


`Mock` protects only against misspelt `assert…` names (`asert`, `assret`,
`assrt`…), which raise `AttributeError` since Python 3.5; any other typo goes
unnoticed. `autospec` (below) fixes that properly.

#### `MagicMock`

`MagicMock` also configures the **magic methods**, so it can stand in for a
container, an iterator or a context manager. Set their return values through the
dunder names.

In [178]:
connection = mock.MagicMock()
connection.__len__.return_value = 3
connection.__iter__.return_value = iter(["TEA-500", "MUG-02", "JAR-750"])
connection.__getitem__.side_effect = lambda sku: {"TEA-500": 240}.get(sku, 0)
connection.__enter__.return_value = connection

with connection as open_connection:
    print(f"{len(open_connection)=}, {list(open_connection)=}, {open_connection['TEA-500']=}")
print(f"{connection.__exit__.called=}, {bool(connection)=}")
print(raised_message(lambda: len(mock.Mock())))

len(open_connection)=3, list(open_connection)=['TEA-500', 'MUG-02', 'JAR-750'], open_connection['TEA-500']=240
connection.__exit__.called=True, bool(connection)=True
TypeError: object of type 'Mock' has no len()


#### `AsyncMock`

For `async def` collaborators (Phase 5–6: HTTP clients, database drivers), calling
an `AsyncMock` returns a coroutine, and there are await-specific assertions
(`assert_awaited_once_with`). `patch` creates an `AsyncMock` automatically when the
target is an async function. The kernel already runs an event loop, so the cell can
use `await` at top level.

In [179]:
async def notify_customer(client, order_id: int) -> str:
    """Post a notification and return its status."""
    response = await client.post("/notifications", json={"order_id": order_id})
    return response["status"]


http_client = mock.AsyncMock()
http_client.post.return_value = {"status": "sent"}
status = await notify_customer(http_client, 7)
print(f"{status=}")
http_client.post.assert_awaited_once_with("/notifications", json={"order_id": 7})
print(f"{http_client.post.await_count=}, {inspect.iscoroutinefunction(http_client.post)=}")

status='sent'
http_client.post.await_count=1, inspect.iscoroutinefunction(http_client.post)=True


#### `side_effect`

`side_effect` overrides `return_value` and has three forms: an **exception**
(class or instance) to raise, an **iterable** of results returned one per call
(exceptions in it are raised), or a **function** called with the same arguments,
whose result is returned. The iterable form is how you test retry logic.

In [180]:
failing = mock.Mock(side_effect=ConnectionError("rates service down"))
print(raised_message(failing))

flaky = mock.Mock(side_effect=[TimeoutError("slow"), TimeoutError("slow"), 83.12])
for attempt in range(1, 4):
    print(f"attempt {attempt}: {raised_message(flaky) if attempt < 3 else flaky()}")
print(raised_message(flaky))  # the iterable is used up

lookup = mock.Mock(side_effect=lambda currency: {"USD": 83.12, "EUR": 90.4}[currency])
print(f"{lookup('EUR')=}, {raised_message(lambda: lookup('JPY'))}")

ConnectionError: rates service down
attempt 1: TimeoutError: slow
attempt 2: TimeoutError: slow
attempt 3: 83.12
StopIteration: 
lookup('EUR')=90.4, KeyError: 'JPY'


#### Call assertions

A mock records every call in `call_args_list` (and every call on it *and its
children* in `mock_calls`). The assertion helpers compare against `mock.call(...)`
objects; `mock.ANY` matches anything. `reset_mock()` clears the record between
phases of a test.

In [181]:
audit = mock.Mock()
audit.log("order created", order_id=7)
audit.log("payment received", order_id=7, amount=679)
audit.flush()

audit.log.assert_called_with("payment received", order_id=7, amount=679)  # the last call
audit.log.assert_any_call("order created", order_id=7)
audit.log.assert_has_calls(
    [
        mock.call("order created", order_id=7),
        mock.call("payment received", order_id=7, amount=mock.ANY),
    ]
)
print(f"{audit.log.call_count=}")
print(f"{audit.log.call_args_list=}")
print(f"{audit.mock_calls=}")
print(raised_message(lambda: audit.log.assert_called_once_with("order created", order_id=7)))
audit.reset_mock()
audit.log.assert_not_called()
print(f"after reset: {audit.log.call_count=}")

audit.log.call_count=2
audit.log.call_args_list=[call('order created', order_id=7),
 call('payment received', order_id=7, amount=679)]
audit.mock_calls=[call.log('order created', order_id=7),
 call.log('payment received', order_id=7, amount=679),
 call.flush()]
AssertionError: Expected 'log' to be called once. Called 2 times.
Calls: [call('order created', order_id=7),
 call('payment received', order_id=7, amount=679)].
after reset: audit.log.call_count=0


#### `patch`: replace a name where it is looked up

`mock.patch("package.module.name")` replaces an attribute for the duration of a
`with` block, a decorated function or a test, and then restores it. The rule that
trips everyone up: **patch the name in the module that uses it**, not where it is
defined. `from shop_rates import fetch_rate` copies the reference into
`shop_checkout`'s namespace at import time, so patching `shop_rates.fetch_rate`
afterwards changes a name `shop_checkout` never looks at again.

The two modules are created in memory for the demo (`types.ModuleType` registered in
`sys.modules`) and removed afterwards.

In [182]:
rates_module = types.ModuleType("shop_rates")
exec(
    "def fetch_rate(currency):\n    raise ConnectionError('real network call')\n",
    rates_module.__dict__,
)
sys.modules["shop_rates"] = rates_module

checkout_module = types.ModuleType("shop_checkout")
exec(
    "from shop_rates import fetch_rate\n\n"
    "def total_in(currency, amount_in_inr):\n"
    "    return round(amount_in_inr / fetch_rate(currency), 2)\n",
    checkout_module.__dict__,
)
sys.modules["shop_checkout"] = checkout_module

with mock.patch("shop_rates.fetch_rate", return_value=83.0):
    print(f"patched where defined: {raised_message(lambda: checkout_module.total_in('USD', 830))}")
with mock.patch("shop_checkout.fetch_rate", return_value=83.0) as fake_rate:
    print(f"patched where used:    {checkout_module.total_in('USD', 830)=}")
    fake_rate.assert_called_once_with("USD")
print(f"restored: {checkout_module.fetch_rate is rates_module.fetch_rate=}")

patched where defined: ConnectionError: real network call
patched where used:    checkout_module.total_in('USD', 830)=10.0
restored: checkout_module.fetch_rate is rates_module.fetch_rate=True


`patch` has several forms: a **decorator** (the mock arrives as an extra argument),
`patch.object(obj, "attribute")` when you hold the object, and
`patch.dict(mapping, values, clear=...)` for dicts such as `os.environ`. All of
them restore the original, even when the code raises.

In [183]:
@mock.patch("shop_checkout.fetch_rate", return_value=90.0)
def check_eur_total(fake_rate: mock.Mock) -> float:
    """A decorated function receives the mock as its last argument."""
    return checkout_module.total_in("EUR", 900)


print(f"{check_eur_total()=}")
with mock.patch.object(checkout_module, "fetch_rate", side_effect=TimeoutError("slow")):
    print(raised_message(lambda: checkout_module.total_in("USD", 100)))
with mock.patch.dict(os.environ, {"SHOP_CURRENCY": "USD"}):
    print(f"inside:  {os.environ['SHOP_CURRENCY']=}")
print(f"outside: {os.environ.get('SHOP_CURRENCY')=}")
for module_name in ("shop_rates", "shop_checkout"):
    del sys.modules[module_name]

check_eur_total()=10.0
TimeoutError: slow
inside:  os.environ['SHOP_CURRENCY']='USD'
outside: os.environ.get('SHOP_CURRENCY')=None


#### `autospec`

A plain `Mock` accepts any arguments, so a test keeps passing after the real
signature changes. `mock.create_autospec(real)` (or `patch(..., autospec=True)`)
builds a double with the **real signature and attributes**: wrong arguments raise
`TypeError`, unknown attributes raise `AttributeError`. `spec_set=True` also refuses
setting attributes the real object does not have. Prefer autospec for every mock of
your own code.

In [184]:
class Mailer:
    """The real collaborator."""

    def send(self, to: str, subject: str, *, body: str = "") -> str:
        raise ConnectionError("SMTP server not available in tests")


loose = mock.Mock()
loose.send("asha@example.com")  # missing subject: accepted
loose.sendd("typo")  # misspelt method: accepted

strict = mock.create_autospec(Mailer, instance=True)
print(raised_message(lambda: strict.send("asha@example.com")))
print(raised_message(lambda: strict.sendd("typo")))
strict.send("asha@example.com", "Order shipped")
strict.send.assert_called_once_with("asha@example.com", subject="Order shipped")
print("assert_called_once_with matched a positional call with a keyword expectation")

TypeError: missing a required argument: 'subject'
AttributeError: Mock object has no attribute 'sendd'
assert_called_once_with matched a positional call with a keyword expectation


The last line shows a second benefit: an autospecced mock **binds calls to the real
signature**, so `send("a", "b")` and `send("a", subject="b")` count as the same
call.

#### Fakes vs stubs vs mocks, side by side

One service, three tests. `PriceAlerter` checks a product's price and e-mails
subscribers when it drops. The **stub** gives a canned price; the **fake** is a
working in-memory price store; the **mock** verifies the e-mail interaction.
Prefer stubs and fakes (they test *outcomes*) and use mocks for interactions that
*are* the outcome, such as "an e-mail was sent".

In [185]:
class InMemoryPriceStore:
    """A fake: a real, simple implementation of the price-store interface."""

    def __init__(self) -> None:
        self._prices: dict[str, int] = {}

    def save(self, sku: str, price: int) -> None:
        self._prices[sku] = price

    def latest(self, sku: str) -> int:
        return self._prices[sku]


class PriceAlerter:
    def __init__(self, store, mailer) -> None:
        self.store, self.mailer = store, mailer

    def check(self, sku: str, old_price: int, subscribers: list[str]) -> bool:
        new_price = self.store.latest(sku)
        if new_price >= old_price:
            return False
        for address in subscribers:
            self.mailer.send(address, f"{sku} is now {new_price}")
        return True


stub_store = mock.Mock(latest=mock.Mock(return_value=199))  # stub
print(f"stub:  {PriceAlerter(stub_store, mock.Mock()).check('MUG-02', 249, [])=}")

fake_store = InMemoryPriceStore()  # fake
fake_store.save("MUG-02", 259)
print(f"fake:  {PriceAlerter(fake_store, mock.Mock()).check('MUG-02', 249, [])=}")

mock_mailer = mock.create_autospec(Mailer, instance=True)  # mock
PriceAlerter(stub_store, mock_mailer).check("MUG-02", 249, ["asha@example.com"])
mock_mailer.send.assert_called_once_with("asha@example.com", "MUG-02 is now 199")
print(f"mock:  {mock_mailer.send.call_args=}")

stub:  PriceAlerter(stub_store, mock.Mock()).check('MUG-02', 249, [])=True
fake:  PriceAlerter(fake_store, mock.Mock()).check('MUG-02', 249, [])=False
mock:  mock_mailer.send.call_args=call('asha@example.com', 'MUG-02 is now 199')


#### Freezing time

Code that calls `datetime.now()` or `time.time()` gives a different answer every
run. The options, best first:

1. **Inject a clock**: take `clock: Callable[[], datetime] = lambda: datetime.now(UTC)`
   as a parameter (the Phase 2 `InventoryService` does this). Tests pass a lambda.
2. **Patch the name in your module** with `monkeypatch` or `mock.patch`, for
   example replace `shop.hours.datetime` with a subclass whose `now()` is fixed.
3. **A library** such as `time-machine` or `freezegun` (third party, not installed
   here) patches the clock functions everywhere.

You cannot patch `datetime.datetime.now` itself: `datetime` is a built-in
(immutable) type, so assigning to its attributes raises `TypeError`.

In [186]:
def is_happy_hour(clock: Callable[[], datetime] = lambda: datetime.now(UTC)) -> bool:
    """Happy hour is 16:00-18:00 IST, which is 10:30-12:30 UTC."""
    now = clock()
    return (10, 30) <= (now.hour, now.minute) < (12, 30)


print(f"{is_happy_hour(lambda: datetime(2026, 11, 7, 11, 0, tzinfo=UTC))=}")
print(f"{is_happy_hour(lambda: datetime(2026, 11, 7, 13, 0, tzinfo=UTC))=}")
print(raised_message(lambda: setattr(datetime, "now", lambda tz=None: None)))

is_happy_hour(lambda: datetime(2026, 11, 7, 11, 0, tzinfo=UTC))=True
is_happy_hour(lambda: datetime(2026, 11, 7, 13, 0, tzinfo=UTC))=False
TypeError: cannot set 'now' attribute of immutable type 'datetime.datetime'


Option 2 in a test file: the module does `from datetime import datetime`, so the
test patches **`shop.hours.datetime`** (where the name is looked up) with a
subclass that overrides the `now` classmethod.

In [187]:
frozen_time = project(
    "frozen_time",
    {
        "shop/__init__.py": "",
        "shop/hours.py": """
            from datetime import UTC, datetime


            def greeting() -> str:
                hour = datetime.now(UTC).hour
                return "Good morning" if hour < 12 else "Good afternoon"
        """,
        "tests/test_hours.py": """
            from datetime import UTC, datetime

            from shop import hours


            class MorningClock(datetime):
                @classmethod
                def now(cls, tz=None):
                    return datetime(2026, 11, 7, 9, 30, tzinfo=tz or UTC)


            def test_morning_greeting(monkeypatch):
                monkeypatch.setattr(hours, "datetime", MorningClock)
                assert hours.greeting() == "Good morning"
        """,
    },
)
run_pytest(frozen_time, "-q", "--no-header")

[<scratch>/frozen_time]$ pytest -q --no-header
  .                                                                        [100%]
  1 passed in <time>


### 4.3.6 Coverage

**Coverage** measures which code your tests executed. `pytest-cov` runs
`coverage.py` around pytest: `--cov=shop` selects what to measure,
`--cov-report=term-missing` lists the line numbers never run.

| Kind | Counts | Misses |
| --- | --- | --- |
| **Line** (statement) coverage | each line executed at least once | an `if` with no `else` whose condition was never false |
| **Branch** coverage (`--cov-branch`) | each possible jump taken (both outcomes of every `if`, loop entered and skipped) | fewer things: it is the stricter measure |

One test for a member discount executes every line, so line coverage says 100%.
The non-member path (the `if` being false) never ran, and branch coverage reports
it as the partial branch `2->4` (line 2 never jumped straight to line 4):

In [188]:
if importlib.util.find_spec("pytest_cov") is None:
    print("SKIP: pytest-cov is not installed (uv add --dev pytest-cov)")
else:
    coverage_demo = project(
        "coverage_demo",
        {
            "shop/__init__.py": "",
            "shop/discounts.py": """
                def member_price(total_in_paise: int, is_member: bool) -> int:
                    if is_member:
                        total_in_paise = total_in_paise * 90 // 100
                    return total_in_paise
            """,
            "tests/test_discounts.py": """
                from shop.discounts import member_price


                def test_member_gets_ten_percent_off():
                    assert member_price(10_000, is_member=True) == 9_000
            """,
        },
    )
    run_pytest(coverage_demo, "-q", "--no-header", "--cov=shop", "--cov-report=term-missing")
    run_pytest(
        coverage_demo,
        "-q",
        "--no-header",
        "--cov=shop",
        "--cov-branch",
        "--cov-report=term-missing",
    )

[<scratch>/coverage_demo]$ pytest -q --no-header --cov=shop --cov-report=term-missing
  .                                                                        [100%]
  ================================ tests coverage ================================
  _______________ coverage: platform linux, python 3.15.1-final-0 ________________
  
  Name                Stmts   Miss  Cover   Missing
  -------------------------------------------------
  shop/__init__.py        0      0   100%
  shop/discounts.py       4      0   100%
  -------------------------------------------------
  TOTAL                   4      0   100%
  1 passed in <time>


[<scratch>/coverage_demo]$ pytest -q --no-header --cov=shop --cov-branch --cov-report=term-missing
  .                                                                        [100%]
  ================================ tests coverage ================================
  _______________ coverage: platform linux, python 3.15.1-final-0 ________________
  
  Name                Stmts   Miss Branch BrPart  Cover   Missing
  ---------------------------------------------------------------
  shop/__init__.py        0      0      0      0   100%
  shop/discounts.py       4      0      2      1    83%   2->4
  ---------------------------------------------------------------
  TOTAL                   4      0      2      1    83%
  1 passed in <time>


**Thresholds.** `--cov-fail-under=90` (or `fail_under` in the config) makes the run
fail when total coverage drops below the number, which is how CI enforces it. Keep
the settings in `pyproject.toml` so every developer and CI measure the same way:
`[tool.coverage.run]` (`branch`, `source`) and `[tool.coverage.report]`
(`fail_under`, `show_missing`, `exclude_also` for lines that should never count,
such as `if TYPE_CHECKING:`). A `# pragma: no cover` comment excludes one line or
block.

In [189]:
if importlib.util.find_spec("pytest_cov") is not None:
    (coverage_demo / "pyproject.toml").write_text(
        textwrap.dedent("""
        [tool.pytest.ini_options]
        pythonpath = ["."]
        testpaths = ["tests"]
        addopts = "--cov=shop --cov-report=term-missing"

        [tool.coverage.run]
        branch = true

        [tool.coverage.report]
        fail_under = 90
        exclude_also = ["if TYPE_CHECKING:", "raise NotImplementedError"]
    """)
    )
    run_pytest(coverage_demo, "-q", "--no-header")  # the test passes, the run fails

[<scratch>/coverage_demo]$ pytest -q --no-header
  .
  ERROR: Coverage failure: total of 83 is less than fail-under=90
                                                                           [100%]
  ================================ tests coverage ================================
  _______________ coverage: platform linux, python 3.15.1-final-0 ________________
  
  Name                Stmts   Miss Branch BrPart  Cover   Missing
  ---------------------------------------------------------------
  shop/__init__.py        0      0      0      0   100%
  shop/discounts.py       4      0      2      1    83%   2->4
  ---------------------------------------------------------------
  TOTAL                   4      0      2      1    83%
  FAIL Required test coverage of 90.0% not reached. Total coverage: 83.33%
  1 passed in <time>
  (exit code 1)


**The limits of coverage.** Coverage tells you what was **executed**, not what was
**checked**. A test without a meaningful assertion reaches 100% while the code is
wrong. It also cannot see code that *should* exist (an unhandled case), and one
input per branch says nothing about boundaries. Use coverage to find untested
code, not as proof of quality; a very high target tends to produce tests like the
one below. **Mutation testing** (tools such as `mutmut` or `cosmic-ray`) asks the
better question, "would a test fail if this line were wrong?", by planting small
bugs; Lab 4.3 checks your tests that way.

In [190]:
if importlib.util.find_spec("pytest_cov") is not None:
    useless = project(
        "useless_coverage",
        {
            "shop/__init__.py": "",
            "shop/gst.py": """
                def with_gst(amount_in_paise: int) -> int:
                    return amount_in_paise * 112 // 100  # bug: GST is 18%
            """,
            "tests/test_gst.py": """
                from shop.gst import with_gst


                def test_with_gst_runs():
                    with_gst(10_000)  # executes every line, checks nothing
            """,
        },
    )
    run_pytest(useless, "-q", "--no-header", "--cov=shop", "--cov-branch", "--cov-report=term")

[<scratch>/useless_coverage]$ pytest -q --no-header --cov=shop --cov-branch --cov-report=term
  .                                                                        [100%]
  ================================ tests coverage ================================
  _______________ coverage: platform linux, python 3.15.1-final-0 ________________
  
  Name               Stmts   Miss Branch BrPart  Cover
  ----------------------------------------------------
  shop/__init__.py       0      0      0      0   100%
  shop/gst.py            2      0      0      0   100%
  ----------------------------------------------------
  TOTAL                  2      0      0      0   100%
  1 passed in <time>


**Under the hood: how coverage sees lines.** Since Python 3.12 the interpreter
offers `sys.monitoring` (PEP 669): a tool registers for events such as `LINE` on
chosen code objects and pays nothing elsewhere. coverage.py uses it on 3.14+ (the
`SysMonitor` core); older versions used `sys.settrace`. A ten-line coverage tool
for one function:

In [191]:
def shipping_fee(total_in_paise: int, express: bool) -> int:
    if express:
        return 9_900
    if total_in_paise >= 50_000:
        return 0
    return 4_900


monitoring = sys.monitoring
TOOL = monitoring.COVERAGE_ID
executed_lines: set[int] = set()
monitoring.use_tool_id(TOOL, "mini-coverage")
monitoring.register_callback(
    TOOL, monitoring.events.LINE, lambda code, line: executed_lines.add(line)
)
monitoring.set_local_events(TOOL, shipping_fee.__code__, monitoring.events.LINE)
shipping_fee(60_000, express=False)
monitoring.set_local_events(TOOL, shipping_fee.__code__, 0)
monitoring.free_tool_id(TOOL)

first_line = shipping_fee.__code__.co_firstlineno
body_lines = range(first_line + 1, first_line + 6)
print(f"executed body lines: {sorted(line - first_line for line in executed_lines)}")
print(
    f"never executed:      "
    f"{[line - first_line for line in body_lines if line not in executed_lines]}"
)

executed body lines: [1, 3, 4]
never executed:      [2, 5]


coverage.py reports which measurement core it uses with `--debug=sys`:

In [192]:
if importlib.util.find_spec("pytest_cov") is not None:
    shell(
        "coverage",
        "run",
        "--debug=sys",
        "-m",
        "pytest",
        "-q",
        "-p",
        "no:cacheprovider",
        cwd=useless,
        keep=r"^\s+(core|coverage_version|CTracer):",
    )

[<scratch>/useless_coverage]$ coverage run --debug=sys -m pytest -q -p no:cacheprovider
                 coverage_version: 7.17.2
                             core: SysMonitor
                          CTracer: available from <repo>/.venv/lib/python3.15/site-packages/coverage/tracer.cpython-315-x86_64-linux-gnu.so


### 4.3.7 Property-based testing

An example-based test checks the cases you thought of. A **property-based** test
states a rule that must hold for *every* input ("the shares always add up to the
total") and lets a library generate hundreds of inputs, including the nasty ones
(empty lists, zero, huge numbers, odd Unicode). **Hypothesis** is the Python
library (fast-check is the JavaScript equivalent). Two ideas make it work:

1. **Strategies** describe how to generate values: `st.integers(min_value=1)`,
   `st.lists(st.text())`, and combinators to build your own types.
2. **Shrinking**: when a property fails, Hypothesis simplifies the failing input
   step by step and reports the **smallest** example that still fails.

| Strategy | Generates |
| --- | --- |
| `st.integers(min_value, max_value)`, `st.floats(allow_nan=False)`, `st.decimals(places=2)` | numbers |
| `st.text(alphabet, min_size, max_size)`, `st.from_regex(r"[A-Z]{3}-\d{3}", fullmatch=True)` | strings |
| `st.booleans()`, `st.none()`, `st.just(x)`, `st.sampled_from([...])` | fixed choices |
| `st.lists(s, min_size, max_size, unique)`, `st.tuples(a, b)`, `st.dictionaries(k, v)` | collections |
| `st.datetimes(timezones=st.timezones())`, `st.dates()` | dates and times |
| `st.builds(Cls, field=...)`, `st.from_type(Cls)` | your classes |
| `s.map(f)`, `s.filter(pred)`, `st.one_of(a, b)`, `@st.composite` | combinators |

A `@given` test is an ordinary function, so the kernel can call it. With
`derandomize=True` the "random" inputs are the same on every run, which keeps this
notebook's output stable. Note how Hypothesis starts with the simplest values:

In [193]:
if importlib.util.find_spec("hypothesis") is None:
    print("SKIP: hypothesis is not installed (uv add --dev hypothesis)")
else:
    from hypothesis import example, find, given, settings
    from hypothesis import strategies as st

    generated: list[list[int]] = []

    @settings(derandomize=True, max_examples=8, database=None)
    @given(st.lists(st.integers(min_value=-5, max_value=5), max_size=4))
    def record_examples(values: list[int]) -> None:
        generated.append(values)

    record_examples()
    print(*generated, sep="\n")

[]
[-3, 0, 2]
[-5, 4, 0, -3]
[4, -1, 3]
[-5]
[2, 2]
[3]
[5]


**Building strategies for your own data.** `@st.composite` receives a `draw`
function and can make later choices depend on earlier ones; `.map` transforms
values; `st.builds` calls a constructor with generated arguments. Here: order lines
with valid SKUs, positive quantities and prices in paise.

In [194]:
if importlib.util.find_spec("hypothesis") is not None:
    skus = st.from_regex(r"[A-Z]{3}-[0-9]{3}", fullmatch=True)

    @st.composite
    def order_lines(draw: st.DrawFn) -> tuple[str, int, Decimal]:
        sku = draw(skus)
        quantity = draw(st.integers(min_value=1, max_value=10))
        price = draw(
            st.integers(min_value=1, max_value=500_000).map(lambda paise: Decimal(paise) / 100)
        )
        return sku, quantity, price

    drawn_lines: list[tuple[str, int, Decimal]] = []

    @settings(derandomize=True, max_examples=5, database=None)
    @given(order_lines())
    def record_lines(line: tuple[str, int, Decimal]) -> None:
        drawn_lines.append(line)

    record_lines()
    print(*drawn_lines, sep="\n")

('AAA-000', 1, Decimal('0.01'))
('OIA-374', 3, Decimal('7.68'))
('OLH-262', 3, Decimal('228.39'))
('VUI-848', 1, Decimal('2730.58'))
('QTV-575', 6, Decimal('17.34'))


**Properties of `split_bill`**, run with pytest. A `conftest.py` registers a
Hypothesis *profile* (deterministic, no example database) so the demo is
reproducible; in a real project you would keep the database, which replays past
failures first. Each `@given` test runs 100 generated examples by default;
`--hypothesis-show-statistics` reports what happened.

In [195]:
HYPOTHESIS_CONFTEST = """
    from hypothesis import settings

    settings.register_profile("course", derandomize=True, database=None, deadline=None)
    settings.load_profile("course")
"""
SPLIT_BILL_PROPERTIES = """
    from decimal import Decimal

    from hypothesis import given
    from hypothesis import strategies as st

    from shop.billing import split_bill

    amounts = st.decimals(min_value=0, max_value=100_000, places=2).map(str)
    people = st.integers(min_value=1, max_value=50)


    @given(amounts, people)
    def test_shares_add_up_to_the_total(total_amount, people_count):
        assert sum(split_bill(total_amount, people_count)) == Decimal(total_amount)


    @given(amounts, people)
    def test_shares_differ_by_at_most_one_paisa(total_amount, people_count):
        shares = split_bill(total_amount, people_count)
        assert len(shares) == people_count
        assert max(shares) - min(shares) <= Decimal("0.01")
"""
if importlib.util.find_spec("hypothesis") is not None:
    properties = project(
        "properties",
        {
            **shop_files("billing"),
            "tests/conftest.py": HYPOTHESIS_CONFTEST,
            "tests/test_split_bill_properties.py": SPLIT_BILL_PROPERTIES,
        },
    )
    run_pytest(
        properties,
        "-q",
        "--no-header",
        "--hypothesis-show-statistics",
        keep=r"passed|test_|examples|Stopped",
    )

[<scratch>/properties]$ pytest -q --no-header --hypothesis-show-statistics
  tests/test_split_bill_properties.py::test_shares_add_up_to_the_total:
    - Stopped because settings.max_examples=100
  tests/test_split_bill_properties.py::test_shares_differ_by_at_most_one_paisa:
    - Stopped because settings.max_examples=100
  2 passed in <time>


**Shrinking in action.** A plausible-looking `split_bill` that divides with floats
and rounds each share. Hypothesis finds totals where the rounded shares no longer
add up, then shrinks the failing input to a minimal one: the smallest total and
number of people that still break the rule. That example is your new regression
test (pin it with `@example(...)`).

In [196]:
FLOAT_SPLIT_BILL = """
    from decimal import Decimal


    def split_bill(total_amount: str, people_count: int) -> list[Decimal]:
        share = round(float(total_amount) / people_count, 2)
        return [Decimal(str(share))] * people_count
"""
if importlib.util.find_spec("hypothesis") is not None:
    write_files(properties, {"shop/billing.py": FLOAT_SPLIT_BILL})
    run_pytest(
        properties,
        "-q",
        "--no-header",
        "--tb=short",
        keep=r"Falsifying|total_amount=|people_count=|^E  |passed|failed",
    )

[<scratch>/properties]$ pytest -q --no-header --tb=short
  E   AssertionError: assert Decimal('0.02') == Decimal('0.01')
  E    +  where Decimal('0.02') = sum([Decimal('0.01'), Decimal('0.01')])
  E    +    where [Decimal('0.01'), Decimal('0.01')] = split_bill('0.01', 2)
  E    +  and   Decimal('0.01') = Decimal('0.01')
  E   Failing test case: test_shares_add_up_to_the_total(
  E       total_amount='0.01',
  E       people_count=2,
  E   )
  1 failed, 1 passed in <time>
  (exit code 1)


`find(strategy, condition)` exposes shrinking directly: it searches for any value
that satisfies the condition, then shrinks it. The first list found with a sum
over 100 might be long and messy; the reported one is minimal. `assume(...)`
discards inputs inside a test, and `@example(...)` always runs a specific case in
addition to the generated ones.

In [197]:
if importlib.util.find_spec("hypothesis") is not None:
    quiet = settings(derandomize=True, database=None)
    print(f"{find(st.lists(st.integers()), lambda values: sum(values) > 100, settings=quiet)=}")
    print(f"{find(st.text(), lambda text: len(text.encode()) > len(text), settings=quiet)=}")
    print(f"{find(st.integers(), lambda number: number * number > 1_000, settings=quiet)=}")

    @settings(derandomize=True, database=None, max_examples=50)
    @example(total_amount="0.01", people_count=3)  # always tried first
    @given(st.decimals(min_value=0, max_value=1_000, places=2).map(str), st.integers(1, 9))
    def property_shares_are_never_negative(total_amount: str, people_count: int) -> None:
        total_in_paise = int(Decimal(total_amount) * 100)
        base, leftover = divmod(total_in_paise, people_count)
        assert base >= 0 and 0 <= leftover < people_count

    property_shares_are_never_negative()
    print("property held for the pinned example and 50 generated ones")

find(st.lists(st.integers()), lambda values: sum(values) > 100, settings=quiet)=[101]
find(st.text(), lambda text: len(text.encode()) > len(text), settings=quiet)='\x80'
find(st.integers(), lambda number: number * number > 1_000, settings=quiet)=32
property held for the pinned example and 50 generated ones


### 4.3.8 Debugging

**`breakpoint()`** (Python 3.7+) is Python's `debugger;` statement: it calls
`sys.breakpointhook()`, which by default starts **pdb**, the debugger in the
standard library, at that line. The environment variable `PYTHONBREAKPOINT`
chooses what happens instead:

| `PYTHONBREAKPOINT` | `breakpoint()` does |
| --- | --- |
| unset or empty | `pdb.set_trace()` |
| `0` | nothing (leave stray breakpoints in CI without hanging it) |
| `module.function` | imports and calls it, e.g. `ipdb.set_trace`, `pudb.set_trace`, `debugpy.breakpoint` |

pdb is interactive, and this notebook runs unattended, so the demos start pdb in a
child process and **feed it commands on standard input**. pdb does not echo
input, so `pdb_session` prints each command after its `(Pdb)` prompt, as you
would see it in a terminal.

In [198]:
def pdb_session(
    *command: str, commands: list[str], cwd: Path, env: dict[str, str] | None = None
) -> None:
    """Run command (which starts pdb) with commands on stdin; print the transcript."""
    completed = subprocess.run(
        [sys.executable if command[0] == "python" else str(VENV_BIN / command[0]), *command[1:]],
        cwd=cwd,
        env={**(env or tool_env()), "PYTHONUNBUFFERED": "1"},
        input="\n".join(commands) + "\n",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=120,
    )
    pieces = tidy(completed.stdout).split("(Pdb) ")
    transcript = [pieces[0].rstrip()] if pieces[0].strip() else []
    for index, piece in enumerate(pieces[1:]):
        transcript.append(f"(Pdb) {commands[index] if index < len(commands) else '<end of input>'}")
        if piece.strip():
            transcript.append(piece.rstrip())
    print(f"[{tidy(str(cwd))}]$ {shlex.join(command)}   # stdin: {commands}")
    print(textwrap.indent("\n".join(transcript), "  ", lambda line: True))


debug_demo = write_files(
    fresh_dir("debug_demo"),
    {
        "cart.py": """
        def line_total(price, quantity):
            subtotal = price * quantity
            breakpoint()
            return subtotal


        def cart_total(lines):
            total = 0
            for price, quantity in lines:
                total += line_total(price, quantity)
            return total


        print(cart_total([(240, 2), (12, "3")]))
    """,
    },
)
print("cart.py written; line 14 passes one quantity as a string")

cart.py written; line 14 passes one quantity as a string


The script stops at `breakpoint()` on every call of `line_total`. The session
prints the arguments, shows the call stack (`w`) and the whole function (`ll`),
steps one line (`n`), continues (`c`) to the second call, and finds the bug: the
quantity `"3"` is a string, so `12 * "3"` repeated the string instead of
multiplying, and adding that string to the running total fails.

In [199]:
pdb_session(
    "python",
    "cart.py",
    cwd=debug_demo,
    commands=[
        "p price, quantity",
        "w",
        "ll",
        "n",
        "p subtotal",
        "c",
        "p price, quantity",
        "p subtotal",
        "c",
    ],
)

[<scratch>/debug_demo]$ python cart.py   # stdin: ['p price, quantity', 'w', 'll', 'n', 'p subtotal', 'c', 'p price, quantity', 'p subtotal', 'c']
  > <scratch>/debug_demo/cart.py(3)line_total()
  -> breakpoint()
  (Pdb) p price, quantity
  (240, 2)
  (Pdb) w
    <scratch>/debug_demo/cart.py(14)<module>()
  -> print(cart_total([(240, 2), (12, "3")]))
    <scratch>/debug_demo/cart.py(10)cart_total()
  -> total += line_total(price, quantity)
  > <scratch>/debug_demo/cart.py(3)line_total()
  -> breakpoint()
  (Pdb) ll
    1  	def line_total(price, quantity):
    2  	    subtotal = price * quantity
    3  ->	    breakpoint()
    4  	    return subtotal
  (Pdb) n
  > <scratch>/debug_demo/cart.py(4)line_total()
  -> return subtotal
  (Pdb) p subtotal
  480
  (Pdb) c
  > <scratch>/debug_demo/cart.py(3)line_total()
  -> breakpoint()
  (Pdb) p price, quantity
  (12, '3')
  (Pdb) p subtotal
  '333333333333'
  (Pdb) c
  Traceback (most recent call last):
    File "<scratch>/debug_demo/cart.py", l

`PYTHONBREAKPOINT=0` disables every `breakpoint()` call, and
`PYTHONBREAKPOINT=module.function` replaces pdb with any callable. Here a small
hook prints the caller's local variables instead of stopping, a poor man's
logpoint:

In [200]:
write_files(
    debug_demo,
    {
        "show_locals.py": """
        import sys


        def report():
            caller = sys._getframe(1)
            print(f"breakpoint in {caller.f_code.co_name}: {caller.f_locals}")
    """,
    },
)
shell("python", "cart.py", cwd=debug_demo, env=tool_env(PYTHONBREAKPOINT="0"), keep=r"^\d|Error")
shell(
    "python",
    "cart.py",
    cwd=debug_demo,
    env=tool_env(PYTHONBREAKPOINT="show_locals.report", PYTHONPATH=str(debug_demo)),
    keep=r"breakpoint in|^\d|Error",
)

[<scratch>/debug_demo]$ python cart.py
  TypeError: unsupported operand type(s) for +=: 'int' and 'str'
  (exit code 1)
[<scratch>/debug_demo]$ python cart.py
  breakpoint in line_total: {'price': 240, 'quantity': 2, 'subtotal': 480}
  breakpoint in line_total: {'price': 12, 'quantity': '3', 'subtotal': '333333333333'}
  TypeError: unsupported operand type(s) for +=: 'int' and 'str'
  (exit code 1)


In the kernel the same mechanism is visible: `breakpoint()` simply calls
`sys.breakpointhook()`. The default hook (always kept in `sys.__breakpointhook__`)
reads `PYTHONBREAKPOINT` and normally calls `pdb.set_trace()`; tools can install
their own, and in Jupyter ipykernel may do so for its debugger, so the first value
below depends on the frontend. Swapping the hook temporarily makes `breakpoint()`
harmless here:

In [201]:
print(f"{sys.breakpointhook is sys.__breakpointhook__=}, {sys.__breakpointhook__!r}")
calls: list[str] = []
sys.breakpointhook = lambda *args, **kwargs: calls.append(sys._getframe(1).f_code.co_name)
try:

    def checkout() -> None:
        breakpoint()

    checkout()
finally:
    sys.breakpointhook = sys.__breakpointhook__
print(f"{calls=}")

sys.breakpointhook is sys.__breakpointhook__=False, <built-in function breakpointhook>
calls=['checkout']


**pdb commands.** Short forms in brackets; anything that is not a command is
evaluated as Python in the current frame (prefix with `!` when it clashes with a
command name, e.g. `!n = 3`).

| Command | Does | Command | Does |
| --- | --- | --- | --- |
| `n`(ext) | run the current line, stepping **over** calls | `s`(tep) | step **into** a call |
| `c`(ontinue) | run to the next breakpoint | `r`(eturn) | run until the current function returns |
| `unt`(il) `N` | run until line N (or past a loop) | `j`(ump) `N` | set the next line to run |
| `l`(ist), `ll` | source around the line / whole function | `w`(here), `bt` | the call stack |
| `u`(p), `d`(own) | move between stack frames | `p expr`, `pp expr` | print / pretty-print |
| `b`(reak) `file:line`, `b func` | set a breakpoint (`b` alone lists them) | `tbreak` | one-shot breakpoint |
| `b N, condition` / `condition N expr` | conditional breakpoint | `cl`(ear) `N` | delete breakpoint N |
| `display expr` | print expr whenever it changes | `interact` | a Python REPL in the current frame |
| `a`(rgs) | the current function's arguments | `q`(uit) | stop the program |

A session that never calls `breakpoint()`: `python -m pdb script.py` stops before
the first line, then sets a **conditional breakpoint** inside the loop, uses
`display`, steps **into** a call with `s`, returns with `r`, and moves **up** the
stack to inspect the caller.

In [202]:
write_files(
    debug_demo,
    {
        "invoice.py": """
        def gst(amount_in_paise):
            return amount_in_paise * 18 // 100


        def invoice_total(lines):
            total = 0
            for sku, amount in lines:
                total += amount + gst(amount)
            return total


        print(invoice_total([("TEA-500", 24000), ("MUG-02", 19900), ("JAR-750", 35000)]))
    """,
    },
)
pdb_session(
    "python",
    "-m",
    "pdb",
    "invoice.py",
    cwd=debug_demo,
    commands=[
        "b 8, sku == 'MUG-02'",
        "c",
        "display total",
        "a",
        "s",
        "ll",
        "r",
        "u",
        "p sku, amount",
        "d",
        "n",
        "cl 1",
        "c",
    ],
)

[<scratch>/debug_demo]$ python -m pdb invoice.py   # stdin: ["b 8, sku == 'MUG-02'", 'c', 'display total', 'a', 's', 'll', 'r', 'u', 'p sku, amount', 'd', 'n', 'cl 1', 'c']
  > <scratch>/debug_demo/invoice.py(1)<module>()
  -> def gst(amount_in_paise):
  (Pdb) b 8, sku == 'MUG-02'
  Breakpoint 1 at <scratch>/debug_demo/invoice.py:8
  (Pdb) c
  > <scratch>/debug_demo/invoice.py(8)invoice_total()
  -> total += amount + gst(amount)
  (Pdb) display total
  display total: 28320
  (Pdb) a
  lines = [('TEA-500', 24000), ('MUG-02', 19900), ('JAR-750', 35000)]
  (Pdb) s
  --Call--
  > <scratch>/debug_demo/invoice.py(1)gst()
  -> def gst(amount_in_paise):
  (Pdb) ll
    1  ->	def gst(amount_in_paise):
    2  	    return amount_in_paise * 18 // 100
  (Pdb) r
  --Return--
  > <scratch>/debug_demo/invoice.py(2)gst()->3582
  -> return amount_in_paise * 18 // 100
  (Pdb) u
  > <scratch>/debug_demo/invoice.py(8)invoice_total()
  -> total += amount + gst(amount)
  (Pdb) p sku, amount
  ('MUG-02', 19900

**Post-mortem debugging** opens the debugger at the frame where an exception was
raised, *after* the crash, with all local variables intact:

| Where | How |
| --- | --- |
| Any script | `python -m pdb script.py`: after an uncaught exception pdb enters post-mortem mode |
| REPL / your own code | `pdb.pm()` (uses `sys.last_exc`, set by the REPL after an error) or `pdb.post_mortem(exception)` |
| IPython / Jupyter | `%debug` right after the failing cell (`%pdb on` starts it automatically) |
| pytest | `pytest --pdb` (post-mortem on each failure), `pytest --trace` (stop at the start of each test) |

`python -m pdb` on a crashing script: `c` runs it, the exception puts pdb in
post-mortem mode at the failing line, and the values explain the crash.

In [203]:
write_files(
    debug_demo,
    {
        "average.py": """
        def average_order_value(totals):
            return sum(totals) / len(totals)


        daily_totals = {"Mon": [679, 240], "Tue": []}
        for day, totals in daily_totals.items():
            print(day, average_order_value(totals))
    """,
    },
)
pdb_session(
    "python",
    "-m",
    "pdb",
    "average.py",
    cwd=debug_demo,
    commands=["c", "p totals", "u", "p day", "q"],
)

[<scratch>/debug_demo]$ python -m pdb average.py   # stdin: ['c', 'p totals', 'u', 'p day', 'q']
  > <scratch>/debug_demo/average.py(1)<module>()
  -> def average_order_value(totals):
  (Pdb) c
  Mon 459.5
  Traceback (most recent call last):
    File "<python>/lib/python3.15/pdb.py", line 3873, in main
      pdb._run(target)
      ~~~~~~~~^^^^^^^^
    File "<python>/lib/python3.15/pdb.py", line 2770, in _run
      self.run(target.code)
      ~~~~~~~~^^^^^^^^^^^^^
    File "<python>/lib/python3.15/bdb.py", line 912, in run
      exec(cmd, globals, locals)
      ~~~~^^^^^^^^^^^^^^^^^^^^^^
    File "<string>", line 1, in <module>
    File "<scratch>/debug_demo/average.py", line 7, in <module>
      print(day, average_order_value(totals))
                 ~~~~~~~~~~~~~~~~~~~^^^^^^^^
    File "<scratch>/debug_demo/average.py", line 2, in average_order_value
      return sum(totals) / len(totals)
             ~~~~~~~~~~~~^~~~~~~~~~~~~
  ZeroDivisionError: division by zero
  Uncaught excepti

`pdb.pm()` in your own code: the REPL stores the last uncaught exception in
`sys.last_exc` (*New in 3.12*; `sys.last_traceback` before), and `pm()` opens it. A
script can do the same in its top-level error handler:

In [204]:
write_files(
    debug_demo,
    {
        "report_job.py": """
        import pdb
        import sys


        def parse_quantity(text):
            return int(text)


        try:
            quantities = [parse_quantity(text) for text in ["3", "2", "two"]]
        except ValueError as error:
            sys.last_exc = error  # what the interactive REPL does for you
            pdb.pm()
    """,
    },
)
pdb_session("python", "report_job.py", cwd=debug_demo, commands=["p text", "w", "q"])

[<scratch>/debug_demo]$ python report_job.py   # stdin: ['p text', 'w', 'q']
  > <scratch>/debug_demo/report_job.py(6)parse_quantity()
  -> return int(text)
  (Pdb) p text
  'two'
  (Pdb) w
    <scratch>/debug_demo/report_job.py(10)<module>()
  -> quantities = [parse_quantity(text) for text in ["3", "2", "two"]]
  > <scratch>/debug_demo/report_job.py(6)parse_quantity()
  -> return int(text)
  (Pdb) q


**`%debug` in IPython and Jupyter.** Run it in the cell after an exception to open
`ipdb` (pdb with IPython's colours and completion) at the failing frame. It needs an
interactive frontend, so this notebook shows it in an IPython child process with
the input piped in (the prompt is `ipdb>` instead of `(Pdb)`; IPython's profile
and history go to a scratch folder):

In [205]:
ipython_input = (
    "def average(values):\n    return sum(values) / len(values)\n\n"
    "average([])\n%debug\np values\nq\nexit\n"
)
print(f"piped into ipython: {ipython_input.splitlines()}")
exit_code, ipython_output = run_tool(
    "ipython",
    "--simple-prompt",
    "--no-banner",
    "--colors=NoColor",
    input_text=ipython_input,
    cwd=debug_demo,
)
for line in ipython_output.splitlines():
    if re.search(r"ZeroDivisionError|ipdb>|---->|^> ", line):
        print(f"  {line}")
print("(after each ipdb> prompt comes the output of the next piped command: `p values` printed [])")

piped into ipython: ['def average(values):', '    return sum(values) / len(values)', '', 'average([])', '%debug', 'p values', 'q', 'exit']


  ZeroDivisionError                         Traceback (most recent call last)
  ----> 1 average([])
  ----> 2     return sum(values) / len(values)
  ZeroDivisionError: division by zero
  ----> 2     return sum(values) / len(values)
  ipdb> []
  ipdb> 
(after each ipdb> prompt comes the output of the next piped command: `p values` printed [])


**`pytest --pdb`** drops into post-mortem pdb on the first failing test, with the
test's locals available (pytest turns output capture off while pdb runs).
`pytest --trace` instead stops at the first line of every selected test.

In [206]:
pdb_test_project = project(
    "pdb_test",
    {
        **shop_files("billing"),
        "tests/test_bill.py": """
            from decimal import Decimal

            from shop.billing import split_bill


            def test_three_way_split():
                shares = split_bill("10", 3)
                assert shares == [Decimal("3.33")] * 3
        """,
    },
)
pdb_session(
    "pytest",
    "-q",
    "--no-header",
    "-p",
    "no:cacheprovider",
    "--pdb",
    cwd=pdb_test_project,
    commands=["p shares", "p sum(shares)", "c"],
)

[<scratch>/pdb_test]$ pytest -q --no-header -p no:cacheprovider --pdb   # stdin: ['p shares', 'p sum(shares)', 'c']
  F
  >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> traceback >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
  
      def test_three_way_split():
          shares = split_bill("10", 3)
  >       assert shares == [Decimal("3.33")] * 3
  E       AssertionError: assert [Decimal('3.3...cimal('3.33')] == [Decimal('3.3...cimal('3.33')]
  E         
  E         At index 0 diff: Decimal('3.34') != Decimal('3.33')
  E         Use -v to get more diff
  
  tests/test_bill.py:8: AssertionError
  >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> entering PDB >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
  
  >>>>>>>>>>>>>>>>>> PDB post_mortem (IO-capturing turned off) >>>>>>>>>>>>>>>>>>>
  > <scratch>/pdb_test/tests/test_bill.py(8)test_three_way_split()
  -> assert shares == [Decimal("3.33")] * 3
  (Pdb) p shares
  [Decimal('3.34'), Decimal('3.33'), Decimal('3.33')]
  (Pdb) p sum(shares)
  Decimal('10.00')
  (Pdb) c
  
  >>>>>>>>>>>>

**IDE debuggers** do the same with a GUI: click in the gutter to set breakpoints,
hover to inspect variables, and use the call-stack and watch panels. VS Code's
Python extension and PyCharm both offer **conditional breakpoints**, **logpoints**
(print a message without stopping or editing code), "just my code" stepping, and
attaching to a running process. VS Code (and Jupyter's own debugger) use
**debugpy**, which ipykernel installs; a minimal `.vscode/launch.json` entry:

```json
{
  "name": "Debug tests",
  "type": "debugpy",
  "request": "launch",
  "module": "pytest",
  "args": ["-x", "tests/test_bill.py"],
  "justMyCode": true
}
```

*New in 3.14:* `python -m pdb -p PID` attaches pdb to an already running Python
process (PEP 768), no restart needed.

In [207]:
for module_name in ("debugpy",):
    if importlib.util.find_spec(module_name) is None:
        print(f"{module_name}: not installed")
    else:
        print(
            f"{module_name}: {importlib.metadata.version(module_name)} (installed with ipykernel)"
        )

debugpy: 1.8.22 (installed with ipykernel)


**`faulthandler`** prints the Python traceback when the interpreter itself crashes
(a segmentation fault in a C extension, an abort) or hangs, cases where a normal
traceback never appears. Enable it with `python -X faulthandler`,
`PYTHONFAULTHANDLER=1` or `faulthandler.enable()`; pytest enables it automatically.
The child process below reads memory address 0 through `ctypes` to crash on
purpose. *New in 3.14:* the report also includes the C stack, which is long and
machine-specific, so only the Python part is printed.

In [208]:
write_files(
    debug_demo,
    {
        "crash.py": (
            "import ctypes\n\n\ndef read_config():\n"
            "    return ctypes.string_at(0)\n\n\nread_config()\n"
        )
    },
)
completed = subprocess.run(
    [sys.executable, "-X", "faulthandler", "crash.py"],
    cwd=debug_demo,
    capture_output=True,
    text=True,
    env=tool_env(),
)
python_part = tidy(completed.stderr).split("Current thread's C stack")[0]
print(re.sub(r"0x[0-9a-f]+", "0x...", python_part).rstrip())
print(
    f"exit code: {completed.returncode} "
    f"(negative = killed by signal {-completed.returncode}, SIGSEGV)"
)

Fatal Python error: Segmentation fault

Current thread 0x... [python] (most recent call first):
  File "<python>/lib/python3.15/ctypes/__init__.py", line 613 in string_at
  File "<scratch>/debug_demo/crash.py", line 5 in read_config
  File "<scratch>/debug_demo/crash.py", line 8 in <module>
exit code: -11 (negative = killed by signal 11, SIGSEGV)


For **hangs**, `faulthandler.dump_traceback_later(seconds, exit=True)` dumps every
thread's stack if the program is still running after the timeout, showing *where*
it is stuck (here, a loop waiting for a lock that never frees):

In [209]:
write_files(
    debug_demo,
    {
        "hang.py": """
        import faulthandler
        import time


        def wait_for_payment_lock():
            while True:
                time.sleep(0.1)


        faulthandler.dump_traceback_later(1, exit=True)
        wait_for_payment_lock()
    """,
    },
)
completed = subprocess.run(
    [sys.executable, "hang.py"],
    cwd=debug_demo,
    capture_output=True,
    text=True,
    env=tool_env(),
    timeout=30,
)
print(re.sub(r"0x[0-9a-f]+", "0x...", tidy(completed.stderr)).rstrip())
print(f"exit code: {completed.returncode}")

Timeout (0:00:01)!
Thread 0x... [python] (most recent call first):
  File "<scratch>/debug_demo/hang.py", line 7 in wait_for_payment_lock
  File "<scratch>/debug_demo/hang.py", line 11 in <module>
exit code: 1


### 4.3.9 Logging architecture

Unit 1.14 used `logging` for error reporting; this section covers how it is put
together and configured. Four kinds of object cooperate:

| Object | Role | Created with |
| --- | --- | --- |
| **Logger** | the API your code calls (`logger.info(...)`); named, arranged in a dotted hierarchy | `logging.getLogger(__name__)` |
| **Handler** | sends records somewhere: console, file, queue, network | `StreamHandler()`, `RotatingFileHandler(...)` |
| **Formatter** | turns a record into text (or JSON) | `Formatter("%(levelname)s %(message)s")` |
| **Filter** | drops or enriches records; attach to a logger or a handler | `Filter` subclass or any callable |

```mermaid
flowchart LR
    CALL["logger.info(...)"] --> LVL{"logger level<br/>enabled?"}
    LVL -- no --> DROP["dropped<br/>(cheap)"]
    LVL -- yes --> REC["LogRecord"] --> LF["logger filters"] --> H1["each handler:<br/>level, filters, formatter"]
    H1 --> OUT["stream / file / queue"]
    REC --> PROP{"propagate?"}
    PROP -- yes --> PARENT["parent logger's handlers<br/>... up to root"]
```

<details><summary>ASCII version</summary>

```text
logger.info() -> level check -> LogRecord -> logger filters -> handlers (level, filters, formatter) -> output
                                           \-> propagate -> parent's handlers -> ... -> root's handlers
```

</details>

**The logger hierarchy.** Logger names are dotted paths, normally the module's
`__name__`, so `shop.orders.api` is a child of `shop.orders`, which is a child of
`shop`, whose parent is the **root** logger. `getLogger` returns the same object for
the same name. The helper below resets the demo loggers so cells can be rerun.

In [210]:
def reset_loggers(*names: str) -> None:
    """Remove handlers and filters and restore defaults for the given loggers."""
    for name in names:
        logger = logging.getLogger(name)
        for handler in logger.handlers[:]:
            logger.removeHandler(handler)
            handler.close()
        logger.filters.clear()
        logger.setLevel(logging.NOTSET)
        logger.propagate, logger.disabled = True, False


DEMO_LOGGERS = ("shop", "shop.orders", "shop.orders.api", "shop.payments", "shop.audit")
reset_loggers(*DEMO_LOGGERS)
api_logger = logging.getLogger("shop.orders.api")
chain, node = [], api_logger
while node is not None:
    chain.append(node.name)
    node = node.parent
print(f"{' -> '.join(chain)}")
print(f"{logging.getLogger('shop.orders.api') is api_logger=}")
print(f"{logging.getLogger().name=}, {logging.getLevelName(logging.getLogger().level)=}")

shop.orders.api -> shop.orders -> shop -> root
logging.getLogger('shop.orders.api') is api_logger=True
logging.getLogger().name='root', logging.getLevelName(logging.getLogger().level)='WARNING'


**Under the hood: a `LogRecord`.** Every enabled call creates a `LogRecord` whose
attributes are what formatters can show (`%(name)s`, `%(lineno)d`, …). `vars()`
lists them; `extra={...}` adds your own keys as attributes. The message is stored
unformatted (`msg` + `args`) and only merged by `getMessage()` when a handler needs
it.

In [211]:
record = api_logger.makeRecord(
    "shop.orders.api",
    logging.WARNING,
    "orders.py",
    42,
    "order %s is %d days late",
    (7, 3),
    None,
    func="check_delays",
    extra={"order_id": 7},
)
stable_keys = [
    "name",
    "levelname",
    "levelno",
    "pathname",
    "lineno",
    "funcName",
    "msg",
    "args",
    "order_id",
]
print({key: vars(record)[key] for key in stable_keys})
print(f"{record.getMessage()=}")
print(f"{sorted(set(vars(record)) - set(stable_keys))=}")

{'name': 'shop.orders.api', 'levelname': 'WARNING', 'levelno': 30, 'pathname': 'orders.py', 'lineno': 42, 'funcName': 'check_delays', 'msg': 'order %s is %d days late', 'args': (7, 3), 'order_id': 7}
record.getMessage()='order 7 is 3 days late'
sorted(set(vars(record)) - set(stable_keys))=['created', 'exc_info', 'exc_text', 'filename', 'module', 'msecs', 'process', 'processName', 'relativeCreated', 'stack_info', 'taskName', 'thread', 'threadName']


**Levels.** Each level is a number; a record passes when its level is at least the
threshold. There are **two thresholds**: the logger's (checked first, before a
record even exists) and each handler's. A logger with level `NOTSET` (0) uses its
nearest ancestor's level, its *effective level*; the root logger defaults to
`WARNING`.

| Level | Number | Use for |
| --- | --- | --- |
| `DEBUG` | 10 | details for diagnosing problems |
| `INFO` | 20 | confirmation that things work ("order 7 shipped") |
| `WARNING` | 30 | something unexpected, the program carries on (default threshold) |
| `ERROR` | 40 | an operation failed |
| `CRITICAL` | 50 | the program may not be able to continue |

In [212]:
reset_loggers(*DEMO_LOGGERS)
shop_logger = logging.getLogger("shop")
console = logging.StreamHandler(sys.stdout)
console.setFormatter(logging.Formatter("%(levelname)-8s %(name)s: %(message)s"))
console.setLevel(logging.INFO)  # handler threshold
shop_logger.addHandler(console)
shop_logger.setLevel(logging.DEBUG)  # logger threshold
shop_logger.propagate = False

orders_logger = logging.getLogger("shop.orders")
print(
    f"{orders_logger.level=} (NOTSET), {logging.getLevelName(orders_logger.getEffectiveLevel())=}"
)
orders_logger.debug("cart recalculated")  # passes the logger, blocked by the handler
orders_logger.info("order %s placed", 7)
orders_logger.warning("order %s has no delivery slot", 7)
print(f"{orders_logger.isEnabledFor(logging.DEBUG)=}")

orders_logger.level=0 (NOTSET), logging.getLevelName(orders_logger.getEffectiveLevel())='DEBUG'
INFO     shop.orders: order 7 placed


WARNING  shop.orders: order 7 has no delivery slot


orders_logger.isEnabledFor(logging.DEBUG)=True


**Predict the output.**

```python
# a fresh script, logging never configured
import logging

logger = logging.getLogger("shop.orders")
logger.info("order 7 placed")
logger.warning("order 7 has no delivery slot")
logger.error("order %s failed", 8)
```

<details><summary>Answer</summary>

Only two lines, on **stderr**, with no level or logger name:
`order 7 has no delivery slot` and `order 8 failed`. Nothing configured the root
logger, so its default level `WARNING` drops the INFO record, and records that
find no handler go to `logging.lastResort`, a stderr handler with level WARNING and
the bare `%(message)s` format. `basicConfig()` (or `dictConfig`) replaces this.

</details>

Run the next cell to check your prediction.

In [213]:
script = (
    "import logging\n"
    "logger = logging.getLogger('shop.orders')\n"
    "logger.info('order 7 placed')\n"
    "logger.warning('order 7 has no delivery slot')\n"
    "logger.error('order %s failed', 8)\n"
)
completed = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
print(f"{completed.stdout=}")
print(f"{completed.stderr=}")

completed.stdout=''
completed.stderr='order 7 has no delivery slot\norder 8 failed\n'


**Handlers.** The standard library has handlers for most destinations
(`logging.handlers`): `StreamHandler` (console), `FileHandler`,
`RotatingFileHandler` (by size), `TimedRotatingFileHandler` (by time),
`WatchedFileHandler` (works with external logrotate), `QueueHandler`, `SysLogHandler`,
`SMTPHandler`, `HTTPHandler`, `MemoryHandler`. Three of them in detail.

**`StreamHandler`** writes to a stream, `sys.stderr` by default. Console logs
usually go to stderr so they do not mix with a program's real output on stdout;
this notebook passes `sys.stdout` so the output appears inline.

In [214]:
reset_loggers(*DEMO_LOGGERS)
stream_target = io.StringIO()
to_stderr_like = logging.StreamHandler(stream_target)
to_stderr_like.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
payments_logger = logging.getLogger("shop.payments")
payments_logger.addHandler(to_stderr_like)
payments_logger.propagate = False
payments_logger.error("card declined for order %s", 7)
print(f"{logging.StreamHandler().stream is sys.stderr=}")
print(f"captured: {stream_target.getvalue()!r}")

logging.StreamHandler().stream is sys.stderr=True
captured: 'ERROR:shop.payments:card declined for order 7\n'


**`RotatingFileHandler`** starts a new file when the current one would exceed
`maxBytes`, keeping `backupCount` old files (`shop.log.1` is the newest backup).
Without rotation, a long-running service fills the disk. The demo uses a tiny limit
to force rotation.

In [215]:
reset_loggers(*DEMO_LOGGERS)
log_folder = fresh_dir("rotating_logs")
rotating = logging.handlers.RotatingFileHandler(
    log_folder / "shop.log", maxBytes=120, backupCount=2, encoding="utf-8"
)
rotating.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
audit_logger = logging.getLogger("shop.audit")
audit_logger.addHandler(rotating)
audit_logger.setLevel(logging.INFO)
audit_logger.propagate = False
for order_id in range(1, 9):
    audit_logger.info("order %s shipped to warehouse PNQ1", order_id)
reset_loggers("shop.audit")
for path in sorted(log_folder.iterdir()):
    print(f"{path.name:<11} {path.read_text().splitlines()}")

shop.log    ['INFO order 7 shipped to warehouse PNQ1', 'INFO order 8 shipped to warehouse PNQ1']
shop.log.1  ['INFO order 4 shipped to warehouse PNQ1', 'INFO order 5 shipped to warehouse PNQ1', 'INFO order 6 shipped to warehouse PNQ1']
shop.log.2  ['INFO order 1 shipped to warehouse PNQ1', 'INFO order 2 shipped to warehouse PNQ1', 'INFO order 3 shipped to warehouse PNQ1']


**`QueueHandler` + `QueueListener`.** Writing to a file or the network inside a
request handler makes the request wait for slow I/O. A `QueueHandler` only puts the
record on a queue (fast); a `QueueListener` thread takes records off the queue and
passes them to the real handlers. This is the recommended setup for web servers and
for asyncio code. Since 3.12, `dictConfig` can build the pair for you
(`"class": "logging.handlers.QueueHandler"` with a `"handlers"` list).

In [216]:
reset_loggers(*DEMO_LOGGERS)
log_queue: queue.Queue[logging.LogRecord] = queue.Queue()
slow_console = logging.StreamHandler(sys.stdout)
slow_console.setFormatter(logging.Formatter("[%(threadName)s] %(levelname)s %(message)s"))
listener = logging.handlers.QueueListener(log_queue, slow_console)

orders_logger = logging.getLogger("shop.orders")
orders_logger.addHandler(logging.handlers.QueueHandler(log_queue))
orders_logger.setLevel(logging.INFO)
orders_logger.propagate = False

listener.start()
orders_logger.info("order 7 placed")  # returns immediately; the listener writes
orders_logger.warning("order 8 payment pending")
listener.stop()  # flushes the queue and joins the thread
print(f"{log_queue.empty()=}")

[MainThread] INFO order 7 placed


[MainThread] WARNING order 8 payment pending


log_queue.empty()=True


**Formatters.** A format string picks record attributes. `%`-style is the default;
`style="{"` allows `{levelname:<8}`. `datefmt` formats `%(asctime)s` with
`time.strftime` codes, and `converter = time.gmtime` makes timestamps UTC instead of
the machine's local time (do this on servers). Every timestamp in this section comes
from the `FrozenClock` filter defined in the unit setup, which stamps each record
with 2026-11-07 09:30:00 UTC so the output never changes.

| Attribute | Example |
| --- | --- |
| `%(asctime)s` | `2026-11-07T09:30:00Z` (with `datefmt`) |
| `%(levelname)s` / `%(levelno)d` | `INFO` / `20` |
| `%(name)s` | `shop.orders` |
| `%(message)s` | the merged message |
| `%(module)s`, `%(funcName)s`, `%(lineno)d` | where the call happened |
| `%(process)d`, `%(thread)d`, `%(threadName)s`, `%(taskName)s` (3.12) | concurrency context |

In [217]:
reset_loggers(*DEMO_LOGGERS)
timestamped = logging.StreamHandler(sys.stdout)
utc_formatter = logging.Formatter(
    "%(asctime)s %(levelname)-7s %(name)s %(funcName)s:%(lineno)d %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%SZ",
)
utc_formatter.converter = time.gmtime
timestamped.setFormatter(utc_formatter)
timestamped.addFilter(FrozenClock())

brace_handler = logging.StreamHandler(sys.stdout)
brace_handler.setFormatter(logging.Formatter("{levelname:<7} | {name:>12} | {message}", style="{"))

orders_logger = logging.getLogger("shop.orders")
orders_logger.handlers[:] = [timestamped, brace_handler]
orders_logger.propagate = False


def ship_order(order_id: int) -> None:
    orders_logger.warning("order %s shipped late", order_id)


ship_order(7)

2026-11-07T09:30:00Z WARNING shop.orders ship_order:20 order 7 shipped late


WARNING |  shop.orders | order 7 shipped late


**Filters** decide per record whether it passes, and may **add attributes** to it.
A filter is a `logging.Filter` subclass or any callable taking a record; on a
logger it applies to records created by that logger, on a handler to everything the
handler receives. Two typical filters: one adds the current request id (from a
`contextvars.ContextVar`, which is how web apps attach request context), the other
drops noisy health-check messages.

In [218]:
import contextvars

request_id: contextvars.ContextVar[str] = contextvars.ContextVar("request_id", default="-")


class RequestIdFilter(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        record.request_id = request_id.get()
        return True


def drop_health_checks(record: logging.LogRecord) -> bool:
    return "/health" not in record.getMessage()


reset_loggers(*DEMO_LOGGERS)
request_handler = logging.StreamHandler(sys.stdout)
request_handler.setFormatter(logging.Formatter("[%(request_id)s] %(levelname)s %(message)s"))
request_handler.addFilter(RequestIdFilter())
request_handler.addFilter(drop_health_checks)
api_logger = logging.getLogger("shop.orders.api")
api_logger.addHandler(request_handler)
api_logger.setLevel(logging.INFO)
api_logger.propagate = False

api_logger.info("GET /health 200")
token = request_id.set("req-7f3a")
api_logger.info("POST /orders 201")
request_id.reset(token)
api_logger.info("GET /products 200")

[req-7f3a] INFO POST /orders 201


[-] INFO GET /products 200


**Propagation.** After a logger's own handlers run, the record is passed to the
parent's handlers, and so on up to the root (the logger levels of ancestors are
*not* checked again). That is why you normally attach handlers **only to the root**
(or one top-level application logger) and let module loggers propagate. Attaching a
handler to both a child and an ancestor prints every record twice;
`propagate = False` stops the climb.

In [219]:
reset_loggers(*DEMO_LOGGERS)
for logger_name in ("shop", "shop.orders"):
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(f"via {logger_name!r:<14} %(name)s: %(message)s"))
    logging.getLogger(logger_name).addHandler(handler)
logging.getLogger("shop").propagate = False  # keep the demo away from the root logger
logging.getLogger("shop").setLevel(logging.INFO)

logging.getLogger("shop.orders").info("order 7 placed (printed twice)")
logging.getLogger("shop.orders").propagate = False
logging.getLogger("shop.orders").info("order 8 placed (printed once)")
logging.getLogger("shop.payments").info("payment captured (no own handler: the parent prints it)")

via 'shop.orders'  shop.orders: order 7 placed (printed twice)


via 'shop'         shop.orders: order 7 placed (printed twice)


via 'shop.orders'  shop.orders: order 8 placed (printed once)


via 'shop'         shop.payments: payment captured (no own handler: the parent prints it)


**`dictConfig`**: configure everything from one dictionary, once, at the
application's entry point (`main()`, the ASGI app factory), never at import time.
The dict can come from a TOML, YAML or JSON file. Keys: `version: 1` (required),
`formatters`, `filters`, `handlers` (each names its formatter, filters and level),
`loggers`, `root`. `"()"` gives a factory (your own class) instead of a built-in
one; `"ext://sys.stdout"` refers to an object by import path.

In [220]:
class UtcFormatter(logging.Formatter):
    converter = time.gmtime  # timestamps in UTC, whatever the server's time zone


reset_loggers(*DEMO_LOGGERS)
LOGGING_CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "console": {
            "()": UtcFormatter,
            "fmt": "%(asctime)s %(levelname)-7s %(name)s: %(message)s",
            "datefmt": "%H:%M:%S",
        },
    },
    "filters": {
        "frozen_clock": {"()": FrozenClock},  # demo only: stable timestamps
        "request_id": {"()": RequestIdFilter},
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "stream": "ext://sys.stdout",
            "formatter": "console",
            "filters": ["frozen_clock", "request_id"],
            "level": "INFO",
        },
    },
    "loggers": {
        "shop": {"handlers": ["console"], "level": "DEBUG", "propagate": False},
        "shop.payments": {"level": "WARNING"},
    },
}
logging.config.dictConfig(LOGGING_CONFIG)
logging.getLogger("shop.orders").info("order 7 placed")
logging.getLogger("shop.payments").info("payment authorised (below WARNING: dropped)")
logging.getLogger("shop.payments").warning("payment retry 2/3")
console_handler = logging.getHandlerByName("console")  # New in 3.12
print(
    f"{type(console_handler).__name__}, {console_handler.name=}, "
    f"{logging.getLevelName(console_handler.level)=}"
)

09:30:00 INFO    shop.orders: order 7 placed


09:30:00 WARNING shop.payments: payment retry 2/3


StreamHandler, console_handler.name='console', logging.getLevelName(console_handler.level)='INFO'


> **Warning:** `disable_existing_loggers` defaults to **`True`**: every logger that
> already exists when `dictConfig` runs, and is not named in the config, is
> silently disabled. Modules create their loggers at import time, so importing your
> code *before* configuring logging and forgetting this flag loses their messages.
> A child process shows it:

In [221]:
write_files(
    fresh_dir("disable_existing"),
    {
        "app.py": """
        import logging
        import logging.config

        inventory_logger = logging.getLogger("shop.inventory")  # created at import time

        for disable in (True, False):
            logging.config.dictConfig({
                "version": 1,
                "disable_existing_loggers": disable,
                "handlers": {
                    "out": {"class": "logging.StreamHandler", "stream": "ext://sys.stdout"}
                },
                "root": {"handlers": ["out"], "level": "INFO"},
            })
            print(f"disable_existing_loggers={disable}: logging...")
            inventory_logger.warning("stock for TEA-500 is low")
    """,
    },
)
shell("python", "app.py", cwd=UNIT43_DIR / "disable_existing")

[<scratch>/disable_existing]$ python app.py
  disable_existing_loggers=True: logging...
  disable_existing_loggers=False: logging...
  stock for TEA-500 is low


**Rules for libraries.** A library (anything other people import) must not
configure logging: no `basicConfig`, no handlers on the root, no levels. It creates
module loggers and, by convention, adds a `NullHandler` to its top-level logger.
The **application** decides where logs go.

Why `NullHandler`: when a record finds no handler anywhere in the hierarchy,
`logging` falls back to `logging.lastResort`, which prints WARNING and above to
stderr. A `NullHandler` counts as a handler, so the library stays silent until the
application configures logging, and then its records propagate to the
application's handlers as usual.

In [222]:
write_files(
    fresh_dir("library_rules"),
    {
        "rupee_tools/__init__.py": """
        import logging

        logger = logging.getLogger(__name__)


        def parse_amount(text):
            if "," in text:
                logger.warning("ignoring thousands separators in %r", text)
            return int(text.replace(",", ""))
    """,
        "rupee_tools_quiet/__init__.py": """
        import logging

        logger = logging.getLogger(__name__)
        logger.addHandler(logging.NullHandler())  # the library convention


        def parse_amount(text):
            if "," in text:
                logger.warning("ignoring thousands separators in %r", text)
            return int(text.replace(",", ""))
    """,
        "app.py": """
        import logging
        import sys

        import rupee_tools
        import rupee_tools_quiet

        print("app has not configured logging:")
        rupee_tools.parse_amount("1,200")        # lastResort prints to stderr
        rupee_tools_quiet.parse_amount("1,200")  # silent
        logging.basicConfig(stream=sys.stdout, format="  app handler: %(name)s %(message)s")
        print("app configured logging:")
        rupee_tools_quiet.parse_amount("2,400")  # now reaches the app's handler
    """,
    },
)
shell("python", "app.py", cwd=UNIT43_DIR / "library_rules", merged=True)

[<scratch>/library_rules]$ python app.py
  app has not configured logging:
  ignoring thousands separators in '1,200'
  app configured logging:
    app handler: rupee_tools_quiet ignoring thousands separators in '2,400'


**`logger.exception(...)`** logs at ERROR level **with the current traceback**
appended; call it only inside an `except` block. For other levels pass
`exc_info=True` (`logger.warning("...", exc_info=True)`), and `stack_info=True` adds
the current call stack even without an exception. Never `print(error)` in a
handler: the traceback is the most useful part. (Run in a child process so the
traceback shows stable file paths.)

In [223]:
write_files(
    fresh_dir("exception_logging"),
    {
        "worker.py": """
        import logging
        import sys

        logging.basicConfig(
            stream=sys.stdout, level=logging.INFO, format="%(levelname)s %(name)s: %(message)s"
        )
        logger = logging.getLogger("shop.worker")


        def charge(order_id, amount):
            return amount / 0  # payment gateway bug


        for order_id in (7, 8):
            try:
                charge(order_id, 679)
            except ZeroDivisionError:
                logger.exception("charging order %s failed", order_id)
                break
    """,
    },
)
shell("python", "worker.py", cwd=UNIT43_DIR / "exception_logging")

[<scratch>/exception_logging]$ python worker.py
  ERROR shop.worker: charging order 7 failed
  Traceback (most recent call last):
    File "<scratch>/exception_logging/worker.py", line 16, in <module>
      charge(order_id, 679)
      ~~~~~~^^^^^^^^^^^^^^^
    File "<scratch>/exception_logging/worker.py", line 11, in charge
      return amount / 0  # payment gateway bug
             ~~~~~~~^~~
  ZeroDivisionError: division by zero


**Lazy `%s` formatting.** Pass arguments separately, `logger.debug("total %s",
total)`, instead of an f-string. The message is only built if a handler will
actually emit the record, so disabled DEBUG calls cost almost nothing, and log
aggregators can group records by their constant template. With an f-string, the
formatting (and every `__str__`/`__repr__` call in it) happens before `logging`
even checks the level. For genuinely expensive data, guard with
`logger.isEnabledFor(logging.DEBUG)`. (ruff's `G004` rule flags f-strings in
logging calls.)

In [224]:
class ExpensiveReport:
    """Counts how often it is rendered."""

    renders = 0

    def __str__(self) -> str:
        ExpensiveReport.renders += 1
        return "<1 MB of stock data>"


reset_loggers(*DEMO_LOGGERS)
lazy_logger = logging.getLogger("shop.orders")
lazy_logger.setLevel(logging.INFO)  # DEBUG is disabled
lazy_logger.addHandler(logging.NullHandler())
report = ExpensiveReport()

for _ in range(1_000):
    lazy_logger.debug("stock report: %s", report)
print(f"lazy %s style: {ExpensiveReport.renders} renders")
for _ in range(1_000):
    lazy_logger.debug(f"stock report: {report}")  # noqa: G004 (the anti-pattern)
print(f"f-string:      {ExpensiveReport.renders} renders")

lazy %s style: 0 renders
f-string:      1000 renders


**`LoggerAdapter`** wraps a logger and adds the same context to every record, for
example the order being processed, without repeating `extra=` on each call.
Override `process()` for custom behaviour. *New in 3.13:* `merge_extra=True` merges
a call's own `extra=` with the adapter's instead of discarding it.

In [225]:
reset_loggers(*DEMO_LOGGERS)
context_handler = logging.StreamHandler(sys.stdout)
context_handler.setFormatter(logging.Formatter("order=%(order_id)s %(levelname)s %(message)s"))
orders_logger = logging.getLogger("shop.orders")
orders_logger.addHandler(context_handler)
orders_logger.setLevel(logging.INFO)
orders_logger.propagate = False

order_log = logging.LoggerAdapter(orders_logger, {"order_id": 7})
order_log.info("payment captured")
order_log.warning("courier not assigned")

if sys.version_info >= (3, 13):
    context_handler.setFormatter(logging.Formatter("order=%(order_id)s step=%(step)s %(message)s"))
    merged = logging.LoggerAdapter(orders_logger, {"order_id": 8}, merge_extra=True)
    merged.info("packed", extra={"step": "warehouse"})

order=7 INFO payment captured


order=7 WARNING courier not assigned


order=8 step=warehouse packed


**JSON logs.** In production, logs are read by machines (Loki, Elasticsearch,
CloudWatch), so each record is written as one JSON object per line with
consistent keys, and context goes into fields rather than the message text. A
formatter only has to return a string, so a JSON formatter is a few lines; libraries
such as `python-json-logger` and `structlog` (Phase 6) add more. Lab 4.3 builds a
complete one with `dictConfig`.

In [226]:
class MinimalJsonFormatter(logging.Formatter):
    """One JSON object per record: time, level, logger, message and selected extras."""

    EXTRA_KEYS = ("order_id", "request_id")

    def format(self, record: logging.LogRecord) -> str:
        payload = {
            "time": datetime.fromtimestamp(record.created, UTC).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        payload |= {key: getattr(record, key) for key in self.EXTRA_KEYS if hasattr(record, key)}
        return json.dumps(payload)


reset_loggers(*DEMO_LOGGERS)
json_handler = logging.StreamHandler(sys.stdout)
json_handler.setFormatter(MinimalJsonFormatter())
json_handler.addFilter(FrozenClock())
orders_logger = logging.getLogger("shop.orders")
orders_logger.addHandler(json_handler)
orders_logger.propagate = False
orders_logger.warning("order %s shipped late", 7, extra={"order_id": 7})
reset_loggers(*DEMO_LOGGERS)

{"time": "2026-11-07T09:30:00+00:00", "level": "WARNING", "logger": "shop.orders", "message": "order 7 shipped late", "order_id": 7}


### 4.3.10 Quality gates

A **quality gate** is an automatic check that code must pass before it is merged:
locally before each commit (pre-commit), and again on every push and pull request
(CI). The usual Python gate: **ruff** (lint + format), a **type checker** (Unit 4.2),
and **pytest with a coverage threshold**.

**ruff** is a linter and formatter in one binary (written in Rust), replacing
flake8, isort, pyupgrade, black and dozens of plugins. Rules are grouped by prefix;
this course enables `E, W, F, I, UP, B`.

| Prefix | Origin | Catches |
| --- | --- | --- |
| `E`, `W` | pycodestyle | style errors and warnings (whitespace, line length) |
| `F` | Pyflakes | unused imports and variables, undefined names, real bugs |
| `I` | isort | import order |
| `UP` | pyupgrade | outdated syntax for your `target-version` |
| `B` | flake8-bugbear | likely bugs: mutable defaults, `except:` without type, loop-variable closures |
| `SIM`, `RET`, `PL` | flake8-simplify, flake8-return, Pylint | simplifications, return style, many checks |
| `S` | flake8-bandit | security smells: `shell=True`, `pickle`, hard-coded passwords |
| `G`, `LOG` | flake8-logging-format, flake8-logging | f-strings in logging calls, misuse of `logging` |
| `RUF` | ruff's own | ambiguous Unicode characters, unused `noqa`, … |

A file with typical problems, checked with the course rules plus `S` and `G`:

In [227]:
lint_demo = write_files(
    fresh_dir("lint_demo"),
    {
        "pyproject.toml": """
        [tool.ruff]
        line-length = 100
        target-version = "py315"

        [tool.ruff.lint]
        select = ["E", "W", "F", "I", "UP", "B", "S", "G"]
    """,
        "orders.py": """
        import sys, os
        import logging
        import subprocess
        from typing import List, Optional

        logger = logging.getLogger(__name__)


        def add_line(sku: str, lines: List[str] = []) -> Optional[List[str]]:
            lines.append(sku)
            logger.info(f"added {sku}")
            return lines


        def archive(name: str) -> None:
            try:
                subprocess.run(f"tar czf {name}.tgz orders/", shell=True)
            except:
                pass
    """,
    },
)
shell("ruff", "check", "--output-format=concise", ".", cwd=lint_demo)

[<scratch>/lint_demo]$ ruff check --output-format=concise .
  orders.py:1:1: E401 [*] Multiple imports on one line
  orders.py:1:1: I001 [*] Import block is un-sorted or un-formatted
  orders.py:1:8: F401 [*] `sys` imported but unused
  orders.py:1:13: F401 [*] `os` imported but unused
  orders.py:4:1: UP035 `typing.List` is deprecated, use `list` instead
  orders.py:9:31: UP006 [*] Use `list` instead of `List` for type annotation
  orders.py:9:43: B006 Do not use mutable data structures for argument defaults
  orders.py:9:50: UP045 [*] Use `X | None` for type annotations
  orders.py:9:59: UP006 [*] Use `list` instead of `List` for type annotation
  orders.py:11:17: G004 Logging statement uses f-string
  orders.py:17:9: S602 `subprocess` call with `shell=True` identified, security issue
  orders.py:18:5: E722 Do not use bare `except`
  orders.py:18:5: S110 `try`-`except`-`pass` detected, consider logging the exception
  Found 13 errors.
  [*] 7 fixable with `--fix` (1 unsafe fix availa

`ruff rule CODE` explains any rule. `--fix` applies the safe automatic fixes
(import sorting, unused imports, modern syntax); `--statistics` summarizes what is
left. Bugs such as the mutable default (`B006`) and `shell=True` (`S602`) still need
a human.

In [228]:
shell("ruff", "rule", "B006", cwd=lint_demo, max_lines=12)
shell("ruff", "check", "--fix", "--statistics", ".", cwd=lint_demo)

[<scratch>/lint_demo]$ ruff rule B006
  # mutable-argument-default (B006)
  
  Derived from the **flake8-bugbear** linter.
  
  Fix is sometimes available.
  
  ## What it does
  Checks for uses of mutable objects as function argument defaults.
  
  ## Why is this bad?
  Function defaults are evaluated once, when the function is defined.
  
  ... (53 more lines)


[<scratch>/lint_demo]$ ruff check --fix --statistics .
  1	E722	bare-except
  1	G004	logging-f-string
  1	B006	mutable-argument-default
  1	S602	subprocess-popen-with-shell-equals-true
  1	S110	try-except-pass
  Found 13 errors (8 fixed, 5 remaining).
  1 unsafe fix available with `--fix --unsafe-fixes`.
  (exit code 1)


`ruff format` is the formatter (Black-compatible): it rewrites layout, never
behaviour. `--diff` shows what it would change, `--check` only reports (for CI).

In [229]:
write_files(
    lint_demo,
    {
        "cart.py": (
            "def total( prices ):\n    return sum( [ p*1.18 for p in prices ] )\n"
            "TAX_RATES={'GST':18,'cess':1}\n"
        )
    },
)
shell("ruff", "format", "--diff", "cart.py", cwd=lint_demo)
shell("ruff", "format", "--check", ".", cwd=lint_demo)

[<scratch>/lint_demo]$ ruff format --diff cart.py
  --- cart.py
  +++ cart.py
  @@ -1,3 +1,5 @@
  -def total( prices ):
  -    return sum( [ p*1.18 for p in prices ] )
  -TAX_RATES={'GST':18,'cess':1}
  +def total(prices):
  +    return sum([p * 1.18 for p in prices])
  +
  +
  +TAX_RATES = {"GST": 18, "cess": 1}
  
  1 file would be reformatted
  (exit code 1)


[<scratch>/lint_demo]$ ruff format --check .
  unformatted: File would be reformatted
   --> cart.py:1:11
    |
    - def total( prices ):
    -     return sum( [ p*1.18 for p in prices ] )
    - TAX_RATES={'GST':18,'cess':1}
  1 + def total(prices):
  2 +     return sum([p * 1.18 for p in prices])
  3 +
  4 +
  5 + TAX_RATES = {"GST": 18, "cess": 1}
    |
  
  1 file would be reformatted, 1 file already formatted
  (exit code 1)


**pre-commit** runs checks on the files you are committing, from a git hook, and
blocks the commit if one fails (husky + lint-staged in the JavaScript world). Hooks
are listed in `.pre-commit-config.yaml`. Normally each hook comes from a git
repository pinned to a tag, which pre-commit downloads into its cache once:

```yaml
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.17.3
    hooks:
      - id: ruff-check
        args: [--fix]
      - id: ruff-format
```

That needs the network, so the demo uses **local hooks** (`repo: local`,
`language: system`) that run the ruff already installed in the course
environment. A scratch git repository, then `pre-commit install` writes the hook
into `.git/hooks/pre-commit`:

In [230]:
HOOK_ENV = tool_env(PATH=f"{VENV_BIN}{os.pathsep}{os.environ['PATH']}")
hooks_repo = write_files(
    fresh_dir("precommit_demo"),
    {
        ".pre-commit-config.yaml": """
        repos:
          - repo: local
            hooks:
              - id: ruff-check
                name: ruff check
                entry: ruff check --fix
                language: system
                types: [python]
              - id: ruff-format
                name: ruff format
                entry: ruff format
                language: system
                types: [python]
    """,
        "pyproject.toml": '[tool.ruff.lint]\nselect = ["E", "F", "I", "UP", "B"]\n',
        "shop.py": "import os\nimport sys\ndef discount( total ):\n  return total==None\n",
    },
)
for command in (("git", "init", "-q", "-b", "main"), ("pre-commit", "install")):
    shell(*command, cwd=hooks_repo, env=HOOK_ENV)

[<scratch>/precommit_demo]$ git init -q -b main


[<scratch>/precommit_demo]$ pre-commit install
  pre-commit installed at .git/hooks/pre-commit


The first commit attempt fails: ruff fixed the unused imports and the formatting
(pre-commit reports "files were modified by this hook"), but `== None` (`E711`) has
no safe automatic fix. The commit is blocked until the file is fixed and staged
again.

In [231]:
shell("git", "add", "-A", cwd=hooks_repo, env=HOOK_ENV)
shell("git", "commit", "-q", "-m", "Add discount rule", cwd=hooks_repo, env=HOOK_ENV)
show_file(hooks_repo / "shop.py")

[<scratch>/precommit_demo]$ git add -A


[<scratch>/precommit_demo]$ git commit -q -m 'Add discount rule'
  ruff check...............................................................Failed
  - hook id: ruff-check
  - exit code: 1
  - files were modified by this hook
  
  E711 Comparison to `None` should be `cond is None`
   --> shop.py:2:17
    |
  1 | def discount( total ):
  2 |   return total==None
    |                 ^^^^
  help: Replace with `cond is None`
    |
  1 | def discount( total ):
    -   return total==None
  2 +   return total is None
    |
  note: This is an unsafe fix and may change runtime behavior
  
  Found 3 errors (2 fixed, 1 remaining).
  1 unsafe fix available with `--fix --unsafe-fixes`.
  
  ruff format..............................................................Failed
  - hook id: ruff-format
  - files were modified by this hook
  
  1 file reformatted
  (exit code 1)
--- shop.py ---
def discount(total):
    return total == None


After fixing the comparison by hand, the hooks pass and the commit goes through:

In [232]:
(hooks_repo / "shop.py").write_text(
    (hooks_repo / "shop.py").read_text().replace("total == None", "total is None")
)
shell("git", "add", "-A", cwd=hooks_repo, env=HOOK_ENV)
shell("git", "commit", "-m", "Add discount rule", cwd=hooks_repo, env=HOOK_ENV)
shell("pre-commit", "run", "--all-files", cwd=hooks_repo, env=HOOK_ENV)

[<scratch>/precommit_demo]$ git add -A


[<scratch>/precommit_demo]$ git commit -m 'Add discount rule'
  [main (root-commit) 6407ba6] Add discount rule
   3 files changed, 17 insertions(+)
   create mode 100644 .pre-commit-config.yaml
   create mode 100644 pyproject.toml
   create mode 100644 shop.py
  ruff check...............................................................Passed
  ruff format..............................................................Passed


[<scratch>/precommit_demo]$ pre-commit run --all-files
  ruff check...............................................................Passed
  ruff format..............................................................Passed


**CI with GitHub Actions.** pre-commit can be skipped (`git commit --no-verify`), so
the same checks run again on GitHub for every push and pull request, on a clean
machine. A workflow is a YAML file in `.github/workflows/`. This one installs uv,
syncs the locked environment, and runs lint, format check, type check and tests
with a coverage threshold on two Python versions. Branch protection then requires
the job to pass before merging.

In [233]:
CI_WORKFLOW = """
    name: CI

    on:
      push:
        branches: [main]
      pull_request:

    jobs:
      quality:
        runs-on: ubuntu-latest
        strategy:
          matrix:
            python-version: ["3.14", "3.15"]
        steps:
          - uses: actions/checkout@v5
          - uses: astral-sh/setup-uv@v7
            with:
              python-version: ${{ matrix.python-version }}
          - run: uv sync --locked
          - run: uv run ruff check .
          - run: uv run ruff format --check .
          - run: uv run mypy src
          - run: uv run pytest --cov=shop --cov-branch --cov-fail-under=90
"""
ci_project = write_files(fresh_dir("ci_demo"), {".github/workflows/ci.yml": CI_WORKFLOW})
if importlib.util.find_spec("yaml") is None:
    print("SKIP: PyYAML is not installed, so the workflow is not parsed")
else:
    import yaml

    workflow = yaml.safe_load((ci_project / ".github" / "workflows" / "ci.yml").read_text())
    quality_job = workflow["jobs"]["quality"]
    print(f"triggers: {list(workflow[True])}  (YAML 1.1 reads the key 'on' as True)")
    print(f"matrix:   {quality_job['strategy']['matrix']}")
    for step in quality_job["steps"]:
        print(f"  step: {step.get('uses') or step.get('run')}")

triggers: ['push', 'pull_request']  (YAML 1.1 reads the key 'on' as True)
matrix:   {'python-version': ['3.14', '3.15']}
  step: actions/checkout@v5
  step: astral-sh/setup-uv@v7
  step: uv sync --locked
  step: uv run ruff check .
  step: uv run ruff format --check .
  step: uv run mypy src
  step: uv run pytest --cov=shop --cov-branch --cov-fail-under=90


The quirk in the output is real: PyYAML follows YAML 1.1, where `on` means `True`,
so the trigger key is parsed as the boolean (GitHub's own parser treats it as the
string). You cannot run GitHub's runners locally (the third-party `act` tool
imitates them with Docker), but you can run the same steps. A tiny "local CI" runs
the workflow's `run:` commands (minus `uv sync` and the `mypy src` step, which need
a full project) against the coverage demo project from 4.3.6:

In [234]:
print(f"act installed: {shutil.which('act') is not None}")
if importlib.util.find_spec("pytest_cov") is not None:
    local_ci = [
        ("ruff", "check", "."),
        ("ruff", "format", "--check", "."),
        (
            "pytest",
            "-q",
            "--no-header",
            "-p",
            "no:cacheprovider",
            "--cov=shop",
            "--cov-branch",
            "--cov-fail-under=80",
        ),
    ]
    for step in local_ci:
        exit_code, output = run_tool(*step, cwd=coverage_demo)
        status = "PASS" if exit_code == 0 else f"FAIL (exit {exit_code})"
        print(f"{status:<14} {shlex.join(step)}")
        if exit_code:
            print(textwrap.indent("\n".join(output.splitlines()[-6:]), "    "))

act installed: False
PASS           ruff check .
PASS           ruff format --check .


PASS           pytest -q --no-header -p no:cacheprovider --cov=shop --cov-branch --cov-fail-under=80


### Lab 4.3

Seven exercises. In exercises 1–4 and 7 you write **tests**, so the checks test
your tests: each check cell takes the source of your test functions
(`source_of`), adds the imports listed below, runs them with the real pytest in a
child process against the `shop` package, and then against **mutants**, copies of
the package with one small bug planted. A good suite passes on the real code and
fails on every mutant ("catches" it). Exercise 5 runs in the kernel; exercise 6
runs pdb.

Every lab test file starts with these imports, so your tests can use the names
directly: `pytest`, `mock` (from `unittest`), `Decimal`, `datetime`, `Callable`,
Hypothesis's `given`, `example`, `assume`, `settings` and `st`, and from `shop`:
`Money`, `CurrencyMismatchError`, `split_bill`, `ExchangeRateClient`,
`RateLookupError` and every name of the inventory model (`Product`, `Warehouse`,
`StockMovement`, `MovementType`, `InMemoryInventoryRepository`, `EventBus`,
`InventoryService`, `LowStockAlert`, `StockChanged`, the `…Error` classes).
Hypothesis runs deterministically (a fixed profile). Until you finish, the checks
print FAIL; that is expected.

**The code under test.** The next cell writes the `shop` package to
`notebooks/_scratch/unit43/lab/shop_src/` and, for the lab only, puts that folder
on `sys.path` so you can also try things in the kernel. `shop.rates` is new: an
exchange-rate client that receives its HTTP client from outside (dependency
injection), so tests can pass a mock. `shop.inventory` is the Phase 2 inventory
model.

In [235]:
RATES_MODULE = """
    \"\"\"Exchange rates from an HTTP service; the HTTP client is injected.\"\"\"

    from decimal import Decimal


    class RateLookupError(Exception):
        \"\"\"The rate service failed or returned something unusable.\"\"\"


    class ExchangeRateClient:
        def __init__(self, http, base_url: str = "https://rates.example.com/v1") -> None:
            self.http = http  # anything with .get(url, *, params, timeout) -> response
            self.base_url = base_url.rstrip("/")

        def rate(self, base: str, quote: str) -> Decimal:
            \"\"\"How many units of quote one unit of base buys.\"\"\"
            response = self.http.get(
                f"{self.base_url}/latest", params={"base": base, "symbols": quote}, timeout=5.0
            )
            if response.status_code != 200:
                raise RateLookupError(f"rate service returned HTTP {response.status_code}")
            try:
                return Decimal(str(response.json()["rates"][quote]))
            except (KeyError, TypeError) as error:
                raise RateLookupError(f"no {quote} rate in the response") from error

        def convert(self, amount: Decimal, base: str, quote: str) -> Decimal:
            \"\"\"amount in base currency -> quote currency, rounded to 2 places.\"\"\"
            if base == quote:
                return amount
            return (amount * self.rate(base, quote)).quantize(Decimal("0.01"))
"""

INVENTORY_MODULE = """
    \"\"\"The Phase 2 inventory domain model (Phase 2 Project, Steps 2-8).\"\"\"

    from __future__ import annotations

    import re
    from collections import defaultdict
    from collections.abc import Callable, Iterator
    from dataclasses import dataclass
    from datetime import datetime
    from enum import StrEnum, auto
    from typing import Any, Protocol, runtime_checkable

    from shop.money import Money

    SKU_PATTERN = re.compile(r"[A-Z]{2,5}-\\d{2,4}")


    @dataclass(frozen=True, slots=True)
    class Product:
        sku: str
        name: str
        unit_price: Money
        reorder_level: int = 5

        def __post_init__(self) -> None:
            if not SKU_PATTERN.fullmatch(self.sku):
                raise ValueError(f"invalid SKU {self.sku!r}: expected e.g. 'TEA-500'")
            if not self.name.strip():
                raise ValueError("product name is required")
            if not isinstance(self.unit_price, Money) or self.unit_price.amount <= 0:
                raise ValueError("unit_price must be a positive Money")
            if self.reorder_level < 0:
                raise ValueError("reorder_level must be >= 0")


    class MovementType(StrEnum):
        RECEIVED = auto()
        SHIPPED = auto()
        RETURNED = auto()
        ADJUSTED = auto()

        @property
        def direction(self) -> int:
            match self:
                case MovementType.RECEIVED | MovementType.RETURNED:
                    return 1
                case MovementType.SHIPPED:
                    return -1
                case _:
                    return 0


    @dataclass(frozen=True, slots=True)
    class StockMovement:
        warehouse_code: str
        sku: str
        movement_type: MovementType
        quantity: int
        occurred_at: datetime
        note: str = ""

        def __post_init__(self) -> None:
            if self.movement_type is MovementType.ADJUSTED:
                if self.quantity == 0:
                    raise ValueError("an adjustment must change the quantity")
            elif self.quantity <= 0:
                raise ValueError(f"{self.movement_type} quantity must be positive")

        @property
        def delta(self) -> int:
            if self.movement_type is MovementType.ADJUSTED:
                return self.quantity
            return self.quantity * self.movement_type.direction


    class InventoryError(Exception):
        error_code = "inventory_error"

        def to_dict(self) -> dict[str, str]:
            return {"error": self.error_code, "message": str(self)}


    class UnknownWarehouseError(InventoryError, LookupError):
        error_code = "unknown_warehouse"

        def __init__(self, warehouse_code: str) -> None:
            super().__init__(f"no warehouse with code {warehouse_code!r}")
            self.warehouse_code = warehouse_code


    class UnknownProductError(InventoryError, LookupError):
        error_code = "unknown_product"

        def __init__(self, sku: str, warehouse_code: str) -> None:
            super().__init__(f"{sku} is not stocked in {warehouse_code}")
            self.sku, self.warehouse_code = sku, warehouse_code


    class DuplicateProductError(InventoryError, ValueError):
        error_code = "duplicate_product"

        def __init__(self, sku: str, warehouse_code: str) -> None:
            super().__init__(f"{sku} is already stocked in {warehouse_code}")
            self.sku, self.warehouse_code = sku, warehouse_code


    class InsufficientStockError(InventoryError):
        error_code = "insufficient_stock"

        def __init__(self, sku: str, available: int, requested: int) -> None:
            super().__init__(f"not enough {sku}: {available} available, {requested} requested")
            self.sku, self.available, self.requested = sku, available, requested


    class Warehouse:
        def __init__(self, code: str, city: str) -> None:
            self.code, self.city = code, city
            self._products: dict[str, Product] = {}
            self._quantities: dict[str, int] = {}

        def __repr__(self) -> str:
            return f"Warehouse({self.code!r}, {self.city!r}, products={len(self)})"

        def __eq__(self, other: object) -> bool:
            return self.code == other.code if isinstance(other, Warehouse) else NotImplemented

        def __hash__(self) -> int:
            return hash(self.code)

        def __len__(self) -> int:
            return len(self._products)

        def __contains__(self, item: object) -> bool:
            sku = item.sku if isinstance(item, Product) else item
            return sku in self._products

        def __getitem__(self, sku: str) -> int:
            if sku not in self._products:
                raise UnknownProductError(sku, self.code)
            return self._quantities[sku]

        def __iter__(self) -> Iterator[Product]:
            return iter(sorted(self._products.values(), key=lambda product: product.sku))

        def __bool__(self) -> bool:
            return any(self._quantities.values())

        def product(self, sku: str) -> Product:
            if sku not in self._products:
                raise UnknownProductError(sku, self.code)
            return self._products[sku]

        def add_product(self, product: Product) -> None:
            if product.sku in self._products:
                raise DuplicateProductError(product.sku, self.code)
            self._products[product.sku] = product
            self._quantities[product.sku] = 0

        def apply(self, movement: StockMovement) -> int:
            available = self[movement.sku]
            if available + movement.delta < 0:
                raise InsufficientStockError(movement.sku, available, -movement.delta)
            self._quantities[movement.sku] = available + movement.delta
            return self._quantities[movement.sku]

        def stock_value(self) -> Money:
            return sum((product.unit_price * self[product.sku] for product in self), Money(0))

        def low_stock(self) -> list[Product]:
            return [product for product in self if self[product.sku] <= product.reorder_level]


    @runtime_checkable
    class InventoryRepository(Protocol):
        def get_warehouse(self, code: str) -> Warehouse: ...
        def save_warehouse(self, warehouse: Warehouse) -> None: ...
        def list_warehouses(self) -> list[Warehouse]: ...
        def record_movement(self, movement: StockMovement) -> None: ...
        def movements(self, *, sku: str | None = None) -> list[StockMovement]: ...


    class InMemoryInventoryRepository:
        def __init__(self) -> None:
            self._warehouses: dict[str, Warehouse] = {}
            self._movements: list[StockMovement] = []

        def get_warehouse(self, code: str) -> Warehouse:
            try:
                return self._warehouses[code]
            except KeyError:
                raise UnknownWarehouseError(code) from None

        def save_warehouse(self, warehouse: Warehouse) -> None:
            self._warehouses[warehouse.code] = warehouse

        def list_warehouses(self) -> list[Warehouse]:
            return sorted(self._warehouses.values(), key=lambda warehouse: warehouse.code)

        def record_movement(self, movement: StockMovement) -> None:
            self._movements.append(movement)

        def movements(self, *, sku: str | None = None) -> list[StockMovement]:
            return [movement for movement in self._movements if sku is None or movement.sku == sku]


    @dataclass(frozen=True)
    class StockChanged:
        warehouse_code: str
        sku: str
        old_quantity: int
        new_quantity: int
        movement_type: MovementType


    @dataclass(frozen=True)
    class LowStockAlert:
        warehouse_code: str
        sku: str
        quantity: int
        reorder_level: int


    class EventBus:
        def __init__(self) -> None:
            self._handlers: defaultdict[type, list[Callable[[Any], None]]] = defaultdict(list)

        def subscribe(self, event_type: type, handler: Callable[[Any], None]) -> None:
            self._handlers[event_type].append(handler)

        def publish(self, event: object) -> int:
            handlers = self._handlers.get(type(event), [])
            for handler in handlers:
                handler(event)
            return len(handlers)


    class InventoryService:
        def __init__(
            self,
            repository: InventoryRepository,
            events: EventBus,
            clock: Callable[[], datetime] = datetime.now,
        ) -> None:
            self.repository, self.events, self.clock = repository, events, clock

        def open_warehouse(self, code: str, city: str) -> Warehouse:
            warehouse = Warehouse(code, city)
            self.repository.save_warehouse(warehouse)
            return warehouse

        def register_product(self, warehouse_code: str, product: Product) -> None:
            warehouse = self.repository.get_warehouse(warehouse_code)
            warehouse.add_product(product)
            self.repository.save_warehouse(warehouse)

        def receive(self, warehouse_code: str, sku: str, quantity: int, note: str = "") -> int:
            return self._record(warehouse_code, sku, MovementType.RECEIVED, quantity, note)

        def ship(self, warehouse_code: str, sku: str, quantity: int, note: str = "") -> int:
            return self._record(warehouse_code, sku, MovementType.SHIPPED, quantity, note)

        def accept_return(
            self, warehouse_code: str, sku: str, quantity: int, note: str = ""
        ) -> int:
            return self._record(warehouse_code, sku, MovementType.RETURNED, quantity, note)

        def adjust(self, warehouse_code: str, sku: str, change: int, note: str) -> int:
            return self._record(warehouse_code, sku, MovementType.ADJUSTED, change, note)

        def transfer(self, from_code: str, to_code: str, sku: str, quantity: int) -> None:
            source = self.repository.get_warehouse(from_code)
            destination = self.repository.get_warehouse(to_code)
            if source[sku] < quantity:
                raise InsufficientStockError(sku, source[sku], quantity)
            if sku not in destination:
                self.register_product(to_code, source.product(sku))
            self.ship(from_code, sku, quantity, note=f"transfer to {to_code}")
            self.receive(to_code, sku, quantity, note=f"transfer from {from_code}")

        def total_value(self) -> Money:
            return sum(
                (warehouse.stock_value() for warehouse in self.repository.list_warehouses()),
                Money(0),
            )

        def stock_report(self, warehouse_code: str) -> list[str]:
            warehouse = self.repository.get_warehouse(warehouse_code)
            lines = [f"{warehouse.code} ({warehouse.city})"]
            for product in warehouse:
                flag = "  LOW" if product in warehouse.low_stock() else ""
                lines.append(f"{product.sku:<9}{warehouse[product.sku]:>5}{flag}")
            lines.append(f"total {warehouse.stock_value()}")
            return lines

        def _record(
            self,
            warehouse_code: str,
            sku: str,
            movement_type: MovementType,
            quantity: int,
            note: str,
        ) -> int:
            warehouse = self.repository.get_warehouse(warehouse_code)
            old_quantity = warehouse[sku]
            movement = StockMovement(
                warehouse_code, sku, movement_type, quantity, self.clock(), note
            )
            new_quantity = warehouse.apply(movement)
            self.repository.record_movement(movement)
            self.repository.save_warehouse(warehouse)
            self.events.publish(
                StockChanged(warehouse_code, sku, old_quantity, new_quantity, movement_type)
            )
            reorder_level = warehouse.product(sku).reorder_level
            if old_quantity > reorder_level >= new_quantity:
                self.events.publish(LowStockAlert(warehouse_code, sku, new_quantity, reorder_level))
            return new_quantity
"""

LAB_MODULES = {
    "shop/__init__.py": "",
    "shop/money.py": MONEY_MODULE,
    "shop/billing.py": BILLING_MODULE,
    "shop/rates.py": RATES_MODULE,
    "shop/inventory.py": INVENTORY_MODULE,
}
LAB43_DIR = find_repo_root() / "notebooks" / "_scratch" / "unit43" / "lab"
LAB_SHOP_DIR = LAB43_DIR / "shop_src"
for relative_path, module_source in LAB_MODULES.items():
    target = LAB_SHOP_DIR / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(textwrap.dedent(module_source).lstrip("\n"))
if str(LAB_SHOP_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_SHOP_DIR))  # lab only: lets you import the shop package here too

from shop.billing import split_bill  # noqa: E402
from shop.inventory import (  # noqa: E402
    EventBus,
    InMemoryInventoryRepository,
    InsufficientStockError,
    InventoryService,
    LowStockAlert,
    MovementType,
    Product,
    StockMovement,
    UnknownProductError,
    UnknownWarehouseError,
    Warehouse,
)
from shop.money import CurrencyMismatchError, Money  # noqa: E402
from shop.rates import ExchangeRateClient, RateLookupError  # noqa: E402

LAB_NAMES = (
    EventBus,
    InMemoryInventoryRepository,
    InsufficientStockError,
    InventoryService,
    LowStockAlert,
)
LAB_NAMES += (
    MovementType,
    Product,
    StockMovement,
    UnknownProductError,
    UnknownWarehouseError,
    Warehouse,
)
LAB_NAMES += (CurrencyMismatchError, Money, ExchangeRateClient, RateLookupError, split_bill)
print(f"shop package at <scratch>/lab/shop_src: {sorted(LAB_MODULES)}")
print(f"imported into the kernel: {', '.join(obj.__name__ for obj in LAB_NAMES)}")
print(f"{split_bill('100', 3)=}")

shop package at <scratch>/lab/shop_src: ['shop/__init__.py', 'shop/billing.py', 'shop/inventory.py', 'shop/money.py', 'shop/rates.py']
imported into the kernel: EventBus, InMemoryInventoryRepository, InsufficientStockError, InventoryService, LowStockAlert, MovementType, Product, StockMovement, UnknownProductError, UnknownWarehouseError, Warehouse, CurrencyMismatchError, Money, ExchangeRateClient, RateLookupError, split_bill
split_bill('100', 3)=[Decimal('33.34'), Decimal('33.33'), Decimal('33.33')]


**The lab runner.** `run_lab_tests(source, conftest, mutation, coverage_target)`
runs a test file and returns a `LabRun` (outcome per test, coverage percent, raw
output); results are cached by content, so each distinct run happens once.
`catches(source, mutation)` is True when the tests pass on the real package and
fail on the mutant. A mutation is `(file, old text, new text)`.

In [236]:
import xml.etree.ElementTree as ElementTree
from dataclasses import dataclass

import pytest

LAB_TEST_PRELUDE = """
from collections.abc import Callable
from datetime import datetime
from decimal import Decimal
from unittest import mock

import pytest
from hypothesis import assume, example, given, settings
from hypothesis import strategies as st

from shop.billing import split_bill
from shop.inventory import (
    DuplicateProductError, EventBus, InMemoryInventoryRepository, InsufficientStockError,
    InventoryError, InventoryRepository, InventoryService, LowStockAlert, MovementType, Product,
    StockChanged, StockMovement, UnknownProductError, UnknownWarehouseError, Warehouse,
)
from shop.money import CurrencyMismatchError, Money
from shop.rates import ExchangeRateClient, RateLookupError
"""
LAB_CONFTEST = """
from hypothesis import settings

settings.register_profile("lab", derandomize=True, database=None, deadline=None)
settings.load_profile("lab")
"""
LAB_PYPROJECT = """
[tool.pytest.ini_options]
pythonpath = ["."]
testpaths = ["tests"]
"""


@dataclass(frozen=True)
class LabRun:
    """Outcome of one pytest run: test name -> "passed" / "failed" / "skipped", plus coverage."""

    outcomes: dict[str, str]
    coverage_percent: float | None
    output: str

    @property
    def green(self) -> bool:
        """At least one test ran and nothing failed or errored."""
        return bool(self.outcomes) and set(self.outcomes.values()) <= {"passed"}


def lab_test_source(objects: list[object]) -> str:
    """The prelude followed by the source of each test or fixture function (never raises)."""
    try:
        return LAB_TEST_PRELUDE + "\n\n" + "\n\n".join(source_of(obj) for obj in objects)
    except (OSError, TypeError) as error:
        return (
            LAB_TEST_PRELUDE
            + f"\n\ndef test_source_unavailable():\n    raise RuntimeError({str(error)!r})\n"
        )


def run_lab_tests(
    test_source: str,
    conftest_source: str = "",
    mutation: tuple[str, str, str] | None = None,
    coverage_target: str | None = None,
) -> LabRun:
    """Run test_source with pytest in a child process against the shop package (or a mutant)."""
    return _cached_lab_run(test_source, conftest_source, mutation, coverage_target)


@functools.cache
def _cached_lab_run(
    test_source: str,
    conftest_source: str,
    mutation: tuple[str, str, str] | None,
    coverage_target: str | None,
) -> LabRun:
    """The run behind run_lab_tests; identical runs are done once."""
    modules = dict(LAB_MODULES)
    if mutation is not None:
        path, old, new = mutation
        if old not in modules[path]:
            raise ValueError(f"mutation target not found in {path}: {old!r}")
        modules[path] = modules[path].replace(old, new)
    root = LAB43_DIR / "run"
    shutil.rmtree(root, ignore_errors=True)
    files = {**modules, "pyproject.toml": LAB_PYPROJECT, "tests/test_lab.py": test_source}
    files["tests/conftest.py"] = LAB_CONFTEST + "\n" + conftest_source
    for relative_path, text in files.items():
        (root / relative_path).parent.mkdir(parents=True, exist_ok=True)
        (root / relative_path).write_text(
            textwrap.dedent(text).lstrip("\n") if relative_path.startswith("shop") else text
        )
    command = [
        str(Path(sys.executable).parent / "pytest"),
        "-q",
        "-p",
        "no:cacheprovider",
        "--junit-xml=report.xml",
    ]
    if coverage_target:
        command += [f"--cov={coverage_target}", "--cov-branch", "--cov-report=json:coverage.json"]
    environment = {
        key: value
        for key, value in os.environ.items()
        if not key.startswith(("PYTEST_", "PYTHONPATH", "COV"))
    }
    completed = subprocess.run(
        command,
        cwd=root,
        capture_output=True,
        text=True,
        timeout=600,
        env={**environment, "NO_COLOR": "1", "PY_COLORS": "0", "PYTHONDONTWRITEBYTECODE": "1"},
    )
    outcomes: dict[str, str] = {}
    if (root / "report.xml").exists():
        for case in ElementTree.parse(root / "report.xml").getroot().iter("testcase"):
            failed = case.find("failure") is not None or case.find("error") is not None
            skipped = case.find("skipped") is not None
            outcomes[case.get("name", "?")] = (
                "failed" if failed else "skipped" if skipped else "passed"
            )
    coverage_percent = None
    if coverage_target and (root / "coverage.json").exists():
        report = json.loads((root / "coverage.json").read_text())
        coverage_percent = round(report["totals"]["percent_covered"], 1)
    return LabRun(outcomes, coverage_percent, completed.stdout + completed.stderr)


def catches(test_source: str, mutation: tuple[str, str, str], conftest_source: str = "") -> bool:
    """True if the tests pass on the real shop package and fail on the mutant."""
    return (
        run_lab_tests(test_source, conftest_source).green
        and not run_lab_tests(test_source, conftest_source, mutation).green
    )


print("lab runner ready: run_lab_tests(source), catches(source, mutation)")

lab runner ready: run_lab_tests(source), catches(source, mutation)


#### Exercise 1 — parametrized tests for `Money`

Turn the three stubs into real tests of the shared `Money` type.
`test_money_rounding` and `test_money_arithmetic` must each be parametrized with
**at least 4 cases and readable ids** (`ids=[...]` or `pytest.param(..., id=...)`);
`test_money_rejects_bad_input` uses `pytest.raises`. Change the signatures as
needed. The suite must pass on the real `Money` and catch four planted bugs:
rounding half-even instead of half-up, a missing currency check in `+`, float
amounts accepted, and subtraction computed the wrong way round.

In [237]:
def test_money_rounding() -> None:
    """Parametrize: raw amount text -> expected Decimal amount (4+ cases, with ids)."""
    raise NotImplementedError


def test_money_arithmetic() -> None:
    """Parametrize: + and - on Money values -> expected Money (4+ cases, with ids)."""
    raise NotImplementedError


def test_money_rejects_bad_input() -> None:
    """pytest.raises: float amounts (TypeError), mixed currencies (CurrencyMismatchError)."""
    raise NotImplementedError


MONEY_TESTS = [test_money_rounding, test_money_arithmetic, test_money_rejects_bad_input]

Check exercise 1:

In [238]:
def parametrized_with_ids(test: object, minimum_cases: int = 4) -> bool:
    """Does test carry a parametrize mark with enough cases, each with a readable id?"""
    for mark in getattr(test, "pytestmark", []):
        if mark.name != "parametrize" or len(mark.args) < 2:
            continue
        cases = list(mark.args[1])
        explicit_ids = "ids" in mark.kwargs or all(getattr(case, "id", None) for case in cases)
        if len(cases) >= minimum_cases and explicit_ids:
            return True
    return False


MONEY_MUTANTS = {
    "half-even rounding": ("shop/money.py", "rounding=ROUND_HALF_UP", 'rounding="ROUND_HALF_EVEN"'),
    "no currency check": ("shop/money.py", "if other._currency != self._currency:", "if False:"),
    "floats accepted": ("shop/money.py", "if isinstance(amount, float):", "if False:"),
    "reversed subtraction": (
        "shop/money.py",
        "Money(self._amount - other._amount",
        "Money(other._amount - self._amount",
    ),
}
money_source = lab_test_source(MONEY_TESTS)
run_checks(
    [
        ("tests pass on the real Money", lambda: run_lab_tests(money_source).green, True),
        (
            "test_money_rounding: 4+ cases with ids",
            lambda: parametrized_with_ids(MONEY_TESTS[0]),
            True,
        ),
        (
            "test_money_arithmetic: 4+ cases with ids",
            lambda: parametrized_with_ids(MONEY_TESTS[1]),
            True,
        ),
        *[
            (f"catches: {name}", lambda mutation=mutation: catches(money_source, mutation), True)
            for name, mutation in MONEY_MUTANTS.items()
        ],
    ]
)

FAIL  tests pass on the real Money: expected True, got False
FAIL  test_money_rounding: 4+ cases with ids: expected True, got False
FAIL  test_money_arithmetic: 4+ cases with ids: expected True, got False
FAIL  catches: half-even rounding: expected True, got False
FAIL  catches: no currency check: expected True, got False
FAIL  catches: floats accepted: expected True, got False
FAIL  catches: reversed subtraction: expected True, got False
0/7 checks passed


#### Exercise 2 — repository fixtures

Write four fixtures for testing the inventory service. The check puts them in a
`conftest.py` and runs its own hidden tests that use them:

| Fixture | Scope | Provides |
| --- | --- | --- |
| `catalogue` | session | `dict[str, Product]`: `TEA-500` (₹240, reorder level 10), `MUG-02` (₹199, 5), `JAR-750` (₹350, 3) |
| `repository(catalogue)` | function | a fresh `InMemoryInventoryRepository` with warehouses `PNQ1` (Pune) and `BLR1` (Bengaluru), each carrying every catalogue product with zero stock |
| `clock` | function | a callable that always returns `datetime(2026, 11, 7, 9, 30)` |
| `make_service(repository, clock)` | function | a **factory**: `make_service(stock=None)` returns an `InventoryService` on that repository and clock with a new `EventBus`, after receiving each `{sku: quantity}` of `stock` into `PNQ1` |

In [239]:
@pytest.fixture(scope="session")
def catalogue() -> dict[str, Product]:
    """TEA-500 (240, reorder 10), MUG-02 (199, reorder 5), JAR-750 (350, reorder 3)."""
    raise NotImplementedError


@pytest.fixture
def repository(catalogue: dict[str, Product]) -> InMemoryInventoryRepository:
    """PNQ1 (Pune) and BLR1 (Bengaluru), each carrying every product, zero stock."""
    raise NotImplementedError


@pytest.fixture
def clock() -> Callable[[], datetime]:
    """Always returns datetime(2026, 11, 7, 9, 30)."""
    raise NotImplementedError


@pytest.fixture
def make_service(
    repository: InMemoryInventoryRepository, clock: Callable[[], datetime]
) -> Callable[..., InventoryService]:
    """Factory: make_service(stock=None) -> InventoryService, stock received into PNQ1."""
    raise NotImplementedError


INVENTORY_FIXTURES = [catalogue, repository, clock, make_service]

Check exercise 2 (each line is one hidden test that uses your fixtures):

In [240]:
FIXTURE_CHECKS = textwrap.dedent("""
SEEN_CATALOGUES = []


def test_catalogue_contents(catalogue):
    assert sorted(catalogue) == ["JAR-750", "MUG-02", "TEA-500"]
    tea = catalogue["TEA-500"]
    assert (tea.unit_price, tea.reorder_level) == (Money("240"), 10)
    mug, jar = catalogue["MUG-02"], catalogue["JAR-750"]
    assert (mug.unit_price, jar.reorder_level) == (Money("199"), 3)
    SEEN_CATALOGUES.append(catalogue)


def test_catalogue_is_session_scoped(catalogue):
    assert SEEN_CATALOGUES and SEEN_CATALOGUES[0] is catalogue


def test_repository_warehouses(repository):
    places = [(w.code, w.city) for w in repository.list_warehouses()]
    assert places == [("BLR1", "Bengaluru"), ("PNQ1", "Pune")]
    assert all(len(w) == 3 and not w for w in repository.list_warehouses())


def test_clock_is_fixed(clock):
    assert clock() == datetime(2026, 11, 7, 9, 30) == clock()


def test_make_service_receives_stock(make_service, clock):
    service = make_service({"TEA-500": 40, "MUG-02": 6})
    pune = service.repository.get_warehouse("PNQ1")
    assert (pune["TEA-500"], pune["MUG-02"], pune["JAR-750"]) == (40, 6, 0)
    assert {movement.occurred_at for movement in service.repository.movements()} == {clock()}


def test_isolation_part_1(make_service):
    service = make_service({"TEA-500": 5})
    service.ship("PNQ1", "TEA-500", 2)
    assert service.repository.get_warehouse("PNQ1")["TEA-500"] == 3


def test_isolation_part_2(repository):
    assert repository.get_warehouse("PNQ1")["TEA-500"] == 0 and repository.movements() == []


def test_factory_shares_the_repository(make_service, repository):
    first, second = make_service({"TEA-500": 40}), make_service({"TEA-500": 2})
    assert first.repository is repository is second.repository
    assert repository.get_warehouse("PNQ1")["TEA-500"] == 42


def test_each_service_has_its_own_event_bus(make_service):
    first, second = make_service(), make_service()
    alerts = []
    first.events.subscribe(LowStockAlert, alerts.append)
    first.receive("PNQ1", "MUG-02", 6)
    first.ship("PNQ1", "MUG-02", 2)
    assert first.events is not second.events
    assert [(alert.sku, alert.quantity) for alert in alerts] == [("MUG-02", 4)]
""")  # dedent: the solutions file runs this cell inside a function


def fixture_outcome(test_name: str) -> str | None:
    """Outcome of one hidden test run against the learner's fixtures."""
    return run_lab_tests(
        LAB_TEST_PRELUDE + FIXTURE_CHECKS, lab_test_source(INVENTORY_FIXTURES)
    ).outcomes.get(test_name)


run_checks(
    [
        (
            name.removeprefix("test_").replace("_", " "),
            lambda name=name: fixture_outcome(name),
            "passed",
        )
        for name in re.findall(r"^def (test_\w+)", FIXTURE_CHECKS, re.MULTILINE)
    ]
)

FAIL  catalogue contents: expected 'passed', got 'failed'
FAIL  catalogue is session scoped: expected 'passed', got 'failed'
FAIL  repository warehouses: expected 'passed', got 'failed'
FAIL  clock is fixed: expected 'passed', got 'failed'
FAIL  make service receives stock: expected 'passed', got 'failed'
FAIL  isolation part 1: expected 'passed', got 'failed'
FAIL  isolation part 2: expected 'passed', got 'failed'
FAIL  factory shares the repository: expected 'passed', got 'failed'
FAIL  each service has its own event bus: expected 'passed', got 'failed'
0/9 checks passed


#### Exercise 3 — mock an HTTP client

`ExchangeRateClient` (in `shop.rates`, imported above) calls
`http.get(url, params=..., timeout=...)` on an injected client and expects a
response with `.status_code` and `.json()`. The real service answers JSON such as
`{"rates": {"USD": 0.012}}` (rates are JSON **numbers**). Write tests that use
`unittest.mock` doubles (no network) and pass on the real client, and that catch
these planted bugs:

- the request is sent without `timeout=5.0`, or with the wrong query parameters
  (it must be `params={"base": ..., "symbols": ...}` to `<base_url>/latest`);
- the rate is converted with `Decimal(float)` (`Decimal(0.012)` is not `0.012`);
- a non-200 response is not rejected **before reading the body** (assert that
  `.json()` was not called and `RateLookupError` was raised);
- `convert` calls the service even when `base == quote`.

Put all your test functions in `RATE_TESTS`.

In [241]:
def test_rate_is_parsed_exactly() -> None:
    raise NotImplementedError


def test_rate_request_is_well_formed() -> None:
    raise NotImplementedError


def test_http_error_is_rejected_before_reading_the_body() -> None:
    raise NotImplementedError


def test_same_currency_conversion_skips_the_service() -> None:
    raise NotImplementedError


RATE_TESTS = [
    test_rate_is_parsed_exactly,
    test_rate_request_is_well_formed,
    test_http_error_is_rejected_before_reading_the_body,
    test_same_currency_conversion_skips_the_service,
]

Check exercise 3:

In [242]:
RATE_MUTANTS = {
    "missing timeout": ("shop/rates.py", ", timeout=5.0\n", "\n"),
    "wrong query parameter": ("shop/rates.py", '"symbols": quote', '"symbol": quote'),
    "Decimal(float)": (
        "shop/rates.py",
        'Decimal(str(response.json()["rates"][quote]))',
        'Decimal(response.json()["rates"][quote])',
    ),
    "no status check": ("shop/rates.py", "if response.status_code != 200:", "if False:"),
    "service called for same currency": ("shop/rates.py", "if base == quote:", "if False:"),
}
rate_source = lab_test_source(RATE_TESTS)
run_checks(
    [
        ("tests pass on the real client", lambda: run_lab_tests(rate_source).green, True),
        (
            "tests use unittest.mock",
            lambda: bool(re.search(r"\bmock\.|Mock\(|patch", "".join(map(source_of, RATE_TESTS)))),
            True,
        ),
        *[
            (f"catches: {name}", lambda mutation=mutation: catches(rate_source, mutation), True)
            for name, mutation in RATE_MUTANTS.items()
        ],
    ]
)

FAIL  tests pass on the real client: expected True, got False
FAIL  tests use unittest.mock: expected True, got False
FAIL  catches: missing timeout: expected True, got False
FAIL  catches: wrong query parameter: expected True, got False
FAIL  catches: Decimal(float): expected True, got False
FAIL  catches: no status check: expected True, got False
FAIL  catches: service called for same currency: expected True, got False
0/7 checks passed


#### Exercise 4 — Hypothesis tests for `split_bill`

Write **property-based** tests (each decorated with `@given`) for `split_bill` from
`shop.billing`. Generate totals as strings with two decimal places (for example
`st.decimals(min_value=0, max_value=100_000, places=2).map(str)`) and 1–50
people. State these properties:

1. there are exactly `people_count` shares;
2. the shares add up **exactly** to the total;
3. no two shares differ by more than one paisa;
4. the shares never increase from first to last (the leftover paise go to the
   first people);
5. `people_count < 1` raises `ValueError` (generate zero and negative counts).

The check runs them on the real function and on four mutants: the leftover goes to
the last people, the leftover is dropped, zero people are not rejected, and the
total is converted with `float`.

In [243]:
def test_share_count() -> None:
    raise NotImplementedError


def test_shares_add_up_exactly() -> None:
    raise NotImplementedError


def test_shares_differ_by_at_most_one_paisa() -> None:
    raise NotImplementedError


def test_shares_never_increase() -> None:
    raise NotImplementedError


def test_invalid_people_count_is_rejected() -> None:
    raise NotImplementedError


SPLIT_BILL_TESTS = [
    test_share_count,
    test_shares_add_up_exactly,
    test_shares_differ_by_at_most_one_paisa,
    test_shares_never_increase,
    test_invalid_people_count_is_rejected,
]

Check exercise 4:

In [244]:
BILL_MUTANTS = {
    "leftover to the last people": (
        "shop/billing.py",
        "(1 if index < leftover_paise else 0)",
        "(1 if index >= people_count - leftover_paise else 0)",
    ),
    "leftover dropped": ("shop/billing.py", "(1 if index < leftover_paise else 0)", "0"),
    "zero people accepted": ("shop/billing.py", "if people_count < 1:", "if people_count < 0:"),
    "float conversion": (
        "shop/billing.py",
        "int(Decimal(total_amount).quantize(PAISE, rounding=ROUND_HALF_UP) * 100)",
        "int(float(total_amount) * 100)",
    ),
}
bill_source = lab_test_source(SPLIT_BILL_TESTS)
hypothesis_ready = importlib.util.find_spec("hypothesis") is not None
if not hypothesis_ready:
    print("SKIP: hypothesis is not installed (uv add --dev hypothesis)")
run_checks(
    [
        (
            "every test uses @given",
            lambda: all(getattr(test, "is_hypothesis_test", False) for test in SPLIT_BILL_TESTS),
            True,
        ),
        ("tests pass on the real split_bill", lambda: run_lab_tests(bill_source).green, True),
        *[
            (f"catches: {name}", lambda mutation=mutation: catches(bill_source, mutation), True)
            for name, mutation in BILL_MUTANTS.items()
        ],
    ]
)

FAIL  every test uses @given: expected True, got False


FAIL  tests pass on the real split_bill: expected True, got False
FAIL  catches: leftover to the last people: expected True, got False
FAIL  catches: leftover dropped: expected True, got False
FAIL  catches: zero people accepted: expected True, got False
FAIL  catches: float conversion: expected True, got False
0/6 checks passed


#### Exercise 5 — a `dictConfig` with JSON output

1. `JsonFormatter.format(record)` returns **one line** of JSON with the keys
   `time` (the record's creation time in UTC, ISO 8601, e.g.
   `2026-11-07T09:30:00.123456+00:00`), `level`, `logger`, `message` (the merged
   message), every attribute added with `extra=...` (anything not a standard
   `LogRecord` attribute), and `exception` (the formatted traceback, via
   `self.formatException`) only when the record carries exception info.
2. `build_logging_config(stream)` returns a `dictConfig` dictionary: version 1,
   existing loggers kept enabled, a formatter using `JsonFormatter` (through the
   `"()"` key), one `StreamHandler` writing to `stream` at level `INFO` with a
   filter that drops records whose message contains `/health`, the logger `shop`
   at `DEBUG` using that handler without propagating, and `shop.payments` at
   `WARNING`.

The check applies your config, logs six records through `shop.orders` and
`shop.payments`, and expects exactly three JSON lines. Afterwards it removes the
handlers again.

In [245]:
from typing import IO


class JsonFormatter(logging.Formatter):
    """Format each record as one line of JSON."""

    def format(self, record: logging.LogRecord) -> str:
        raise NotImplementedError


def build_logging_config(stream: IO[str]) -> dict[str, object]:
    """A dictConfig dict: JSON lines to stream for the shop loggers."""
    raise NotImplementedError

Check exercise 5:

In [246]:
def json_log_lines() -> list[dict[str, object]]:
    """Apply the learner's config, emit six records, return the parsed JSON lines."""
    stream = io.StringIO()
    logging.config.dictConfig(build_logging_config(stream))
    orders, payments = logging.getLogger("shop.orders"), logging.getLogger("shop.payments")
    try:
        orders.debug("cart recalculated")
        orders.info("order %s placed", 7, extra={"order_id": 7, "customer": "Asha"})
        orders.info("GET /health 200")
        payments.info("payment authorised")
        payments.warning("payment retry %d/3", 2)
        try:
            _ = 1 / 0
        except ZeroDivisionError:
            orders.exception("charging order %s failed", 8)
    finally:
        for name in ("shop", "shop.orders", "shop.payments"):
            logger = logging.getLogger(name)
            for handler in logger.handlers[:]:
                logger.removeHandler(handler)
            logger.setLevel(logging.NOTSET)
            logger.propagate = True
    return [json.loads(line) for line in stream.getvalue().splitlines()]


STANDARD_KEYS = set(vars(logging.makeLogRecord({}))) - {"message"}
UTC_ISO = r"\d{4}-\d\d-\d\dT\d\d:\d\d:\d\d(\.\d+)?\+00:00"
run_checks(
    [
        (
            "three lines (DEBUG, /health and payments INFO dropped)",
            lambda: len(json_log_lines()),
            3,
        ),
        (
            "line 1 fields",
            lambda: {key: json_log_lines()[0].get(key) for key in ("level", "logger", "message")},
            {"level": "INFO", "logger": "shop.orders", "message": "order 7 placed"},
        ),
        (
            "line 1 extras",
            lambda: (json_log_lines()[0].get("order_id"), json_log_lines()[0].get("customer")),
            (7, "Asha"),
        ),
        (
            "time is UTC ISO 8601",
            lambda: all(
                re.fullmatch(UTC_ISO, str(line.get("time"))) is not None
                for line in json_log_lines()
            ),
            True,
        ),
        (
            "no standard attributes leak",
            lambda: sorted(STANDARD_KEYS & set(json_log_lines()[0])),
            [],
        ),
        ("no exception key without exc_info", lambda: "exception" in json_log_lines()[0], False),
        (
            "line 2 from shop.payments",
            lambda: (
                json_log_lines()[1].get("logger"),
                json_log_lines()[1].get("message"),
                json_log_lines()[1].get("level"),
            ),
            ("shop.payments", "payment retry 2/3", "WARNING"),
        ),
        (
            "line 3 has the traceback",
            lambda: (
                json_log_lines()[2].get("level"),
                "ZeroDivisionError" in str(json_log_lines()[2].get("exception")),
            ),
            ("ERROR", True),
        ),
    ]
)

FAIL  three lines (DEBUG, /health and payments INFO dropped): raised NotImplementedError: 
FAIL  line 1 fields: raised NotImplementedError: 
FAIL  line 1 extras: raised NotImplementedError: 
FAIL  time is UTC ISO 8601: raised NotImplementedError: 
FAIL  no standard attributes leak: raised NotImplementedError: 
FAIL  no exception key without exc_info: raised NotImplementedError: 
FAIL  line 2 from shop.payments: raised NotImplementedError: 
FAIL  line 3 has the traceback: raised NotImplementedError: 
0/8 checks passed


#### Exercise 6 — a pdb debugging walkthrough

`coupon.py` (written by the next cell) should print `61110`: two teas at ₹240 and a
mug at ₹199 are 67,900 paise, minus 10% for the `DIWALI10` coupon. It prints `0`.
Find the bug **with pdb**, not by reading:

1. `coupon_debug_commands()` returns the commands of your session. They are fed to
   `python -m pdb coupon.py`, which stops before line 1. Your session must stop
   inside `apply_coupon`, print `percent` (10) and, after the next line has run,
   `discount`, and show the call stack (`w`).
2. `coupon_bug_report()` returns `(line number of the bug, the corrected line)`.
   The check replaces that line and runs the script, which must print `61110`.

The transcript of your session is printed below the checks.

In [247]:
COUPON_SCRIPT = """
COUPONS = {"DIWALI10": 10, "FLAT5": 5}


def apply_coupon(total_paise, coupon):
    percent = COUPONS.get(coupon, 0)
    discount = total_paise * percent // 10
    return total_paise - discount


def checkout(cart, coupon):
    total = sum(price * quantity for price, quantity in cart)
    return apply_coupon(total, coupon)


print(checkout([(24000, 2), (19900, 1)], "DIWALI10"))
"""
COUPON_DIR = LAB43_DIR / "coupon"
COUPON_DIR.mkdir(parents=True, exist_ok=True)
(COUPON_DIR / "coupon.py").write_text(textwrap.dedent(COUPON_SCRIPT).lstrip("\n"))
for number, line in enumerate((COUPON_DIR / "coupon.py").read_text().splitlines(), start=1):
    print(f"{number:>2}  {line}")


def coupon_debug_commands() -> list[str]:
    """The pdb commands of your session, in order."""
    raise NotImplementedError


def coupon_bug_report() -> tuple[int, str]:
    """(line number of the bug, the corrected line)."""
    raise NotImplementedError

 1  COUPONS = {"DIWALI10": 10, "FLAT5": 5}
 2  
 3  
 4  def apply_coupon(total_paise, coupon):
 5      percent = COUPONS.get(coupon, 0)
 6      discount = total_paise * percent // 10
 7      return total_paise - discount
 8  
 9  
10  def checkout(cart, coupon):
11      total = sum(price * quantity for price, quantity in cart)
12      return apply_coupon(total, coupon)
13  
14  
15  print(checkout([(24000, 2), (19900, 1)], "DIWALI10"))


Check exercise 6:

In [248]:
def lab_pdb_session(commands: list[str]) -> list[tuple[str, str]]:
    """Run `python -m pdb coupon.py` with commands; return (command, its output) pairs."""
    completed = subprocess.run(
        [sys.executable, "-m", "pdb", "coupon.py"],
        cwd=COUPON_DIR,
        input="\n".join(commands) + "\n",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=60,
        env={**os.environ, "PYTHON_COLORS": "0", "NO_COLOR": "1"},
    )
    transcript = completed.stdout.replace(str(COUPON_DIR), "<lab>").replace(
        sys.base_prefix, "<python>"
    )
    pieces = transcript.split("(Pdb) ")
    return [
        (command, pieces[index + 1] if index + 1 < len(pieces) else "")
        for index, command in enumerate(commands)
    ]


def run_fixed_coupon(line_number: int, fixed_line: str) -> str:
    """Replace one line of coupon.py (keeping its indentation) and run the result."""
    lines = (COUPON_DIR / "coupon.py").read_text().splitlines()
    original = lines[line_number - 1]
    lines[line_number - 1] = original[: len(original) - len(original.lstrip())] + fixed_line.strip()
    (COUPON_DIR / "coupon_fixed.py").write_text("\n".join(lines) + "\n")
    completed = subprocess.run(
        [sys.executable, "coupon_fixed.py"],
        cwd=COUPON_DIR,
        capture_output=True,
        text=True,
        timeout=60,
    )
    return (completed.stdout + completed.stderr).strip()


def printed(session: list[tuple[str, str]], name: str, value: str) -> bool:
    """Did a p/pp/display command mentioning name print value?"""
    return any(
        re.match(rf"(p|pp|print|display)\b.*\b{name}\b", command.strip())
        and re.search(rf"\b{value}\b", output)
        for command, output in session
    )


run_checks(
    [
        (
            "stopped inside apply_coupon",
            lambda: any(
                "apply_coupon()" in output for _, output in lab_pdb_session(coupon_debug_commands())
            ),
            True,
        ),
        (
            "printed percent (10)",
            lambda: printed(lab_pdb_session(coupon_debug_commands()), "percent", "10"),
            True,
        ),
        (
            "printed discount (67900)",
            lambda: printed(lab_pdb_session(coupon_debug_commands()), "discount", "67900"),
            True,
        ),
        (
            "showed the call stack",
            lambda: any(
                command.strip() in {"w", "where", "bt"} and "checkout()" in output
                for command, output in lab_pdb_session(coupon_debug_commands())
            ),
            True,
        ),
        ("bug line", lambda: coupon_bug_report()[0], 6),
        ("fixed script prints 61110", lambda: run_fixed_coupon(*coupon_bug_report()), "61110"),
    ]
)
try:
    for command, output in lab_pdb_session(coupon_debug_commands()):
        print(
            f"(Pdb) {command}\n{textwrap.indent(output.rstrip(), '      ')}"
            if output.strip()
            else f"(Pdb) {command}"
        )
except Exception as error:  # the transcript must never stop Run All
    print(f"(no session yet: {type(error).__name__})")

FAIL  stopped inside apply_coupon: raised NotImplementedError: 
FAIL  printed percent (10): raised NotImplementedError: 
FAIL  printed discount (67900): raised NotImplementedError: 
FAIL  showed the call stack: raised NotImplementedError: 
FAIL  bug line: raised NotImplementedError: 
FAIL  fixed script prints 61110: raised NotImplementedError: 
0/6 checks passed
(no session yet: NotImplementedError)


#### Exercise 7 — a 90%+ coverage suite for the Phase 2 model

Write a test suite for `shop/inventory.py` (the Phase 2 model: `Product`,
`StockMovement`, `Warehouse`, the repository, `EventBus` and `InventoryService`).
Put every test function in `INVENTORY_TESTS`; you may include fixtures (for example
your exercise 2 fixtures, which pytest will find in the same file). Requirements:
at least 8 tests, all passing, **branch coverage of at least 90%** of
`shop/inventory.py`, and the suite must catch two mutants: a low-stock alert sent on
*every* sale at or below the reorder level (instead of only when the level is
crossed), and adjustments that ignore their sign.

In [249]:
def test_inventory_placeholder() -> None:
    raise NotImplementedError


INVENTORY_TESTS = [test_inventory_placeholder]

Check exercise 7 (this runs pytest with `--cov-branch` a few times; allow some seconds):

In [250]:
INVENTORY_MUTANTS = {
    "alert on every low sale": (
        "shop/inventory.py",
        "if old_quantity > reorder_level >= new_quantity:",
        "if reorder_level >= new_quantity:",
    ),
    "adjustment sign ignored": (
        "shop/inventory.py",
        "if self.movement_type is MovementType.ADJUSTED:\n                return self.quantity",
        "if self.movement_type is MovementType.ADJUSTED:\n"
        "                return abs(self.quantity)",
    ),
}
inventory_source = lab_test_source(INVENTORY_TESTS)


def inventory_coverage_run() -> LabRun:
    return run_lab_tests(inventory_source, coverage_target="shop.inventory")


run_checks(
    [
        (
            "at least 8 tests",
            lambda: (
                len(
                    [
                        test
                        for test in INVENTORY_TESTS
                        if getattr(test, "__name__", "").startswith("test_")
                    ]
                )
                >= 8
            ),
            True,
        ),
        ("all tests pass", lambda: inventory_coverage_run().green, True),
        (
            "branch coverage >= 90%",
            lambda: (inventory_coverage_run().coverage_percent or 0) >= 90,
            True,
        ),
        *[
            (
                f"catches: {name}",
                lambda mutation=mutation: catches(inventory_source, mutation),
                True,
            )
            for name, mutation in INVENTORY_MUTANTS.items()
        ],
    ]
)
try:
    print(f"branch coverage of shop/inventory.py: {inventory_coverage_run().coverage_percent}%")
except Exception as error:  # never stop Run All
    print(f"coverage not measured: {type(error).__name__}")

FAIL  at least 8 tests: expected True, got False


FAIL  all tests pass: expected True, got False
FAIL  branch coverage >= 90%: expected True, got False


FAIL  catches: alert on every low sale: expected True, got False
FAIL  catches: adjustment sign ignored: expected True, got False
0/5 checks passed
branch coverage of shop/inventory.py: 40.4%


### Recap — Unit 4.3

| Concept | One-line takeaway | Jest / JS equivalent |
| --- | --- | --- |
| Test pyramid, AAA | Many fast unit tests, fewer integration, few end-to-end; arrange, act once, assert | same |
| pytest basics | `test_*` functions, plain `assert` (rewritten), `pytest.raises`, `pytest.approx`, `-k -x -v --lf -s` | `test`, `expect`, `toThrow`, `toBeCloseTo` |
| Fixtures | Injected by parameter name; scopes; `yield` teardown; `conftest.py`; factories; `tmp_path`, `monkeypatch`, `capsys`, `caplog` | `beforeEach`/`afterEach` |
| Parametrize, markers | One function, many cases with ids; `skip`, `skipif`, `xfail(strict=True)`, registered custom markers + `-m` | `test.each`, `test.skip`, tags |
| Test doubles | Stubs and fakes check outcomes, mocks check interactions; `patch` where looked up; always `autospec`; inject clocks | `jest.fn`, `jest.mock`, fake timers |
| Coverage | Branch coverage over line coverage; enforce a threshold; coverage shows execution, not verification | `--coverage` |
| Hypothesis | Properties over generated inputs; failures shrink to minimal examples; pin them with `@example` | fast-check |
| Debugging | `breakpoint()` + pdb commands, `PYTHONBREAKPOINT`, post-mortem (`python -m pdb`, `pdb.pm`, `%debug`, `--pdb`), `faulthandler` | `debugger;`, Chrome DevTools |
| Logging | `getLogger(__name__)` everywhere, configure once with `dictConfig`; handlers, formatters, filters, propagation; `NullHandler` in libraries; lazy `%s`; JSON in production | `console.log`, pino, winston |
| Quality gates | ruff (lint + format), pre-commit locally, the same checks in CI on every push | ESLint + Prettier, husky, GitHub Actions |

**Further reading**

- [`unittest.mock`](https://docs.python.org/3/library/unittest.mock.html),
  [`pdb`](https://docs.python.org/3/library/pdb.html),
  [`faulthandler`](https://docs.python.org/3/library/faulthandler.html),
  [`sys.monitoring`](https://docs.python.org/3/library/sys.monitoring.html)
- [`logging`](https://docs.python.org/3/library/logging.html),
  [`logging.config`](https://docs.python.org/3/library/logging.config.html),
  [`logging.handlers`](https://docs.python.org/3/library/logging.handlers.html),
  [Logging HOWTO](https://docs.python.org/3/howto/logging.html),
  [Logging Cookbook](https://docs.python.org/3/howto/logging-cookbook.html)
- [pytest documentation](https://docs.pytest.org/),
  [coverage.py](https://coverage.readthedocs.io/),
  [pytest-cov](https://pytest-cov.readthedocs.io/),
  [Hypothesis](https://hypothesis.readthedocs.io/)
- [ruff rules](https://docs.astral.sh/ruff/rules/), [pre-commit](https://pre-commit.com/),
  [GitHub Actions for Python](https://docs.github.com/en/actions/use-cases-and-examples/building-and-testing/building-and-testing-python)
- [PEP 282](https://peps.python.org/pep-0282/) (logging),
  [PEP 391](https://peps.python.org/pep-0391/) (`dictConfig`),
  [PEP 553](https://peps.python.org/pep-0553/) (`breakpoint()`),
  [PEP 669](https://peps.python.org/pep-0669/) (`sys.monitoring`),
  [PEP 768](https://peps.python.org/pep-0768/) (remote debugging)

## Unit 4.4 — Standard Library Essentials

**Objectives**

- Work with files and folders through `pathlib`, and know the `os`, `sys` and
  `shutil` functions around it.
- Run other programs safely with `subprocess`.
- Handle dates and times correctly: aware datetimes, time zones, UTC storage, DST,
  ISO 8601, and the right clock for measuring time.
- Read and write regular expressions, including verbose patterns and lookarounds,
  and avoid catastrophic backtracking.
- Serialize JSON and CSV without losing precision or data.
- Reach for `heapq`, `bisect`, `statistics`, `math` and `graphlib` instead of
  writing them yourself.
- Tell `random` from `secrets`; hash, sign, encode and parse URLs.
- Build a command-line interface with `argparse`; format text with the text
  utilities.

**JS bridge: Node built-ins → the standard library**

| Node / browser | Python | The difference that bites |
| --- | --- | --- |
| `path.join`, `fs.readFileSync` | `Path("a") / "b"`, `path.read_text(encoding="utf-8")` | Paths are **objects**; always pass `encoding` |
| `child_process.execFile` / `exec` | `subprocess.run([...])` / `shell=True` | Pass a **list**; `shell=True` with user input is an injection hole |
| `Date` (always UTC inside, local outside) | `datetime` naive or **aware** | A naive datetime has no time zone at all; use `ZoneInfo` |
| `Intl.DateTimeFormat` | `strftime` | Format codes such as `%Y-%m-%d` |
| `/\d+/g`, `str.match(re)` | `re.findall(r"\d+", text)` | Raw strings; `re.match` anchors at the start; no `g` flag |
| `JSON.parse(text)` | `json.loads(text, parse_float=Decimal)` | Python can keep decimals exact |
| (npm `heap`, `lodash.sortedIndex`) | `heapq`, `bisect` | Batteries included |
| `Math.random()` | `random` (seedable) / `secrets` (crypto) | Never use `random` for tokens |
| `crypto.createHmac` | `hmac.new(key, body, "sha256")` | `hmac.compare_digest` for checks |
| `crypto.randomUUID()` | `uuid.uuid4()`, `uuid.uuid7()` (3.14) | v7 sorts by creation time |
| `new URL(...)`, `URLSearchParams` | `urllib.parse.urlsplit`, `parse_qs`, `urlencode` | Functions, not classes |
| `process.argv` + commander/yargs | `sys.argv` + `argparse` | Built in, with `--help` for free |

**Determinism.** Dates use fixed example datetimes with explicit time zones
(`FIXED_MOMENT` is 2026-11-07 09:30 UTC), `random` is seeded, file demos use a
scratch tree with fixed modification times under `notebooks/_scratch/unit44/`.
Only the `secrets` and `uuid4` demos print properties instead of values: being
unpredictable is their whole point.

In [251]:
import argparse
import base64
import bisect
import contextlib
import csv
import dataclasses
import difflib
import enum
import glob
import graphlib
import hashlib
import heapq
import hmac
import inspect
import io
import json
import math
import os
import pprint
import random
import re
import secrets
import shlex
import shutil
import statistics
import string
import subprocess
import sys
import textwrap
import time
import urllib.parse
import uuid
import warnings
from collections.abc import Callable, Iterable, Mapping
from datetime import UTC, date, datetime, timedelta
from decimal import Decimal
from pathlib import Path, PurePosixPath, PureWindowsPath
from zoneinfo import ZoneInfo

REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
UNIT44_DIR = scratch_dir / "unit44"
UNIT44_DIR.mkdir(parents=True, exist_ok=True)
IST = ZoneInfo("Asia/Kolkata")
NEW_YORK = ZoneInfo("America/New_York")
LONDON = ZoneInfo("Europe/London")
FIXED_MOMENT = datetime(2026, 11, 7, 9, 30, tzinfo=UTC)  # every "now" in this unit


def tidy(text: str) -> str:
    """Show absolute paths as <scratch>, <repo> or ~."""
    for folder, label in ((UNIT44_DIR, "<scratch>"), (REPO_ROOT, "<repo>"), (Path.home(), "~")):
        text = text.replace(str(folder), label)
    return text


def fresh_dir(name: str) -> Path:
    """An empty folder <scratch>/<name>, recreated on every run."""
    folder = UNIT44_DIR / name
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    return folder


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


SAMPLE_FILES = {
    "src/shop/__init__.py": "",
    "src/shop/orders.py": "def total(prices):\n    return sum(prices)\n",
    "tests/test_orders.py": "from shop.orders import total\n",
    "data/sales_2026-10.csv": "sku,quantity\nTEA-500,12\n",
    "data/sales_2026-11.csv": "sku,quantity\nTEA-500,7\nMUG-02,3\n",
    "docs/README.md": "# Shop\n\nPrices in ₹ (rupees).\n",
}


def make_sample_tree(name: str = "sample_tree") -> Path:
    """A small project tree with fixed contents and fixed modification times."""
    root = fresh_dir(name)
    for offset, (relative_path, text) in enumerate(sorted(SAMPLE_FILES.items())):
        path = root / relative_path
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        stamp = (FIXED_MOMENT - timedelta(days=offset)).timestamp()
        os.utime(path, (stamp, stamp))
    return root


sample_root = make_sample_tree()
print(f"{sample_root.relative_to(REPO_ROOT).as_posix()}: {len(SAMPLE_FILES)} files")
print(f"time zones: {IST}, {NEW_YORK}, {LONDON}; fixed moment {FIXED_MOMENT.isoformat()}")

notebooks/_scratch/unit44/sample_tree: 6 files
time zones: Asia/Kolkata, America/New_York, Europe/London; fixed moment 2026-11-07T09:30:00+00:00


### 4.4.1 Paths and the OS

**`pathlib`** (Python 3.4+) represents a path as an object. `Path` is the concrete
class for the current OS (`PosixPath` here, `WindowsPath` on Windows);
`PurePosixPath` and `PureWindowsPath` manipulate paths of either system without
touching the disk. The `/` operator joins.

**Joining and taking apart:**

In [252]:
report = Path("data") / "reports" / "sales_2026-11.tar.gz"
print(f"{report=}")
print(f"{report.name=}, {report.stem=}, {report.suffix=}, {report.suffixes=}")
print(f"{report.parent=}, {list(report.parents)=}")
print(f"{report.parts=}, {Path('/srv/shop').anchor=}")
print(f"{report.with_suffix('.zip')=}, {report.with_name('summary.txt')=}")
print(f"{Path('data').joinpath('2026', 'nov.csv')=}")
print(f"{PureWindowsPath(r'C:\Shop\data\nov.csv').parts=}")
print(f"{PurePosixPath('/srv/shop/data/nov.csv').relative_to('/srv/shop')=}")

report=PosixPath('data/reports/sales_2026-11.tar.gz')
report.name='sales_2026-11.tar.gz', report.stem='sales_2026-11.tar', report.suffix='.gz', report.suffixes=['.tar', '.gz']
report.parent=PosixPath('data/reports'), list(report.parents)=[PosixPath('data/reports'), PosixPath('data'), PosixPath('.')]
report.parts=('data', 'reports', 'sales_2026-11.tar.gz'), Path('/srv/shop').anchor='/'
report.with_suffix('.zip')=PosixPath('data/reports/sales_2026-11.tar.zip'), report.with_name('summary.txt')=PosixPath('data/reports/summary.txt')
Path('data').joinpath('2026', 'nov.csv')=PosixPath('data/2026/nov.csv')
PureWindowsPath(r'C:\Shop\data\nov.csv').parts=('C:\\', 'Shop', 'data', 'nov.csv')
PurePosixPath('/srv/shop/data/nov.csv').relative_to('/srv/shop')=PurePosixPath('data/nov.csv')


**Predict the output.**

```python
from pathlib import Path

archive = Path("backups/shop.tar.gz")
print(archive.suffix, archive.suffixes, archive.stem)
```

<details><summary>Answer</summary>

`.gz ['.tar', '.gz'] shop.tar`. `suffix` is only the **last** extension and `stem`
is the name without it, so multi-part extensions need `suffixes` (or
`name.removesuffix(".tar.gz")`).

</details>

Run the next cell to check your prediction.

In [253]:
archive = Path("backups/shop.tar.gz")
print(archive.suffix, archive.suffixes, archive.stem)

.gz ['.tar', '.gz'] shop.tar


**Searching: `glob`, `rglob`, `walk`.** `glob(pattern)` matches in one folder
(`**` recurses); `rglob(pattern)` is `glob("**/" + pattern)`. `Path.walk()` (*New
in 3.12*) is the pathlib version of `os.walk`: it yields `(folder, subfolders,
files)` top-down, and you can prune `subfolders` in place to skip them. Results
are sorted here because the OS returns entries in no particular order.

In [254]:
print(f"{sorted(path.name for path in sample_root.glob('data/*.csv'))=}")
print(f"{sorted(path.relative_to(sample_root).as_posix() for path in sample_root.rglob('*.py'))=}")
print(f"{sorted(path.name for path in sample_root.glob('**/sales_*'))=}")
for folder, subfolders, files in sample_root.walk():
    subfolders[:] = sorted(name for name in subfolders if name != "tests")  # prune
    print(
        f"  {folder.relative_to(sample_root).as_posix():<10} "
        f"dirs={subfolders} files={sorted(files)}"
    )

sorted(path.name for path in sample_root.glob('data/*.csv'))=['sales_2026-10.csv', 'sales_2026-11.csv']
sorted(path.relative_to(sample_root).as_posix() for path in sample_root.rglob('*.py'))=['src/shop/__init__.py', 'src/shop/orders.py', 'tests/test_orders.py']
sorted(path.name for path in sample_root.glob('**/sales_*'))=['sales_2026-10.csv', 'sales_2026-11.csv']
  .          dirs=['data', 'docs', 'src'] files=[]
  data       dirs=[] files=['sales_2026-10.csv', 'sales_2026-11.csv']
  docs       dirs=[] files=['README.md']
  src        dirs=['shop'] files=[]
  src/shop   dirs=[] files=['__init__.py', 'orders.py']


**Reading and writing.** `read_text` / `write_text` open, read or write and close
in one call; `read_bytes` / `write_bytes` for binary data. Always pass
`encoding="utf-8"`: the default depends on the platform (*Changed in 3.15*: UTF-8
mode is on by default, PEP 686, but explicit is still clearer and works on every
version). For large files, use `open()` and iterate line by line.

In [255]:
readme = sample_root / "docs" / "README.md"
print(f"{readme.read_text(encoding='utf-8')!r}")
notes = sample_root / "docs" / "notes.txt"
characters_written = notes.write_text("Diwali sale: 10% off ₹500+\n", encoding="utf-8")
print(f"{characters_written=}, {len(notes.read_bytes())=} bytes (₹ is 3 bytes in UTF-8)")
with (sample_root / "data" / "sales_2026-11.csv").open(encoding="utf-8") as sales_file:
    print(f"{[line.rstrip() for line in sales_file]=}")
print(f"{sys.flags.utf8_mode=}")

'# Shop\n\nPrices in ₹ (rupees).\n'
characters_written=27, len(notes.read_bytes())=29 bytes (₹ is 3 bytes in UTF-8)
[line.rstrip() for line in sales_file]=['sku,quantity', 'TEA-500,7', 'MUG-02,3']
sys.flags.utf8_mode=1


**`resolve`, `absolute`, relative paths and symlinks.** `absolute()` prefixes the
current folder without touching the disk; `resolve()` also follows symlinks and
removes `..`. `relative_to` computes a relative path (`walk_up=True`, *New in
3.12*, allows `..` steps), `is_relative_to` tests containment, which is how you
stop a user-supplied path from escaping a folder (`../../etc/passwd`).

In [256]:
latest = sample_root / "data" / "latest.csv"
latest.unlink(missing_ok=True)
latest.symlink_to("sales_2026-11.csv")
print(f"{latest.is_symlink()=}, {latest.resolve().name=}")
messy = sample_root / "src" / ".." / "docs" / "README.md"
print(f"{tidy(str(messy))}\n -> {tidy(str(messy.resolve()))}")
print(f"{(sample_root / 'docs').relative_to(sample_root / 'src', walk_up=True)=}")
uploads = sample_root / "uploads"
for requested in ("invoice.pdf", "../../etc/passwd"):
    inside = (uploads / requested).resolve().is_relative_to(uploads.resolve())
    print(f"  {requested!r:<20} safe={inside}")

latest.is_symlink()=True, latest.resolve().name='sales_2026-11.csv'
<scratch>/sample_tree/src/../docs/README.md
 -> <scratch>/sample_tree/docs/README.md
(sample_root / 'docs').relative_to(sample_root / 'src', walk_up=True)=PosixPath('../docs')
  'invoice.pdf'        safe=True
  '../../etc/passwd'   safe=False


**`stat`, `exists`, `is_file`, `is_dir`.** `stat()` returns size, timestamps and
permissions (one system call; *New in 3.14*: `path.info` caches the file-type
checks of a directory listing). Modification times are POSIX timestamps (seconds
since 1970 UTC): convert with `datetime.fromtimestamp(ts, UTC)`.

In [257]:
sales = sample_root / "data" / "sales_2026-11.csv"
file_status = sales.stat()
print(f"{file_status.st_size=} bytes, {file_status.st_mtime=}")
print(f"modified: {datetime.fromtimestamp(file_status.st_mtime, UTC).isoformat()}")
print(
    f"{sales.exists()=}, {sales.is_file()=}, "
    f"{sales.parent.is_dir()=}, {(sample_root / 'nope').exists()=}"
)
if sys.version_info >= (3, 14):
    entry = next(
        path for path in (sample_root / "data").iterdir() if path.name == "sales_2026-10.csv"
    )
    print(f"New in 3.14: {entry.info.is_file()=}, {entry.info.is_symlink()=}")

file_status.st_size=32 bytes, file_status.st_mtime=1793957400.0
modified: 2026-11-06T09:30:00+00:00
sales.exists()=True, sales.is_file()=True, sales.parent.is_dir()=True, (sample_root / 'nope').exists()=False
New in 3.14: entry.info.is_file()=True, entry.info.is_symlink()=False


**Creating, copying, moving, deleting.** `mkdir(parents=True, exist_ok=True)`,
`touch`, `rename`/`replace`, `unlink`, `rmdir` (empty folders only). *New in 3.14:*
`Path.copy`, `Path.copy_into` and `Path.move` cover what used to need `shutil`.

In [258]:
archive_dir = sample_root / "archive" / "2026" / "10"
archive_dir.mkdir(parents=True, exist_ok=True)
october = sample_root / "data" / "sales_2026-10.csv"
if sys.version_info >= (3, 14):
    copied = october.copy_into(archive_dir)  # New in 3.14
    moved = copied.move(archive_dir / "sales.csv")  # New in 3.14
    print(f"{copied.name=}, {moved.relative_to(sample_root).as_posix()=}")
else:
    shutil.copy2(october, archive_dir / "sales.csv")
(archive_dir / "sales.csv").unlink()
archive_dir.rmdir()
print(f"{sorted(path.name for path in (sample_root / 'archive' / '2026').iterdir())=}")

copied.name='sales_2026-10.csv', moved.relative_to(sample_root).as_posix()='archive/2026/10/sales.csv'
sorted(path.name for path in (sample_root / 'archive' / '2026').iterdir())=[]


**Under the hood: one class per OS.** `Path(...)` returns an instance of the
platform's subclass, and pure paths never touch the filesystem. Path objects use
`__slots__` (no per-instance `__dict__`) and are immutable and hashable, so they
work as dict keys.

In [259]:
print([klass.__name__ for klass in type(Path("x")).__mro__])
print(f"{hasattr(Path('x'), '__dict__')=}, {'__slots__' in vars(Path.__mro__[-2])=}")
print(f"{Path('data/nov.csv') == Path('data//nov.csv')=}, {len({Path('a/b'), Path('a') / 'b'})=}")
print(
    f"{PureWindowsPath('C:/Shop/Data.CSV') == PureWindowsPath('c:/shop/data.csv')=}"
    "  (Windows is case-insensitive)"
)

['PosixPath', 'Path', 'PurePosixPath', 'PurePath', 'object']
hasattr(Path('x'), '__dict__')=False, '__slots__' in vars(Path.__mro__[-2])=True
Path('data/nov.csv') == Path('data//nov.csv')=True, len({Path('a/b'), Path('a') / 'b'})=1
PureWindowsPath('C:/Shop/Data.CSV') == PureWindowsPath('c:/shop/data.csv')=True  (Windows is case-insensitive)


**`os` and `sys`.** `os` talks to the operating system (environment, processes,
low-level files); `sys` talks to the interpreter. The ones you meet most:

| Call | Gives |
| --- | --- |
| `os.environ`, `os.getenv(name, default)` | environment variables (strings) |
| `os.sep`, `os.linesep`, `os.pathsep` | `/`, `\n`, `:` here (`\`, `\r\n`, `;` on Windows) |
| `os.cpu_count()`, `os.process_cpu_count()` (3.13) | CPUs in the machine / usable by this process |
| `os.getpid()` | this process's id |
| `os.scandir(folder)` | fast directory listing with cached type checks |
| `os.replace(src, dst)` | atomic rename, the basis of safe file writes |
| `sys.argv`, `sys.exit(code)` | command-line arguments, exit status |
| `sys.platform`, `sys.version_info`, `sys.executable` | where and what is running |
| `sys.stdin`, `sys.stdout`, `sys.stderr` | the standard streams |

In [260]:
print(f"{os.getenv('SHOP_CURRENCY', 'INR')=}  (default used: the variable is not set)")
print(f"{os.sep=}, {os.linesep=}, {os.pathsep=}")
print(
    f"{isinstance(os.cpu_count(), int)=}, {isinstance(os.getpid(), int)=}  (values vary by machine)"
)
with os.scandir(sample_root / "data") as entries:
    print(f"{sorted((entry.name, entry.is_file()) for entry in entries)=}")
print(f"{sys.platform=}, {sys.version_info[:2]=}, {Path(sys.executable).name=}")
print(f"{sys.getrecursionlimit()=}, {sys.stdout.isatty()=}")

os.getenv('SHOP_CURRENCY', 'INR')='INR'  (default used: the variable is not set)
os.sep='/', os.linesep='\n', os.pathsep=':'
isinstance(os.cpu_count(), int)=True, isinstance(os.getpid(), int)=True  (values vary by machine)
sorted((entry.name, entry.is_file()) for entry in entries)=[('latest.csv', True), ('sales_2026-10.csv', True), ('sales_2026-11.csv', True)]
sys.platform='linux', sys.version_info[:2]=(3, 15), Path(sys.executable).name='python'
sys.getrecursionlimit()=1000, sys.stdout.isatty()=False


**Atomic writes with `os.replace`.** Writing a file in place can leave it
half-written if the program crashes. Write a temporary file in the same folder,
then `os.replace` it over the target: readers see either the old file or the new
one, never a mix (the rename is atomic on POSIX and Windows).

In [261]:
def write_atomically(target: Path, text: str) -> None:
    """Replace target's contents in one atomic step."""
    temporary = target.with_name(f".{target.name}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, target)


settings_file = sample_root / "settings.json"
write_atomically(settings_file, json.dumps({"currency": "INR", "gst": 18}))
write_atomically(settings_file, json.dumps({"currency": "INR", "gst": 12}))
print(f"{settings_file.read_text()=}, {sorted(path.name for path in sample_root.glob('.*.tmp'))=}")

settings_file.read_text()='{"currency": "INR", "gst": 12}', sorted(path.name for path in sample_root.glob('.*.tmp'))=[]


**`shutil`: high-level file operations.** `copy2` copies contents **and** metadata
(timestamps), `copytree` copies a folder (with `ignore=shutil.ignore_patterns(...)`),
`move`, `rmtree`, `make_archive` (zip/tar), `which` (find a program on `PATH`) and
`disk_usage`.

In [262]:
backup = UNIT44_DIR / "backup_tree"
shutil.rmtree(backup, ignore_errors=True)
shutil.copytree(sample_root, backup, ignore=shutil.ignore_patterns("tests", "*.tmp", "latest.csv"))
print(
    f"{sorted(path.relative_to(backup).as_posix() for path in backup.rglob('*') if path.is_file())}"
)
copied = shutil.copy2(sample_root / "docs" / "README.md", backup / "README.copy.md")
same_mtime = Path(copied).stat().st_mtime == (sample_root / "docs" / "README.md").stat().st_mtime
print(f"copy2 keeps the modification time: {same_mtime}")
archive_path = shutil.make_archive(str(UNIT44_DIR / "backup"), "zip", backup)
print(f"{Path(archive_path).name=}, {shutil.which('git') is not None=}")
usage = shutil.disk_usage(UNIT44_DIR)
print(f"{usage.used <= usage.total=}  (disk_usage values vary)")

['data/sales_2026-10.csv', 'data/sales_2026-11.csv', 'docs/README.md', 'docs/notes.txt', 'settings.json', 'src/shop/__init__.py', 'src/shop/orders.py']
copy2 keeps the modification time: True
Path(archive_path).name='backup.zip', shutil.which('git') is not None=True
usage.used <= usage.total=True  (disk_usage values vary)


#### `os.path` → `pathlib` mapping

Older code (and many answers online) use the string functions of `os.path`. Each
has a `pathlib` equivalent; one demo per row follows, grouped by purpose.

| `os.path` / `os` / `glob` | `pathlib` | Demo group |
| --- | --- | --- |
| `os.path.join(a, b)` | `Path(a) / b` | Building paths |
| `os.path.abspath(p)` | `Path(p).absolute()` | Building paths |
| `os.path.realpath(p)` | `Path(p).resolve()` | Building paths |
| `os.path.expanduser("~/x")` | `Path("~/x").expanduser()`, `Path.home()` | Building paths |
| `os.path.relpath(p, start)` | `Path(p).relative_to(start, walk_up=True)` | Building paths |
| `os.path.basename(p)` | `Path(p).name` | Taking paths apart |
| `os.path.dirname(p)` | `Path(p).parent` | Taking paths apart |
| `os.path.splitext(p)` | `Path(p).stem`, `Path(p).suffix` | Taking paths apart |
| `os.path.isabs(p)` | `Path(p).is_absolute()` | Taking paths apart |
| `os.path.exists(p)` | `Path(p).exists()` | Querying the filesystem |
| `os.path.isfile(p)`, `os.path.isdir(p)` | `.is_file()`, `.is_dir()` | Querying the filesystem |
| `os.path.getsize(p)` | `Path(p).stat().st_size` | Querying the filesystem |
| `os.path.getmtime(p)` | `Path(p).stat().st_mtime` | Querying the filesystem |
| `os.path.samefile(a, b)` | `Path(a).samefile(b)` | Querying the filesystem |
| `os.listdir(p)` | `Path(p).iterdir()` | Listing and walking |
| `glob.glob("**/*.py", recursive=True)` | `Path().glob("**/*.py")`, `rglob("*.py")` | Listing and walking |
| `os.walk(p)` | `Path(p).walk()` (3.12) | Listing and walking |
| `os.makedirs(p, exist_ok=True)` | `Path(p).mkdir(parents=True, exist_ok=True)` | Changing the filesystem |
| `os.rename(a, b)`, `os.replace(a, b)` | `Path(a).rename(b)`, `.replace(b)` | Changing the filesystem |
| `os.remove(p)` | `Path(p).unlink()` | Changing the filesystem |
| `open(p).read()` | `Path(p).read_text(encoding=...)` | Changing the filesystem |

#### Building paths

`os.path.join(a, b)` → `Path(a) / b`. Both treat an absolute second part as a
restart.

In [263]:
print(f"{os.path.join('data', 'nov.csv')=}, {Path('data') / 'nov.csv'=}")
print(f"{os.path.join('data', '/etc/hosts')=}, {Path('data') / '/etc/hosts'=}")

os.path.join('data', 'nov.csv')='data/nov.csv', Path('data') / 'nov.csv'=PosixPath('data/nov.csv')
os.path.join('data', '/etc/hosts')='/etc/hosts', Path('data') / '/etc/hosts'=PosixPath('/etc/hosts')


`os.path.abspath(p)` → `Path(p).absolute()` (no symlink resolution, no `..` removal for `absolute`):

In [264]:
with contextlib.chdir(sample_root):  # contextlib.chdir: New in 3.11
    print(f"{tidy(os.path.abspath('docs/../data'))=}")
    print(f"{tidy(str(Path('docs/../data').absolute()))=}")

tidy(os.path.abspath('docs/../data'))='<scratch>/sample_tree/data'
tidy(str(Path('docs/../data').absolute()))='<scratch>/sample_tree/docs/../data'


`os.path.realpath(p)` → `Path(p).resolve()` (follows symlinks, removes `..`):

In [265]:
link = sample_root / "data" / "latest.csv"
print(f"{Path(os.path.realpath(link)).name=}, {link.resolve().name=}")

Path(os.path.realpath(link)).name='sales_2026-11.csv', link.resolve().name='sales_2026-11.csv'


`os.path.expanduser` → `Path.expanduser()` / `Path.home()`:

In [266]:
print(f"{tidy(os.path.expanduser('~/.config/shop'))=}")
print(
    f"{tidy(str(Path('~/.config/shop').expanduser()))=}, {Path.home() == Path('~').expanduser()=}"
)

tidy(os.path.expanduser('~/.config/shop'))='~/.config/shop'
tidy(str(Path('~/.config/shop').expanduser()))='~/.config/shop', Path.home() == Path('~').expanduser()=True


`os.path.relpath(p, start)` → `Path(p).relative_to(start, walk_up=True)`:

In [267]:
print(f"{os.path.relpath('/srv/shop/docs', '/srv/shop/src')=}")
print(f"{Path('/srv/shop/docs').relative_to('/srv/shop/src', walk_up=True)=}")
print(raised_message(lambda: Path("/srv/shop/docs").relative_to("/srv/shop/src")))

os.path.relpath('/srv/shop/docs', '/srv/shop/src')='../docs'
Path('/srv/shop/docs').relative_to('/srv/shop/src', walk_up=True)=PosixPath('../docs')
ValueError: '/srv/shop/docs' is not in the subpath of '/srv/shop/src'


#### Taking paths apart

`os.path.basename(p)` → `Path(p).name` (note the trailing-slash difference):

In [268]:
print(f"{os.path.basename('data/nov.csv')=}, {Path('data/nov.csv').name=}")
print(f"{os.path.basename('data/')=}, {Path('data/').name=}")

os.path.basename('data/nov.csv')='nov.csv', Path('data/nov.csv').name='nov.csv'
os.path.basename('data/')='', Path('data/').name='data'


`os.path.dirname(p)` → `Path(p).parent`:

In [269]:
print(f"{os.path.dirname('data/2026/nov.csv')=}, {Path('data/2026/nov.csv').parent=}")
print(f"{os.path.dirname('nov.csv')=}, {Path('nov.csv').parent=}  ('' vs '.')")

os.path.dirname('data/2026/nov.csv')='data/2026', Path('data/2026/nov.csv').parent=PosixPath('data/2026')
os.path.dirname('nov.csv')='', Path('nov.csv').parent=PosixPath('.')  ('' vs '.')


`os.path.splitext(p)` → `.stem` and `.suffix`:

In [270]:
print(f"{os.path.splitext('reports/sales.tar.gz')=}")
print(f"{Path('reports/sales.tar.gz').stem=}, {Path('reports/sales.tar.gz').suffix=}")

os.path.splitext('reports/sales.tar.gz')=('reports/sales.tar', '.gz')
Path('reports/sales.tar.gz').stem='sales.tar', Path('reports/sales.tar.gz').suffix='.gz'


`os.path.isabs(p)` → `Path(p).is_absolute()`:

In [271]:
print(
    f"{os.path.isabs('/srv/shop')=}, {Path('/srv/shop').is_absolute()=}, "
    f"{Path('srv/shop').is_absolute()=}"
)
print(
    f"{PureWindowsPath('C:/shop').is_absolute()=}, "
    f"{PureWindowsPath('/shop').is_absolute()=}  (no drive)"
)

os.path.isabs('/srv/shop')=True, Path('/srv/shop').is_absolute()=True, Path('srv/shop').is_absolute()=False
PureWindowsPath('C:/shop').is_absolute()=True, PureWindowsPath('/shop').is_absolute()=False  (no drive)


#### Querying the filesystem

`os.path.exists(p)` → `Path(p).exists()`:

In [272]:
print(
    f"{os.path.exists(sample_root / 'docs')=}, {(sample_root / 'docs').exists()=}, "
    f"{(sample_root / 'missing').exists()=}"
)

os.path.exists(sample_root / 'docs')=True, (sample_root / 'docs').exists()=True, (sample_root / 'missing').exists()=False


`os.path.isfile` / `isdir` → `.is_file()` / `.is_dir()`:

In [273]:
readme = sample_root / "docs" / "README.md"
print(
    f"{os.path.isfile(readme)=}, {readme.is_file()=}, {readme.is_dir()=}, {readme.parent.is_dir()=}"
)

os.path.isfile(readme)=True, readme.is_file()=True, readme.is_dir()=False, readme.parent.is_dir()=True


`os.path.getsize(p)` → `Path(p).stat().st_size`:

In [274]:
print(f"{os.path.getsize(readme)=}, {readme.stat().st_size=}")

os.path.getsize(readme)=32, readme.stat().st_size=32


`os.path.getmtime(p)` → `Path(p).stat().st_mtime`:

In [275]:
print(f"{os.path.getmtime(readme) == readme.stat().st_mtime=}")
print(f"{datetime.fromtimestamp(readme.stat().st_mtime, UTC).date()=}")

os.path.getmtime(readme) == readme.stat().st_mtime=True
datetime.fromtimestamp(readme.stat().st_mtime, UTC).date()=datetime.date(2026, 11, 5)


`os.path.samefile(a, b)` → `Path(a).samefile(b)`:

In [276]:
link, target = sample_root / "data" / "latest.csv", sample_root / "data" / "sales_2026-11.csv"
print(f"{os.path.samefile(link, target)=}, {link.samefile(target)=}, {link == target=}")

os.path.samefile(link, target)=True, link.samefile(target)=True, link == target=False


#### Listing and walking

`os.listdir(p)` → `Path(p).iterdir()` (names vs `Path` objects):

In [277]:
print(f"{sorted(os.listdir(sample_root / 'docs'))=}")
print(f"{sorted(path.name for path in (sample_root / 'docs').iterdir())=}")

sorted(os.listdir(sample_root / 'docs'))=['README.md', 'notes.txt']
sorted(path.name for path in (sample_root / 'docs').iterdir())=['README.md', 'notes.txt']


`glob.glob(..., recursive=True)` → `Path.glob` / `rglob`:

In [278]:
with contextlib.chdir(sample_root):
    print(f"{sorted(glob.glob('**/*.py', recursive=True))=}")
    print(f"{sorted(path.as_posix() for path in Path().rglob('*.py'))=}")

sorted(glob.glob('**/*.py', recursive=True))=['src/shop/__init__.py', 'src/shop/orders.py', 'tests/test_orders.py']
sorted(path.as_posix() for path in Path().rglob('*.py'))=['src/shop/__init__.py', 'src/shop/orders.py', 'tests/test_orders.py']


`os.walk(p)` → `Path(p).walk()` (strings vs `Path` objects for the folder):

In [279]:
walked_with_os = sorted(
    os.path.relpath(folder, sample_root) for folder, _, _ in os.walk(sample_root)
)
walked_with_path = sorted(
    folder.relative_to(sample_root).as_posix() for folder, _, _ in sample_root.walk()
)
print(f"{walked_with_os=}")
print(f"{walked_with_os == walked_with_path=}")

walked_with_os=['.', 'archive', 'archive/2026', 'data', 'docs', 'src', 'src/shop', 'tests']
walked_with_os == walked_with_path=True


#### Changing the filesystem

`os.makedirs(p, exist_ok=True)` → `Path(p).mkdir(parents=True, exist_ok=True)`:

In [280]:
os.makedirs(sample_root / "out" / "a", exist_ok=True)
(sample_root / "out" / "b" / "c").mkdir(parents=True, exist_ok=True)
out_folders = [
    folder.relative_to(sample_root).as_posix() for folder, _, _ in (sample_root / "out").walk()
]
print(f"{sorted(out_folders)=}")

sorted(out_folders)=['out', 'out/a', 'out/b', 'out/b/c']


`os.rename` / `os.replace` → `Path.rename` / `Path.replace` (both return the new path):

In [281]:
draft = sample_root / "out" / "draft.txt"
draft.write_text("v1", encoding="utf-8")
final = draft.rename(sample_root / "out" / "final.txt")
(sample_root / "out" / "draft.txt").write_text("v2", encoding="utf-8")
os.replace(sample_root / "out" / "draft.txt", final)
print(f"{final.name=}, {final.read_text()=}, {(sample_root / 'out' / 'draft.txt').exists()=}")

final.name='final.txt', final.read_text()='v2', (sample_root / 'out' / 'draft.txt').exists()=False


`os.remove(p)` → `Path(p).unlink()` (`missing_ok=True` ignores a missing file):

In [282]:
final.unlink()
print(tidy(raised_message(lambda: os.remove(final))))
final.unlink(missing_ok=True)
print(f"{final.exists()=}")

FileNotFoundError: [Errno 2] No such file or directory: '<scratch>/sample_tree/out/final.txt'
final.exists()=False


`open(p).read()` → `Path(p).read_text(encoding=...)` (the file is closed for you):

In [283]:
with open(readme, encoding="utf-8") as readme_file:
    via_open = readme_file.read()
print(
    f"{via_open == readme.read_text(encoding='utf-8')=}, "
    f"{readme.read_text(encoding='utf-8').splitlines()[0]=}"
)

via_open == readme.read_text(encoding='utf-8')=True, readme.read_text(encoding='utf-8').splitlines()[0]='# Shop'


### 4.4.2 Processes

**`subprocess.run`** starts a program, waits for it and returns a
`CompletedProcess`. Pass the command as a **list** (program, then each argument);
`capture_output=True` collects stdout and stderr, `text=True` decodes them to
`str`. The demos run Python children (`sys.executable`) so they behave the same on
every machine.

In [284]:
completed = subprocess.run(
    [sys.executable, "-c", "import sys; print('stock ok'); print('2 warnings', file=sys.stderr)"],
    capture_output=True,
    text=True,
)
print(f"{completed.returncode=}")
print(f"{completed.stdout=}")
print(f"{completed.stderr=}")
print(f"{completed.args[1:]=}")

completed.returncode=0
completed.stdout='stock ok\n'
completed.stderr='2 warnings\n'
completed.args[1:]=['-c', "import sys; print('stock ok'); print('2 warnings', file=sys.stderr)"]


**`check=True`** raises `CalledProcessError` on a non-zero exit code (the exception
carries `returncode`, `stdout` and `stderr`), so failures cannot be ignored by
accident. Without it you must test `returncode` yourself.

In [285]:
try:
    subprocess.run(
        [sys.executable, "-c", "import sys; sys.exit('database migration failed')"],
        capture_output=True,
        text=True,
        check=True,
    )
except subprocess.CalledProcessError as error:
    print(f"{type(error).__name__}: exit {error.returncode}, stderr={error.stderr.strip()!r}")

CalledProcessError: exit 1, stderr='database migration failed'


**`timeout=`** kills the child if it runs too long and raises `TimeoutExpired`.
Always set one for programs that talk to the network or might hang.

In [286]:
started = time.monotonic()
try:
    subprocess.run([sys.executable, "-c", "import time; time.sleep(10)"], timeout=0.5)
except subprocess.TimeoutExpired as error:
    print(
        f"{type(error).__name__}: {error.timeout=}s, "
        f"gave up after ~{time.monotonic() - started:.1f}s"
    )

TimeoutExpired: error.timeout=0.5s, gave up after ~0.5s


`input=` sends data to the child's stdin, `cwd=` sets its working folder and
`env=` its environment (start from `os.environ` and add to it; an `env` without
`PATH` breaks most programs).

In [287]:
counter = (
    "import os, sys; lines = sys.stdin.read().splitlines(); "
    "print(os.environ['SHOP_REGION'], os.getcwd().endswith('data'), len(lines))"
)
completed = subprocess.run(
    [sys.executable, "-c", counter],
    input="TEA-500\nMUG-02\nJAR-750\n",
    cwd=sample_root / "data",
    env={**os.environ, "SHOP_REGION": "west"},
    capture_output=True,
    text=True,
    check=True,
)
print(f"{completed.stdout.strip()=}")

completed.stdout.strip()='west True 3'


**`Popen`** is the lower-level class `run` is built on. Use it to **stream** output
while the child runs (a progress log, a long build), to talk to a child
interactively, or to connect two processes with a pipe. `poll()` checks without
waiting, `wait()` waits, `communicate()` sends input and collects all output.

In [288]:
worker = (
    "import time\nfor step in range(1, 4):\n"
    "    print(f'imported batch {step}/3', flush=True)\n    time.sleep(0.1)\n"
)
with subprocess.Popen([sys.executable, "-c", worker], stdout=subprocess.PIPE, text=True) as process:
    print(f"running: {process.poll() is None}")
    for line in process.stdout:
        print(f"  live: {line.rstrip()}")
print(f"{process.returncode=}")

producer = subprocess.Popen(
    [sys.executable, "-c", "print('TEA-500\\nMUG-02\\nTEA-500')"], stdout=subprocess.PIPE
)
consumer = subprocess.Popen(
    [
        sys.executable,
        "-c",
        "import sys, collections; print(dict(collections.Counter(sys.stdin.read().split())))",
    ],
    stdin=producer.stdout,
    stdout=subprocess.PIPE,
    text=True,
)
producer.stdout.close()  # so the producer gets SIGPIPE if the consumer exits early
output, _ = consumer.communicate(timeout=30)
producer.wait()
print(f"pipeline (producer | consumer): {output.strip()}")

running: True
  live: imported batch 1/3
  live: imported batch 2/3


  live: imported batch 3/3
process.returncode=0
pipeline (producer | consumer): {'TEA-500': 2, 'MUG-02': 1}


**The `shell=True` injection risk.** With `shell=True` the command is a single
string run by `/bin/sh`, so shell syntax in any part you interpolate is
*executed*: `;`, `&&`, `$(...)`, backticks. A file name supplied by a user becomes
a command. (`wc -c` counts bytes; the "file name" below smuggles in a second
command, a harmless `echo` here, `rm -rf` in an attack.)

In [289]:
user_supplied_name = "notes.txt; echo INJECTED: this command was not supposed to run"
unsafe = subprocess.run(
    f"wc -c {user_supplied_name}", shell=True, capture_output=True, text=True, cwd=sample_root
)
print("shell=True:")
print(textwrap.indent((unsafe.stdout + unsafe.stderr).strip(), "  "))
safe = subprocess.run(
    ["wc", "-c", user_supplied_name], capture_output=True, text=True, cwd=sample_root
)
print("list form:")
print(textwrap.indent((safe.stdout + safe.stderr).strip(), "  "))
print(f"if you really need a shell, quote: {shlex.quote(user_supplied_name)}")

shell=True:
  INJECTED: this command was not supposed to run
  wc: notes.txt: No such file or directory
list form:
  wc: 'notes.txt; echo INJECTED: this command was not supposed to run': No such file or directory
if you really need a shell, quote: 'notes.txt; echo INJECTED: this command was not supposed to run'


In the list form the whole string is one argument, a (non-existent) file name, and
no shell ever sees it. Rules: pass a list; never build a shell string from input;
if you need shell features (pipes, globbing), do them in Python (`Popen` pipes,
`Path.glob`) or `shlex.quote` every interpolated value. ruff's `S602`/`S604`
rules flag `shell=True` (4.3.10). `shutil.which("git")` finds a program before you
run it. For many concurrent children, `asyncio.create_subprocess_exec` (Phase 5).

### 4.4.3 Dates and times

A `datetime` is either **naive** (no `tzinfo`: "09:30", but where?) or **aware**
(carries a time zone, so it denotes one exact instant). JavaScript's `Date` is
always an instant; Python's naive datetimes are a common source of bugs. Rules:
use aware datetimes for anything that happened or will happen, store and compute
in **UTC**, and convert to a local zone only for display.

Naive and aware datetimes cannot be compared or subtracted:

In [290]:
naive = datetime(2026, 11, 7, 9, 30)
aware = datetime(2026, 11, 7, 9, 30, tzinfo=UTC)
print(f"{naive.tzinfo=}, {aware.tzinfo=}, {aware.utcoffset()=}")
print(f"{naive.isoformat()=}, {aware.isoformat()=}")
print(raised_message(lambda: naive < aware))
print(raised_message(lambda: aware - naive))

naive.tzinfo=None, aware.tzinfo=datetime.timezone.utc, aware.utcoffset()=datetime.timedelta(0)
naive.isoformat()='2026-11-07T09:30:00', aware.isoformat()='2026-11-07T09:30:00+00:00'
TypeError: can't compare offset-naive and offset-aware datetimes
TypeError: can't subtract offset-naive and offset-aware datetimes


**Getting "now".** `datetime.now(UTC)` gives an aware UTC timestamp.
`datetime.utcnow()` returned a **naive** datetime that merely happened to show UTC,
and is deprecated since 3.12. This unit never prints the real time; it uses
`FIXED_MOMENT` (in real code, inject a clock, Unit 4.3.5).

In [291]:
print(f"{datetime.now(UTC).tzinfo=}  (the value itself changes every run)")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    legacy = datetime.utcnow()
print(f"{legacy.tzinfo=}, warning: {caught[0].category.__name__}: {str(caught[0].message)[:60]}...")

datetime.now(UTC).tzinfo=datetime.timezone.utc  (the value itself changes every run)
legacy.tzinfo=None, warning: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for r...


**`zoneinfo`** (3.9+) provides real time zones from the IANA database (the system's,
or the `tzdata` package on Windows). India Standard Time is `Asia/Kolkata`: UTC+05:30
all year, no daylight saving. `astimezone(zone)` converts an instant to another
zone; the instant stays the same, only the wall-clock reading changes.

In [292]:
meeting_ist = datetime(2026, 11, 7, 15, 0, tzinfo=IST)
print(f"{meeting_ist.isoformat()=}, {meeting_ist.tzname()=}, {meeting_ist.utcoffset()=}")
print(f"{meeting_ist.astimezone(UTC).isoformat()=}")
for zone in (LONDON, NEW_YORK):
    local = meeting_ist.astimezone(zone)
    print(f"  {str(zone):<17} {local:%a %d %b %H:%M} {local.tzname()}")
print(f"{meeting_ist == meeting_ist.astimezone(NEW_YORK)=}  (same instant)")

meeting_ist.isoformat()='2026-11-07T15:00:00+05:30', meeting_ist.tzname()='IST', meeting_ist.utcoffset()=datetime.timedelta(seconds=19800)
meeting_ist.astimezone(UTC).isoformat()='2026-11-07T09:30:00+00:00'
  Europe/London     Sat 07 Nov 09:30 GMT
  America/New_York  Sat 07 Nov 04:30 EST
meeting_ist == meeting_ist.astimezone(NEW_YORK)=True  (same instant)


**Store UTC, display local.** Databases, logs and APIs should hold UTC (or an ISO
8601 string with an offset); convert at the edges. A naive `replace(tzinfo=...)`
*labels* a datetime without converting it, which is right only when you know the
naive value was in that zone.

In [293]:
order_placed_utc = FIXED_MOMENT
stored = order_placed_utc.isoformat()
print(f"{stored=}  (what goes into the database)")
loaded = datetime.fromisoformat(stored)
print(f"shown to an Indian customer: {loaded.astimezone(IST):%d %b %Y, %I:%M %p %Z}")
form_input = datetime(2026, 11, 7, 15, 0)  # naive, typed by a user in Pune
print(f"{form_input.replace(tzinfo=IST).astimezone(UTC).isoformat()=}  (label, then convert)")
print(f"{form_input.replace(tzinfo=UTC).isoformat()=}  (wrong label: 5h30m off)")

stored='2026-11-07T09:30:00+00:00'  (what goes into the database)
shown to an Indian customer: 07 Nov 2026, 03:00 PM IST
form_input.replace(tzinfo=IST).astimezone(UTC).isoformat()='2026-11-07T09:30:00+00:00'  (label, then convert)
form_input.replace(tzinfo=UTC).isoformat()='2026-11-07T15:00:00+00:00'  (wrong label: 5h30m off)


**`timedelta`** is a duration (days, seconds, microseconds internally). Adding it to
a datetime moves it; subtracting two datetimes gives one. `total_seconds()` gives
the whole duration as a float; dividing two timedeltas gives a ratio.

In [294]:
delivery_window = timedelta(days=2, hours=6)
print(f"{delivery_window=}, {delivery_window.total_seconds()=}")
print(f"{FIXED_MOMENT + delivery_window=}")
print(f"{timedelta(hours=25)=}  (normalized to days + seconds)")
print(
    f"{date(2026, 12, 25) - date(2026, 11, 7)=}, {(date(2026, 12, 25) - date(2026, 11, 7)).days=}"
)
print(
    f"{timedelta(hours=8) / timedelta(minutes=45)=}, {timedelta(hours=8) // timedelta(minutes=45)=}"
)

delivery_window=datetime.timedelta(days=2, seconds=21600), delivery_window.total_seconds()=194400.0
FIXED_MOMENT + delivery_window=datetime.datetime(2026, 11, 9, 15, 30, tzinfo=datetime.timezone.utc)
timedelta(hours=25)=datetime.timedelta(days=1, seconds=3600)  (normalized to days + seconds)
date(2026, 12, 25) - date(2026, 11, 7)=datetime.timedelta(days=48), (date(2026, 12, 25) - date(2026, 11, 7)).days=48
timedelta(hours=8) / timedelta(minutes=45)=10.666666666666666, timedelta(hours=8) // timedelta(minutes=45)=10


**Predict the output.**

```python
from datetime import timedelta

print(timedelta(hours=-1))
```

<details><summary>Answer</summary>

`-1 day, 23:00:00`. A `timedelta` is normalized so that only `days` can be negative;
`seconds` is always `0 <= s < 86400`. Minus one hour is stored as minus one day plus
23 hours. Use `total_seconds()` (`-3600.0`) when you need the plain number.

</details>

Run the next cell to check your prediction.

In [295]:
print(timedelta(hours=-1), timedelta(hours=-1).total_seconds())

-1 day, 23:00:00 -3600.0


**DST pitfalls.** Zones with daylight saving time skip an hour in spring and repeat
one in autumn. In New York in 2026 clocks jump from 02:00 to 03:00 on 8 March and
fall back from 02:00 to 01:00 on 1 November. Three consequences:

1. **Non-existent times.** 02:30 on 8 March never happens there. `ZoneInfo` still
   builds the datetime, with an offset that does not round-trip; detect it by
   converting to UTC and back.
2. **Ambiguous times.** 01:30 on 1 November happens twice. `fold=0` picks the
   first (EDT, -04:00), `fold=1` the second (EST, -05:00).
3. **Arithmetic is wall-clock arithmetic** within one zone: adding `timedelta(days=1)`
   keeps the local time and changes the offset, so only 23 hours pass across the
   spring change. Subtracting two datetimes **with the same `tzinfo`** also ignores
   the offset change. Convert to UTC for elapsed time.

In [296]:
gap = datetime(2026, 3, 8, 2, 30, tzinfo=NEW_YORK)
round_trip = gap.astimezone(UTC).astimezone(NEW_YORK)
print(
    f"non-existent: {gap.isoformat()} -> {round_trip.isoformat()} "
    f"(exists: {gap.replace(tzinfo=None) == round_trip.replace(tzinfo=None)})"
)

first = datetime(2026, 11, 1, 1, 30, tzinfo=NEW_YORK, fold=0)
second = first.replace(fold=1)
print(
    f"ambiguous: fold=0 {first.isoformat()} {first.tzname()}, "
    f"fold=1 {second.isoformat()} {second.tzname()}"
)

saturday_noon = datetime(2026, 3, 7, 12, 0, tzinfo=NEW_YORK)
sunday_noon = saturday_noon + timedelta(days=1)
print(f"{sunday_noon.isoformat()=}")
print(f"same-zone subtraction: {sunday_noon - saturday_noon}")
print(f"real elapsed (UTC):    {sunday_noon.astimezone(UTC) - saturday_noon.astimezone(UTC)}")

non-existent: 2026-03-08T02:30:00-05:00 -> 2026-03-08T03:30:00-04:00 (exists: False)
ambiguous: fold=0 2026-11-01T01:30:00-04:00 EDT, fold=1 2026-11-01T01:30:00-05:00 EST
sunday_noon.isoformat()='2026-03-08T12:00:00-04:00'
same-zone subtraction: 1 day, 0:00:00
real elapsed (UTC):    23:00:00


**ISO 8601** is the interchange format: `2026-11-07T09:30:00+00:00`.
`isoformat()` writes it (`timespec=` controls precision), `fromisoformat()` reads it
(since 3.11 almost any ISO 8601 form, including a trailing `Z`).
`isocalendar()` gives ISO year, week and weekday (weeks start on Monday).

In [297]:
print(f"{FIXED_MOMENT.isoformat()=}")
print(f"{FIXED_MOMENT.isoformat(timespec='minutes')=}, {FIXED_MOMENT.date().isoformat()=}")
print(f"{datetime.fromisoformat('2026-11-07T09:30:00Z')=}")
print(f"{datetime.fromisoformat('2026-11-07T15:00+05:30').astimezone(UTC)=}")
print(f"{date.fromisoformat('2026-11-07').isocalendar()=}")

FIXED_MOMENT.isoformat()='2026-11-07T09:30:00+00:00'
FIXED_MOMENT.isoformat(timespec='minutes')='2026-11-07T09:30+00:00', FIXED_MOMENT.date().isoformat()='2026-11-07'
datetime.fromisoformat('2026-11-07T09:30:00Z')=datetime.datetime(2026, 11, 7, 9, 30, tzinfo=datetime.timezone.utc)
datetime.fromisoformat('2026-11-07T15:00+05:30').astimezone(UTC)=datetime.datetime(2026, 11, 7, 9, 30, tzinfo=datetime.timezone.utc)
date.fromisoformat('2026-11-07').isocalendar()=datetime.IsoCalendarDate(year=2026, week=45, weekday=6)


**`strftime` / `strptime`** format and parse with codes (the same codes as C):

| Code | Meaning | Example |
| --- | --- | --- |
| `%Y` `%m` `%d` | year, month, day (zero-padded) | `2026` `11` `07` |
| `%H` `%M` `%S` `%f` | hour (24h), minute, second, microsecond | `09` `30` `00` `000000` |
| `%I` `%p` | hour (12h), AM/PM | `09` `AM` |
| `%a` `%A` `%b` `%B` | weekday and month names (locale-dependent) | `Sat` `Saturday` `Nov` `November` |
| `%j` | day of the year | `311` |
| `%z` `%Z` | UTC offset, zone name | `+0000` `UTC` |
| `%%` | a literal `%` | `%` |

f-strings accept the same codes after a colon: `f"{moment:%d %b %Y}"`.
`strptime` returns a naive datetime unless the text has `%z`. *New in 3.14:*
`date.strptime` and `time.strptime` classmethods.

In [298]:
print(f"{FIXED_MOMENT.strftime('%A %d %B %Y, %H:%M %Z')=}")
print(f"{FIXED_MOMENT:%Y-%m-%d %I:%M %p (day %j)}")
parsed = datetime.strptime("07/11/2026 15:00 +0530", "%d/%m/%Y %H:%M %z")
print(f"{parsed=}, {parsed.astimezone(UTC).isoformat()=}")
print(raised_message(lambda: datetime.strptime("2026-13-01", "%Y-%m-%d")))
if sys.version_info >= (3, 14):
    print(f"New in 3.14: {date.strptime('07 Nov 2026', '%d %b %Y')=}")

FIXED_MOMENT.strftime('%A %d %B %Y, %H:%M %Z')='Saturday 07 November 2026, 09:30 UTC'
2026-11-07 09:30 AM (day 311)
parsed=datetime.datetime(2026, 11, 7, 15, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=19800))), parsed.astimezone(UTC).isoformat()='2026-11-07T09:30:00+00:00'
ValueError: time data '2026-13-01' does not match format '%Y-%m-%d'
New in 3.14: date.strptime('07 Nov 2026', '%d %b %Y')=datetime.date(2026, 11, 7)


**Measuring time: `time.monotonic` vs `perf_counter` (vs `time.time`).**
`time.time()` is the wall clock: it can jump when NTP corrects it or an
administrator changes it, so never use it for durations. `monotonic()` never goes
backwards: use it for timeouts and deadlines. `perf_counter()` is monotonic with
the highest available resolution: use it for benchmarks. Their absolute values
mean nothing; only differences do. (The clock details below vary by OS; on Linux
both use `CLOCK_MONOTONIC`.)

In [299]:
for clock_name in ("time", "monotonic", "perf_counter"):
    info = time.get_clock_info(clock_name)
    print(
        f"{clock_name:<13} monotonic={info.monotonic!s:<5} "
        f"adjustable={info.adjustable!s:<5} resolution={info.resolution:g}"
    )

deadline = time.monotonic() + 0.2  # a timeout: monotonic
started = time.perf_counter()  # a measurement: perf_counter
while time.monotonic() < deadline:
    time.sleep(0.01)
print(f"waited about {time.perf_counter() - started:.1f} s")

time          monotonic=False adjustable=True  resolution=1e-09
monotonic     monotonic=True  adjustable=False resolution=1e-09
perf_counter  monotonic=True  adjustable=False resolution=1e-09


waited about 0.2 s


### 4.4.4 Regular expressions (`re`)

Write patterns as **raw strings** (`r"\d+"`) so backslashes reach `re` unchanged.
Three functions differ only in *where* the match must be:

| Function | Matches | JS equivalent |
| --- | --- | --- |
| `re.match(p, s)` | at the **start** of `s` | `/^p/.exec(s)` |
| `re.search(p, s)` | **anywhere** in `s` (first match) | `/p/.exec(s)` |
| `re.fullmatch(p, s)` | the **whole** of `s` | `/^p$/.test(s)` |

They return a `Match` object or `None`.

In [300]:
order_text = "Order 42 shipped on 2026-11-07"
print(f"{re.match(r'\d+', order_text)=}")
print(f"{re.search(r'\d+', order_text)=}")
print(f"{re.fullmatch(r'\d+', '42')=}, {re.fullmatch(r'\d+', '42a')=}")
found = re.search(r"\d{4}-\d{2}-\d{2}", order_text)
print(f"{found.group()=}, {found.start()=}, {found.end()=}, {found.span()=}")

re.match(r'\d+', order_text)=None
re.search(r'\d+', order_text)=<re.Match object; span=(6, 8), match='42'>
re.fullmatch(r'\d+', '42')=<re.Match object; span=(0, 2), match='42'>, re.fullmatch(r'\d+', '42a')=None
found.group()='2026-11-07', found.start()=20, found.end()=30, found.span()=(20, 30)


**Predict the output.**

```python
import re

print(re.match(r"\d+", "SKU 42"), re.search(r"\d+", "SKU 42").group())
```

<details><summary>Answer</summary>

`None 42`. `re.match` only tries at position 0, where `S` is not a digit; it is
not "match anywhere" as in JavaScript's `String.prototype.match`. `re.search` scans
and finds `42`.

</details>

Run the next cell to check your prediction.

In [301]:
print(re.match(r"\d+", "SKU 42"), re.search(r"\d+", "SKU 42").group())

None 42


**All matches: `findall` and `finditer`.** `findall` returns a list of strings (or
of tuples when the pattern has several groups); `finditer` yields `Match` objects
lazily, with positions. Python has no `g` flag: these functions are "global" by
design.

In [302]:
invoice = "TEA-500 x2 @240, MUG-02 x1 @199, JAR-750 x3 @350"
print(f"{re.findall(r'[A-Z]{3}-\d+', invoice)=}")
print(f"{re.findall(r'([A-Z]{3}-\d+) x(\d+)', invoice)=}")
for line_match in re.finditer(r"@(\d+)", invoice):
    print(f"  price {line_match.group(1)} at {line_match.span()}")

re.findall(r'[A-Z]{3}-\d+', invoice)=['TEA-500', 'MUG-02', 'JAR-750']
re.findall(r'([A-Z]{3}-\d+) x(\d+)', invoice)=[('TEA-500', '2'), ('MUG-02', '1'), ('JAR-750', '3')]
  price 240 at (11, 15)
  price 199 at (27, 31)
  price 350 at (44, 48)


**Groups and named groups.** Parentheses capture; `(?P<name>...)` names a group
(Python syntax; JavaScript's `(?<name>...)` is a syntax error in Python's `re`),
`(?:...)` groups without capturing. `groupdict()` returns the named groups as a
dict; `(?P=name)` matches the same text again.

In [303]:
line_pattern = re.compile(r"(?P<sku>[A-Z]{3}-\d+) x(?P<quantity>\d+) @(?P<price>\d+)")
first_line = line_pattern.search(invoice)
print(f"{first_line.group('sku')=}, {first_line['quantity']=}, {first_line.groups()=}")
print(f"{first_line.groupdict()=}")
print(f"{re.findall(r'(?:TEA|MUG)-(\d+)', invoice)=}  (non-capturing group)")
print(f"{re.search(r'(?P<word>\b\w+\b) (?P=word)', 'please ship ship today').group()=}")
print(raised_message(lambda: re.compile(r"(?<sku>[A-Z]+)")))

first_line.group('sku')='TEA-500', first_line['quantity']='2', first_line.groups()=('TEA-500', '2', '240')
first_line.groupdict()={'sku': 'TEA-500', 'quantity': '2', 'price': '240'}
re.findall(r'(?:TEA|MUG)-(\d+)', invoice)=['500', '02']  (non-capturing group)
re.search(r'(?P<word>\b\w+\b) (?P=word)', 'please ship ship today').group()='ship ship'
PatternError: unknown extension ?<s at position 1


**Lazy quantifiers.** `*`, `+`, `?` and `{m,n}` are **greedy**: they take as much as
possible and give back only if the rest fails. Adding `?` (`*?`, `+?`) makes them
**lazy**: as little as possible.

In [304]:
html = "<b>Masala tea</b> and <b>Steel mug</b>"
print(f"{re.findall(r'<b>(.*)</b>', html)=}   greedy")
print(f"{re.findall(r'<b>(.*?)</b>', html)=}   lazy")
print(f"{re.findall(r'<b>([^<]*)</b>', html)=}   negated class (often the best choice)")

re.findall(r'<b>(.*)</b>', html)=['Masala tea</b> and <b>Steel mug']   greedy
re.findall(r'<b>(.*?)</b>', html)=['Masala tea', 'Steel mug']   lazy
re.findall(r'<b>([^<]*)</b>', html)=['Masala tea', 'Steel mug']   negated class (often the best choice)


**Anchors.** `^` and `$` match at the start and end of the string (and of each line
with `re.MULTILINE`); `\A` and `\Z` always mean the whole string; `\b` is a word
boundary. Python's `$` also matches **before a final newline**, so validate input
with `fullmatch` or `\Z`.

In [305]:
log = "ERROR disk full\nINFO started\nERROR timeout"
print(f"{re.findall(r'^ERROR (.+)$', log)=}  (no MULTILINE: none)")
print(f"{re.findall(r'^ERROR (.+)$', log, re.MULTILINE)=}")
print(f"{bool(re.match(r'\d{6}$', '411001\n'))=}  ($ before a trailing newline)")
print(f"{bool(re.match(r'\d{6}\Z', '411001\n'))=}, {bool(re.fullmatch(r'\d{6}', '411001\n'))=}")
print(f"{re.findall(r'\bcat\b', 'cat category concat cat.')=}")

re.findall(r'^ERROR (.+)$', log)=[]  (no MULTILINE: none)
re.findall(r'^ERROR (.+)$', log, re.MULTILINE)=['disk full', 'timeout']
bool(re.match(r'\d{6}$', '411001\n'))=True  ($ before a trailing newline)
bool(re.match(r'\d{6}\Z', '411001\n'))=False, bool(re.fullmatch(r'\d{6}', '411001\n'))=False
re.findall(r'\bcat\b', 'cat category concat cat.')=['cat', 'cat']


**Flags** change how the pattern is read: `re.IGNORECASE` (`re.I`), `re.MULTILINE`
(`re.M`), `re.DOTALL` (`re.S`: `.` also matches newlines), `re.ASCII` (`re.A`:
`\d`, `\w`, `\b` ASCII-only), `re.VERBOSE` (`re.X`). Combine with `|`, or put them
inline at the start of the pattern: `(?i)`, `(?m)`, `(?s)`.

In [306]:
note = "Coupon: diwali10\nvalid till Nov"
print(f"{re.findall(r'coupon: (\w+)', note)=}, {re.findall(r'coupon: (\w+)', note, re.I)=}")
print(f"{re.search(r'Coupon.*Nov', note)=}, {re.search(r'Coupon.*Nov', note, re.S).group()!r}")
print(f"{re.findall(r'(?i)NOV', note)=}  (inline flag)")
print(f"{re.findall(r'\d+', 'Order ४२ and 42')=}, {re.findall(r'\d+', 'Order ४२ and 42', re.A)=}")

re.findall(r'coupon: (\w+)', note)=[], re.findall(r'coupon: (\w+)', note, re.I)=['diwali10']
re.search(r'Coupon.*Nov', note)=None, 'Coupon: diwali10\nvalid till Nov'
re.findall(r'(?i)NOV', note)=['Nov']  (inline flag)
re.findall(r'\d+', 'Order ४२ and 42')=['४२', '42'], re.findall(r'\d+', 'Order ४२ and 42', re.A)=['42']


**`re.VERBOSE`** ignores whitespace in the pattern and allows `#` comments, so a
long pattern can be laid out and explained. Spaces you *want* to match must be
escaped (`\ `) or written as `[ ]` or `\s`. `re.compile` once, reuse many times
(Python caches recently used patterns too).

In [307]:
PHONE_PATTERN = re.compile(
    r"""
    ^(?:\+91[ -]?)?        # optional country code
    (?P<operator>[6-9]\d{4})  # mobile numbers start with 6-9
    [ -]?                  # optional separator
    (?P<subscriber>\d{5})$
    """,
    re.VERBOSE,
)
for phone in ("+91 98765 43210", "9876543210", "+91-98765-43210", "12345 67890"):
    phone_match = PHONE_PATTERN.match(phone)
    print(f"  {phone:<17} {phone_match.groupdict() if phone_match else None}")
print(f"{bool(PHONE_PATTERN.flags & re.VERBOSE)=}, {PHONE_PATTERN.groupindex=}")

  +91 98765 43210   {'operator': '98765', 'subscriber': '43210'}
  9876543210        {'operator': '98765', 'subscriber': '43210'}
  +91-98765-43210   {'operator': '98765', 'subscriber': '43210'}
  12345 67890       None
bool(PHONE_PATTERN.flags & re.VERBOSE)=True, PHONE_PATTERN.groupindex=mappingproxy({'operator': 1, 'subscriber': 2})


**Lookarounds** check what comes before or after **without consuming it**:
`(?=...)` lookahead, `(?!...)` negative lookahead, `(?<=...)` lookbehind, `(?<!...)`
negative lookbehind. Python requires lookbehinds to be **fixed-width** (JavaScript
allows variable width).

In [308]:
prices = "₹240, $3, ₹1,199 and 15 items"
print(f"{re.findall(r'(?<=₹)\d+(?:,\d+)*', prices)=}   lookbehind: amounts after ₹")
print(f"{re.findall(r'\d+(?= items)', prices)=}   lookahead: the number before ' items'")
print(
    f"{re.findall(r'\b\d+\b(?!,| items)', prices)=}"
    "   negative lookahead: not before ',' or ' items'"
)
strong = re.compile(r"^(?=.*[A-Z])(?=.*\d)(?=.*[^\w\s]).{10,}$")
for password in ("chai-time-2026", "Chai-time-2026", "Ch@i1"):
    print(f"  {password:<16} strong={bool(strong.match(password))}")
print(raised_message(lambda: re.compile(r"(?<=₹\s*)\d+")))

re.findall(r'(?<=₹)\d+(?:,\d+)*', prices)=['240', '1,199']   lookbehind: amounts after ₹
re.findall(r'\d+(?= items)', prices)=['15']   lookahead: the number before ' items'
re.findall(r'\b\d+\b(?!,| items)', prices)=['199']   negative lookahead: not before ',' or ' items'
  chai-time-2026   strong=False
  Chai-time-2026   strong=True
  Ch@i1            strong=False
PatternError: look-behind requires fixed-width pattern


**`sub` with a function.** `re.sub(pattern, replacement, text)` replaces every
match; the replacement can use `\1` or `\g<name>` (JavaScript: `$1`, `$<name>`),
or be a **function** that receives each `Match` and returns the replacement text.
`subn` also returns the count; `re.split` splits on a pattern.

In [309]:
print(f"{re.sub(r'(\d{4})-(\d{2})-(\d{2})', r'\3/\2/\1', 'due 2026-11-07')=}")
print(f"{re.sub(r'(?P<sku>[A-Z]{3})-(?P<n>\d+)', r'\g<sku>#\g<n>', 'TEA-500 MUG-02')=}")


def mask_card(card_match: re.Match[str]) -> str:
    digits = re.sub(r"\D", "", card_match.group())
    return "**** **** **** " + digits[-4:]


print(f"{re.sub(r'\b(?:\d{4}[ -]?){3}\d{4}\b', mask_card, 'card 4111 1111 1111 1234 ok')=}")
print(f"{re.subn(r'\s+', ' ', 'too    many   spaces')=}")
print(f"{re.split(r'[;,]\s*', 'tea, mug;jar,  spoon')=}")

re.sub(r'(\d{4})-(\d{2})-(\d{2})', r'\3/\2/\1', 'due 2026-11-07')='due 07/11/2026'
re.sub(r'(?P<sku>[A-Z]{3})-(?P<n>\d+)', r'\g<sku>#\g<n>', 'TEA-500 MUG-02')='TEA#500 MUG#02'
re.sub(r'\b(?:\d{4}[ -]?){3}\d{4}\b', mask_card, 'card 4111 1111 1111 1234 ok')='card **** **** **** 1234 ok'
re.subn(r'\s+', ' ', 'too    many   spaces')=('too many spaces', 2)
re.split(r'[;,]\s*', 'tea, mug;jar,  spoon')=['tea', 'mug', 'jar', 'spoon']


**Catastrophic backtracking.** Python's `re` is a backtracking engine (like
JavaScript's). Nested quantifiers such as `(a+)+` can match the same text in
exponentially many ways, so a *failing* match tries them all: each extra character
roughly doubles the time. With user-controlled input this is a denial-of-service
hole (ReDoS). Timings below vary by machine; the doubling does not.

In [310]:
nested = re.compile(r"(a+)+$")
for length in (16, 18, 20, 22):
    started = time.perf_counter()
    nested.match("a" * length + "!")
    print(f"  length {length}: {(time.perf_counter() - started) * 1000:8.1f} ms")

  length 16:      4.1 ms
  length 18:     10.8 ms
  length 20:     29.4 ms


  length 22:    136.4 ms


The fixes: remove the nesting (`a+$` means the same here), or use a
**possessive** quantifier `a++` or an **atomic group** `(?>...)` (*New in 3.11*),
which never give back what they matched. Both are linear even for 50,000
characters. For untrusted patterns, the third-party `regex` module supports a
`timeout`.

In [311]:
attack = "a" * 50_000 + "!"
for pattern in (r"a+$", r"(a++)+$", r"(?>a+)+$"):
    started = time.perf_counter()
    result = re.match(pattern, attack)
    print(f"  {pattern:<10} {result=}, fast: {time.perf_counter() - started < 0.1}")

  a+$        result=None, fast: True
  (a++)+$    result=None, fast: True
  (?>a+)+$   result=None, fast: True


**JavaScript regex differences**, each shown in a cell above or below:

| Topic | JavaScript | Python |
| --- | --- | --- |
| Literal | `/\d+/g` | `re.compile(r"\d+")` (raw string) |
| All matches | `g` flag + `matchAll` | `findall` / `finditer` (no `g` flag) |
| Named group | `(?<name>...)` | `(?P<name>...)` |
| Replacement | `$1`, `$<name>` | `\1`, `\g<name>` (or a function) |
| `\d`, `\w` | ASCII only | Unicode by default (`re.ASCII` for ASCII) |
| `$` | end of string | end, **or before a final `\n`** (use `\Z` / `fullmatch`) |
| Lookbehind | variable width allowed | fixed width only |
| Sticky `y` flag | `re.lastIndex` | `pattern.match(text, pos)` |
| Possessive / atomic | not supported | `a++`, `(?>...)` (3.11+) |
| Start-only match | (none) | `re.match` |

The sticky-flag equivalent: a compiled pattern's `match(text, pos)` anchors at
`pos`, which is how simple tokenizers are written:

In [312]:
TOKEN = re.compile(r"\s*(?:(?P<number>\d+)|(?P<operator>[+*-]))")
expression, position, tokens = "12 + 30 * 2", 0, []
while position < len(expression):
    token = TOKEN.match(expression, position)
    tokens.append((token.lastgroup, token.group(token.lastgroup)))
    position = token.end()
print(tokens)

[('number', '12'), ('operator', '+'), ('number', '30'), ('operator', '*'), ('number', '2')]


**Under the hood: the compiled pattern.** `re.compile` parses the pattern into a
tree of opcodes for the C matching engine; `re.DEBUG` prints that tree, which shows
how quantifiers and groups are represented (`MAX_REPEAT`, `SUBPATTERN`,
`POSSESSIVE_REPEAT`). The output goes to stdout, captured here.

In [313]:
parse_tree = io.StringIO()
with contextlib.redirect_stdout(parse_tree):
    re.compile(r"(?P<sku>[A-Z]{3})-\d++", re.DEBUG)
print("\n".join(parse_tree.getvalue().splitlines()[:12]))
print(f"{re.compile(r'\d+') is re.compile(r'\d+')=}  (re caches compiled patterns)")

SUBPATTERN 1 0 0
  MAX_REPEAT 3 3
    IN
      RANGE (65, 90)
LITERAL 45
POSSESSIVE_REPEAT 1 MAXREPEAT
  IN
    CATEGORY CATEGORY_DIGIT

 0. INFO 4 0b0 5 MAXREPEAT (to 5)
 5: MARK 0
 7. REPEAT_ONE 10 3 3 (to 18)
re.compile(r'\d+') is re.compile(r'\d+')=True  (re caches compiled patterns)


### 4.4.5 Data formats

**`json`.** `dumps`/`loads` work with strings, `dump`/`load` with files. Python
types map to JSON as `dict`→object, `list`/`tuple`→array, `str`, `int`/`float`→number,
`True`/`False`/`None`→`true`/`false`/`null`. Options worth knowing: `indent`,
`sort_keys`, `ensure_ascii=False` (keep `₹` readable), `separators` for compact
output. Errors carry the line and column.

In [314]:
order = {
    "id": 7,
    "customer": "Asha",
    "items": [("TEA-500", 2)],
    "paid": True,
    "note": None,
    "currency": "₹",
}
print(json.dumps(order))
print(json.dumps(order, ensure_ascii=False, sort_keys=True, indent=2))
print(json.dumps(order, separators=(",", ":"), ensure_ascii=False))
print(f"{json.loads(json.dumps(order))['items']=}  (tuples come back as lists)")
print(raised_message(lambda: json.loads('{"id": 7, "paid": true,}')))

{"id": 7, "customer": "Asha", "items": [["TEA-500", 2]], "paid": true, "note": null, "currency": "\u20b9"}
{
  "currency": "₹",
  "customer": "Asha",
  "id": 7,
  "items": [
    [
      "TEA-500",
      2
    ]
  ],
  "note": null,
  "paid": true
}
{"id":7,"customer":"Asha","items":[["TEA-500",2]],"paid":true,"note":null,"currency":"₹"}
json.loads(json.dumps(order))['items']=[['TEA-500', 2]]  (tuples come back as lists)
JSONDecodeError: Illegal trailing comma before end of object: line 1 column 23 (char 22)


**Custom encoders.** `json` refuses types it does not know (`Decimal`, `datetime`,
`set`, dataclasses, enums). Either pass `default=function` (called for each unknown
object, returns something serializable) or subclass `JSONEncoder` and override
`default`. Decide deliberately how each type is represented: money as a
**string** keeps every digit.

In [315]:
class OrderStatus(enum.StrEnum):
    PAID = "paid"
    SHIPPED = "shipped"


@dataclasses.dataclass
class OrderLine:
    sku: str
    quantity: int
    unit_price: Decimal


class ShopEncoder(json.JSONEncoder):
    def default(self, value: object) -> object:
        if isinstance(value, Decimal):
            return str(value)
        if isinstance(value, datetime | date):
            return value.isoformat()
        if isinstance(value, set | frozenset):
            return sorted(value)
        if dataclasses.is_dataclass(value) and not isinstance(value, type):
            return dataclasses.asdict(value)
        return super().default(value)  # raises TypeError for anything else


shipment = {
    "placed_at": FIXED_MOMENT,
    "status": OrderStatus.SHIPPED,
    "tags": {"gift", "express"},
    "lines": [OrderLine("TEA-500", 2, Decimal("240.00"))],
    "total": Decimal("480.00"),
}
print(raised_message(lambda: json.dumps(shipment)))
print(json.dumps(shipment, cls=ShopEncoder))
print(raised_message(lambda: json.dumps({"lock": io.StringIO()}, cls=ShopEncoder)))

TypeError: Object of type datetime is not JSON serializable
{"placed_at": "2026-11-07T09:30:00+00:00", "status": "shipped", "tags": ["express", "gift"], "lines": [{"sku": "TEA-500", "quantity": 2, "unit_price": "240.00"}], "total": "480.00"}
TypeError: Object of type StringIO is not JSON serializable


**`object_hook`** is the decoding counterpart: it is called with every decoded JSON
object (dict) and can return anything, for example turning ISO strings back into
datetimes or tagged dicts into your own classes. (`object_pairs_hook` receives the
key-value pairs in order, e.g. to reject duplicate keys.)

In [316]:
def decode_shop(obj: dict[str, object]) -> object:
    """Revive datetimes and order lines from their JSON form."""
    if "placed_at" in obj:
        obj["placed_at"] = datetime.fromisoformat(str(obj["placed_at"]))
    if obj.keys() == {"sku", "quantity", "unit_price"}:
        return OrderLine(str(obj["sku"]), int(obj["quantity"]), Decimal(str(obj["unit_price"])))
    return obj


revived = json.loads(json.dumps(shipment, cls=ShopEncoder), object_hook=decode_shop)
print(f"{revived['placed_at']=}")
print(f"{revived['lines']=}")


def reject_duplicates(pairs: list[tuple[str, object]]) -> dict[str, object]:
    keys = [key for key, _ in pairs]
    if len(keys) != len(set(keys)):
        raise ValueError(f"duplicate keys in {keys}")
    return dict(pairs)


print(f"{json.loads('{\"qty\": 1, \"qty\": 5}')=}  (last one silently wins)")
print(
    raised_message(lambda: json.loads('{"qty": 1, "qty": 5}', object_pairs_hook=reject_duplicates))
)

revived['placed_at']=datetime.datetime(2026, 11, 7, 9, 30, tzinfo=datetime.timezone.utc)
revived['lines']=[OrderLine(sku='TEA-500', quantity=2, unit_price=Decimal('240.00'))]
json.loads('{\"qty\": 1, \"qty\": 5}')={'qty': 5}  (last one silently wins)
ValueError: duplicate keys in ['qty', 'qty']


**`parse_float=Decimal`.** By default JSON numbers with a decimal point become
`float`, with the usual binary rounding (`0.1 + 0.2`). For money, parse them
straight into `Decimal` so no digit is ever lost (JavaScript's `JSON.parse` has no
such option). `parse_int` does the same for integers.

In [317]:
payload = '{"items": [{"price": 0.1}, {"price": 0.2}], "discount": 0.3}'
as_floats = json.loads(payload)
as_decimals = json.loads(payload, parse_float=Decimal)
print(f"{sum(item['price'] for item in as_floats['items'])=}")
print(f"{sum(item['price'] for item in as_decimals['items'])=}")
print(f"{sum(item['price'] for item in as_decimals['items']) == as_decimals['discount']=}")
print(f"{json.loads('{\"id\": 12345678901234567890}')['id']=}  (Python ints never overflow)")

sum(item['price'] for item in as_floats['items'])=0.30000000000000004
sum(item['price'] for item in as_decimals['items'])=Decimal('0.3')
sum(item['price'] for item in as_decimals['items']) == as_decimals['discount']=True
json.loads('{\"id\": 12345678901234567890}')['id']=12345678901234567890  (Python ints never overflow)


**`csv`.** `csv.DictWriter` writes dicts as rows (`fieldnames` fixes the column
order; `writeheader()` writes the first row); `csv.DictReader` yields one dict per
row, keyed by the header. Everything read back is a **string**: convert types
yourself. Always open CSV files with **`newline=""`** (explained below).

In [318]:
csv_path = UNIT44_DIR / "orders.csv"
rows = [
    {
        "order_id": 7,
        "customer": "Asha Rao",
        "total": Decimal("679.00"),
        "note": 'gift wrap, "fragile"',
    },
    {"order_id": 8, "customer": "Ravi", "total": Decimal("240.00"), "note": ""},
]
with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["order_id", "customer", "total", "note"])
    writer.writeheader()
    writer.writerows(rows)
print(csv_path.read_text(encoding="utf-8"))
with csv_path.open(newline="", encoding="utf-8") as csv_file:
    loaded_rows = list(csv.DictReader(csv_file))
print(f"{loaded_rows[0]=}")
print(f"{sum(Decimal(row['total']) for row in loaded_rows)=}")

order_id,customer,total,note
7,Asha Rao,679.00,"gift wrap, ""fragile"""
8,Ravi,240.00,

loaded_rows[0]={'order_id': '7', 'customer': 'Asha Rao', 'total': '679.00', 'note': 'gift wrap, "fragile"'}
sum(Decimal(row['total']) for row in loaded_rows)=Decimal('919.00')


Note the quoting in the printed file: the `note` field contains a comma and quotes,
so the writer wrapped it in quotes and doubled the inner ones. `DictWriter` refuses keys not in `fieldnames` unless
`extrasaction="ignore"`; other dialects (`csv.excel_tab`, `delimiter=";"`) and
`quoting=csv.QUOTE_NONNUMERIC` cover the variants you meet in practice.

**Why `newline=""`.** A quoted CSV field may contain line breaks, and the CSV
format uses `\r\n` between rows. Text mode normally translates line endings, which
corrupts both: on reading, an embedded `\r\n` becomes `\n`; on writing on
Windows, every `\r\n` becomes `\r\r\n` (blank rows in Excel). `newline=""` turns
the translation off and lets the `csv` module handle line endings itself.

In [319]:
address_path = UNIT44_DIR / "addresses.csv"
with address_path.open("w", newline="", encoding="utf-8") as csv_file:
    csv.writer(csv_file).writerow(["Asha", "Flat 4\r\nMG Road\r\nPune"])
print(f"{address_path.read_bytes()=}")
with address_path.open(encoding="utf-8") as csv_file:  # wrong
    print(f"without newline='': {next(csv.reader(csv_file))[1]!r}")
with address_path.open(newline="", encoding="utf-8") as csv_file:  # right
    print(f"with newline='':    {next(csv.reader(csv_file))[1]!r}")
print(
    raised_message(
        lambda: csv.DictWriter(io.StringIO(), fieldnames=["a"]).writerow({"a": 1, "b": 2})
    )
)

address_path.read_bytes()=b'Asha,"Flat 4\r\nMG Road\r\nPune"\r\n'
without newline='': 'Flat 4\nMG Road\nPune'
with newline='':    'Flat 4\r\nMG Road\r\nPune'
ValueError: dict contains fields not in fieldnames: 'b'


### 4.4.6 Algorithm helpers

**`heapq`** keeps a plain list arranged as a binary **min-heap**: `heap[0]` is
always the smallest item, `heappush` and `heappop` are O(log n). That makes a
**priority queue**. Store tuples `(priority, sequence, item)`: the sequence number
breaks ties in insertion order and stops Python from ever comparing the items
themselves (dicts cannot be compared).

In [320]:
import itertools

task_queue: list[tuple[int, int, dict[str, str]]] = []
sequence = itertools.count()
for priority, task in [
    (2, {"job": "send invoice"}),
    (1, {"job": "refund order 8"}),
    (3, {"job": "weekly report"}),
    (1, {"job": "restock TEA-500"}),
]:
    heapq.heappush(task_queue, (priority, next(sequence), task))
print(f"{task_queue[0]=}  (smallest first)")
while task_queue:
    priority, _, task = heapq.heappop(task_queue)
    print(f"  priority {priority}: {task['job']}")
print(raised_message(lambda: heapq.heappush([(1, {"a": 1})], (1, {"b": 2}))))

task_queue[0]=(1, 1, {'job': 'refund order 8'})  (smallest first)
  priority 1: refund order 8
  priority 1: restock TEA-500
  priority 2: send invoice
  priority 3: weekly report
TypeError: '<' not supported between instances of 'dict' and 'dict'


`heapify` turns a list into a heap in O(n). `nlargest(k, items, key=)` and
`nsmallest` find the top k in O(n log k) without sorting everything (for k = 1 use
`max`/`min`; for k close to n, `sorted`). `heappushpop` keeps a bounded heap;
`heapq.merge` lazily merges already-sorted inputs. *New in 3.14:* `heappush_max`,
`heappop_max`, `heapify_max` for max-heaps (before, you negated the priorities).

In [321]:
sales = {"TEA-500": 480, "MUG-02": 199, "JAR-750": 1050, "SPOON-1": 60, "KETTLE-9": 899}
print(f"{heapq.nlargest(3, sales.items(), key=lambda item: item[1])=}")
print(f"{heapq.nsmallest(2, sales, key=sales.get)=}")
numbers = [9, 4, 7, 1, 8]
heapq.heapify(numbers)
print(f"{numbers=}  (heap order, not sorted)")
print(f"{list(heapq.merge([1, 4, 9], [2, 3, 10], [5]))=}")
if sys.version_info >= (3, 14):
    max_heap = [3, 9, 1]
    heapq.heapify_max(max_heap)
    print(f"New in 3.14: {heapq.heappop_max(max_heap)=}")

heapq.nlargest(3, sales.items(), key=lambda item: item[1])=[('JAR-750', 1050), ('KETTLE-9', 899), ('TEA-500', 480)]
heapq.nsmallest(2, sales, key=sales.get)=['SPOON-1', 'MUG-02']
numbers=[1, 4, 7, 9, 8]  (heap order, not sorted)
list(heapq.merge([1, 4, 9], [2, 3, 10], [5]))=[1, 2, 3, 4, 5, 9, 10]
New in 3.14: heapq.heappop_max(max_heap)=9


**Under the hood: a C accelerator with a Python fallback.** `heapq` is written in
Python, but at import time it replaces its functions with C versions from `_heapq`
when available. The pure-Python helpers remain, and `inspect` can show them: a
push appends at the end, then sifts the new item up towards the root.

In [322]:
print(f"{heapq.heappush.__module__=}, {type(heapq.heappush).__name__=}")
print(inspect.getsource(heapq._siftdown))

heapq.heappush.__module__='_heapq', type(heapq.heappush).__name__='builtin_function_or_method'
def _siftdown(heap, startpos, pos):
    newitem = heap[pos]
    # Follow the path to the root, moving parents down until finding a place
    # newitem fits.
    while pos > startpos:
        parentpos = (pos - 1) >> 1
        parent = heap[parentpos]
        if newitem < parent:
            heap[pos] = parent
            pos = parentpos
            continue
        break
    heap[pos] = newitem



**`bisect`** finds positions in a **sorted** list by binary search, O(log n):
`bisect_left` / `bisect_right` (where an item would go), `insort` (insert keeping
order). The classic use: mapping a number to a band, such as income-tax slabs or
shipping rates. `key=` (*New in 3.10*) searches records by a field.

In [323]:
slab_limits = [400_000, 800_000, 1_200_000, 1_600_000, 2_000_000, 2_400_000]  # rupees
slab_rates = [0, 5, 10, 15, 20, 25, 30]  # percent


def marginal_rate(income: int) -> int:
    """Rate of the slab an income falls into (illustrative new-regime slabs)."""
    return slab_rates[bisect.bisect_right(slab_limits, income)]


print({income: marginal_rate(income) for income in (300_000, 400_000, 950_000, 2_500_000)})
scores = [55, 70, 85]
bisect.insort(scores, 77)
print(f"{scores=}, {bisect.bisect_left(scores, 77)=}, {bisect.bisect_right(scores, 77)=}")
orders = [(3, "MUG-02"), (5, "TEA-500"), (9, "JAR-750")]
print(f"{bisect.bisect_left(orders, 6, key=lambda order: order[0])=}")

{300000: 0, 400000: 5, 950000: 10, 2500000: 30}
scores=[55, 70, 77, 85], bisect.bisect_left(scores, 77)=2, bisect.bisect_right(scores, 77)=3
bisect.bisect_left(orders, 6, key=lambda order: order[0])=2


**`statistics`** covers descriptive statistics without NumPy: averages, spread,
quantiles, correlation, simple linear regression and the normal distribution.
`fmean` is the fast float mean; `mean` keeps exact types (`Decimal`, `Fraction`).

In [324]:
daily_orders = [42, 38, 51, 47, 38, 60, 55, 38, 49, 52]
print(f"{statistics.mean(daily_orders)=}, {statistics.fmean(daily_orders)=}")
print(
    f"{statistics.median(daily_orders)=}, {statistics.mode(daily_orders)=}, "
    f"{statistics.multimode([1, 1, 2, 2])=}"
)
print(
    f"{statistics.stdev(daily_orders):.2f} (sample), "
    f"{statistics.pstdev(daily_orders):.2f} (population)"
)
print(f"{statistics.quantiles(daily_orders, n=4)=}")
ad_spend = [10, 9, 14, 12, 8, 18, 16, 9, 13, 15]
print(f"{statistics.correlation(ad_spend, daily_orders):.3f}")
slope, intercept = statistics.linear_regression(ad_spend, daily_orders)
print(f"orders ≈ {slope:.2f} × spend + {intercept:.2f}")
delivery_days = statistics.NormalDist(mu=3, sigma=0.8)
print(f"P(delivery within 4 days) = {delivery_days.cdf(4):.3f}")
print(f"{statistics.mean([Decimal('0.10'), Decimal('0.20')])=}")

statistics.mean(daily_orders)=47, statistics.fmean(daily_orders)=47.0
statistics.median(daily_orders)=48.0, statistics.mode(daily_orders)=38, statistics.multimode([1, 1, 2, 2])=[1, 2]
7.79 (sample), 7.39 (population)
statistics.quantiles(daily_orders, n=4)=[38.0, 48.0, 52.75]
0.994
orders ≈ 2.29 × spend + 18.54
P(delivery within 4 days) = 0.894
statistics.mean([Decimal('0.10'), Decimal('0.20')])=Decimal('0.15')


**`math`** beyond the basics of Unit 1.3: exactly rounded float sums (`fsum`;
*Changed in 3.12:* built-in `sum()` of floats also compensates for rounding, but
only `fsum` is exact in every case), products,
combinatorics (`comb`, `perm`), integer roots (`isqrt`), `gcd`/`lcm`, distances,
`isclose` for float comparisons, and `fma` (fused multiply-add, *New in 3.13*).

In [325]:
running_total = 0.0
for _ in range(10):
    running_total += 0.1
print(
    f"{running_total=}, {sum([0.1] * 10)=} (Changed in 3.12: compensated), {math.fsum([0.1] * 10)=}"
)
print(
    f"{sum([2**53, 1.0, 1.0])=}, {math.fsum([2**53, 1.0, 1.0])=}  (fsum is always exactly rounded)"
)
print(f"{math.prod([2, 3, 4])=}, {math.comb(10, 3)=}, {math.perm(10, 3)=}")
print(f"{math.isqrt(10**20 + 1)=}, {math.gcd(240, 360)=}, {math.lcm(4, 6)=}")
print(f"{math.dist((0, 0), (3, 4))=}, {math.hypot(3, 4)=}, {math.log2(1024)=}")
print(f"{math.isclose(0.1 + 0.2, 0.3)=}, {math.ceil(4.01)=}, {math.floor(-4.5)=}")
if sys.version_info >= (3, 13):
    print(f"New in 3.13: {math.fma(2.0, 3.0, 1.0)=}")

running_total=0.9999999999999999, sum([0.1] * 10)=1.0 (Changed in 3.12: compensated), math.fsum([0.1] * 10)=1.0
sum([2**53, 1.0, 1.0])=9007199254740992.0, math.fsum([2**53, 1.0, 1.0])=9007199254740994.0  (fsum is always exactly rounded)
math.prod([2, 3, 4])=24, math.comb(10, 3)=120, math.perm(10, 3)=720
math.isqrt(10**20 + 1)=10000000000, math.gcd(240, 360)=120, math.lcm(4, 6)=12
math.dist((0, 0), (3, 4))=5.0, math.hypot(3, 4)=5.0, math.log2(1024)=10.0
math.isclose(0.1 + 0.2, 0.3)=True, math.ceil(4.01)=5, math.floor(-4.5)=-5
New in 3.13: math.fma(2.0, 3.0, 1.0)=7.0


**`graphlib.TopologicalSorter`** orders the nodes of a dependency graph so every
node comes after its dependencies: build steps, database migrations, task
pipelines. You give `{node: dependencies}`. `static_order()` returns one valid
order; a cycle raises `CycleError` with the cycle in its arguments.

In [326]:
build_steps = {
    "package": {"test", "docs"},
    "test": {"compile"},
    "docs": {"compile"},
    "compile": {"fetch"},
    "fetch": set(),
}
print(f"{list(graphlib.TopologicalSorter(build_steps).static_order())=}")
try:
    list(graphlib.TopologicalSorter({"migrate": {"seed"}, "seed": {"migrate"}}).static_order())
except graphlib.CycleError as error:
    print(f"{type(error).__name__}: {error.args[1]}")

list(graphlib.TopologicalSorter(build_steps).static_order())=['fetch', 'compile', 'test', 'docs', 'package']
CycleError: ['migrate', 'seed', 'migrate']


For **parallel** execution use the incremental API: `prepare()`, then repeatedly
`get_ready()` (everything whose dependencies are done, which can run at the same
time) and `done(node)` as each finishes. Sorting each batch makes the output
deterministic.

In [327]:
sorter = graphlib.TopologicalSorter(build_steps)
sorter.prepare()
batch_number = 0
while sorter.is_active():
    ready = sorted(sorter.get_ready())
    batch_number += 1
    print(f"  batch {batch_number}: run {ready} in parallel")
    sorter.done(*ready)

  batch 1: run ['fetch'] in parallel
  batch 2: run ['compile'] in parallel
  batch 3: run ['docs', 'test'] in parallel
  batch 4: run ['package'] in parallel


### 4.4.7 Randomness and security

**`random`** is a Mersenne Twister pseudo-random generator: fast, good statistical
quality, and **reproducible** with `random.seed(...)`, which is what simulations,
sampling and tests need. Every call below follows `random.seed(42)`, so the output
is the same on every run (for a given Python version).

In [328]:
random.seed(42)
print(f"{random.random()=:.6f}, {random.randint(1, 6)=}, {random.uniform(100, 200)=:.2f}")
customers = ["Asha", "Ravi", "Meera", "John", "Fatima"]
print(f"{random.choice(customers)=}")
print(f"{random.choices(['card', 'upi', 'cash'], weights=[5, 4, 1], k=8)=}  (with replacement)")
print(f"{random.sample(customers, 3)=}  (without replacement)")
deck = list(range(1, 11))
random.shuffle(deck)  # in place, returns None
print(f"{deck=}")
print(f"{random.gauss(mu=3, sigma=0.5)=:.3f}")

random.random()=0.639427, random.randint(1, 6)=1, random.uniform(100, 200)=174.16
random.choice(customers)='Ravi'
random.choices(['card', 'upi', 'cash'], weights=[5, 4, 1], k=8)=['card', 'upi', 'upi', 'upi', 'card', 'card', 'card', 'card']  (with replacement)
random.sample(customers, 3)=['Fatima', 'Asha', 'Meera']  (without replacement)
deck=[8, 6, 1, 3, 5, 10, 2, 7, 9, 4]
random.gauss(mu=3, sigma=0.5)=2.854


A `random.Random(seed)` instance is an **independent** generator: give each
simulation or test its own, so code elsewhere calling `random.random()` cannot
shift its sequence.

In [329]:
simulation_a, simulation_b = random.Random(7), random.Random(7)
random.random()  # global calls do not disturb the instances
print(f"{[simulation_a.randint(1, 100) for _ in range(3)]=}")
print(f"{[simulation_b.randint(1, 100) for _ in range(3)]=}  (same seed, same stream)")

[simulation_a.randint(1, 100) for _ in range(3)]=[42, 20, 51]
[simulation_b.randint(1, 100) for _ in range(3)]=[42, 20, 51]  (same seed, same stream)


**Never use `random` for security.** Its whole state can be reconstructed from 624
consecutive 32-bit outputs, after which every future "random" token is predictable.
**`secrets`** draws from the operating system's cryptographically secure source
(`os.urandom`): use it for passwords, reset tokens, API keys, session ids. Its
output is unpredictable by design, so this cell prints only properties.

In [330]:
reset_token = secrets.token_urlsafe(32)  # 32 random bytes, URL-safe base64
api_key = secrets.token_hex(16)  # 16 bytes as 32 hex characters
print(
    f"{len(reset_token)=}, {set(reset_token) <= set(string.ascii_letters + string.digits + '-_')=}"
)
print(f"{len(api_key)=}, {all(character in string.hexdigits for character in api_key)=}")
print(f"{secrets.token_hex(16) != secrets.token_hex(16)=}")
otp = "".join(secrets.choice(string.digits) for _ in range(6))
print(f"{len(otp)=}, {otp.isdigit()=}, {0 <= secrets.randbelow(10) < 10=}")
print(f"{secrets.compare_digest('abc123', 'abc123')=}")

len(reset_token)=43, set(reset_token) <= set(string.ascii_letters + string.digits + '-_')=True
len(api_key)=32, all(character in string.hexdigits for character in api_key)=True
secrets.token_hex(16) != secrets.token_hex(16)=True
len(otp)=6, otp.isdigit()=True, 0 <= secrets.randbelow(10) < 10=True
secrets.compare_digest('abc123', 'abc123')=True


**`hashlib`**: cryptographic hash functions. A hash is a fixed-size fingerprint of
data: the same input always gives the same digest, any change gives a completely
different one, and you cannot go back. `update()` hashes data in pieces (same
result as one call), `hashlib.file_digest` (*New in 3.11*) hashes a file
efficiently. `blake2b` is fast and supports keys and custom digest sizes.

In [331]:
print(f"{hashlib.sha256(b'order 7: 679.00 INR').hexdigest()=}")
print(f"{hashlib.sha256(b'order 7: 679.01 INR').hexdigest()[:16]=}...  (one character changed)")
incremental = hashlib.sha256()
for chunk in (b"order 7: ", b"679.00 INR"):
    incremental.update(chunk)
print(f"{incremental.hexdigest() == hashlib.sha256(b'order 7: 679.00 INR').hexdigest()=}")
with (sample_root / "docs" / "README.md").open("rb") as readme_file:
    print(f"{hashlib.file_digest(readme_file, 'sha256').hexdigest()[:16]=}...")
print(f"{hashlib.blake2b(b'TEA-500', digest_size=8).hexdigest()=}")
print(f"{sorted(hashlib.algorithms_guaranteed)[:6]=}")

hashlib.sha256(b'order 7: 679.00 INR').hexdigest()='26b8effe0932ca6c09a61cbbbc5b5ca720ff23a32bd0dd1b502c5baeb1702213'
hashlib.sha256(b'order 7: 679.01 INR').hexdigest()[:16]='969fe0fdb1f54456'...  (one character changed)
incremental.hexdigest() == hashlib.sha256(b'order 7: 679.00 INR').hexdigest()=True
hashlib.file_digest(readme_file, 'sha256').hexdigest()[:16]='a3fc79668d370fb0'...
hashlib.blake2b(b'TEA-500', digest_size=8).hexdigest()='1051c55287c48003'
sorted(hashlib.algorithms_guaranteed)[:6]=['blake2b', 'blake2s', 'md5', 'sha1', 'sha224', 'sha256']


**Passwords are not hashed with plain SHA-256** (too fast: attackers try billions
per second). Use a deliberately slow, salted key-derivation function:
`hashlib.scrypt` or `hashlib.pbkdf2_hmac` in the standard library, or argon2
(Phase 6 uses `pwdlib[argon2]`). The salt is random per user and stored next to the
hash; it is fixed here only so the output is stable.

In [332]:
demo_salt = b"fixed-demo-salt!"  # real code: secrets.token_bytes(16) per user
stored_hash = hashlib.scrypt(b"correct horse battery staple", salt=demo_salt, n=2**14, r=8, p=1)
print(f"{stored_hash.hex()[:32]=}...")
attempt = hashlib.scrypt(b"correct horse battery staple", salt=demo_salt, n=2**14, r=8, p=1)
print(f"{hmac.compare_digest(stored_hash, attempt)=}")
print(f"{hashlib.pbkdf2_hmac('sha256', b'chai123', demo_salt, 600_000).hex()[:32]=}...")

stored_hash.hex()[:32]='34c8eadd0886418ca7fe87648dacd467'...


hmac.compare_digest(stored_hash, attempt)=True


hashlib.pbkdf2_hmac('sha256', b'chai123', demo_salt, 600_000).hex()[:32]='2118a5069e0ccfe4a3aa8da3c492230d'...


**`hmac`: message authentication, e.g. webhook signatures.** A payment provider
that calls your webhook signs the request body with a secret you share:
`HMAC-SHA256(secret, timestamp + "." + body)`. You recompute the signature and
compare. A plain hash would not do, because anyone can hash a forged body; only
the secret holder can produce the HMAC. Compare with **`hmac.compare_digest`**,
which takes the same time whatever the input, so an attacker cannot learn the
signature byte by byte from response timings. Include a timestamp to stop replays.

In [333]:
webhook_secret = b"whsec_demo_only_not_a_real_secret"
body = b'{"event": "payment.captured", "order_id": 7, "amount": 67900}'
timestamp = int(FIXED_MOMENT.timestamp())
signature = hmac.new(webhook_secret, f"{timestamp}.".encode() + body, hashlib.sha256).hexdigest()
print(f"X-Signature: t={timestamp},v1={signature}")


def signature_is_valid(received_body: bytes, received_signature: str) -> bool:
    expected = hmac.new(
        webhook_secret, f"{timestamp}.".encode() + received_body, hashlib.sha256
    ).hexdigest()
    return hmac.compare_digest(expected, received_signature)


print(f"{signature_is_valid(body, signature)=}")
print(f"{signature_is_valid(body.replace(b'67900', b'100'), signature)=}  (tampered amount)")

X-Signature: t=1794043800,v1=9caf97ce8fab841f8e91e558602a686eda41bc2395bb6de65ccb251223995603
signature_is_valid(body, signature)=True
signature_is_valid(body.replace(b'67900', b'100'), signature)=False  (tampered amount)


**`uuid`.** `uuid4()` is 122 random bits: unique without coordination but in random
order, which scatters database index inserts. **`uuid7()`** (*New in 3.14*, RFC
9562) starts with a 48-bit Unix timestamp in milliseconds, so ids **sort by
creation time** and index well; prefer it for primary keys. `uuid5` is
deterministic (a namespace + a name), useful for stable ids derived from data.
Random ids are shown only through their properties.

In [334]:
random_id = uuid.uuid4()
print(f"{random_id.version=}, {random_id.variant=}, {len(str(random_id))=}")
if sys.version_info >= (3, 14):
    time_ordered = [uuid.uuid7() for _ in range(5)]
    print(f"{time_ordered == sorted(time_ordered)=}, {time_ordered[0].version=}")
    milliseconds = time_ordered[0].int >> 80
    print(
        f"timestamp inside: within a minute of now = {abs(milliseconds / 1000 - time.time()) < 60}"
    )
    print(f"{uuid.NIL=}, {uuid.MAX=}")
print(f"{uuid.uuid5(uuid.NAMESPACE_URL, 'https://shop.example.com/orders/7')=}  (always the same)")

random_id.version=4, random_id.variant='specified in RFC 4122', len(str(random_id))=36
time_ordered == sorted(time_ordered)=True, time_ordered[0].version=7
timestamp inside: within a minute of now = True
uuid.NIL=UUID('00000000-0000-0000-0000-000000000000'), uuid.MAX=UUID('ffffffff-ffff-ffff-ffff-ffffffffffff')
uuid.uuid5(uuid.NAMESPACE_URL, 'https://shop.example.com/orders/7')=UUID('698f48f9-2dd0-5d41-910f-02f905659890')  (always the same)


**`base64`** turns bytes into ASCII text so binary data fits in JSON, URLs, e-mail
or HTTP headers. It is an **encoding, not encryption**: anyone can decode it.
`urlsafe_b64encode` uses `-` and `_` instead of `+` and `/`; the `=` padding makes
the length a multiple of 4. (`b32encode`, `b16encode` and `b85encode` exist too.)

In [335]:
credentials = base64.b64encode(b"asha:chai123").decode("ascii")
print(f"Authorization: Basic {credentials}")
print(f"{base64.b64decode(credentials)=}  (anyone can read it)")
token_bytes = bytes([251, 255, 0, 62])
print(f"{base64.b64encode(token_bytes)=}, {base64.urlsafe_b64encode(token_bytes)=}")
print(f"{[base64.b64encode(b'x' * size) for size in (1, 2, 3)]=}  (padding)")
print(f"{base64.b16encode(b'TEA')=}, {base64.b32encode(b'TEA')=}")

Authorization: Basic YXNoYTpjaGFpMTIz
base64.b64decode(credentials)=b'asha:chai123'  (anyone can read it)
base64.b64encode(token_bytes)=b'+/8APg==', base64.urlsafe_b64encode(token_bytes)=b'-_8APg=='
[base64.b64encode(b'x' * size) for size in (1, 2, 3)]=[b'eA==', b'eHg=', b'eHh4']  (padding)
base64.b16encode(b'TEA')=b'544541', base64.b32encode(b'TEA')=b'KRCUC==='


**`urllib.parse`** splits, builds and escapes URLs (JavaScript's `URL` and
`URLSearchParams`). `urlsplit` gives the parts; `parse_qs` returns lists (a key can
repeat), `parse_qsl` pairs; `urlencode` builds a query string (`doseq=True` for list
values); `quote` escapes a path segment, `quote_plus` a form value; `urljoin`
resolves a relative link the way a browser does.

In [336]:
url = "https://shop.example.com:8443/products/search?q=masala+tea&tag=organic&tag=gift#reviews"
parts = urllib.parse.urlsplit(url)
print(f"{parts.scheme=}, {parts.hostname=}, {parts.port=}, {parts.path=}, {parts.fragment=}")
print(f"{urllib.parse.parse_qs(parts.query)=}")
print(f"{urllib.parse.parse_qsl(parts.query)=}")
print(
    f"{urllib.parse.urlencode({'q': 'chai & biscuits', 'page': 2, 'tag': ['a', 'b']}, doseq=True)=}"
)
print(
    f"{urllib.parse.quote('reports/Nov 2026/₹.csv')=}, "
    f"{urllib.parse.quote_plus('chai & biscuits')=}"
)
print(f"{urllib.parse.unquote('%E2%82%B9500')=}")
print(f"{urllib.parse.urljoin('https://shop.example.com/products/tea/', '../mugs?page=2')=}")
print(f"{urllib.parse.urlunsplit(parts._replace(query='q=mug', fragment=''))=}")

parts.scheme='https', parts.hostname='shop.example.com', parts.port=8443, parts.path='/products/search', parts.fragment='reviews'
urllib.parse.parse_qs(parts.query)={'q': ['masala tea'], 'tag': ['organic', 'gift']}
urllib.parse.parse_qsl(parts.query)=[('q', 'masala tea'), ('tag', 'organic'), ('tag', 'gift')]
urllib.parse.urlencode({'q': 'chai & biscuits', 'page': 2, 'tag': ['a', 'b']}, doseq=True)='q=chai+%26+biscuits&page=2&tag=a&tag=b'
urllib.parse.quote('reports/Nov 2026/₹.csv')='reports/Nov%202026/%E2%82%B9.csv', urllib.parse.quote_plus('chai & biscuits')='chai+%26+biscuits'
urllib.parse.unquote('%E2%82%B9500')='₹500'
urllib.parse.urljoin('https://shop.example.com/products/tea/', '../mugs?page=2')='https://shop.example.com/products/mugs?page=2'
urllib.parse.urlunsplit(parts._replace(query='q=mug', fragment=''))='https://shop.example.com:8443/products/search?q=mug'


### 4.4.8 Command-line interfaces

**`argparse`** turns `sys.argv` into a namespace of typed values and generates
`--help` and error messages. Positional arguments are required; options start with
`-`/`--`. Per argument you choose a `type` (any callable), `default`, `choices`,
`nargs`, `action` (`store_true`, `count`, `append`) and `help`. **Subcommands**
(`git commit`, `uv add`) come from `add_subparsers()`, each with its own parser;
`set_defaults(handler=...)` attaches the function that runs it.

In a notebook, pass an explicit list to `parse_args([...])`; a real script calls
`parse_args()` with no argument and reads `sys.argv[1:]`. Since 3.14 argparse
colours help and errors when it thinks the output is a terminal, and the notebook
frontend can look like one, so the cell sets `PYTHON_COLORS=0` for plain text.

In [337]:
os.environ["PYTHON_COLORS"] = "0"  # plain argparse output in this notebook (3.14+ colours it)


def report_command(arguments: argparse.Namespace) -> str:
    return f"report for {arguments.month} as {arguments.format}"


def restock_command(arguments: argparse.Namespace) -> str:
    verb = "would add" if arguments.dry_run else "added"
    return f"{verb} {arguments.quantity} x {arguments.sku}"


def build_shopctl() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(prog="shopctl", description="Shop admin tool.")
    verbosity = parser.add_mutually_exclusive_group()
    verbosity.add_argument("-v", "--verbose", action="count", default=0, help="more output")
    verbosity.add_argument("-q", "--quiet", action="store_true", help="errors only")
    commands = parser.add_subparsers(dest="command", required=True, metavar="COMMAND")

    report = commands.add_parser("report", help="sales report for a month")
    report.add_argument("month", help="YYYY-MM")
    report.add_argument("--format", choices=["table", "csv", "json"], default="table")
    report.set_defaults(handler=report_command)

    restock = commands.add_parser("restock", help="add stock for a SKU")
    restock.add_argument("sku")
    restock.add_argument("-n", "--quantity", type=int, default=1)
    restock.add_argument("--dry-run", action="store_true")
    restock.set_defaults(handler=restock_command)
    return parser


shopctl = build_shopctl()
for argv in (["report", "2026-11"], ["-vv", "restock", "TEA-500", "-n", "24", "--dry-run"]):
    arguments = shopctl.parse_args(argv)
    print(f"{argv} -> {vars(arguments) | {'handler': arguments.handler.__name__}}")
    print(f"   {arguments.handler(arguments)}")

['report', '2026-11'] -> {'verbose': 0, 'quiet': False, 'command': 'report', 'month': '2026-11', 'format': 'table', 'handler': 'report_command'}
   report for 2026-11 as table
['-vv', 'restock', 'TEA-500', '-n', '24', '--dry-run'] -> {'verbose': 2, 'quiet': False, 'command': 'restock', 'sku': 'TEA-500', 'quantity': 24, 'dry_run': True, 'handler': 'restock_command'}
   would add 24 x TEA-500


**Help and errors for free.** `format_help()` returns what `--help` prints. On a
bad command line argparse prints usage plus an error to stderr and exits with
status 2; `exit_on_error=False` raises `ArgumentError` instead (useful in tests).
*New in 3.14:* `suggest_on_error=True` suggests the closest valid choice, and help
output is coloured in terminals (`color=False` turns that off).

In [338]:
print(build_shopctl().format_help())
with contextlib.redirect_stdout(io.StringIO()) as restock_help, contextlib.suppress(SystemExit):
    build_shopctl().parse_args(["restock", "--help"])  # --help prints, then exits
print(restock_help.getvalue())

usage: shopctl [-h] [-v | -q] COMMAND ...

Shop admin tool.

positional arguments:
  COMMAND
    report       sales report for a month
    restock      add stock for a SKU

options:
  -h, --help     show this help message and exit
  -v, --verbose  more output
  -q, --quiet    errors only

usage: shopctl restock [-h] [-n QUANTITY] [--dry-run] sku

positional arguments:
  sku

options:
  -h, --help            show this help message and exit
  -n, --quantity QUANTITY
  --dry-run



The real command-line experience, in a child process: the script runs the same
parser as `shopctl.py`. A typo in the subcommand, a bad type and a successful call:

In [339]:
cli_dir = fresh_dir("cli")
cli_source = inspect.getsource(report_command) + "\n\n" + inspect.getsource(restock_command)
parser_source = inspect.getsource(build_shopctl)
if sys.version_info >= (3, 14):  # suggest_on_error: New in 3.14
    parser_source = parser_source.replace(
        'description="Shop admin tool."', 'description="Shop admin tool.", suggest_on_error=True'
    )
cli_source += "\n\n" + parser_source
cli_source += "\n\narguments = build_shopctl().parse_args()\nprint(arguments.handler(arguments))\n"
(cli_dir / "shopctl.py").write_text("import argparse\n\n\n" + cli_source, encoding="utf-8")
for argv in (
    ["reprot", "2026-11"],
    ["restock", "TEA-500", "-n", "many"],
    ["restock", "MUG-02", "-n", "6"],
):
    completed = subprocess.run(
        [sys.executable, "shopctl.py", *argv],
        cwd=cli_dir,
        capture_output=True,
        text=True,
        env={**os.environ, "PYTHON_COLORS": "0", "NO_COLOR": "1"},
    )
    print(f"$ python shopctl.py {shlex.join(argv)}   (exit {completed.returncode})")
    print(textwrap.indent((completed.stdout + completed.stderr).strip(), "  "))

$ python shopctl.py reprot 2026-11   (exit 2)
  usage: shopctl [-h] [-v | -q] COMMAND ...
  shopctl: error: argument COMMAND: invalid choice: 'reprot', maybe you meant 'report'? (choose from 'report', 'restock')


$ python shopctl.py restock TEA-500 -n many   (exit 2)
  usage: shopctl restock [-h] [-n QUANTITY] [--dry-run] sku
  shopctl restock: error: argument -n/--quantity: invalid int value: 'many'


$ python shopctl.py restock MUG-02 -n 6   (exit 0)
  added 6 x MUG-02


For tests, `exit_on_error=False` turns most usage errors into an exception you can
assert on (some errors, such as missing required arguments, still exit; catch
`SystemExit` for those):

In [340]:
strict_parser = argparse.ArgumentParser(prog="shopctl", exit_on_error=False)
strict_parser.add_argument("--quantity", type=int)
print(raised_message(lambda: strict_parser.parse_args(["--quantity", "many"])))
with contextlib.redirect_stderr(io.StringIO()) as usage_message:
    try:
        build_shopctl().parse_args([])
    except SystemExit as exit_request:
        print(
            f"SystemExit({exit_request.code}): {usage_message.getvalue().strip().splitlines()[-1]}"
        )

ArgumentError: argument --quantity: invalid int value: 'many'
SystemExit(2): shopctl: error: the following arguments are required: COMMAND


**`typer` and `click`.** For larger CLIs, third-party libraries trade a dependency
for less code: `click` builds commands from decorators; `typer` builds them from
**type hints** (the same idea as FastAPI) on top of click. Unit 4.6 uses typer; the
`restock` command would look like this:

```python
import typer

app = typer.Typer()


@app.command()
def restock(sku: str, quantity: int = 1, dry_run: bool = False) -> None:
    verb = "would add" if dry_run else "added"
    print(f"{verb} {quantity} x {sku}")
```

### 4.4.9 Text utilities

**`textwrap`**: `wrap` (list of lines) and `fill` (one string) re-flow text to a
width; `shorten` truncates on word boundaries with a placeholder; `dedent` removes
common indentation (handy for triple-quoted strings, as this course's build scripts
do); `indent` adds a prefix to lines.

In [341]:
description = (
    "Hand-picked Assam tea leaves blended with cardamom, ginger and cinnamon "
    "for a strong, spicy cup."
)
print(textwrap.fill(description, width=40))
print(f"{textwrap.wrap(description, width=40)[:2]=}")
print(f"{textwrap.shorten(description, width=45, placeholder=' [...]')=}")
block = """
    Dear Asha,
        Your order has shipped.
"""
print(repr(textwrap.dedent(block).strip("\n")))
print(textwrap.indent("line one\nline two", "> "))

Hand-picked Assam tea leaves blended
with cardamom, ginger and cinnamon for a
strong, spicy cup.
textwrap.wrap(description, width=40)[:2]=['Hand-picked Assam tea leaves blended', 'with cardamom, ginger and cinnamon for a']
textwrap.shorten(description, width=45, placeholder=' [...]')='Hand-picked Assam tea leaves blended [...]'
'Dear Asha,\n    Your order has shipped.'
> line one
> line two


**`string.Template`** does `$name` / `${name}` substitution and nothing else: no
attribute access, no indexing, no format specs. That makes it the safe choice when
the **template** comes from users or translators. `str.format` with a
user-supplied template can read attributes of the objects you pass, all the way to
module globals (a real vulnerability class). `substitute` raises `KeyError` for a
missing name; `safe_substitute` leaves it in place.

In [342]:
@dataclasses.dataclass
class Customer:
    name: str
    city: str

    def __post_init__(self) -> None:
        self.api_secret = "sk_live_demo_secret"


asha = Customer("Asha", "Pune")
user_template = "Hi {customer.name}, secret: {customer.api_secret}"  # typed by a user
print(f"str.format leaks: {user_template.format(customer=asha)!r}")

greeting = string.Template("Hi $name, your order ships to ${city}. Code: $coupon")
print(f"{greeting.safe_substitute(name=asha.name, city=asha.city)=}")
print(raised_message(lambda: greeting.substitute(name=asha.name, city=asha.city)))
print(
    f"{string.Template('$customer.api_secret').safe_substitute(customer=asha)=}"
    "  (no attribute access)"
)

str.format leaks: 'Hi Asha, secret: sk_live_demo_secret'
greeting.safe_substitute(name=asha.name, city=asha.city)='Hi Asha, your order ships to Pune. Code: $coupon'
KeyError: 'coupon'
string.Template('$customer.api_secret').safe_substitute(customer=asha)="Customer(name='Asha', city='Pune').api_secret"  (no attribute access)


*New in 3.14:* template strings (`t"Hi {name}"`, PEP 750, `string.templatelib`)
produce a `Template` object with the static parts and the interpolated values kept
separate, so a library can escape values for HTML or SQL before combining them.
Unlike `string.Template`, the template is written in your code, not supplied at run
time.

**`difflib`** compares sequences: `unified_diff` produces `git diff`-style output,
`ndiff` a character-level view, `SequenceMatcher.ratio()` a similarity score, and
`get_close_matches` powers "did you mean ...?" suggestions.

In [343]:
old_config = ["currency = INR", "gst = 18", "free_shipping_above = 500"]
new_config = ["currency = INR", "gst = 12", "free_shipping_above = 500", "cod = true"]
print(
    "\n".join(
        difflib.unified_diff(old_config, new_config, "settings.old", "settings.new", lineterm="")
    )
)
print(f"{difflib.SequenceMatcher(None, 'masala tea', 'masala chai').ratio():.2f}")
catalogue_names = ["masala tea", "green tea", "steel mug", "pickle jar", "tea strainer"]
print(f"{difflib.get_close_matches('masla tee', catalogue_names)=}")
print(f"{difflib.get_close_matches('stel mug', catalogue_names, n=1)=}")
print(list(difflib.ndiff(["gst = 18"], ["gst = 12"])))

--- settings.old
+++ settings.new
@@ -1,3 +1,4 @@
 currency = INR
-gst = 18
+gst = 12
 free_shipping_above = 500
+cod = true
0.76
difflib.get_close_matches('masla tee', catalogue_names)=['masala tea']
difflib.get_close_matches('stel mug', catalogue_names, n=1)=['steel mug']
['- gst = 18', '?        ^\n', '+ gst = 12', '?        ^\n']


**`pprint`** prints nested data readably: it wraps at `width`, can limit `depth`,
and sorts dict keys unless `sort_dicts=False`. `pformat` returns the string;
`pprint.pp` (3.8) is `pprint` with `sort_dicts=False` by default.

In [344]:
store_config = {
    "store": {"name": "Pune Central", "city": "Pune", "opens": "09:00"},
    "payments": {"methods": ["upi", "card", "cash"], "gateway": {"name": "demo-pay", "timeout": 5}},
    "inventory": {"warehouses": ["PNQ1", "BLR1"], "reorder": {"TEA-500": 10, "MUG-02": 5}},
}
pprint.pprint(store_config, width=70)
print("--- depth=1, sort_dicts=False ---")
pprint.pprint(store_config, depth=1, sort_dicts=False)
print(f"{len(pprint.pformat(store_config, width=40).splitlines())=} lines at width 40")
pprint.pp({"z": 1, "a": 2})

{'inventory': {'reorder': {'MUG-02': 5, 'TEA-500': 10},
               'warehouses': ['PNQ1', 'BLR1']},
 'payments': {'gateway': {'name': 'demo-pay', 'timeout': 5},
              'methods': ['upi', 'card', 'cash']},
 'store': {'city': 'Pune', 'name': 'Pune Central', 'opens': '09:00'}}
--- depth=1, sort_dicts=False ---
{'store': {...}, 'payments': {...}, 'inventory': {...}}
len(pprint.pformat(store_config, width=40).splitlines())=12 lines at width 40
{'z': 1, 'a': 2}


### Lab 4.4

Seven exercises; replace each `raise NotImplementedError`, then run the check cell.
The checks build their own test folders under `notebooks/_scratch/unit44/lab/`.
Until you finish, they print FAIL; that is expected.

In [345]:
LAB44_DIR = find_repo_root() / "notebooks" / "_scratch" / "unit44" / "lab"
LAB44_DIR.mkdir(parents=True, exist_ok=True)
MIB = 1024 * 1024
print(f"lab files go to {LAB44_DIR.relative_to(find_repo_root()).as_posix()}/")

lab files go to notebooks/_scratch/unit44/lab/


#### Exercise 1 — `find_large_files(root, min_size_mb)`

Return every **regular file** under `root` (recursively; skip symlinks) whose size
is at least `min_size_mb` mebibytes (`min_size_mb * 1024 * 1024` bytes; the
argument may be a float). Each result is `(path relative to root as a POSIX
string, size in bytes)`, sorted by size (largest first), then by path. Raise
`NotADirectoryError` if `root` is not a directory.

In [346]:
def find_large_files(root: Path, min_size_mb: float) -> list[tuple[str, int]]:
    """Files of at least min_size_mb MiB under root, largest first."""
    raise NotImplementedError

Check exercise 1 (the test files are *sparse*: they report a size but use almost no disk):

In [347]:
def make_large_file_tree() -> Path:
    """big.iso 5 MiB, two 2.5 MiB logs, edge.bin exactly 1 MiB, small files, a symlink."""
    root = LAB44_DIR / "large_files"
    shutil.rmtree(root, ignore_errors=True)
    sizes = {
        "big.iso": 5 * MIB,
        "logs/app.log": 5 * MIB // 2,
        "logs/old/app.1.log": 5 * MIB // 2,
        "edge.bin": MIB,
        "notes.txt": 10,
        "logs/old/tiny.gz": 2048,
    }
    for relative_path, size in sizes.items():
        path = root / relative_path
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("wb") as handle:
            handle.truncate(size)
    (root / "big-link.iso").symlink_to("big.iso")
    return root


large_root = make_large_file_tree()
run_checks(
    [
        (
            "at least 1 MiB",
            lambda: find_large_files(large_root, 1),
            [
                ("big.iso", 5 * MIB),
                ("logs/app.log", 5 * MIB // 2),
                ("logs/old/app.1.log", 5 * MIB // 2),
                ("edge.bin", MIB),
            ],
        ),
        ("at least 3 MiB", lambda: find_large_files(large_root, 3), [("big.iso", 5 * MIB)]),
        (
            "float threshold",
            lambda: [path for path, _ in find_large_files(large_root, 2.5)],
            ["big.iso", "logs/app.log", "logs/old/app.1.log"],
        ),
        ("nothing that large", lambda: find_large_files(large_root, 10), []),
        (
            "symlink skipped",
            lambda: "big-link.iso" in [path for path, _ in find_large_files(large_root, 0)],
            False,
        ),
        (
            "root is a file",
            lambda: raised_message(lambda: find_large_files(large_root / "notes.txt", 1)).split(
                ":"
            )[0],
            "NotADirectoryError",
        ),
    ]
)

FAIL  at least 1 MiB: raised NotImplementedError: 
FAIL  at least 3 MiB: raised NotImplementedError: 
FAIL  float threshold: raised NotImplementedError: 
FAIL  nothing that large: raised NotImplementedError: 
FAIL  symlink skipped: raised NotImplementedError: 
FAIL  root is a file: expected 'NotADirectoryError', got 'NotImplementedError'
0/6 checks passed


#### Exercise 2 — a `VERBOSE` nginx-log regex with named groups

nginx's default "combined" access-log format looks like this:

```text
203.0.113.7 - - [07/Nov/2026:09:30:00 +0530] "GET /products/TEA-500?ref=home HTTP/1.1" 200 5120 "https://shop.example.com/" "Mozilla/5.0 (X11; Linux x86_64)"
```

1. `NGINX_LOG_PATTERN`: a pattern compiled with `re.VERBOSE` (laid out over several
   lines with comments) with exactly these named groups: `ip`, `user`, `time`,
   `method`, `path`, `protocol`, `status`, `size`, `referrer`, `user_agent`.
2. `parse_access_line(line)` returns `None` for a line that does not match, else
   the named groups as a dict with `status` and `size` converted to `int` (`-` as
   size means 0) and `user` / `referrer` set to `None` when they are `-`.

In [348]:
NGINX_LOG_PATTERN = re.compile(r"replace me")


def parse_access_line(line: str) -> dict[str, str | int | None] | None:
    """One combined-format log line -> dict of fields, or None."""
    raise NotImplementedError

Check exercise 2:

In [349]:
GET_LINE = (
    '203.0.113.7 - - [07/Nov/2026:09:30:00 +0530] "GET /products/TEA-500?ref=home HTTP/1.1" '
    '200 5120 "https://shop.example.com/" "Mozilla/5.0 (X11; Linux x86_64)"'
)
POST_LINE = (
    '198.51.100.23 - asha [07/Nov/2026:09:31:12 +0530] "POST /api/orders HTTP/2.0" '
    '201 - "-" "curl/8.10.1"'
)
LOG_GROUPS = {
    "ip",
    "user",
    "time",
    "method",
    "path",
    "protocol",
    "status",
    "size",
    "referrer",
    "user_agent",
}
run_checks(
    [
        ("compiled with re.VERBOSE", lambda: bool(NGINX_LOG_PATTERN.flags & re.VERBOSE), True),
        ("named groups", lambda: set(NGINX_LOG_PATTERN.groupindex), LOG_GROUPS),
        (
            "GET line",
            lambda: parse_access_line(GET_LINE),
            {
                "ip": "203.0.113.7",
                "user": None,
                "time": "07/Nov/2026:09:30:00 +0530",
                "method": "GET",
                "path": "/products/TEA-500?ref=home",
                "protocol": "HTTP/1.1",
                "status": 200,
                "size": 5120,
                "referrer": "https://shop.example.com/",
                "user_agent": "Mozilla/5.0 (X11; Linux x86_64)",
            },
        ),
        (
            "POST line: user, size '-', referrer '-'",
            lambda: {
                key: parse_access_line(POST_LINE)[key]
                for key in ("user", "method", "status", "size", "referrer", "user_agent")
            },
            {
                "user": "asha",
                "method": "POST",
                "status": 201,
                "size": 0,
                "referrer": None,
                "user_agent": "curl/8.10.1",
            },
        ),
        ("malformed line", lambda: parse_access_line("GET /index.html 200"), None),
        (
            "status must be 3 digits",
            lambda: parse_access_line(GET_LINE.replace(" 200 ", " 20 ")),
            None,
        ),
    ]
)

FAIL  compiled with re.VERBOSE: expected True, got False
FAIL  named groups: expected {'method', 'size', 'referrer', 'user', 'user_agent', 'protocol', 'time', 'path', 'status', 'ip'}, got set()
FAIL  GET line: raised NotImplementedError: 
FAIL  POST line: user, size '-', referrer '-': raised NotImplementedError: 
FAIL  malformed line: raised NotImplementedError: 
FAIL  status must be 3 digits: raised NotImplementedError: 
0/6 checks passed


#### Exercise 3 — IST/UTC conversion, including a DST case

1. `ist_to_utc("2026-11-07 15:00")` treats the text as India Standard Time and
   returns the UTC instant in ISO format: `"2026-11-07T09:30:00+00:00"`.
2. `utc_to_zone(utc_text, zone_name)` converts an ISO timestamp **with an offset**
   to the zone and returns ISO text; text without an offset raises `ValueError`.
3. `local_to_utc(local_text, zone_name)` treats `"YYYY-MM-DD HH:MM"` as wall-clock
   time in the zone: a time that does not exist there (skipped by a DST change)
   raises `ValueError`; an ambiguous time (repeated by a DST change) uses the
   **earlier** instant.

In [350]:
def ist_to_utc(local_text: str) -> str:
    """'YYYY-MM-DD HH:MM' in IST -> ISO 8601 in UTC."""
    raise NotImplementedError


def utc_to_zone(utc_text: str, zone_name: str) -> str:
    """ISO 8601 with offset -> ISO 8601 in zone_name."""
    raise NotImplementedError


def local_to_utc(local_text: str, zone_name: str) -> str:
    """'YYYY-MM-DD HH:MM' wall-clock time in zone_name -> ISO 8601 in UTC."""
    raise NotImplementedError

Check exercise 3 (New York's clocks change on 8 March and 1 November 2026):

In [351]:
run_checks(
    [
        ("IST afternoon", lambda: ist_to_utc("2026-11-07 15:00"), "2026-11-07T09:30:00+00:00"),
        (
            "IST crosses midnight",
            lambda: ist_to_utc("2026-01-01 03:00"),
            "2025-12-31T21:30:00+00:00",
        ),
        (
            "UTC -> IST",
            lambda: utc_to_zone("2026-11-07T09:30:00+00:00", "Asia/Kolkata"),
            "2026-11-07T15:00:00+05:30",
        ),
        (
            "UTC -> New York before DST",
            lambda: utc_to_zone("2026-03-06T14:00:00+00:00", "America/New_York"),
            "2026-03-06T09:00:00-05:00",
        ),
        (
            "UTC -> New York after DST",
            lambda: utc_to_zone("2026-03-09T14:00:00+00:00", "America/New_York"),
            "2026-03-09T10:00:00-04:00",
        ),
        (
            "IST input -> London",
            lambda: utc_to_zone("2026-07-01T19:30:00+05:30", "Europe/London"),
            "2026-07-01T15:00:00+01:00",
        ),
        (
            "naive input rejected",
            lambda: raised_message(
                lambda: utc_to_zone("2026-11-07T09:30:00", "Asia/Kolkata")
            ).split(":")[0],
            "ValueError",
        ),
        (
            "non-existent New York time",
            lambda: raised_message(
                lambda: local_to_utc("2026-03-08 02:30", "America/New_York")
            ).split(":")[0],
            "ValueError",
        ),
        (
            "ambiguous New York time: earlier",
            lambda: local_to_utc("2026-11-01 01:30", "America/New_York"),
            "2026-11-01T05:30:00+00:00",
        ),
        (
            "London summer time",
            lambda: local_to_utc("2026-07-01 09:00", "Europe/London"),
            "2026-07-01T08:00:00+00:00",
        ),
    ]
)

FAIL  IST afternoon: raised NotImplementedError: 
FAIL  IST crosses midnight: raised NotImplementedError: 
FAIL  UTC -> IST: raised NotImplementedError: 
FAIL  UTC -> New York before DST: raised NotImplementedError: 
FAIL  UTC -> New York after DST: raised NotImplementedError: 
FAIL  IST input -> London: raised NotImplementedError: 
FAIL  naive input rejected: expected 'ValueError', got 'NotImplementedError'
FAIL  non-existent New York time: expected 'ValueError', got 'NotImplementedError'
FAIL  ambiguous New York time: earlier: raised NotImplementedError: 
FAIL  London summer time: raised NotImplementedError: 
0/10 checks passed


#### Exercise 4 — HMAC webhook verification

A payment provider signs each webhook as `"sha256=" + hex(HMAC-SHA256(secret,
f"{timestamp}." + body))`, where `timestamp` is Unix seconds (as text) and `body`
the raw request bytes.

1. `sign_webhook(secret, timestamp, body)` returns that signature string.
2. `verify_webhook(secret, timestamp, body, signature, now, tolerance_seconds=300)`
   returns `True` only if the signature matches (compare with
   `hmac.compare_digest`) **and** `timestamp` is within `tolerance_seconds` of
   `now` in either direction (replay protection).

In [352]:
def sign_webhook(secret: bytes, timestamp: int, body: bytes) -> str:
    """'sha256=<hex>' signature of timestamp + '.' + body."""
    raise NotImplementedError


def verify_webhook(
    secret: bytes,
    timestamp: int,
    body: bytes,
    signature: str,
    now: int,
    tolerance_seconds: int = 300,
) -> bool:
    """True if the signature is valid and the timestamp is fresh."""
    raise NotImplementedError

Check exercise 4:

In [353]:
SECRET, OTHER_SECRET = b"whsec_lab_secret", b"whsec_attacker_guess"
BODY = b'{"event": "payment.captured", "order_id": 7}'
SENT_AT = int(datetime(2026, 11, 7, 9, 30, tzinfo=UTC).timestamp())
EXPECTED_SIGNATURE = (
    "sha256=" + hmac.new(SECRET, f"{SENT_AT}.".encode() + BODY, hashlib.sha256).hexdigest()
)
run_checks(
    [
        (
            "signature format and value",
            lambda: sign_webhook(SECRET, SENT_AT, BODY),
            EXPECTED_SIGNATURE,
        ),
        (
            "valid webhook",
            lambda: verify_webhook(SECRET, SENT_AT, BODY, EXPECTED_SIGNATURE, SENT_AT + 30),
            True,
        ),
        (
            "tampered body",
            lambda: verify_webhook(
                SECRET, SENT_AT, BODY.replace(b"7", b"8"), EXPECTED_SIGNATURE, SENT_AT
            ),
            False,
        ),
        (
            "wrong secret",
            lambda: verify_webhook(OTHER_SECRET, SENT_AT, BODY, EXPECTED_SIGNATURE, SENT_AT),
            False,
        ),
        (
            "timestamp is signed too",
            lambda: verify_webhook(SECRET, SENT_AT + 1, BODY, EXPECTED_SIGNATURE, SENT_AT),
            False,
        ),
        (
            "stale (replayed) webhook",
            lambda: verify_webhook(SECRET, SENT_AT, BODY, EXPECTED_SIGNATURE, SENT_AT + 301),
            False,
        ),
        (
            "from the future",
            lambda: verify_webhook(SECRET, SENT_AT, BODY, EXPECTED_SIGNATURE, SENT_AT - 301),
            False,
        ),
        (
            "custom tolerance",
            lambda: verify_webhook(
                SECRET, SENT_AT, BODY, EXPECTED_SIGNATURE, SENT_AT + 600, tolerance_seconds=900
            ),
            True,
        ),
        (
            "uses hmac.compare_digest",
            lambda: "compare_digest" in inspect.getsource(verify_webhook),
            True,
        ),
    ]
)

FAIL  signature format and value: raised NotImplementedError: 
FAIL  valid webhook: raised NotImplementedError: 
FAIL  tampered body: raised NotImplementedError: 
FAIL  wrong secret: raised NotImplementedError: 
FAIL  timestamp is signed too: raised NotImplementedError: 
FAIL  stale (replayed) webhook: raised NotImplementedError: 
FAIL  from the future: raised NotImplementedError: 
FAIL  custom tolerance: raised NotImplementedError: 
FAIL  uses hmac.compare_digest: expected True, got False
0/9 checks passed


#### Exercise 5 — top-K with `heapq`

`top_k(scores, k)` takes any iterable of `(name, score)` pairs (possibly a
generator too large to sort in memory) and returns the `k` highest as a list,
highest score first, ties by name (alphabetical). `k <= 0` returns `[]`. Use
`heapq` (`nlargest`, or a bounded heap of size `k`).

In [354]:
def top_k(scores: Iterable[tuple[str, int]], k: int) -> list[tuple[str, int]]:
    """The k highest (name, score) pairs, highest first, ties by name."""
    raise NotImplementedError

Check exercise 5:

In [355]:
LEADERBOARD = [("ravi", 75), ("meera", 90), ("john", 60), ("asha", 90), ("fatima", 82)]


def big_stream(count: int = 100_000):
    """A seeded generator of (name, score) pairs; never materialized as a list."""
    generator = random.Random(44)
    for number in range(count):
        yield f"customer-{number:06d}", generator.randint(0, 1_000_000)


def expected_top(pairs: list[tuple[str, int]], k: int) -> list[tuple[str, int]]:
    return sorted(pairs, key=lambda pair: (-pair[1], pair[0]))[:k]


run_checks(
    [
        (
            "top 3 with a tie",
            lambda: top_k(LEADERBOARD, 3),
            [("asha", 90), ("meera", 90), ("fatima", 82)],
        ),
        ("k = 1", lambda: top_k(LEADERBOARD, 1), [("asha", 90)]),
        ("k larger than the input", lambda: top_k(LEADERBOARD, 10), expected_top(LEADERBOARD, 10)),
        ("k = 0", lambda: top_k(LEADERBOARD, 0), []),
        ("generator input", lambda: top_k(iter(LEADERBOARD), 2), [("asha", 90), ("meera", 90)]),
        (
            "100,000-item stream",
            lambda: top_k(big_stream(), 5),
            expected_top(list(big_stream()), 5),
        ),
        ("uses heapq", lambda: "heapq" in inspect.getsource(top_k), True),
    ]
)

FAIL  top 3 with a tie: raised NotImplementedError: 
FAIL  k = 1: raised NotImplementedError: 
FAIL  k larger than the input: raised NotImplementedError: 
FAIL  k = 0: raised NotImplementedError: 
FAIL  generator input: raised NotImplementedError: 
FAIL  100,000-item stream: raised NotImplementedError: 
FAIL  uses heapq: expected True, got False
0/7 checks passed


#### Exercise 6 — build-step ordering with `graphlib`

`steps` maps each build step to the steps it depends on. A dependency that is not
itself a key is a step with no dependencies.

1. `build_batches(steps)` returns the steps in **batches** that can run in
   parallel: each batch contains every step whose dependencies are all in earlier
   batches, sorted alphabetically.
2. `build_order(steps)` returns one flat list: the batches concatenated.
3. Both raise `ValueError` with a message starting `"cycle: "` followed by the
   cycle joined with `" -> "` (as `graphlib.CycleError` reports it) when the
   dependencies form a cycle.

In [356]:
def build_batches(steps: Mapping[str, Iterable[str]]) -> list[list[str]]:
    """Parallel batches of steps, each sorted alphabetically."""
    raise NotImplementedError


def build_order(steps: Mapping[str, Iterable[str]]) -> list[str]:
    """All steps in a valid order (the batches, concatenated)."""
    raise NotImplementedError

Check exercise 6:

In [357]:
PIPELINE = {
    "package": ["test", "lint", "docs"],
    "test": ["compile"],
    "lint": [],
    "docs": ["fetch"],
    "compile": ["fetch"],
}
run_checks(
    [
        (
            "batches",
            lambda: build_batches(PIPELINE),
            [["fetch", "lint"], ["compile", "docs"], ["test"], ["package"]],
        ),
        (
            "order",
            lambda: build_order(PIPELINE),
            ["fetch", "lint", "compile", "docs", "test", "package"],
        ),
        ("implicit leaf step", lambda: build_order({"deploy": ["build"]}), ["build", "deploy"]),
        (
            "independent steps sorted",
            lambda: build_batches({"b": [], "c": [], "a": []}),
            [["a", "b", "c"]],
        ),
        ("empty", lambda: build_order({}), []),
        (
            "cycle message",
            lambda: (
                re.fullmatch(
                    r"ValueError: cycle: \w+( -> \w+)+",
                    raised_message(lambda: build_order({"migrate": ["seed"], "seed": ["migrate"]})),
                )
                is not None
            ),
            True,
        ),
        (
            "cycle in batches too",
            lambda: raised_message(
                lambda: build_batches({"a": ["b"], "b": ["c"], "c": ["a"]})
            ).split(":")[0],
            "ValueError",
        ),
    ]
)

FAIL  batches: raised NotImplementedError: 
FAIL  order: raised NotImplementedError: 
FAIL  implicit leaf step: raised NotImplementedError: 
FAIL  independent steps sorted: raised NotImplementedError: 
FAIL  empty: raised NotImplementedError: 
FAIL  cycle message: expected True, got False
FAIL  cycle in batches too: expected 'ValueError', got 'NotImplementedError'
0/7 checks passed


#### Exercise 7 — a file-organizer CLI with `--dry-run`

`organizer_main(argv)` implements a command-line tool with two `argparse`
subcommands and returns an exit code (it never calls `sys.exit`):

- `organize FOLDER [--by {extension,month}] [--dry-run]` moves every regular,
  non-hidden file directly inside `FOLDER` (not in subfolders) into a subfolder:
  `--by extension` (the default) uses the lower-case extension without the dot
  (`no_extension` when there is none); `--by month` uses the file's modification
  month in UTC as `YYYY-MM`. Files are processed in sorted name order, printing
  `moved NAME -> SUBFOLDER/NAME` for each, then `N files moved`. With
  `--dry-run` nothing changes on disk and the lines say `would move ...` and
  `N files would be moved`.
- `summary FOLDER` prints `EXTENSION: COUNT` lines for the same files, sorted by
  extension (using the same extension rule).
- If `FOLDER` is not a directory, print `error: FOLDER is not a directory` to
  **stderr** and return 2. Invalid arguments (argparse errors) also return 2.
  Success returns 0.

In [358]:
def organizer_main(argv: list[str]) -> int:
    """The file-organizer CLI; returns the exit code."""
    raise NotImplementedError

Check exercise 7:

In [359]:
ORGANIZER_FILES = {
    "Invoice.PDF": (2026, 10, 30),
    "photo.jpg": (2026, 11, 2),
    "notes.txt": (2026, 11, 5),
    "README": (2026, 9, 1),
    "data.csv": (2026, 11, 6),
    ".hidden": (2026, 11, 7),
}


def make_messy_folder() -> Path:
    """Six files (one hidden) with fixed modification times, plus a subfolder."""
    folder = LAB44_DIR / "messy"
    shutil.rmtree(folder, ignore_errors=True)
    (folder / "existing").mkdir(parents=True)
    (folder / "existing" / "keep.txt").write_text("untouched", encoding="utf-8")
    for name, (year, month, day) in ORGANIZER_FILES.items():
        (folder / name).write_text(name, encoding="utf-8")
        stamp = datetime(year, month, day, 12, tzinfo=UTC).timestamp()
        os.utime(folder / name, (stamp, stamp))
    return folder


def run_organizer(*arguments: str) -> tuple[int, list[str], str]:
    """(exit code, stdout lines, stderr); "messy"/"missing" are folders in the lab folder."""
    stdout, stderr = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(stdout), contextlib.redirect_stderr(stderr):
        exit_code = organizer_main(
            [
                str(LAB44_DIR / argument) if argument in ("messy", "missing") else argument
                for argument in arguments
            ]
        )
    return exit_code, stdout.getvalue().splitlines(), stderr.getvalue()


def folder_listing() -> list[str]:
    folder = LAB44_DIR / "messy"
    return sorted(
        path.relative_to(folder).as_posix() for path in folder.rglob("*") if path.is_file()
    )


def dry_run_then_listing() -> tuple[tuple[int, list[str], str], list[str]]:
    make_messy_folder()
    return run_organizer("organize", "messy", "--dry-run"), folder_listing()


def organize_then_listing(*options: str) -> tuple[list[str], list[str]]:
    make_messy_folder()
    return run_organizer("organize", "messy", *options)[1], folder_listing()


BY_EXTENSION = [
    "Invoice.PDF -> pdf/Invoice.PDF",
    "README -> no_extension/README",
    "data.csv -> csv/data.csv",
    "notes.txt -> txt/notes.txt",
    "photo.jpg -> jpg/photo.jpg",
]
run_checks(
    [
        (
            "summary",
            lambda: (make_messy_folder(), run_organizer("summary", "messy"))[1],
            (0, ["csv: 1", "jpg: 1", "no_extension: 1", "pdf: 1", "txt: 1"], ""),
        ),
        (
            "dry run output",
            lambda: dry_run_then_listing()[0],
            (0, [f"would move {line}" for line in BY_EXTENSION] + ["5 files would be moved"], ""),
        ),
        (
            "dry run changes nothing",
            lambda: dry_run_then_listing()[1],
            sorted([*ORGANIZER_FILES, "existing/keep.txt"]),
        ),
        (
            "organize by extension",
            lambda: organize_then_listing()[0],
            [f"moved {line}" for line in BY_EXTENSION] + ["5 files moved"],
        ),
        (
            "files really moved",
            lambda: organize_then_listing()[1],
            [
                ".hidden",
                "csv/data.csv",
                "existing/keep.txt",
                "jpg/photo.jpg",
                "no_extension/README",
                "pdf/Invoice.PDF",
                "txt/notes.txt",
            ],
        ),
        (
            "organize by month",
            lambda: organize_then_listing("--by", "month")[0],
            [
                "moved Invoice.PDF -> 2026-10/Invoice.PDF",
                "moved README -> 2026-09/README",
                "moved data.csv -> 2026-11/data.csv",
                "moved notes.txt -> 2026-11/notes.txt",
                "moved photo.jpg -> 2026-11/photo.jpg",
                "5 files moved",
            ],
        ),
        (
            "missing folder",
            lambda: run_organizer("organize", "missing"),
            (2, [], f"error: {LAB44_DIR / 'missing'} is not a directory\n"),
        ),
        ("bad --by value", lambda: run_organizer("organize", "messy", "--by", "size")[0], 2),
    ]
)

FAIL  summary: raised NotImplementedError: 
FAIL  dry run output: raised NotImplementedError: 
FAIL  dry run changes nothing: raised NotImplementedError: 
FAIL  organize by extension: raised NotImplementedError: 
FAIL  files really moved: raised NotImplementedError: 
FAIL  organize by month: raised NotImplementedError: 
FAIL  missing folder: raised NotImplementedError: 
FAIL  bad --by value: raised NotImplementedError: 
0/8 checks passed


### Recap — Unit 4.4

| Concept | One-line takeaway | JS / Node equivalent |
| --- | --- | --- |
| `pathlib`, `os`, `sys`, `shutil` | Paths are objects (`/`, `.name`, `.glob`, `.walk`); always pass `encoding`; atomic writes with `os.replace` | `path`, `fs`, `process` |
| `subprocess` | `run([...], capture_output=True, text=True, check=True, timeout=...)`; `Popen` to stream; never `shell=True` with input | `child_process` |
| Dates and times | Aware datetimes, `ZoneInfo("Asia/Kolkata")`, store UTC, ISO 8601; DST: gaps, folds, wall-clock arithmetic; `monotonic`/`perf_counter` for durations | `Date`, `Intl`, `performance.now()` |
| `re` | `match`/`search`/`fullmatch`, named groups `(?P<n>)`, lazy `*?`, `VERBOSE`, lookarounds, `sub` with a function; beware nested quantifiers | `RegExp` |
| `json`, `csv` | `default=`/`JSONEncoder`, `object_hook`, `parse_float=Decimal`; `DictReader`/`DictWriter` with `newline=""` | `JSON`, csv libraries |
| Algorithms | `heapq` priority queues and top-k, `bisect` bands, `statistics`, `math`, `graphlib` for dependency order | npm packages |
| Randomness and security | `random` (seeded, reproducible) vs `secrets`; `hashlib`, slow KDFs for passwords, `hmac` + `compare_digest`, `uuid7`, `base64` is not encryption, `urllib.parse` | `Math.random`, `crypto`, `URL` |
| CLIs | `argparse` subcommands, `set_defaults(handler=...)`, `exit_on_error=False` for tests; typer/click in Unit 4.6 | commander, yargs |
| Text utilities | `textwrap`, `string.Template` (safe for user templates), `difflib`, `pprint` | template literals, `diff` |

**Further reading**

- [`pathlib`](https://docs.python.org/3/library/pathlib.html),
  [`os`](https://docs.python.org/3/library/os.html),
  [`shutil`](https://docs.python.org/3/library/shutil.html),
  [`subprocess`](https://docs.python.org/3/library/subprocess.html)
- [`datetime`](https://docs.python.org/3/library/datetime.html),
  [`zoneinfo`](https://docs.python.org/3/library/zoneinfo.html),
  [`time`](https://docs.python.org/3/library/time.html)
- [`re`](https://docs.python.org/3/library/re.html),
  [Regular Expression HOWTO](https://docs.python.org/3/howto/regex.html),
  [`json`](https://docs.python.org/3/library/json.html),
  [`csv`](https://docs.python.org/3/library/csv.html)
- [`heapq`](https://docs.python.org/3/library/heapq.html),
  [`bisect`](https://docs.python.org/3/library/bisect.html),
  [`statistics`](https://docs.python.org/3/library/statistics.html),
  [`graphlib`](https://docs.python.org/3/library/graphlib.html)
- [`random`](https://docs.python.org/3/library/random.html),
  [`secrets`](https://docs.python.org/3/library/secrets.html),
  [`hashlib`](https://docs.python.org/3/library/hashlib.html),
  [`hmac`](https://docs.python.org/3/library/hmac.html),
  [`uuid`](https://docs.python.org/3/library/uuid.html),
  [`base64`](https://docs.python.org/3/library/base64.html),
  [`urllib.parse`](https://docs.python.org/3/library/urllib.parse.html)
- [`argparse`](https://docs.python.org/3/library/argparse.html),
  [`textwrap`](https://docs.python.org/3/library/textwrap.html),
  [`string`](https://docs.python.org/3/library/string.html),
  [`difflib`](https://docs.python.org/3/library/difflib.html),
  [`pprint`](https://docs.python.org/3/library/pprint.html)
- [PEP 428](https://peps.python.org/pep-0428/) (pathlib),
  [PEP 495](https://peps.python.org/pep-0495/) (`fold`),
  [PEP 615](https://peps.python.org/pep-0615/) (zoneinfo),
  [PEP 506](https://peps.python.org/pep-0506/) (secrets),
  [PEP 686](https://peps.python.org/pep-0686/) (UTF-8 mode),
  [PEP 750](https://peps.python.org/pep-0750/) (template strings),
  [RFC 9562](https://www.rfc-editor.org/rfc/rfc9562) (UUIDv7)

## Unit 4.5 — File I/O, Encodings and Serialization

**Objectives**

- Explain what `open()` builds (raw file, buffer, text wrapper) and choose modes,
  buffering, encodings and newline handling deliberately.
- Read and write files safely: line by line, in binary chunks, with `seek`/`tell`,
  and atomically.
- Use in-memory and temporary files for tests and pipelines.
- Parse binary formats with `struct`, `memoryview` and `mmap`.
- Understand encodings well enough to fix mojibake and choose error handlers.
- Pick a serialization format (JSON, CSV, TOML, YAML, XML, MessagePack, pickle)
  and know which ones are safe for untrusted input.
- Compress and archive files, and extract archives without path traversal.
- Store data in SQLite with parameterized queries and transactions.

**JS bridge: Node `fs` and friends → Python**

| Node / browser | Python | The difference that bites |
| --- | --- | --- |
| `fs.readFileSync(p, "utf8")` | `open(p, encoding="utf-8").read()` | Text vs binary is the **mode** (`"r"` vs `"rb"`); always pass `encoding` |
| `fs.createReadStream` | iterate the file object, or `read(chunk_size)` | Files are iterators of lines |
| `Buffer` | `bytes` / `bytearray` / `memoryview` | `str` and `bytes` never mix implicitly |
| `TextDecoder("utf-16")` | `bytes.decode("utf-16")`, `codecs` | Error handlers: `strict`, `replace`, `ignore`, `surrogateescape` |
| `DataView.getUint32(o, false)` | `struct.unpack_from(">I", data, o)` | Format strings choose endianness and sizes |
| `JSON.stringify` / `structuredClone` | `json`, `pickle` | **Never unpickle untrusted data** |
| `zlib`, `tar` packages | `gzip`, `bz2`, `lzma`, `compression.zstd`, `zipfile`, `tarfile` | All in the standard library |
| `better-sqlite3` | `sqlite3` | Built in; use `?` placeholders |

Every demo writes under `notebooks/_scratch/unit45/` (paths are shown as
`<scratch>`). The file contents are fixed, so outputs are the same on every run.

In [360]:
import bz2
import codecs
import csv
import dataclasses
import gzip
import importlib.util
import io
import json
import lzma
import mmap
import os
import pickle
import shutil
import sqlite3
import struct
import sys
import tarfile
import tempfile
import textwrap
import tomllib
import uuid
import wave
import zipfile
from collections.abc import Callable
from datetime import UTC, date, datetime
from decimal import Decimal
from pathlib import Path

REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
UNIT45_DIR = scratch_dir / "unit45"
UNIT45_DIR.mkdir(parents=True, exist_ok=True)
FIXED_MOMENT = datetime(2026, 11, 7, 9, 30, tzinfo=UTC)


def tidy(text: str) -> str:
    """Show absolute paths as <scratch>, <repo> or ~."""
    for folder, label in ((UNIT45_DIR, "<scratch>"), (REPO_ROOT, "<repo>"), (Path.home(), "~")):
        text = text.replace(str(folder), label)
    return text


def fresh_dir(name: str) -> Path:
    """An empty folder <scratch>/<name>, recreated on every run."""
    folder = UNIT45_DIR / name
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    return folder


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return tidy(f"{type(error).__name__}: {error}")
    return "no error"


io_demo = fresh_dir("io_demo")
print(f"scratch folder: {io_demo.relative_to(REPO_ROOT).as_posix()}")
print(f"{sys.getdefaultencoding()=}, {sys.flags.utf8_mode=}")

scratch folder: notebooks/_scratch/unit45/io_demo
sys.getdefaultencoding()='utf-8', sys.flags.utf8_mode=1


### 4.5.1 `open()` in depth

**Modes** combine an action with text or binary:

| Mode | Meaning | If the file exists | If it does not |
| --- | --- | --- | --- |
| `"r"` | read (default) | read from the start | `FileNotFoundError` |
| `"w"` | write | **truncate** to empty | create |
| `"a"` | append | write at the end | create |
| `"x"` | exclusive create | `FileExistsError` | create |
| `"+"` | read **and** write (`"r+"`, `"w+"`, `"a+"`) | as the base mode | as the base mode |
| `"b"` / `"t"` | binary (`bytes`) / text (`str`, the default) | | |

`"x"` is the safe way to create a file only if nobody else has.

In [361]:
orders_path = io_demo / "orders.txt"
with open(orders_path, "w", encoding="utf-8") as orders_file:
    orders_file.write("TEA-500,2\n")
with open(orders_path, "a", encoding="utf-8") as orders_file:
    orders_file.write("MUG-02,1\n")
print(f"{orders_path.read_text(encoding='utf-8')!r}")
print(raised_message(lambda: open(orders_path, "x", encoding="utf-8")))
print(raised_message(lambda: open(io_demo / "missing.txt", encoding="utf-8")))
with open(orders_path, "rb") as binary_file:
    print(f"{binary_file.read()!r}  (binary mode: bytes)")

'TEA-500,2\nMUG-02,1\n'
FileExistsError: [Errno 17] File exists: '<scratch>/io_demo/orders.txt'
FileNotFoundError: [Errno 2] No such file or directory: '<scratch>/io_demo/missing.txt'
b'TEA-500,2\nMUG-02,1\n'  (binary mode: bytes)


**Predict the output.**

```python
from pathlib import Path

notes = Path("notes.txt")
notes.write_text("order 7: paid\n", encoding="utf-8")
with open(notes, "w", encoding="utf-8") as notes_file:
    pass  # opened for writing, nothing written
print(repr(notes.read_text(encoding="utf-8")))
```

<details><summary>Answer</summary>

`''`. Mode `"w"` **truncates** the file the moment it is opened, before you write
anything, so opening a file for writing by mistake destroys its contents. Use `"a"`
to append, `"x"` to refuse existing files, or the atomic-write pattern (4.5.2) to
replace a file safely.

</details>

Run the next cell to check your prediction.

In [362]:
notes = io_demo / "notes.txt"
notes.write_text("order 7: paid\n", encoding="utf-8")
with open(notes, "w", encoding="utf-8") as notes_file:
    pass  # opened for writing, nothing written
print(repr(notes.read_text(encoding="utf-8")))

''


**The I/O stack.** `open()` builds up to three layers, and the object you get back
depends on the mode:

```mermaid
flowchart LR
    CODE["your code<br/>str"] --> TEXT["TextIOWrapper<br/>encoding, newlines"]
    TEXT --> BUF["BufferedReader / BufferedWriter<br/>buffer (io.DEFAULT_BUFFER_SIZE)"]
    BUF --> RAW["FileIO<br/>OS read()/write() calls"]
    RAW --> OS["file descriptor"]
```

<details><summary>ASCII version</summary>

```text
text mode:   TextIOWrapper -> BufferedReader/Writer -> FileIO -> OS
"rb"/"wb":                    BufferedReader/Writer -> FileIO -> OS
buffering=0:                                           FileIO -> OS
```

</details>

Each layer is reachable through an attribute (`.buffer`, `.raw`), which is how you
get binary access to a text file such as `sys.stdout.buffer`. The buffer size is
`io.DEFAULT_BUFFER_SIZE`: 128 KiB on this 3.15 build (it used to be 8 KiB).

In [363]:
with open(orders_path, encoding="utf-8") as text_file:
    print(
        f"{type(text_file).__name__} -> {type(text_file.buffer).__name__} -> "
        f"{type(text_file.buffer.raw).__name__}"
    )
    print(f"{text_file.encoding=}, {text_file.buffer.raw.fileno() >= 0=}")
with open(orders_path, "rb") as binary_file:
    print(f"'rb': {type(binary_file).__name__} -> {type(binary_file.raw).__name__}")
with open(orders_path, "rb", buffering=0) as unbuffered_file:
    print(f"'rb', buffering=0: {type(unbuffered_file).__name__}")
print(f"{io.DEFAULT_BUFFER_SIZE=}")

TextIOWrapper -> BufferedReader -> FileIO
text_file.encoding='utf-8', text_file.buffer.raw.fileno() >= 0=True
'rb': BufferedReader -> FileIO
'rb', buffering=0: FileIO
io.DEFAULT_BUFFER_SIZE=131072


**Buffering.** Writes collect in the buffer and reach the OS when it fills, on
`flush()` or on `close()`. That is why output can seem to "disappear" when a
program crashes, and why another reader may not see data yet. `buffering=1`
means line-buffered (text mode only); `buffering=0` means unbuffered (binary
only).

In [364]:
log_path = io_demo / "buffered.log"
writer = open(log_path, "w", encoding="utf-8")
writer.write("order 7 placed\n")
print(f"before flush: {log_path.read_text(encoding='utf-8')!r}")
writer.flush()
print(f"after flush:  {log_path.read_text(encoding='utf-8')!r}")
writer.close()
with open(log_path, "a", encoding="utf-8", buffering=1) as line_buffered:
    line_buffered.write("order 8 placed\n")  # a newline flushes
    print(f"line-buffered: {log_path.read_text(encoding='utf-8').splitlines()}")

before flush: ''
after flush:  'order 7 placed\n'
line-buffered: ['order 7 placed', 'order 8 placed']


**Encodings.** A text file is bytes on disk; the `encoding` turns them into `str`.
Without the argument Python used the platform's locale encoding (cp1252 on many
Windows machines), so code worked on one machine and broke on another. *Changed
in 3.15:* UTF-8 mode is the default (PEP 686), but passing
`encoding="utf-8"` keeps code correct on every version, and linters (ruff's
`PLW1514`) can enforce it. `encoding="locale"` asks for the old behaviour
explicitly.

In [365]:
price_path = io_demo / "price.txt"
price_path.write_text("Masala tea: ₹240\n", encoding="utf-8")
print(f"{price_path.read_bytes()=}")
print(f"{price_path.read_text(encoding='utf-8')=}")
print(f"{price_path.read_text(encoding='cp1252')=}  (wrong encoding: mojibake)")
print(raised_message(lambda: price_path.read_text(encoding="ascii")))

price_path.read_bytes()=b'Masala tea: \xe2\x82\xb9240\n'
price_path.read_text(encoding='utf-8')='Masala tea: ₹240\n'
price_path.read_text(encoding='cp1252')='Masala tea: â‚¹240\n'  (wrong encoding: mojibake)
UnicodeDecodeError: 'ascii' codec can't decode byte 0xe2 in position 12: ordinal not in range(128)


**Newline handling.** In text mode, `newline=None` (the default) translates `\r\n`
and `\r` to `\n` when reading and writes `os.linesep` when writing. `newline=""`
turns translation off (the `csv` module needs this, Unit 4.4.5); `newline="\n"`
writes Unix line endings on every OS. Binary mode never translates.

In [366]:
mixed_path = io_demo / "mixed_endings.txt"
mixed_path.write_bytes(b"windows\r\nunix\nold mac\r")
with open(mixed_path, encoding="utf-8") as default_mode:
    print(f"newline=None: {default_mode.read()!r}")
with open(mixed_path, encoding="utf-8", newline="") as untranslated:
    print(f"newline='':   {untranslated.read()!r}")
with open(mixed_path, encoding="utf-8") as default_mode:
    default_mode.read()
    print(f"endings seen: {default_mode.newlines!r}")
with open(io_demo / "unix.txt", "w", encoding="utf-8", newline="\n") as unix_file:
    unix_file.write("a\nb\n")
print(f"{(io_demo / 'unix.txt').read_bytes()=}")

newline=None: 'windows\nunix\nold mac\n'
newline='':   'windows\r\nunix\nold mac\r'
endings seen: ('\r', '\n', '\r\n')
(io_demo / 'unix.txt').read_bytes()=b'a\nb\n'


### 4.5.2 Reading and writing patterns

**Line iteration.** A text file object is an iterator of lines (each ending in
`\n`, except perhaps the last). Iterating reads one buffer at a time, so a 10 GB
log needs only a few KB of memory. `read()` and `readlines()` load everything.

In [367]:
access_log = io_demo / "access.log"
access_log.write_text(
    "".join(f"2026-11-07 GET /products/{n} {200 if n % 7 else 500}\n" for n in range(1, 1001)),
    encoding="utf-8",
)
with access_log.open(encoding="utf-8") as log_file:
    errors = [line.split()[2] for line in log_file if line.rstrip().endswith("500")]
print(f"{len(errors)=}, {errors[:3]=}")
with access_log.open(encoding="utf-8") as log_file:
    print(f"{next(log_file)!r}, {log_file.readline()!r}")

len(errors)=142, errors[:3]=['/products/7', '/products/14', '/products/21']
'2026-11-07 GET /products/1 200\n', '2026-11-07 GET /products/2 200\n'


**Chunked binary reads.** For binary data (or text without useful lines) read a
fixed number of bytes at a time. The walrus operator makes the loop compact;
`read()` returns `b""` at the end of the file. This is how you hash or copy large
files.

In [368]:
import hashlib


def sha256_of(path: Path, chunk_size: int = 64 * 1024) -> tuple[str, int]:
    """Hash a file in chunks; return (hex digest, number of chunks read)."""
    digest, chunks = hashlib.sha256(), 0
    with path.open("rb") as binary_file:
        while chunk := binary_file.read(chunk_size):
            digest.update(chunk)
            chunks += 1
    return digest.hexdigest()[:16], chunks


print(f"{access_log.stat().st_size=}")
print(f"{sha256_of(access_log)=}")
print(f"{sha256_of(access_log, chunk_size=1024)=}  (same digest, more reads)")

access_log.stat().st_size=32893
sha256_of(access_log)=('523280e876776c3c', 1)
sha256_of(access_log, chunk_size=1024)=('523280e876776c3c', 33)  (same digest, more reads)


**`seek` and `tell`.** `tell()` gives the current position; `seek(offset,
whence)` moves it: `whence=0` from the start (`os.SEEK_SET`), `1` from the current
position, `2` from the end. In **binary** mode positions are byte offsets. In text
mode `tell()` returns an opaque cookie and only `seek(cookie)` or `seek(0, 2)` are
allowed, because characters and bytes are not the same size.

In [369]:
with access_log.open("rb") as binary_file:
    binary_file.seek(-40, os.SEEK_END)
    print(f"{binary_file.tell()=}, {binary_file.read()!r}")
    binary_file.seek(11)
    print(f"{binary_file.read(3)=}, {binary_file.tell()=}")
    binary_file.seek(5, os.SEEK_CUR)
    print(f"{binary_file.read(9)=}")
with access_log.open(encoding="utf-8") as text_file:
    print(raised_message(lambda: text_file.seek(-10, os.SEEK_END)))

binary_file.tell()=32853, b'9 200\n2026-11-07 GET /products/1000 200\n'
binary_file.read(3)=b'GET', binary_file.tell()=14
binary_file.read(9)=b'ducts/1 2'
UnsupportedOperation: can't do nonzero end-relative seeks


**`flush` and `fsync`.** `flush()` moves Python's buffer into the operating
system; the OS may still keep the data in its page cache for seconds.
`os.fsync(fd)` asks the OS to write it to the physical disk, which matters before
you tell someone "saved" (a payment record, a database journal). It is slow, so
call it only where durability matters.

In [370]:
ledger_path = io_demo / "ledger.txt"
with ledger_path.open("a", encoding="utf-8") as ledger:
    ledger.write("2026-11-07 order 7 paid 679.00\n")
    ledger.flush()  # Python buffer -> OS
    os.fsync(ledger.fileno())  # OS cache -> disk
print(f"{ledger_path.read_text(encoding='utf-8')!r}")

'2026-11-07 order 7 paid 679.00\n'


**Atomic writes with `os.replace`.** Writing over a file in place leaves a
truncated or half-written file if the program crashes or the disk fills mid-write.
The safe pattern (Unit 4.4.1 showed the idea): write a temporary file **in the
same folder** (same file system), flush and `fsync` it, then `os.replace` it over
the target. The rename is atomic: readers see the old file or the new one. If
anything fails, the target is untouched and the temporary file is removed.

In [371]:
def write_text_atomically(target: Path, text: str) -> None:
    """Replace target's contents in one atomic step, or not at all."""
    handle, temporary_name = tempfile.mkstemp(dir=target.parent, prefix=f".{target.name}.")
    try:
        with os.fdopen(handle, "w", encoding="utf-8") as temporary_file:
            temporary_file.write(text)
            temporary_file.flush()
            os.fsync(temporary_file.fileno())
        os.replace(temporary_name, target)
    except BaseException:
        os.unlink(temporary_name)
        raise


stock_path = io_demo / "stock.csv"
write_text_atomically(stock_path, "sku,quantity\nTEA-500,40\n")
print(f"{stock_path.read_text(encoding='utf-8')!r}")
print(raised_message(lambda: write_text_atomically(stock_path, "sku,quantity\n" + "MUG-02," + 7)))
print(f"unchanged: {stock_path.read_text(encoding='utf-8')!r}")
print(f"leftover temp files: {sorted(path.name for path in io_demo.glob('.stock.csv.*'))}")

'sku,quantity\nTEA-500,40\n'
TypeError: can only concatenate str (not "int") to str
unchanged: 'sku,quantity\nTEA-500,40\n'
leftover temp files: []


The second call failed while *building* the text (`str + int`), before anything was
written. A failure during the write (a full disk, an encoding error) is handled the
same way by the `except` branch: the temporary file is deleted and the target
never changes.

### 4.5.3 In-memory and temporary files

**`io.StringIO`** and **`io.BytesIO`** are file objects backed by memory. Any code
that accepts a file (the `csv` module, `json.dump`, a parser, a logger handler)
works with them, which makes them ideal for tests and for building a payload
before sending it over the network.

In [372]:
report = io.StringIO()
writer = csv.writer(report)
writer.writerows([["sku", "quantity"], ["TEA-500", 2], ["MUG-02", 1]])
print(f"{report.getvalue()!r}")
report.seek(0)
print(f"{list(csv.DictReader(report))=}")

payload = io.BytesIO()
payload.write(b"\x89PNG")
payload.write(struct.pack(">I", 13))
print(f"{payload.getvalue()=}, {payload.tell()=}")
print(f"{io.StringIO('line 1\nline 2\n').readlines()=}")

'sku,quantity\r\nTEA-500,2\r\nMUG-02,1\r\n'
list(csv.DictReader(report))=[{'sku': 'TEA-500', 'quantity': '2'}, {'sku': 'MUG-02', 'quantity': '1'}]
payload.getvalue()=b'\x89PNG\x00\x00\x00\r', payload.tell()=8
io.StringIO('line 1\nline 2\n').readlines()=['line 1\n', 'line 2\n']


**`tempfile`** creates files and folders that do not clash with anything else and
are cleaned up for you:

| Function | Gives | Cleaned up |
| --- | --- | --- |
| `TemporaryFile()` | an anonymous file (no visible name on POSIX) | on close |
| `NamedTemporaryFile(delete=...)` | a file with a real path (`.name`) | on close, or with `delete_on_close=False` (3.12) when the `with` block ends |
| `TemporaryDirectory()` | a folder | when the `with` block ends |
| `mkstemp()` / `mkdtemp()` | a low-level handle + path / a folder path | never: you delete it |
| `SpooledTemporaryFile(max_size)` | memory until `max_size`, then a real file | on close |

They live in `tempfile.gettempdir()` unless you pass `dir=`.

In [373]:
with tempfile.TemporaryDirectory(prefix="shop-") as workspace:
    workspace_path = Path(workspace)
    (workspace_path / "export.csv").write_text("sku\nTEA-500\n", encoding="utf-8")
    print(
        f"inside:  {workspace_path.name.startswith('shop-')=}, "
        f"{[p.name for p in workspace_path.iterdir()]}"
    )
print(f"after:   {workspace_path.exists()=}")

with tempfile.NamedTemporaryFile(
    "w+", suffix=".json", encoding="utf-8", delete_on_close=False
) as named:
    named.write('{"ok": true}')
    named.close()  # other programs can now open it by name (needed on Windows)
    print(
        f"{Path(named.name).suffix=}, {json.loads(Path(named.name).read_text(encoding='utf-8'))=}"
    )
print(f"deleted at block end: {not Path(named.name).exists()}")

with tempfile.SpooledTemporaryFile(max_size=100) as spooled:
    spooled.write(b"x" * 50)
    print(f"50 bytes -> in memory: {isinstance(spooled._file, io.BytesIO)}")
    spooled.write(b"x" * 100)
    print(f"150 bytes -> on disk: {not isinstance(spooled._file, io.BytesIO)}")

inside:  workspace_path.name.startswith('shop-')=True, ['export.csv']
after:   workspace_path.exists()=False
Path(named.name).suffix='.json', json.loads(Path(named.name).read_text(encoding='utf-8'))={'ok': True}
deleted at block end: True
50 bytes -> in memory: True
150 bytes -> on disk: True


### 4.5.4 Binary formats

**`struct`** converts between Python values and packed C-style binary data,
described by a format string. The first character sets byte order and alignment;
the rest are field codes.

| Prefix | Byte order | | Code | C type | Size |
| --- | --- | --- | --- | --- | --- |
| `<` | little-endian (x86, ARM, most file formats from Microsoft) | | `B` / `b` | unsigned / signed char | 1 |
| `>` or `!` | big-endian ("network order": PNG, TCP/IP) | | `H` / `h` | unsigned / signed short | 2 |
| `=` | native order, standard sizes | | `I` / `i` | unsigned / signed int | 4 |
| `@` (default) | native order **and** native alignment (padding) | | `Q` / `q` | unsigned / signed long long | 8 |
| | | | `f` / `d` | float / double | 4 / 8 |
| | | | `4s` | 4 bytes | 4 |

**Endianness** is the order of the bytes of a multi-byte number: the integer
`0x01020304` is stored as `01 02 03 04` big-endian and `04 03 02 01`
little-endian. Always use an explicit prefix for file formats.

In [374]:
number = 0x01020304
print(f"big-endian:    {struct.pack('>I', number).hex(' ')}")
print(f"little-endian: {struct.pack('<I', number).hex(' ')}")
print(f"this machine:  {sys.byteorder}")
record = struct.pack("<HIf4s", 7, 67900, 2.5, b"PNQ1")  # order id, paise, weight kg, warehouse
print(f"{record.hex(' ')}  ({len(record)} bytes, {struct.calcsize('<HIf4s')=})")
print(f"{struct.unpack('<HIf4s', record)=}")
print(f"{struct.calcsize('@HI')=} (native alignment pads), {struct.calcsize('<HI')=}")
print(raised_message(lambda: struct.pack("<H", 70_000)))

big-endian:    01 02 03 04
little-endian: 04 03 02 01
this machine:  little
07 00 3c 09 01 00 00 00 20 40 50 4e 51 31  (14 bytes, struct.calcsize('<HIf4s')=14)
struct.unpack('<HIf4s', record)=(7, 67900, 2.5, b'PNQ1')
struct.calcsize('@HI')=8 (native alignment pads), struct.calcsize('<HI')=6
error: 'H' format requires 0 <= number <= 65535


**Parsing a PNG header.** Every PNG starts with an 8-byte signature followed by
chunks: a 4-byte big-endian length, a 4-byte type, the data and a CRC. The first
chunk, `IHDR`, holds width, height, bit depth, colour type, compression, filter
and interlace. `struct.unpack_from(fmt, buffer, offset)` reads at an offset without
slicing. The demo image is written by hand with `zlib`, so no image library is
needed.

In [375]:
import zlib


def tiny_png(width: int, height: int) -> bytes:
    """A valid RGB PNG (solid colour) built by hand, no image library needed."""

    def chunk(kind: bytes, data: bytes) -> bytes:
        return (
            struct.pack(">I", len(data)) + kind + data + struct.pack(">I", zlib.crc32(kind + data))
        )

    header = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    pixels = b"".join(b"\x00" + b"\xff\x99\x00" * width for _ in range(height))
    return (
        b"\x89PNG\r\n\x1a\n"
        + chunk(b"IHDR", header)
        + chunk(b"IDAT", zlib.compress(pixels))
        + chunk(b"IEND", b"")
    )


PNG_SIGNATURE = b"\x89PNG\r\n\x1a\n"


def png_header(data: bytes) -> dict[str, int]:
    if data[:8] != PNG_SIGNATURE:
        raise ValueError("not a PNG file")
    length, kind = struct.unpack_from(">I4s", data, 8)
    if kind != b"IHDR" or length != 13:
        raise ValueError("first chunk is not IHDR")
    width, height, depth, colour, _, _, interlace = struct.unpack_from(">IIBBBBB", data, 16)
    return {
        "width": width,
        "height": height,
        "bit_depth": depth,
        "colour_type": colour,
        "interlace": interlace,
    }


png_bytes = tiny_png(640, 480)
(io_demo / "banner.png").write_bytes(png_bytes)
print(f"{png_bytes[:24].hex(' ')}")
print(f"{png_header(png_bytes)=}")
print(raised_message(lambda: png_header(b"GIF89a" + bytes(30))))

89 50 4e 47 0d 0a 1a 0a 00 00 00 0d 49 48 44 52 00 00 02 80 00 00 01 e0
png_header(png_bytes)={'width': 640, 'height': 480, 'bit_depth': 8, 'colour_type': 2, 'interlace': 0}
ValueError: not a PNG file


**`memoryview`** gives a zero-copy view of a buffer (`bytes`, `bytearray`, `mmap`,
NumPy arrays): slicing it does not copy the data, which matters when parsing large
binary files or network packets. On a `bytearray` the view is writable. `.cast()`
reinterprets the bytes as another C type.

In [376]:
big_buffer = bytearray(png_bytes)
header_view = memoryview(big_buffer)[16:29]  # no copy
print(f"{header_view.nbytes=}, {header_view.readonly=}, {bytes(header_view[:4]).hex()=}")
header_view[0:4] = struct.pack(">I", 1280)  # writes into big_buffer
print(f"{png_header(bytes(big_buffer))['width']=}")
sample_rates = memoryview(struct.pack("<4H", 8000, 16000, 44100, 48000)).cast("H")
print(f"{sample_rates.tolist()=}, {sample_rates.format=}, {sample_rates.itemsize=}")
print(
    f"{sys.getsizeof(png_bytes[16:29_000])=} (slice of bytes: a copy), "
    f"{sys.getsizeof(memoryview(png_bytes)[16:29_000])=} (view)"
)

header_view.nbytes=13, header_view.readonly=False, bytes(header_view[:4]).hex()='00000280'
png_header(bytes(big_buffer))['width']=1280
sample_rates.tolist()=[8000, 16000, 44100, 48000], sample_rates.format='H', sample_rates.itemsize=2
sys.getsizeof(png_bytes[16:29_000])=2143 (slice of bytes: a copy), sys.getsizeof(memoryview(png_bytes)[16:29_000])=184 (view)


**`mmap`** maps a file into memory: you index and slice it like a `bytearray`, and
the OS loads only the pages you touch. Good for random access in large files
(search for a marker, read a record at an offset) and for sharing memory between
processes. Opened with `r+b`, writes go back to the file.

In [377]:
records_path = io_demo / "records.bin"
records_path.write_bytes(
    b"".join(struct.pack("<I8s", n, f"SKU{n:05d}".encode()) for n in range(10_000))
)
RECORD = struct.Struct("<I8s")
with records_path.open("r+b") as records_file, mmap.mmap(records_file.fileno(), 0) as mapped:
    print(f"{len(mapped)=} bytes, record size {RECORD.size}")
    print(f"record 4321: {RECORD.unpack_from(mapped, 4321 * RECORD.size)}")
    offset = mapped.find(b"SKU09999")
    print(f"{offset=}, found record {offset // RECORD.size}")
    mapped[0:4] = struct.pack("<I", 424242)  # write through the map
with records_path.open("rb") as records_file:
    print(f"after writing via mmap: {RECORD.unpack(records_file.read(RECORD.size))}")

len(mapped)=120000 bytes, record size 12
record 4321: (4321, b'SKU04321')
offset=119992, found record 9999
after writing via mmap: (424242, b'SKU00000')


### 4.5.5 Encodings

A `str` is a sequence of Unicode **code points**; an encoding maps them to bytes.

| Encoding | Bytes per character | Notes |
| --- | --- | --- |
| UTF-8 | 1–4 (ASCII is 1) | the default for files, the web and this course |
| UTF-16 | 2 or 4 | Windows APIs, JavaScript strings internally; needs a byte order (`-le`/`-be`) or a BOM |
| latin-1 (ISO-8859-1) | 1 | maps bytes 0–255 to the first 256 code points: **decodes any bytes**, never errors |
| cp1252 | 1 | Windows' variant of latin-1 (`€`, smart quotes at 0x80–0x9F) |
| ASCII | 1 | only 128 characters |

In [378]:
text = "₹240 café 🍵"
for encoding in ("utf-8", "utf-16-le", "utf-16", "utf-32-le"):
    encoded = text.encode(encoding)
    print(f"{encoding:<10} {len(encoded):>3} bytes  {encoded[:12].hex(' ')}...")
print(f"{len(text)=} code points, {[f'U+{ord(character):04X}' for character in '₹é🍵']}")
print(f"{bytes(range(0x80, 0x86)).decode('latin-1')!r}  (latin-1 decodes anything)")
print(raised_message(lambda: text.encode("latin-1")))

utf-8       17 bytes  e2 82 b9 32 34 30 20 63 61 66 c3 a9...
utf-16-le   24 bytes  b9 20 32 00 34 00 30 00 20 00 63 00...
utf-16      26 bytes  ff fe b9 20 32 00 34 00 30 00 20 00...
utf-32-le   44 bytes  b9 20 00 00 32 00 00 00 34 00 00 00...
len(text)=11 code points, ['U+20B9', 'U+00E9', 'U+1F375']
'\x80\x81\x82\x83\x84\x85'  (latin-1 decodes anything)
UnicodeEncodeError: 'latin-1' codec can't encode character '\u20b9' in position 0: ordinal not in range(256)


**Predict the output.**

```python
print("€".encode("cp1252"), "€".encode("utf-8"))
print("€".encode("latin-1"))
```

<details><summary>Answer</summary>

`b'\x80' b'\xe2\x82\xac'`, then a `UnicodeEncodeError`. cp1252 (Windows) put `€` at
byte 0x80, UTF-8 needs three bytes, and latin-1 has no euro sign at all (it predates
the euro); its 0x80 is an invisible control character. That difference is why "latin-1"
and "cp1252" files are not interchangeable.

</details>

Run the next cell to check your prediction.

In [379]:
print("€".encode("cp1252"), "€".encode())
print(raised_message(lambda: "€".encode("latin-1")))

b'\x80' b'\xe2\x82\xac'
UnicodeEncodeError: 'latin-1' codec can't encode character '\u20ac' in position 0: ordinal not in range(256)


**BOMs (byte order marks).** UTF-16 files start with `FF FE` or `FE FF` so the
reader knows the byte order. Some Windows tools also put `EF BB BF` (the UTF-8
BOM) at the start of UTF-8 files, which then shows up as a stray `\ufeff`
character, often breaking the first CSV header. The `utf-8-sig` codec removes it
when reading (and writes it, if you ask, for Excel).

In [380]:
excel_export = io_demo / "excel_export.csv"
excel_export.write_bytes(codecs.BOM_UTF8 + b"sku,quantity\nTEA-500,2\n")
print(f"{excel_export.read_bytes()[:6]=}")
with excel_export.open(encoding="utf-8", newline="") as wrong:
    print(f"utf-8:     {next(csv.reader(wrong))!r}")
with excel_export.open(encoding="utf-8-sig", newline="") as right:
    print(f"utf-8-sig: {next(csv.reader(right))!r}")
print(f"{'hi'.encode('utf-16')=}  (BOM + little-endian on this machine)")
print(f"{codecs.BOM_UTF16_LE=}, {codecs.BOM_UTF16_BE=}")

excel_export.read_bytes()[:6]=b'\xef\xbb\xbfsku'
utf-8:     ['\ufeffsku', 'quantity']
utf-8-sig: ['sku', 'quantity']
'hi'.encode('utf-16')=b'\xff\xfeh\x00i\x00'  (BOM + little-endian on this machine)
codecs.BOM_UTF16_LE=b'\xff\xfe', codecs.BOM_UTF16_BE=b'\xfe\xff'


**Mojibake** is text decoded with the wrong encoding: `₹` written as UTF-8 and read
as cp1252 becomes `â‚¹`. If the wrong decoding lost nothing, you can reverse it:
encode back with the wrong codec, decode with the right one. (The third-party
`ftfy` library automates this.) Prevention is better: declare the encoding wherever
bytes cross a boundary (files, HTTP `Content-Type`, databases).

In [381]:
original = "Café ₹240 – Diwali offer"  # every UTF-8 byte here is defined in cp1252
garbled = original.encode("utf-8").decode("cp1252")
print(f"{garbled=}")
repaired = garbled.encode("cp1252").decode("utf-8")
print(f"{repaired=}, {repaired == original=}")
double = garbled.encode("utf-8").decode("cp1252")
print(f"double mojibake: {double[:24]!r}...")

garbled='CafÃ© â‚¹240 â€“ Diwali offer'
repaired='Café ₹240 – Diwali offer', repaired == original=True
double mojibake: 'CafÃƒÂ© Ã¢â€šÂ¹240 Ã¢â‚¬'...


**Error handlers** decide what happens when bytes cannot be decoded (or characters
cannot be encoded). Pass `errors=` to `open()`, `bytes.decode` or `str.encode`.

| Handler | Decoding bad bytes | Encoding unencodable characters | Use when |
| --- | --- | --- | --- |
| `strict` (default) | `UnicodeDecodeError` | `UnicodeEncodeError` | data must be correct |
| `replace` | `�` (U+FFFD) | `?` | display, logs |
| `ignore` | dropped | dropped | rarely: silent data loss |
| `backslashreplace` | `\xff` | `\u20b9` | debugging, readable escapes |
| `surrogateescape` | lone surrogates that round-trip | restores the original bytes | file names, pass-through of unknown bytes |
| `xmlcharrefreplace` | (encode only) | `&#8377;` | HTML/XML output in a narrow encoding |

In [382]:
damaged = "₹240 café".encode()[:-1] + b"\xff more"
for handler in ("replace", "ignore", "backslashreplace", "surrogateescape"):
    print(f"decode {handler:<17} {damaged.decode('utf-8', errors=handler)!r}")
print(raised_message(lambda: damaged.decode("utf-8")))
escaped = damaged.decode("utf-8", errors="surrogateescape")
print(
    f"surrogateescape round-trips: {escaped.encode('utf-8', errors='surrogateescape') == damaged}"
)
for handler in ("replace", "backslashreplace", "xmlcharrefreplace"):
    print(f"encode {handler:<17} {'₹240'.encode('ascii', errors=handler)!r}")

decode replace           '₹240 caf�� more'
decode ignore            '₹240 caf more'
decode backslashreplace  '₹240 caf\\xc3\\xff more'
decode surrogateescape   '₹240 caf\udcc3\udcff more'
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc3 in position 10: invalid continuation byte
surrogateescape round-trips: True
encode replace           b'?240'
encode backslashreplace  b'\\u20b9240'
encode xmlcharrefreplace b'&#8377;240'


**Predict the output.**

```python
data = "naïve".encode()  # UTF-8 is the default
print(len("naïve"), len(data), data.decode("latin-1"))
```

<details><summary>Answer</summary>

`5 6 naÃ¯ve`. `str.encode()` uses UTF-8 unless told otherwise. `ï` is one code
point but two bytes in UTF-8 (`C3 AF`), so the bytes
are one longer than the string. latin-1 decodes every byte as one character, so it
never fails: it silently produces mojibake instead.

</details>

Run the next cell to check your prediction.

In [383]:
data = "naïve".encode()  # UTF-8 is the default
print(len("naïve"), len(data), data.decode("latin-1"))

5 6 naÃ¯ve


### 4.5.6 Serialization formats

| Format | Module | Human-editable | Types | Safe for untrusted input | Typical use |
| --- | --- | --- | --- | --- | --- |
| JSON | `json`, `orjson` | yes | dict, list, str, number, bool, null | yes | APIs, configs, logs |
| CSV | `csv` | yes (spreadsheets) | strings only | yes | tabular exchange |
| TOML | `tomllib` (read only) | yes | + dates, tables, arrays | yes | config (`pyproject.toml`) |
| YAML | PyYAML `safe_load` | yes | + anchors, many scalar types | **only** with `safe_load` | config (CI, Kubernetes) |
| XML | `xml.etree`, `defusedxml` | yes | text + attributes | **only** with `defusedxml` | legacy systems, feeds, office files |
| MessagePack | `msgpack` | no (binary) | like JSON + bytes | yes | compact JSON-like data |
| pickle | `pickle` | no | almost any Python object | **never** (4.5.7) | caches between your own processes |

**JSON** in brief (Unit 4.4.5 covered encoders, hooks and `Decimal`): `json.dump`
and `json.load` take file objects.

In [384]:
order = {
    "id": 7,
    "placed_at": FIXED_MOMENT.isoformat(),
    "lines": [{"sku": "TEA-500", "quantity": 2}],
    "paid": True,
}
order_path = io_demo / "order.json"
with order_path.open("w", encoding="utf-8") as json_file:
    json.dump(order, json_file, indent=2)
with order_path.open(encoding="utf-8") as json_file:
    print(f"{json.load(json_file) == order=}")
print(order_path.read_text(encoding="utf-8")[:60] + "...")

json.load(json_file) == order=True
{
  "id": 7,
  "placed_at": "2026-11-07T09:30:00+00:00",
  "...


**CSV** in brief (Unit 4.4.5): `newline=""`, `DictReader`/`DictWriter`, and every
value comes back as a string. Here a CSV is streamed into typed records:

In [385]:
@dataclasses.dataclass(frozen=True)
class StockRow:
    sku: str
    quantity: int
    unit_price: Decimal


stock_csv = io_demo / "stock.csv"
stock_csv.write_text(
    "sku,quantity,unit_price\nTEA-500,40,240.00\nMUG-02,6,199.00\n", encoding="utf-8"
)
with stock_csv.open(newline="", encoding="utf-8") as csv_file:
    rows = [
        StockRow(row["sku"], int(row["quantity"]), Decimal(row["unit_price"]))
        for row in csv.DictReader(csv_file)
    ]
print(rows)
print(f"stock value: {sum(row.quantity * row.unit_price for row in rows)}")

[StockRow(sku='TEA-500', quantity=40, unit_price=Decimal('240.00')), StockRow(sku='MUG-02', quantity=6, unit_price=Decimal('199.00'))]
stock value: 10794.00


**TOML** (`tomllib`, *New in 3.11*) is the format of `pyproject.toml`. The standard
library only **reads** it (open the file in binary mode); writing needs a
third-party package such as `tomli-w`. TOML has real types: integers, floats,
booleans, dates and datetimes, arrays and nested tables.

In [386]:
settings_toml = io_demo / "shop.toml"
settings_toml.write_text(
    textwrap.dedent("""
    [store]
    name = "Pune Central"
    opened = 2019-04-01
    gst_percent = 18

    [store.hours]
    weekday = ["09:00", "21:00"]

    [[warehouses]]
    code = "PNQ1"
    capacity = 5_000
"""),
    encoding="utf-8",
)
with settings_toml.open("rb") as toml_file:
    settings = tomllib.load(toml_file)
print(settings)
print(f"{type(settings['store']['opened']).__name__=}, {settings['warehouses'][0]['capacity']=}")
with (REPO_ROOT / "pyproject.toml").open("rb") as project_file:
    print(f"course project: {tomllib.load(project_file)['project']['name']!r}")
print(raised_message(lambda: tomllib.loads("name = 'unterminated")))

{'store': {'name': 'Pune Central', 'opened': datetime.date(2019, 4, 1), 'gst_percent': 18, 'hours': {'weekday': ['09:00', '21:00']}}, 'warehouses': [{'code': 'PNQ1', 'capacity': 5000}]}
type(settings['store']['opened']).__name__='date', settings['warehouses'][0]['capacity']=5000
course project: 'python-course'
TOMLDecodeError: Expected "'" (at end of document)


**YAML** (PyYAML) is popular for configuration (GitHub Actions, Docker Compose,
Kubernetes). Always use **`yaml.safe_load`**: the full loader can build arbitrary
Python objects from tags such as `!!python/object/apply`, which turns a config file
into code execution. `safe_load` refuses those tags. YAML's implicit typing has
surprises: in YAML 1.1 (PyYAML) `no` and `off` are booleans and `0755` is an octal
number.

In [387]:
if importlib.util.find_spec("yaml") is None:
    print("SKIP: PyYAML is not installed (uv add pyyaml)")
else:
    import yaml

    compose_text = textwrap.dedent("""
        service: shop-api
        replicas: 3
        debug: off
        country: NO
        file_mode: 0755
        ports: ["8000:8000"]
    """)
    print(yaml.safe_load(compose_text))
    print(yaml.safe_dump({"service": "shop-api", "replicas": 3, "country": "NO"}, sort_keys=False))
    tagged = "!!python/object/apply:builtins.print ['this would run on load']"
    print(raised_message(lambda: yaml.safe_load(tagged)).splitlines()[0])

{'service': 'shop-api', 'replicas': 3, 'debug': False, 'country': False, 'file_mode': 493, 'ports': ['8000:8000']}
service: shop-api
replicas: 3
country: 'NO'

ConstructorError: could not determine a constructor for the tag 'tag:yaml.org,2002:python/object/apply:builtins.print'


`debug: off` became `False` and `country: NO` (Norway) became `False` too: quote
values that must stay strings. The tagged document was refused by `safe_load`.

**XML.** The standard library parses XML with `xml.etree.ElementTree`, but it is
vulnerable to malicious documents (entity-expansion "billion laughs" attacks that
exhaust memory, external entities in some parsers). **`defusedxml`** wraps the same
API and refuses those constructs; use it for any XML you did not write yourself.
`defusedxml` is not an Appendix A package; here it happens to be installed as a
dependency of `nbconvert`, so the cell checks first.

In [388]:
import xml.etree.ElementTree as ElementTree

feed = """<orders currency="INR">
  <order id="7"><customer>Asha</customer><total>679.00</total></order>
  <order id="8"><customer>Ravi</customer><total>240.00</total></order>
</orders>"""
root = ElementTree.fromstring(feed)
print(f"{root.tag=}, {root.attrib=}")
for order_element in root.iter("order"):
    print(
        f"  order {order_element.get('id')}: {order_element.findtext('customer')} "
        f"{Decimal(order_element.findtext('total'))}"
    )
print(f"{[element.text for element in root.findall('./order/customer')]=}")

entity_bomb = (
    '<!DOCTYPE d [<!ENTITY a "aaaaaaaaaa"><!ENTITY b "&a;&a;&a;&a;&a;&a;&a;&a;&a;&a;">]><d>&b;</d>'
)
if importlib.util.find_spec("defusedxml") is None:
    print("SKIP: defusedxml is not installed (not an Appendix A package)")
else:
    import defusedxml.ElementTree as SafeElementTree

    print(f"defusedxml, normal feed: {SafeElementTree.fromstring(feed).tag!r}")
    print(
        "defusedxml, entities:    "
        f"{raised_message(lambda: SafeElementTree.fromstring(entity_bomb))}"
    )

root.tag='orders', root.attrib={'currency': 'INR'}
  order 7: Asha 679.00
  order 8: Ravi 240.00
[element.text for element in root.findall('./order/customer')]=['Asha', 'Ravi']
defusedxml, normal feed: 'orders'
defusedxml, entities:    EntitiesForbidden: EntitiesForbidden(name='a', system_id=None, public_id=None)


**MessagePack and orjson.** MessagePack is "binary JSON": the same data model plus
raw bytes, smaller and faster to parse. `orjson` (an Appendix A package, Unit 4.6.5)
is a fast JSON library written in Rust that returns `bytes` and natively handles
`datetime`, `UUID`, dataclasses and NumPy arrays. `msgpack` is not in Appendix A;
it is installed here as a dependency of another tool, so the cell checks first.

In [389]:
sample = {"id": 7, "sku": "TEA-500", "quantity": 2, "tags": ["gift", "express"], "paid": True}
as_json = json.dumps(sample, separators=(",", ":")).encode()
print(f"json:    {len(as_json)} bytes")
if importlib.util.find_spec("msgpack") is None:
    print("SKIP: msgpack is not installed (not an Appendix A package)")
else:
    import msgpack

    packed = msgpack.packb(sample)
    print(
        f"msgpack: {len(packed)} bytes, {packed[:12].hex(' ')}..., "
        f"round trip {msgpack.unpackb(packed) == sample}"
    )
    print(f"msgpack keeps bytes: {msgpack.unpackb(msgpack.packb({'png': png_bytes[:4]}))}")
if importlib.util.find_spec("orjson") is None:
    print("SKIP: orjson is not installed (uv add orjson)")
else:
    import orjson

    print(f"orjson:  {orjson.dumps(sample)[:40]!r}... ({type(orjson.dumps(sample)).__name__})")
    print(f"orjson datetime + UUID: {orjson.dumps({'at': FIXED_MOMENT, 'id': uuid.UUID(int=7)})!r}")

json:    75 bytes
msgpack: 52 bytes, 85 a2 69 64 07 a3 73 6b 75 a7 54 45..., round trip True
msgpack keeps bytes: {'png': b'\x89PNG'}
orjson:  b'{"id":7,"sku":"TEA-500","quantity":2,"ta'... (bytes)
orjson datetime + UUID: b'{"at":"2026-11-07T09:30:00+00:00","id":"00000000-0000-0000-0000-000000000007"}'


### 4.5.7 `pickle`

`pickle` serializes almost any Python object graph (instances of your classes,
nested containers, shared references, cycles) to bytes and back. It is
Python-specific, and the bytes are **instructions** for rebuilding objects.

**Protocols** are format versions: 0 is ASCII, 2 added efficient classes, 4 (3.4)
large objects, 5 (3.8) out-of-band buffers. `pickle.DEFAULT_PROTOCOL` is 5 since
Python 3.14 (4 before). Higher protocols are smaller and faster; older Pythons
cannot read newer protocols.

In [390]:
@dataclasses.dataclass
class Basket:
    customer: str
    items: dict[str, int]
    created: datetime


basket = Basket("Asha", {"TEA-500": 2, "MUG-02": 1}, FIXED_MOMENT)
shared = [basket, basket]  # the same object twice
for protocol in (0, 2, pickle.DEFAULT_PROTOCOL):
    print(f"protocol {protocol}: {len(pickle.dumps(shared, protocol=protocol))} bytes")
restored = pickle.loads(pickle.dumps(shared))
print(f"{restored[0] == basket=}, {restored[0] is restored[1]=}  (shared reference kept)")
print(f"{pickle.DEFAULT_PROTOCOL=}, {pickle.HIGHEST_PROTOCOL=}")
print(f"{pickle.dumps(42, protocol=0)=}")

protocol 0: 376 bytes
protocol 2: 270 bytes
protocol 5: 197 bytes
restored[0] == basket=True, restored[0] is restored[1]=True  (shared reference kept)
pickle.DEFAULT_PROTOCOL=5, pickle.HIGHEST_PROTOCOL=5
pickle.dumps(42, protocol=0)=b'I42\n.'


**Customizing: `__getstate__` / `__setstate__`.** By default pickle saves the
instance `__dict__`. Override `__getstate__` to leave out what cannot or should not
be saved (open connections, locks, caches, secrets) and `__setstate__` to rebuild
it on load.

In [391]:
class PriceCache:
    """Holds an open SQLite connection (unpicklable) plus cached prices."""

    def __init__(self, database: str) -> None:
        self.database = database
        self.prices = {"TEA-500": 240}
        self.connection = sqlite3.connect(database)

    def __getstate__(self) -> dict[str, object]:
        state = self.__dict__.copy()
        del state["connection"]  # not picklable; reopened on load
        return state

    def __setstate__(self, state: dict[str, object]) -> None:
        self.__dict__.update(state)
        self.connection = sqlite3.connect(self.database)


print(raised_message(lambda: pickle.dumps(sqlite3.connect(":memory:"))))
cache_copy = pickle.loads(pickle.dumps(PriceCache(":memory:")))
print(f"{cache_copy.prices=}, {type(cache_copy.connection).__name__=}")

TypeError: cannot pickle 'sqlite3.Connection' object
cache_copy.prices={'TEA-500': 240}, type(cache_copy.connection).__name__='Connection'


**`__reduce__`** is the lower-level hook: it returns `(callable, arguments)`, and
unpickling **calls** `callable(*arguments)`. Extension types and classes whose
constructor must run use it.

In [392]:
import pickletools


class Sku:
    """Validated in __init__, so unpickling must go through __init__ too."""

    def __init__(self, code: str) -> None:
        if not code.isupper():
            raise ValueError(f"invalid SKU {code!r}")
        self.code = code

    def __reduce__(self) -> tuple[type, tuple[str]]:
        return (Sku, (self.code,))

    def __repr__(self) -> str:
        return f"Sku({self.code!r})"


print(f"{pickle.loads(pickle.dumps(Sku('TEA-500')))=}")
opcode_listing = io.StringIO()  # a StringIO target also keeps the listing free of colour codes
pickletools.dis(pickle.dumps(Sku("TEA-500"), protocol=2), out=opcode_listing)
print(opcode_listing.getvalue())  # GLOBAL names the callable, REDUCE calls it

pickle.loads(pickle.dumps(Sku('TEA-500')))=Sku('TEA-500')
    0: \x80 PROTO      2
    2: c    GLOBAL     '__main__ Sku'
   16: q    BINPUT     0
   18: X    BINUNICODE 'TEA-500'
   30: q    BINPUT     1
   32: \x85 TUPLE1
   33: q    BINPUT     2
   35: R    REDUCE
   36: q    BINPUT     3
   38: .    STOP
highest protocol among opcodes = 2



**The remote-code-execution risk.** Because unpickling calls whatever callable the
data names, **loading a pickle runs code chosen by whoever wrote the bytes**. The
harmless demonstration below names a function that only appends a message to a
list; a real attacker would name something dangerous instead, and `loads` would
call it before your code ever sees the "object". There is no safe way to unpickle
untrusted data: signatures (`hmac`) can prove the bytes are yours, but the rule is
simply to never accept pickles from users, the network or shared storage you do
not control.

In [393]:
side_effects: list[str] = []


def record_side_effect(message: str) -> str:
    """Stands in for any callable an attacker could name."""
    side_effects.append(message)
    return "not the object you expected"


class Innocent:
    def __reduce__(self) -> tuple[Callable[[str], str], tuple[str]]:
        return (record_side_effect, ("ran during pickle.loads",))


untrusted_bytes = pickle.dumps(Innocent())  # pretend this arrived over the network
result = pickle.loads(untrusted_bytes)  # the callable runs here
print(f"{result=}")
print(f"{side_effects=}")

result='not the object you expected'
side_effects=['ran during pickle.loads']


**Legitimate uses** are the ones where both ends are your own trusted code:
`multiprocessing` and `concurrent.futures` send arguments to worker processes with
pickle (Phase 5), caches on local disk (`shelve`, `joblib`), and saving
intermediate results of a long computation. For anything that crosses a trust
boundary or must outlive a Python version, use JSON, MessagePack or a database.
`shelve` is a persistent dict built on pickle:

In [394]:
import shelve

with shelve.open(str(io_demo / "report_cache")) as cache:
    cache["2026-11"] = {"orders": 412, "revenue": Decimal("187340.50")}
with shelve.open(str(io_demo / "report_cache")) as cache:
    print(f"{dict(cache)=}")

dict(cache)={'2026-11': {'orders': 412, 'revenue': Decimal('187340.50')}}


### 4.5.8 Compression and archives

Four compressors in the standard library share a file-like API (`open`,
`compress`, `decompress`):

| Module | Algorithm | Speed / ratio | Extension |
| --- | --- | --- | --- |
| `gzip` (`zlib`) | DEFLATE | fast, moderate ratio; universal | `.gz` |
| `bz2` | Burrows–Wheeler | slower, better ratio | `.bz2` |
| `lzma` | LZMA2 (xz) | slowest to compress, best ratio | `.xz` |
| `compression.zstd` | Zstandard (*New in 3.14*) | very fast at good ratios; tunable levels | `.zst` |

*New in 3.14:* the `compression` package also re-exports the others
(`compression.gzip`, …). Sizes below are deterministic for this input; times vary.

In [395]:
log_bytes = access_log.read_bytes()
compressors = {"gzip": gzip, "bz2": bz2, "lzma": lzma}
if sys.version_info >= (3, 14):
    from compression import zstd

    compressors["zstd"] = zstd
print(f"original: {len(log_bytes):>6} bytes")
for name, module in compressors.items():
    packed = module.compress(log_bytes)
    assert module.decompress(packed) == log_bytes
    print(f"{name:<8}: {len(packed):>6} bytes ({len(packed) / len(log_bytes):.1%})")

original:  32893 bytes
gzip    :   2549 bytes (7.7%)
bz2     :   1548 bytes (4.7%)


lzma    :   1152 bytes (3.5%)
zstd    :   1267 bytes (3.9%)


The `open()` functions return file objects, so compressed files are read and
written like plain ones, line by line, in text mode with an encoding:

In [396]:
gzip_path = io_demo / "access.log.gz"
with gzip.open(gzip_path, "wt", encoding="utf-8") as compressed:
    compressed.write(access_log.read_text(encoding="utf-8"))
with gzip.open(gzip_path, "rt", encoding="utf-8") as compressed:
    server_errors = sum(1 for line in compressed if line.rstrip().endswith("500"))
print(f"{gzip_path.stat().st_size=}, {server_errors=}")
print(f"{gzip_path.read_bytes()[:2].hex(' ')=}  (gzip magic number 1f 8b)")
if sys.version_info >= (3, 14):
    with zstd.open(io_demo / "access.log.zst", "wt", encoding="utf-8") as compressed:
        compressed.write("TEA-500,2\n")
    print(
        "zstd text round trip: "
        f"{zstd.open(io_demo / 'access.log.zst', 'rt', encoding='utf-8').read()!r}"
    )

gzip_path.stat().st_size=2566, server_errors=142
gzip_path.read_bytes()[:2].hex(' ')='1f 8b'  (gzip magic number 1f 8b)
zstd text round trip: 'TEA-500,2\n'


**`zipfile`** reads and writes ZIP archives (also `.docx`, `.xlsx`, `.whl`, `.jar`).
Choose the compression per archive (`ZIP_DEFLATED`, `ZIP_BZIP2`, `ZIP_LZMA`,
*New in 3.14* `ZIP_ZSTANDARD`); `infolist()` shows each member's sizes; `writestr`
adds data without a file on disk.

In [397]:
bundle_path = io_demo / "monthly_report.zip"
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(access_log, arcname="logs/access.log")
    bundle.writestr("summary.json", json.dumps({"orders": 1000, "errors": len(errors)}))
with zipfile.ZipFile(bundle_path) as bundle:
    for member in bundle.infolist():
        print(f"  {member.filename:<16} {member.file_size:>6} -> {member.compress_size:>5} bytes")
    print(f"{json.loads(bundle.read('summary.json'))=}")
    print(f"{bundle.testzip() is None=}  (all CRCs good)")

  logs/access.log   32893 ->  2531 bytes
  summary.json         31 ->    28 bytes
json.loads(bundle.read('summary.json'))={'orders': 1000, 'errors': 142}
bundle.testzip() is None=True  (all CRCs good)


**`tarfile`** handles `.tar`, `.tar.gz`, `.tar.xz` (and `.tar.zst` in 3.14). Tar
members can be absolute paths, `..` paths, symlinks or device files, so extracting
an untrusted archive could overwrite files anywhere. Since Python 3.12,
`extractall(..., filter="data")` refuses all of that, and *Changed in 3.14:*
`"data"` is the **default** filter. Always pass it explicitly so the intent is clear
and older versions behave the same.

In [398]:
tar_path = io_demo / "backup.tar.gz"
with tarfile.open(tar_path, "w:gz") as archive:
    archive.add(access_log, arcname="logs/access.log")
    archive.add(stock_csv, arcname="data/stock.csv")
with tarfile.open(tar_path) as archive:
    print(f"{archive.getnames()=}")
    archive.extractall(fresh_dir("restored"), filter="data")
restored = UNIT45_DIR / "restored"
restored_files = sorted(
    p.relative_to(restored).as_posix() for p in restored.rglob("*") if p.is_file()
)
print(f"{restored_files=}")

hostile_path = io_demo / "hostile.tar"
with tarfile.open(hostile_path, "w") as archive:
    member = tarfile.TarInfo("../../outside.txt")
    member.size = 4
    archive.addfile(member, io.BytesIO(b"oops"))
with tarfile.open(hostile_path) as archive:
    print(raised_message(lambda: archive.extractall(fresh_dir("hostile_out"), filter="data")))
print(f"{(UNIT45_DIR / 'outside.txt').exists()=}")

archive.getnames()=['logs/access.log', 'data/stock.csv']
restored_files=['data/stock.csv', 'logs/access.log']
OutsideDestinationError: '../../outside.txt' would be extracted to '<repo>/notebooks/_scratch/outside.txt', which is outside the destination
(UNIT45_DIR / 'outside.txt').exists()=False


### 4.5.9 `sqlite3`

SQLite is a complete SQL database in one file (or in memory), built into Python.
It is the right choice for local apps, caches, tests and prototypes, and the
habits carry over to PostgreSQL in Phase 6. **Connections**: `sqlite3.connect(path)`
(`":memory:"` for a throwaway database); `connection.execute(sql, parameters)`
returns a cursor; `fetchone`/`fetchall` or iteration read rows.

In [399]:
database_path = io_demo / "shop.db"
database_path.unlink(missing_ok=True)
connection = sqlite3.connect(database_path)
connection.execute(
    "CREATE TABLE products (sku TEXT PRIMARY KEY, name TEXT NOT NULL, price_paise INTEGER NOT NULL)"
)
connection.execute("INSERT INTO products VALUES (?, ?, ?)", ("TEA-500", "Masala tea 500 g", 24000))
connection.commit()
print(f"{connection.execute('SELECT * FROM products').fetchall()=}")
print(
    f"{sqlite3.sqlite_version=}, {connection.execute('SELECT COUNT(*) FROM products').fetchone()=}"
)

connection.execute('SELECT * FROM products').fetchall()=[('TEA-500', 'Masala tea 500 g', 24000)]
sqlite3.sqlite_version='3.53.1', connection.execute('SELECT COUNT(*) FROM products').fetchone()=(1,)


**Parameterized queries.** Values go in as **parameters**: `?` placeholders with a
tuple, or `:name` placeholders with a dict. The driver sends them separately from
the SQL text, so a value can never change the query's structure. **`executemany`**
runs one statement for every item of an iterable, far faster than a Python loop of
`execute` calls.

In [400]:
catalogue = [
    ("MUG-02", "Steel mug", 19900),
    ("JAR-750", "Pickle jar 750 ml", 35000),
    ("CHAI-1", "O'Brien's chai", 15000),
]
connection.executemany("INSERT INTO products VALUES (?, ?, ?)", catalogue)
connection.commit()
cheap = connection.execute(
    "SELECT sku, price_paise FROM products WHERE price_paise < :limit ORDER BY price_paise",
    {"limit": 20000},
)
print(f"{cheap.fetchall()=}")
print(
    f"{connection.execute('SELECT name FROM products WHERE sku = ?', ('CHAI-1',)).fetchone()=}"
    "  (the quote is just data)"
)

cheap.fetchall()=[('CHAI-1', 15000), ('MUG-02', 19900)]
connection.execute('SELECT name FROM products WHERE sku = ?', ('CHAI-1',)).fetchone()=("O'Brien's chai",)  (the quote is just data)


**SQL injection** happens when user input is pasted into the SQL **text**. The
demonstration runs against this throwaway database only: a crafted "SKU" turns the
`WHERE` clause into something that is always true and returns every row. The
parameterized version treats the same input as an ordinary (non-matching) value.
Never build SQL with f-strings, `%` or `+` from input; for dynamic identifiers
(column names) use an allow-list.

In [401]:
user_input = "nothing' OR '1'='1"
unsafe_sql = f"SELECT sku FROM products WHERE sku = '{user_input}'"  # never do this
print(f"{unsafe_sql=}")
print(f"string-built query returns: {connection.execute(unsafe_sql).fetchall()}")
print(
    "parameterized query returns: "
    f"{connection.execute('SELECT sku FROM products WHERE sku = ?', (user_input,)).fetchall()}"
)

SORTABLE_COLUMNS = {"sku", "name", "price_paise"}


def products_sorted_by(column: str) -> list[tuple[str, ...]]:
    if column not in SORTABLE_COLUMNS:  # identifiers cannot be parameters
        raise ValueError(f"cannot sort by {column!r}")
    return connection.execute(f"SELECT sku FROM products ORDER BY {column}").fetchall()


print(f"{products_sorted_by('price_paise')=}")
print(raised_message(lambda: products_sorted_by("price_paise; DROP TABLE products")))

unsafe_sql="SELECT sku FROM products WHERE sku = 'nothing' OR '1'='1'"
string-built query returns: [('CHAI-1',), ('JAR-750',), ('MUG-02',), ('TEA-500',)]
parameterized query returns: []
products_sorted_by('price_paise')=[('CHAI-1',), ('MUG-02',), ('TEA-500',), ('JAR-750',)]
ValueError: cannot sort by 'price_paise; DROP TABLE products'


**Transactions.** Changes are invisible to other connections and not durable until
`commit()`; `rollback()` discards them. Using the connection as a **context
manager** commits if the block succeeds and rolls back if it raises (it does *not*
close the connection). Multi-step changes, such as moving stock between
warehouses, must happen in one transaction so they cannot half-happen. *New in
3.12:* `autocommit=` gives explicit control of the transaction mode.

In [402]:
connection.execute(
    "CREATE TABLE stock (warehouse TEXT, sku TEXT, quantity INTEGER CHECK (quantity >= 0),"
    " PRIMARY KEY (warehouse, sku))"
)
connection.executemany(
    "INSERT INTO stock VALUES (?, ?, ?)", [("PNQ1", "TEA-500", 10), ("BLR1", "TEA-500", 0)]
)
connection.commit()


def transfer(quantity: int) -> None:
    with connection:  # commit on success, rollback on error
        connection.execute(
            "UPDATE stock SET quantity = quantity + ? WHERE warehouse = 'BLR1' AND sku = 'TEA-500'",
            (quantity,),
        )
        connection.execute(
            "UPDATE stock SET quantity = quantity - ? WHERE warehouse = 'PNQ1' AND sku = 'TEA-500'",
            (quantity,),
        )


transfer(4)
print(
    "after transferring 4:  "
    f"{connection.execute('SELECT warehouse, quantity FROM stock ORDER BY warehouse').fetchall()}"
)
print(raised_message(lambda: transfer(50)))
print(
    "after failed transfer: "
    f"{connection.execute('SELECT warehouse, quantity FROM stock ORDER BY warehouse').fetchall()}"
    "  (rolled back)"
)

after transferring 4:  [('BLR1', 4), ('PNQ1', 6)]
IntegrityError: CHECK constraint failed: quantity >= 0
after failed transfer: [('BLR1', 4), ('PNQ1', 6)]  (rolled back)


**`sqlite3.Row`.** Set `connection.row_factory = sqlite3.Row` and rows support
access by column name and `keys()` as well as by index, so code no longer depends
on column order. (Any callable works as a row factory, for example one that builds
a dataclass.)

In [403]:
connection.row_factory = sqlite3.Row
product = connection.execute("SELECT * FROM products WHERE sku = ?", ("TEA-500",)).fetchone()
print(f"{product['name']=}, {product[2]=}, {product.keys()=}, {dict(product)=}")
connection.row_factory = lambda cursor, row: StockRow(row[0], row[1], Decimal(row[2]) / 100)
first_two = connection.execute(
    "SELECT sku, 1, price_paise FROM products ORDER BY sku LIMIT 2"
).fetchall()
print(f"{first_two=}")
connection.row_factory = None
connection.close()

product['name']='Masala tea 500 g', product[2]=24000, product.keys()=['sku', 'name', 'price_paise'], dict(product)={'sku': 'TEA-500', 'name': 'Masala tea 500 g', 'price_paise': 24000}
first_two=[StockRow(sku='CHAI-1', quantity=1, unit_price=Decimal('150')), StockRow(sku='JAR-750', quantity=1, unit_price=Decimal('350'))]


**Under the hood: the query plan.** `EXPLAIN QUERY PLAN` shows how SQLite will run
a query: a primary-key lookup uses the index (`SEARCH ... USING INDEX`), a filter on
an unindexed column reads every row (`SCAN`). Adding an index changes the plan.

In [404]:
with sqlite3.connect(database_path) as inspector:
    for sql in (
        "SELECT * FROM products WHERE sku = 'MUG-02'",
        "SELECT * FROM products WHERE price_paise > 20000",
    ):
        print(f"{sql}\n  -> {[row[3] for row in inspector.execute('EXPLAIN QUERY PLAN ' + sql)]}")
    inspector.execute("CREATE INDEX IF NOT EXISTS products_by_price ON products (price_paise)")
    plan_rows = inspector.execute("EXPLAIN QUERY PLAN " + sql).fetchall()  # sql: the price query
    print(f"with an index -> {[row[3] for row in plan_rows]}")
inspector.close()

SELECT * FROM products WHERE sku = 'MUG-02'
  -> ['SEARCH products USING INDEX sqlite_autoindex_products_1 (sku=?)']
SELECT * FROM products WHERE price_paise > 20000
  -> ['SCAN products']


with an index -> ['SEARCH products USING INDEX products_by_price (price_paise>?)']


### 4.5.10 File-system safety

Whenever a path comes from outside (an upload's file name, a URL segment, a zip
member, a CLI argument), check that it stays inside the folder you meant before
touching the disk. The robust check: **resolve** the joined path (which removes
`..` and follows symlinks) and test `Path.is_relative_to(base)` (*New in 3.9*).
String checks such as `".." in name` or `startswith(base)` miss absolute paths,
symlinks and prefixes like `/srv/uploads-old`.

In [405]:
uploads = fresh_dir("uploads")
(uploads / "invoices").mkdir()
(uploads / "shortcut").symlink_to(UNIT45_DIR)  # a symlink pointing outside


def safe_path(base: Path, requested: str) -> Path:
    """The resolved path of requested inside base; ValueError if it would escape."""
    candidate = (base / requested).resolve()
    if not candidate.is_relative_to(base.resolve()):
        raise ValueError(f"{requested!r} escapes {base.name}/")
    return candidate


for requested in (
    "invoices/7.pdf",
    "../shop.db",
    "/etc/passwd",
    "invoices/../../shop.db",
    "shortcut/shop.db",
):
    print(
        f"  {requested!r:<26} "
        f"{raised_message(lambda requested=requested: safe_path(uploads, requested))}"
    )
print(
    "naive startswith check is fooled: "
    f"{str(UNIT45_DIR / 'uploads-old' / 'x').startswith(str(uploads))}"
)

  'invoices/7.pdf'           no error
  '../shop.db'               ValueError: '../shop.db' escapes uploads/
  '/etc/passwd'              ValueError: '/etc/passwd' escapes uploads/
  'invoices/../../shop.db'   ValueError: 'invoices/../../shop.db' escapes uploads/
  'shortcut/shop.db'         ValueError: 'shortcut/shop.db' escapes uploads/
naive startswith check is fooled: True


`raised_message` prints `no error` for the allowed path; every escape attempt is
refused, including the symlink, which only `resolve()` exposes. The last line shows
why prefix checks on strings are wrong: `uploads-old` "starts with" `uploads`.
The same check protects zip extraction (Lab 4.5, exercise 6); `tarfile` has it built
in through `filter="data"` (4.5.8).

### Lab 4.5

Six exercises; replace each `raise NotImplementedError`, then run the check cell.
The checks build their own files under `notebooks/_scratch/unit45/lab/`. Until you
finish, they print FAIL; that is expected.

In [406]:
LAB45_DIR = find_repo_root() / "notebooks" / "_scratch" / "unit45" / "lab"
LAB45_DIR.mkdir(parents=True, exist_ok=True)
print(f"lab files go to {LAB45_DIR.relative_to(find_repo_root()).as_posix()}/")

lab files go to notebooks/_scratch/unit45/lab/


#### Exercise 1 — `tail(path, n)` reading backwards with `seek`

Return the last `n` lines of a UTF-8 text file as a list of strings without their
line endings, **without reading the whole file**: open it in binary mode, `seek`
to the end and read blocks of `chunk_size` bytes backwards until you have more than
`n` newlines (or reach the start), then decode. A final line without a trailing
newline counts as a line. `n <= 0` returns `[]`. Decode only after joining the
blocks: a multi-byte character can straddle two blocks.

In [407]:
def tail(path: Path, n: int, chunk_size: int = 4096) -> list[str]:
    """The last n lines of a UTF-8 file, read backwards in binary chunks."""
    raise NotImplementedError

Check exercise 1:

In [408]:
big_log = LAB45_DIR / "big.log"
big_log.write_text(
    "".join(f"line {number} ₹{number * 10}\n" for number in range(1, 20_001)), encoding="utf-8"
)
no_newline = LAB45_DIR / "no_newline.txt"
no_newline.write_text("first\nsecond\nthird", encoding="utf-8")
(LAB45_DIR / "empty.txt").write_text("", encoding="utf-8")
expected_tail = big_log.read_text(encoding="utf-8").splitlines()[-5:]
run_checks(
    [
        ("last 5 lines", lambda: tail(big_log, 5), expected_tail),
        ("tiny chunks (multi-byte split)", lambda: tail(big_log, 5, chunk_size=7), expected_tail),
        ("no trailing newline", lambda: tail(no_newline, 2), ["second", "third"]),
        ("n larger than the file", lambda: tail(no_newline, 10), ["first", "second", "third"]),
        ("empty file", lambda: tail(LAB45_DIR / "empty.txt", 3), []),
        ("n = 0", lambda: tail(big_log, 0), []),
        (
            "uses seek, not read_text",
            lambda: (
                "seek" in inspect.getsource(tail) and "read_text" not in inspect.getsource(tail)
            ),
            True,
        ),
    ]
)

FAIL  last 5 lines: raised NotImplementedError: 
FAIL  tiny chunks (multi-byte split): raised NotImplementedError: 
FAIL  no trailing newline: raised NotImplementedError: 
FAIL  n larger than the file: raised NotImplementedError: 
FAIL  empty file: raised NotImplementedError: 
FAIL  n = 0: raised NotImplementedError: 
FAIL  uses seek, not read_text: expected True, got False
0/7 checks passed


#### Exercise 2 — atomic `write_json(path, data)`

Write `json.dumps(data, indent=2, ensure_ascii=False) + "\n"` to `path`
atomically: serialize first, write to a temporary file in the same folder, flush
and `fsync` it, then `os.replace` it over `path`. If anything fails (for example
`data` is not JSON-serializable), re-raise the error, leave any existing file
unchanged and leave no temporary file behind.

In [409]:
def write_json(path: Path, data: object) -> None:
    """Atomically replace path with data as pretty-printed JSON."""
    raise NotImplementedError

Check exercise 2:

In [410]:
json_dir = LAB45_DIR / "json_out"
shutil.rmtree(json_dir, ignore_errors=True)
json_dir.mkdir()
settings_target = json_dir / "settings.json"


def write_then_read(data: object) -> str:
    write_json(settings_target, data)
    return settings_target.read_text(encoding="utf-8")


def failed_write_keeps_old() -> tuple[str, str, list[str]]:
    write_json(settings_target, {"currency": "₹"})
    error = raised_message(lambda: write_json(settings_target, {"bad": {1, 2}}))
    leftovers = sorted(path.name for path in json_dir.iterdir() if path.name != "settings.json")
    return error.split(":")[0], settings_target.read_text(encoding="utf-8"), leftovers


run_checks(
    [
        (
            "writes pretty JSON",
            lambda: write_then_read({"gst": 18, "city": "Pune"}),
            '{\n  "gst": 18,\n  "city": "Pune"\n}\n',
        ),
        ("keeps non-ASCII", lambda: write_then_read(["₹"]), '[\n  "₹"\n]\n'),
        (
            "overwrites",
            lambda: (write_then_read({"v": 1}), write_then_read({"v": 2}))[1],
            '{\n  "v": 2\n}\n',
        ),
        (
            "failure leaves old file, no temp files",
            failed_write_keeps_old,
            ("TypeError", '{\n  "currency": "₹"\n}\n', []),
        ),
        (
            "uses os.replace and fsync",
            lambda: all(word in inspect.getsource(write_json) for word in ("os.replace", "fsync")),
            True,
        ),
    ]
)

FAIL  writes pretty JSON: raised NotImplementedError: 
FAIL  keeps non-ASCII: raised NotImplementedError: 
FAIL  overwrites: raised NotImplementedError: 
FAIL  failure leaves old file, no temp files: raised NotImplementedError: 
FAIL  uses os.replace and fsync: expected True, got False
0/5 checks passed


#### Exercise 3 — a CSV → SQLite importer

`import_orders(csv_path, connection)` reads a CSV with the header
`order_id,customer,sku,quantity,unit_price` and inserts every row into a table
`orders(order_id INTEGER PRIMARY KEY, customer TEXT, sku TEXT, quantity INTEGER,
unit_price TEXT)` (create it if it does not exist; keep `unit_price` as text so
`Decimal` values survive). Use **parameterized** `executemany` inside **one
transaction**: if any row is invalid (quantity not a positive integer, or a
duplicate `order_id`), raise `ValueError` and insert **nothing** from that file.
Return the number of rows inserted.

In [411]:
def import_orders(csv_path: Path, connection: sqlite3.Connection) -> int:
    """Import a CSV of orders in one transaction; return the number of rows."""
    raise NotImplementedError

Check exercise 3:

In [412]:
HEADER = "order_id,customer,sku,quantity,unit_price\n"
good_csv, bad_csv, duplicate_csv = (
    LAB45_DIR / name for name in ("orders_good.csv", "orders_bad.csv", "orders_dup.csv")
)
good_csv.write_text(
    HEADER
    + '1,Asha,TEA-500,2,240.00\n2,O\'Brien,MUG-02,1,199.00\n3,"Rao, Ravi",JAR-750,3,350.50\n',
    encoding="utf-8",
)
bad_csv.write_text(
    HEADER + "4,Meera,TEA-500,1,240.00\n5,John,MUG-02,two,199.00\n", encoding="utf-8"
)
duplicate_csv.write_text(
    HEADER + "6,Fatima,TEA-500,1,240.00\n1,Asha,TEA-500,9,240.00\n", encoding="utf-8"
)


def import_story() -> dict[str, object]:
    database = sqlite3.connect(":memory:")
    story: dict[str, object] = {"good": import_orders(good_csv, database)}
    story["bad"] = raised_message(lambda: import_orders(bad_csv, database)).split(":")[0]
    story["duplicate"] = raised_message(lambda: import_orders(duplicate_csv, database)).split(":")[
        0
    ]
    story["rows"] = database.execute(
        "SELECT order_id, customer, quantity, unit_price FROM orders ORDER BY order_id"
    ).fetchall()
    database.close()
    return story


run_checks(
    [
        ("rows inserted", lambda: import_story()["good"], 3),
        (
            "quotes and commas stored as data",
            lambda: [row[1] for row in import_story()["rows"]],
            ["Asha", "O'Brien", "Rao, Ravi"],
        ),
        ("types", lambda: import_story()["rows"][2], (3, "Rao, Ravi", 3, "350.50")),
        ("invalid quantity -> ValueError", lambda: import_story()["bad"], "ValueError"),
        ("duplicate id -> ValueError", lambda: import_story()["duplicate"], "ValueError"),
        ("failed files insert nothing", lambda: len(import_story()["rows"]), 3),
        (
            "uses executemany with placeholders",
            lambda: (
                "executemany" in inspect.getsource(import_orders)
                and "?" in inspect.getsource(import_orders)
            ),
            True,
        ),
    ]
)

FAIL  rows inserted: raised NotImplementedError: 
FAIL  quotes and commas stored as data: raised NotImplementedError: 
FAIL  types: raised NotImplementedError: 
FAIL  invalid quantity -> ValueError: raised NotImplementedError: 
FAIL  duplicate id -> ValueError: raised NotImplementedError: 
FAIL  failed files insert nothing: raised NotImplementedError: 
FAIL  uses executemany with placeholders: expected True, got False
0/7 checks passed


#### Exercise 4 — a binary header parser with `struct`

A WAV file starts with a little-endian RIFF header: `b"RIFF"`, a 4-byte file size
minus 8, `b"WAVE"`, then a `b"fmt "` chunk (4-byte size 16, 2-byte audio format
(1 = PCM), 2-byte channels, 4-byte sample rate, 4-byte byte rate, 2-byte block
align, 2-byte bits per sample), then a `b"data"` chunk header (4-byte size).
`parse_wav_header(data)` returns a dict with `channels`, `sample_rate`,
`bits_per_sample`, `data_size` and `duration_seconds` (`data_size / byte_rate`,
rounded to 3 places). Raise `ValueError` if the magic values are wrong or `data`
is shorter than 44 bytes.

In [413]:
def parse_wav_header(data: bytes) -> dict[str, int | float]:
    """Fields of a canonical 44-byte PCM WAV header."""
    raise NotImplementedError

Check exercise 4 (the WAV files are written with the standard `wave` module):

In [414]:
def make_wav(channels: int, sample_rate: int, sample_width: int, frames: int) -> bytes:
    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as wav_file:
        wav_file.setnchannels(channels)
        wav_file.setsampwidth(sample_width)
        wav_file.setframerate(sample_rate)
        wav_file.writeframes(bytes(frames * channels * sample_width))
    return buffer.getvalue()


run_checks(
    [
        (
            "mono 8 kHz 16-bit",
            lambda: parse_wav_header(make_wav(1, 8000, 2, 4000)),
            {
                "channels": 1,
                "sample_rate": 8000,
                "bits_per_sample": 16,
                "data_size": 8000,
                "duration_seconds": 0.5,
            },
        ),
        (
            "stereo 44.1 kHz 16-bit",
            lambda: parse_wav_header(make_wav(2, 44_100, 2, 44_100)),
            {
                "channels": 2,
                "sample_rate": 44_100,
                "bits_per_sample": 16,
                "data_size": 176_400,
                "duration_seconds": 1.0,
            },
        ),
        ("8-bit", lambda: parse_wav_header(make_wav(1, 22_050, 1, 11_025))["bits_per_sample"], 8),
        (
            "not RIFF",
            lambda: raised_message(
                lambda: parse_wav_header(b"RIFX" + make_wav(1, 8000, 2, 10)[4:])
            ).split(":")[0],
            "ValueError",
        ),
        (
            "not WAVE",
            lambda: raised_message(
                lambda: parse_wav_header(make_wav(1, 8000, 2, 10).replace(b"WAVE", b"AVI "))
            ).split(":")[0],
            "ValueError",
        ),
        (
            "too short",
            lambda: raised_message(lambda: parse_wav_header(b"RIFF")).split(":")[0],
            "ValueError",
        ),
        ("uses struct", lambda: "struct" in inspect.getsource(parse_wav_header), True),
    ]
)

FAIL  mono 8 kHz 16-bit: raised NotImplementedError: 
FAIL  stereo 44.1 kHz 16-bit: raised NotImplementedError: 
FAIL  8-bit: raised NotImplementedError: 
FAIL  not RIFF: expected 'ValueError', got 'NotImplementedError'
FAIL  not WAVE: expected 'ValueError', got 'NotImplementedError'
FAIL  too short: expected 'ValueError', got 'NotImplementedError'
FAIL  uses struct: expected True, got False
0/7 checks passed


#### Exercise 5 — a JSON encoder for `Decimal`, `datetime`, `UUID` and dataclasses

`RichJSONEncoder` (a `json.JSONEncoder` subclass) encodes `Decimal` as a string
(`"679.00"`), `datetime` and `date` with `isoformat()`, `UUID` as its string form,
and dataclass instances as dicts of their fields (nested values encoded the same
way). Anything else raises `TypeError` as usual.

In [415]:
class RichJSONEncoder(json.JSONEncoder):
    """JSON for Decimal, datetime/date, UUID and dataclasses."""

    def default(self, value: object) -> object:
        raise NotImplementedError

Check exercise 5:

In [416]:
@dataclasses.dataclass
class LabLine:
    sku: str
    price: Decimal


@dataclasses.dataclass
class LabOrder:
    order_id: uuid.UUID
    placed_at: datetime
    lines: list[LabLine]


LAB_MOMENT = datetime(2026, 11, 7, 9, 30, tzinfo=UTC)
ORDER_ID = uuid.UUID("12345678-1234-5678-1234-567812345678")
sample_order = LabOrder(ORDER_ID, LAB_MOMENT, [LabLine("TEA-500", Decimal("240.00"))])
run_checks(
    [
        (
            "Decimal",
            lambda: json.dumps({"total": Decimal("679.00")}, cls=RichJSONEncoder),
            '{"total": "679.00"}',
        ),
        (
            "datetime and date",
            lambda: json.dumps([LAB_MOMENT, date(2026, 11, 7)], cls=RichJSONEncoder),
            '["2026-11-07T09:30:00+00:00", "2026-11-07"]',
        ),
        (
            "UUID",
            lambda: json.dumps(ORDER_ID, cls=RichJSONEncoder),
            '"12345678-1234-5678-1234-567812345678"',
        ),
        (
            "nested dataclasses",
            lambda: json.loads(json.dumps(sample_order, cls=RichJSONEncoder)),
            {
                "order_id": str(ORDER_ID),
                "placed_at": "2026-11-07T09:30:00+00:00",
                "lines": [{"sku": "TEA-500", "price": "240.00"}],
            },
        ),
        (
            "dataclass class itself is not encoded",
            lambda: raised_message(lambda: json.dumps(LabLine, cls=RichJSONEncoder)).split(":")[0],
            "TypeError",
        ),
        (
            "other types still fail",
            lambda: raised_message(lambda: json.dumps({1, 2}, cls=RichJSONEncoder)).split(":")[0],
            "TypeError",
        ),
    ]
)

FAIL  Decimal: raised NotImplementedError: 
FAIL  datetime and date: raised NotImplementedError: 
FAIL  UUID: raised NotImplementedError: 
FAIL  nested dataclasses: raised NotImplementedError: 
FAIL  dataclass class itself is not encoded: expected 'TypeError', got 'NotImplementedError'
FAIL  other types still fail: expected 'TypeError', got 'NotImplementedError'
0/6 checks passed


#### Exercise 6 — safe zip extraction that rejects path traversal

`safe_extract_zip(archive, destination)` extracts a ZIP file into `destination`
(create it if needed) and returns the sorted list of extracted file paths relative
to `destination` (POSIX strings, directories excluded). Before extracting
**anything**, check every member: if any would land outside `destination`
(absolute names, `..` components, or a resolved path not relative to the resolved
destination), raise `ValueError` naming the member and extract nothing.

In [417]:
def safe_extract_zip(archive: Path, destination: Path) -> list[str]:
    """Extract archive into destination, refusing any member that escapes it."""
    raise NotImplementedError

Check exercise 6:

In [418]:
def make_zip(name: str, members: dict[str, str]) -> Path:
    path = LAB45_DIR / name
    with zipfile.ZipFile(path, "w") as archive:
        for member_name, text in members.items():
            archive.writestr(member_name, text)
    return path


def extract_into(archive: Path, folder_name: str) -> tuple[str, list[str]]:
    destination = LAB45_DIR / folder_name
    shutil.rmtree(destination, ignore_errors=True)
    outcome = raised_message(lambda: safe_extract_zip(archive, destination))
    written = (
        sorted(
            path.relative_to(destination).as_posix()
            for path in destination.rglob("*")
            if path.is_file()
        )
        if destination.exists()
        else []
    )
    return outcome.split(":")[0], written


def read_if_exists(path: Path) -> str | None:
    return path.read_text() if path.exists() else None


shutil.rmtree(LAB45_DIR / "extract_good", ignore_errors=True)  # start clean on every run
good_zip = make_zip(
    "good.zip",
    {"invoices/7.txt": "679.00", "invoices/8.txt": "240.00", "readme.txt": "hi", "empty/": ""},
)
run_checks(
    [
        (
            "good archive",
            lambda: safe_extract_zip(good_zip, LAB45_DIR / "extract_good"),
            ["invoices/7.txt", "invoices/8.txt", "readme.txt"],
        ),
        (
            "contents",
            lambda: read_if_exists(LAB45_DIR / "extract_good" / "invoices" / "7.txt"),
            "679.00",
        ),
        (
            "'..' member",
            lambda: extract_into(
                make_zip("dotdot.zip", {"ok.txt": "1", "../evil.txt": "x"}), "extract_dotdot"
            ),
            ("ValueError", []),
        ),
        (
            "nested '..'",
            lambda: extract_into(
                make_zip("nested.zip", {"a/../../evil.txt": "x"}), "extract_nested"
            ),
            ("ValueError", []),
        ),
        (
            "absolute member",
            lambda: extract_into(
                make_zip("absolute.zip", {"ok.txt": "1", "/tmp/evil.txt": "x"}), "extract_absolute"
            ),
            ("ValueError", []),
        ),
        ("nothing escaped", lambda: (LAB45_DIR.parent / "evil.txt").exists(), False),
    ]
)

FAIL  good archive: raised NotImplementedError: 
FAIL  contents: expected '679.00', got None
FAIL  '..' member: expected ('ValueError', []), got ('NotImplementedError', [])
FAIL  nested '..': expected ('ValueError', []), got ('NotImplementedError', [])
FAIL  absolute member: expected ('ValueError', []), got ('NotImplementedError', [])
PASS  nothing escaped
1/6 checks passed


### Recap — Unit 4.5

| Concept | One-line takeaway | JS / Node equivalent |
| --- | --- | --- |
| `open()` | Mode = action + text/binary; `TextIOWrapper` → buffer → `FileIO`; pass `encoding`; `newline=""` for CSV | `fs.open`, streams |
| Patterns | Iterate lines; `read(chunk)` with walrus; byte offsets with `seek`/`tell` in binary; `flush` + `fsync`; temp file + `os.replace` | `createReadStream`, `fs.rename` |
| In-memory / temp | `StringIO`, `BytesIO` for tests; `tempfile` for real files that clean up | `Buffer`, `os.tmpdir()` |
| Binary | `struct` with explicit `<`/`>`; `memoryview` for zero-copy slices; `mmap` for random access | `DataView`, `Buffer.subarray` |
| Encodings | UTF-8 everywhere; `utf-8-sig` for BOMs; fix mojibake by re-encoding; choose `errors=` deliberately | `TextDecoder` |
| Formats | JSON/CSV/TOML/MessagePack are data; YAML only with `safe_load`; XML only with `defusedxml` | `JSON`, `js-yaml` |
| `pickle` | Rebuilds objects by calling code named in the data: never load untrusted pickles; customize with `__getstate__`/`__reduce__` | `structuredClone` (no code) |
| Compression | `gzip`/`bz2`/`lzma`/`zstd` share an API; `zipfile`; `tarfile` with `filter="data"` | `zlib`, `tar` |
| `sqlite3` | `?` parameters, `executemany`, `with connection:` transactions, `sqlite3.Row` | `better-sqlite3` |
| Path safety | `resolve()` + `is_relative_to(base)` for any outside path | `path.resolve` + checks |

**Further reading**

- [`io`](https://docs.python.org/3/library/io.html),
  [`open()`](https://docs.python.org/3/library/functions.html#open),
  [`tempfile`](https://docs.python.org/3/library/tempfile.html),
  [`struct`](https://docs.python.org/3/library/struct.html),
  [`mmap`](https://docs.python.org/3/library/mmap.html)
- [`codecs`](https://docs.python.org/3/library/codecs.html),
  [Unicode HOWTO](https://docs.python.org/3/howto/unicode.html)
- [`tomllib`](https://docs.python.org/3/library/tomllib.html),
  [`xml.etree.ElementTree`](https://docs.python.org/3/library/xml.etree.elementtree.html)
  and [XML vulnerabilities](https://docs.python.org/3/library/xml.html#xml-vulnerabilities),
  [PyYAML](https://pyyaml.org/wiki/PyYAMLDocumentation),
  [defusedxml](https://github.com/tiran/defusedxml)
- [`pickle`](https://docs.python.org/3/library/pickle.html) (read the warning box),
  [`shelve`](https://docs.python.org/3/library/shelve.html)
- [`gzip`](https://docs.python.org/3/library/gzip.html),
  [`compression.zstd`](https://docs.python.org/3/library/compression.zstd.html),
  [`zipfile`](https://docs.python.org/3/library/zipfile.html),
  [`tarfile` extraction filters](https://docs.python.org/3/library/tarfile.html#extraction-filters)
- [`sqlite3`](https://docs.python.org/3/library/sqlite3.html)
- [PEP 686](https://peps.python.org/pep-0686/) (UTF-8 mode default),
  [PEP 706](https://peps.python.org/pep-0706/) (tar extraction filters),
  [PEP 784](https://peps.python.org/pep-0784/) (Zstandard),
  [PEP 3154](https://peps.python.org/pep-3154/) / [PEP 574](https://peps.python.org/pep-0574/) (pickle protocols 4 and 5)

## Unit 4.6 — Essential Third-Party Libraries

**Objectives**

- Evaluate a library before depending on it.
- Call HTTP APIs with `requests` and `httpx` (sync and async) and make them
  resilient with timeouts and `tenacity` retries.
- Validate untrusted data and configuration with Pydantic, `pydantic-settings` and
  `python-dotenv`.
- Build command-line apps with `typer` and `rich`.
- Parse dates with `python-dateutil`; serialize fast with `orjson`.
- Analyse tabular data with NumPy, pandas and Polars, and know when to use which.
- Scrape HTML politely with BeautifulSoup and `lxml`.
- Render HTML and e-mail with Jinja2; process images with Pillow; talk to S3 with
  `boto3`.
- Know the developer utilities `loguru`, `attrs`, `more-itertools` and `cachetools`.

**JS bridge: npm packages → PyPI packages**

| npm | PyPI | Note |
| --- | --- | --- |
| `axios`, `fetch` | `requests` (sync), `httpx` (sync + async) | Python HTTP calls **block** unless you use `httpx.AsyncClient` |
| `p-retry`, `axios-retry` | `tenacity` | Decorator-based retry policies |
| `zod` | `pydantic` | Models are classes with type hints |
| `dotenv` | `python-dotenv`, `pydantic-settings` | Settings are typed and validated |
| `commander`, `yargs`, `chalk`, `ora` | `typer`, `click`, `rich` | |
| `date-fns`, `dayjs` | `python-dateutil` (+ stdlib `datetime`) | |
| `lodash` (arrays) | `numpy`, `more-itertools` | |
| `danfo.js`, `arquero` | `pandas`, `polars` | Python's data ecosystem is far larger |
| `cheerio`, `jsdom` | `beautifulsoup4`, `lxml` | |
| `nunjucks`, `handlebars` | `jinja2` | Nunjucks was modelled on Jinja2 |
| `sharp` | `pillow` | |
| `@aws-sdk/client-s3` | `boto3` | |
| `pino`, `winston` | `loguru` (or stdlib `logging`) | |
| `lru-cache` | `cachetools`, `functools.lru_cache` | |

**No external network.** Every HTTP demo talks to a tiny API that the setup cell
starts on `127.0.0.1` in a background thread (`SHOP_API`): exchange rates, orders
(with three invalid records planted), a "flaky" endpoint that fails the first N
requests, a slow endpoint, an HTML page and a `robots.txt`. S3 calls are stubbed.
Outputs are deterministic except timings, which the markdown points out.

**The local shop API.** The next cell defines the server: a
`BaseHTTPRequestHandler` subclass with a few routes, served by
`ThreadingHTTPServer` in a daemon thread on a fixed port (18046, falling back to
any free port). The setup cell after it starts the server and prints versions.

In [419]:
import http.server
import json
import random
import threading
import time
import urllib.parse

SHOP_API_PORT = 18046  # fixed so outputs (URLs in messages) are stable; falls back to any free port


def demo_orders() -> list[dict[str, object]]:
    """40 seeded orders as an API would return them, with 3 invalid records planted."""
    generator = random.Random(46)
    cities = ["Pune", "Mumbai", "Bengaluru", "Hubli & Dharwad"]
    prices = {"TEA-500": "240.00", "MUG-02": "199.00", "JAR-750": "350.50", "KETTLE-9": "899.00"}
    orders: list[dict[str, object]] = []
    for order_id in range(1, 41):
        sku = generator.choice(sorted(prices))
        orders.append(
            {
                "order_id": order_id,
                "customer": f"customer-{order_id:03d}",
                "city": generator.choice(cities),
                "sku": sku,
                "quantity": generator.randint(1, 5),
                "unit_price": prices[sku],
                "placed_at": f"2026-11-{generator.randint(1, 7):02d}"
                f"T{generator.randint(9, 20):02d}:00:00+05:30",
            }
        )
    orders[6]["quantity"] = -2  # invalid: not positive
    del orders[17]["city"]  # invalid: missing field
    orders[29]["unit_price"] = "two hundred"  # invalid: not a number
    return orders


PRODUCTS_HTML = """<!doctype html>
<html><head><title>Chai Corner - Products</title></head>
<body>
  <h1>Products</h1>
  <ul id="catalogue">
    <li class="product" data-sku="TEA-500">
      <a href="/products/tea-500">Masala tea 500 g</a> <span class="price">₹240.00</span>
    </li>
    <li class="product" data-sku="MUG-02">
      <a href="/products/mug-02">Steel mug</a> <span class="price">₹199.00</span>
    </li>
    <li class="product sold-out" data-sku="JAR-750">
      <a href="/products/jar-750">Pickle jar 750 ml</a> <span class="price">₹350.50</span>
    </li>
  </ul>
  <p class="note">Prices include GST &amp; free delivery above ₹500.</p>
</body></html>
"""
ROBOTS_TXT = "User-agent: *\nDisallow: /admin/\nCrawl-delay: 1\n\nUser-agent: BadBot\nDisallow: /\n"


class ShopAPIHandler(http.server.BaseHTTPRequestHandler):
    """A tiny local HTTP API used by every HTTP demo (no external network)."""

    protocol_version = "HTTP/1.1"  # keep-alive, so clients can reuse connections
    hits: dict[str, int] = {}
    connections = 0

    def setup(self) -> None:
        type(self).connections += 1  # one per TCP connection, not per request
        super().setup()

    def log_message(self, format: str, *args: object) -> None:  # keep the notebook quiet
        pass

    def _send(self, status: int, body: object, content_type: str = "application/json") -> None:
        data = body.encode() if isinstance(body, str) else json.dumps(body).encode()
        try:
            self.send_response(status)
            self.send_header("Content-Type", f"{content_type}; charset=utf-8")
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        except BrokenPipeError, ConnectionResetError:  # the client gave up (a timeout demo)
            self.close_connection = True

    def _count(self, key: str) -> int:
        self.hits[key] = self.hits.get(key, 0) + 1
        return self.hits[key]

    def do_GET(self) -> None:
        url = urllib.parse.urlsplit(self.path)
        query = dict(urllib.parse.parse_qsl(url.query))
        parts = url.path.strip("/").split("/")
        fail_first = int(query.get("fail", 0))
        if url.path == "/rates":
            base = query.get("base", "INR")
            self._send(
                200,
                {
                    "base": base,
                    "date": "2026-11-07",
                    "rates": {"USD": 0.012, "EUR": 0.011, "GBP": 0.0094},
                },
            )
        elif url.path == "/orders":
            if "key" in query and self._count(query["key"]) <= fail_first:
                self._send(503, {"error": "service busy, try again"})
            else:
                self._send(200, demo_orders())
        elif parts[0] == "flaky" and len(parts) == 2:
            attempt = self._count(parts[1])
            if attempt <= fail_first:
                self._send(503, {"error": "busy", "attempt": attempt})
            else:
                self._send(200, {"ok": True, "attempt": attempt})
        elif url.path == "/hits":
            self._send(
                200, {"key": query.get("key"), "hits": self.hits.get(query.get("key", ""), 0)}
            )
        elif url.path == "/connections":
            self._send(200, {"connections": type(self).connections})
        elif url.path == "/reset":
            self.hits.clear()
            self._send(200, {"reset": True})
        elif url.path == "/slow":
            time.sleep(float(query.get("seconds", "2")))
            self._send(200, {"slow": True})
        elif parts[0] == "status" and len(parts) == 2:
            self._send(int(parts[1]), {"status": int(parts[1])})
        elif url.path == "/headers":
            self._send(
                200,
                {
                    name.lower(): value
                    for name, value in self.headers.items()
                    if name.lower() in ("user-agent", "x-request-id", "authorization")
                },
            )
        elif url.path == "/products.html":
            self._send(200, PRODUCTS_HTML, "text/html")
        elif url.path == "/robots.txt":
            self._send(200, ROBOTS_TXT, "text/plain")
        else:
            self._send(404, {"error": f"no route {url.path}"})

    def do_POST(self) -> None:
        length = int(self.headers.get("Content-Length", 0))
        payload = json.loads(self.rfile.read(length) or b"null")
        self._send(201 if self.path == "/orders" else 200, {"received": payload})


def start_shop_api() -> str:
    """Start the local API once (in a daemon thread) and return its base URL."""
    server = globals().get("_SHOP_API_SERVER")
    if server is None:
        try:
            server = http.server.ThreadingHTTPServer(("127.0.0.1", SHOP_API_PORT), ShopAPIHandler)
        except OSError:
            server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), ShopAPIHandler)
        server.daemon_threads = True
        threading.Thread(target=server.serve_forever, daemon=True).start()
        globals()["_SHOP_API_SERVER"] = server
    return f"http://127.0.0.1:{server.server_address[1]}"

Unit setup: imports, scratch folder, the API's base URL (`SHOP_API`) and package versions.

In [420]:
import asyncio
import contextlib
import importlib.metadata
import importlib.util
import inspect
import io
import os
import re
import shutil
import subprocess
import sys
import textwrap
from collections.abc import Callable
from datetime import UTC, date, datetime, timedelta
from decimal import Decimal
from pathlib import Path
from unittest import mock

REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
UNIT46_DIR = scratch_dir / "unit46"
UNIT46_DIR.mkdir(parents=True, exist_ok=True)
FIXED_MOMENT = datetime(2026, 11, 7, 9, 30, tzinfo=UTC)
SHOP_API = start_shop_api()


def tidy(text: str) -> str:
    """Show absolute paths as <scratch>, <repo>, <site-packages> or ~."""
    site_packages = Path(inspect.getfile(importlib.metadata)).parent.parent
    for folder, label in ((UNIT46_DIR, "<scratch>"), (REPO_ROOT, "<repo>"), (Path.home(), "~")):
        text = text.replace(str(folder), label)
    return re.sub(r" at 0x[0-9a-f]+", " at 0x...", text.replace(str(site_packages), "<stdlib>"))


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return tidy(f"{type(error).__name__}: {error}")
    return "no error"


def installed(module_name: str, package: str | None = None) -> bool:
    """True if module_name can be imported; otherwise print a skip message."""
    if importlib.util.find_spec(module_name) is not None:
        return True
    print(f"SKIP: {package or module_name} is not installed")
    return False


print(f"local shop API: {SHOP_API}")
for package in (
    "requests",
    "httpx",
    "tenacity",
    "pydantic",
    "pydantic-settings",
    "python-dotenv",
    "typer",
    "rich",
    "python-dateutil",
    "orjson",
    "numpy",
    "pandas",
    "polars",
    "beautifulsoup4",
    "lxml",
    "jinja2",
    "pillow",
    "boto3",
    "loguru",
    "attrs",
    "more-itertools",
    "cachetools",
):
    try:
        version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        version = "not installed (its cells print a skip message)"
    print(f"  {package:<18} {version}")

local shop API: http://127.0.0.1:18046
  requests           2.34.2
  httpx              0.28.1
  tenacity           9.2.1
  pydantic           2.14.3


  pydantic-settings  2.16.2
  python-dotenv      1.2.4
  typer              0.28.0
  rich               15.0.0
  python-dateutil    2.9.0.post0
  orjson             3.13.0
  numpy              2.5.4
  pandas             3.1.1
  polars             2.1.0


  beautifulsoup4     4.15.0
  lxml               6.1.3
  jinja2             3.1.6
  pillow             13.0.0
  boto3              1.43.126
  loguru             0.7.3
  attrs              26.1.0
  more-itertools     11.2.1
  cachetools         8.0.0


### 4.6.1 Choosing a library

Every dependency is code you ship, update and answer for. Check before you add:

| Question | Where to look | Warning signs |
| --- | --- | --- |
| Is it maintained? | release history on PyPI, recent commits, open/closed issues and PRs, number of maintainers | no release in 2+ years, issues ignored, one maintainer who left |
| Is it used? | download stats (pypistats.org, libraries.io), dependents on GitHub | a fork or a typo-squatting name (Unit 4.1) |
| Is it typed? | a `py.typed` marker (PEP 561) or stubs on typeshed | `Any` everywhere in your code |
| Licence? | `License-Expression` / classifiers in the metadata | GPL/AGPL in a closed product, no licence at all |
| Known vulnerabilities? | `pip-audit` (Unit 4.1), GitHub/OSV advisories | open advisories without fixes |
| How heavy? | its dependency tree, wheel size, compiled parts, Python-version support | pulls in dozens of packages for one function |
| Can you read it? | the docs **and the source** | magic you cannot debug |

Much of this is in the installed package's metadata. A small report for packages
this unit uses (download stats and advisories need the network, so they are not
shown here):

In [421]:
from importlib.metadata import distribution, requires

from packaging.requirements import Requirement


def dependency_tree(package: str, seen: set[str] | None = None) -> set[str]:
    """All runtime dependencies of an installed package (extras and markers ignored)."""
    seen = set() if seen is None else seen
    for requirement_text in requires(package) or []:
        requirement = Requirement(requirement_text)
        if requirement.marker and not requirement.marker.evaluate({"extra": ""}):
            continue
        name = requirement.name.lower()
        if name not in seen:
            seen.add(name)
            with contextlib.suppress(importlib.metadata.PackageNotFoundError):
                dependency_tree(name, seen)
    return seen


print(
    f"{'package':<12}{'version':<10}{'licence':<26}"
    f"{'typed':<7}{'deps':>5}{'all':>5}  requires-python"
)
for package in ("requests", "httpx", "tenacity", "pydantic", "polars", "pandas", "boto3"):
    metadata = distribution(package).metadata
    classifiers = [
        value.rsplit(" :: ", 1)[-1]
        for value in metadata.get_all("Classifier") or []
        if value.startswith("License ::")
    ]
    licence = (
        metadata.get("License-Expression")
        or (classifiers or [(metadata.get("License") or "?").splitlines()[0]])[0]
    )
    typed = any(str(file).endswith("py.typed") for file in distribution(package).files or [])
    direct = [text for text in requires(package) or [] if "extra ==" not in text]
    print(
        f"{package:<12}{metadata['Version']:<10}{licence[:24]:<26}{str(typed):<7}{len(direct):>5}"
        f"{len(dependency_tree(package)):>5}  {metadata.get('Requires-Python')}"
    )

package     version   licence                   typed   deps  all  requires-python
requests    2.34.2    Apache Software License   True       4    4  >=3.10
httpx       0.28.1    BSD License               True       4    5  >=3.8
tenacity    9.2.1     Apache-2.0                True       0    0  >=3.10
pydantic    2.14.3    MIT                       True       4    4  >=3.10
polars      2.1.0     MIT License               True       1    1  >=3.11


pandas      3.1.1     BSD-3-Clause              False      5    3  >=3.11


boto3       1.43.126  Apache-2.0                False      3    6  >= 3.10


**Reading the source** is the fastest way to answer "what does this actually do?".
`inspect.getsource` works on any pure-Python library; here it shows that
`requests.get` is a two-line wrapper that builds a throwaway `Session` for every
call (one reason to use a `Session` yourself, 4.6.2):

In [422]:
import requests

print(tidy(inspect.getsourcefile(requests.get)))
get_lines = inspect.getsource(requests.get).splitlines()
print(f"{get_lines[0]}\n    ...docstring...\n{get_lines[-1]}")
print("\n".join(inspect.getsource(requests.api.request).splitlines()[-3:]))

<repo>/.venv/lib/python3.15/site-packages/requests/api.py
def get(
    ...docstring...
    return request("get", url, params=params, **kwargs)
    # cases, and look like a memory leak in others.
    with sessions.Session() as session:
        return session.request(method=method, url=url, **kwargs)


### 4.6.2 HTTP and resilience

**`requests`** is the classic synchronous HTTP library. Three habits matter more
than any API detail:

1. **Always pass `timeout=`.** The default is *no timeout*: a stalled server hangs
   your program forever. Use `(connect, read)` seconds.
2. **Call `raise_for_status()`.** A 404 or 500 is a normal response, not an
   exception, until you ask.
3. **Use a `Session`** for repeated calls: it reuses TCP connections (much faster),
   and keeps headers, cookies and auth in one place.

In [423]:
response = requests.get(f"{SHOP_API}/rates", params={"base": "INR"}, timeout=(3.05, 10))
print(f"{response.status_code=}, {response.headers['Content-Type']=}")
print(f"{response.url=}")
print(f"{response.json()=}")
missing = requests.get(f"{SHOP_API}/status/404", timeout=5)
print(f"{missing.ok=}, {missing.status_code=}  (no exception yet)")
print(raised_message(missing.raise_for_status))
print(raised_message(lambda: requests.get(f"{SHOP_API}/slow", params={"seconds": 2}, timeout=0.2)))

response.status_code=200, response.headers['Content-Type']='application/json; charset=utf-8'
response.url='http://127.0.0.1:18046/rates?base=INR'
response.json()={'base': 'INR', 'date': '2026-11-07', 'rates': {'USD': 0.012, 'EUR': 0.011, 'GBP': 0.0094}}
missing.ok=False, missing.status_code=404  (no exception yet)
HTTPError: 404 Client Error: Not Found for url: http://127.0.0.1:18046/status/404


ReadTimeout: HTTPConnectionPool(host='127.0.0.1', port=18046): Read timed out. (read timeout=0.2)


A `Session` with default headers (user agent, auth token) for every request, and a
context manager that closes its connection pool. Connection reuse is the main speed
win: across the internet every new connection costs a TCP and TLS handshake (tens to
hundreds of milliseconds). The local server counts TCP connections, which shows the
reuse directly: twenty requests through a session use one connection, twenty
`requests.get` calls use twenty.

In [424]:
def connections_opened(make_calls: Callable[[], None]) -> int:
    """TCP connections the server accepted while make_calls ran."""
    before = requests.get(f"{SHOP_API}/connections", timeout=5).json()["connections"]
    make_calls()
    after = requests.get(f"{SHOP_API}/connections", timeout=5).json()["connections"]
    return after - before - 1  # the second check opened one connection itself


def twenty_with_session() -> None:
    with requests.Session() as session:
        session.headers.update(
            {"User-Agent": "chai-corner-reports/1.0", "Authorization": "Bearer demo-token"}
        )
        print(f"{session.get(f'{SHOP_API}/headers', timeout=5).json()=}")
        for _ in range(19):
            session.get(f"{SHOP_API}/rates", timeout=5).raise_for_status()


reused = connections_opened(twenty_with_session)
fresh = connections_opened(
    lambda: [requests.get(f"{SHOP_API}/rates", timeout=5).raise_for_status() for _ in range(20)]
)
print(f"20 requests: {reused} connection(s) with a Session, {fresh} without")
created = requests.post(f"{SHOP_API}/orders", json={"sku": "TEA-500", "quantity": 2}, timeout=5)
print(f"POST: {created.status_code=}, {created.json()=}")

session.get(f'{SHOP_API}/headers', timeout=5).json()={'user-agent': 'chai-corner-reports/1.0', 'authorization': 'Bearer demo-token'}


20 requests: 1 connection(s) with a Session, 20 without
POST: created.status_code=201, created.json()={'received': {'sku': 'TEA-500', 'quantity': 2}}


**`httpx`** has a `requests`-like API plus: **async** support, HTTP/2, strict
timeouts by default (5 s), and pluggable transports for testing. The recommended
pattern is **one shared `Client`** for the whole application (connection pool,
base URL, default headers and timeouts), created once and closed at shutdown, not
a new client per request.

In [425]:
import httpx

with httpx.Client(
    base_url=SHOP_API,
    timeout=httpx.Timeout(5.0, connect=2.0),
    headers={"User-Agent": "chai-corner/1.0"},
) as client:
    rates = client.get("/rates", params={"base": "INR"})
    rates.raise_for_status()
    print(f"{rates.status_code=}, {rates.json()['rates']=}")
    print(f"{rates.http_version=}, {rates.elapsed.total_seconds() < 1=}")
    print(raised_message(lambda: client.get("/status/503").raise_for_status()).splitlines()[0])
    print(raised_message(lambda: client.get("/slow", params={"seconds": 2}, timeout=0.2)))
print(f"{httpx.Timeout(5.0)=}  (requests has no default timeout at all)")

rates.status_code=200, rates.json()['rates']={'USD': 0.012, 'EUR': 0.011, 'GBP': 0.0094}
rates.http_version='HTTP/1.1', rates.elapsed.total_seconds() < 1=True
HTTPStatusError: Server error '503 Service Unavailable' for url 'http://127.0.0.1:18046/status/503'


ReadTimeout: timed out
httpx.Timeout(5.0)=Timeout(timeout=5.0)  (requests has no default timeout at all)


**Async httpx.** An `AsyncClient` lets one thread wait for many requests at once
(Phase 5 explains the event loop). Five calls to the slow endpoint, each taking
0.3 s, finish in about 0.3 s together instead of 1.5 s one after another. The kernel
already runs an event loop, so the cell can `await` at top level.

In [426]:
async def fetch_all(paths: list[str]) -> list[dict[str, object]]:
    async with httpx.AsyncClient(base_url=SHOP_API, timeout=5) as client:
        responses = await asyncio.gather(*(client.get(path) for path in paths))
        return [response.raise_for_status().json() for response in responses]


started = time.perf_counter()
results = await fetch_all(["/slow?seconds=0.3"] * 5)
elapsed = time.perf_counter() - started
print(f"{len(results)} responses in {elapsed:.1f} s (concurrent; sequential would be ~1.5 s)")

5 responses in 0.3 s (concurrent; sequential would be ~1.5 s)


**Testing without a server: `httpx.MockTransport`.** A transport is the layer that
actually sends bytes; a mock transport is a function that receives the `Request`
and returns a `Response`. Code under test keeps using a normal `Client`.

In [427]:
def fake_rates_service(request: httpx.Request) -> httpx.Response:
    if request.url.path != "/rates":
        return httpx.Response(404, json={"error": "not found"})
    return httpx.Response(
        200, json={"rates": {"USD": 0.012}, "asked_for": request.url.params["base"]}
    )


with httpx.Client(
    transport=httpx.MockTransport(fake_rates_service), base_url="https://rates.example.com"
) as fake_client:
    print(f"{fake_client.get('/rates', params={'base': 'INR'}).json()=}")
    print(f"{fake_client.get('/nope').status_code=}")

fake_client.get('/rates', params={'base': 'INR'}).json()={'rates': {'USD': 0.012}, 'asked_for': 'INR'}
fake_client.get('/nope').status_code=404


**`tenacity`: retry policies.** Networks fail transiently (timeouts, 502/503
responses, dropped connections). Retrying blindly makes outages worse, so a policy
states **what** to retry (`retry=`), **how long to wait** between attempts
(`wait=`, usually exponential backoff with jitter, so thousands of clients do not
retry in lockstep), and **when to stop** (`stop=`). `reraise=True` raises the last
real error instead of tenacity's `RetryError`. Waits are tiny here so the demo is
fast; real values are seconds.

| Building block | Examples |
| --- | --- |
| stop | `stop_after_attempt(5)`, `stop_after_delay(30)`, combine with `\|` |
| wait | `wait_fixed(1)`, `wait_exponential(multiplier=0.5, max=10)`, `wait_random_exponential(...)` (jitter) |
| retry | `retry_if_exception_type(...)`, `retry_if_result(predicate)`, `retry_if_exception(predicate)` |
| hooks | `before_sleep=before_sleep_log(logger, logging.WARNING)`, `after=` |

In [428]:
from tenacity import (
    RetryError,
    retry,
    retry_if_exception,
    stop_after_attempt,
    wait_exponential,
)


def is_transient(error: BaseException) -> bool:
    """Retry timeouts, connection errors and 5xx responses; never 4xx."""
    if isinstance(error, httpx.HTTPStatusError):
        return error.response.status_code >= 500
    return isinstance(error, httpx.TransportError)


attempts_log: list[str] = []


@retry(
    retry=retry_if_exception(is_transient),
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=0.01, max=0.05),
    before_sleep=lambda state: attempts_log.append(
        f"attempt {state.attempt_number} failed: {state.outcome.exception().response.status_code}"
    ),
    reraise=True,
)
def get_json(client: httpx.Client, path: str) -> dict[str, object]:
    return client.get(path).raise_for_status().json()


with httpx.Client(base_url=SHOP_API, timeout=5) as client:
    client.get("/reset")
    print(f"{get_json(client, '/flaky/demo-a?fail=2')=}")
    print(f"{attempts_log=}")
    print(raised_message(lambda: get_json(client, "/flaky/demo-b?fail=10")).splitlines()[0])
    print(f"{client.get('/hits', params={'key': 'demo-b'}).json()=}  (gave up after 4)")
    print(
        raised_message(lambda: get_json(client, "/status/404")).splitlines()[0]
        + "  (4xx: not retried)"
    )
    print(f"{get_json.statistics['attempt_number']=}  (statistics of the last call)")

get_json(client, '/flaky/demo-a?fail=2')={'ok': True, 'attempt': 3}
attempts_log=['attempt 1 failed: 503', 'attempt 2 failed: 503']


HTTPStatusError: Server error '503 Service Unavailable' for url 'http://127.0.0.1:18046/flaky/demo-b?fail=10'
client.get('/hits', params={'key': 'demo-b'}).json()={'key': 'demo-b', 'hits': 4}  (gave up after 4)
HTTPStatusError: Client error '404 Not Found' for url 'http://127.0.0.1:18046/status/404'  (4xx: not retried)
get_json.statistics['attempt_number']=1  (statistics of the last call)


The same policy as an object (`Retrying`), retrying on a **result** instead of an
exception (poll until a job is ready), with a time-based stop:

In [429]:
from tenacity import Retrying, retry_if_result, stop_after_delay, wait_fixed

job_states = iter(["queued", "running", "running", "done"])
poller = Retrying(
    retry=retry_if_result(lambda state: state != "done"),
    wait=wait_fixed(0.01),
    stop=stop_after_delay(2),
)
print(f"{poller(lambda: next(job_states))=}, attempts: {poller.statistics['attempt_number']}")
print(
    raised_message(
        lambda: Retrying(stop=stop_after_attempt(2), retry=retry_if_result(lambda value: True))(
            lambda: "never done"
        )
    )
)
print(f"{RetryError.__mro__[1].__name__=}")

poller(lambda: next(job_states))='done', attempts: 4
RetryError: RetryError[<Future at 0x... state=finished returned str>]
RetryError.__mro__[1].__name__='Exception'


### 4.6.3 Validation and configuration

**Pydantic v2** turns untrusted data (an API response, a config file, user input)
into typed objects, or tells you exactly what is wrong. Unit 4.2.13 showed it reads
type hints at run time; here it is used outside any web framework. Models declare
fields with hints and `Field` constraints; validation **coerces** compatible input
(`"3"` → `3` in the default lax mode) and collects every error.

In [430]:
from typing import Annotated

from pydantic import (
    AwareDatetime,
    BaseModel,
    ConfigDict,
    Field,
    PositiveInt,
    TypeAdapter,
    ValidationError,
    field_validator,
)


class ApiOrder(BaseModel):
    model_config = ConfigDict(frozen=True, extra="ignore")

    order_id: PositiveInt
    customer: str
    city: str
    sku: Annotated[str, Field(pattern=r"^[A-Z]{2,6}-\d{1,4}$")]
    quantity: PositiveInt
    unit_price: Annotated[Decimal, Field(gt=0, max_digits=8, decimal_places=2)]
    placed_at: AwareDatetime

    @field_validator("city")
    @classmethod
    def city_is_title_case(cls, value: str) -> str:
        return value.strip()


raw_orders = httpx.get(f"{SHOP_API}/orders", timeout=5).json()
first = ApiOrder.model_validate(raw_orders[0])
print(repr(first))
print(f"{first.unit_price * first.quantity=}, {first.placed_at.utcoffset()=}")

ApiOrder(order_id=1, customer='customer-001', city='Hubli & Dharwad', sku='JAR-750', quantity=1, unit_price=Decimal('350.50'), placed_at=datetime.datetime(2026, 11, 5, 18, 0, tzinfo=TzInfo(19800)))
first.unit_price * first.quantity=Decimal('350.50'), first.placed_at.utcoffset()=datetime.timedelta(seconds=19800)


**Validating a whole API response.** `TypeAdapter(list[ApiOrder])` validates a list
in one go; a `ValidationError` lists every problem with its location
(`[index, field]`), message and type. Validating record by record instead keeps the
good ones and reports the bad ones, the usual choice for imports.

In [431]:
print(
    raised_message(lambda: TypeAdapter(list[ApiOrder]).validate_python(raw_orders)).splitlines()[0]
)
valid_orders, problems = [], []
for index, record in enumerate(raw_orders):
    try:
        valid_orders.append(ApiOrder.model_validate(record))
    except ValidationError as error:
        problems.extend(
            f"record {index}: {'.'.join(map(str, issue['loc']))}: {issue['msg']}"
            for issue in error.errors()
        )
print(f"{len(valid_orders)} valid, {len(problems)} problems:")
print(textwrap.indent("\n".join(problems), "  "))
print(f"{first.model_dump(mode='json')}")

ValidationError: 3 validation errors for list[ApiOrder]
37 valid, 3 problems:
  record 6: quantity: Input should be greater than 0
  record 17: city: Field required
  record 29: unit_price: Input should be a valid decimal
{'order_id': 1, 'customer': 'customer-001', 'city': 'Hubli & Dharwad', 'sku': 'JAR-750', 'quantity': 1, 'unit_price': '350.50', 'placed_at': '2026-11-05T18:00:00+05:30'}


**Config files.** The same models validate configuration loaded from TOML, YAML or
JSON, so a typo or a wrong type fails at startup with a clear message instead of
deep inside the program. `strict=True` turns off coercion where it would hide
mistakes.

In [432]:
import tomllib


class WarehouseConfig(BaseModel):
    code: Annotated[str, Field(min_length=4, max_length=4)]
    capacity: PositiveInt


class ShopConfig(BaseModel):
    model_config = ConfigDict(extra="forbid")

    store_name: str
    gst_percent: Annotated[int, Field(ge=0, le=28)]
    currency: str = "INR"
    warehouses: list[WarehouseConfig]
    debug: Annotated[bool, Field(strict=True)] = False


good_toml = (
    'store_name = "Pune Central"\ngst_percent = 18\n'
    '[[warehouses]]\ncode = "PNQ1"\ncapacity = 5000\n'
)
print(repr(ShopConfig.model_validate(tomllib.loads(good_toml))))
bad_toml = (
    'store_name = "Pune"\ngst_percent = 45\ndebug = "yes"\nwarehose = []\n'
    '[[warehouses]]\ncode = "PNQ"\ncapacity = 0\n'
)
try:
    ShopConfig.model_validate(tomllib.loads(bad_toml))
except ValidationError as error:
    for issue in error.errors():
        print(f"  {'.'.join(map(str, issue['loc'])):<22} {issue['msg']}")

ShopConfig(store_name='Pune Central', gst_percent=18, currency='INR', warehouses=[WarehouseConfig(code='PNQ1', capacity=5000)], debug=False)
  gst_percent            Input should be less than or equal to 28
  warehouses.0.code      String should have at least 4 characters
  warehouses.0.capacity  Input should be greater than 0
  debug                  Input should be a valid boolean
  warehose               Extra inputs are not permitted


**`pydantic-settings`** reads settings from **environment variables** (and `.env`
files, secrets folders, CLI arguments), validates them with the same machinery, and
gives you one typed object, the twelve-factor way of configuring an app. Field
`api_base_url` is read from `SHOP_API_BASE_URL` because of `env_prefix`; nested
values use a delimiter; `SecretStr` hides secrets in reprs and logs.

In [433]:
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="SHOP_", env_file=None)

    api_base_url: str = "http://127.0.0.1:8000"
    request_timeout_seconds: float = 5.0
    max_retries: Annotated[int, Field(ge=0, le=10)] = 3
    api_token: SecretStr
    debug: bool = False


environment = {
    "SHOP_API_BASE_URL": SHOP_API,
    "SHOP_MAX_RETRIES": "5",
    "SHOP_API_TOKEN": "tok_demo_123",
    "SHOP_DEBUG": "true",
}
with mock.patch.dict(os.environ, environment):
    settings = AppSettings()
print(repr(settings))
print(f"{settings.max_retries=} (int from a string), {settings.api_token.get_secret_value()=}")
with mock.patch.dict(os.environ, {"SHOP_MAX_RETRIES": "50"}, clear=False):
    error_lines = raised_message(lambda: AppSettings()).splitlines()
print("\n".join(line for line in error_lines if "further information" not in line))

AppSettings(api_base_url='http://127.0.0.1:18046', request_timeout_seconds=5.0, max_retries=5, api_token=SecretStr('**********'), debug=True)
settings.max_retries=5 (int from a string), settings.api_token.get_secret_value()='tok_demo_123'
ValidationError: 2 validation errors for AppSettings
max_retries
  Input should be less than or equal to 10 [type=less_than_equal, input_value='50', input_type=str]
api_token
  Field required [type=missing, input_value={'max_retries': '50'}, input_type=dict]


**`.env` files and `python-dotenv`.** A `.env` file holds local development
settings (never commit real secrets; the course's `.gitignore` excludes `.env`).
`dotenv_values` parses one into a dict without touching the environment;
`load_dotenv` copies the values into `os.environ` and by default does **not**
override variables that are already set, so real environment variables win.
`pydantic-settings` reads the same files with `env_file=`.

In [434]:
from dotenv import dotenv_values, load_dotenv

env_file = UNIT46_DIR / "demo.env"
env_file.write_text(
    "# local development only\nSHOP_API_TOKEN=tok_from_dotenv\nSHOP_MAX_RETRIES=2\n"
    'SHOP_API_BASE_URL="http://localhost:9000"  # quoted\n',
    encoding="utf-8",
)
print(f"{dotenv_values(env_file)=}")
with mock.patch.dict(os.environ, {"SHOP_MAX_RETRIES": "7"}):
    load_dotenv(env_file)  # does not override SHOP_MAX_RETRIES=7
    print(f"{os.environ['SHOP_MAX_RETRIES']=}, {os.environ['SHOP_API_TOKEN']=}")
    print(f"{AppSettings(_env_file=env_file).max_retries=}  (environment beats the file)")
print(f"after the block: {'SHOP_API_TOKEN' in os.environ=}")

dotenv_values(env_file)=OrderedDict({'SHOP_API_TOKEN': 'tok_from_dotenv', 'SHOP_MAX_RETRIES': '2', 'SHOP_API_BASE_URL': 'http://localhost:9000'})
os.environ['SHOP_MAX_RETRIES']='7', os.environ['SHOP_API_TOKEN']='tok_from_dotenv'
AppSettings(_env_file=env_file).max_retries=7  (environment beats the file)
after the block: 'SHOP_API_TOKEN' in os.environ=False


**Predict the output.**

```python
from pydantic import BaseModel

class Line(BaseModel):
    quantity: int

print(Line(quantity="3").quantity + 1, Line.model_validate({"quantity": 3.0}).quantity)
```

<details><summary>Answer</summary>

`4 3`. In the default **lax** mode Pydantic coerces `"3"` to `3` and the float `3.0`
(which has no fractional part) to the int `3`; `3.5` or `"three"` would fail. Use
`ConfigDict(strict=True)` or `Field(strict=True)` to reject anything that is not
already an `int`.

</details>

Run the next cell to check your prediction.

In [435]:
class Line(BaseModel):
    quantity: int


print(Line(quantity="3").quantity + 1, Line.model_validate({"quantity": 3.0}).quantity)
print(raised_message(lambda: Line(quantity=3.5)).splitlines()[2].strip())

4 3
Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=3.5, input_type=float]


### 4.6.4 Command-line apps

**`typer`** builds a CLI from **type hints**, the same idea as FastAPI: parameters
without defaults become arguments, parameters with defaults become `--options`,
`bool` options become flags, and `--help` is generated from the docstrings.
`typer.testing.CliRunner` invokes the app in-process and captures its output,
which is how CLIs are tested. (Compare the hand-written `argparse` version in Unit
4.4.8.)

In [436]:
import typer
from typer.testing import CliRunner

shopctl = typer.Typer(help="Chai Corner admin tool.", add_completion=False, rich_markup_mode=None)


@shopctl.command()
def restock(sku: str, quantity: int = 1, dry_run: bool = False) -> None:
    """Add stock for SKU."""
    verb = "would add" if dry_run else "added"
    typer.echo(f"{verb} {quantity} x {sku}")


@shopctl.command()
def rates(base: str = typer.Option("INR", help="Currency to convert from.")) -> None:
    """Show exchange rates from the shop API."""
    for currency, rate in (
        httpx.get(f"{SHOP_API}/rates", params={"base": base}, timeout=5).json()["rates"].items()
    ):
        typer.echo(f"1 {base} = {rate} {currency}")


runner = CliRunner()
plain = {"NO_COLOR": "1", "TERM": "dumb", "COLUMNS": "80"}
for arguments in (
    ["restock", "TEA-500", "--quantity", "24", "--dry-run"],
    ["rates"],
    ["restock", "TEA-500", "--quantity", "many"],
):
    result = runner.invoke(shopctl, arguments, env=plain, prog_name="shopctl")
    print(f"$ shopctl {' '.join(arguments)}   (exit {result.exit_code})")
    print(textwrap.indent(result.output.rstrip(), "  "))

$ shopctl restock TEA-500 --quantity 24 --dry-run   (exit 0)
  would add 24 x TEA-500
$ shopctl rates   (exit 0)
  1 INR = 0.012 USD
  1 INR = 0.011 EUR
  1 INR = 0.0094 GBP
$ shopctl restock TEA-500 --quantity many   (exit 2)
  Usage: shopctl restock [OPTIONS] {sku}
  Try 'shopctl restock --help' for help.

  Error: Invalid value for '--quantity': 'many' is not a valid int.


The generated help for one command (rich draws the boxes):

In [437]:
print(runner.invoke(shopctl, ["restock", "--help"], env=plain, prog_name="shopctl").output)

Usage: shopctl restock [OPTIONS] {sku}

  Add stock for SKU.

Arguments:
  sku  [required]

Options:
  --quantity <int>          [default: 1]
  --dry-run / --no-dry-run  [default: no-dry-run]
  --help                    Show this message and exit.



**`click`** is the library typer is built on: commands and options are declared with
decorators instead of type hints, and it is still common in existing code (Flask's
CLI, many tools). Recent typer releases ship their own copy of click inside
`typer` (`typer._click`), so a standalone `click` package is only installed if
something else needs it. `click` is not an Appendix A package; the cell checks
first.

```python
import click

@click.command()
@click.argument("sku")
@click.option("--quantity", default=1, type=int, help="Units to add.")
@click.option("--dry-run", is_flag=True)
def restock(sku: str, quantity: int, dry_run: bool) -> None:
    click.echo(f"{'would add' if dry_run else 'added'} {quantity} x {sku}")
```

In [438]:
print(f"typer's bundled click: {importlib.util.find_spec('typer._click') is not None}")
if installed("click", "click (not an Appendix A package; typer bundles its own copy)"):
    import click

    @click.command()
    @click.argument("sku")
    @click.option("--quantity", default=1, type=int)
    def click_restock(sku: str, quantity: int) -> None:
        click.echo(f"added {quantity} x {sku}")

    print(click.testing.CliRunner().invoke(click_restock, ["TEA-500", "--quantity", "3"]).output)

typer's bundled click: True
SKIP: click (not an Appendix A package; typer bundles its own copy) is not installed


**`rich`** renders tables, progress bars, syntax-highlighted tracebacks and
coloured logs in the terminal. A `Console(file=..., color_system=None)` writes plain
text to any file, which keeps this notebook's output deterministic (in a real
terminal you just use `rich.print` or `Console()`). **Tables:**

In [439]:
from rich.console import Console
from rich.table import Table


def plain_console(width: int = 80) -> Console:
    """A rich Console that writes plain text (no colours) to a buffer."""
    return Console(
        file=io.StringIO(),
        width=width,
        color_system=None,
        force_terminal=False,
        force_jupyter=False,
    )


table = Table(title="Low stock", show_lines=False)
table.add_column("SKU", style="bold")
table.add_column("Product")
table.add_column("Qty", justify="right")
table.add_column("Value (INR)", justify="right")
for sku, product, quantity, price in [
    ("MUG-02", "Steel mug", 3, Decimal("199")),
    ("JAR-750", "Pickle jar 750 ml", 1, Decimal("350.50")),
]:
    table.add_row(sku, product, str(quantity), f"{quantity * price:,.2f}")
console = plain_console()
console.print(table)
print(console.file.getvalue())

                     Low stock                     
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━━━━━━━━━┓
┃ SKU     ┃ Product           ┃ Qty ┃ Value (INR) ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━━━━━━━━━┩
│ MUG-02  │ Steel mug         │   3 │      597.00 │
│ JAR-750 │ Pickle jar 750 ml │   1 │      350.50 │
└─────────┴───────────────────┴─────┴─────────────┘



**Progress bars** (`rich.progress.track` or `Progress` for several tasks) show
work as it happens; in a non-terminal they print only the final state. **Logging
handler:** `RichHandler` formats stdlib `logging` records with levels, times and
paths (the options below switch time and path off for stable output).

In [440]:
import logging

from rich.logging import RichHandler
from rich.progress import track

console = plain_console()
processed = sum(
    1 for _ in track(range(250), description="Importing orders", console=console, transient=False)
)
print(f"{processed=}")
print(console.file.getvalue().strip().splitlines()[-1])

console = plain_console()
rich_logger = logging.getLogger("unit46.rich")
rich_logger.handlers[:] = [RichHandler(console=console, show_time=False, show_path=False)]
rich_logger.propagate = False
rich_logger.warning("stock for %s is low (%d left)", "MUG-02", 3)
rich_logger.error("payment gateway timed out")
print(console.file.getvalue())
rich_logger.handlers.clear()

processed=250
Importing orders ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
WARNING  stock for MUG-02 is low (3 left)                                       
ERROR    payment gateway timed out                                              



**Pretty tracebacks.** `rich.traceback.install()` replaces the default exception
hook for a whole program; `console.print_exception()` renders the current exception
with the surrounding source lines (and local variables with `show_locals=True`). It
runs here in a small script in a child process, so the frames and paths are clean:

In [441]:
trace_script = UNIT46_DIR / "rich_traceback_demo.py"
trace_script.write_text(
    textwrap.dedent("""
    from decimal import Decimal

    from rich.console import Console


    def apply_discount(total, percent):
        return total * (100 - percent) / 100 / 0  # bug


    console = Console(color_system=None, width=150)
    try:
        apply_discount(Decimal("679.00"), 10)
    except ArithmeticError:
        console.print_exception(show_locals=True, width=150)
"""),
    encoding="utf-8",
)
completed = subprocess.run(
    [sys.executable, str(trace_script)],
    capture_output=True,
    text=True,
    timeout=60,
    env={**os.environ, "NO_COLOR": "1"},
)
long_name, short_name = str(trace_script), "<scratch>/rich_traceback_demo.py"
pad = " " * (len(long_name) - len(short_name))  # keep rich's box borders aligned
print(
    re.sub(
        re.escape(long_name) + r"(:\d+ in \S+)",
        lambda found: short_name + found.group(1) + pad,
        completed.stdout,
    ).rstrip()
)

╭──────────────────────────────────────────────────────── Traceback (most recent call last) ─────────────────────────────────────────────────────────╮
│ <scratch>/rich_traceback_demo.py:13 in <module>                                                                                                    │
│                                                                                                                                                    │
│   10                                                                                          ╭────────────── locals ──────────────╮               │
│   11 console = Console(color_system=None, width=150)                                          │ console = <console width=150 None> │               │
│   12 try:                                                                                     ╰────────────────────────────────────╯               │
│ ❱ 13 │   apply_discount(Decimal("679.00"), 10)                                              

### 4.6.5 Dates and serialization

**`python-dateutil`** fills two gaps in `datetime`. `parser.parse` reads almost any
human-written date without a format string (`dayfirst=True` for Indian/European
`07/11/2026`; `fuzzy=True` skips surrounding words). Beware: `dayfirst=True` also
swaps the parts of an ISO date such as `2026-11-07`, so apply it only to input
known to be day-first. Use it at the edges for messy
input; for data you control, `datetime.fromisoformat` is stricter and faster.

In [442]:
from dateutil import parser as date_parser
from dateutil import tz

for text in (
    "7 November 2026 3pm",
    "Sat, 07 Nov 2026 09:30:00 GMT",
    "Nov 7 2026 15:00 IST",
    "2026-11-07T15:00:00+05:30",
):
    parsed = date_parser.parse(text, tzinfos={"IST": tz.gettz("Asia/Kolkata")})
    print(f"  {text!r:<34} -> {parsed.isoformat()}")
print(f"{date_parser.parse('07/11/2026')=}  (default is month first!)")
print(f"{date_parser.parse('07/11/2026', dayfirst=True)=}")
print(
    f"{date_parser.parse('2026-11-07', dayfirst=True)=}"
    "  (dayfirst even flips ISO dates: use fromisoformat)"
)
print(f"{date_parser.parse('Order placed on 7th Nov 2026 at 9:30', fuzzy=True)=}")
print(raised_message(lambda: date_parser.parse("next Diwali")))

  '7 November 2026 3pm'              -> 2026-11-07T15:00:00
  'Sat, 07 Nov 2026 09:30:00 GMT'    -> 2026-11-07T09:30:00+00:00
  'Nov 7 2026 15:00 IST'             -> 2026-11-07T15:00:00+05:30
  '2026-11-07T15:00:00+05:30'        -> 2026-11-07T15:00:00+05:30
date_parser.parse('07/11/2026')=datetime.datetime(2026, 7, 11, 0, 0)  (default is month first!)
date_parser.parse('07/11/2026', dayfirst=True)=datetime.datetime(2026, 11, 7, 0, 0)
date_parser.parse('2026-11-07', dayfirst=True)=datetime.datetime(2026, 7, 11, 0, 0)  (dayfirst even flips ISO dates: use fromisoformat)
date_parser.parse('Order placed on 7th Nov 2026 at 9:30', fuzzy=True)=datetime.datetime(2026, 11, 7, 9, 30)
ParserError: Unknown string format: next Diwali


**`relativedelta`** does calendar arithmetic that `timedelta` cannot: "one month
later" (clamped to the end of shorter months), "last day of the month", ages and
subscription renewals. `rrule` generates recurring dates (every second Tuesday,
the last working day of each month).

In [443]:
from dateutil.relativedelta import FR, relativedelta
from dateutil.rrule import MONTHLY, rrule

print(f"{date(2026, 1, 31) + relativedelta(months=1)=}  (clamped to Feb 28)")
print(f"{date(2026, 11, 7) + relativedelta(months=+1, day=31)=}  (last day of next month)")
print(f"{relativedelta(date(2026, 11, 7), date(1990, 2, 14))=}")
print(f"{relativedelta(date(2026, 11, 7), date(1990, 2, 14)).years=} years old")
print(f"{date(2026, 11, 7) + relativedelta(weekday=FR(+1))=}  (next Friday)")
payroll = rrule(
    MONTHLY, count=3, bysetpos=-1, byweekday=(0, 1, 2, 3, 4), dtstart=datetime(2026, 11, 1)
)
print(f"last working day of the next 3 months: {[run.date().isoformat() for run in payroll]}")

date(2026, 1, 31) + relativedelta(months=1)=datetime.date(2026, 2, 28)  (clamped to Feb 28)
date(2026, 11, 7) + relativedelta(months=+1, day=31)=datetime.date(2026, 12, 31)  (last day of next month)
relativedelta(date(2026, 11, 7), date(1990, 2, 14))=relativedelta(years=+36, months=+8, days=+24)
relativedelta(date(2026, 11, 7), date(1990, 2, 14)).years=36 years old
date(2026, 11, 7) + relativedelta(weekday=FR(+1))=datetime.date(2026, 11, 13)  (next Friday)
last working day of the next 3 months: ['2026-11-30', '2026-12-31', '2027-01-29']


**Predict the output.**

```python
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

start = date(2026, 1, 31)
print(start + relativedelta(months=1), start + timedelta(days=30))
```

<details><summary>Answer</summary>

`2026-02-28 2026-03-02`. `relativedelta(months=1)` moves to the same day of the next
month and clamps to its last day (February 2026 has 28 days). `timedelta` only
knows fixed durations: 30 days after 31 January is 2 March.

</details>

Run the next cell to check your prediction.

In [444]:
start = date(2026, 1, 31)
print(start + relativedelta(months=1), start + timedelta(days=30))

2026-02-28 2026-03-02


**`orjson`** is a JSON library written in Rust, often 5–10× faster than `json`. It
returns **bytes**, serializes `datetime`, `date`, `UUID`, dataclasses, enums and
NumPy arrays natively, and takes options as bit flags. It does not serialize
`Decimal` by default (pass `default=`). Use it when JSON is a hot path (APIs, log
pipelines); FastAPI can use it too.

In [445]:
import dataclasses
import uuid

import orjson


@dataclasses.dataclass
class Shipment:
    shipment_id: uuid.UUID
    dispatched_at: datetime
    weights_kg: list[float]
    declared_value: Decimal


shipment = Shipment(uuid.UUID(int=46), FIXED_MOMENT, [1.5, 0.25], Decimal("679.00"))
print(raised_message(lambda: orjson.dumps(shipment)))
encoded = orjson.dumps(shipment, default=str)
print(f"{encoded=}")
print(
    orjson.dumps(
        {"b": 1, "a": {"z": 2, 10: "non-str key"}},
        option=orjson.OPT_INDENT_2 | orjson.OPT_SORT_KEYS | orjson.OPT_NON_STR_KEYS,
    ).decode()
)
print(f"{orjson.loads(encoded)['dispatched_at']=}")

TypeError: Type is not JSON serializable: decimal.Decimal
encoded=b'{"shipment_id":"00000000-0000-0000-0000-00000000002e","dispatched_at":"2026-11-07T09:30:00+00:00","weights_kg":[1.5,0.25],"declared_value":"679.00"}'
{
  "a": {
    "10": "non-str key",
    "z": 2
  },
  "b": 1
}
orjson.loads(encoded)['dispatched_at']='2026-11-07T09:30:00+00:00'


A rough speed comparison on 2,000 orders (timings vary by machine; the ratio is the
point). **`msgspec`**, another fast library, validates while decoding into typed
`Struct` classes, like a much faster, stricter Pydantic for serialization. It is
not an Appendix A package, so its cell checks first.

In [446]:
import json
import timeit

big_payload = [dict(order, placed_at=FIXED_MOMENT.isoformat()) for order in raw_orders * 50]
stdlib_seconds = min(timeit.repeat(lambda: json.loads(json.dumps(big_payload)), number=5, repeat=3))
orjson_seconds = min(
    timeit.repeat(lambda: orjson.loads(orjson.dumps(big_payload)), number=5, repeat=3)
)
print(
    f"{len(big_payload)} records round trip: json {stdlib_seconds * 200:.1f} ms, "
    f"orjson {orjson_seconds * 200:.1f} ms, speed-up ~{stdlib_seconds / orjson_seconds:.0f}x"
)
if installed("msgspec", "msgspec (not an Appendix A package)"):
    import msgspec

    class Rate(msgspec.Struct):
        base: str
        rates: dict[str, float]

    print(msgspec.json.decode(b'{"base": "INR", "rates": {"USD": 0.012}}', type=Rate))

2000 records round trip: json 1.9 ms, orjson 0.8 ms, speed-up ~2x
SKIP: msgspec (not an Appendix A package) is not installed


### 4.6.6 Data analysis

**NumPy** provides the `ndarray`: a block of same-typed numbers with vectorized
operations implemented in C. Arithmetic applies element-wise without Python loops,
**broadcasting** stretches smaller arrays to match, boolean masks select elements,
and reductions (`sum`, `mean`, `max`) run at C speed. pandas, Polars, scikit-learn
and PyTorch all build on the same idea.

In [447]:
import numpy as np

quantities = np.array([2, 1, 3, 5, 1, 4])
unit_prices = np.array([240.0, 199.0, 350.5, 240.0, 899.0, 199.0])
line_totals = quantities * unit_prices  # element-wise
print(f"{line_totals=}")
print(f"{line_totals.sum()=}, {line_totals.mean()=:.2f}, {line_totals.max()=}")
print(f"{line_totals[line_totals > 700]=}  (boolean mask)")
with_gst = line_totals[:, np.newaxis] * np.array([1.05, 1.12, 1.18])  # broadcasting: 6x1 * 3 -> 6x3
print(f"{with_gst.shape=}\n{with_gst.round(2)[:2]}")
print(f"{np.arange(12).reshape(3, 4).sum(axis=0)=}, {quantities.dtype=}, {unit_prices.dtype=}")

line_totals=array([ 480. ,  199. , 1051.5, 1200. ,  899. ,  796. ])
line_totals.sum()=np.float64(4625.5), line_totals.mean()=770.92, line_totals.max()=np.float64(1200.0)
line_totals[line_totals > 700]=array([1051.5, 1200. ,  899. ,  796. ])  (boolean mask)
with_gst.shape=(6, 3)
[[504.   537.6  566.4 ]
 [208.95 222.88 234.82]]
np.arange(12).reshape(3, 4).sum(axis=0)=array([12, 15, 18, 21]), quantities.dtype=dtype('int64'), unit_prices.dtype=dtype('float64')


**Under the hood: why arrays are fast and small.** A Python list of floats is an
array of pointers to separate `float` objects (24 bytes each plus the pointer); a
NumPy array stores the raw 8-byte values contiguously. Seeded random numbers come
from a `Generator` (`np.random.default_rng(seed)`).

In [448]:
rng = np.random.default_rng(46)
daily_orders = rng.poisson(lam=40, size=1_000_000)
as_list = daily_orders.tolist()
list_bytes = sys.getsizeof(as_list) + sum(sys.getsizeof(value) for value in as_list[:1000]) * 1000
print(
    f"{daily_orders.nbytes / 1e6:.1f} MB as int64 array "
    f"vs ~{list_bytes / 1e6:.0f} MB as a list of ints"
)
print(f"{daily_orders[:8]=}, {daily_orders.mean()=:.3f}")
started = time.perf_counter()
numpy_total = int(daily_orders.sum())
numpy_ms = (time.perf_counter() - started) * 1000
started = time.perf_counter()
python_total = sum(as_list)
python_ms = (time.perf_counter() - started) * 1000
print(
    f"sum: numpy {numpy_ms:.2f} ms, Python {python_ms:.1f} ms, "
    f"equal: {numpy_total == python_total}  (timings vary)"
)

8.0 MB as int64 array vs ~36 MB as a list of ints


daily_orders[:8]=array([50, 35, 52, 42, 40, 49, 46, 44]), daily_orders.mean()=40.013
sum: numpy 0.23 ms, Python 2.4 ms, equal: True  (timings vary)


**pandas** is the long-established DataFrame library: labelled columns, an index,
rich time-series support and the widest ecosystem (plotting, Excel, SQL,
statistics). The usual cycle: read, filter, group-by, join, write. The data is the
seeded order list from the local API, saved as CSV.

In [449]:
import pandas as pd

orders_csv = UNIT46_DIR / "orders.csv"
pd.DataFrame([order for order in raw_orders if "city" in order]).to_csv(orders_csv, index=False)
orders = pd.read_csv(orders_csv, parse_dates=["placed_at"])
orders = orders[
    pd.to_numeric(orders["unit_price"], errors="coerce").notna() & (orders["quantity"] > 0)
].copy()
orders["unit_price"] = orders["unit_price"].astype(float)
orders["revenue"] = orders["quantity"] * orders["unit_price"]
print(orders[["order_id", "city", "sku", "quantity", "revenue"]].head(4).to_string(index=False))
by_city = (
    orders.groupby("city")
    .agg(orders=("order_id", "count"), units=("quantity", "sum"), revenue=("revenue", "sum"))
    .sort_values("revenue", ascending=False)
)
print(by_city.round(2))

 order_id            city      sku  quantity  revenue
        1 Hubli & Dharwad  JAR-750         1    350.5
        2          Mumbai KETTLE-9         5   4495.0
        3            Pune  JAR-750         3   1051.5
        4          Mumbai  JAR-750         5   1752.5
                 orders  units  revenue
city                                   
Mumbai               11     38  16335.0
Pune                 12     42  14045.0
Bengaluru             7     17   9257.5
Hubli & Dharwad       7     18   6878.0


A join with a product table, a filter on dates, and writing the result. Parquet
(a compressed, typed, columnar file format) needs the `pyarrow` engine in pandas,
which is not an Appendix A package, so that step checks first; CSV always works.

In [450]:
products = pd.DataFrame(
    {
        "sku": ["TEA-500", "MUG-02", "JAR-750", "KETTLE-9"],
        "category": ["tea", "kitchen", "pantry", "kitchen"],
    }
)
enriched = orders.merge(products, on="sku", how="left", validate="many_to_one")
first_week = enriched[enriched["placed_at"].dt.day <= 3]
print(first_week.groupby("category")["revenue"].sum().round(2).to_dict())
enriched.to_csv(UNIT46_DIR / "orders_enriched.csv", index=False)
if installed("pyarrow", "pyarrow (pandas' Parquet engine; not an Appendix A package)"):
    enriched.to_parquet(UNIT46_DIR / "orders_enriched_pandas.parquet")
    print("written with pandas + pyarrow")

{'kitchen': 11776.0, 'pantry': 2453.5, 'tea': 4320.0}
SKIP: pyarrow (pandas' Parquet engine; not an Appendix A package) is not installed


**Polars** is a newer DataFrame library written in Rust: multi-threaded, strict
about types, with an expression API (`pl.col(...)`) and a **lazy** mode that
optimizes the whole query before running it, like a database. It reads and writes
Parquet natively. The same analysis:

In [451]:
import polars as pl

pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_hide_dataframe_shape(True)
frame = pl.read_csv(orders_csv, try_parse_dates=True)
clean = (
    frame.with_columns(pl.col("unit_price").cast(pl.Float64, strict=False))
    .filter(pl.col("unit_price").is_not_null() & (pl.col("quantity") > 0))
    .with_columns((pl.col("quantity") * pl.col("unit_price")).alias("revenue"))
)
summary = (
    clean.group_by("city")
    .agg(
        pl.len().alias("orders"),
        pl.col("quantity").sum().alias("units"),
        pl.col("revenue").sum().round(2).alias("revenue"),
    )
    .sort("revenue", descending=True)
)
print(summary)

| city            | orders | units | revenue |
| ---             | ---    | ---   | ---     |
| str             | u32    | i64   | f64     |
|-----------------|--------|-------|---------|
| Mumbai          | 11     | 38    | 16335.0 |
| Pune            | 12     | 42    | 14045.0 |
| Bengaluru       | 7      | 17    | 9257.5  |
| Hubli & Dharwad | 7      | 18    | 6878.0  |


Lazy mode (`scan_csv` … `collect()`) builds a plan first; `explain()` shows that the
filter was pushed down into the CSV scan and only the needed columns are read.
Then a join and a Parquet round trip:

In [452]:
plan = (
    pl.scan_csv(orders_csv)
    .filter(pl.col("quantity") > 3)
    .group_by("sku")
    .agg(pl.col("quantity").sum())
    .sort("sku")
)
print("\n".join(plan.explain().splitlines()[-3:]))
print(plan.collect())
product_frame = pl.DataFrame(
    {"sku": products["sku"].tolist(), "category": products["category"].tolist()}
)
joined = clean.join(product_frame, on="sku", how="left")
parquet_path = UNIT46_DIR / "orders.parquet"
joined.write_parquet(parquet_path)
print(
    f"{parquet_path.stat().st_size} bytes of Parquet, {pl.read_parquet(parquet_path).shape=}, "
    f"columns: {pl.read_parquet(parquet_path).columns[-3:]}"
)

    PROJECT 2/7 COLUMNS
    SELECTION: col("quantity") > 3
    ESTIMATED ROWS: 36
| sku      | quantity |
| ---      | ---      |
| str      | i64      |
|----------|----------|
| JAR-750  | 15       |
| KETTLE-9 | 18       |
| MUG-02   | 32       |
| TEA-500  | 9        |
4124 bytes of Parquet, pl.read_parquet(parquet_path).shape=(37, 9), columns: ['placed_at', 'revenue', 'category']


**When to use which**

| Use | When |
| --- | --- |
| Plain Python / `csv` / `statistics` | a few thousand rows, simple aggregates, no dependency wanted |
| NumPy | numeric arrays, matrices, vectorized maths, the base for scientific code |
| pandas | interactive analysis, time series, the widest ecosystem (plotting, Excel, statsmodels, scikit-learn), existing code |
| Polars | larger-than-comfortable data, pipelines that must be fast and type-strict, lazy queries, multi-core machines |
| a database (SQLite, DuckDB, PostgreSQL) | data that outlives the process, concurrent writers, joins across many large tables |

### 4.6.7 Parsing and scraping

**BeautifulSoup** parses HTML (even broken HTML) into a tree you can search with
CSS selectors (`select`, `select_one`) or by tag and attributes (`find`,
`find_all`). Pass a parser explicitly: `"html.parser"` (built in) or `"lxml"`
(faster). The page comes from the local shop API.

In [453]:
from bs4 import BeautifulSoup

page = httpx.get(f"{SHOP_API}/products.html", timeout=5)
soup = BeautifulSoup(page.text, "lxml")
print(f"{soup.title.string=}")
for item in soup.select("ul#catalogue li.product"):
    link = item.find("a")
    price = Decimal(item.select_one(".price").get_text().lstrip("₹"))
    sold_out = "sold-out" in item["class"]
    print(
        f"  {item['data-sku']:<9} {link.get_text():<20} {price:>7} "
        f"{link['href']:<20} sold out: {sold_out}"
    )
print(f"{soup.select_one('p.note').get_text(strip=True)=}  (entities decoded)")

soup.title.string='Chai Corner - Products'


  TEA-500   Masala tea 500 g      240.00 /products/tea-500    sold out: False
  MUG-02    Steel mug             199.00 /products/mug-02     sold out: False
  JAR-750   Pickle jar 750 ml     350.50 /products/jar-750    sold out: True
soup.select_one('p.note').get_text(strip=True)='Prices include GST & free delivery above ₹500.'  (entities decoded)


**`lxml`** is a fast C-based parser for XML and HTML with full **XPath**. Use it
directly for XML (feeds, sitemaps, SOAP) or when BeautifulSoup is too slow.

In [454]:
from lxml import etree
from lxml import html as lxml_html

tree = lxml_html.fromstring(page.text)
print(f"{tree.xpath('//li[contains(@class, \"product\")]/@data-sku')=}")
print(f"{tree.xpath('//li[not(contains(@class, \"sold-out\"))]/a/text()')=}")
sitemap = b"""<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">
  <url><loc>https://chai.example.com/products/tea-500</loc><lastmod>2026-11-01</lastmod></url>
  <url><loc>https://chai.example.com/products/mug-02</loc><lastmod>2026-11-05</lastmod></url>
</urlset>"""
root = etree.fromstring(sitemap)
namespaces = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}
entries = [
    (
        url.findtext("sm:loc", namespaces=namespaces),
        url.findtext("sm:lastmod", namespaces=namespaces),
    )
    for url in root.xpath("//sm:url", namespaces=namespaces)
]
print(f"{entries=}")
print(
    "changed after 3 Nov: "
    f"{[location for location, modified in entries if modified > '2026-11-03']}"
)
print(
    f"{root.xpath('//sm:url[sm:lastmod > \"2026-11-03\"]', namespaces=namespaces)=}"
    "  (XPath 1.0 compares strings as numbers)"
)

tree.xpath('//li[contains(@class, \"product\")]/@data-sku')=['TEA-500', 'MUG-02', 'JAR-750']
tree.xpath('//li[not(contains(@class, \"sold-out\"))]/a/text()')=['Masala tea 500 g', 'Steel mug']
entries=[('https://chai.example.com/products/tea-500', '2026-11-01'), ('https://chai.example.com/products/mug-02', '2026-11-05')]
changed after 3 Nov: ['https://chai.example.com/products/mug-02']
root.xpath('//sm:url[sm:lastmod > \"2026-11-03\"]', namespaces=namespaces)=[]  (XPath 1.0 compares strings as numbers)


**Scraping politely and legally.** Prefer an official API. If you scrape: read the
site's terms; obey **`robots.txt`** (the standard library's `urllib.robotparser`
reads it); identify yourself with a descriptive `User-Agent`; **rate-limit**
(honour `Crawl-delay`, never hammer a server, back off on 429/503); cache pages you
already fetched; and never collect personal data you have no right to.

In [455]:
from urllib import robotparser

robots = robotparser.RobotFileParser()
robots.parse(httpx.get(f"{SHOP_API}/robots.txt", timeout=5).text.splitlines())
for agent, path in [
    ("chai-corner-bot", "/products.html"),
    ("chai-corner-bot", "/admin/orders"),
    ("BadBot", "/products.html"),
]:
    print(f"  {agent:<16} {path:<16} allowed: {robots.can_fetch(agent, f'{SHOP_API}{path}')}")
print(f"{robots.crawl_delay('chai-corner-bot')=} seconds")

  chai-corner-bot  /products.html   allowed: True
  chai-corner-bot  /admin/orders    allowed: False
  BadBot           /products.html   allowed: False
robots.crawl_delay('chai-corner-bot')=1 seconds


A minimal **rate limiter**: remember when the last request was sent and sleep until
the minimum interval has passed (`time.monotonic`, never the wall clock). The demo
uses 0.2 s instead of the site's 1 s crawl delay to stay quick; the elapsed time
shows the spacing was enforced.

In [456]:
class Throttle:
    """Allow at most one call per min_interval seconds."""

    def __init__(self, min_interval: float) -> None:
        self.min_interval = min_interval
        self.last_call = -float("inf")

    def wait(self) -> None:
        pause = self.last_call + self.min_interval - time.monotonic()
        if pause > 0:
            time.sleep(pause)
        self.last_call = time.monotonic()


throttle = Throttle(min_interval=0.2)
started = time.monotonic()
with httpx.Client(
    base_url=SHOP_API,
    headers={"User-Agent": "chai-corner-bot/1.0 (+mailto:dev@chai.example.com)"},
    timeout=5,
) as polite_client:
    for path in ("/products.html", "/rates", "/robots.txt", "/rates"):
        throttle.wait()
        polite_client.get(path).raise_for_status()
print(f"4 requests took {time.monotonic() - started:.1f} s (at least 3 x 0.2 s)")

4 requests took 0.6 s (at least 3 x 0.2 s)


### 4.6.8 Templating

**Jinja2** renders text from templates: `{{ expression }}` outputs a value,
`{% statement %}` runs control flow, `{# comment #}` is ignored. An `Environment`
holds the configuration: where templates come from (a `FileSystemLoader` in real
projects, a `DictLoader` here), **autoescaping** for HTML, and `StrictUndefined`
so a misspelt variable is an error instead of an empty string.

In [457]:
from jinja2 import DictLoader, Environment, StrictUndefined, select_autoescape


def inr(amount: object) -> str:
    """Custom filter: 1234.5 -> '₹1,234.50'."""
    return f"₹{Decimal(str(amount)):,.2f}"


TEMPLATES = {
    "base.html": (
        "<html><head><title>{% block title %}Chai Corner{% endblock %}</title></head>\n"
        "<body>\n{% block content %}{% endblock %}\n"
        "<footer>Generated {{ generated_at.strftime('%Y-%m-%d %H:%M %Z') }}</footer>\n"
        "</body></html>"
    ),
    "orders.html": (
        "{% extends 'base.html' %}\n"
        "{% block title %}Orders for {{ city }}{% endblock %}\n"
        "{% block content %}<h1>{{ city | upper }}: {{ orders | length }} orders</h1>\n<ul>\n"
        "{% for order in orders %}  <li class=\"{{ loop.cycle('odd', 'even') }}\">"
        "#{{ loop.index }} {{ order.customer }}: {{ order.total | inr }}"
        "{% if order.total > 1000 %} (big){% endif %}</li>\n"
        "{% else %}  <li>No orders</li>\n{% endfor %}</ul>\n"
        "<p>Tags: {{ tags | join(', ') | default('none', true) }}</p>{% endblock %}"
    ),
    "receipt.txt": (
        "Dear {{ customer }},\nThank you for order #{{ order_id }} ({{ total | inr }}).\n"
        "{% for line in lines %}- {{ line.quantity }} x {{ line.sku }}\n{% endfor %}"
    ),
}
template_env = Environment(
    loader=DictLoader(TEMPLATES), autoescape=select_autoescape(["html"]), undefined=StrictUndefined
)
template_env.filters["inr"] = inr
print(f"{sorted(template_env.list_templates())=}")

sorted(template_env.list_templates())=['base.html', 'orders.html', 'receipt.txt']


**Variables, loops, filters and inheritance** in one page: `orders.html` *extends*
`base.html` and fills its blocks; `loop.index` and `loop.cycle` come from the `for`
loop, `{% else %}` handles an empty list; `upper`, `length`, `join`, `default` are
built-in filters and `inr` is ours.

In [458]:
orders_for_page = [
    {"customer": "Asha", "total": Decimal("679.00")},
    {"customer": "Ravi", "total": Decimal("1798.50")},
]
print(
    template_env.get_template("orders.html").render(
        city="Pune", orders=orders_for_page, tags=["gift"], generated_at=FIXED_MOMENT
    )
)
print("---")
print(
    template_env.get_template("orders.html")
    .render(city="Mumbai", orders=[], tags=[], generated_at=FIXED_MOMENT)
    .split("<body>")[1]
)

<html><head><title>Orders for Pune</title></head>
<body>
<h1>PUNE: 2 orders</h1>
<ul>
  <li class="odd">#1 Asha: ₹679.00</li>
  <li class="even">#2 Ravi: ₹1,798.50 (big)</li>
</ul>
<p>Tags: gift</p>
<footer>Generated 2026-11-07 09:30 UTC</footer>
</body></html>
---

<h1>MUMBAI: 0 orders</h1>
<ul>
  <li>No orders</li>
</ul>
<p>Tags: none</p>
<footer>Generated 2026-11-07 09:30 UTC</footer>
</body></html>


**Autoescaping.** In HTML templates every `{{ value }}` is HTML-escaped, so data
such as a customer name containing `<script>` is displayed, not executed (the
defence against cross-site scripting). Mark a value `|safe` (or `Markup`) only when
you produced the HTML yourself. Text templates (e-mails, `.txt`) are not escaped:
`select_autoescape(["html"])` decides by file extension.

In [459]:
hostile = [{"customer": "<script>alert('hi')</script> & Co", "total": Decimal("10")}]
rendered = template_env.get_template("orders.html").render(
    city="Pune", orders=hostile, tags=["<b>vip</b>"], generated_at=FIXED_MOMENT
)
print([line for line in rendered.splitlines() if "<li" in line or "Tags" in line])
print(
    template_env.get_template("receipt.txt").render(
        customer="Asha & family",
        order_id=7,
        total=Decimal("679"),
        lines=[{"quantity": 2, "sku": "TEA-500"}],
    )
)
print(raised_message(lambda: template_env.get_template("receipt.txt").render(customer="Asha")))
print(
    f"{template_env.from_string('{{ name }}').render(name='<b>') = }"
    "  (strings are escaped too: select_autoescape's default_for_string)"
)

['  <li class="odd">#1 &lt;script&gt;alert(&#39;hi&#39;)&lt;/script&gt; &amp; Co: ₹10.00</li>', '<p>Tags: &lt;b&gt;vip&lt;/b&gt;</p>']
Dear Asha & family,
Thank you for order #7 (₹679.00).
- 2 x TEA-500

UndefinedError: 'order_id' is undefined
template_env.from_string('{{ name }}').render(name='<b>') = '&lt;b&gt;'  (strings are escaped too: select_autoescape's default_for_string)


**Predict the output.**

```python
from jinja2 import Environment

print(Environment().from_string("Hi {{ name }}").render(name="<b>Asha</b>"))
print(Environment(autoescape=True).from_string("Hi {{ name }}").render(name="<b>Asha</b>"))
```

<details><summary>Answer</summary>

`Hi <b>Asha</b>` then `Hi &lt;b&gt;Asha&lt;/b&gt;`. A bare `Environment()` does
**not** autoescape (Jinja2 is also used for non-HTML text), so templates rendering
HTML from user data are vulnerable unless you turn escaping on, ideally with
`select_autoescape(...)` as in the cells above.

</details>

Run the next cell to check your prediction.

In [460]:
print(Environment().from_string("Hi {{ name }}").render(name="<b>Asha</b>"))
print(Environment(autoescape=True).from_string("Hi {{ name }}").render(name="<b>Asha</b>"))

Hi <b>Asha</b>
Hi &lt;b&gt;Asha&lt;/b&gt;


### 4.6.9 Files, images and cloud storage

**Pillow** (the maintained fork of PIL) opens, edits and saves images. Typical
jobs: product thumbnails, converting uploads to a standard format, stripping or
honouring EXIF orientation (`ImageOps.exif_transpose`). `resize` sets an exact size;
`thumbnail` shrinks **in place** keeping the aspect ratio. The image is drawn here,
so the demo needs no files.

In [461]:
from PIL import Image, ImageDraw, features

banner = Image.new("RGB", (1200, 600), (250, 240, 225))
draw = ImageDraw.Draw(banner)
for stripe in range(0, 1200, 100):
    draw.rectangle([stripe, 0, stripe + 50, 600], fill=(230, 120, 30))
draw.ellipse([450, 150, 750, 450], fill=(120, 60, 20))
images_dir = UNIT46_DIR / "images"
images_dir.mkdir(exist_ok=True)
banner.save(images_dir / "banner.png")
with Image.open(images_dir / "banner.png") as opened:
    print(f"{opened.format=}, {opened.size=}, {opened.mode=}")
resized = banner.resize((600, 300))
thumbnail = banner.copy()
thumbnail.thumbnail((256, 256))  # in place, keeps 2:1
print(f"{resized.size=}, {thumbnail.size=}")

opened.format='PNG', opened.size=(1200, 600), opened.mode='RGB'
resized.size=(600, 300), thumbnail.size=(256, 128)


**Formats.** The extension (or `format=`) picks the encoder. JPEG is lossy and has
no alpha channel (convert `RGBA` to `RGB` first); PNG is lossless; WebP does both.
File sizes below depend on the Pillow and codec versions.

In [462]:
thumbnail.save(images_dir / "thumb.png", optimize=True)
thumbnail.convert("RGB").save(images_dir / "thumb.jpg", quality=85)
if features.check("webp"):
    thumbnail.save(images_dir / "thumb.webp", quality=80)
for path in sorted(images_dir.glob("thumb.*")):
    with Image.open(path) as image:
        print(f"  {path.name:<11} {image.format:<5} {image.size} {path.stat().st_size:>6} bytes")
transparent = Image.new("RGBA", (10, 10), (0, 0, 0, 0))
print(raised_message(lambda: transparent.save(io.BytesIO(), format="JPEG")))

  thumb.jpg   JPEG  (256, 128)   7407 bytes
  thumb.png   PNG   (256, 128)   2579 bytes
  thumb.webp  WEBP  (256, 128)   1170 bytes
OSError: cannot write mode RGBA as JPEG


**`boto3`** is the AWS SDK; S3-compatible services (MinIO, Cloudflare R2,
DigitalOcean Spaces) work with the same client and an `endpoint_url`. The three
everyday operations: **upload** (`put_object`, or `upload_file` for large files
with multipart), **download** (`get_object`, `download_file`) and **presigned
URLs**, which let a browser upload or download one object directly for a limited
time without your credentials.

No real bucket is used. A presigned URL is computed locally (no network). Upload
and download go through botocore's **`Stubber`**, which checks the request
parameters and returns canned responses: the standard way to unit-test AWS code.
Fake credentials, and empty config files so nothing is read from `~/.aws`.

In [463]:
import urllib.parse

import boto3
from botocore.config import Config

aws_isolation = {"AWS_CONFIG_FILE": os.devnull, "AWS_SHARED_CREDENTIALS_FILE": os.devnull}
with mock.patch.dict(os.environ, aws_isolation):
    s3 = boto3.client(
        "s3",
        region_name="ap-south-1",
        aws_access_key_id="AKIAEXAMPLEEXAMPLE",
        aws_secret_access_key="example-secret-not-real",
        config=Config(signature_version="s3v4"),
    )
download_url = s3.generate_presigned_url(
    "get_object", Params={"Bucket": "chai-reports", "Key": "2026/11/report.html"}, ExpiresIn=900
)
parts = urllib.parse.urlsplit(download_url)
query = urllib.parse.parse_qs(parts.query)
print(f"{parts.netloc=}, {parts.path=}")
print(f"query keys: {sorted(query)}")
print(
    f"{query['X-Amz-Expires']=}, {query['X-Amz-Algorithm']=}  (date and signature change with time)"
)

parts.netloc='chai-reports.s3.amazonaws.com', parts.path='/2026/11/report.html'
query keys: ['X-Amz-Algorithm', 'X-Amz-Credential', 'X-Amz-Date', 'X-Amz-Expires', 'X-Amz-Signature', 'X-Amz-SignedHeaders']
query['X-Amz-Expires']=['900'], query['X-Amz-Algorithm']=['AWS4-HMAC-SHA256']  (date and signature change with time)


Upload, list and download against the stub; `assert_no_pending_responses` proves every expected call happened:

In [464]:
from botocore.response import StreamingBody
from botocore.stub import ANY, Stubber

report_bytes = b"<html>November report</html>"
with Stubber(s3) as stubber:
    stubber.add_response(
        "put_object",
        {"ETag": '"abc123"'},
        {
            "Bucket": "chai-reports",
            "Key": "2026/11/report.html",
            "Body": ANY,
            "ContentType": "text/html",
        },
    )
    stubber.add_response(
        "list_objects_v2",
        {"KeyCount": 1, "Contents": [{"Key": "2026/11/report.html", "Size": len(report_bytes)}]},
        {"Bucket": "chai-reports", "Prefix": "2026/11/"},
    )
    stubber.add_response(
        "get_object",
        {
            "Body": StreamingBody(io.BytesIO(report_bytes), len(report_bytes)),
            "ContentLength": len(report_bytes),
        },
        {"Bucket": "chai-reports", "Key": "2026/11/report.html"},
    )

    upload = s3.put_object(
        Bucket="chai-reports", Key="2026/11/report.html", Body=report_bytes, ContentType="text/html"
    )
    listing = s3.list_objects_v2(Bucket="chai-reports", Prefix="2026/11/")
    downloaded = s3.get_object(Bucket="chai-reports", Key="2026/11/report.html")["Body"].read()
    stubber.assert_no_pending_responses()
print(f"{upload['ETag']=}, {[item['Key'] for item in listing['Contents']]=}, {downloaded=}")
with Stubber(s3) as stubber:
    stubber.add_client_error("get_object", service_error_code="NoSuchKey", http_status_code=404)
    print(raised_message(lambda: s3.get_object(Bucket="chai-reports", Key="missing.html")))

upload['ETag']='"abc123"', [item['Key'] for item in listing['Contents']]=['2026/11/report.html'], downloaded=b'<html>November report</html>'
NoSuchKey: An error occurred (NoSuchKey) when calling the GetObject operation: 


### 4.6.10 Developer utilities

**`loguru` vs stdlib `logging`.** loguru gives one ready-made `logger` with a
readable default format, `{}`-style messages, easy file sinks with rotation and
retention, context binding and exception catching, all without handlers, formatters
and `dictConfig` (Unit 4.3.9).

| | stdlib `logging` | `loguru` |
| --- | --- | --- |
| Setup | `getLogger(__name__)` + handlers/formatters/`dictConfig` | `logger.add(sink, format=..., level=...)` |
| Context | `extra=`, `LoggerAdapter`, filters | `logger.bind(order_id=7)`, `contextualize()` |
| Files | `RotatingFileHandler` | `logger.add("app.log", rotation="10 MB", retention="7 days")` |
| Ecosystem | every library logs through it | an extra dependency; intercept stdlib logs with a handler |
| Libraries | the right choice | do not use in a library you publish |

In [465]:
from loguru import logger as loguru_logger

loguru_logger.remove()  # drop the default stderr sink
sink_id = loguru_logger.add(
    sys.stdout,
    format="{level: <8} | {function}:{line} | {message} | {extra}",
    colorize=False,
    backtrace=False,
    diagnose=False,
)


def ship(order_id: int) -> None:
    log = loguru_logger.bind(order_id=order_id)
    log.info("packing {} items", 3)
    log.warning("courier slot missing")


ship(7)
try:
    Decimal("679") / 0
except ArithmeticError:
    loguru_logger.opt(exception=False).error("charging failed")
loguru_logger.remove(sink_id)

INFO     | ship:15 | packing 3 items | {'order_id': 7}


WARNING  | ship:16 | courier slot missing | {'order_id': 7}


ERROR    | <module>:23 | charging failed | {}


**`attrs`** is the library `dataclasses` was modelled on, and still does more:
validators and converters on fields, `__slots__` by default (`@define`), frozen
classes with `evolve`, and keyword-only or private-attribute handling. Choose
`dataclasses` for plain records, `attrs` when you need validation without
Pydantic's parsing.

In [466]:
import attrs


@attrs.define(frozen=True)
class StockLevel:
    sku: str = attrs.field(validator=attrs.validators.matches_re(r"[A-Z]{2,6}-\d{1,4}"))
    quantity: int = attrs.field(converter=int, validator=attrs.validators.ge(0))
    reorder_level: int = 5

    @property
    def is_low(self) -> bool:
        return self.quantity <= self.reorder_level


level = StockLevel("TEA-500", "12")  # converter: "12" -> 12
print(f"{level=}, {level.is_low=}")
print(f"{attrs.evolve(level, quantity=3).is_low=}, {attrs.asdict(level)=}")
try:
    StockLevel("tea", 1)
except ValueError as error:
    print(f"ValueError: {error.args[0]}")  # attrs adds the attribute and validator as extra args
print(raised_message(lambda: StockLevel("TEA-500", -1)))
print(f"{hasattr(level, '__dict__')=}  (slotted)")

level=StockLevel(sku='TEA-500', quantity=12, reorder_level=5), level.is_low=False
attrs.evolve(level, quantity=3).is_low=True, attrs.asdict(level)={'sku': 'TEA-500', 'quantity': 12, 'reorder_level': 5}
ValueError: 'sku' must match regex '[A-Z]{2,6}-\\d{1,4}' ('tea' doesn't)
ValueError: 'quantity' must be >= 0: -1
hasattr(level, '__dict__')=False  (slotted)


**`more-itertools`** collects the iteration recipes from the `itertools` docs and
many more, tested and named: batching, windows, partitioning, "exactly one"
checks, de-duplication preserving order.

In [467]:
import more_itertools as mit

skus = ["TEA-500", "MUG-02", "TEA-500", "JAR-750", "KETTLE-9", "MUG-02", "SPOON-1"]
print(f"{list(mit.chunked(skus, 3))=}")
print(f"{list(mit.windowed([42, 38, 51, 47], 2))=}")
print(f"{list(mit.unique_everseen(skus))=}")
cheap, expensive = mit.partition(lambda price: price >= 300, [240, 199, 350, 899, 60])
print(f"{list(cheap)=}, {list(expensive)=}")
print(f"{mit.first([], default='none')=}, {mit.one(['only'])=}")
print(raised_message(lambda: mit.one(["a", "b"])))
print(f"{list(mit.pairwise('ABC'))=}, {mit.ilen(x for x in range(10) if x % 3 == 0)=}")

list(mit.chunked(skus, 3))=[['TEA-500', 'MUG-02', 'TEA-500'], ['JAR-750', 'KETTLE-9', 'MUG-02'], ['SPOON-1']]
list(mit.windowed([42, 38, 51, 47], 2))=[(42, 38), (38, 51), (51, 47)]
list(mit.unique_everseen(skus))=['TEA-500', 'MUG-02', 'JAR-750', 'KETTLE-9', 'SPOON-1']
list(cheap)=[240, 199, 60], list(expensive)=[350, 899]
mit.first([], default='none')='none', mit.one(['only'])='only'
ValueError: Expected exactly one item in iterable, but got 'a', 'b', and perhaps more.
list(mit.pairwise('ABC'))=[('A', 'B'), ('B', 'C')], mit.ilen(x for x in range(10) if x % 3 == 0)=4


**`cachetools`** adds cache types the standard library lacks: `TTLCache` (entries
expire after `ttl` seconds), `LRUCache`, `LFUCache`, and a `@cached` decorator that
takes any of them (plus a `lock` for threads). `functools.lru_cache` has no
expiry, which is wrong for data that changes, such as exchange rates. A fake timer
makes expiry visible without waiting.

In [468]:
from cachetools import LRUCache, TTLCache, cached

fake_now = [0.0]
rate_lookups: list[str] = []


@cached(cache=TTLCache(maxsize=100, ttl=60, timer=lambda: fake_now[0]))
def exchange_rate(currency: str) -> float:
    rate_lookups.append(currency)
    return httpx.get(f"{SHOP_API}/rates", timeout=5).json()["rates"][currency]


print(f"{exchange_rate('USD')=}, {exchange_rate('USD')=}, lookups: {rate_lookups}")
fake_now[0] += 61  # a minute later: expired
print(f"{exchange_rate('USD')=}, lookups: {rate_lookups}")

recent = LRUCache(maxsize=2)
recent["TEA-500"], recent["MUG-02"] = 240, 199
recent["TEA-500"]  # touch: now most recent
recent["JAR-750"] = 350  # evicts the least recently used
print(f"{sorted(recent)=}")

exchange_rate('USD')=0.012, exchange_rate('USD')=0.012, lookups: ['USD']
exchange_rate('USD')=0.012, lookups: ['USD', 'USD']
sorted(recent)=['JAR-750', 'TEA-500']


### Lab 4.6

One exercise in six parts that together form a small reporting tool. The lab setup
cell imports what you need and makes sure the local shop API is running
(`LAB_API`). Until you finish, the checks print FAIL; that is expected.

#### Exercise 1 — a `typer` + `rich` CLI: `httpx` + `tenacity` → Pydantic → Polars → `rich` table → Jinja2 report

1. **`fetch_orders(client, path="/orders")`** GETs `path` with the given
   `httpx.Client` and returns the decoded JSON list. Decorate it with
   `tenacity.retry(...)` (the setup imports the module as `tenacity`):
   retry only transient failures (HTTP 5xx and `httpx.TransportError`, never 4xx),
   at most **4 attempts**, exponential backoff with tiny waits (e.g.
   `wait_exponential(multiplier=0.01, max=0.05)`), and `reraise=True`.
2. **`OrderRecord`**, a Pydantic model: `order_id` and `quantity` positive ints,
   `customer`, `city`, `sku` strings, `unit_price` a `Decimal` greater than 0,
   `placed_at` a timezone-aware datetime.
3. **`validate_orders(raw)`** returns `(valid_records, problems)`, where each problem
   is a string starting `"record {index}:"` (index in `raw`), one per invalid
   record.
4. **`summarize(records)`** returns a Polars DataFrame with exactly the columns
   `city`, `orders`, `units`, `revenue` (float, rounded to 2 places), one row per
   city, sorted by revenue (highest first).
5. **`build_table(summary)`** returns a `rich.table.Table` titled
   `"Revenue by city"` with columns `City`, `Orders`, `Units`, `Revenue (INR)` and one
   row per city (revenue formatted like `12,345.50`).
6. **`render_report(summary, generated_at)`** returns an HTML page rendered with
   Jinja2 **with autoescaping on**: a `<table>` with a header row and one `<tr>` per
   city, and the text `Generated 2026-11-07 09:30 UTC` (format
   `%Y-%m-%d %H:%M %Z`).
7. **`app`**, a `typer.Typer()` with one command taking `--base-url` (required),
   `--path` (default `/orders`) and `--output` (default `report.html`): fetch,
   validate, summarize, print the rich table, write the report, then print
   `"{valid} valid orders, {rejected} rejected; report written to {output name}"`.
   On an HTTP error print a line starting `error:` and exit with code 1.

In [469]:
import httpx
import polars as pl
import tenacity  # noqa: F401  (use tenacity.retry(...), tenacity.stop_after_attempt(...), ...)
import typer
from jinja2 import Environment, select_autoescape
from pydantic import AwareDatetime, BaseModel, Field, PositiveInt, ValidationError
from rich.console import Console
from rich.table import Table

LAB46_DIR = find_repo_root() / "notebooks" / "_scratch" / "unit46" / "lab"
LAB46_DIR.mkdir(parents=True, exist_ok=True)
LAB_API = start_shop_api()
print(f"lab API: {LAB_API}; lab files go to {LAB46_DIR.relative_to(find_repo_root()).as_posix()}/")

lab API: http://127.0.0.1:18046; lab files go to notebooks/_scratch/unit46/lab/


The stubs (the order follows the numbered parts above):

In [470]:
def fetch_orders(client: httpx.Client, path: str = "/orders") -> list[dict[str, object]]:
    """GET path and return the JSON list (retried by tenacity)."""
    raise NotImplementedError


class OrderRecord(BaseModel):
    """One validated order."""


def validate_orders(raw: list[dict[str, object]]) -> tuple[list[OrderRecord], list[str]]:
    """(valid records, one 'record {index}: ...' message per invalid record)."""
    raise NotImplementedError


def summarize(records: list[OrderRecord]) -> pl.DataFrame:
    """city, orders, units, revenue; sorted by revenue, highest first."""
    raise NotImplementedError


def build_table(summary: pl.DataFrame) -> Table:
    """A rich table titled 'Revenue by city'."""
    raise NotImplementedError


def render_report(summary: pl.DataFrame, generated_at: datetime) -> str:
    """An autoescaped Jinja2 HTML report."""
    raise NotImplementedError


app = typer.Typer(add_completion=False)

Check exercise 1 (it calls your code against the local API several times):

In [471]:
from typer.testing import CliRunner


def hits(key: str) -> int:
    return httpx.get(f"{LAB_API}/hits", params={"key": key}, timeout=5).json()["hits"]


def fetch_story(key: str, fail: int) -> tuple[str, int]:
    with httpx.Client(base_url=LAB_API, timeout=5) as client:
        client.get("/reset")
        try:
            outcome = str(len(fetch_orders(client, f"/orders?key={key}&fail={fail}")))
        except httpx.HTTPStatusError as error:
            outcome = f"HTTPStatusError {error.response.status_code}"
    return outcome, hits(key)


def not_found_story() -> str:
    with httpx.Client(base_url=LAB_API, timeout=5) as client:
        started = time.monotonic()
        try:
            fetch_orders(client, "/status/404")
        except httpx.HTTPStatusError as error:
            return f"{error.response.status_code} fast={time.monotonic() - started < 0.5}"
    return "no error"


LAB_RAW = demo_orders()
VALID_RAW = [record for index, record in enumerate(LAB_RAW) if index not in (6, 17, 29)]
EXPECTED_REVENUE = {}
for record in VALID_RAW:
    EXPECTED_REVENUE[record["city"]] = EXPECTED_REVENUE.get(record["city"], Decimal(0)) + record[
        "quantity"
    ] * Decimal(record["unit_price"])
EXPECTED_ORDER = sorted(EXPECTED_REVENUE, key=lambda city: -EXPECTED_REVENUE[city])


def lab_summary() -> pl.DataFrame:
    return summarize(validate_orders(LAB_RAW)[0])


def table_facts() -> tuple[object, ...]:
    table = build_table(lab_summary())
    return table.title, [str(column.header) for column in table.columns], table.row_count


def report_facts() -> tuple[object, ...]:
    html = render_report(lab_summary(), datetime(2026, 11, 7, 9, 30, tzinfo=UTC))
    return (
        html.count("<tr"),
        "Hubli &amp; Dharwad" in html,
        "Hubli & Dharwad" in html,
        "Generated 2026-11-07 09:30 UTC" in html,
    )


def cli_story(path: str) -> tuple[int, bool, bool, bool]:
    output = LAB46_DIR / "report.html"
    output.unlink(missing_ok=True)
    httpx.get(f"{LAB_API}/reset", timeout=5)
    result = CliRunner().invoke(
        app,
        ["--base-url", LAB_API, "--path", path, "--output", str(output)],
        env={"COLUMNS": "100", "NO_COLOR": "1"},
    )
    text = result.output
    return (
        result.exit_code,
        "Revenue by city" in text,
        "37 valid orders, 3 rejected; report written to report.html" in text,
        output.exists(),
    )


run_checks(
    [
        ("fetch: 2 failures then success", lambda: fetch_story("lab-ok", 2), ("40", 3)),
        (
            "fetch: gives up after 4 attempts",
            lambda: fetch_story("lab-down", 9),
            ("HTTPStatusError 503", 4),
        ),
        ("fetch: 404 is not retried", not_found_story, "404 fast=True"),
        ("validate: 37 valid", lambda: len(validate_orders(LAB_RAW)[0]), 37),
        (
            "validate: problems name records 6, 17, 29",
            lambda: [problem.split(":")[0] for problem in validate_orders(LAB_RAW)[1]],
            ["record 6", "record 17", "record 29"],
        ),
        (
            "OrderRecord types",
            lambda: (
                type(validate_orders(LAB_RAW)[0][0].unit_price).__name__,
                validate_orders(LAB_RAW)[0][0].placed_at.utcoffset() is not None,
            ),
            ("Decimal", True),
        ),
        ("summary columns", lambda: lab_summary().columns, ["city", "orders", "units", "revenue"]),
        (
            "summary order and revenue",
            lambda: [(row["city"], row["revenue"]) for row in lab_summary().iter_rows(named=True)],
            [(city, float(round(EXPECTED_REVENUE[city], 2))) for city in EXPECTED_ORDER],
        ),
        (
            "summary counts",
            lambda: (lab_summary()["orders"].sum(), lab_summary()["units"].sum()),
            (37, sum(record["quantity"] for record in VALID_RAW)),
        ),
        (
            "rich table",
            table_facts,
            ("Revenue by city", ["City", "Orders", "Units", "Revenue (INR)"], len(EXPECTED_ORDER)),
        ),
        (
            "report: rows, escaping, timestamp",
            report_facts,
            (len(EXPECTED_ORDER) + 1, True, False, True),
        ),
        ("CLI success", lambda: cli_story("/orders"), (0, True, True, True)),
        ("CLI HTTP error -> exit 1", lambda: cli_story("/status/404")[0], 1),
    ]
)

FAIL  fetch: 2 failures then success: raised NotImplementedError: 
FAIL  fetch: gives up after 4 attempts: raised NotImplementedError: 
FAIL  fetch: 404 is not retried: raised NotImplementedError: 
FAIL  validate: 37 valid: raised NotImplementedError: 
FAIL  validate: problems name records 6, 17, 29: raised NotImplementedError: 
FAIL  OrderRecord types: raised NotImplementedError: 
FAIL  summary columns: raised NotImplementedError: 
FAIL  summary order and revenue: raised NotImplementedError: 
FAIL  summary counts: raised NotImplementedError: 
FAIL  rich table: raised NotImplementedError: 
FAIL  report: rows, escaping, timestamp: raised NotImplementedError: 


FAIL  CLI success: raised RuntimeError: Could not get a command for this Typer instance
FAIL  CLI HTTP error -> exit 1: raised RuntimeError: Could not get a command for this Typer instance
0/13 checks passed


### Recap — Unit 4.6

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Choosing a library | Check maintenance, usage, `py.typed`, licence, advisories, dependency weight; read the source | npm trends, Snyk |
| HTTP | `requests` or `httpx`; always a timeout and `raise_for_status()`; one shared client/session; `AsyncClient` for concurrency; `MockTransport` for tests | `axios`, `fetch` |
| Resilience | `tenacity`: retry only transient errors, exponential backoff + jitter, a stop condition, `reraise=True` | `p-retry` |
| Validation | Pydantic models for API data and config; per-record validation for imports; `pydantic-settings` + `.env` for settings | `zod`, `dotenv` |
| CLIs | `typer` (hints) on top of `click`; `rich` tables, progress, logging, tracebacks | `commander`, `chalk` |
| Dates, JSON | `dateutil.parser`, `relativedelta`, `rrule`; `orjson` for speed (bytes; `default=` for Decimal) | `date-fns`, `fast-json-stringify` |
| Data | NumPy arrays; pandas for ecosystem and exploration; Polars for speed and lazy queries; Parquet for storage | `danfo.js` |
| Scraping | BeautifulSoup/lxml; obey robots.txt, identify yourself, rate-limit | `cheerio` |
| Templating | Jinja2 with autoescaping and `StrictUndefined`; inheritance and filters | `nunjucks` |
| Images, S3 | Pillow `thumbnail`/formats; `boto3` put/get/presigned URLs, tested with `Stubber` | `sharp`, AWS SDK |
| Utilities | `loguru` for apps (not libraries), `attrs`, `more-itertools`, `cachetools` TTL caches | `pino`, `lodash`, `lru-cache` |

**Further reading**

- [requests](https://requests.readthedocs.io/), [HTTPX](https://www.python-httpx.org/),
  [Tenacity](https://tenacity.readthedocs.io/)
- [Pydantic](https://docs.pydantic.dev/), [pydantic-settings](https://docs.pydantic.dev/latest/concepts/pydantic_settings/),
  [python-dotenv](https://saurabh-kumar.com/python-dotenv/)
- [Typer](https://typer.tiangolo.com/), [Click](https://click.palletsprojects.com/), [Rich](https://rich.readthedocs.io/)
- [dateutil](https://dateutil.readthedocs.io/), [orjson](https://github.com/ijl/orjson), [msgspec](https://jcristharif.com/msgspec/)
- [NumPy](https://numpy.org/doc/stable/user/absolute_beginners.html), [pandas](https://pandas.pydata.org/docs/user_guide/10min.html),
  [Polars](https://docs.pola.rs/user-guide/)
- [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/), [lxml](https://lxml.de/),
  [`urllib.robotparser`](https://docs.python.org/3/library/urllib.robotparser.html)
- [Jinja](https://jinja.palletsprojects.com/), [Pillow](https://pillow.readthedocs.io/),
  [Boto3 S3 guide](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/s3.html),
  [botocore Stubber](https://botocore.amazonaws.com/v1/documentation/api/latest/reference/stubber.html)
- [loguru](https://loguru.readthedocs.io/), [attrs](https://www.attrs.org/),
  [more-itertools](https://more-itertools.readthedocs.io/), [cachetools](https://cachetools.readthedocs.io/)
- [PEP 561](https://peps.python.org/pep-0561/) (`py.typed`), [PEP 517](https://peps.python.org/pep-0517/)

## Phase 4 Project — Publish the Log Analyzer

The Phase 1 Project built a log analyzer inside a notebook. This project turns the
same code into a **package**: something anyone can install with one command, that
checks its own types, tests itself on four Python versions, and publishes itself to
TestPyPI when you push a release tag.

| Requirement (syllabus) | Where it lives | Unit |
| --- | --- | --- |
| Turn the Phase 1 log analyzer into a package published to TestPyPI | `projects/log-analyzer/`, distribution `log-analyzer-psbcg433` | 4.1 |
| `src/` layout | `projects/log-analyzer/src/log_analyzer/` | 4.1.7 |
| `mypy --strict` clean | `[tool.mypy] strict = true`, `py.typed` | 4.2 |
| pytest + Hypothesis at 90%+ coverage | `tests/`, `--cov-fail-under=90` | 4.3 |
| `dictConfig` logging | `log_analyzer/logging_config.py` | 4.3.9 |
| An `argparse` CLI with subcommands | `log_analyzer/cli.py` | 4.4.8 |
| `.gz` input support | `log_analyzer/parsing.py` (`open_log`) | 4.5.8 |
| CSV / JSON / SQLite export | `log_analyzer/export.py` | 4.5.6, 4.5.9 |
| A GitHub Actions workflow with trusted publishing | `.github/workflows/log-analyzer.yml` | 4.1.8, 4.3.10 |

```mermaid
graph LR
    A[push or pull request] --> T[test: ruff, mypy --strict, pytest on 3.12-3.15]
    G[push tag log-analyzer-v0.1.0] --> T
    T --> B[build: tag = version?, uv build, smoke test]
    B -->|tag or manual run only| P[publish: OIDC token, upload to TestPyPI]
```

<details><summary>ASCII fallback</summary>

```text
push / pull request ----+
                        +--> test (ruff, mypy --strict, pytest x 4 Pythons)
push tag log-analyzer-v* +          |
                                   v
                        build (tag = version?, uv build, smoke test)
                                   |  only for a tag or a manual run
                                   v
                        publish (OIDC token -> TestPyPI)
```

</details>

**How this section works**

- The package has its **own** `pyproject.toml`, `uv.lock` and `.venv`, separate from
  the course environment. Step 3 creates that environment; the CLI and the dev tools
  run from it in child processes, and the cells print their real output.
- CLI demos write their files to `notebooks/_scratch/phase4_project/`. Paths are shown
  as `<scratch>`, `<package>` (`projects/log-analyzer`), `<repo>` or `~`, timings as
  `<time>`.
- **Nothing is published from this notebook.** Publishing happens on GitHub when you
  push a tag; *How to publish* at the end lists the steps you do yourself.

The setup cell defines the paths and puts `projects/log-analyzer/src` on `sys.path`,
so the kernel can import the package for the code tour (the kernel's environment
does not have the package *installed*; Step 4 shows what that changes).

In [472]:
import argparse
import csv
import datetime as dt
import gzip
import inspect
import json
import os
import re
import shlex
import shutil
import sqlite3
import subprocess
import sys
import tarfile
import textwrap
import tomllib
import zipfile
from collections import Counter
from contextlib import closing
from pathlib import Path

REPO_ROOT = find_repo_root()
PACKAGE_DIR = REPO_ROOT / "projects" / "log-analyzer"  # the package, with its own pyproject.toml
PACKAGE_BIN = PACKAGE_DIR / ".venv" / "bin"  # the package's own environment (Step 3)
WORKFLOW_PATH = REPO_ROOT / ".github" / "workflows" / "log-analyzer.yml"
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
PROJECT_DIR = scratch_dir / "phase4_project"  # where the CLI demos write their files
shutil.rmtree(PROJECT_DIR, ignore_errors=True)
PROJECT_DIR.mkdir(parents=True)
UV = shutil.which("uv")
if str(PACKAGE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR / "src"))  # so the kernel can import log_analyzer
print(f"{sys.version_info[:2]=}, uv found: {UV is not None}")

sys.version_info[:2]=(3, 15), uv found: True


Two helpers in the style of Units 4.1 and 4.3: `run` executes a command in a child
process with a clean environment (not the course venv, no colours, unbuffered so
stderr and stdout keep their order, `HYPOTHESIS_PROFILE=ci`) and returns its exit
code and tidied output. `log-analyzer` and `python` resolve to the package's own
`.venv/bin/`.

In [473]:
TIMING = re.compile(r"\b(in|after) \d+(?:\.\d+)? ?(?:ms|s|µs)\b")
LOG_STAMP = re.compile(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}[+-]\d{4}")
ANSI_ESCAPE = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")  # mypy colours its output even when piped
SKIPPED_NAMES = {".venv", "__pycache__", "dist", ".pytest_cache", ".mypy_cache", ".ruff_cache",
                 ".hypothesis", ".coverage"}  # fmt: skip


def package_env() -> dict[str, str]:
    """Child environment: not the course venv, no colours, unbuffered, reproducible Hypothesis."""
    dropped = ("UV_", "VIRTUAL_ENV", "PYTHONPATH", "PYTEST_", "COV", "HYPOTHESIS")
    inherited = {key: value for key, value in os.environ.items() if not key.startswith(dropped)}
    return {**inherited, "NO_COLOR": "1", "PY_COLORS": "0", "TERM": "dumb", "COLUMNS": "100",
            "UV_NO_PROGRESS": "1", "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
            "HYPOTHESIS_PROFILE": "ci"}  # fmt: skip


def tidy(text: str) -> str:
    """Show paths as <scratch>, <package>, <repo> or ~; timings and log times as placeholders."""
    for folder, label in ((PROJECT_DIR, "<scratch>"), (PACKAGE_DIR, "<package>"),
                          (REPO_ROOT, "<repo>"), (Path.home(), "~")):  # fmt: skip
        text = text.replace(str(folder), label)
    return ANSI_ESCAPE.sub("", LOG_STAMP.sub("<timestamp>", TIMING.sub(r"\1 <time>", text)))


def run(*command: str, cwd: Path = PROJECT_DIR) -> tuple[int, str]:
    """Run a command in a child process; return (exit code, tidied stdout + stderr in order)."""
    name = command[0]
    known = {"log-analyzer": PACKAGE_BIN / "log-analyzer", "python": PACKAGE_BIN / "python"}
    executable = Path(UV) if name == "uv" and UV else known.get(name, Path(name))
    if not executable.exists():
        return 127, f"SKIP: {name} is not available (needs uv and the Step 3 environment)"
    completed = subprocess.run(
        [str(executable), *command[1:]], cwd=cwd, env=package_env(), text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=600,
    )  # fmt: skip
    return completed.returncode, tidy(completed.stdout.rstrip())

`shell` prints a command and its output like a terminal session; `show_tree` and
`show_lines` display files.

In [474]:
def shell(*command: str, cwd: Path = PROJECT_DIR, keep: str | None = None,
          max_lines: int | None = None) -> None:  # fmt: skip
    """Print `[folder]$ command` and the real output (only lines matching keep, if given)."""
    exit_code, output = run(*command, cwd=cwd)
    print(f"[{tidy(str(cwd))}]$ {tidy(shlex.join(command))}")
    lines = output.splitlines()
    if keep is not None:
        lines = [line for line in lines if re.search(keep, line)]
    if max_lines is not None and len(lines) > max_lines:
        lines = [*lines[:max_lines], f"... ({len(lines) - max_lines} more lines)"]
    if lines:
        print(textwrap.indent("\n".join(lines), "  ", lambda line: True))
    if exit_code:
        print(f"  (exit code {exit_code})")


def show_tree(root: Path, prefix: str = "") -> None:
    """Print a folder tree, skipping environments, caches and build output."""
    entries = sorted((entry for entry in root.iterdir() if entry.name not in SKIPPED_NAMES),
                     key=lambda entry: (entry.is_file(), entry.name))  # fmt: skip
    for index, entry in enumerate(entries):
        last = index == len(entries) - 1
        print(f"{prefix}{'└── ' if last else '├── '}{entry.name}{'/' if entry.is_dir() else ''}")
        if entry.is_dir():
            show_tree(entry, prefix + ("    " if last else "│   "))


def show_lines(path: Path, first: int | None = None) -> None:
    """Print a text file (or its first lines), paths tidied."""
    lines = path.read_text(encoding="utf-8").splitlines()
    print("\n".join(tidy(line) for line in lines[:first]))

### Step 1 — Packaging decisions

Every line of the package's configuration is a decision. These are the ones that
matter, with the unit that taught the trade-off:

| Decision | Choice | Why | Unit |
| --- | --- | --- | --- |
| Names | distribution `log-analyzer-psbcg433`, import `log_analyzer`, command `log-analyzer` | Index names are global and first-come; a GitHub-username suffix makes the name unique on TestPyPI. The import name stays short. The index normalizes `-`, `_` and `.` (PEP 503) | 4.1.4 |
| Layout | `src/` | Tests import the *installed* package, never the folder next to them, so a file missing from the wheel fails the tests | 4.1.7 |
| Build backend | `uv_build` (pinned `<0.14`) | The backend Unit 4.1 used; zero configuration for a pure-Python `src/` layout | 4.1.8 |
| Runtime dependencies | none | `gzip`, `csv`, `json`, `sqlite3`, `argparse`, `logging.config` are standard library: nothing to audit, nothing to break | 4.1.9 |
| Python versions | `requires-python = ">=3.12"`, CI on 3.12–3.15 | Nothing needs 3.15; 3.12 brings the `type` statement used for aliases. A wider range means more users can install it | 4.1.5 |
| Version | `version = "0.1.0"` only in `pyproject.toml`; `__version__` read with `importlib.metadata` | One source of truth; CI refuses a tag that does not match it | 4.1.4 |
| Typing | `py.typed`, `mypy --strict` on `src` and `tests` | Users' type checkers read the package's hints (PEP 561) | 4.2.12 |
| Environment | own `pyproject.toml` + `uv.lock` | The package does not inherit the course's 160 packages; CI installs exactly the lock | 4.1.6 |
| Logging | library: `getLogger(__name__)` + `NullHandler`; CLI: `dictConfig` to stderr | Libraries never configure logging; stdout stays clean for pipes | 4.3.9 |
| Exit codes | 0 success, 1 malformed lines (`validate`), 2 error | Scripts and CI can branch on them; argparse already uses 2 for usage errors | 4.4.8 |
| sdist contents | `src/`, `tests/`, README | Anyone building from source (a Linux distribution, say) can run the tests | 4.1.8 |
| Publishing | trusted publishing (OIDC) to TestPyPI, `testpypi` environment | No long-lived API token stored in GitHub | 4.1.8 |
| Licence | none declared yet | Choosing a licence is the author's decision; add `license = "..."` (PEP 639) before a real PyPI release | 4.1.9 |

The package folder. The modules are the Phase 1 project's steps, split by
responsibility; the tests mirror them:

In [475]:
print("projects/log-analyzer/")
show_tree(PACKAGE_DIR)

projects/log-analyzer/
├── src/
│   └── log_analyzer/
│       ├── __init__.py
│       ├── __main__.py
│       ├── analysis.py
│       ├── cli.py
│       ├── export.py
│       ├── generate.py
│       ├── logging_config.py
│       ├── models.py
│       ├── parsing.py
│       ├── py.typed
│       └── report.py
├── tests/
│   ├── data/
│   │   └── phase1_report_seed42_20000.txt
│   ├── __init__.py
│   ├── conftest.py
│   ├── samples.py
│   ├── test_analysis.py
│   ├── test_cli.py
│   ├── test_export.py
│   ├── test_generate.py
│   ├── test_logging_config.py
│   ├── test_parsing.py
│   ├── test_properties.py
│   └── test_report.py
├── .gitignore
├── README.md
├── pyproject.toml
└── uv.lock


| Module | Responsibility | Phase 1 Project step |
| --- | --- | --- |
| `models.py` | `AccessRecord`, `AccessSummary`, the exception hierarchy | Steps 3–5 |
| `parsing.py` | regex + `match` parser, lazy `read_lines`, **gzip detection** | Steps 2–4 |
| `analysis.py` | `aggregate`, `percentile`, `top_endpoints`, strict mode | Steps 5, 7 |
| `report.py` | 60-column text report, JSON document (`TypedDict`) | Step 6 |
| `export.py` | streaming CSV / JSON / SQLite writers | new |
| `generate.py` | seeded synthetic logs, reproducible `.gz` output | Step 1 |
| `logging_config.py` | the `dictConfig` dictionary | new (replaces `@timer` / `Timer` prints) |
| `cli.py` | `argparse` subcommands, exit codes | new |
| `__init__.py` / `__main__.py` | public API, `__version__`, `python -m log_analyzer` | new |

The algorithms are unchanged, so Step 5 can prove that the package prints exactly
the report the Phase 1 notebook printed.

### Step 2 — `pyproject.toml`

One file holds the metadata (`[project]`), the console script
(`[project.scripts]`), the dev tools (`[dependency-groups]`, PEP 735: installed for
development, never shipped), the build backend (`[build-system]`) and every tool's
configuration (`[tool.*]`). Note `addopts`: plain `pytest` always measures coverage
and **fails under 90%**, locally and in CI alike.

In [476]:
show_lines(PACKAGE_DIR / "pyproject.toml")

[project]
name = "log-analyzer-psbcg433"
version = "0.1.0"
description = "Analyze web-server access logs (plain or gzip) and export them to CSV, JSON or SQLite."
readme = "README.md"
requires-python = ">=3.12"
authors = [{ name = "psbcg433" }]
keywords = ["access-log", "log-analysis", "cli", "sqlite", "csv"]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Environment :: Console",
    "Intended Audience :: Developers",
    "Intended Audience :: System Administrators",
    "Operating System :: OS Independent",
    "Programming Language :: Python :: 3",
    "Programming Language :: Python :: 3 :: Only",
    "Programming Language :: Python :: 3.12",
    "Programming Language :: Python :: 3.13",
    "Programming Language :: Python :: 3.14",
    "Programming Language :: Python :: 3.15",
    "Topic :: System :: Logging",
    "Typing :: Typed",
]
dependencies = []

[project.urls]
Source = "https://github.com/psbcg433/python-atlas/tree/main/projects/log-analyzer"
Issues = "https://g

**Under the hood.** Tools read this file with `tomllib` (standard library since
3.11), exactly as the next cell does. `dependencies = []` is what makes the wheel
installable offline from a single file.

In [477]:
config = tomllib.loads((PACKAGE_DIR / "pyproject.toml").read_text(encoding="utf-8"))
project = config["project"]
print(f"{project['name']=}, {project['version']=}")
print(f"{project['requires-python']=}, {project['dependencies']=}")
print(f"{project['scripts']=}")
print(f"{config['build-system']['build-backend']=}")
print(f"dev tools: {[spec.split('>=')[0] for spec in config['dependency-groups']['dev']]}")
print(f"{config['tool']['mypy']=}")

project['name']='log-analyzer-psbcg433', project['version']='0.1.0'
project['requires-python']='>=3.12', project['dependencies']=[]
project['scripts']={'log-analyzer': 'log_analyzer.cli:main'}
config['build-system']['build-backend']='uv_build'
dev tools: ['hypothesis', 'mypy', 'pytest', 'pytest-cov', 'ruff']
config['tool']['mypy']={'strict': True, 'files': ['src', 'tests'], 'warn_unreachable': True}


### Step 3 — The package's own environment

uv finds the project by walking up from the working directory to the nearest
`pyproject.toml`. Run inside `projects/log-analyzer/`, it therefore manages the
*package* (its own `uv.lock` and `.venv`), not the course. `uv sync --locked`
installs the package in editable mode plus the `dev` group, and **fails** instead of
silently re-locking if `uv.lock` no longer matches `pyproject.toml`; CI runs the
same command. (`.venv/` is git-ignored at every depth by the repository's
`.gitignore`.)

The first run lists what it installed, later runs only audit, so the cell prints just
the exit code, then the dependency tree, which comes from the lockfile and is the
same every time.

In [478]:
sync_exit_code, _ = run("uv", "sync", "--locked", cwd=PACKAGE_DIR)
print(f"uv sync --locked -> exit code {sync_exit_code}")
shell("uv", "tree", "--locked", "--depth", "1", cwd=PACKAGE_DIR)

uv sync --locked -> exit code 0
[<package>]$ uv tree --locked --depth 1
  Resolved 18 packages in <time>
  log-analyzer-psbcg433 v0.1.0
  ├── hypothesis v6.170.15 (group: dev)
  ├── mypy v2.4.1 (group: dev)
  ├── pytest v9.2.0 (group: dev)
  ├── pytest-cov v7.1.0 (group: dev)
  └── ruff v0.17.4 (group: dev)


**Under the hood: an editable install.** The environment holds no copy of the code.
The package's `.dist-info` folder records where the source lives
(`direct_url.json`, PEP 610) and which console scripts to create
(`entry_points.txt`); `.venv/bin/log-analyzer` is a tiny generated script that
imports `log_analyzer.cli` and calls `main()`.

In [479]:
site_packages = next((PACKAGE_DIR / ".venv" / "lib").glob("python3.*/site-packages"))
dist_info = next(site_packages.glob("log_analyzer_psbcg433-*.dist-info"))
print(dist_info.name)
print("direct_url.json:", tidy((dist_info / "direct_url.json").read_text()))
print("entry_points.txt:", (dist_info / "entry_points.txt").read_text().strip())
console_script = (PACKAGE_BIN / "log-analyzer").read_text().splitlines()
print("bin/log-analyzer:", *console_script[1:], sep="\n  ")

log_analyzer_psbcg433-0.1.0.dist-info
direct_url.json: {"url":"file://<package>","dir_info":{"editable":true}}
entry_points.txt: [console_scripts]
log-analyzer = log_analyzer.cli:main
bin/log-analyzer:
  # -*- coding: utf-8 -*-
  import sys
  from log_analyzer.cli import main
  if __name__ == "__main__":
      if sys.argv[0].endswith("-script.pyw"):
          sys.argv[0] = sys.argv[0][:-11]
      elif sys.argv[0].endswith(".exe"):
          sys.argv[0] = sys.argv[0][:-4]
      sys.exit(main())


### Step 4 — A tour of the key code

The kernel imports the package from `src/` (the setup cell put it on `sys.path`).
Look at `__version__`: the code asks `importlib.metadata` for the *installed*
distribution's version, and the kernel's environment has no such distribution, so
the fallback `0.0.0+unknown` appears. Installed (Step 3, Step 8) it says `0.1.0`.
This is the price of a single-sourced version, and the reason the fallback branch
is excluded from coverage.

In [480]:
import log_analyzer
from log_analyzer import analysis, cli, export, logging_config, parsing, report

print(f"{log_analyzer.__version__=}")
print(f"imported from: {tidy(log_analyzer.__file__ or '')}")
print(f"public API: {len(log_analyzer.__all__)} names, e.g. {log_analyzer.__all__[-3:]}")

log_analyzer.__version__='0.0.0+unknown'
imported from: <package>/src/log_analyzer/__init__.py
public API: 27 names, e.g. ['write_json', 'write_sqlite', 'write_synthetic_log']


#### `.gz` input

Compression is detected from the first two bytes (`1f 8b`), not from the file name,
so a gzip file named `access.log` works and a plain file named `.gz` does too.
`errors="replace"` turns undecodable bytes into U+FFFD, so one corrupt byte makes
one malformed line instead of a crash. A truncated or corrupt gzip file becomes a
`LogFileError` (exit code 2 in the CLI) instead of a raw `EOFError` or
`zlib.error`.

In [481]:
for function in (parsing.is_gzip_file, parsing.open_log, parsing.read_lines):
    print(inspect.getsource(function))

def is_gzip_file(path: Path) -> bool:
    """True if the file starts with the gzip magic bytes (the extension is not trusted)."""
    with path.open("rb") as raw_file:
        return raw_file.read(len(GZIP_MAGIC)) == GZIP_MAGIC

def open_log(path: Path) -> TextIO:
    """Open a log for reading as text, decompressing gzip transparently.

    Bytes that are not valid UTF-8 become U+FFFD instead of raising, so one bad byte turns
    one line into a malformed line rather than stopping the whole run.
    """
    if is_gzip_file(path):
        return gzip.open(path, "rt", encoding="utf-8", errors="replace")
    return path.open(encoding="utf-8", errors="replace")

def read_lines(path: Path) -> Iterator[tuple[int, str]]:
    """Yield ``(line_number, line)`` for each non-blank line, without the newline.

    Raises:
        LogFileError: The file is gzip-compressed but truncated or corrupt.
    """
    logger.debug("reading %s (%s)", path, "gzip" if is_gzip_file(path) else "plain text")
    tr

#### Analysis, logging and strict mode

`analyze_access_log` is the Phase 1 pipeline plus logging: one INFO record with the
counts and the time taken, one WARNING if any line was malformed. The library only
*emits* records; whether anyone sees them is the application's decision.

In [482]:
print(inspect.getsource(analysis.analyze_access_log))

def analyze_access_log(
    path: Path, *, strict: bool = False
) -> tuple[AccessSummary, list[MalformedLineError]]:
    """Stream the log through ``read_lines -> parse_records -> aggregate``.

    Args:
        path: Plain-text or gzip-compressed access log.
        strict: If true, raise every malformed line together as one ``ExceptionGroup``.

    Returns:
        The summary and the list of malformed lines.
    """
    malformed: list[MalformedLineError] = []
    started = time.perf_counter()
    summary = aggregate(parse_records(read_lines(path), malformed))
    elapsed_ms = (time.perf_counter() - started) * 1000
    logger.info(
        "analyzed %s: %d records, %d malformed in %.1f ms",
        path.name,
        summary.record_count,
        len(malformed),
        elapsed_ms,
    )
    if malformed:
        total_lines = summary.record_count + len(malformed)
        logger.warning(
            "%d of %d lines in %s are malformed (%.1f%%)",
            len(malformed),
         

#### The `dictConfig` dictionary

The CLI configures logging with one dictionary (Unit 4.3.9): a stderr handler at the
level chosen by `-q` / `-v` / `-vv`, and with `--log-file` a second handler that
records everything from DEBUG up. The package logger's own level is the *lowest*
any handler needs; each handler then filters by its own level. `propagate: False`
keeps records away from the root logger, so they are not printed twice.

In [483]:
logging_dict = logging_config.build_logging_config("WARNING", log_file=Path("run.log"))
print(json.dumps(logging_dict, indent=2))

{
  "version": 1,
  "disable_existing_loggers": false,
  "formatters": {
    "console": {
      "format": "%(levelname)s %(name)s: %(message)s"
    },
    "file": {
      "format": "%(asctime)s %(levelname)-8s %(name)s:%(lineno)d %(message)s",
      "datefmt": "%Y-%m-%dT%H:%M:%S%z"
    }
  },
  "handlers": {
    "console": {
      "class": "logging.StreamHandler",
      "stream": "ext://sys.stderr",
      "formatter": "console",
      "level": "WARNING"
    },
    "file": {
      "class": "logging.FileHandler",
      "filename": "run.log",
      "encoding": "utf-8",
      "formatter": "file",
      "level": "DEBUG"
    }
  },
  "loggers": {
    "log_analyzer": {
      "level": "DEBUG",
      "handlers": [
        "console",
        "file"
      ],
      "propagate": false
    }
  }
}


#### The CLI: strict mode with `except*`

`validate` uses the library's strict mode, which raises every malformed line at once
as an `ExceptionGroup` (Phase 1 Project, Step 8). Python does not allow `return`
inside an `except*` clause, so the handler records the exit code and returns after
the `try`. `main` turns expected failures (`LogAnalyzerError`, `OSError`) into a
one-line message on stderr and exit code 2; anything else is a bug and keeps its
traceback.

In [484]:
print(inspect.getsource(cli.run_validate))
print(inspect.getsource(cli.main))

def run_validate(args: argparse.Namespace) -> int:
    """``validate``: strict mode; list malformed lines and exit 1 if there are any."""
    exit_code = EXIT_OK
    try:
        summary, _ = analyze_access_log(args.log, strict=True)
    except* MalformedLineError as group:
        problems = [error for error in group.exceptions if isinstance(error, MalformedLineError)]
        print(f"{args.log}: FAILED, {len(problems):,} malformed lines")
        for error in problems[: args.show]:
            print(f"  line {error.line_number}: {error.reason}: {error.line[:50]!r}")
        if len(problems) > args.show:
            print(f"  ... and {len(problems) - args.show:,} more")
        exit_code = EXIT_MALFORMED
    else:
        print(f"{args.log}: OK, {summary.record_count:,} records, no malformed lines")
    return exit_code

def main(argv: Sequence[str] | None = None) -> int:
    """Entry point of the ``log-analyzer`` console script; returns the exit status."""
    args = build_parser().p

#### Streaming export to SQLite

`executemany` accepts a generator, so records flow from the gzip file into SQLite one
at a time. The `malformed` list is filled *while* the records are consumed, which is
why it is inserted afterwards. `closing(...)` closes the connection;
`with connection:` only commits (predict question 17 comes back to this).

In [485]:
print(inspect.getsource(export.write_sqlite))

def write_sqlite(
    records: Iterable[AccessRecord],
    output: Path,
    malformed: list[MalformedLineError] | None = None,
) -> int:
    """Replace ``output`` with a database holding ``records`` (and ``malformed``) tables.

    ``malformed`` is read after every record has been inserted, so it can be the list that
    ``parse_records`` fills while ``records`` is consumed.
    """
    output.unlink(missing_ok=True)
    placeholders = ", ".join("?" * len(RECORD_COLUMNS))
    with closing(sqlite3.connect(output)) as connection, connection:
        connection.executescript(SQLITE_SCHEMA)
        connection.executemany(
            f"INSERT INTO records ({', '.join(RECORD_COLUMNS)}) VALUES ({placeholders})",
            (record_row(record) for record in records),
        )
        connection.executemany(
            "INSERT INTO malformed (line_number, reason, line) VALUES (?, ?, ?)",
            ((error.line_number, error.reason, error.line) for error in malformed or []),
        )
   

The same functions are a library API. One line through the pipeline: parsed into a
record, normalized to an endpoint, flattened into an export row:

In [486]:
sample_line = '2026-11-06T09:00:00Z 10.0.2.17 "GET /api/orders/42?expand=items HTTP/1.1" 200 23ms'
sample_record = parsing.parse_access_line(1, sample_line)
print(sample_record)
print(analysis.normalize_endpoint(sample_record.method, sample_record.path))
print(export.record_row(sample_record))
print(f"{parsing.format_access_line(sample_record) == sample_line=}")

AccessRecord(timestamp='2026-11-06T09:00:00Z', client_ip='10.0.2.17', method='GET', path='/api/orders/42?expand=items', status=200, latency_ms=23)
GET /api/orders/{id}
('2026-11-06T09:00:00Z', '10.0.2.17', 'GET', '/api/orders/42?expand=items', 'GET /api/orders/{id}', 200, 23)
parsing.format_access_line(sample_record) == sample_line=True


### Step 5 — Run the CLI

From here on, `log-analyzer` is the console script installed in the package's
environment: exactly what a user gets after `pip install`. Global options (`-v`,
`-q`, `--log-file`) come before the subcommand.

In [487]:
shell("log-analyzer", "--help")

[<scratch>]$ log-analyzer --help
  usage: log-analyzer [-h] [--version] [-v | -q] [--log-file LOG_FILE] COMMAND ...
  
  Analyze web-server access logs (plain text or gzip).
  
  positional arguments:
    COMMAND
      generate           write a seeded synthetic access log
      report             print a summary report
      export             export parsed records to CSV, JSON or SQLite
      validate           exit 1 if any line is malformed
  
  options:
    -h, --help           show this help message and exit
    --version            show program's version number and exit
    -v, --verbose        log more on stderr (-v, -vv)
    -q, --quiet          log only errors
    --log-file LOG_FILE  also write DEBUG logs to this file
  
  Exit status: 0 success, 1 malformed lines found (validate), 2 error.


#### `generate`

The Phase 1 generator, now a subcommand. A `.gz` name writes compressed output, and
the output is **byte-for-byte reproducible**: gzip headers normally store the file
name and the current time, so `generate.py` writes through
`GzipFile(filename="", mtime=0)`. The cell writes the same 20 000 lines plain and
compressed and compares them.

In [488]:
shell("log-analyzer", "generate", "access.log.gz", "-n", "20000")
shell("log-analyzer", "generate", "access.log", "-n", "20000")
for name in ("access.log", "access.log.gz"):
    path = PROJECT_DIR / name
    print(f"{name:<14}{path.stat().st_size:>10,} bytes, starts with {path.read_bytes()[:2]!r}")
compressed_text = gzip.decompress((PROJECT_DIR / "access.log.gz").read_bytes())
print(f"same text inside: {compressed_text == (PROJECT_DIR / 'access.log').read_bytes()}")

[<scratch>]$ log-analyzer generate access.log.gz -n 20000
  wrote 20,000 lines to access.log.gz (431 malformed: garbage 100, status out of range 110, truncated 116, unknown method 105)


[<scratch>]$ log-analyzer generate access.log -n 20000
  wrote 20,000 lines to access.log (431 malformed: garbage 100, status out of range 110, truncated 116, unknown method 105)
access.log     1,433,929 bytes, starts with b'20'
access.log.gz    201,302 bytes, starts with b'\x1f\x8b'
same text inside: True


#### `report`

The WARNING line comes from the library's logger through the CLI's `dictConfig`, on
stderr; the report itself is on stdout.

In [489]:
shell("log-analyzer", "report", "access.log.gz")

[<scratch>]$ log-analyzer report access.log.gz
  WARNING log_analyzer.analysis: 431 of 20000 lines in access.log.gz are malformed (2.2%)
  =====================ACCESS LOG REPORT======================
  Lines read                                            20,000
  Records parsed                                        19,569
  Malformed lines                                          431
  Overall p95 latency (ms)                                 225
  Status codes -----------------------------------------------
  Status  Class                        Requests          Share
  200     success                        15,690          80.2%
  201     success                           982           5.0%
  204     success                           384           2.0%
  304     redirect                          612           3.1%
  400     client error                      575           2.9%
  401     client error                      187           1.0%
  404     client error                      

**Same algorithm, same answer.** The package's test data holds the report the Phase 1
notebook printed for the same seed and line count. Reading the gzip file instead of
plain text changes nothing:

In [490]:
_, package_report = run("log-analyzer", "-q", "report", "access.log.gz")
phase1_report = (PACKAGE_DIR / "tests" / "data" / "phase1_report_seed42_20000.txt").read_text(
    encoding="utf-8"
)
print(f"identical to the Phase 1 notebook's report: {package_report + chr(10) == phase1_report}")

identical to the Phase 1 notebook's report: True


`--format json` prints the same facts as data, for scripts. Status codes become
string keys, because JSON object keys are always strings (predict question 14);
`summary_document` converts them explicitly so the `TypedDict` says so too.

In [491]:
shell("log-analyzer", "-q", "report", "access.log.gz", "--top", "2", "--format", "json")

[<scratch>]$ log-analyzer -q report access.log.gz --top 2 --format json
  {
    "lines_read": 20000,
    "records_parsed": 19569,
    "malformed_lines": 431,
    "p95_latency_ms": 225,
    "status_counts": {
      "200": 15690,
      "201": 982,
      "204": 384,
      "304": 612,
      "400": 575,
      "401": 187,
      "404": 580,
      "429": 197,
      "500": 164,
      "502": 98,
      "503": 100
    },
    "top_endpoints": [
      {
        "endpoint": "GET /api/products",
        "requests": 5904,
        "p95_ms": 120
      },
      {
        "endpoint": "GET /api/products/{id}",
        "requests": 4798,
        "p95_ms": 74
      }
    ],
    "malformed_by_reason": {
      "status out of range": 110,
      "unknown method": 105,
      "unrecognized format": 216
    }
  }


#### `export`: CSV, JSON and SQLite

The format comes from the extension (or `--format`). CSV and JSON hold the records;
SQLite also stores the malformed lines in their own table. All three stream: one
record in memory at a time.

In [492]:
shell("log-analyzer", "-q", "export", "access.log.gz", "records.csv")
show_lines(PROJECT_DIR / "records.csv", first=3)

[<scratch>]$ log-analyzer -q export access.log.gz records.csv
  exported 19,569 records to records.csv (csv); 431 malformed lines skipped
timestamp,client_ip,method,path,endpoint,status,latency_ms
2026-11-06T09:00:01Z,10.0.0.152,GET,/api/products?page=4,GET /api/products,200,216
2026-11-06T09:00:01Z,10.0.3.57,GET,/api/products,GET /api/products,200,17


The JSON export is one array, written incrementally (`json.dump` per record), so it
never builds a 20 000-element list in memory:

In [493]:
shell("log-analyzer", "-q", "export", "access.log.gz", "records.json")
show_lines(PROJECT_DIR / "records.json", first=3)
exported_records = json.loads((PROJECT_DIR / "records.json").read_text(encoding="utf-8"))
print(f"{len(exported_records)=}, keys: {list(exported_records[0])}")

[<scratch>]$ log-analyzer -q export access.log.gz records.json
  exported 19,569 records to records.json (json); 431 malformed lines skipped
[
  {"timestamp": "2026-11-06T09:00:01Z", "client_ip": "10.0.0.152", "method": "GET", "path": "/api/products?page=4", "endpoint": "GET /api/products", "status": 200, "latency_ms": 216},
  {"timestamp": "2026-11-06T09:00:01Z", "client_ip": "10.0.3.57", "method": "GET", "path": "/api/products", "endpoint": "GET /api/products", "status": 200, "latency_ms": 17},
len(exported_records)=19569, keys: ['timestamp', 'client_ip', 'method', 'path', 'endpoint', 'status', 'latency_ms']


SQLite turns the log into something you can query. Which endpoints fail with 5xx
most often, and how slow were the failures? (`p95` has no SQL function, which is
one reason the report computes it in Python.)

In [494]:
shell("log-analyzer", "-q", "export", "access.log.gz", "records.db")
with closing(sqlite3.connect(PROJECT_DIR / "records.db")) as connection:
    table_rows = connection.execute("SELECT name FROM sqlite_master WHERE type = 'table'")
    print(f"tables: {sorted(name for (name,) in table_rows)}")
    failures = connection.execute(
        "SELECT endpoint, COUNT(*), MAX(latency_ms) FROM records WHERE status >= 500"
        " GROUP BY endpoint ORDER BY 2 DESC LIMIT 3"
    )
    for endpoint, failure_count, slowest_ms in failures:
        print(f"  {endpoint:<26}{failure_count:>5} failures, slowest {slowest_ms:>5} ms")
    reasons = connection.execute("SELECT reason, COUNT(*) FROM malformed GROUP BY reason")
    print(f"malformed by reason: {sorted(reasons)}")

[<scratch>]$ log-analyzer -q export access.log.gz records.db
  exported 19,569 records to records.db (sqlite); 431 malformed lines stored in table 'malformed'
tables: ['malformed', 'records']
  GET /api/products           112 failures, slowest  1445 ms
  GET /api/products/{id}       88 failures, slowest   575 ms
  POST /api/cart/items         41 failures, slowest  1210 ms
malformed by reason: [('status out of range', 110), ('unknown method', 105), ('unrecognized format', 216)]


#### `validate` and exit codes

`validate` is the strict mode: it lists the first malformed lines and exits with 1.
A clean log exits with 0. A CI step or a shell `&&` chain can rely on that.

In [495]:
shell("log-analyzer", "-q", "validate", "access.log.gz", "--show", "3")
shell("log-analyzer", "-q", "generate", "clean.log", "-n", "1000", "--malformed-rate", "0")
shell("log-analyzer", "validate", "clean.log")

[<scratch>]$ log-analyzer -q validate access.log.gz --show 3
  access.log.gz: FAILED, 431 malformed lines
    line 64: unrecognized format: '-'
    line 127: status out of range: '2026-11-06T09:01:46Z 10.0.2.31 "GET /api/products '
    line 232: unrecognized format: '-'
    ... and 428 more
  (exit code 1)
[<scratch>]$ log-analyzer -q generate clean.log -n 1000 --malformed-rate 0
  wrote 1,000 lines to clean.log (no malformed lines)


[<scratch>]$ log-analyzer validate clean.log
  clean.log: OK, 1,000 records, no malformed lines


#### Logging levels and the log file

`-vv` lowers the stderr handler to DEBUG (every skipped line is logged), and
`--log-file` adds a file handler with timestamps and line numbers. The timestamps
depend on when the cell runs, so they are shown as `<timestamp>`.

In [496]:
shell("log-analyzer", "-vv", "--log-file", "run.log", "validate", "access.log.gz", "--show", "1",
      max_lines=6)  # fmt: skip
log_file_lines = (PROJECT_DIR / "run.log").read_text(encoding="utf-8").splitlines()
level_counts = Counter(line.split()[1] for line in log_file_lines)
print(f"run.log: {len(log_file_lines)} lines, by level {dict(level_counts)}")
print(tidy(log_file_lines[1]))

[<scratch>]$ log-analyzer -vv --log-file run.log validate access.log.gz --show 1
  DEBUG log_analyzer.cli: arguments: {'verbose': 2, 'quiet': False, 'log_file': PosixPath('run.log'), 'command': 'validate', 'log': PosixPath('access.log.gz'), 'show': 1}
  DEBUG log_analyzer.parsing: reading access.log.gz (gzip)
  DEBUG log_analyzer.parsing: skipping line 64: unrecognized format
  DEBUG log_analyzer.parsing: skipping line 127: status out of range
  DEBUG log_analyzer.parsing: skipping line 232: unrecognized format
  DEBUG log_analyzer.parsing: skipping line 279: unknown method
  ... (432 more lines)
  (exit code 1)
run.log: 435 lines, by level {'DEBUG': 433, 'INFO': 1, 'WARNING': 1}
<timestamp> DEBUG    log_analyzer.parsing:78 reading access.log.gz (gzip)


#### Errors

Expected failures give one line on stderr and exit code 2, never a traceback: a
truncated gzip file (the first 50 000 bytes of the real one), a missing file, and an
export format that cannot be inferred. `python -m log_analyzer` runs the same CLI
through `__main__.py`.

In [497]:
(PROJECT_DIR / "cut.log.gz").write_bytes((PROJECT_DIR / "access.log.gz").read_bytes()[:50_000])
shell("log-analyzer", "-q", "report", "cut.log.gz")
shell("log-analyzer", "report", "missing.log")
shell("log-analyzer", "export", "access.log.gz", "records.txt")
shell("python", "-m", "log_analyzer", "--version")

[<scratch>]$ log-analyzer -q report cut.log.gz
  log-analyzer: error: cut.log.gz: compressed file ended early (truncated gzip)
  (exit code 2)


[<scratch>]$ log-analyzer report missing.log
  log-analyzer: error: [Errno 2] No such file or directory: 'missing.log'
  (exit code 2)
[<scratch>]$ log-analyzer export access.log.gz records.txt
  log-analyzer: error: cannot infer the export format from 'records.txt'; use --format or one of the extensions .csv, .json, .db, .sqlite, .sqlite3
  (exit code 2)


[<scratch>]$ python -m log_analyzer --version
  log-analyzer 0.1.0


### Step 6 — Tests: pytest, Hypothesis and coverage

| File | What it covers |
| --- | --- |
| `conftest.py` | fixtures (`sample_log`, `clean_log`, `generated_log`), Hypothesis profiles, an autouse fixture that undoes `dictConfig` after each test |
| `test_parsing.py` | every malformed reason (parametrized), gzip by content, invalid UTF-8, truncated and corrupt gzip |
| `test_analysis.py` | helpers, one-pass aggregation, log records with `caplog`, the strict `ExceptionGroup` |
| `test_report.py` | alignment, empty logs, the JSON document, the Phase 1 golden report |
| `test_export.py` | CSV quoting, JSON, SQLite tables, format inference, streaming |
| `test_generate.py` | reproducible bytes (also gzip), every injected line accounted for |
| `test_logging_config.py` | the dictionary, level routing to stderr and file (`capsys`), library silence |
| `test_cli.py` | every subcommand and exit code through `main(argv)`, `--version`, `python -m` |
| `test_properties.py` | Hypothesis: round trips, invariants, export round trips |

Example tests check one input each. **Properties** state a rule for every input and
let Hypothesis search for a counter-example (Unit 4.3.7). Here are the strategies and
the first two properties: formatting a record and parsing it back gives the same
record, and the parser never raises anything but `MalformedLineError`, whatever
text it gets.

In [498]:
property_source = (PACKAGE_DIR / "tests" / "test_properties.py").read_text(encoding="utf-8")
start = property_source.index("# isoformat()")
stop = property_source.index("@given(st.lists(st.one_of")
print(property_source[start:stop].rstrip())

# isoformat() always writes a 4-digit year; strftime("%Y") gives "999" for year 999 on
# Python 3.12-3.13 (glibc), which Hypothesis found in the CI matrix.
timestamps = st.datetimes().map(lambda moment: moment.isoformat(timespec="seconds") + "Z")
ip_addresses = st.tuples(*[st.integers(0, 255)] * 4).map(lambda parts: ".".join(map(str, parts)))
paths = st.from_regex(r"/[A-Za-z0-9/._~?=&%-]{0,40}", fullmatch=True)
records = st.builds(
    AccessRecord,
    timestamp=timestamps,
    client_ip=ip_addresses,
    method=st.sampled_from(sorted(HTTP_METHODS)),
    path=paths,
    status=st.integers(100, 599),
    latency_ms=st.integers(0, 10**6),
)
latency_lists = st.lists(st.integers(0, 10**6), min_size=1)
log_lines = st.text(alphabet=st.characters(exclude_characters="\r\n\x0b\x0c\x1c\x1d\x1e\x85"))


@given(records)
def test_format_then_parse_round_trips(record: AccessRecord) -> None:
    assert parse_access_line(1, format_access_line(record)) == record


@given(log_lines)
def test_parser_onl

> **Note: what the CI matrix caught.** The first version of the `timestamps`
> strategy used `moment.strftime("%Y-...")`. It passed on 3.14 and 3.15, but on
> 3.12 and 3.13 Hypothesis found `datetime(999, 1, 1)`: those versions print the
> year as `999` on Linux (glibc), the regex wants four digits, and the round trip
> failed. The package itself only ever formats 2026 timestamps; the *test* was wrong.
> `isoformat()` always pads the year. This is exactly what a version matrix plus
> property-based testing is for.

In [499]:
year_999 = dt.datetime(999, 1, 1)
print(f"Python {sys.version_info[:2]}: strftime -> {year_999.strftime('%Y-%m-%d')!r}, "
      f"isoformat -> {year_999.isoformat()!r}")  # fmt: skip

Python (3, 15): strftime -> '0999-01-01', isoformat -> '0999-01-01T00:00:00'


The whole suite, with the configuration from `pyproject.toml`: coverage measured
with branches, missing lines listed, and the run **fails under 90%**.
`HYPOTHESIS_PROFILE=ci` makes Hypothesis derandomized, so this output (and a CI
failure) is reproducible. The only partial branch, `44->exit` in `parsing.py`, is the
`match` statement: coverage.py counts "no `case` matched" as a possible exit, even
though the last case always matches.

In [500]:
shell("uv", "run", "--locked", "pytest", cwd=PACKAGE_DIR)

[<package>]$ uv run --locked pytest
  ======================================= test session starts ========================================
  platform linux -- Python 3.15.1, pytest-9.2.0, pluggy-1.7.0
  rootdir: <package>
  configfile: pyproject.toml
  testpaths: tests
  plugins: hypothesis-6.170.15, cov-7.1.0
  collected 108 items
  
  tests/test_analysis.py ..................                                                    [ 16%]
  tests/test_cli.py ....................                                                       [ 35%]
  tests/test_export.py ................                                                        [ 50%]
  tests/test_generate.py ........                                                              [ 57%]
  tests/test_logging_config.py ..........                                                      [ 66%]
  tests/test_parsing.py .................                                                      [ 82%]
  tests/test_properties.py ............            

**The gate works.** Running only one test file leaves most of the CLI and the
exporters unexecuted, the total drops below 90%, and pytest exits with 1 even though
every selected test passed:

In [501]:
shell("uv", "run", "--locked", "pytest", "tests/test_report.py", cwd=PACKAGE_DIR,
      keep=r"TOTAL|Required|passed|failed")  # fmt: skip

[<package>]$ uv run --locked pytest tests/test_report.py
  TOTAL                                  434    172     90      8    60%
  FAIL Required test coverage of 90% not reached. Total coverage: 60.31%
  ======================================== 7 passed in <time> =========================================
  (exit code 1)


### Step 7 — `mypy --strict`, ruff and `py.typed`

`[tool.mypy] strict = true` with `files = ["src", "tests"]`, so plain `mypy` is
already strict; the cell (and CI) pass `--strict` anyway, to make the gate visible.
The tests are typed too (`pytest.CaptureFixture[str]`, `pytest.LogCaptureFixture`,
typed Hypothesis strategies). Then the linter and the formatter, as CI runs them.

In [502]:
shell("uv", "run", "--locked", "mypy", "--strict", cwd=PACKAGE_DIR)
shell("uv", "run", "--locked", "ruff", "check", ".", cwd=PACKAGE_DIR)
shell("uv", "run", "--locked", "ruff", "format", "--check", ".", cwd=PACKAGE_DIR)

[<package>]$ uv run --locked mypy --strict
  Success: no issues found in 21 source files
[<package>]$ uv run --locked ruff check .
  All checks passed!
[<package>]$ uv run --locked ruff format --check .
  22 files already formatted


**`py.typed` in action (PEP 561).** Without the marker file, mypy treats an installed
package as untyped and checks nothing about how you call it. With it, a *user's*
code is checked against the package's hints. A small consumer script, outside the
package, checked by the mypy in the package's environment:

In [503]:
(PROJECT_DIR / "consumer.py").write_text(
    textwrap.dedent("""
        from pathlib import Path

        from log_analyzer import analyze_access_log, percentile

        summary, malformed = analyze_access_log(Path("access.log.gz"))
        reveal_type(summary.status_counts)
        percentile(["120", "15"], 95)
        analyze_access_log("access.log.gz")
    """).lstrip()
)
shell("uv", "run", "--project", str(PACKAGE_DIR), "--locked", "mypy", "--strict", "consumer.py")

[<scratch>]$ uv run --project <package> --locked mypy --strict consumer.py
  consumer.py:6: note: Revealed type is "collections.Counter[int]"
  consumer.py:7: error: List item 0 has incompatible type "str"; expected "int"  [list-item]
  consumer.py:7: error: List item 1 has incompatible type "str"; expected "int"  [list-item]
  consumer.py:8: error: Argument 1 to "analyze_access_log" has incompatible type "str"; expected "Path"  [arg-type]
  Found 3 errors in 1 file (checked 1 source file)
  (exit code 1)


### Step 8 — Build the sdist and the wheel

`uv build` builds the sdist first, then the wheel *from the sdist*, which proves the
sdist is complete. The build runs `uv_build` in an isolated environment (PEP 517).

In [504]:
shutil.rmtree(PACKAGE_DIR / "dist", ignore_errors=True)
shell("uv", "build", cwd=PACKAGE_DIR)
wheel_path = next((PACKAGE_DIR / "dist").glob("*.whl"))
sdist_path = next((PACKAGE_DIR / "dist").glob("*.tar.gz"))
print(f"{wheel_path.name=}\n{sdist_path.name=}")

[<package>]$ uv build
  Building source distribution...
  Successfully built dist/log_analyzer_psbcg433-0.1.0.tar.gz
  Successfully built dist/log_analyzer_psbcg433-0.1.0-py3-none-any.whl
wheel_path.name='log_analyzer_psbcg433-0.1.0-py3-none-any.whl'
sdist_path.name='log_analyzer_psbcg433-0.1.0.tar.gz'


**Under the hood: a wheel is a zip file** (Unit 4.1.8). It holds the package
(including `py.typed`) and a `.dist-info` folder: `METADATA` (the core metadata the
index displays), `entry_points.txt` (the console script), `WHEEL` (the tag) and
`RECORD` (a hash per file). No tests, no `pyproject.toml`.

In [505]:
dist_info_name = "log_analyzer_psbcg433-0.1.0.dist-info"
with zipfile.ZipFile(wheel_path) as wheel:
    wheel_names = wheel.namelist()
    metadata_text = wheel.read(f"{dist_info_name}/METADATA").decode()
    entry_points_text = wheel.read(f"{dist_info_name}/entry_points.txt").decode()
print(*wheel_names, sep="\n")
print("--- entry_points.txt\n" + entry_points_text.strip())

log_analyzer/
log_analyzer/__init__.py
log_analyzer/__main__.py
log_analyzer/analysis.py
log_analyzer/cli.py
log_analyzer/export.py
log_analyzer/generate.py
log_analyzer/logging_config.py
log_analyzer/models.py
log_analyzer/parsing.py
log_analyzer/py.typed
log_analyzer/report.py
log_analyzer_psbcg433-0.1.0.dist-info/
log_analyzer_psbcg433-0.1.0.dist-info/WHEEL
log_analyzer_psbcg433-0.1.0.dist-info/entry_points.txt
log_analyzer_psbcg433-0.1.0.dist-info/METADATA
log_analyzer_psbcg433-0.1.0.dist-info/RECORD
--- entry_points.txt
[console_scripts]
log-analyzer = log_analyzer.cli:main


The `METADATA` headers (the README follows them as the long description). The
classifiers are summarized to keep the output short:

In [506]:
metadata_headers = metadata_text.split("\n\n", 1)[0].splitlines()
print(*(line for line in metadata_headers if not line.startswith("Classifier")), sep="\n")
classifier_count = sum(line.startswith("Classifier") for line in metadata_headers)
print(f"... and {classifier_count} Classifier lines")
with tarfile.open(sdist_path) as sdist:
    sdist_names = sdist.getnames()
tests_in_sdist = any(name.endswith("/tests/test_cli.py") for name in sdist_names)
print(f"sdist: {len(sdist_names)} entries, tests included: {tests_in_sdist}")

Metadata-Version: 2.3
Name: log-analyzer-psbcg433
Version: 0.1.0
Summary: Analyze web-server access logs (plain or gzip) and export them to CSV, JSON or SQLite.
Keywords: access-log,log-analysis,cli,sqlite,csv
Author: psbcg433
Requires-Python: >=3.12
Project-URL: Source, https://github.com/psbcg433/python-atlas/tree/main/projects/log-analyzer
Project-URL: Issues, https://github.com/psbcg433/python-atlas/issues
Description-Content-Type: text/markdown
... and 13 Classifier lines
sdist: 32 entries, tests included: True


**The user's view.** A brand-new virtual environment, the wheel installed into it
(no index needed: there are no dependencies), and the console script run from there.
This is what `pip install log-analyzer-psbcg433` will do once the package is on
TestPyPI.

In [507]:
consumer_venv = PROJECT_DIR / "consumer-venv"
shell("uv", "venv", "--quiet", "--python", "3.15", str(consumer_venv))
shell("uv", "pip", "install", "--python", str(consumer_venv), str(wheel_path), keep=r"^ \+")
shell(str(consumer_venv / "bin" / "log-analyzer"), "--version")
shell(str(consumer_venv / "bin" / "log-analyzer"), "-q", "validate", "clean.log")

[<scratch>]$ uv venv --quiet --python 3.15 <scratch>/consumer-venv
[<scratch>]$ uv pip install --python <scratch>/consumer-venv <package>/dist/log_analyzer_psbcg433-0.1.0-py3-none-any.whl
   + log-analyzer-psbcg433==0.1.0 (from file://<package>/dist/log_analyzer_psbcg433-0.1.0-py3-none-any.whl)
[<scratch>]$ <scratch>/consumer-venv/bin/log-analyzer --version
  log-analyzer 0.1.0
[<scratch>]$ <scratch>/consumer-venv/bin/log-analyzer -q validate clean.log
  clean.log: OK, 1,000 records, no malformed lines


### Step 9 — The GitHub Actions workflow and trusted publishing

`.github/workflows/log-analyzer.yml` lives at the repository root (GitHub only reads
workflows there) and sets `working-directory: projects/log-analyzer` for every
`run` step.

| Event | test | build | publish |
| --- | --- | --- | --- |
| push or pull request touching `projects/log-analyzer/**` or the workflow | ✓ (3.12–3.15) | ✓ | — |
| push of a tag `log-analyzer-v*` (path filters do not apply to tags) | ✓ | ✓ + tag = version check | ✓ |
| manual *Run workflow* (`workflow_dispatch`) | ✓ | ✓ | ✓ |
| push of any other tag (for example `phase-4`) | — | — | — |

**Trusted publishing** replaces the API token. TestPyPI is told in advance: "trust
uploads for `log-analyzer-psbcg433` from repository `psbcg433/python-atlas`,
workflow `log-analyzer.yml`, environment `testpypi`". At run time:

```mermaid
sequenceDiagram
    participant J as publish job
    participant G as GitHub OIDC provider
    participant T as TestPyPI
    J->>G: request an ID token (needs permissions: id-token: write)
    G-->>J: signed token: repo, workflow, environment, ref
    J->>T: exchange the token
    T-->>J: short-lived upload token, if the claims match the trusted publisher
    J->>T: upload sdist + wheel
```

<details><summary>ASCII fallback</summary>

```text
publish job --(id-token: write)--> GitHub OIDC: "give me an ID token"
publish job <-- signed token (repo, workflow file, environment, ref)
publish job --> TestPyPI: exchange token
publish job <-- upload token valid for minutes, only if the claims match
publish job --> TestPyPI: upload dist/*
```

</details>

Nothing secret is stored in the repository, and a token leaked from a log expires
within minutes. Other design points: the publish job gets `id-token: write` and
nothing else; it downloads the artifact the build job made, so what was tested is
what is uploaded; `--locked` everywhere means CI installs exactly `uv.lock`.

In [508]:
show_lines(WORKFLOW_PATH)

# CI and release for the log-analyzer package in projects/log-analyzer/.
#
# - test:    ruff, mypy --strict and pytest (coverage must stay >= 90%) on Python 3.12-3.15,
#            for pushes and pull requests that touch the package (and for releases).
# - build:   sdist + wheel, smoke-tested, uploaded as an artifact. On a release tag it also
#            checks that the tag matches the version in pyproject.toml.
# - publish: uploads the artifact to TestPyPI with trusted publishing (OpenID Connect):
#            no API token is stored anywhere. Runs only for tags log-analyzer-v* and for a
#            manual "Run workflow", and only after test and build pass.
name: log-analyzer

on:
  push:
    branches: ["**"]
    tags: ["log-analyzer-v*"] # path filters are not applied to tag pushes
    paths:
      - "projects/log-analyzer/**"
      - ".github/workflows/log-analyzer.yml"
  pull_request:
    paths:
      - "projects/log-analyzer/**"
      - ".github/workflows/log-analyzer.yml"
  work

> **Warning: YAML 1.1's `on`.** PyYAML implements YAML 1.1, where a bare `on`, `yes`
> or `off` is a **boolean**. The workflow's `on:` key loads as `True`, so
> `workflow["on"]` raises `KeyError`. GitHub's own parser does not have this
> problem; scripts that read workflows do.

In [509]:
import yaml  # PyYAML, in the course environment since Unit 4.5

workflow = yaml.safe_load(WORKFLOW_PATH.read_text(encoding="utf-8"))
print(f"top-level keys: {[repr(key) for key in workflow]}")
triggers = workflow[True]
print(f"events: {list(triggers)}, tags: {triggers['push']['tags']}")
publish_job = workflow["jobs"]["publish"]
print(f"publish: needs={publish_job['needs']!r}, permissions={publish_job['permissions']}")
print(f"publish: environment={publish_job['environment']['name']!r}")
print(f"publish: if={publish_job['if']!r}")
print(f"publish steps: {[step['uses'] for step in publish_job['steps']]}")

top-level keys: ["'name'", 'True', "'permissions'", "'concurrency'", "'defaults'", "'env'", "'jobs'"]
events: ['push', 'pull_request', 'workflow_dispatch'], tags: ['log-analyzer-v*']
publish: needs='build', permissions={'id-token': 'write'}
publish: environment='testpypi'
publish: if="startsWith(github.ref, 'refs/tags/log-analyzer-v') || github.event_name == 'workflow_dispatch'"
publish steps: ['actions/download-artifact@v8', 'pypa/gh-action-pypi-publish@release/v1']


The build job's tag check, in Python: the shell expression
`${GITHUB_REF_NAME#log-analyzer-v}` strips the prefix, like `str.removeprefix`. A
tag that does not match `pyproject.toml` stops the release before anything is
uploaded.

In [510]:
def tag_matches_version(ref_name: str, version: str) -> bool:
    """The build job's check: tag log-analyzer-vX.Y.Z must equal the package version."""
    return ref_name.removeprefix("log-analyzer-v") == version


for ref_name in ("log-analyzer-v0.1.0", "log-analyzer-v0.2.0"):
    print(f"{ref_name}: {tag_matches_version(ref_name, project['version'])}")

log-analyzer-v0.1.0: True
log-analyzer-v0.2.0: False


### Step 10 — Self-check

Each syllabus requirement, checked against the real artefacts: the tool runs, the
wheel, the exports and the workflow file. All checks should pass.

In [511]:
_, mypy_output = run("uv", "run", "--locked", "mypy", "--strict", cwd=PACKAGE_DIR)
_, pytest_output = run("uv", "run", "--locked", "pytest", "-q", cwd=PACKAGE_DIR)
coverage_match = re.search(r"Total coverage: ([\d.]+)%", pytest_output)
total_coverage = float(coverage_match[1]) if coverage_match else 0.0
with (PROJECT_DIR / "records.csv").open(encoding="utf-8", newline="") as csv_file:
    csv_row_count = sum(1 for _ in csv.DictReader(csv_file))
with closing(sqlite3.connect(PROJECT_DIR / "records.db")) as connection:
    (sqlite_row_count,) = connection.execute("SELECT COUNT(*) FROM records").fetchone()
commands = (["generate", "x"], ["report", "x"], ["export", "x", "y"], ["validate", "x"])
exit_code_cases = (
    ["validate", "clean.log"],
    ["validate", "access.log.gz"],
    ["report", "missing.log"],
)
pytest_addopts = config["tool"]["pytest"]["ini_options"]["addopts"]
publish_condition = (
    "startsWith(github.ref, 'refs/tags/log-analyzer-v') || github.event_name == 'workflow_dispatch'"
)
print(f"{total_coverage=}, {csv_row_count=}, {sqlite_row_count=}")

total_coverage=99.81, csv_row_count=19569, sqlite_row_count=19569


**Package checks:** layout, typing, tests, logging, CLI and exports.

In [512]:
run_checks(
    [
        ("src/ layout", lambda: (PACKAGE_DIR / "src" / "log_analyzer").is_dir(), True),
        ("no package next to the tests", lambda: (PACKAGE_DIR / "log_analyzer").exists(), False),
        ("py.typed is in the wheel", lambda: "log_analyzer/py.typed" in wheel_names, True),
        ("mypy strict configured", lambda: config["tool"]["mypy"]["strict"], True),
        ("mypy --strict clean", lambda: mypy_output.startswith("Success: no issues"), True),
        ("coverage >= 90%", lambda: total_coverage >= 90, True),
        ("coverage gate configured", lambda: "--cov-fail-under=90" in pytest_addopts, True),
        ("Hypothesis properties exist", lambda: property_source.count("@given("), 11),
        ("dictConfig logging", lambda: "dictConfig" in inspect.getsource(logging_config), True),
        (
            "argparse subcommands",
            lambda: [cli.build_parser().parse_args(argv).command for argv in commands],
            ["generate", "report", "export", "validate"],
        ),
        (
            ".gz gives the same report as plain text",
            lambda: (
                run("log-analyzer", "-q", "report", "access.log.gz")
                == run("log-analyzer", "-q", "report", "access.log")
            ),
            True,
        ),
        (
            "CSV / JSON / SQLite hold every record",
            lambda: (csv_row_count, len(exported_records), sqlite_row_count),
            (19_569, 19_569, 19_569),
        ),
        (
            "exit codes 0 / 1 / 2",
            lambda: [run("log-analyzer", "-q", *argv)[0] for argv in exit_code_cases],
            [0, 1, 2],
        ),
    ]
)

PASS  src/ layout
PASS  no package next to the tests
PASS  py.typed is in the wheel
PASS  mypy strict configured
PASS  mypy --strict clean
PASS  coverage >= 90%
PASS  coverage gate configured
PASS  Hypothesis properties exist
PASS  dictConfig logging
PASS  argparse subcommands
PASS  .gz gives the same report as plain text
PASS  CSV / JSON / SQLite hold every record


PASS  exit codes 0 / 1 / 2
13/13 checks passed


**Workflow checks:** where it runs and how it publishes.

In [513]:
run_checks(
    [
        (
            "workflow working directory",
            lambda: workflow["defaults"]["run"]["working-directory"],
            "projects/log-analyzer",
        ),
        ("publish job permissions", lambda: publish_job["permissions"], {"id-token": "write"}),
        ("publish environment", lambda: publish_job["environment"]["name"], "testpypi"),
        (
            "trusted-publishing action",
            lambda: publish_job["steps"][-1]["uses"],
            "pypa/gh-action-pypi-publish@release/v1",
        ),
        ("publish only on tags or manual runs", lambda: publish_job["if"], publish_condition),
    ]
)

PASS  workflow working directory
PASS  publish job permissions
PASS  publish environment
PASS  trusted-publishing action
PASS  publish only on tags or manual runs
5/5 checks passed


### How to publish

The notebook built and checked everything; the upload itself happens on GitHub.
These are the steps you do yourself, once:

1. **Create a TestPyPI account** at <https://test.pypi.org/account/register/>. It is
   separate from a PyPI account. Verify the email address and turn on two-factor
   authentication.
2. **Add a pending trusted publisher.** On TestPyPI: *Your account → Publishing →
   Add a new pending publisher → GitHub*, with:

   | Field | Value |
   | --- | --- |
   | PyPI project name | `log-analyzer-psbcg433` |
   | Owner | `psbcg433` |
   | Repository name | `python-atlas` |
   | Workflow name | `log-analyzer.yml` |
   | Environment name | `testpypi` |

   "Pending" because the project does not exist yet: the first successful upload
   creates it and turns the pending publisher into a normal one.
3. **Create the `testpypi` environment** on GitHub: *repository Settings →
   Environments → New environment → `testpypi`*. GitHub would create it on first use,
   but creating it yourself lets you add protection, for example *Deployment
   branches and tags → Selected → tag rule `log-analyzer-v*`*, or a required
   reviewer.
4. **Push the commits:** `git push origin main`. The workflow's test and build jobs
   run (publish is skipped); check that the *Actions* tab is green.
5. **Push the release tag.** Annotated tags are pushed explicitly:

   ```bash
   git tag -a log-analyzer-v0.1.0 -m "log-analyzer 0.1.0"
   git push origin log-analyzer-v0.1.0
   ```

   test → build (the tag must equal `version = "0.1.0"`) → publish.
6. **Check the result** at <https://test.pypi.org/project/log-analyzer-psbcg433/>,
   then install it from there:

   ```bash
   uvx --index-url https://test.pypi.org/simple/ --from log-analyzer-psbcg433 log-analyzer --version
   ```

> **Note:** a version can be uploaded only once, even if you delete it later. For the
> next release, bump the version (`uv version --bump patch` in
> `projects/log-analyzer/`), commit, and tag `log-analyzer-v0.1.1`. Pushing the
> course's own `phase-4` tag never publishes: it does not match `log-analyzer-v*`.

### Phase 4 review — 20 predict-the-output questions

Twenty short snippets, at least three from every unit of the phase, several of them
using the package you just built. For each one, decide what it prints **before**
opening the answer, then run the proof cell. If a question surprises you, go back to
the unit it names.

| Unit | Questions |
| --- | --- |
| 4.1 | 4 |
| 4.2 | 3 |
| 4.3 | 3 |
| 4.4 | 3 |
| 4.5 | 4 |
| 4.6 | 3 |

#### Question 1 — Unit 4.1

**Predict the output.**

```python
import re


def normalize(name: str) -> str:
    return re.sub(r"[-_.]+", "-", name).lower()


print(normalize("Log_Analyzer.psbcg433"), normalize("log--analyzer__PSBCG433"))
```

<details><summary>Answer</summary>

`log-analyzer-psbcg433 log-analyzer-psbcg433`. This is the PEP 503 normalization the
index applies: runs of `-`, `_` and `.` become one `-`, and case is ignored. Both
spellings name **the same project**, so nobody can register a look-alike that differs
only in separators. File names use the other spelling: the wheel is
`log_analyzer_psbcg433-0.1.0-py3-none-any.whl`.

</details>

Run the next cell to check your prediction.

In [514]:
import re


def normalize(name: str) -> str:
    return re.sub(r"[-_.]+", "-", name).lower()


print(normalize("Log_Analyzer.psbcg433"), normalize("log--analyzer__PSBCG433"))

log-analyzer-psbcg433 log-analyzer-psbcg433


#### Question 2 — Unit 4.1

**Predict the output.**

```python
from packaging.version import Version

releases = ["0.1.0", "0.1.0rc1", "0.1.0.post1", "0.1.0.dev2", "0.1.0a1"]
print(sorted(releases, key=Version))
```

<details><summary>Answer</summary>

`['0.1.0.dev2', '0.1.0a1', '0.1.0rc1', '0.1.0', '0.1.0.post1']`. PEP 440 order:
development releases, then pre-releases (alpha, beta, release candidate), then the
final release, then post-releases. A plain string sort would put `0.1.0` first.
`>=0.1.0` excludes the dev and pre-releases unless you opt in.

</details>

Run the next cell to check your prediction.

In [515]:
from packaging.version import Version

releases = ["0.1.0", "0.1.0rc1", "0.1.0.post1", "0.1.0.dev2", "0.1.0a1"]
print(sorted(releases, key=Version))

['0.1.0.dev2', '0.1.0a1', '0.1.0rc1', '0.1.0', '0.1.0.post1']


#### Question 3 — Unit 4.1

**Predict the output.**

```python
from importlib.metadata import PackageNotFoundError, version

for name in ("pytest", "log-analyzer-psbcg433"):
    try:
        version(name)
        print(f"{name}: installed here")
    except PackageNotFoundError as error:
        print(f"{name}: {type(error).__name__}")
```

<details><summary>Answer</summary>

```text
pytest: installed here
log-analyzer-psbcg433: PackageNotFoundError
```

`importlib.metadata` reads the `.dist-info` folders of the **running** environment.
The kernel runs in the course's `.venv`; the package is installed only in its own
`projects/log-analyzer/.venv`. Being importable (the setup cell put `src/` on
`sys.path`) is not the same as being installed, which is why `log_analyzer.__version__`
showed `0.0.0+unknown` in Step 4.

</details>

Run the next cell to check your prediction.

In [516]:
from importlib.metadata import PackageNotFoundError, version

for name in ("pytest", "log-analyzer-psbcg433"):
    try:
        version(name)
        print(f"{name}: installed here")
    except PackageNotFoundError as error:
        print(f"{name}: {type(error).__name__}")

pytest: installed here
log-analyzer-psbcg433: PackageNotFoundError


#### Question 4 — Unit 4.1

**Predict the output.**

```python
wheel_name = "log_analyzer_psbcg433-0.1.0-py3-none-any.whl"
print(wheel_name.removesuffix(".whl").split("-"))
```

<details><summary>Answer</summary>

`['log_analyzer_psbcg433', '0.1.0', 'py3', 'none', 'any']`: name, version, then the
three-part tag. `py3` = any Python 3, `none` = no compiled ABI, `any` = any platform. A
pure-Python wheel installs everywhere, which is why one wheel is enough here, while a
package with C extensions publishes one wheel per platform. Dashes inside the name
become underscores precisely so this split works.

</details>

Run the next cell to check your prediction.

In [517]:
wheel_name = "log_analyzer_psbcg433-0.1.0-py3-none-any.whl"
print(wheel_name.removesuffix(".whl").split("-"))

['log_analyzer_psbcg433', '0.1.0', 'py3', 'none', 'any']


#### Question 5 — Unit 4.2

**Predict the output.**

```python
from log_analyzer.models import AccessRecord

record = AccessRecord("2026-11-06T09:00:00Z", "10.0.0.1", "GET", "/", "200", 5)
print(repr(record.status), record.status == 200)
```

<details><summary>Answer</summary>

`'200' False`. Annotations are **not enforced at runtime**: `NamedTuple` stores
whatever it gets. `mypy --strict` would reject this call
(`Argument 5 has incompatible type "str"; expected "int"`), which is exactly why the
package runs mypy in CI. Runtime validation is a different tool (question 18).

</details>

Run the next cell to check your prediction.

In [518]:
from log_analyzer.models import AccessRecord

record = AccessRecord("2026-11-06T09:00:00Z", "10.0.0.1", "GET", "/", "200", 5)
print(repr(record.status), record.status == 200)

'200' False


#### Question 6 — Unit 4.2

**Predict the output.**

```python
from log_analyzer.report import EndpointStats

row = {"endpoint": "GET /health", "requests": 2, "p95_ms": 6}
try:
    print(isinstance(row, EndpointStats))
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
```

<details><summary>Answer</summary>

`TypeError: TypedDict does not support instance and class checks`. A `TypedDict`
describes the *shape* of a plain `dict` for the type checker; at runtime the row is just
a `dict` and there is no class to check against. Narrow with a `TypeGuard`/`TypeIs`
function or validate with Pydantic instead.

</details>

Run the next cell to check your prediction.

In [519]:
from log_analyzer.report import EndpointStats

row = {"endpoint": "GET /health", "requests": 2, "p95_ms": 6}
try:
    print(isinstance(row, EndpointStats))
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

TypeError: TypedDict does not support instance and class checks


#### Question 7 — Unit 4.2

**Predict the output.**

```python
from log_analyzer.export import ExportFormat

print(type(ExportFormat).__name__, ExportFormat.__value__)
```

<details><summary>Answer</summary>

`TypeAliasType typing.Literal['csv', 'json', 'sqlite']`. The `type` statement
(*New in 3.12*, PEP 695) creates a `TypeAliasType` object, and its value is evaluated
**lazily**, when `__value__` is first read. That is why `requires-python` is `>=3.12`:
on 3.11 `export.py` would be a `SyntaxError`.

</details>

Run the next cell to check your prediction.

In [520]:
from log_analyzer.export import ExportFormat

print(type(ExportFormat).__name__, ExportFormat.__value__)

TypeAliasType typing.Literal['csv', 'json', 'sqlite']


#### Question 8 — Unit 4.3

**Predict the output.**

```python
import logging.config

from log_analyzer.logging_config import build_logging_config

logging.config.dictConfig(build_logging_config("INFO"))
child = logging.getLogger("log_analyzer.export")
print(child.level, child.getEffectiveLevel(), child.isEnabledFor(logging.DEBUG))
```

<details><summary>Answer</summary>

`0 20 False`. `dictConfig` set the level on `log_analyzer` only; the child's own level
stays `NOTSET` (0), so its **effective** level is inherited from the nearest ancestor
with a level: INFO (20). DEBUG is below that, so `logger.debug(...)` calls in
`export.py` return almost immediately. This is why each module only needs
`getLogger(__name__)`.

</details>

Run the next cell to check your prediction.

In [521]:
import logging
import logging.config

from log_analyzer.logging_config import LOGGER_NAME, build_logging_config

logging.config.dictConfig(build_logging_config("INFO"))
child = logging.getLogger("log_analyzer.export")
print(child.level, child.getEffectiveLevel(), child.isEnabledFor(logging.DEBUG))

package_logger = logging.getLogger(LOGGER_NAME)  # restore the library default
for handler in package_logger.handlers[:]:
    package_logger.removeHandler(handler)
package_logger.addHandler(logging.NullHandler())
package_logger.setLevel(logging.NOTSET)
package_logger.propagate = True

0 20 False


#### Question 9 — Unit 4.3

**Predict the output.**

```python
import logging

# The root logger has no handlers (a script that never configured logging).
library_logger = logging.getLogger("demo.library")
library_logger.addHandler(logging.NullHandler())
library_logger.warning("disk almost full (library)")
logging.getLogger("demo.script").warning("disk almost full (script)")
```

<details><summary>Answer</summary>

Only `disk almost full (script)` appears (on stderr). When no handler at all is found
on the way up to the root, `logging` falls back to `logging.lastResort`, which prints
WARNING and above to stderr. The library logger *has* a handler, a `NullHandler` that
discards everything, so the fallback never kicks in. That is the recommended set-up for
a library: silent until the application configures logging. The proof cell empties the
root logger's handlers first (and restores them), because earlier cells in this
notebook may have added some.

</details>

Run the next cell to check your prediction.

In [522]:
import logging

root_logger = logging.getLogger()
saved_handlers = root_logger.handlers[:]
root_logger.handlers.clear()  # as in a script that never configured logging
try:
    library_logger = logging.getLogger("demo.library")
    library_logger.addHandler(logging.NullHandler())
    library_logger.warning("disk almost full (library)")
    logging.getLogger("demo.script").warning("disk almost full (script)")
finally:
    root_logger.handlers[:] = saved_handlers

disk almost full (script)


#### Question 10 — Unit 4.3

**Predict the output.**

```python
from hypothesis import given, settings
from hypothesis import strategies as st

calls: list[bool] = []


@settings(max_examples=100, database=None)
@given(st.booleans())
def test_flag(flag: bool) -> None:
    calls.append(flag)


test_flag()
print(len(calls), sorted(set(calls)))
```

<details><summary>Answer</summary>

`2 [False, True]`, not 100. `max_examples` is an upper bound: Hypothesis notices that
`st.booleans()` has only two possible values, tries both, and stops because the search
space is exhausted. Calling a `@given` test directly runs the whole search, as pytest
does.

</details>

Run the next cell to check your prediction.

In [523]:
from hypothesis import given, settings
from hypothesis import strategies as st

calls: list[bool] = []


@settings(max_examples=100, database=None)
@given(st.booleans())
def test_flag(flag: bool) -> None:
    calls.append(flag)


test_flag()
print(len(calls), sorted(set(calls)))

2 [False, True]


#### Question 11 — Unit 4.4

**Predict the output.**

```python
from log_analyzer.cli import build_parser

args = build_parser().parse_args(["-vv", "report", "access.log", "--top", "3"])
print(args.verbose, args.quiet, args.command, args.top, type(args.log).__name__)
```

<details><summary>Answer</summary>

`2 False report 3 PosixPath`. `action="count"` counts repeated flags (`-vv` → 2),
`store_true` defaults to `False`, `dest="command"` records which subparser ran, and
`type=` converts the strings (`positive_int`, `Path`). The class is `WindowsPath` on
Windows.

</details>

Run the next cell to check your prediction.

In [524]:
from log_analyzer.cli import build_parser

args = build_parser().parse_args(["-vv", "report", "access.log", "--top", "3"])
print(args.verbose, args.quiet, args.command, args.top, type(args.log).__name__)

2 False report 3 PosixPath


#### Question 12 — Unit 4.4

**Predict the output.**

```python
from log_analyzer.analysis import ID_SEGMENT

print(ID_SEGMENT.sub("/{id}", "/api/v2/orders/42/items/7"))
print(ID_SEGMENT.sub("/{id}", "/reports/2026.csv"))
```

<details><summary>Answer</summary>

```text
/api/v2/orders/{id}/items/{id}
/reports/2026.csv
```

The pattern is `/\d+(?=/|$)`: a slash, digits, then a **lookahead** for another slash
or the end of the string. `v2` does not start with a digit after the slash, and
`2026.csv` has `.` after the digits, so neither is an id. The lookahead checks the next
character without consuming it, so `/42/items` keeps its slash for the next match.

</details>

Run the next cell to check your prediction.

In [525]:
from log_analyzer.analysis import ID_SEGMENT

print(ID_SEGMENT.sub("/{id}", "/api/v2/orders/42/items/7"))
print(ID_SEGMENT.sub("/{id}", "/reports/2026.csv"))

/api/v2/orders/{id}/items/{id}
/reports/2026.csv


#### Question 13 — Unit 4.4

**Predict the output.**

```python
import datetime as dt

moment = dt.datetime(999, 12, 31, 23, 59)
print(moment.strftime("%Y-%m-%d"), moment.isoformat(timespec="seconds"))
```

<details><summary>Answer</summary>

On 3.15: `0999-12-31 0999-12-31T23:59:00`. On 3.12 and 3.13 (Linux, glibc) the first
part is `999-12-31`: `strftime` delegates to the C library, and before 3.14 Python did
not pad `%Y` to four digits there. `isoformat()` is Python's own code and always pads.
This is the bug Hypothesis found in the CI matrix (Step 6).

</details>

Run the next cell to check your prediction.

In [526]:
import datetime as dt

moment = dt.datetime(999, 12, 31, 23, 59)
print(moment.strftime("%Y-%m-%d"), moment.isoformat(timespec="seconds"))

0999-12-31 0999-12-31T23:59:00


#### Question 14 — Unit 4.5

**Predict the output.**

```python
import json

status_counts = {200: 15690, 404: 580}
round_trip = json.loads(json.dumps(status_counts))
print(round_trip, round_trip == status_counts)
```

<details><summary>Answer</summary>

`{'200': 15690, '404': 580} False`. JSON object keys are always strings, so
`json.dumps` converts the `int` keys and `json.loads` cannot know they were numbers. A
JSON round trip is not an identity. `summary_document` therefore converts the status
codes to strings itself, and its `TypedDict` says `dict[str, int]`.

</details>

Run the next cell to check your prediction.

In [527]:
import json

status_counts = {200: 15690, 404: 580}
round_trip = json.loads(json.dumps(status_counts))
print(round_trip, round_trip == status_counts)

{'200': 15690, '404': 580} False


#### Question 15 — Unit 4.5

**Predict the output.**

```python
import csv
import io

buffer = io.StringIO()
csv.writer(buffer).writerow(["GET", "/search?q=a,b", 200])
print(repr(buffer.getvalue()))
```

<details><summary>Answer</summary>

`'GET,"/search?q=a,b",200\r\n'`. The field containing a comma is quoted, and the row
ends with `\r\n` (RFC 4180), not `\n`. That is why CSV files must be opened with
`newline=""`: otherwise text mode on Windows would turn `\n` into `\r\n` and the file
would end every row with `\r\r\n`.

</details>

Run the next cell to check your prediction.

In [528]:
import csv
import io

buffer = io.StringIO()
csv.writer(buffer).writerow(["GET", "/search?q=a,b", 200])
print(repr(buffer.getvalue()))

'GET,"/search?q=a,b",200\r\n'


#### Question 16 — Unit 4.5

**Predict the output.**

```python
import sqlite3

connection = sqlite3.connect(":memory:")
with connection:
    connection.execute("CREATE TABLE hits (n INTEGER)")
    connection.execute("INSERT INTO hits VALUES (7)")
print(connection.execute("SELECT n * 2, n / 2 FROM hits").fetchone())
connection.close()
```

<details><summary>Answer</summary>

`(14, 3)`. Two surprises: `with connection:` **commits** (or rolls back) but does *not*
close the connection, so the query after the block works; and SQLite divides two
integers as integers (`7 / 2` is `3`). `write_sqlite` uses `closing(...)` for the close
and `with connection:` for the transaction.

</details>

Run the next cell to check your prediction.

In [529]:
import sqlite3

connection = sqlite3.connect(":memory:")
with connection:
    connection.execute("CREATE TABLE hits (n INTEGER)")
    connection.execute("INSERT INTO hits VALUES (7)")
print(connection.execute("SELECT n * 2, n / 2 FROM hits").fetchone())
connection.close()

(14, 3)


#### Question 17 — Unit 4.5

**Predict the output.**

```python
import gzip

first = gzip.compress(b"GET /health", mtime=0)
second = gzip.compress(b"GET /health", mtime=0)
print(first[:2], first == second)
```

<details><summary>Answer</summary>

`b'\x1f\x8b' True`. Every gzip stream starts with the magic bytes `1f 8b`, which is how
`is_gzip_file` recognizes compressed logs whatever their name. The header also stores a
modification time; with the default (`mtime=None`, the current time) two compressions of
the same data can differ. Fixing it at 0 makes the output reproducible, the trick
`generate.py` uses.

</details>

Run the next cell to check your prediction.

In [530]:
import gzip

first = gzip.compress(b"GET /health", mtime=0)
second = gzip.compress(b"GET /health", mtime=0)
print(first[:2], first == second)

b'\x1f\x8b' True


#### Question 18 — Unit 4.6

**Predict the output.**

```python
from pydantic import BaseModel


class Hit(BaseModel):
    status: int
    latency_ms: int


hit = Hit(status="200", latency_ms=23.0)
print(repr(hit.status), repr(hit.latency_ms))
```

<details><summary>Answer</summary>

`200 23`. Unlike `NamedTuple` (question 5), Pydantic **validates and converts** at
runtime. In its default *lax* mode, the string `"200"` and the float `23.0` (no
fractional part) are accepted and converted to `int`; `23.5` or `"abc"` would raise
`ValidationError`.

</details>

Run the next cell to check your prediction.

In [531]:
from pydantic import BaseModel


class Hit(BaseModel):
    status: int
    latency_ms: int


hit = Hit(status="200", latency_ms=23.0)
print(repr(hit.status), repr(hit.latency_ms))

200 23


#### Question 19 — Unit 4.6

**Predict the output.**

```python
from pydantic import BaseModel, ConfigDict, ValidationError


class StrictHit(BaseModel):
    model_config = ConfigDict(strict=True)
    status: int
    latency_ms: int


try:
    StrictHit(status="200", latency_ms=23.0)
except ValidationError as error:
    print(error.error_count(), [detail["type"] for detail in error.errors()])
```

<details><summary>Answer</summary>

`2 ['int_type', 'int_type']`. Strict mode turns the conversions off: only real `int`
values pass. Pydantic reports **every** failing field in one `ValidationError`, not
just the first: the same "collect everything" idea as the log analyzer's strict mode
and its `ExceptionGroup`.

</details>

Run the next cell to check your prediction.

In [532]:
from pydantic import BaseModel, ConfigDict, ValidationError


class StrictHit(BaseModel):
    model_config = ConfigDict(strict=True)
    status: int
    latency_ms: int


try:
    StrictHit(status="200", latency_ms=23.0)
except ValidationError as error:
    print(error.error_count(), [detail["type"] for detail in error.errors()])

2 ['int_type', 'int_type']


#### Question 20 — Unit 4.6

**Predict the output.**

```python
import json

import orjson

document = {"status_counts": {"200": 3}, "p95_latency_ms": None}
compact = orjson.dumps(document)
print(type(compact).__name__, compact == json.dumps(document).encode())
```

<details><summary>Answer</summary>

`bytes False`. `orjson.dumps` returns **bytes** (UTF-8), not `str`. Even after encoding,
the outputs differ: `json.dumps` puts a space after `:` and `,` by default, while orjson
writes the compact form `{"status_counts":{"200":3},"p95_latency_ms":null}`. Same data,
different bytes, so never compare JSON documents as text.

</details>

Run the next cell to check your prediction.

In [533]:
import json

import orjson

document = {"status_counts": {"200": 3}, "p95_latency_ms": None}
compact = orjson.dumps(document)
print(type(compact).__name__, compact == json.dumps(document).encode())

bytes False


### Phase 4 complete

You took notebook code and shipped it: a `src/`-layout package with its own lockfile,
a typed public API that `mypy --strict` and its users' type checkers can rely on, a
pytest + Hypothesis suite that fails the build under 90% coverage, logging that
stays silent in a library and configurable in a CLI, an `argparse` tool with
subcommands, gzip input and three export formats, and a CI pipeline that tests on
four Python versions and publishes with short-lived OIDC credentials instead of
stored secrets.

**Next: Phase 5 — Concurrency, Async and Performance.** Threads, processes,
`asyncio`, the free-threaded build and profiling. `log-analyzer generate` can produce
logs of any size, which makes the analyzer a ready-made benchmark: is parsing CPU-bound
or I/O-bound, and what does a process pool do to it?